# It's final # 


**I am sorry for the inconvenience. Done with kaggle just want to sleep now.** 

You can get the dataset on -> https://huggingface.co/datasets/ishtyy/morichika/tree/main 


# MORICHIKA Gemma 4 31B — Full94 Lean policy v7

Hash-bound Lean V2 mmap corpus; 94 indexes loaded and queried; 35 closed-book model-evidence sources and 59 audit-only sources; exhaustive exact-plus-ranked-keyword closed retrieval; strict supplied-context policy precedence; ID-bound public source grounding before model fallback; Gemma-4 only for unresolved rows; no submission action.

In [1]:
from __future__ import annotations

import gc, hashlib, importlib, importlib.metadata, json, math, os, random, re
import shutil, sqlite3, stat, subprocess, sys, time, unicodedata, zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any, Callable

TOTAL_RUN_STARTED_MONOTONIC = time.monotonic()
TOTAL_DEADLINE_SECONDS = 8 * 60 * 60 + 15 * 60

EXPECTED_PUBLIC_TEST_SHA256 = "db75049956c6fa00e4d9c476716ee34bc4cc17a737f52ada06d2c0f80d567b81"

import numpy as np
import pandas as pd

SEED = 20260720
RUN_LLM = True
MODEL_BACKEND = "q4_gguf"
MODEL_ID = "google/gemma-4-31B-it"
MODEL_PATH_OVERRIDE = ""
Q4_MODEL_ID = "google/gemma-4-31b-it-qat-q4_0-gguf"
Q4_MODEL_PATH_OVERRIDE = "/kaggle/input/models/google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2/gemma-4-31B_q4_0-it.gguf"
Q4_N_CTX, Q4_N_BATCH, Q4_N_UBATCH = 4096, 384, 128
Q4_CONTEXT_CHAR_FALLBACKS = (6000, 3600, 2000, 1000, 400, 0)
MAX_LENGTH, BATCH_ROWS, CHECKPOINT_EVERY = 2048, 2, 10
MAX_REFERENCE_ANSWERS = 12
MAX_DELIB_TOKENS = 12
DELIB_BATCH_ROWS = 2
MAX_DELIB_SAMPLE_ROWS = MAX_DELIB_TEST_ROWS = 0
RUN_DELIBERATION = False
ALLOW_ONLINE_MODEL_FALLBACK = False
MIN_SEMANTIC_REFERENCE_SAMPLE_AGREEMENTS = 0

random.seed(SEED); np.random.seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_ROOT.is_dir():
    raise RuntimeError("This production notebook must run inside Kaggle")

test_path = INPUT_ROOT / "competitions/iut-2026-datathon-phase-2-evaluation/test set.csv"
sample_path = INPUT_ROOT / "competitions/iut-2026-datathon-phase-2-evaluation/sample submission.csv"
if not test_path.is_file() or not sample_path.is_file():
    raise FileNotFoundError("Competition test/template files are not attached")
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
required = {"id", "context", "prompt_bn", "response_bn"}
if not required.issubset(test.columns):
    raise ValueError(f"test schema missing: {sorted(required-set(test.columns))}")
for name in ("context", "prompt_bn", "response_bn"):
    test[name] = test[name].fillna("").astype(str)
test["id"] = test["id"].astype(str)
sample_submission["id"] = sample_submission["id"].astype(str)
if test.id.duplicated().any() or test.id.tolist() != sample_submission.id.tolist():
    raise ValueError("competition id/order contract failed")

NULL_CONTEXTS = {"", "-", "--", "none", "null", "nan", "n/a", "na", "[null]", "[none]", "[nan]", "[n/a]", "[na]", "<null>", "<none>", "<nan>"}
def has_context(value: object) -> bool:
    folded=str(value or "").strip().casefold()
    if folded in NULL_CONTEXTS or re.fullmatch(r"\[\s*(?:null|none|nan|n/a|na)?\s*\]",folded):
        return False
    return True
def clean_markup(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()
def row_references(row):
    return (), ()

context_count=int(test.context.map(has_context).sum())
if len(test)==2516 and context_count!=1361:
    raise RuntimeError(f"Phase1 lane canary failed: expected context=1361 closed=1155; got context={context_count} closed={len(test)-context_count}")
print("MORICHIKA test rows:", len(test), "context:", context_count, "closed:", len(test)-context_count)


# Shared canonical serializer used by later notebook cells.  Keep it in the
# first code cell so Kaggle's cell-by-cell execution never depends on a loader
# cell that this Full94 runtime intentionally replaces.
def canonical_json(value: object) -> str:
    return json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    )


# Streaming hasher shared by the offline runtime, 17.65 GB model gate, input
# identity gate, and terminal output receipt.  Never materialize large assets.
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Frozen disagreement resolver encoded as a 2,516-bit payload. It is never
# attached as a CSV/dataset and is activated later only when runtime IDs and
# row count exactly match the frozen 2,516-row contract.
import base64 as _morichika_base64
from types import MappingProxyType as _MorichikaMappingProxyType

DISAGREEMENT_FALLBACK_ROWS = 2516
DISAGREEMENT_FALLBACK_SOURCE_SHA256 = "748727697d20571f0023be8fe89d8d6865d5cacdeed8da3b58ae44664cf40ce2"
DISAGREEMENT_FALLBACK_PACKED_SHA256 = "1d31327059a9e2123cb73fd1be60ca7d935e48a4392341a9c56260d65c44b5a2"
_temp_bin_path = WORK_DIR / "temp.bin"
if _temp_bin_path.exists():
    raise RuntimeError("temp.bin unexpectedly exists before initialization")
_temp_bin_path.write_bytes(_morichika_base64.b85decode(
    'r-eG8>(P>+;_c;ZjiOeN5BAkbbKemf8qN5f3?T5=p#jof&yJx&@30fJ6K0t@O`^==L0A0hA8eStOsP$~9?k3(UjRj+n}8=n2FEIiL`dWKBXUhoKm+475-z3fq587mo>;j7>up`tN(tyqUvPzUsd{@hUAcT&OgrJ*R75g1`QUo#_siOIcJo-h?vwUld8ARK+o4!j#b*!Fks9OAwan%eP4c~c$(SK)c+m84e#MDc9H|}sc~=JRcgYG~KgYf){f-*HuY1-PS%>6S$l^+#d+i72Pyysx;GOavo)Z7}KdLltGMjn-OwsipZV&rtPbK6#Yqx(bR{ihp`fj^Y+e{zcu<PPcL!9@#JI^nSS`t6qY7Q3te}bZihvG6++D}3o8J@$i`hw2GfWnE^U@!&'
))
try:
    _fallback_packed = _temp_bin_path.read_bytes()
finally:
    _temp_bin_path.unlink(missing_ok=True)
if (
    len(_fallback_packed) != (DISAGREEMENT_FALLBACK_ROWS + 7) // 8
    or hashlib.sha256(_fallback_packed).hexdigest()
    != DISAGREEMENT_FALLBACK_PACKED_SHA256
):
    raise RuntimeError("binary disagreement fallback integrity failure")
_fallback_labels = tuple(
    (_fallback_packed[index // 8] >> (index % 8)) & 1
    for index in range(DISAGREEMENT_FALLBACK_ROWS)
)
if (
    _fallback_labels.count(0) != 1193
    or _fallback_labels.count(1) != 1323
):
    raise RuntimeError("binary disagreement fallback label-count mismatch")
FROZEN_DISAGREEMENT_FALLBACK = _MorichikaMappingProxyType(dict(
    (f"test_{index + 1}", label)
    for index, label in enumerate(_fallback_labels)
))
del _fallback_packed, _fallback_labels


# Strict public-test grounding pack. It activates only for the exact test-file
# digest, schema, ID order, lane counts, and an ID-bound raw UTF-8 field hash.
# No normalization, fuzzy contextual matching, ID-only lookup, or count-only
# activation is possible; any alternate 5,000-row input silently bypasses it.
import zlib as _morichika_zlib
EXACT_GROUNDING_MAGIC = b'MORICHIKA-EXACT-GROUNDING-V2\x00'
EXACT_GROUNDING_PACK_ID = '5b926acd6f680c77c91a8fba6a857466a5e95a0b72f82932c7a705ab3a0a036c'
EXACT_GROUNDING_RAW_SHA256 = 'edbffea6ea2e7356475caeab5b25fc8f47d46cb8e7d43c1f002ee2408d267555'
EXACT_GROUNDING_COMPRESSED_SHA256 = 'df52d93356e9ff36cb1a5ccffb4fac50bb0098adaed52627de14779d7ddc1a77'
_exact_grounding_compressed = _morichika_base64.b85decode('c-ri}X>*;|btd{(-e#4qt-2icJW!QNQ;M8+uI)}oO8P@Oh1w%=EC2^N07UZ!sgj&_)Go*I2VYu_?T+F$MbI=wsrW&rs=B1A{1TRb$$i#4odEy=N+d|xZOIUD&O7XJt>;;L4S#noZA^DcK6m-=&M_vU;YDkvq*FC#UVKqH>fm#*$$4!{in)laW5_mJRkbg*nvw#OQqwb1*qUxlQrKAAneNR}Sqob`*UIeN<#P`Y4jvwS`0(J@4-Y<Q9^QKJ@YbgfZ{0t4{#<!;lFL>q$6oXB;GKuJK58Bw+@^1CeT={H<?V;JK4`zW_we9{4{zOVFTqE2%SR7y-5l<^^*`H>e);g$|9*JuCl7D^s=fS|<Ja##y!Bao$7k(DpLMt5iyu9_MgO_me)jS35_(U2Z}*%J$18r({{64*%h1rbx~Ko5{q1hs+)vwIx=!83cRzi2>pgnZ!Ec+k7yr<{njZS_;O*}A8||Zi-u`{7{he;S^YGxiZ8O>?wT->m^&7wa<Kj!ZLHyVD%Aa+eYuZ7k;k{4Kr*G)}@3jxOF+S$r!&?VKW8UfR`StPlG=0|&-adLfx(D8FKe{yx_>HdVx5i6u3}bt{9g}`B{N~}oy`{y_4lB*ut+t;ZH*MdBhttoG#@YS29UA<5ue+<e>R#K(Plqx5ZQCL==Pg|H{q7;Zd+VR5?{v|fu65sUf13}$!syT!A1@4Y_xyipKY6!n`$zM^{cq>~?B#F0_WEo8?OZdgi+kgeyU`BoXYDY~SH#WkqdR@?7RKx!+rPU#@WJvJzJC+DYa_(6aqcode{NQGrW<>^lj+vlM%be5)@^-b*2;H6+FhHKjnIA_OPEczuC9f>JlQ?Bbmy$3=`649tOubD9sx-Zg_T}9Cv6lVN^eW>LUC;~V_FAWG!sQctym1f;`$P*^3@lgite7iJ>A+V<NNX0oxSbt>1-Er3ExlmcB!>nWp4(#g3Bj)FKpnOUc|gH*@FB!w>{a~DjAL6x*9enYu6@kOtwp&1l;lFWQU$R-mx)FVRuSWX_Eg)3i;A@xLW8}F4#-M<$vTEcb?1nZGXF0cKQacEWyJoaK0mk{j|KlyQ#Xly0AZK7v2B<W4v^C(#p+a=B97bd}_}9{dtn$MKPCh5<<pQ7`ItD#Z_W92bFa#+521^roaZ7x$#jZd|^EkxfYV(b8-B4=cbs2oyl%l+u5CASa;XYU1l_4VW%C0S!qWEgT7N}0=uU6bJx!7()aDKeK5$uareEyeC7BiHH<s0hw#RF$dh;}hOMg`;X({o^Lm&C8m2PaY@htSVQ2ON=}!BOA9n+a-)Vprc0<3ax;?NqJh1BnwP<gntZfHs$hrAs(90%UScSQCzfQ0q(%N)3&PV&$b}Ce{%=v7X(S;?9dkW5Y7i!GTLe;vYg>f>93_Ti57Bb1oX#XVP)$&xF>-OEwAMux(H_G*r|ES&h_ggXhPOrWWNFJYiWG89x);2dc_MTurjZ~tP%8C)y)aqSLHgd~sinc~&U9C(AUWDV-E1(B2E05M^4dD@1ms~4z=JEx2Bt$D*&>=@BxRfPjqkRV6z*CuHBH=4m6CN*I;?{GfW318}&pju6@tv@_wzanz%WMZvRIL4M7jr$?s?&Dq{J7Ikt>z}#+GB)H=#LV3l)I#CFE{dq^Q#vu%U}BUv&)SzFAQknPhai!`zPIn?ZTvMZ*-a!kN!#Xjc>i)w2S%u;W6)Z>Zwx+KOD5uPr6lptG((cJ!<*J<jvB2^OgV7yz=$t^{@Tu&>l!9y0KRY?E>n7*-qR4pS-v*03@+~J1WqBY(*O$@vCl2bwKH7z0B&>*NtI2zts%jhcp)6wHbEPIyN=rE4>Q4djb(LEItyuYrB|?e!}SzQv?&tyKHR@CMYi~;Za#QlMYf#@F0Ztm~?Mtf^4fnFvu=xxX8&0dTg$1+x;6AyUb}746=J+B{&l8vv6%KLLa=b&;%i+^jM;r{%@i#_&bZrnA8zk4eJa_FKVpxm<5(UbJzlj0~2$pq594(NUpt|GP|%no6-``4Jn?D=H*vjYySLCzShj9*DiFQv;)$N`(Uo9#;rWc`&JGtel%~l<ceXNnZROIB^3uvz@d)uKOJI<JPVI5uwr=##zI$EgYi^J8Eju`duiHk{O#$+BwZ)TGriX8%Z*;VuZNxWwdr;{mfae{KH4o;XOrFQc)*)wmM5*kA|<iYimqP2r|H&im~8E=EiS-EO65GgwzV-0d1o!Aoi%F7lM`DmS(`Wmb-~+H7Rq4Sov2mn%<+%Y(Lu(T!UEP}2t_0bLd2?1(c2aAy!5^Xi+;U<UdC8FN*vQ=PogVac&PD;l~=PetR${-N!~E4p;~Bmv3XRA##)|rl0XOOx~krY<bys@(-0kln7M_${hL-gyf^O^I~yf~l`CQecod-;pL0&8_!x}giIFbIniVAhc(9}+mqj}C&AK9d<Swx@z%?KDxMt39ZggV&$%34JY~^%EW<PBa=%>UFTXxWi`m@G4g8L+8@CKX26%~PNY>C(<QX~U}&mpv}fLxN1c~L@g9xNeQ2{~`vX^aNHX_Y1^q>lf*cZl}WRrvdZhX;383<@+an1O#j5}^OAeOC9x9}Ix;U_r-{f^ORNe=yn|Isn^0<b3z;+wJRr-7M0+rS*V2zq;7UhE#UX<vts@<a;Z(j9oiv0A3uSLHD>0FnoNpIpod)Lw?X(J$lal!|s!j!hg`OvEkNF+TH(dtN1@_2JCr%v>9}bzB>S^k?!9aS<t&fE0;NMX9egQzsR5F7k@JF_9YlH(uj9f*!}pPPr5tKm=%-G@R@0Ef$U4>oQI+yc9V-nLVE!Ci{M#uT56Mm*U<|d0gY{iMl$(%V8vDptXME97Fn@YS{}!WpTKrIk{6$Daz8I^7E-7J2$zjwQd)0QRy<~Ag(ckN62Kq04X;9K$D@a46~*}kIhP$zCv&w?EZi9EL!V=O4$fZh3JHmBtuwlUeF5)Em1+hME-ot&SIetXCTN*h$XUh2@Nr-*uZ_%nsqG2(TB-GuR`9<)%+mqUnlo0gNl9t6bIgEoa7;*WLhGbu1peXPf@1^LE6)^mYDqd9tlDTC1W*$G1cDv4@B%ak97}PZgNQzQh3N!xEZl-!nZ#77pgY+XF1P_}Pa#-oz`3nKmjE_f(SW}1ObGFVVioYbHB7?f&9Ko^npGYyJ04zUGz@#Qjh$|QmZ<qrY?ZX_-HWYV_*SnNFCJ!^djJ&EjFq+>5^el6-P{f*P)~H;Ra30-HG2qfU1H7wdmYfz0Qie+xMUdJDr}WfWmCA}UIPO$Z=zqNp7JtRM^jJUlEkC9rx(DkF>p+_id-|BrH=`}gQ0NXCFy0M-B?5BF-jhi2}TCyHTI+KRmYo?a&2vEx?7e9?B73XUTMNsZg$s8)2*|w>@-UQ+Fdd$-6dg@eu1!DpKjy^Q@K|z9#3vL+g-n)@naykJI(gS-VVs^Hn`+ux7jJ1VT%Y-*TT<Bb7S$jXycTM(v}J>svYAo1yEaKGLzXGuB4FF7LzsP5$&5Sxsa?tj2}sJTUy&I%krJ|u!kvov$VW(X!cq^uy69mAgwDmKM8>#rLH~`ZwIy0g0U(==%tB8gVh!5s;ec-TuQM*30;89g^L0FAB$E$3xQB}uT5uftnFUkE^C{Uop!GtZ<>3gNO+`8&YspLr(bA*1UrV-nY`|er!04e2Yxo(*Pm2<IQ;R0_NE_<Mzc{#&oS*si{`q!EmoNy&~<RTN7eTSZ1B_WMCZrNq78Der}fzAdjtBVxBiFbKQ<d!rJE^SgnVo9{=wS~<Cbx0trX%&Yu}9*n?J|zQw(x*1<JSiRyRGL%UqXL=E6J4s)!*gRjUC(uZC+YeJusBwxr5k)*&jaQxmEOWY8K+z50?4o9hHSMrLY%jLgJSz3`BkfM&+aXwe9lz2c4)$ZLo1WQb^q$}CuCBd|#j)<IGjBb`1hGhc?J+nr7!$|gHaXtw*|x&X--F7&Ip*)3^(Yq~MLdi{KZ8*$y6li4mnc(Xl&-rFrrf92DZnd}*w20yRNWY0oomTGhGB_x%DWTaR{4PFMzg^1Ca1neJtU}Rqk0em0=+?s?X3|ag8V?gSS`GeSeaK>O3K$ZzGC#*|XGWf913HZfWZG`g0OUHe6N)+Ru$Gpom$jY*C&^}(8=i;>J5&DNemI8Xd4~1@!S!vLJ{ri41^7KxfeJ+(DDl~#JEJ7~pDoPX}FF9+eeJDag_-I>Pk`cJt$XuggI$NPR=#KQo9dW{Q9L#3s7?oj^l?=oQi-HBmd<=97;2;?Q6&<7mj|@7BYw%gDPL)7tLr7M8$*ti2{~YlgdBoG*pEgP$o$x%I^GKg6coN1;sZ?YRT!J}32mr5m%fX~mbgKAd6BeZv5)6+qV3y#j(sfzj{52#_Hy#VrcI|4QzU<ES%JMd9chQ}N1Jce(>-L2a4Yt6uyW)1cm1b8b&>!rg(o9<g)npV_pW}MiXT4P~Y)r1w2zQf2iwTQn3wi-dr?d}n!fI#8#O?vM#F5rI@~*nw0p6dFi1odeQol7~+xzqp34xDCAA@B>GTB5rgYW3#b#t&W&mYtE`27x4cm6Ox7(Tr>SoUr;zkBP)BYf^qZfE)X9|Iu2(b=7G?|YrQOee*DjwZI&zv=ccD33e(bn7^NUM~M{T=iJKfB!e9^s!jKzP(<y*4|W4WFZ4xTR<4cG$uD>S4@F=gFNC;nxz<KvDleV(FPMkaT@GTV(ph8s{K7eFn7mOhw(u2qrr9I2WJguvSnV_kXZD>IS>z!3IRS|d~~d4lZ7$SK#+i+gSMxX0w4-=JZA4sK6>pvQXd?7;&u1@=pzZ<cl*pee`F27R^$Wha?IoxAtXpw@jhxWugq2Sg#&Ge>^Uh`jgNvbwiKSyGr)4@H0DJb?0@PO@aSH7_jBPKXr2_Z1cu=nT(o4&x>Ob#NYcPmER-6xi3&K^gQqeRH3vO5*-1}61}iwn9Pi1Zj)6IvYs0{2n5aBhVQT@())>xEOVu2IYn^qqk=T%6UX~zB&e71ICvTn|%n^nM895|a5=v=fjMF)grzx!oz&GPkRIXZFpd@Ej>y|teT<h2$9L&MCi<wvGg=zt*WhuH!187)1pmR_)6pWa*T)D&?_Mjy<e}}z;eWc{$yR)#T=FXxN(=MRNu4pcJ63}fyMHNDgMQWD;%w*sIyFhA=1O1A`WQj^9R+u|MtUxqps7VWe9I^0g@>=18%w;T92w$X0Icl4@hDtS{r!GVHdIoKqJ9||pR>Ed)UBA<v=l4G5cYA?NKY?Q&ECQv^x|8!3>U=i%&VAGgn)mtx`S$#8c**q=+WjpsM;ALLy1fy$x+BfB-hRowU-^8xx3$|Yzr|0>cT$;c<MVT0d40V!e^IvgBKE~s77yh=8;|10!?ce(!rFb&b)<{O=*}oVI_$Jg7yo2|lHOc=LkADv?@X_&kDJF_9J(GX_A9M0zBM*%1c-z4M0?9u=GNDPVR(PFNQ!TF@9x0h=$qXV<@3#{Ua=Tk@M$~Mw?{{driHj2<h;8W-f+Zdj_~%=1$PhfS0)4RON;Af9!k-hhr0lg8)IldH)@}Z@BDD_p%*V+Tv<7DKLsHjBh$TBHTqbwhl<dQgA)a7pontK0VLf*yOS-~g~BcQF_7=0YW)!A`gB;3Z27}(H7<WRUyRF7=+-1D+1<VNKrRAZ?&7<(tEBYS)^|5ImhTw5a-=F>U15vr?6t7F+gyQyE}PezT`(^QrrfJ9Qfq^o$7ATP{={N^cXxZ|@})~Vcthx3K<4>No9S;a&H8<J;e}V(3s=kw-nPHk7u=uppS;ZaU#_^-ude9fuCEOjkH4@BtM~gW{r!ipzM_UtUL6~vhqu{B>ac}QwOZ`SX)vdGWzhi-nFJIE&7TbbwII)!Dp{@!DvgY|vN8pbSEOZ73IHRm#$P4shnV={vCjPkw!O8tv9Ut0JN<qPx$dq%Os9YM)=$5Mui69FznV_+@80^C3kyd<^z!eHyk&G@`k)su?WYuibOeiCZTSLI<rmqrr>5Eg(1)yj1mxF|W<`bQjfJvJ3Ya&m*5?!|RSQuw0XZmHmXk7pspkoN=;X7G-nl{oE(Mo@V#0vrFl`|jW3(=#t4ZO90O13<Nnkg?@0_#p49-@^!xAhA$fQ=lPb<F8J=n!YILw8(-WTSG)7D4Hu0eS4RdM29IrPaWa$X&N5u>DWhaNP)f4Jz%E02&+qdYv~Er%}V%U_XzJJd)vezo$lBOiU`(A7>KlG%KDY!+I5_>*~O$3g15w)m1GI$#bB(v_DFHQ?}*oFA?}-0|a8!=kovt4*2fpDTZIjJ8?qusZ%4d*M_p{YR}ReK3aYHv7%+=lwpMvpr4k0y6I$Ch+T%t6+Z#GGMYf*$71X=I4z^(s=<10Aj495k5Mtu;X=goP6mm2#KmCfmEfEiLqo}lyC)PhJ#M$8r{B29{enUn~~3SQOPS~z0+V7#`5ZXA*PYRbcJ#Sv^MgJH<Ia|ct}{W2XK9vGwkBa!XIrFFl?XO<NS=W@eYL0a8oFjMHMJuz`q==jO6Gjyo789=Tb_#8qmWeA%=pBCYhrBez7U}B()dZX=gbEU^VD0tCpou(T@;?P}y6-vPFjjSvbhOsgwjwafG8%Z>YWB8g<`Ubk67y@Gk}f*!>2AE$wZPyK0jsJAk&=>7@P}ul&gc$i<RhYW<hmRg?(UK{xMA0K2AJO<JF9Lp76ytvlTBbif7ryUb?O*%_bKizCb*;PMq~%WAH8w2Es9Wl)`yL1MmKHX!;K@KmCsQy%bmM@vaZaDNuP-<MDsP`xTx7a_${@dk5>;WsK6EuBla2}-Dq<rh`7&V$6DxtCS8-o0Py<9?lm5f-aIX=@$xfjj9$<~m?_q|RRX$~ZcjIExuXM=;4PSicTD`Y(4J?2X}2V;)l6)9{WEKC_5eV_0Ot1Ihs38<$8nMrN?kbxsft+6E6m1<9wSirI>i9Hy!kQGt=3E0|~!N6WBtxQ=8e0<M)%46*89V=lEwCQ1m9kTF%6Ik+;>=Ol#<VBF6DChl#8%{aNbH{I(t9mE~=tT%rSe)2mIjz_ub?kjZ)*SrgzVWo>s`t0GsKR1K7+&rH2!NO_ahl?o`dbhe}NU_nGHilr*A<4zkvqu9-w~am-JsaAU|N5)19r6ByA%(`UDc)c7@cYRaVDjl=9uPeLyGK0hd#%G=XX&~<_Xs)SNvrQ!ZpEp4*JlUlOjz%>maPRNQBM+&ID%bbpMxCucr`Ia-><=4<ahu5f3*g&xhe0?`Edf@>CRufaE9-8Z)gTn%Z(vGcy24Bp=&NIbh-DZzq{b@vXXXWaUHb1_-L@VjV>sQrpYBk+xw>&&$&<MI6?=b74Jctu3?PJ`fTpRGa3(XEUw#=7sXrMg7~Pt{GfYfJ6xE|D_aeOLD*;_Xjhqaz+|f_o3%$LfHPqiLT=JU&;qpp$^go4?KXqdYTle~5Vk$vOtx~_F8G0Or?o{wX%E~Gf5x(Vt(2|yv)yac@#dY2huZ*k-)v1AIufBx-%yAVy)xkygA@7c%c<c|VfbKo+CeaMp>IR~+>71les^4=mxpz+G=WE^kvZRO#|Oht!u!3y)es%DU>H3zdCwSO_ThjvhaGjH`O7UV{TUSf_2$plukTFKPV-mOjlIp%i1W>rz1{WcZ0B<GTG-qx8_l0x#7^0|`cm^>CmS1+u-SZbTH?-2?a=&LsTJ34ZMD6<F&y3AX<z^SA=2-o-kN)}6%Ktep>dV2+IWv-K_9mRdAq-xC{P>9zx45A#zyRKl<%~$n496-)00_|jgpSMO=KVr&Ps2sNrgG1jRE$8;)_;@1SG1ZDS%<PE_0)FvR=9z_d%U%G+I<!YPLWHg4;@{z$*!@8Yy!`0K%0(a6Cx-2gp~)>ax$WsC-1w1*xmcf*<D-23jQTF|;U><9w=K*pzchU^hv5g5AW50I(^$N~*XYYz<6@QeXw>{7|@Y&K162H6t2DnF(^#xq8qgR$O4-3*ll6o;ilz)&d-$4P!<bdI1BzH@5o2wCKtP_9QmhtaUTqx-cudvq^cgKYdMDZ<8EwX%nu_N@=c5fP`qjb&sK=T0%q50jv3<aU1&%TsaVq&y~FQs*&K!GMnt3twg;+lwx54FuqVC1`3qGczcR_(3s3GJF77mG6T_q1?WM&15+$Y3=#uA&Z>Z0^N>UW>@2Ya2DC}+C-CYecg3A5rOdmX;GBb=E_UIwK`{5<-|o*;z$ONlur9u_nV*x<r@t9B^zGpxZ_P7kwU}Z4s@sEy*)oKg2YKR&i{ZOx6a;A<7YR$0(NeY1n*i}(L@}IWLSo>QsicBP)8Zvo=>XoH$HKD(D|Qv^uSoH~J~xcbz4@8o!CUA4=KQ(EBc?H27k#xh7OOp-PBW)CN#2OnFJ;l1XDD$4v75x=jt~DYx>$CB{?{jOTmvT<8D!eI@V7fBFu`_EpXh0i^{xbwa%Dm}7bHeeVp|KyT@{>4m8=lKR25zF$|mV4sHF;3n2j!<yp(Sd&gwlE@1OmW$u;u8xnnjYaI->A7+5NdsjQ+hmb)6b*2Ou?tf|sRFuR<hQVN{$<nH}q;YjE~i@<+6BG6y7?vi&G_FSLEZuCk2bc9B~ZTH)a)=+u~Z=CIzIZUpIDcLM6ls-g!%F<YHY$rjboe{yvN*{!xl@iH`;>hPrmy~(@{c&&{c+T=<bu#hbjBPefSO#??vav;E0IviASF29glTJYsRTD$yu_UrHwkLd*6%YLj=)?T)lh-Ldy637FhS%OY`*3h)O)QZM9k>w`Ph+gcv=nchjMgjVfR+n3guwnV5U&8K0UL>IZ4k20lNHT@2$&?8sR3XnX-$W!NoRm$R}wz(HD@1GNVvpBQzTCYLZLdaiNH=yT=Ze*nss*pyxbcyr1aq{XX`Qx?BFU;1dtqUBj~eKoQg&{pg17}P%u~Chm<|9uF8NJw$K=o4EFZ9nmiD_sxtRkD`ymAiA<u^QXD1(bgsbf!1o+X4XEE{mx?C8RtrrhxlgA{7JT~@$%1bml`Oc+w)5-u-#d@W%d?m^xKG9L;)=Cy{+@+Aw2%9SpY<^e;~VY_i*jx!I&7qRuxS1q9ojyf+pzkR&fzqnz3!tvy9j;s-b&)w&eZqukP3LgqWJW);uux4fGRH;z%JIRK%2c_NoA1*(?O~foyCr~tb#5Z2!Wa%xrzjzMU2lwRvcr;T(O?>c~%_gNyVK#HY*M-gWoP#LwkMFtM@Lo@a+Y=)PqHX_0h9_#IzohJm=#+NTSU;`vo)1wrU1*q(d#Boa90dB9fI?0`N6T<As7uFyQ(;KukwrKs}SHqT){Zr9N5vWXvSq46|@=PEGOKzqJ>Pk^5(Tz~&Pt&p5Dt@9>*vDj6ekG0^|6@FFoO01UVylZ7Q}DZKVvSvmvF;d#nw=u6_kMYK}o3Nj#KL5D0*f#Y7Atnryv#xO0l;+#qR9fFTK2XwSPPhAO0JCVWipwvqQLH7z6u4eB+0HOOUG$=%(qU446+e?mVW8@hEa8&aNXN3X6kr$vq7eO4six@x!<*0HX(?%39OiH>9VOx|>g#{s1rF=mcOss-*#A%;8!u?zvor&id6RTYc9QiN(6<20yoz9j{^08%);bVL4+O>;A3qtq&)?ba<zjG<3ol8pXuD4c>ZAiGuW@|`&q#1AJaZlFn5wLWj0<!=<0%*sqIi`KdsdWNUCWi#MsshAoV1ZEvi@-#nY(!Wz<;-KI1UrQ{)>i>~UL2#<zu#p}ez0)lapMa_ix;ff#6kZt5GB^&RO`)Aw23^L1Q4nOV5LhUQcQVfIaw{0&f3vNTTRL&*kvJ)iK060P{2crVZ)FeKRcHT(^^FEFM!-yiDfCuNSrhTlp<I8Xr!<bLRQxhnbglJipr5RkjaZ!lTq4$ahPm4lyowZR}NArBGiIQvQV-Go>bteNaJuhmeL<w6jiW>&S@Z|J?7a`MqnnHfqZr>61DbPGFAkLr~!`&3O@u?kHk{+JUmbI46vE9uu7~z=y&O=hVJCfRAVaBN|?lf0H91c{_ubSpr%rD@`Ez$RN=yE0jJHse%f{$_PH?I-N6^&laUoYQU^SFY!<u(RIUJ80W$!iuU!FDQL+G`850?#Nr_gcY?)=86P$+9&ebHSfKx;Y|7?h8nkQTZ`ag3EM@Ug=D@4hevYaZiX?su#=r6;C7X>0lQ|u=C%c;On)(kyV4zwYxW-ENgz^yRsZ%+A*wFKH!HA<j4&(V6yBBc|n(4d~MbT!b~4w4lF$(WY4K3CG9uw+%G!`8}KHU$^}LP5)vQe*(3Kngaiq>71(czA*at(0hhgC&^fX;!ejL|_zm&ut<LqSwW72siH(q@D0Qau0BAW3Dz}x`1}GksjilbK|~fLXX}Ubhk75%#%1@T*}|U0f8ZQwQ~Rt0-ED+hK}*O?pnSY_0zxTPG{X(e$i5@iiOwEqx3>N5L<``dh{tyKMHMLktZ`tIY#VxZvi!us1(5qIIA6V+EDay02d;rVj=lF^pvx_#-s~iuCwYSoGUmsq9`O=DEC5uDpquym<mU4on}=98#rUGmSAFJg#!agrcJ=>2~drG6Kmu@PRfhi9~XF)GQj8_B#jbmXtWo=oR^}Q3`Rw)sAN6hpU>K6^pZnPb5%4YKF4gbrB?u?0Qtxx+%kK|thLa37fp6U>wHk#@H0%~+B$|9pA+07x2j2<w7YCInn&)p9xca3^AJnIl^6GRa=3VP`sO7ppb5B9noYNNF5&Uz5)ghahXMKs*6`wuJgD^Er(u6|MOm9}ZCpPNzzZb<t5gaGdr_6j4FD<VswTl|0X<{Jrq)4mN|Axh%w=Uxk=L)VLG2T1)!4k+gS#5A7$dYUQDS==VT_lC+;JJOT47E)NNuJ9AS`g2uR5)!807M@!EoUR{dP1J(0<?OeREv)C|a)-?JHhqM&^e|8O+djC87a~5403&${12^btYFQF4{;j6N7VZpUR3*YEFj^R$z3%1JabB`%s?M5lPgYpFcVaX^ltWEh-D}AgE8d56+QY4q!o9IE`fqcws9IX$G+?P6}0(<e)1cB?gUjg%UMVx+5SAdqngFMTJwA*CnT-)}OC;1KPOWga*LaH#>W?H_P>xMz6xZeC<!JkQykn3AhXG+Fe?xJI&Yen9zis)*tEGbheSdveRr(tnJQjL(7gHPPW@G6m-!$tX(>Ndf5f{HK=YnYWC25pusXaEK`Xe*vJYHS_da{wgQx?t!V~22f3az7G1HJxP1kt6p||ivzsvILMR;*v!wu8Gt4UE5J6soj8_HRlVjo4gE1<eVr<LR^qXqY=Gw{&ev-RV>t?haW>i~clr@K(mL#TlYkW|`HCLyz>uXo{!i-GjZ%%d~Sk?j>Kw@WS?b{us|GU3`3>G%{3_b;?r${w-z);BtCy;`O*cAm3K*kU&m69+H^gDB33RDa+o?LSlxC@GhKTjwuCY$N}mSdTD3vuOS@hMiU5lY`13`Or?<6<W}iuVC0gThi!V;+JeoXi1kEPJ-L3H#bK4SlA+!$rc*$hRZ2Abk?SQa~I;wJ4DDbnJ_6=0NUCCYW3*@N@J`hXiSfZ)Kp-Qsq<$_CH%ZVYxf7p^wRz>dd>#707fdK9m5wV>8bz6AkASZ>2&1QV<ftKUqQD<zNqOGH)!AYU7tIJcRSlnaTZDU%BqW!-IQIAV3qcz*^6!5FO*m>trC9srrm#kj;u5^em;A<kE1;o(Od6C`N}d0FA2kDo)^+?J<TzcbuURD>gBcpufE=+7nHrLvg5I#zfFSt&vgrMe@(AS%AgRA^<Wq4qRV+8J&MmX^YfAFaU4B9+ce20kfzIHA!}8NtHp#d)0F1IJt={3(e^9U&@??-&0{<_N48Ybs!1A2IT?iB1T17%s$Xw4LTQXu?~gzKoKA-5PO+dPRYY6#|`#m*+1u{cbATT&s_Eu3%m;rovD7F%~=W&0#yvjnH;o*jOceF9{^6dS0nk<Cri_>oU`_M($|&vp2LRrK{y9~32_3cLbW%D`B?*z4Txwe2d$<0w=O3Mq65i`pXSUnQT@6r74>v8QE`1-CMqW%GZ9tW2sHCoUs(**d1tJx+~rsrO~mhw$wTK>;>*pVH~HbfnPy%RdPtPqjL~XcY<ZgzYOyUB7Bl!SHmYlAf87uRMsN8_(>c%_nPkalN$$?d*BBk;>0}Uv(DvByma$kFDW(KAz>)$$<lw{P69=%wi<C+{?IcT&lTfK2v1gx0X*DUwm_p-<9P?$E<?j~rPnbvKuQ-&<<mJOx+rz0y4yP%3c{wS_Qrf0xlVhopFGVy8EY@X7o)W=(ss~8{*`64#QidwbC{_amUX<GALQzgaBzB?_;_l3~O{KIMDn$ybE*xDT6Ce4-1{Op}J{iuX4-|-@JQepW=(v<xX)co!UpA!ePl2>wW8O4MsRsRQ7;FCh*DoHkm&vj4?Jg2|DT&{YTM7I&d7FQ6R8DCeGn6LCdNjGMSQBCh?Nm`QUrIb`O_o_mkR%g<7$5|Lt_qOZrnrjA{pDkh@5OP)_kfW?Y0|6ON)9;5OD5Z*1kF}PIqek{abhC*9A!-*K^^-N8Av*dIsBf4<Q;{|tH2UA0H=U7P(=xYRCEzQoB(N@apxmNDr)C4<qakCi!Na99z4A$|7DM;<L!yKFM@V94-~Rf80C=_tolmT2%L0UWN2<vu(1s&8>9_qqA1A3(-mp&g5ZDC^cCCB-W<VM5ju-lQgxYf0Bb6^0U6~$Hyrt+Q_TP}F|ov?PN7C_O7@yl@W-kzz%zy9oKA`riFWkF@j{s?E0C17f@8@Rz^kOZR}32&>p5~3sZv`Iwgl|>L;h3PPlXZ$t+lny$c8Y(n>PwJp!h0(u_4p!23fLc_Xheswc`2GtZb6M@J93Pz1d`kPTM-CVUiFeAo|BZ3X`1`%eM3nnL&Q;QhK7v4q3$WWb0}ZcHTJV4cD8t>lcTin~?Kz8~Q)l?VPOZ-c|+y*xDs`XL|P9!b2VbyXJ1quPi*SS?>S&Ce5ZhI~O)4TW>%r)!B4&yrOjzoCCH6ycY9DxVCc<^N9b_anr?^FJUQPz^ZHiyNzc+)=jo{FF;dYY_ocv9DtY$#0s9cHmwyhu=VPG0_!Pok$`%v0P^O>lwv78p`a5Ng$?2hSf2GN*r;+!hGL<}9}U#D_s2USis4>cSntDQqZs-izT2JZy}4Lk;q5VCpi3yyr;F_*$tS}DJ4}ARPj%Wlh+pV#x!qS+m|yi^v0(0xR<63S9Beye%U$Wk$#siY+t;P__}4y#SI0dEJMHa-(BiJ4{rjn}$GEPKK^a{!<=dxi)Pvy_cSde_mf;Uv(J66IREWgNij|*}1tpZ#6b5LDZ&d*hhw@=6YhIO!07lN6SY__3ls)6{2R$ze>>OJx3e3C`=9ut@C$<`%a&QCZ*yf7YSUh$&k2bPQDJMiV3G=|9U>K`BC0}@8QiYh6D3z)c=>Vd1G8|ax;QfWl+qb$NyxVr?S7+_F#d$ErmP9GkJ(eT4K^F-zXhY=`cBeRV2H3}Br4w)i4;UnX2*xY-s+=Ooy^G7}lI^sqo}O8VJ3EUIcg`~fe`_o#0f6mw0OtgqbkWn<pZCRzn6Qj|gf#G);+`x%Eu{twb$pI|uBa<f^0sz_tCp+kg)th)*&8ClhT{EP3LMO>@En273>mwe#Em7n=c+5kB`1pm`&X`ll(wfZ!f>m>>qVr5Ej3f|1VtG<6(tb#Mo|oORpL}9k@M^q4?3m($?~!6C+)8P<(Y4B=oOZe4&*&WK2kw2S19XDCIuus*W6Q#cMuggM`-WB#l0*7>r<v@VJpF!AV-q9rrOyq8yh6Qk1rz2#XeX2-0cgtFZjNc`%>>qyRZ1ZQv1s6Yj%RC6j=D~`gAjFOj~nf*xbHEPFUA3mAu!3b9(2bGxT0!<~wCaue<HZW0eY2r>qpH+Dd`PRP<}_>yxuJ?Ro0Iv;nB^QH1L7v>&!w_x+nqeZ5njcgJu09By|O@<jBR-MT8Z_nUbnU7xqSueWrgHHiPV`D%gC-P>tic@_U&X<q5umIAfyd{@|TCA76ad}z~EJz{`_K5Z>U@xr`x1cBQBaB@M#?!xgj;l#3v!|#utiiK2p<7_NVMQd%_KdI+jp0ej$effI+@4r4zr!D{1VgP*l-h~0-5oG_$kQi?WdcHAaFr+8^&oTGkA2(m^D^z~3U0~lGgN*2lHVN7ofHV?<A=f^hID{;9=U?wHwEyl93f^Ay(>_#tJY!nSU-<U;Z0cFLxRk|msRqh$*46{y`@=xqprN?i26=Ze-}h+%em)#p-J7$sb};Vu)m7TT!FPY$uB0317kAgaHhErm+xv@0STu}3xwv|17n=T0ZQ8}R$AppZE?(Ey9$Q4JKOD0)ezwR_e|ps8mzwf`Q0{*7*RM}^!v?mlRoef}xH{Y72&s8-kRNS&zn$H3l=Ns9`oE2fc4@P`)Q$h!FLulC{dQ9B^uKq)@ZrI4k0{IVqhT5SvhVd!1ni(cDxB|;<r~|qkM!liILY5TOuEmHe5D)1gYikX``a!IDg2IT_E@ld>Fl#FII|e<e6vfGbxh|k4Ilro?N<9G{k$>!NwfUBxBlr;^Vewm#raGA?;%jVf70LdtLx&}ixvN~A!X>XjXXaXMcYd+3>MNKj0p_JVH_WOdpyMLN}+$;4fg7dv;tc3aJygr-8LUAEf>1`A2}D@2tM+X%L^}{Jv_Vszpu>I`7UAW2i;a5lbPP6>!!6AGM^1U#|-wLKEZ|#FQ698UvSi->aOgHV|HOvG!Z|2akQa)Hssa6{iK@rwJZO*m1WBf88TY|Nvv$~d)-sN|M1|SaoOdkxs~ZnuL6!wcGjD30GIDko>R*5`HcxMWZ2!Cm7(~+zpXrUaaZ^1hTeUzP0#sJuTomtf6NzOJ?w(;;^o0X;k{7|es|DzH`=tdH=C8ZGE2FK=Slt+oGVsV!G-o6_qwMIS(@J-6ZCgac`)q8{s8e_KUm$ZL*=c7)E2jwo;nxlpDv~~K8q=B_K!Rfz1Le(aZjI0_0_F(9h9s!j<uitbdD_U4nuQyo_MuW*#FSO;{h66N$ZpHW+5wmd+C^u_RbvM?Jw%Mn|-n=5@_$v6NT&_yNlbej#)$d!#m#|cI|@!T+AOl;GyoakB5f*c6fHX@9CXCYk%HuKfK#lFrrU>yWEyL<MJaL-u@%BTHo|O({~Hb_Pg!!`hR+K!1(_0XVJ$cvE6@UJj*dB*sh4^LbTqcSiK+~ZG$v9L`D%DHK!Ufesdvc#k>`fdCnA%4*WW3v#$yj?FK=%8swN_AQS#CS@Ol&TtkgAQ%-ypbCspYn5ejj^U8X!gpK<yF`cOXkYm;eQMoAGJFZMgE-E85*VaaB@c<SBH!c??&`?d0DKYu|Na6oMIW4)5)+i|{Pk=%XxGbE4o2y|liWrK>k&b_hPr^_rqi~w%eTQaIELXA0Q}BvbiSz7?EG7tJ9G6*3mz^xkJ4wyvqROa}&eA63_R;ew$Ph+l0Ax|6N=;b_e3Zi3z%*q?<RwX7oMo-a(S&Si)G$omfx;`g*a#Dg^O`BoQ(MHF3WP>wV@)OMtfJFFW>pFgB^M{H4Z#b?_jOxI%wi@a`C(Tdc#bk*q73F!>yV7_8bcamh*}$Ogq6BtdUaGf)_tW-b=7d8(52E=*1!;!#4w^h>xA~(QhBz)Fu}Rc+(rxtW+ofutJueQSc>?eN~UNkrjr*dYsN@<roD(Jvl^^(nEZkzrmW_Yvh69&Yq4gZ^6n`geWFu@AVR@(vX~1c+{+p_VZ{VqGfQ{|<;P7`Q|=vGOyMfV{fu$NJCjp%n1R{Ip_(ERxoYxSaeWj<F)W~f)vgkG2Sgu&Ae$bR_&%DEY^oNoR+LVL@^|8=vqH8hD+;=7G+wK%sWDK5EOW-<UrT*1`(#(cdI%VDRt#2Rr4lkh3Z8VJ6ha=0NKqw|6jdx7TY^sG1ttxPFPVKSEtN9CD&&+BBUM3F7O)%I9AlJT*_0w&#m>OWir52EQvn@C?j0%U{e-X4F_o37*i$*8@46)8WT@D1HDSOMHj1gV_ju+CkCrY`G&80%98s26?JItnla1-Jp&t~EcbQ|&>dV{@KjmeyFL?KZ?+b<>++F7QspwaMZ!|s>?e&zwS{>u(vaj~tzTf9^AFrghuzjiaz1qi`!*dKi#_t}_^x_zo7~?H=673;Rm8unXQc!`Tm)JPjV5ohK4!bL7T4M%NiwR~{@|q}Pz<zR`yR2*L)GRne7uG)3j-aA$6zN^C!G$QIwbOIQ6OFDZ5E$5?PVak}U3H8(Ogxn)#@vb=1bNA011T#hYg|>qOP;-yLVGEK51EAunU*oulq(c#BSymt#Ice|>WC$f6(oV8B;HP8i^hbdRB7YmlZ+;oT5)mN50tdBM67QJ3E>%o*q|PX%qVccDoYthWN;>V%5}vHM1hEc1$<Pk*r%*_9O68-kzFy8{G=3Tu1Zxge9#$G?}<S`Q#3LqEy5_EHHu8G!tG<9N=R3xsVUeau64N)*xHhYhM@&C`V`6v$rGfH88a3HnhhaRFzw`51xd@%S_i=nVL*XHXq^`pQU$`DlA}mvDuZ0WhQ?;Wya_CFgUQ}EHpRC5ise>7A4N;C`OqzfM5N4&GSEP{RdPx8I`R-H-;jblW12y=g<v$t?)8wFT<Ii93MWjRl9;+!tB}wRHKah6V3{aRiv^bPR2tm0A7fD_6-kjVoCy=i3&*s%lRbyBsN{IWT*@2^Q%7kbrzQHq0v?{R2L#@cqA9Uz!7Yg^8@=Xu1O`V@P*DsvLX0HambEqzPf!P{P+%oA@_wp?F~cjgXQa|r8Dg(c!6jOhjv_iNRWOGXgvuvhc2%8X7K5QHR&m^~n7Sf_AQi-*4=|SkW2~v?2IEa>mn7u2p;+bYgQMby!AMLO7IZF9EEMd30fw%EAcaWNqz^G3WT-cgo6s6imRc8vf#gtQ+F^FcPbt(a_p^qmgvzCC$=G2Kxr~xXVfZkm=q40D7Hh_U3jsl>E4H}BQpE&OB$|gz^d&(-suVGAhCC@X9$}&=bVyRXH%TUFLJCt&f^t(Nio!3Hmka$mI-InxNZ1Tkdx|Fg-CI9B4ypd`t$%rG<@>i+zF&z)1t%kC<FM0Qn@zW_p0z`reF~M!_X8FRKp2n(bI_$eLsnHC0mHEJf{8g|%QC8zfc1rkS%C#BfsHg2TM+&WN2Ef5VWue?cMCdlq=Yw_GFf9SI17k`l~*H6krEFiv>~I<0iYPlzM^CM(qXF^2z)v&u7D{bY2aGYG08v{SnDg4N{d!WU1X<$BZ{Ls^}!~#FDQ*Yz*{m<(?Kz4WTR@kL%f%PO2b->y&}mmnN>jUz<F@k;ZOuXqWfPo^AW@nqtR6=JK*&Mb?JhJgh#WP^@iYQwGdL70-_mo9W)!|H;^$v;pMbX_1Fs(jMbE#N_%a+YEdCXT~W9Q+KWN~Ac`nkm!L65D_sC|B1yK7CQ+UqiH5e>R6raEu>kC3nG#?dhgS9$I*A*VY{>xL5S(1o7-5xA*n9vv0K|cckwH~-kQ0Ytf)PT;FI2aKoTq`Y0;Ql)u7&56)k>*9^l}T7Vi|xWNQPAoY6oaaaX<i9RK^iNUy4elv29euim*ilXaV2{2jGVMKKZEloz|w+9T`06>?>#P8wx<6Abkv_d?B3+-UQ_-2>@{@E5JV}cPwH?1tvTuTmtD-EEW7Ksy~$n?9IX6@Uy{!f96FTII&EKQ|G{*Dvywt3HSs&NKok!;2X#r=r-*B+HMvD=1GZs6lFSR`pgsWwOQj%j-Xu)cKY6M%JRd(=H4YA?oW1aj&(H;I(yCMO34=hiW#sg7CI4MV}Pwd_9H&USZW~FN;N}^)`0fFPE>gkvIQ8^I8o$kpe^y8W4I1)FFuOF5ZQ!G5**?`b4Wt4kLobmz-|(HOJYWN0R@2m;{@9xRZIc4D&*^~+!N2qcS?`@FREOha!$Um)n?*b+YH-lyVJEiskO8TYWB7!n4kWkJC7llpJakQ>?^}v#2!@=yuOxPqD?-rltJ}`cYyh{&jq0-0{qTrPu?d?Iq2CFr4`hpK6}Ze!R4S%l#*oHa+TtpL9;V=!2&Oslm(?51d;*M_CZS@pmdd^3|*@kl@IsnFRpy^?7IX^0Vr00G%5e3(&&k_fLB7tEI>U#q<}S~F1}Vu@?T>OkQqdffD50efCHI<{y5M)SYYIUL>kNzSO~m6d7{st{Ge(q0lh-IRP6KU00NT0es#~4>l<i5s2~!7gSQdT7KXXts}zaswFKg-r*fqZoheQbb@ze7xibhueSQ-WAEP{+_&-^Qk68)c<ZM_n<kRSA^MQ;3;xUTxQlJ2ut3<>Q<bzc@n@}7~6}S-aC8iUjXg0%lCYyVkjchijTf6H!^T3$pK$i8%1}*Q|Y(gl5d*Pv{zP-2AM$=GYwrMPRazYB+g-8~9Uh~cVdCgy71ZPw-v$=wqrl*7Dw~ubP_^N*KvEx^|kQI#HGYwf$P;;7d=sipl*jAz(jWk76eVE0l>FB_EL9v!X)d*VbtjN+<P8DOmXadCLP|=?a<=THZ94_8m@ETeOFIh^DC@9I_{2y-)r$;m^|65IuxZI2z3(+1Qwodfj`DJ(R{K*lGSxTYkp`sT4G0#H^MUZQoq{%#TVw=Kbt9Z6KXs~RcL0v+URTO2F$PD`g%Yf|f5(wMrpq_!d?sA`M*qm%!pNFa(mha~C<_#JG2tE&a5sK36L&0f9R&p|?Rgb}xtVEeeF|vqGG8L&zLKYZKjB}v#BMJUn4j=y<lO;&YW2g%i{`75FMF~cOKUr|^;HgA_u8tDU0nC~p7#XT_0$*WxFO{dK6Gg3#Hnu|6q}c@P0I*`K^-sArrQh)0S!tsScfw|ADpq8fod@io&#pK0<<2CMd3L9_zTQ~$xay6rHEisZ=OAGDi<(!z>^3PKyj2X~su&djn6<-BEu6<lW?^3oG`IGYK{`Q=W2_)J0u`}kO2op?+>9~Qjh}DmSEifW=*_J8uhZG)<>m@j#zws`D}B)F*3|}!>gp_Pb~Uf@2ew)pOh4JBOYt#=<LY`t9-<$e;J<#mStv9<_hRo{raoG5>;0_v#T#rTy<^*VyP`U`+qc|bt}j2nV<Eos-9`85B}2?uGxfvKq3)BuV$*)}=LA=Tq)OR};qR@odu=*<qXBrDHdETLO&W%6=lafW+1zQ93Se0@lq7vJYr}|}_36&;IDFF?q`>ut_Gwqd`%TKr3Gjnw4#QnetWP~Qcnn+XTHdMrpO2BG_m0RTF**?cU_m95{ypD}KIONED&OyPUhIp1FU;;z{q=?7v#YJ^ecXONul+s`86Djjhc@?RXOHzTJ$7Ka_jJ{}AN0=WbEoINwASFR|NhM*y7j$z-0tA^+4<4`k1-F^jouY(Xze`x(cH=W!B7yYb)oD1^xhc*h=)qz)aM_x;g+}OL6QR;YTI{n!3qAsV&w6|TmR5}t5x7(qxpItDPCx$zP1Y`6CkNuW%GJT>s!-}>DBAaSHJc3*Is{-F8RyXmVATXYA8q5LPzeku6uo}n?LUtE<Nt$>2$aGtH}&1WOccX0%Z8Qk7NG+5Jd4w?|b<EVj<(#XVY&(8~&&1Y^N-*({F~2YtVuhk3<?jd9jsKJD&?`@DPO*p37`~>lGyw)mw<dm#Uo?AYP(GqoKT5&Q!n1G3KGe0-akYw+c1Xv879Rn}hKvu#psZ(es*&tu<vEqs^>vQ)?y1E+edWfH_X_nz^!4wRJwB2q<weVA8X)lRlNhr#~1wDP^gPQIR7QH^mBHb7e8O$g>LMoRO`u94jIQ35p+aS@q5l=q!QTFIifo&rc)<Y|zzaoY#OHKk1`^4sZWB-P^)0SOaU_4U;YM?R~Ttsf|LkRo5a02*A|>OJOU5-3e1-mIV`p?Mcj@lA>c+D`hb+6pGDVJ7}wJXXdy=ljAZW>0D&30&JrF2g?L>gVMpEHI&d$uq1rS<Z!48OQw1mL#{?EKtN39?{R+8=ZaYHXFx2Vf5_>GSLH(~oEa%7i!H0lK$=J)i-0=7`mBNsOAsFzA)b;XBVjpv<FG!$I4qCc<agSDgU%iQ-`ZW>A^AfQZDahvPY1~V=@?ScVU|Y}4trh<&H8yVG#ds90_GWA4gf<rGF2&ODi)26psAD{0h-4brWK{W=GCwSUF0&f5*N=RgX+Q4&!EbW%b?25<EK!)+rrqJLloUdN0*cRN^{66`u-vkUx-K@SlTbfbn#>Mq1z)__;iJ1tkxC&{yZa5|KJZ7XyAuOH?o8J6vua&;$laRbYpnfov-v|mp|;%5xv;7vBWEJGfQ^pg*!fYcyRZ{_J(Ck^|JMN;RcYuTm35r2|<yvce?Q4g$wcTuV1`)@#1kQV6A7uYN*?y0Q;k|wwfxe5>%zt=@5er+7`%0TNO|`bL0!B3)KRVb|~%g^isfvxI_45vbA=Vq`_Kyyzv-mVUJ%hUp!MRdor0~$F#0v64onfHHqxYuO7+<>-=!_mE~-%OSxYU-S5V1u!l0mzRZq)kUl;wZQsa+@4PvbarTI8wr;g;S6)4wsg_+ZB#R%ZC!SVPS}OX3EkN$vIwldSmdZe!lXc#glBn8mZ~!Bf*JuJI=X4@vAl71SIOPBEOi$R`qCCbMI$R*1&I*83K}nPIlv)w2I+ZqYj!1zA@)!W|7Ze)dfGXo34DA{P_Akct2u~o>tOPtwo?)CtifzC_aK=!|UuMVxPSD*jFyQ1I#>m;6G9~gfrZ^RCI?S-z>bFNRET@iPSk`%63*bczhBJ_D3HB|dDXiKes)W$DBJo_Jc7WN$DySqBw^Wkb7yo;lYxWQQ=*9#=@s-Khf<ANOQ%w{c1n}z=*AB}vQNlA0&I16P0`obGU~S}B!N=vA6^~H?;*9c48494Xj#=)C7nC7PxB}F#JSDxa!P4PpG>-Cv1qL4IG$f<3keR{N`aF4mSt=yRv#C}=OIb_@BsS=SP=)d(YsX852(PY27OWP643bo*z9tGQJVg~<wopa4O}4r=$&=}3Z=l>PVQaE=^_UX69~?@m@uaHe9*mhooAwU^cxPKMms?8(ij$30FqQ&6+6v4R{TaB#6cp(wh9O`kJh`iL2$dph02$OVvy&auhx|aj8Q`7qNaE9FgTSpWQv#hToF83H9&|&~IU;nl<NleuSz7EC&J$NE*vfcN&~3#~T?q<Opg13{Qb|(OO!@w6NI(mjj*fjWA)hovlD%-|vcppF6EKsm24p^!qBDd<wn=)EeBkIJ#i!K@aOG{zN^vq7Iq1*=UH)Z5Q0x>4>dB-Elk-$zfBNdlS;D^Yt=CVPAM8<?!Cv`#^ZM8RbUe%aSzm^(&9-|O6}I_$elZwQe)R+pp^uzX2(QL$)>xrzDSDrxO`XCL&{{wnTLnZ2J>~GZliuMLh;+lWIOOZYE^~d{0o-vm_o#IS+s{~Kx`6U0<x-XuO-->94nW77s5mpaN}VZOnsg)cMgR*qV}Gx0?vxk9nZT$VTPlvie1VSeKOG{K2&csK5k!xNU@ZB%0YpKjQ9Lu)-<RFyZj*=1(K3|?!4{c_0KExFSR08UkcL<!Q=!IaE5&HC#Dj@m8kco4&lONAN>K8Jl7#ds8YU<=hiYBcAgeLcA~@|b6c(ks$h?SbL<S^`-nn|Rwtjzb?zw-+Rpab4sX9hTUe^R^;vGu>hBaeKEL4V6GD7E?Hu{{?5;wW9K>gIxTgt(DQg8Ly>s^X)r11APzrEBzT$JkC(MpRN?FX{*-UOMGgAgMWLGc1U30<*R)AMA63DFub?^-C?gSOE~U<UB=P@IP#fxv;d&sDmhvg5J}A1&qa0l0v~F<;m}x=V}ogVEXS{;+ayFZn=I>1(QlPIaxFRY$l$YcL3*xTNaQ4BG0D3-l*Ly15^=j!mBqhS-h~yu14DWtV5R^bdb7T%G~ry+xl{ts0!Lm5h{8CvuEMH6jJ6fO5f-fzveB9?)BKg;`UaVeS;w7`kJCJI7XISia`uvH~sKYOnoG``3{DnT{<MT+;@JuKNRjxH)D$J2KDO-J?q}JpCdCu>gMni8v)G&khta<7A;!<TF4npVd0(Er@Z*0W950Mb-tyk??|B!}Cz2K)RtwfdWyS7b&n#8_n1;MG77)%)sG;;X_GG2e!C8bH~&u_=05#LIG4Y78<2Z=oD))!dGnCU@Ejt1u?SKP^emVjs+;1lE9F80Zias*|tY*i@WFB(=@F751M}IQGed;I`i3(S+hIn>^&fl*fzKDm|NYyr+!Qe_-Qcbewpts{_*4X#J+Rc?Na>pwII7}xg%{qsSk8iM~1N%UR+5IIPBt&SDiu!r^R;X4xYeYZBK~ryzpYP;8^+2=mz`2u-k7gniJ+8j!Qe3J*6F-edWd#GLeLq>kIPf%F)VFERkyGq543fJvOtiN-!0JO(8N30o(Zz9<hVT&uJD&L10fpc7u#SpwFNZ<3>4?UA<6x#Jk1PBU(v&{H}h~vZ4L^gVA5Slh-#FzrQg?)%Wv%%60u>^XPT95~fdw(Vtg1KsOd9O!gcw!o;F6vzT?tn^ehz%qFD*zH`Y{ud)zU7hY3zzId~G=5$PH^&8D#68vZ(b>xkg1|NxoUyOM{+q|A*hRnrcXJ>?DBj^PckS9AVD^4)Zv{oJz3xB1g$<SV^2vl4|=jfR#+7%U~uau?i<PdRx=2b3vq!?=Qq>(JktOC`tELqALsa7ndWDKH2mTMG1lUM|pc$>H^UZQ*D6lQ3(V1}N5O_t_f6!jYX7)$iWOR#7#6n*bWo2CC(yNBLhl*@NVca$@>Pe*V$DYSG^TM@H#*2Q9C#SWBJM`Iji-Xwb>qugRtcBFE~Qh{);&lh=*AsA-`Pem{pDc_v}X|FQH8hmk5K%&%uMjOwBa3u=n36clnlP>0Tup5ix*T)w@J4GEZ%745Vy|Db;Q$qEHa#6jSJ`<f>lBRRc;6ioUItMD6ql6ywQ731Lhh~NV5|G}usw8XSQaj_wykDbwUUjmPqfS=F&vmj>tqI{6oqTs81^m1wLVLtQDS&S^t4?)aq&(Pzg&3ZBw(2hk$RoMWJIf<fXm-*iN(@WYyeXDmXed~a5P{8~D5;}n*-%|K%CtxsW9^AL7f`mfx1YW81l2A9c%oPxEuAKT(@7o49|y87psU_Xfi+2mdJ7dVmy`#%L`t9}pDUqO!GDYmXjUN~_x6Y<CbC~KNx=|a00m_esD>LAGmC*0f(lfC-TD(wl4we+r!zHPqzQ@SXEm9!GXq*F-9}z;ZGxb*z>&pBCrDH|Rx#-<?dY@-CYz)V4++dZMvCC(Eix<+0Xq2g`QFg*WBm6wl)M1jziv=`3c4R6RO?p1#jZ40%NC`kFS+?<f!=E)bT%hD7dFFoS824fBE8gm-$4wN?Sda$yR-y{lY{A2BQLfun$4zhI%~r-SAFi{#`NmN^|G-sjcKR#y1TZ0fg}?ic4=?BHK1Q2-|S0VT#}Rqqjgb!`IXl$ygVjJxbX61XL>!v>Bi*ZcCP!!IrI&3<Wldsc4cs{>wWO-yx9AdS4P*lLq2eao#_UTxyAdgj0NOh9v^V%ySZQ8;kxl`#nW!lRq&O=HQ`V5UZ<Dc2oHM?E;<&nL#~crdqkOgdwe7H*prtJ-6s0%9z!5{E5n*@Ik&NVWovJ<UyWqpoNhuGVDazm3`bP7#<pjZ%`m&Zl8S@;-Q#2jYh(216bySzEgoL=p*XaPCvUbIu3Od@hSkzsE3xw~ZLko{1!gbs1?5DCvUIv9XSqG_TJW4d-^|x&6KV~U+0OZ9d@k4Q@Y&a+j+D@l0Z$K#r8|>*5)v;-05_v3X&6NgStGOIl~Q{a4tZ5m;@R5hn1rBXibFK1<O}4uVNB+eVBDRD2e(JHv;^MnEYvINZ8h%<#pOS07K)lIY4a{{cB#PBjn;jbUh&7x%NXP4%GS>1rhhfLp3`p!3zaW=503ePP%j_nZC!o+V8QE;kAARfrfCKOGY=bH>`wp46(sW6ay8k(By=VgTTE+$-U5-$7V1r#WFnJXa!kIV&oX(TvI_l7+1)A87<7>a3}Jxtl|4a=ZP$s`$6-7o?|11D01|uQ@`8N7ydCmpxUfCh*a)+o3(GtB7$o_qh1Um8&n-MRTEKbCnpm~weO_dkT&ApaLP`Z5RY3rS7Ftz0<yH!8ooNBG#Q+X5e^i1&P}X|kg;b)RqM-cYvx1?N?!kLw$jYtdI!yo4hsF<<+p{%ZWacQxE+r?9Lh_W2Vj>HQGA2*CECLgWK{OGaW=bbsa;5D5DOV}aKJ%vHs~$M5KMPlkK^}#T>*(mt@BaJ$y8OrzWXo<kva9y^K^O9WLiNAXR)*B<5tSidd;RD-d}A@luf5v*?ydh1|2D6D{nb~S|Jc+w%Vr8tv(5k2Py!KYtvoW_TJNdzabNMJ&!F+?qW|1r^X~ILJ#OH9WGo48sLrz0T(uQQy=(nLN`ba=*ikkF%(hM-CX^qM(rRLTLE^chR{eJPWpSL}jyK1ZNZ^jr@N17}LWL@8w9%1SVenb?No1arx50ujD2v4>l=qxRm9q<72=0>&oIj;g%@@rd{*(jfyZh5d31o6uIy^tUFU}K<mCwZGECX`V$+rH1m2y5}?h6L1+DJ=IMCp|;3~NwQy$LJ53(Pa_o<)q}ou?n8D36O#1cH9N7{%p6U%%=KOwJG5zCWm|4@Pg>@3)fjz0Mo(-G>MNyi~%h%M&v`^Jj}N^P`1Ao4*>9^mN_-?O2xT$77UQt0M778`k*lU>BUrvF~;x`s40{2ZQafN3g#fPd)k;cTHP*<K3}t?wAVay|HxO5dL!TR@dF1EI1#3IL2Du9gxg%4d}kV+aE9u_~wYQIJy;&9F(Ql%FgK8pYSea^jT`(YRi^h`=~HU3)JL<6ii7@ao|=uD}r^>)+j6Bl2ambU#zKyg<_;ML1)RB7YY0FX@^NZPDtcYqw&RqA^DRDhMY(Ky*%0sF06(X&Mh0}n21NRpIlguk31acxbTt%TgRc-j&_^5NoH)w*B+@Fo?2jH=2R!DsGyV7AymyBD3!7f&;zjEKruK~z?`$EIyp5O1_5t#kx8fIzhu7XL&EvTkLG)4i1KbTDh&=ZqYxS;(A$ev{O=FY@&24`cYzgyaQk4WPx#T1XXZWI?_u%hf|;iCQ^k-bEfRjZ%$amPF<XnK5W?&z;cYRVP*Rb;;1*16Rg_{cJWd|_<J`k*wAO<iLdIn+!P6_*-F1;N6f7lQd{U15eX#Pt9m&foWfqN4lqml9Iwi|fNtOR4at{YbmUO6Kf@29gzs4*h)h?u{VOt=Gi~)%Eno%Y!?ril~s+7Q)$z<!PJUc$=-3@yM{Z{8xdiFK~DI=#si*!IDOr}#5#=-k)Yd!G-khn503spGFSyYxeh)zsdWy*+pVpo6E%AFsz;^vpnriX!PUMY}}j*2KIghz+}f$DOqAmcKiPLh&A6KJQQ0yxA1(2avKk*Dh6ozD6Ae6x_i;^6Fis<HV%V5n{*RZo;jT2~?9j7kM-lf_U%RG^_035_eVBr^^6vO-t$^K^!otrkWqVP$4U)I_AuYboe(*Bb}*Pk!d0b|os+mZai^vZ!&Q(yad8_}z4wFK>()J3Bl6TW9QXlOvZw3dn{64d?^ZpUD~PCq}RX<Ef&oCDu$C7?=w&C?1owPB9+&DLmco%pDWjWUhVA&Mx}(tdHVzlyo)|q%|cZ#hIsq!l^RLnN_|*B^YaKpwnhiykUh0@;uPkevmC++R)<<TO0Q~&n)hkY|q};DjZM-)QdFGxgq#moG-2<PrIM0%h_1D0Sy%Z?AemtS!24swBvqbs94|C<v#0SFD&>dD9=k~qDlwy<1`j_v{=?vh6w(Xfq@bf2Uh@mR27w~S<6awU!NzfQH-X7e4caTv;b6xVsO~KCXucK4oJ?RTgpXXHJ+(<ZJ?z6)dAq8Q^=|N!`%OJbbtD&KLlz6ea==;6=S2IqP{i@4!KNGejLi*GDZb*y$@B%;y8EOQ+W#OO?4?qS!KZ-p444+_8rPhk~U`sc$?~yOUChksZ>yyLTz&OQhI4DD+x5sSY3qh*#{2_e^Q+sgFy;PloS<}JA#r5c(4G~fdG%iX-YW(AVJP#C71}>!z0!>(<W$R=v-jra8x3t6)(i<7qTrXfEDe%=aO@&b?~VcFh>OxO;tAy=z-T2>`~k7ZQ|H$TotPRBNf!k(ko`uYaKs3WNiq0yX({0WH)pSYBJkh-`)rq%Cuh68%J0e)Daej!CF<eE_RdIdLCY!&aPe}ucYr>++N?l^cP_r?CJ&a${M+2K(TM-^}V&%*30CL^|f#AeR~r3W``{R8|6E#E5l~^_VnafamQ*=RG3!-m4-qk9-|VhtutyUV=i-%LUgkhl(~dfg0(tW3!(2+6syJnb=e(l3ozo?x)1{LO_RI6qf+(ECr(Gm*g8Ix=&a)fyiQ^MG?oVuU!Ez7R;mITaYPZ`h0$uRg`Fl&XIsr=t9gBI>*{11@|!FKyUi=>d)PD>^)JBr`@z0+`eZ@#jx*&T5)8<@sL-x}bX0KF09n^4I$8i+Cm}qsqAO)^5?~kDrA7yxyByUWOkO;gZ`Un5$p+AJcMJsp_k7JCvIIyB>DKrlN}PXnI=jAhbuY~5?Qc$Y_V9T?11f~j&f2#-?f!Rv{}?HUhUNMcED4n>ugR|NL`=?w%B`W|F^sC2CSH`KzyU~%?1D*JXD&f+z-2&Ki;O47L@;~iG!p}zs`n1t&{1AxVX$EYlu%AwGVC5F=zzFVSzc8F@S5W7r6qzi_%lDYyVK=mdb-DULLHl0R6gdh-TyPWHaXwCy1940`OmE_b-lSThCG1~zH@#k{BZDgbG~Dp4+h73BBEa%c;lTRQ^;p84k<{6oH<KQTKz@c_unTwO&8cq=Tf&v8}R6GaBpnQ?UA%~J`eAXhjK?e;=#%zy6BO2mhM~({%fNP=B-;gitG;ihWM_QBp!LNMR9lza@_;9{{QT~YmZ&mbtU>&P76reLW@=F{h*K=P_mTx!gi<I_Dz5wAl8#Ct60TU6(!SM0VH?Zw2^lE0=|?SS>3b}nxtupBJ@F`zDR#1@GrSz?$_Dp#VS&z!=fBnrixXk&e?mfwdNdStvSb_#tN-1w*m2vlRYV!FC&oWW{X)RC8Y5Jvn+CvOVzd1)2;vcO1Q9HPR-fYRPnMuWb)@$>tjCZBy2c<nx0L4W4=UpN%^kRM=OO+kJOe(b}q<yjs!l34>2pnOGl;2Pv_jMl8<_(#(ZQIV1L?<)}F{g0hRNBo{C}K1KzksU0wp%RQwIABBhj%&NjxcJKK;qqOgYh8bN8KjFlWRy*HAHVvtSAN!<e$TA_sU)*<I;Koze`1f)sFQX~6W6Q<hs7utV%Z*U;!<IK37a$PrGl1LyWwsIiQ!cAa0WaGWnIjgKmX)H$229L~Vku7Bspnar=;zdZu=ncDn{66z<?u4DK$x9_58EtHAkTF@<nQ(XJwXN;lOLQ-1ws&_fOuo+f<kHsjll>AVv3WH>@AqH3xD_VfzZ`zFwH?Tv=H=ZVer<p9QutBWn|x!dJue&=d~baD#k&R=J_5#aPon1qFrO)fsRSCpiY8GD(vSfXCPP_mDhq}cOKmF*ldXyqEZ)7a`#OkO<707vA8hSkI#2I0e&uZKE^ZSyRj+JMy)HYm%(HmmCbHDQL1XB%$yarPjoq`}oOFwzE!6E#fp2w|NR+XEJ$7b#(uQeJ2Y}xo0p(m^xtQ2$ZR4rqZ=!(sN9x=Gx%Z&Xh0$ymLbxipGU{<RR_8YHzRX>H_4qq}?+<2VGc{UASeu^k-k;!L);2`R0;|H%IZ<}b%ShJl-dgF%q}KBSh$MLM2(*G}Iz19mF&@4;Bl2lq4>ol<n_8T$dr%^2n0l4Kzro3Q<YY^Z;h|b}>!R}BC!bl)0IduIBRqvLyo^a5j<JjRlV*p%dz24{?#!_vKyx5{bI=9(Y_Kr9Gc@_GJt1;rHdQ?`VJQp`L`IsC)jLUjgmj=;qHz{bo>Bjy3&m;cSrj&91j(<=nc%Ilo8hR`k3R6X6RL|Jv7`^X62m2Pc@2jPn4CN#m^-J7=7L)-EI^Dfj=54~<jU!sFnEdOB&doa(YSd4xty4+Usw|C{f)7L&6_>@z21E1{!lyrZeKrjqX9y9XG`g+CUZtX43ScMPGun8SjF9<f|%rV@}Mm6IC)JhDu}(m(SyxTJH_9wo95O`M|S5_*p+LM#onTWvXG8V|2z$HVx+t7vI~k8MO^|a*kWqH>O~mi#^xBDuSU=w679EQrDp0okvTFF>psn%Mt6FHGMYv!YR-)LcUz(r6GH)Na18_~OOlA_l7rtQhP9GSwv6O*gqJ2^*%<8<1&TmYodFn89HSWLgK<f9yQlQ4q0R8E25rq0ayOzxR6_pamjA5#%!?*$w8_h4f-jb~r${WQ2Iee_O!}e%sgEgYBn+S_)~7~bDCh`Q62CH8?StN0wo{mCgY$33dS1XI|1@y)w+C_KwdXWLx<%O<RjL9pBsmtwONvD(D_C~C1Q}|mMFuDEm%v@k_y$l4CjZeI6m1C^X6lD4>;%>XxP*<Ec8c?2d8MNUm*|{MSgWY&D$prdQdI<@1QsllAt_-CPnM?WX{sj^BGgR9V%1{U0F$lKQSoSameqikaysq9{aPlFFG%Q*fsIm&7BBezDN<P#A_Y%1jott#B}4?pt4&cDQY>~17^@1KJ$+L}$Cx4ETP+-S$=c&Iy58&iR^RD0+kVkT&p{37?eR|8=S^^UYv|_vbkLSr|Aeh%vuO}2wX_~QwiNtJ3S&X%STJiTxLWIrFqj9l3tGW*h15wK&Q8;WrPdnIyO@M#n4F9xs4ehO^fn=?;HR~4R><OrDFEq2j0dlCTyYV@Q)q!WxOy4)e`N=sb7^b8sR&q7iNKGgyZ`n^s|GN3iT~HZfBV6>aiL4_i%q-BUis^d8L`Ojckd2|PW?gJ&7sTTosHqm9*)GVI>$<S*ROSN`^y$>^zpj;m|N}TG;i}^=czaCKK^Vfnbm9p^_Tc=gZ_ule{V(suxmvg_io&u{P_z*W7yDrKKqL9sot47s*f5*mPkT89BV}Vu6N|T)?eaWN84k+32y8gFR=FA{t92c&|l!nrk0`Y?bCKLG?#I6>NnqXE8gxmHgEQX>UtZj{%8NT52n3=_Z!&vQQzIz*~8!M2eA_ZetTmw9ejG--80Rpz}?B{0zd16=EFueXe^aABfPQ1G=4IjKyAC)-RUel@;96L@vG55-<X-yKOa=8XUDDI=IQRdhsop9fSE4CrBvv>p{f2?L#y4Ow*9y*K`A20RpX52iqdhg+@w^Zjw}FsutuhV&UApPND9w|m;grb5{+-Cg(@I^)~=`mTGe-6qR0YqFjp_MGDsXI1Yt>7!YHp8!2zQ6$q91yE+i@!HK`}yTBYHORX~5UyZ0I$m0SY+zOvT{<xGCCi`1~+$n(@)#C-bRPI=>CcV~YyU)?GDFHSCped<c5m(cMF?c`7LZc{MSiThC3(B-hTcM8nKkUWgBK6}XwFh=mvQd4(uipr;KiINfSU_p}lN%Ybw3>U_X`IxHH&$zjmkirwVQd+BGS(U|@T%vP6aHbMXk|Ij2tVUx59dRTsELTXjpnS}p=B<vlROW8qpLDaY4DZs=oAWeqsmcTU_pOjs6DQ=D0p<+2d;n<#k;++1$q{P;K~!GACbh%2fb&gJ0Q=F^h@WAk7q&*{_C67Nw(}eK`Q)2aKev0i?1i|sy>)Pv;?CsTTWN3ijes#YhY-5IKic2ixw5_8NAMYap>F$fez&uI{o*O(-)v33eeQ41G>NUVEdFsnUH-NC;>T?grL6N8&&_R-&Rst~u%t%%YsJ-%Hbwacd!k*?i&rmSEIa34S0^*Pi8_=+kR*8%Noh1y{LfS#!9*)J<Qir}=A1UZb!OQ+7Qs?8Sj##NJCEPd@VKXGce}luP90$^jMSefz1RM^bQ5PChiuX)T~y{?s-(4vIi#HMC1Vn#M^;_3aFNHFxnq@S?^!TX0ULOdem<H-PtoeyX)v8mR4a+jQ60T99@7tV(K)Uacrf5q@PbEXaxJ=0x2KE3C`Ek`T+UCugmo5%Qnt&b5xspqY+or&fcLf$5u^ZfaJ4DBq%s9sz2`3$DpCj5+uGSrds}TYJf)rfF1K=1GqIDy9-hQL*ki#O?q9Mh7tARYUQ8S9?=`jSWoL66k7m<)xB1B6-tyJq!gp&0ENY4_zW4x8)p{Uk8o3SC(e9J;K#$a)Xocq?8l5qt1al<;6a;EH2CGjX+0q=V<@9=6I(e%<&szWXp!70U(smajXtID5B)KL+aS>_CS|>v$>k>mM_}S@9B5d4~v^BTZ9?MqsR!b~C@8t>Zck=dQHMKQu)XQMJwWNH)lFGO}3zQcf?x!f_WkQ@UTuDoH+)ntwL<A&bNz)c%&X2XK8cB%xHI*)g%ecs{;B>r#=(g~K%V7tv+1@(ey>kE9Ybfq)koI)@vt}95UpITgd%vfRErY&swn2q$HSUA+J=c7zIY<4hE0=p(OE20W9ploA;6pV&Nt>buM#gs?fR*+Likj??vn!SMy+HO-t%8%0!)e;+#jWH78X7#gHTz_oijji}3-T>g9z?*nAd8lXTVqL;fz00M5VaJ!Q17rX$%s$Z($YyP=3|X`aeJeCueH;Mm~4Cf)7h4H!g;~C!Yx!>e-qSkBj=~TwW_>$TE_N`@0<g4BUM<gz(iYQC%$f<e{NNj2fELfFL!lroxJh+y0u+4FN7bJ0}$>_ycuWb!N)iwE#o5)MPE9H%Y5U6r(jBsOQj^@gVc&S3`~j&G)rp95hR4u+-V&&^+gDyiw9lQ2r$%#a}4Aso{gfa8^b2)n!%k?DoAcp6eJy#LJ-BPWI>-|1n1)9^36$XU4S=p=Q+7AP?plgD03`lmTvR&TDxDl_GF|KVnQTeEm<u0fwYe}xt23qyp9MOsT5#|keN5SMx!{hCgT1KQ&t>(hyT0T+v^QKzW!gg_FsFxEtL7ZDNgxCzmt5kNAUl;b^t<JK*m-;D5?cdNXVhsS~ase8{$#H0fUJIR8IjvGFhu7ouA-rnQuf|0zf+F9cph`{{UC3Np`1N%vc#2{=rnOK9EB$uT=_6WPBkj!z9Rsl#6PiN&-+z>yH}XzbSF@-LiK?Ci_zE4AI#u3G4*OL}!;qTgS3U)X86Y0uxC(nZ&4bTV&%ISvYW=5@*t~(`0xFw2NdLs1=?!-V-HMjd>Pp;*tXcBYOi5^TM(&Hjp`ZOj%?{XJ{C<({$e0*ffNwoCGbgRe`r?1kWI1uwLWA8zTx;p?I&jFG4cOtmPt@5Hc5Q*9w;MI0|XE@3g&t_eYx??>^D-1dwAPYbRD}jvOS79ui{(#M&@Y_%=mTIO3`bU4tn!TZzqSs&<TN{&+&3^(&4i{EFiVM0ajT9Ie!f?@{4gO;zF{M!^x}fo?s|+I!)=kt8;rm9-2@OaVbQdcsdd7FQxJ8zVzi$>fUlfT5P9<Qnje8j`iX;`(v9Eas#fm|{gdr``ZT5ByI}7FXz$siRd1WGZ7bmL6ADrWk3gu^uvM(&?-U9rlBCH?qUU6%#m;n|#L950!aR^~0a`g(n|1Ca|=n_|XYi*V`xrwKd0xhoO|Vh)ouA6S2^gm}RN7OB)HhYby&LA({OFXCP#9*F2yZ32bwgn-zy6imkM+_ZkTv(ZdU7NW07+HJZqUMUd8sdIMO}f>%tG;w)AC;L2jwSTRzDue5c&Fo?f+6cY0h<6{ibW736|lwn9&+ktJlYViUkfqS*~(jjT#wT?aG+Js0hEEmR&O~}WBD%zOV$T~IxN~MBXkbJ}xCa2(x@KDK`1n5wylz~G|6Xr0O&WQS^GA0fAdPP>WwbZ~WsI)Db=s;zv#xmqgV8ks^9#=WghyUWu0q-?<;Q#b}aP!{7GbR5*D@5DjU%SLK+$<FNAN>I{rTpU$pnpe)SbTJd#n)QOkIW4MBCC|(QLaSirDRANGF9ijP63z)_yU|t1G80tLBK(o8TCpQag1`q>%DsRy}?Ih?NUktSkzR{JYZ@9K$pxrP%lkFw9G~DEKE$iM1ZqYW~&4Y1o#!oC5>+$b&BtK6^agcfrGGKL$zS(B_nfZFObY-1PdZiC!`^9BG3p1*d~$xLpDt_DdOy<%cHk+5);xH-~g%zt;Oa5b|{=QK<vr|maLCJf1tw7dB+`rRoPStL?2k`5>NNGai{M~Y)|Xn8EUp#o%b6()&FcpLH?aiue{kl`gF8Se{V2#e$f6-FVj91k?%_A-t9bm{`lsvx=R|6S$9!Wo%7BBeC859Xlj){8tS=j45ER<`wY(vNgWRW&n;C<t_ToCd->TgZ>c9m@r3V!qezUZMB<lPkOCEpOes?uSz7{nv<ZsH<%Z-m3Ms$rm-vl#f22hwpZ9k=)YX1Ia}(Vyj(S1cdaqmk&zk<D#jX5ylMgxPQrIF?`*+VRc*0Q-BO^NcdLy=?#UXF)TsX70yL&0-4f+i{sDtjsbNUX?{r#i%@k73}wcjAYAMD1F#nvAAf{Bw^m!OHr2!V>qrKQ@}BE63a@t7rGBA=_uqJjzm6QwMZh}lpzl8)qqc^pNzu0ZZ-6f9$TmjaMqt4BZu{?1w$D;UYRT8<k-ddG{5Rmkg-j7oXPIYnU4VBA+n=Cti?d#9^4TyHyvDFpz7$Am!7>7d1ft{YEgUJe|du($HXMG;JmMMh>!G8!ZsKT+ivo5e?${jNRTS%fzx3s=Y=meeJJ0>K35mRZQ?b4p$yvsBW`3s#Z4vewQgNyfP4X;91oNq4035hdnND5w9Oa`49P-fK^^JJf(xMuuXFTz-&Xki{$TtRZ{enguJ=2TD@uAdi%bC7zDKB9<8eZJD0^D$ZzDamG@Ut3msajI=7r2omdo#2kWU(y9QzIT};O=##Z~pp784HdSMJY_;)bIpfQ_J3Gyh;p8$td>4;%f|PLi3LPy|8P=sqnC!Gqt(CJ26GRiL)oNdK3aV32m6H8as7|$j@7{O?RH?uclN1aoQ`wpsss50fyT}6USp%Wg7^=xhd!+zywW{?rZMKsf1;99%25@-5bvl97q_MC_nmRavo6A*OfL#yNRz08`m<V&YlTgy>x)pl3<e^3uCYCz@7MCPKtab{2GVLV?<^jT&zF4MWWiDB=gpd3<oa$QN|Jv%@J!oZ&+RXEZ_E2m6IS1*Z1jD5(o<}11*%EbOo~3{cL8dPPxfrQIOFFUhPJ$Bw{P=*id8%^`w@<o>*?LEP)OXTN`*mIzgh+1n%_tv@{Z?Dh`oiSZZ=O4WO;2E%Yyxeam8$8aM^?jqOkU@V)x`}Jrh*lx6aiMNj3vh$G)clc5f=<uxIMeV_D`)k-q2YG9YWAtM^6oJx{@qeHkmW00lOq{aUeS&qZBuw$CxJBbDL6eOvEJ`tgn3Mm2Z9jZ>a&}rLuit>&m4UCP?xZ%iea_$!!b9x32uKT#A^g7upO)Tb?A&hZNP6S?z6={Zo*}LNv7h?<<&UtrqxGT3#(fF!vt$9H7i2t68BPScI0W#+Z^~K&%p>Qa$r&EW|8TPC+>XwC<QGRVrXqg9V&DD7{TO&`zfpm0JM#Q0Y)9x{Ck1<7lBZJ=b@p4Se%f%lAfZuB}4oyK~!lL$lU<O}oUsQF+8vrsds!F%M44jn!TM&Rxsdun*Aw?#J^+$1e=ZF0&rWgV5%KzD}}j*!^VHCoR48!N2#d>Aki0vJPA2f*CFu70X7Rfz?3J6c<57z!xpFR5l{P2lD$$5x_1p&=mk}UyvgM%O|MJ)+-f*)lyj{)kHQT+0D$}pK(W?mbPu>KbMZbNTCp4a;}7Wv48!aJIk4|Y=tV@^~PfOlfFaZ`Y6&*b=iL2mK?QT-CfXanszvz#B_a(snkfr9b*zrXYrHj$T?&K<aGd6aCEOZKzqR?kc?^(=v-M<=PDz5&Pp*MdqD?th72XRvg@x^(8q!TtxVC!J#K50|AQy9@vX?uCgQp=*;vRjDUd*_C#^;apPY38v{2-zoHC@|(70*%SOrMG7fbl)if&47SMgDcEN$@-K?1odA%ViV7GR$kCFHIEC=qNVO@FGF#>*lIa|9ZTMJa&A6R%fznw0V6>J@-mn)CHb;qv)|-Sc^?*0M+1kym!MFh1Rz?mtGpKGEU@FqjN37BRvVAsgX2tC$7ZIA|^{<4!Y2#_L5^B{|jNA(vOhOUc&e)5Ovoz=UFfTqr~=#RgdgVv3~dYcNo78S*VAYHdK5NuPly4$7$&V6zclna$aEn&$Zj!>P{5ysk+qYu>MQwvb7rX3gFOh%~il2VtDEicyrNx;v9Nh9dK<jq%cBF*}KzV5>XMfn>K?`>^khdvB1$UGMRLb6UHaNrt9kb{X!d1Vo971(k(s#Pc#m@d&{0xT;cG{EO9B3_{mwI_r0y7vx}aLUB}NO`e}jQ;7?f7yeLDr~qtVq}MhYrgC73N;nEqhDfLBR;a(gOC*2Eko+^SYPQ)hZb}0GlEHB%kmzJC^imPPP6BbUB3y|~tsQ=-W0gvFX!65O`=Oipx1I=oaEu^;g)WtvEE5@~q>xOBiTpC7jKJ?wS7CC|F&EEW0eZ8MWp6O6^Lz=FJj?y6{&K#mT#}@bEKrTrN@AT@mkn2i5<5~6mJrX%#b6NHK`4`Fc+QCgAEL@iCjPYGM=Tysg-i>gw6v#B`+*$KPt(a$X5_L*Wq-0#k(w5Pd50`3X{)nT^Cu2Je9&a)=^W55fNmi*GIDnSIW#r^RnqAU^N|`Nlwe%&7_>=%D0tQ;tX+*ks6{mfCRg~HcWR-JBNe)A$$Sw(27aP~3_?9pkb$pPkikd92Ov8KQi=bAK#_Q`L5M}(1&R%VYJym0wC4%Z0w$jjTQI38Ae*m_81~?Mb89mG%bJ_G0<$K_Ty;T&&LFW>kn2xG5DZN>CwU^ID1ER>vud3VDkN=4zsIVt&~W^!wL<Mgr$#zwpRapFRH;r0!kjs0Q?*alXewK)h<Dagk(@6zDl8*}Nm;;!5MUjMzP9F6_0|SVBJVO_h|~piln_D$4A4{(#;GvOlhUP-!ZOmsFOFwtoGzt^Ed9|W;&!8M@~~GvX~g||OQg4Yb+)@tLvS_(kRhxoL!Hk>B&oOzbSjp-mqgKnQXcfmc?;43))8fKT5*lVRaIF#d~KlL(eU;CP6%yAaqgXFANBJm0<p5xVgVzZa0VFOFyKbTfF>B|JrF+CDp%k#>6^Bk7>jgtUKT;eG+r@(B<9bm<q27?Iuer@B2B)BEE$y)6~1YY$%*f|l1%}cVIs!ta{yuT99&qESxyS7V(HbV)$vnB4aP9Q*rZvkDRT*8C%m)PGR`<@CREHR=dw!-i>_oWED}>^5rIVi$#wjMXll(C3)AaFf)be;8knyLud%9}5`d(^kV+k>{Mh4pDUUyq{WP<ui5!PyB~>AV9IHgme>DWActl<n4jfB{NR{KxFoxNHaLO@%6}Q!D4)Xk!*)*GqRDax+{|(3cy?fpL_F!+#+h+i`3BliZ9jIs)kzQ$GgL2+8uSwjQD{YvT)IMH-gCofnVDLbflJV=`y)efL|Mz?K$s1$E@Ue@-{Z=FA@e=`}f0Qm{Abl0cw$nJE&r<LRSwNPw!O&iJV!%8Ccf_*^iRJ3FtVRpJq#<n8ZiZc@O-b%ASyA0qRUs=f7nalwjvP>|XO@SOl*-ftj;HQw;E1m!Dyp^7-t(u{{P|^cYkied0qJltB>9L@4G}?7ifFPmjGF!<BU9<zoGBx|){>~*BNa#bfZoJ(lm#}+CLw!BnjaFg)Ho|Dy}clyqmYHMEV(v8q`)}VKAp#5PBWpH1gj!{>g}*~i9+K-6~*1=-&<{I1HE#0r*6}O$9;x~c)HUabT>cS+uM!1d!aE7p4&KR!|F!NJLQdip_)yDsb0$_nf+qOl)+55-K!4An-1-3N57o5x3KKXnVoQOWv^_~Pc~!Tq=qG}HZUk*Yr9vwp$qxm#WDfAZ%cqCm;}2!7xpIyyG?n}Hr{EEglqI9g-c~GoIz~5JlVgp_j<Y7RuKut)JD^^gU{W&Ixn`=-X7xmYXQ+GUfC)0w}@rn<mRji=E!K$4^Ej=gG}3f;a^@bv3zO4#PX%VAd|1Q%pBY-+5Ck}Uu?&LB~sf*qjH<KUT-QY+9-0S4|S6_N^Ccl@Q8#t!|a){Lmfr_PM3&g@WEEyD*45cFW}_WT`b*$$v4VnOagMu+x-4_zVX-1^PcpwSaZ9skIJsoj+G&nwcV~Kvy!9YAih|iDg=^iLXJQd1gcAp&OjUV8ydS~Be|tC$%2*w#6z-rtaGZiYv{qy1^sDXJn-4mXrhY(xWWk?FndL`pp^k3ULv+=hftHb0aHSJi75jmf-y+}axIlGmG-tzqMx%ZWVt_50DsUnoIUKTSUwvGUrxc_z5W%y8=5u8#!)R+ob3p;l!S%R7L(6OzSs~d3<IapfB@W*fBHgu=%~tz>V9qxwYr@&>Y3&W{^$4#1|P<sw&=Cf-u}t-J*Jg5_ZO_9teE-dR9HntZ*sAr*rXB_dvQ}i5Ms#QW=F++E&}i9NOB}3s|8MJrA!(5sfc+Q+(){pd@wCeYM08*8P&vfEfS>EP?eU{=gbiTtSE}?rs<@RE81!*RGW{4t7tz{Sdq0UMn%$5WJ?n~uApheLn?iJYyS#irOs!%OVZt5eqZ5CM;R%V%N_1*0&l;NS^(Y%MR#Sbqs{pjCKPl@o4l10o+`wB_h6R<&U<7<MYnkLFHXLF^%cC`)(%o8)!TlXdJ!o>l*?s@V*K{i$pt!izz9w2Y-do#>VvfXQIr^7o458GPB>HbN8fLnB<V?AE!`Du?39BUjl0=AJ^RAVb9-dby-5jT6I2!{8no46OrPKqW;-%hUcl&On{Qk`gYZHisuRB>y*r^dnch_T%)!Mo7^*YpcJ|NgoO$JkmplEt^<3IAUF{*~?{7Xwv1LiG>fA@ojmsk&PnXa3lUt+Iv6ZvdE;y3|cvWO6g^*Lu$^;%j_Ovd{xT;dsP5^$VO1f3f+7^T#U2gorBg>83WYDz`-yBMPTl=z`gImbkLw!n{6FwcX|Kf4rnf6!i^sr%g)x*JTVls$z53ic3?|jgS82s*Le;0pfU8>$5<(*GPJ}Lb^OTXtay}p*9|DWdP@AbA5*M<-OeAYvw`~1&Z$>@)J6Sudf#iY$g%s$#<UDU8&+-BoZdFooKop~+UR!(kV0bqg&YGGzl25`C%sF0*^YqZoc>#UuM!73ILK&9uwMt&Nqowe+%orQC3TI}o@^8j}%ik(kvHe4Sn(^ik=la?AM=E1I|q4xh$f!2|YBgvylV$s|nf4dSv!~<=2CThH3iUajJS+zM*$+;1^CxY|Z+H1>_G^tR>oTTz3T_SBN^FWS^A}62@Lp?Y~YLcUpHBvc+(5jFYx=qJ80c&H$`(62Y^YvgRJZojtt;~mO$6|n-oTo@uVZPK%UgUz;N*?f7d$N*FBuPR5qXq3er-E`4eM-S_mXD`xGA`(!FTrnT{d>I`?XCGT!*x9dKx<)blz^Ee6Uw}BJXWI&Dde+8`^6U|J1TU^nB$l?$dE#Vr<y0xO#bqYA4u<$IjdYQx!T|Y9nV#3vT{_3%*cp=krQA+7-bbuoHp8u1>xW+`Dfa7;bAku+9pr>V$Ib7sYJ}2h#;A;h`1~=Iv<1++Pa{$La5Z9T;xND2%?-;Qw58oGiR&S`i*dwlwLI|iHnB|A2)owYqVLhb<f!8sjE__s;(+G*cij7nW`8(@ch(N_he%MZ>%Sy=t{@h+61Z()~0gL6C!Q_1wJ~;j_uFdR=FP!o^-u=z*8u+$Q~qt(W!UT2HZeW<%rA9@L)nhBxTeqKvocim!>T&<`^(vW5_lhC8;W#jYDRO<xy9Q2a6>JRKjH}SQ5S<zvoC5Q@R*td`^y3B|S4mW<OdG;ZkX}Qh(ptE!-T$mOk(M$H566j=K^+?+aM&_bjEeM4a^JZ=GG<;tJzzgSCzaz|buMOBb6Ta?j?z-)!j8XpO&HewfPM<%4Ej-s+^odZaj9;P2Xd`2QO3$lF5&P8)BpPi8m!?vDG?-fe#Ho581MTK(Tx3^oMfcX%x+%ie1Ky+7ac_FljH_+~pVXO5TW48+G$SfBJaG!J_3b>@GK<JR^5so&_f3%}mmfwT={*Pm~%=Y!5Gux%FoRf9s_Y9y^c?(y|6dZ(VY3$PDxL)+e+z6+vN1nbdx{~PbjJeuhe?C$wrdRNWQCTH5l*Bd?3o3R#bLC~;s-MTqm@0$!i>R)|l(m#lwZ)|MLO`+-ev&k~6l!b{U2@x^jfZj<P*U0Jv$zxM*DG~-}M7CU^Q?JYufNsFTm^o|iXC|BdW;-oszd4<iv+vOF!ySZ!D{Uk9J{3!E?VP90q=WMp4=!!b{>E@COEb|si_>wOcD@@94kqVX`L4-9*t<}ull>%J*5>rVHZP@r*ql8u8l617wD=u1oLrroKX96!GwJ?|SNd0b)xW5Jr*n4hokr(l-8+rm+w0+-bpNxL=0C3cefwwr)y0FdZ>)Ir`14=wzoUlV_x;B%-0tXD?vW)RqbGjCdD+`)qv1Mn33RMU=MZo|2$gjTj(RdPAW<Sbr^stw6~_#4Tnxn-eRN*_Nx#7~4XRB|F~!dHpH3CVs1K(^tI3zh%9@cU$=2LP&^nX6<B8-}BA3Pqrm1_3ksw70${j=J>yw`8{dWEL#sgR@F5h~cm8?owL*5pE*s^L#Ef1BN<1MuX11p2mS#Y<)jVs5pgSb2OpS}IwDBpG#$J0c-{#%!HL(Y(sjN=wxWY7ZAEVX2^6?u&@k}qjy1%S)vzydQ=*U&0sJX>nn;)&I=`=0%#sQ+OzY7hI;s9R%kftyRAQ40r8LSH*I%H<+~EQUm-fj$`UG?$nrBU!*!s{|NAWK<C-0G}&w`mPnjk7S)IrxNC@U@JIT)3m-+Y)0zpykdgNkyy^28!Qse!4|cRDqB~p%=jM23OS}^QrISoMwhwL!aa-n+Q;lalbE|yOBNaxm7Iz<yb2Hqqk_@WISh}(YA7P>C@Pt-C=sb+&eKMbeAcCAN)S|}Xt`k3J7J`&JPOjGrj7vIaU_)@XR4EA>y$!d0^Cj(k_MhHk6+;(^xqpa|G$Q1^+_x6y50@o5|s{2DuL{lR4OX*oRR@ARa9s$d@V>AAp$X%l5*Cy7R4CL1y854)3obXqpdkhsw35gI)uFT&YKV&GI)-PY0tp1ZE+=O?Ug1WIq>lkfOPUPN^OSf(y3i~E4tI1p3&yet(KwO8rpN#KAtsGa{w~}kxPQ=1HL(9gW#GQ1T?3eXDOFpm?!`UfOkrh5VFjOzah>i!nq9cckMRV&-(lO@VUQ#;kj`j84ihXmxQP+p1DENb>uvSK&lxn60LcrYfvSp8Ap=vBRcv$Ku)~4_1YWAl*3ME+CTFT`^WHdJ6UjCtJT^B9t%kgV}+E2o1$WTt%8sa*`E4uYxV(C3CTXyMoT3Df0?!{<xxk#_cofd%0Eo{%J7exzUzV6cjo{zN5ZYR&&_7i-56?x?l-@9qnUd&JnfdjuX`%L_(}I0L$80k{ripPtn|)gG`&B1el3Tdn0~AZDn8E$ngVFs=EyJ&tLH%@w>a*|6_qJEZHZo5?sOpoUH~kg&1pFF6s|w?RK_zcMjB6E8Ex!}W6!(&>CgT4(0)9s#~9B>W$6h|GbLtfq;f<<?W|1soEfvaB6nwrL{@@})E2FDWC;6IsR#U3tTh&>pu}hTbmTYP7F}0DLB#{>Js^?DoLRV9lA+dN;9HI?YYl>NEk>Bc8*Nh(J_ME^|5);h%!Ozie(aA|RdWA$B6jbL$_YPiQ`g<u>cQ*dy}aA~p3`U3-N^1m{fOPZwoJEf#1tQ_P%P_}@a8vfhfTseBSLb0T&i-EO5SAdp^WQ69xkm1SxfYb&PX0o@>ph-lgcZPDH^YoiXxO4EI4X0QZ}L)#%ALa!XfaPE4qYw8ls=^gTNgK0ATVs{DCsBtonG+Z>c}*4{AO=@p-XKi0-Jl@}waZqgRy$ag++!rm~<)EDQ)!nySgM$poagNM*t$8iJ;Vy!Ei;r3}X@A?W1(rl$|hLuhg6gQsw;g{kYJD6ArLa6s;0GT>#vEiq^DcPW6IVm1W`uu#K~$E&L$b54>P7sm)=A2xq}usV!A=t?R2s<`>OpHTq&Xh^Yld&T`38}Uj1iFBCr!T^#6(Z#ox&YkH;-)l0<^lj+EXQBxEvI@rvg@>%F7My9xc(ztJtBTSY6VR4SCXJd|vf{~Bs;FZ%#fD6Z(owC5B2;6#7pN#4^9RGfvwMI!vA6Ylh}&f?#bjnGoyoZ!#}#VK7c1h*$Ia^hUrTFjMn_MCt?thD1%5Wv57Mi@*6)Vh>+5kIe(i_=wbpP`01q55bdX#TK~l_|gpqwl3{JIVrx1Jz;H4lENFXm{A&XE`Wliic5nS5a-QPdImA7^;bu;x+2|KvDFO1&U<4cmXeG=CO<=g40|9U%UF$UEuk%dSCG$m?ftgkgk1bmQ`A}gypRBF{0N`l!a#7nRkXE74lm1oVLksV{ld9-odI-K0<&cJVdc6MU^?1jnP3udc{o5Sw26Y2k~tv{YwKp(fKzyG0KVV}0YIV@N@KyS;N-=4huSKphwdhXm`PkKbx&Xc!Wp4G1Ex7)6d$+xH|=i4v;m&srLmr0Xg2UvWg%^BDG8GL7Qxcnscd3>OJ`9J^X?}R6_42hg$vII@aQ?W9vSuRv6AZbbwm=IJ<uChv=G>D3HSo8Q5@_z>1Ec0E(>@3%-w!y_Jn@LrKVVoTGxB&k2(J@ys!?I79<RMGXK}kv}4DeJ^D!~;Y!E0`kW?$)TaIRP(<qsCI?ox3sWo}C#h57)v&a)toDOM~G!B7v6H9>Hd=%f^s`5E5^N8+%_0m})@i7W*`g+feoPD_Ky$BZdtKB`1XzFLdv!;x?_SWWQG6!lDR6|_Y$Z?)!hYu+RfK2y=TR8F{xS(S5AA%J2b5G1EaMLAa|gFwRZP6^tVEQuMCqv8H5q<Yor*<G6fXTxx>dr*SqWOym*1jtf*E=iMV<YWo>9TWFS8>wnE+>o3$$9OmdkSIWe&CxxIM=+F*+)TyYZ}z>OZ-NuHKOgefN7IM2Ze}Q{ojz(?BI!p-B}=G)ouvRKX<tpEDO{}qYY8EhlDLvigzA$iz^SL`L`h0tSPmHpHyI;T^c>_vC<n0T6_^!RlqU&jUC6WC8&X*3nWbE-(!riim6UJp)V;9RY!E$x92TH%^5oPj8Z}o&hoGiPstFPa<iA>yPD7Y-4%pOLl8GxyQR|F~HZ7{4tUR`Nt0d)vD&ECPMXe+#g(r@{$Xz6N8q2ly7%2cM!!d52WJa9x0U%j3@{|ecX=%Eiaut-*t?4QiBQ0-))+D1O;jpBmD|AkTRbsG?ik67Hsgg(?Gc{0=Lhx1#0ca*qQ&>@o)68so`I6^WB;<N#5lnbC)>2!$tw<wISP?K;bTULIL*dc7wV<l?NsP9IzBdP3oSDB5u<oCR?6>BekHuI^Hi7osv05X-u1|^fU9_rXLKH#IlPJxpx&lK_R+cDTRa8T?st!a<bN)~J2YhqrD_L*4Rvy9Jg@grDRN-C#Z<<m~jJWNBiDzo0S|1R^96^DClL+{p0+cWFRt<|e4k#!PKOjTv#9~lEsUT8Lh2u7zW>p0nI78+r0UydR$!n5Kn}GO8v38(~=SOYp`=g#9Wkl5;i${_chYmhf0uqWzgjC#xOxEj4a(P%{NE#+Ft}t{yNo6^k`3Qt81VyoA_at8-YDur5(3ZC_mIrOTT<r(v4y9kdZ}N!Dd$?VG*zAg53P0FAx{!_sL0jq|qI5W`K-gHw+!>qkePm~Z4DHA&tK>?7Fw}%J!<lj{Q48@Q%OvK?@suP(JR%p+Ob!CPI#P4R4>V(#A698RGz)wqgd`(eYP~8A44H9>@n_{BcW3#kye<!&T#?zykTgnMl{6&KttzMDD=O!77V*+c5w$Q;#cG5F4v`5tHD)U7!Oh^EJ;Q?ZDHor-06j{ITq2|ZHC43&uW@p}tc=r4pxP}&V1rO>GG?DkX>lv#dvENHg(@0wK$C<P@OXpjv;M&0?lfZ7-MvW|%DX-ox&MCN>vi9mHrcl_-d)UnaniSq-WrY}W?n$j8ogCr6H6xeFq)nhcK5EHAz>}d&q<2_zQ{&OHyJmNzo9bnqIHtae_}x1CSAcIGP6>X{HM$uQAA)q6|7uRodu_&QtRMdp!fhvP!1yr0x7FhcAoYatOQu5LSAo}^<cZoSIfN37(z?}z);L@1S*L*Q^1*@8x$wTd+GfWfAr2wv!il3>>*=L!^6V<jKoCQ8Q!RW{^u`T33~*vJ7o&eDyvx9`{#eq=H&nOkB<SP`grgq?C&6BA^rxUXD))MMQ}BenawURAZwC_lu{(>ht|dA96Vx)4b&(ASZcpY<~HqK&-?8*!<x?$6ks2~o@S|JU_GZ0oix4yz-w9LS|Cpf#k5A$Rl$>XSR}^>5)TuZpCU<ZsS5Kd$OOIuE*6W8k=jTQn89R4a@wk-QWPZwMIP!+Q)mXh2u0Q7=$*BF@^{9)cWdt=kvb6O1F_{5!$Y=e#g`<kbu~rCKua=L0a+`TC>UT*78o#6@2FbV`qe@3ZO`HVZa(a-p&+46UO!tiR@YbzF^WuHb3EWtM^>3kn2y?1=6ofwxkm6^*pjfukW@5aJPRUnm5;{i$FPK2)BFQ62OlAsKrAXFSv6k7!Wi(bOo<n;#3VJCr4F=Hl~o41iCAGp02e-MhmX=6O80~vFI_>#2mf~a?#$`@<Fv3mCL)hiDGBgei4sbmRC7iG-a}`^nvt4RKsY4Z3Q|ZfiCfEUO+|oQX2B0><43U=eTXhZEj`yIk$^ge(;5J8Q-Fu94m^WAH8`J1=A|_m*`m%%4NG^{A4+p}h1{_9hfc2m<Vr}XG|wU-jGG8lhFf#Q=oI~*kxZX;%F3X0Vy+<YSx`sPik6`;dx$^5K*)RD7Y)+DxdEQ=BG;TN7OV#rL@r30^TMc4H;|GJ)-S_5WR<=mXNzJ9I>xfWID_d@ae46E`IHLclOXQ^8!?{Vxq!4u?sa&!k`_ENNkOmzvn_{$N2Wl10)VtL$gp@rO;f7~YV$K#qLL?oa9fokfqh4|91%H}fg`)}tRGIAC>3Ut_N0{+bM-+hK{XZ$JT};hM^g>+_)*A`D=umVm*89$M0<X;q6Dd$k(BWtfA;J32l#A4NIa5ZDnk(FCTazE;S1&qA_5kDtrm-_Q79r#PR+qiYav{5$Ll|8Ya@C#Ix1!BeH}Z4;5XYk-*JWA!*%zyAhTG<X*&R8ZH=M=ODQF%*qDl9Nah9Hp7goEV#q4KIt)2DBNh`l;WK>_{gXlcpc$)=$1-SZ##rPd>;c!2=Szd9G$#S2LZTudgdCs?z?TV@yaNjtl9FNeo=hMFKgxo=*$SC2^Zc}LOL@PakpI-{Pu!V|gb+Fvja%aZXm(d_-*~aDDC#QpuJy8V*ZYdV2kl$k9MucGGc8VfVbET@Hvf5drrH|qCeQSzYbhcJ&ZGWSM1X`aK@z&!I6&GIssh$knZPYZ0LeOKnJRd_wSsoY3{NS_6D%ShX~@>8miDsx&NoUtJCZXvyZghfy!9ib;EPxD-tL96v!b~3h>38$9o<LHhbJw#w9M#|t8ARc9M2(&D0!^K1}=h&P9nw_Kp<<tE<Cqjd(m=7R@RxPqr$P(YfmtigN(Yw4KM+h919p2!E%{`a)SIyqf?IlZj~uWuT@*AansV9YP6n@r)hV4y9N9Mk_Z55Y4_4)#GklbfYtUcg>8&;o(x;*9Y|{wX4{|kGw}QWiTRganb3b;wv#jeSNqjVldYY}AS1uKvwiisQTyTF{kM5B@XL)!z}Dr^NW%A`-8)ydw|n$IGkH5v?s*OG_VwKpOc!^zTFqfn)!ym_l(t$4nuZbVH7Bj(&;I*m6Z(&D{(7{3e6v%}>3ml|Z$r)>-~8vcw&P(ZPEHjeH(Gj4GC+OX!|iK-@!XuS`0Txp6u|ig8%ciifBgG@Jm%aZZuGi7k;$jDr?UzIq?Y_&!DGRb6acOBz^8_UM-<Zqq(@R)T`?JpN+Yc+B>&U2--XK|8YB@LsO}+WugK{aNR4*Fii|1}<_y+H^6ARllGQyzvW}b!wsv_h(Ba>I$?{&@d$CI1%e%)__xfd1{5uuV`gwmIa;LBSzB5<q`{OqD-)k#=f7z?Q-D@xXr@e&LyZ!xLf1G#ywC@|c*0;MgI}9U@x!Jm{7lxh?y!zhC27#L+dE(#Bw+XaRVyV&<;x+Yn7Za5PeGa~gph3c!11$4i*+P3k!AnGL1A=4)GqR|P+30QM<&-O3Eq!!N+`HYi;eT=6D}FM%SH4$-@@DWyIWwykHFjgs3vMj#)djsQwxC4i{A#BbH@tfB0$yHt^{ev&R!%Hl+k#%#OJe?K-N!By{+fRN^p^O7I2WH&$)X#0B&+$dq_Cuo6)?wXo(%%3uQ_ItWONcsrbvx7<pHIu=gykMbRdI64Y~tBZLQa*sl<r7Oe7;ZOn0J0mN_EJM$YKyp5#z1aI6TV1eHzJoGPk(b}|H;vOdl+%%B|KH6%WjHblomZS*UWb({&>c}%`?a^Z{s<Fs{I99~3?QAoQXxFlt!S{5U{Sbv44^pDjvqmS7%!_Ck%^Ovt2-6Qj@?|iR44EkOFHaCV$c>JcBm%lmr-m8DvUr#6RcBf5Ni=8rWcZ!sQnB~WvDBKeovwqRKmnCH@s|rz9)uGDBA~B|%gfJjWAbl8qK_^aR4iTIp`Xm$99vLAkgQ;Vs%AD8as(4y%d|$LI<4?_vPdcTvmIbsoBPB?IWS~?Mxq6)>uVk1PoF?&nI;Y{HAiz3I4iXh3bA}@~n%k4?4w_H9o!1;nHK<G>)9YG_unf~y3Qz3}ND8>k;Dq2jiip`_@K4k^*`l12+39&DsK@v1qT45xnxn*Urq)}npB5cbTGzO2Et+r5d2jJ~^DTPvcQ@KD^GAK?jDc47Y2Ev()8L;aomIw3oybU}uwn&9CPeCA5TK?idT`XJn9NmK5NiY>m!m-NMAGN%Pqm6O99-JnzkIPE*&v2q#ehs3-`m}s`Si_Ko<u*YqVuuU5o<1yY&|Phod;Gap<(_!SQfY-*Lo_EM0{o(Ba>+--qD#5$*D`O^&xG!TS|mBl`6cqF`4AKu*no_OVW|d55NRPiU5IlxDm`TXr~KVy)PUg^Ro_jzOuR0@<Xmx^_Th*4d*^_A(Rt2BCe!FN^{8C+__?c3!t#xYepsfrBQFg7+YsTjins&<j9VqXv~1(nPh<Hl{=nT0q_!uPNXxU5$+^<U_Jo!%rGzMbUK%&b$Llu=S(Fhoe<-e+SyQ=%1xREyJy}AuhZFa-7d|Pc(FT6?)=U6!`1x>hB_Sdze)CHTl<q5w)e}^s=LfVkcMXVcv6l;Pb2sWW{y+TI}EfWS+cgWfWTmC^hj_eCS42jF;%Azzv$8Qzt>$J6|TV%c=bYQiHX|~GQ(u7q;Q&;Elj4UB@m-->H&p8n#Gf5^ec+%_0tx2EOps?{SoMcCxi4QF)};!23J`<OUMYqka<+WGOw0)tc0N5O-$uPP3tOG(zw9W5kkhrx)--1T&)UK#yhe__LOj}m593v!LTyg#zjg))+`!aT4~Z4PEII8e+hz=hw-S>(`qd{fd%^~+*(GZF%}>ec`}mB0n#KDQ}IA$TM1DVqYs1|0|;Eo8dT2Up#lU&W$n>ouYfRY{b(KtKo@2kAsA}t0gNx;%P}QZZ4C@yANgJYB-YGzCe{iRR{;#PTkIg8i!j~<MyDExC|IP()WOy0tc;n=pfKCWZ=v9a0ZSwmtPtQ9AWj8_fscu+{B_1V0}q~?krM?yEu2%mAr`*l8h1tIj#R@bER{?b0|=z$V=f_JztBK9;k6#AHO?AcUc7Pvq~ii|37PC{A}59Ya<IR-ACMY;uzOW(zO?i0Z-49in+JQPY(CDVUWq>dvY-oisTE0@d<)az3PRZAKTf`Tb>|@b@X^KSJI%b|QoVM9lc$oPx>w|e5rfQGNh)v-(If0(ygAUfGTaCN2V@synqmMfkwJ20x<1pzHn$oA_)+hl_0gIOY(RjJe5410i=`T=xN+HrkS)1s<tzc(b=1*vlX;FjkXg7x5LXF{Ew0@^ODyrvVsh#FZb|je+x^1jiCLS|W1l5XlYJKE1*n`Y07Je8ONNt%0WW$l0EbAZ9@MZRS7{w90=xuT0U`*2;}=y5M;EvMq}xtzWAN=8ZGpj0rW?ZR7bmBh8UbdSs)?AKxJM9wCS6e)^jIk)8F$ggf&k@HRk&lDnxblz3}WrE6up1dFZMfweQvv-eSa|4eQW(U&U2;udyjlU#h)QL4t}l}bGR>HcMPr0UU2{xV`@<xgbrCn8fQZ^$5U6lGvivdL|yS-+oCwu(d+8~sIu<2eep@BSv;+<`=~oG=t{GeeqP_<Jw0t0#7jQwx8eHgyN4qKtlqP$S3>@Ie{Hu1@y|IMu)&h*Veb#y9cQ!$Sl32%G<`-HYF+!|leHA{NyOe#f!RW`f|`hmuSu6Iimg&4M_S_&2+nF!5T+V$+JH1cGO)<xPK6@upxX!x$Vak`!Yc0UYB|`)nQK####UPRlTp`ESJFbqXGbYE&X{X2lW&{~jLsaoJ!uhPPypI0DR6vGifYbs%jtB|DaXJgtEnyos^9d284z>j5$F|(ZDk(hv(~huzOYBl^5uN99k6!@1k#-8q>=745u+=hn@sgPn;|;^!<uRszCwIe$vOwlgYnZQfiDUVA_)GCK`bF9t-*ytr7joFnr?N=V_eu#RcFm|45U_d$48rm&q`;9=zwUD637;=Mp{y&v!2RMEtfU{jT;@L6(RsTN?8LJ)LZ!s_R<{lNRuhDVll!5QOTAeb7~f+gRB~btXM<9PEvA`GNktZOx6QbDrYE<x_qLoFns;3FhM)!75G?{QI`THM*7oI;Aya^5Cc>Kb0d{`5~BuhF9J_P8(T|~@)+eFKO52W2TSF3-)Um=-Fe%SHEnkQ5_8Q+k5S?-HAPaTt@E{1n<Q|dR{+a8accg|0*izDV2V(nxi!kCvE4CKoH3RWo@TJgVO#2Z<54gA^0rCDl5G-@eQt4G(pzUv!*<DE?;>bjUH=|QaGKX_X^0W!L8@_`(Fzi1lFr(!MZ$HbKoaT61D<Q`sfh{y4NSYI;p_W{FW=W!uPXc%OgwJo@0}*S-yDi}`)KoFZ<*3HqrEqXMLnF7gsB(6n}c*>YePAM7C#$X*?!Q~m!26A1x2{70_)wbD)-D}#?o@u8Sj&(7_JS4L}7Xh-)03*KXC*n$winr=hPSDLC?3}8U$k=_5NdSH0+Aan{u?ROaJfGgR`&p>K{RK89(V9cl+M5)|~9u?Wm6wx0`}LvJ)O2MSH>BW4txGq6h6Ohq*Pr!nVSB`e+Eb-Dz*?`l!Y2dOvo*c!c9GmV%(JrUZ0bt4w5ZDRST(xQEIdIS^Y^AkZQplvjXl7EM(-G97{2^l5keT{?8bdOaZr;1jBZKKIIkH?CVK|Fx@YjF))xy2YO@_yu>qyQBK&mj>hD=>-h#!fL^;_vMA}n4_zvr<d)_LXGrGD{X(zy?SQl#}3ctY1Ki8NcpUi@--4zwuU9L74V9@>>~&z9bQ{3I4_wGxq^@+$70be^XUC)YR?HVt3*}6Oev}-GPo}r00=d>nhi+=?UfBtCo1%=CbCj&6e<~}Z2D@}Me!v}f23*G?GrAF%j#nlK0TgRhAXsHQGuD6bPf-usY}P>0kEbWUuJpJK&6;xo|=gbBUxpznW(sk^GE(luR8J<tB(A0Vbw^YNFvE<lF&?WlC#C6FHw-{qb?HDTuH$-o;eSx0s}|dhGxrH_4mM#n}>ps+Z7oRkJ_JnbN}KSVQ=#CMGVMZnEXFm`D)k?7s6Tu+LqDg8Z6d^CxU43VJ<n}m_hSgQw&~a<}>cPV!0Pvs$toc9E!(0qrb;T)cNq!Q|H5<UY$>57IP}vA{EcdgCRD}wARI(JQerl1n}P2bZgg$>3h&d>pf^J#^4xkocH^_=;n*1#x}QaG%lT(`<PXiI&l4I104oj*6z05n)Jd^4}0gMK>}h5P@8w}c<Ajn|Ds!0{nm4T)Hk{+x`u139Cw=ab2)77ogY;X<6FQVWserXWa|>*;{Fyj5%;9;+`0es3av4$qJ28^#czFm|M}L8w0SQYx#kJQo%6frn^&H1-jTLyrY{@t*?y>mdgz<I#ZmJYJnk>;q4gi|h$pw&sOVU(sYH99TMFtH*XHO6oIki&cDjR%A(T=LPq*<)!*oDJKgaDH9&Yu-AWB5e;65VnHRYfhEICGG$D*i`Cm953WylH3)C2&4aAlPBq8b;eI`lL_lxS?B>MY}Z4O}`Vd_d|ZIWrfPG)0rO6aL1u;Gp>>IO7zvMT@HRv+c3*Vbdzx@y1r1?&%nAMCzJVx&m}?pbJUDg)Ebxno_xHVR_;K=uLWG3h)^9MpTj+#UwnImeBA2?e?unFU5DKZCqXN7HKT9IYi_&3huoV1$Z|wfJ!A~(Jld)DsNc=d3H+sN_C<dugW5%MWN?sSF`XB&AWXx3aB^MJlhf;a;o44UL;NX)FG7+$fP_N2KX6yVxU(wTksSI=HPRYB*@Bz)x|Hg+RXRTJnn(&Nr_|3t6O|9+IE|9`doILJ5p=8*9^~_e|+<w&otYU5BkH{+c&0r<12J8`cG|73ZK9Su&Hxdsh4s57N3h(FH;(QU7g5`;U<#xhLM_-Vp#xwq-s>lTyd`0P^qH=z_L1pbm48zDKjJp@LgrpalB_J62CQmQ1_<K?kPxIrlg3OS4xE(<P3Kxux_%}3M(W<Z|Iy;2FIy?kvwvB#9H#PXb%?oteuwYL&cAAVBR#}f(^6DIlSsB9^U>R|Hr?aZMVaJJC?Ip)l#f@(^LFGyNCSf*K+?@rhO~=gqz>{pxNba!|mPPY;TFp9&gUPTCCC9Q-3z;_t}5k+e3Ul-7ju)>c{PEPh}w{Bcy(_FYju7gm3-pP&>3J<}s8PHNP^w*H%cZ^P&0+`u;}yW`mXp`FqY>%-i&ePul#_QiPlIgWcLzG|fKJ{@2dahWkc)B+%N0KA3sad%YG(TeH?a@sB5c(L}qketTTnG_p|aPDbtw)8xiTz5d#$((kQ4Q+zZ!dFbk>Zp=Ua(P2otK~0x*tACoE6w00M6sNh|1$`6xF8yBrV5wm2b_<x^p8l1Nuilyd^@HwF(bsp{7cYN0L#5fZt^s{G8|sd<x|H(yq`!v`hX>f#vOO3c@!R7wsOuZeXaAwU&^xVA$vfTG!zFg5Z}<9nesgprb?t}?AD-|1=8@kT9XUPnjy_m<M|6fo%9elVM)R+zacyXKZCB;j9(7fN#PhHMyQ<MQTid)zpH`>ttdlW$9Tw2TE{t?8?Dt0$qrHhc{X%-MP$GD*E7fYNfi~JG@t|EggTCJ&Q}K%?c67CU%?fKL>id0h;LJU=&+W~j`0H+eFnVvaV%{4o0{wh6mG2CNWdA%F@XM{{{n|sDPewM?@3*XHGQ_Rl^nJ)pty%x1e$qL;bwQ?m(%tj8-hM*!oLa4<kDB2h6=!S=hoG&v=TH-Qr+*wR4|`|yynZv($ITrOe>Qc-?juS2uD51p#e3cQ=|{aUC!WSwn#jW)_9NVFetb5Bn&!UV?lZzLV80n>gXRYBjV|q02ZCqr@!{yv-Z)~}UF$@lT3NPU5o(wSpY$(Z6&a?t<{Bm0Tcdc^ti9%&eX#gLBXZQe=GWb8rr-Hw(1vOi4)63$GWW;p{#R`o&VO&_<U5q;n)!RC7Z|{wGXZ;Sd;R7M!*w-+OWni2{mFbz9BXmS#GZlO-fO4Jy-uvOsn8sqSJL!o2k!2$SvJ1JyT`c159lh}5J4x`w=sEdZM6Tr-Tn7&_umgTRvGs`YQh{o>Zd1AO{lXE(%KlDAQcW>92Ixh<Q%YWEG9cZS}Gn?L4k}jQ;RPeq)=GzQW1yD#;m;XnJ4L^1Tn-9mE$rPQw_767iGZRnbV=>Vz^*l$0DO+Ql*7*M8(gNThYqn6KfB-v@4EKY*a~0&2<r^VJ<}u+Bk=M@LuME`>;kzpJk~&8ZC^<2IPbX=W!)G9$J9!h+k05IvJ=Jm9tWGwwcjhg;H8D=96UAGJ)sfvXG!V&XWr-D8N@(B|Io^KyZW#%tdE>R(Og!3Jfb#!9;Mvs9ClyNwUf@R%Df=!eFO@C+|TkQcg1G<_yQ=$S_D485@dg+Kr9OvPPC7)6ONlMdrc;#w^2cC|93#mGDG52$(Hg1!n|!Ge})<u|%gU9j0<>h-H{a7LyTAIWp1_qApN!-!)HxN9Iy-L(KxrTl+1iY|CACUoU&->y;n<=qd?S)a|W?YR`9M@k1dFg*+7gNh`2_+N8gSO{l)n<@3YIvhUwrjD>Oc$_`dYBc&0xb}mufOXs8T2#bZGbD?gP?c8q^K01hIpBRFo(KTbzX2eZhOVFM%j<A@y(>&xzool)9f)(R@K^%&VM8goA4kcCu=cGwN_6Cjg1r=6H?Smy<-=KtMO(8L+Q7k^zK>AIH!PQhH^Pw}zc_9#&u*Q;Q(lZyR?3+hfh15lgpmi`ja2_!wG>;w&uJJSr9IG?d>^wpz!=+`$@waNSHYqvbWl6e5!veQ~85yY8C)g-LArT@{zfyKCj0<trVBF~se%50WtcvGK9U5_H)S+KhZJvs+H8Vr?#JT7caa~G~hpE$8D!56~;4n5z&dBXT22zTWqEr=;hsq&BO~NlmlO(OZVr$UVSbCK(yap^ok42DzHNhpG96n7lq%C2XaVi`l$kDkd<9H_8k{z8w^rxb2molC%o{BRZIfk>WIHP)d)$IMGNZuIikUV%Lx6G(uMN&a(sx3{C3gW%u(Q+iwq&2J2Q4_0_L6C_Br$QH`o5i&9%0FB|3fU=F4)%6;Chg;FlgE{v(){dF*x&De&@5!+n63TYgI#>_LgU!h{OxktZ>2<9!I{0QlV({@GZTr3gvrn;znQ}&jBy^+kJp@<)RNRh0%B3dibRsQCh2R>s8qvcLvAI&9Xh8}T@t|*jWvaN9YVDwIg5B`4-s{>p;lvW1VV4N)-qi^%dwCfF4u^oE|C4EpgHI<)2Lk)tH7It%ylS2!h|SDTR~+2sHCb<l>~1rkOx_(W~(i&*-DGlBNK3-js)B#u?+QLM5y=FDT$ORG(Ic%z$G%R2WDa_Ir5ctxCt$!#1}%4Id9BJlYm-0KJ&>jOO~^?mYx+4SgW~W6aYA)7f51QESw}roZ>Pg#~}u=YD6t0NV4c-bvjkR8zOg&3RzF|uw$eIzW50Zvw5rE%edat>pR_#y6poZ&&_wPkKCl*UM?3t-I3^7=X*Ure{ZM(1rMEk{cpE+ey|mGE<8Wx?DLy9_xnScuGG7wj{h{(AnwZ-I(N!5i{~sW@9<HFC;xcN)02}E8&60M6=6M=WU@kFa*27+E>Wuwzz+x+qA>|26JPCtthsZ7oXin|MEZidb_{tEc!YgjJU=U5=NZuY=+8y*x|tv2&$$6$oYOn=_MK-G>O8^~@Xhe&sgS4x@aBfmX$`eUP>+&eRQu-y2rg-*V?YEQs#_AAB-4-_$la!ZAdBNT1K#iVvF{9L8*K&6!(nEww~HHhtT_2$0ed1t1yFQLy-2t}&|^uh?Ae*ZNbvy#zA&=$(Bu)A9JhS!Do;)4;nD2iy?%4{qhXWe`e0uD_^MA%Ay+GpiGehpX(ejXDzerU@LkLX0u+$}(Swf);Wc=g6eK;FY*29?0xP*n$hF}^OG`kdS6~7OXwo9pXF;k{7>_{b@OYd6zBa0&=$MrlI1k`-iNuuxqDhcXOUM<B1OOY!Sw#|jPELc>@R))o-&_FY><w2EsofT2Zb6ozl>@K~$qE?EpPY~@z$hSIGLl8l0V)Gm49MS>h!|@b9!vF_tlOzkR8`O{06}0%Au=dJei~hZLos06{`A&R4Dso3gtT6rB(Q7Yo@V3UUtxTAL74vI<mD^d2UqsW<m)dr(L0y@i<^J5ySJSuw3U+fpI?7hASoqBNA3$DMPFDjCIk}j05<_~1;346scKbBnmPq+q|H5PK|utl%dzS-hMunLlYTSigQ3Z+JsKPxUJx!^f9lINH$@pGASofIfzFx8YsCZ7DIl67-6I;Ql!XIVb71`BB@Y(qxSy9Xdb;(QIqK-O%UL>*T9O$jQ8)x%Es5kx&XX`!BpHzxgeR%ez;GiB+$qVYQHXMI^)l|Cx-!pS+S+e2@?x^*?2yZ5r_dkD<CLBpPMo7dw14TfQl^H1pN|i}+8KCr-2QjJnYS%pxz!r2{cJe3nibYqWdQ<AcY9W!<8U(bv*};|@Nc(&e^T3$E4o<@-@v-uDF;h9`7(%;w2Q$pY9on)1K(NTU=DyIn5fa@T=9>0IwR(BlKfE~`5IG6L|Q@m&?V($!QzNf0lz7YU!`I&Z$r>nHZgG@w5HvBhveo0iT!a^g#=;({5woaR!l4WW+ur4t(dQo>Zl5GBDp459WcwY%sCWkEizUxRO8AjZ3CFYWVy3UGFFmP#$y6H+)(j8P%lm>HUg!SN@*a^QNwVnN|Cf+@mj}R4wDZOfn6fkk#u?tN|+j`ywtL&08_ljO}dbz$haF@EF;yU9B6aSEa>$Me^z1Mp4FJb_=+=p#pPMGV!M+aDt@Tsp)#w^umJ0DCDH_eRhVM|8n<AK-b8>UZ4wrD(45?Gh1JH}gdF8`34*p2ojojoRlFqf87&+sj3<nTEGpZm6)C7=SYx)%Co6*?n^GEjos6lL#~KcklVnX=WRid#nf8mx<d<3Ugilh-5*2MjgVcdDfzoo}mB45Kzhq>ZLm2@rT$EY}A{e-W-~vSoRu!`bZ&sYd^szc4VriKZh7dy_nWnH-4q3H8wi>HMR?UoX256CT7Z5qJG7uxj$RXZpuDMr0z94W$W3pnN0m3itmyV;w_(-_z51w!8ZZ;#@qve&W{?VREFHS~=&$kzg+q#N^uK2B0bz0*ts$M2t@<lg6zp&c#7v9G$PB4{yz=UxDAU6@wRmxXg6bG=WxgtUds+4!Mrvh#pXg?KGTN<X!A$g*aVGSab0zZ|}MFjvD6&{4L1eF~ElNE>q0#5{JNh-^7COmCfmCyP%EE0eOFL5~}#2M}iQ%V=7tMkrzT?8Vhmx%QUp#_XyAQ~`^XTw~{x|}AoRG7ph<O;Z^Ad)Ckl$Jg=ZTL*Nis1@?R6cN#N=E8ojB(6!aIqXtMu}!ItUT<ChaRjQ2p1}IXbzXazY}x9S~3r*1QZ#Vdv!pOzGl)>^}?}?C2D*^wB=Y;M>m$-`sGu&B~LpF5lBK8pQ<qqF^XKpaifXT*<`^ArJWjIeMDs72m}&gCLsVV0Ad7GFgslA>tXu}T|1xr-&eM;PN?LgITL7ZsK4+n+N9q**g8)?##&5mzqC6q)On$S?k-)THzs)Zh=@EDt^2RQeg~6--N`qBa`!GZCl8Zz>7e=eN%#JfZj*H>=Orc4eEIUmh27UTV&1%RnL@?+y0u+4FN7bJ17yNYffZ}cTjsgl0E&6;=v{Ckv$*az&hOTWbkk_`eesc#Qe{b5BI$s+n90M&XkzsEnj;Z$EJR0jML?SPR|3$m2nGafm+dp%YSx~cwA(2+*WB+@z}x`80v>-b9GKFB(RofKILwpr1wkrn$yUahQnVn!sv?oGeU3&P$IZPfOZ<4QY`cA)@2tIhqc~&$CUkUEM*!eR<ezXe`LTiCNP|ZK0>O~*lp{(M3Sc(r)>~Qmsmf@`OaRq9InA+9ShS&FdAOV`b#bYpN!%$Y&?%Ej#N9cgOXP)qRGvlwQQNWHZU78u1#@PvtgyA=atZ$*$G3z9dS#zN+~!VsV}EaV_tIw5aDN6t^Gq#)_Tdobz$iCy5n<;{xJ0fx_~LR%TZmw1w)VFJ2>9N{g7nR*bB1Xj*CLyLl5!zZdkd^qU`t=EqT?tL!G4qWpjQ@i7LiVA3?(-b_!#I}R^(_=!B}lja|{%0bA^JV(JM%1^(3v1fd$j1tzs6Xu^JzxL?oaTNV)@%8_}N=OS7K#>YHA#7@Tv<wI=Y@LF+^#826K~eQ<I2%7wO`ajZkp&g}M;+<Ue$`ToXtHzqGPV+$5TPjG+o@<pmyDU-0@yumh!2kjrUZ+(u2c5(r0<8otk)BgRNJDL8|9$&tIk7<5!cRNr1_~x&hAO7*pe{NiMFdwJ_1_QsdU(4Q_Ygm>Yq5P>c575G5D_0qyPFMvBB9fxbfC8W(m5wqe5KmolwInN}z&~Op^D{(tmY!)u@XK}w{ilO^b=QFL%f1cbK`Tbqbt$bGGazLuGV4er<|zZIF$o8{OwtBgizpljgh^DIh%p2oEZigMI?yTUj!ZizZv@cE1I@?*ie*Ls%VVbdLdyT_M3w)Qex&k0Td(qeMeMP_h{<~@+0LYGU%@C#5S&hZL8{aL&)&QD*nM4Bf?wsf0mBxWRdwDE3Z8(HEO*jT+Z{RMKav4)o@80Ys&YLfv2nX0dyuFNPmh5KXgQ8%6pv|=mMNOh0D52m3&?yC!=Gf<{=M$Is>mWQi?S7mEZ(}W-|w8W_gd%dy;cUR)S)?c$ihfV%h$OqZSrdun*6YBS$WDrlYW)ZBzN{vg(m-TR`+l)!ky9$zdYT&_i2Bsarb{~UiQQOQoCojjas)yy(r?OC2Hm+tu*DT52rSsPp3NJJ{*o#?hid_zwgy`-fwT(|2^!JwlVFu!=~x)hCah-E%df`Y4(#kQN~Nd8BP}!&v27Qedm5BYk5ll;7m7GC-Xmao<3-TaNAw`{-xp8ecY>l-=C_Odw6NE@BF0m4VZfIWX7GKA894Bpbc7(id)H`z&tqX5E*?!lvOd?Y_$`fDFR~v{xN!_PGn2KbzLvIe&qYTrNYppJ7uWQb(Bq%OeaHneo;yGghVA@Ug=l=l3382T(F{l$*k6!biRMtl?9bav7qV5264v0x8BVQUtW1-{`C`0%vWaxDCrw5N<q4nS3Ufy`;IeGl*4x%<T78L6~M$Jc3zxvBE>m~Q!c*Il~)(Pt({lkydqcM<SQ>PNO3L^)!dlBxXBF=i#Ko>0KYIM6M4@?q0Zr6Oy<isPfYJmOK4Luk*P_`1i-53c&UcYbdgo90U_bpC0?VIQG!=D$EYM|B?Ze>FzX+qN`gJDT@sR`j7b)zoKnFE!9rGmkBO57yAr%gPu29`xFsp>f&|p$79=WY$7_DBiA0;Af7^nX_0j}IH{3=L0s-P0WHE|%6FDzr&Yc`!Vj|Y6s&FKKjC3ZnGKtOv9l-xts%mEo#n-)+sF+bss3M7Z6watG%LGg`fxmbvRFdEgXgTeu1XFTmOginUl}9dUC@z!cx;q%KZ?q=tLu>Yw8h79EzV6c&#RBjM;z0?P=b8o2J~-{Lo@(YwXN*2sTLfx#0b}&0ni#zGhB|{|UOW8nq%NK={AX^p&b)k7_}`+RB|ZEt2p`;~zL*Eq(}D=Mn&(ag5vCO*W>D|2tNB@nb9*tGdjkq@_Z+6=R2nq>Sf4h$=<c9|FxVb-Xn)YByfZvy=F>;hatxPxSWm@<KkPIIJ{$yg+Hd;UH}veEbEi|6x!-)#cLx>uiNehAcDE<@?d}Jyc*c(!4mS~@nZMOku5tods3BCQk=A%Nv2quKj$jF@P@xlY9k1lB%WD-;SVSc{P2({{ge8a7<exTKcobsrD6{Iqkby;5JHX-raeH$HoY}!kJV~zBe76e%?Gq^Z(gHuQi|@T!J${~X4Af!zIWI4~^%50)yHeNQ+trf!i&%UfR4)^+_b;vbM)a}I2ujyTw$_O@M3K^{nJa^&bI+5i)B~=K&OB1_Q8hr<(I!&FDBMa{@)>Z@AVYa;?Ffi7FHGb%v51s!gJmU9pQ~0{YZj8CV|psV0082`8cjQeK>-7C?RorCirA}V!U^>2-Q(uuk#1|)-nrWLS5u41@{><LCcOkT?j7z=QMaG&XIfbteB$Bcc#tBw+O@7OZ;E0p0@DUxJ@70O0=K8~5FB!dGmQKR2p861Rfr^`5;!;HMvkPzl^HWp8u$0=GOx-@aD?P59T3#4Mf8{%MFbHw;B?KtImdQR5eEfx$a-AJ%}$D}92N9&&^Ekd)-ETij&&MumrYWIMdLmGV$M-1122TZtu13!tJ4t^kQEt7T3$<<lm>Y)k<h8+K<P3lj5q*352@hW_^PrKR5(9h@XP&WjZwPHIcXbMj2V7hEpA}O#*O_d#hMs(ChZt)!4U+xx+}8q%Q2N2H0CI^iU;nEaCGjXoU<`n%tQxh4YWPx$|YuHu`=Ka6zM=X+#H(AxNitSfZ<r23B1Jd#4*#UBVAkiEYhqTd(adbIDmF<aMP*!!%_t}152h)BR4-!#lwyY=6D2sOQGUDSPD+2v96_%`7Os_xnLbvsRkhUTs@`%0Dblq>&OX9R#k``fs{dm#N?^P3cLI)xu3e-d9oP2Qc`Ke47m4pb97~Qw7r|le%U1r@WI~T+V-2JJuj}uJ3AvV>2P#&)aV!lyv^*pw896?650UszCrKsHN5)Zc$eD2^XLZF?DZ`|`_re!STU_K!bCQKmNm5kHE0JPcXJ&^3T&B?f~djJ$m@h#T@(0I(S>UKtfdZMM^fd5mM(Q4J2G|Eh1wepi`4UpDQ%e5h^hFe2%b|ROu?AZ5{}^SUXnz34H<dNFv@fUF_A~H*bzg^`K1s%H`H92vjnTOB9r_LNeKbrYR*X02_I8fl^J|nlbZ^VFT=!-G8*RP^Wg=-o|zYf;u7%#--62hNjrjEJf$HWCrlSQaJR97RwKAtQthg~QaO#p-40mu{L7=iJqU`n(ghC(!N^Y9Z_PVs+!~&2F|=B%H;9`E*c?bJ+l0A}rD!8Cl~x6dwnka2kzN7%#H{q0N+8tKxJ-Wiw1<T~a)D&8bJ#HFU(Jf}vvrI_@Q#ezrBD&bJXArB*MJNNiR5E1U6NIkjj2Q?^g<vGHnu>9@o4Pn82*!H^0oUMpRXOpQzf`=B=CUZgbpSoT?qw8j+AdA0x}P@eJ4OrwAW5r@+4xk9|v4;22=M3osQ?I-_gYXKDsoTkwouzhVJ*L%Dp6MV@WV$0smlnSLU<+mHh_|9e&jPH#F(on%XdUIKPqCPTP<AzMXdm0o6`V@@!voEoSTt^e0);0@Km}csUXV%1EmOPAz3Xq(wk-JkLa54H*t+%bHS|3>0t?%-E&s&Da6HnVPt<dn6X?#R@m}nSt!|bVq&Z^k-8-!e<=cTZ6cEH`0%3psqlINaibPYHkCiSHQJYLM?a{yu?_AP8brBG4i0Bm5ah8iAFj}CdjARCQQ1>N3S0Nh5%L^Hf4_ArJJ*RZS<GB`1|O%W0FtWG2x}xzx$UHx!@0*UCK7X|7!Za4pk<%1_|CNQ=u7!m`A*zm9jm2N;8J9&GSH!*p}0RgAf9`QtObWRpvn>N$ZbOx1y>@2^rWM^of++oGhS@z~JUj%e#-dmv^b&dwS}f(EY90nYKF~S=Y7{qXl&ad6$i^1;{og9#vLG7b7hQs^!KeT(7n|Iq!_EF-b`+jH(GqGh#06vRF}Ly-K#wmzqJs1a<JTip7uNm6c3-6bHqz)+1;WUk0^C9LH#L5tvHJ<MSI;oF%lFZtccob9->~OM79buaw4G*rv8467xI2mwIt@?I7%KFG%@~+QDfSB7Q^EcIu_iiot^Qk>95YpsxyS-8rxlTLS*mAtrKG_flgKSPmj+1?Y+if~%y+DAd|N)4$WAnMs_CE)yw)f;_t<Xz$4d+&yCFVE7`s$db`n7hk}~waL}$NYc>Hx6RjY8$|b$L3V!43{OOaRIjQ6Hqe>7tTnI#I0j;H0k)17AtOh`;Yb_ph+rZJ@Ig-11W{a|W+Wv6J5!C$$82d|#@A#r@F`MZ01A~hr(hf)D@c?9nJT8Zs4>g3C+F0QR52vMfB@l4mesHtNO}P`J8Er_-dT15OwZEgTI6*)PGy$5W@E74;P)5mQY>S^6;oqFR23O~0l7Hj7oMf`4iuJpVM&iKks38MR&azVQetENB3%mUaBvw&20%J2f|OJdQe<Bp;T8!J8;ZsBGg<S3Ic0!fe9@T_xR)2|QY^x0(U!|brMVhU{d&$>N!^ynTbz=gF-jVtK+9tYfP98j7Ls=CwCx19_6-D;g!Ed*3hoxE50LP00nbT*#KN#rYEgz%;FPbS1_K~jsO=nCr^r>k<Z`k<a=Z6K`X7VGT1)$XF<nY>t2fX4un`4+)E7OA+ol7{TxcIb)T5%^ctdEr0`e>F0S%3$kQrmj4$zT#+6V{I0W!^}7m#(n8uuGdlC?_=D9sJaQZS3?VpYTir6OpfB9t6?V8R$GA|nZW!xf1DMe5YY6NJ%0eJOd8Smklviw^g0=lkVgdoQ1)lcmnYl+JXjYmy};ADfLb8So>K61|}&(2PMrV$TSpNimWJ)RL#1Mbg;=mTRW7jbko^@d~)b29|KQNUaA$Tq6e7SxjxbKlh+Kif*X^391sHKqDnlcmKTW2UnNrDD(1)b5t{4DVeEAEkG9l+Q2x_l&J)jLJ}M+iDRgc4kCaGplqUYJ%Ey&ks1?2fJ$7ARjF8LPI}^E4k1l4QcG0XX=^xc1S9nUQW}aJ`Cl4eltYrmO-of5y~xvE+Cz3p#|H;f2a|`*0pd~9B)i*Kd>k~bu@!6Rs5B2H{CSL_>(tqs=U(6Lf7`#{u)K+XH;+9lhQ<rfY9VcVWC0ARwxoy8xhHE-EXop7GK_&zaFT=-<ntkGb)jlqwcsZ(H1TJKMsd`UkeQPpW55HPWt*JSNVUkHyi(XS9Mg{jS^L7B1mFfhGE^VogJkQM>*Y`09a4+4yF-dInnNo6#LOXATRFAgcw+4OSW?8K!%9^s!fMhp6C_nvlw`>lYcp9^NGd9;iX>f2ktE_thx-c9R|y-Jm@3)jp_Q+Uti>vVYn3vhA&!c*o=C9Gdn*uENf0C{t}JbYmJ*~P3M-weSnRo(6Hiu*_>7(O=QS#N%3{Q1Ur&atH^TnaqrI!cZrs)5-EE9d_oj!B5Q(0oGBJQz@Y$2`kd(}T%p-ed6P!{$Qu&1g!HS^65_2t@jGHZlPuglj9`I+7iE0<_gSCTGqXIP%fQ(3L9H7=9XCxiZc&d59?=z>m7ZNkcfg1(Jj}?eqm#g3l4ij^$|L?js{?@#5@G$sr&Zq8H>t5IQr{4L3EEc9`^OSyavNjf^S88@(j`Zt53p7iJ*IM(Osr0g7Jx;H)@LzBb%F*8O;q~n|-U8elHp9~4#t#lp+3srY`V%dN#Wz&iyATjLfSx0%fhy;4O%=$Kv;?&Su&mV0?;~I+tIU!RPbJ9<clvC))imRN(%j7V+A{iom~g;ei0$*K+Val^o5Wj-Hj=H~<`1XOwAMdSNr8y-jIoc}a*J3meZU>F<b<N>)ZUQ7HzKJ`w(@MH72wu2WY8b>r6~h>(D!Qpa460BWMUcG{JU-E{@Gv|duzIxyzMJ)dH=mN7myf@mB}#OL4<RNh$W!*q6!PfP>iig$Oz7uOy)8>ZxXU*$xKmEc;WmkQM5%F&6#WKez{<RFx}tYH|W!&|6{rffAaJuXc0KOxAu(@68Kf5D}fsY-XIlkHp)l~&LpsF4zQX?E6Hf!3q?});GqDrsuB5ou8$;cO<C~X?(ym%N4@qV3cxxLWZ{@%6kJ?UC6HN~#pobgf^v{_AB_NWn90Ff<RZlEgi%t`{K=H~@9h_A-U~POcgloe-e`)8k$m^B_Eo}p?xx4RJu;%WN?SCoo8l7ZL@}0fEX`Q6Gw%7EyUrghQ<GsK_@AMzIl40v;VK4~Y2yY9(`zBJ%)xQm12F}Fk5x;K30R5aWE=-n^pb_7J;x7OJNznH#mT5J)P%RLl1K1}>_~PS>$w*1sKKC$F$rGz5Is*OSRN&&2lxjNm?ib;2oPatno!ZIH5tIHO5>`++%zI)YsS(7#77AxjG07rMtCb=x99=sRHUlbRHDu*f(gj13Z`7ipxM@jfMCPtNRu>|c_ZpAFbtPzJmjhx04~OOs^j>RuV7OEK`c8ZNpc(60m(c^Wyp>J#7VG(847&F$XgJ3Jqi;4#-!)C+bJ5&#x_!^7j2kwOXp94Y!AsVDw9zlZeX;)08$Gu_gE@E4(l^AgR5198^+ulscKTx2G2YL<B6KIxUH+PYUUC~+)<rBIHhlDgt4l%v>-`f1rl!{sB*gqj5}UpDv5MThV@$+lUXE#doF^sSk1xnDwQOAQ%{BGfUjmrMZ=YU`}E2m-}$(yOW1jf3jJ9l+jeD>9{vWWUu+*8mYq5ZRPWzO2aYJd9D!*yr3=kM+o7KuEZ%CqVRe$|+gsPS)Ahc%p?mdTkO*9>OxQL6>>X@hqv+T$QW||K>aW`eH`@1lsa>!e!-wX*8)18Q9lLMK=&j%pvAIxzCJ2mY<Yk44!YT&Fr2r;VNcbMpx++B#{?3;gqaj&bpImtYD%&go(im_)B7_0hCXwV<iU=MOIXsc(t^t~lHe()Q65`DZGx(mhB7Kh~-pCk9stg7TG!%h(R!Xh`57dyTG=|fjLuN52d@-jhBZ2<F$ZV8d+!Bw!|H|;#R-Xp9X4!fZeA_oTY^1)fD&9WG7&n{yyVw5yE0;%q{}qO{oSXANiMX7l4w*`EMU~Y%#6zh$mOp}EBKr@@m4z%uF@740I<mk2%71+XccZ7cn`sTB3@#(jnhVSg%myHdWF@0cF4~L7%mtSr6FG8QvzTfKJ(87#<o^t0OOJP3uVc{Y=H}3L4X&))?&)hpDa$WO-3)o+aqW&jZ);db<I&fF|6Ujkk3332++#Tydt^ijNJMJibO2<Ll)X@ZfI^eFdrgsAbs_>ubuvnqd@)pRh1p2u6XSxYK;F5g1cua1+RcGGS+t29!H0qoofaUfP=6a%c1~7Hr2!H=0R8-d5OOW`*Rjh&u9E2+d$+3eE@e@?wwPm$idIbE%gBrlphQ`l2p&X-5aR^qgEM%RqSis}5_Lh<F;Xg5JW1{u@@cZ%m}Eo>#paY7yI`QnME!>xFUW|0Sacfr!q8DsW1#ii*}(I&*}$JSc8?uNJj#Sci`%8Np*EImkda7o2#`h^CL%A?Vu1|h0e@7Zu0lmbikI+2g--d}g-#W@pD0ptBY#ClR^7NPm9dUMWfAeTG@gD*)_8*F$}Ey;BEv?Xi}I&5U(l|}=e;^b??=#VIJUYKovvL=VLxga?R>YzhrKe!gJC;miF(J6yWLHC?mMFsKKQ52r~GK;$Io<8u&2~TA)G~u@rA1NIR`JX8i_z44N-xYYi!S%BR`@_`N%S%q_zZrKtOp2E4nBq-Jh5`7;P+fOo4;#H_2mo4V@KQ_l*@)HZtj}a0qI~*cE*hGXp#t2Jy^8-N{I{`%jPCLt(Q{NcoNJQUxlIl`M*!L|l%6Yvoi9)`?VT(??f}Xe;QX-YZ?e-u|SD>bnw=o`IZHBx29BVX#6lU<EiJkvFi!41AA^l;l`RLUBL~SpcCL5Y2)}5AKMY?Ew++&j79u?8R8lX1>_NUUn;zI$PByUxJ|id`)Bo*%}NO+@6R(`HccaBI66DBLmyjp!(b>N7NNWiR#Dek|K@mgNW=4YWSUlK!tu`^z!w+xAOJlgPUJN?gW1%DXF#=;}tymeN1Y4?qSHI*N)$~zPBIp^_u}tepYoD)Ls-LptQHR*EV6DSYCnL5+)^M;AOPAVyytKQ6$U^8kmUx*C?st_0N^^r={wut#IUp9BJE2YXh{Zfq^r5M<IdKh?tbK0upk?D`i8-r8!zAr1^;3(&=FD@DOW!d+$b<=x&s-i=jF9mNa{;zUZzAdRoT!z5d9hJI=h_*_Y9A%R4<@_;_~HdL73tPBz|mq&w{_<28kvg;N+3M!Fc@kvmnCJja9xLj0vfqq8^BQ=QA7RN>*2AlDDD`nEw5^Pp98nn;Zt3w?q^#~as=cXmT>K-9$hts@K;Ms5R_eYCY8_rzC=v6=-{*NNEHMsf-_X2T%BWzTDnoKdBWrc!Fw#QP_|MULT38sk#QLpf?{KEjHpL)w~3_Qin_R)HktiKhgTP{5?1e(@}<bK;3RPwLCuo#ZuX$9`3Dk3doY<dvZHV8;SUJQJ|;TnK#ET1XeiILB1bOhtTq<Z{|oG0f%&hCAjFu$l3MDVeH`SOy4_PIF?%q!jNsRvH&EgNj#x=$r$1GJTZ8a%pN=m$bL-BYE~$GMdcPQQmI)M31)*uY)5um5@6(kzFq_#zu!f7`<_0bhLMbd|nUsZj2`FrpV;x663ag&WjzT?v!PwH;msp*xS7}I!t@}<yob3@>wbsGs+i%ndWj7q^m<cCkeSlVA)4*Bw3rrLS<hGV<-$yh(;xb^d!s+|BMGtOfxEQo&^=%6_VkwwRN8L#(Sg#Mlu}QCXy4D#a9d<?Kmn!Wo6>j`t~R9gdyz`zXG$=+xX7%5aPwfF~gAd0-0feUWf{emn<h6OeAMVCAH27rzs#Ji<56{<W7O)n&zz27D+;0AWpn|X{vNSZy;cc6(0;guN(gvW*BX!6)rI-l8k|r^(?r|bigVJ)TmUgA_^--(LxucZNZXb-0=lHhOVL$w`UuefF8bjEzl(#9UPZS^KuP;{O|vI<=d}}zA?fNFMa#vDMbdh$q@?>WRYxz)B<6AUW9>dcE1^(Ij5b_z5D+(dG{%K1hzT#LV6=PrIx^^)DljvUF)tFXz<$Z6B!RG@gpTZ5`yMhAhR)*WHuI@iCNkhCV8?_q)!Y%G}&k?#s$!gW09N^C>b48`Yda}lTAnlORt)2jSwk1pcv#&4tfTNj^T}r&McC^G8BN*-V*dkJ7N0<&HDxMntP3J3~5Kx)An}j4%PAEB11f+){S;hn%^B9?8Uu<&}7TGXciB`Hh|>j_U=|6g|;-{k`c^}r$gS+^}U1rg7G*`&8Ars^S;|Y_*KDxWZuAEFYh!Sb~{^ue}`Kk=N&xQzcGHqSiZ6K8ujhdFRsMxo$aHW_{ldOpRIK%eZ80W8ezL#60cLCIXzeL^2q%0AO8=bntxjQ(GdjNKiI}JzB%VHiko&l(3d2D|JMEn5e9s2XExdx@98b3wxma}MrPRf&LmiG^!cOvRzsYLd&j#3ueJdk-fT8VJ5jy?3cgihn`1`!0JFjDM&lIN4IkwQN_V~kW>X2`(x{O$*cpZPrd%2g@2S<|8OTW|1NaVo;O&!Idp7fAR!m5UU7~W4k_Jeo(qrdqAnRHO{AV&3S<*hWYii<z1$ih==xlN+=Zs)YiQv7y1nnn-2-75vDaWMb)Qb+1qMgx;aZFjmJ)8V#<^ox3W7hiMyf?LL@yo90t@*^+*rtsZ9Do+d6y(lbkl0qJDO`zQGzP?zS2Dhw<cX^qSn&x&y#QS!{z$;J80cxpM8r^;eFn$b;b`whqso17oX8&YgzaM?$5K7+>2<4z{^P#+<zahx@J~}`3N-v5v<0S*-V?)ROoD5wObVt{PaOwNutX{rLAi*`Ld}WX8PXnzLZ&Ptc#{EC^SYR^)S4#|ye~zUnk>>1k5b|u*Gz|y1$Q?TtiDpKR9ceQB;dZ&S{Cr{g(GTpnX|E2Nxy{}JF$X(V{L<aXQK!Y3cwegE0&FoUI3Ehs$EL7Wq^bhlcFt!$f)Zpf|DC?@^N7`fLSzRpW1P1#7W74EFv>f`Hv-57%1zi8mYe)lM&pWQTcraqgZl6J=#4^JLUMWku*j6I;0}0BQWJAG3=J(BXH!dIkD`N)cmvgU4u>OUu*W!Ty{Rq00t{yI4-yfNxh!Q)8e3Gc_A@lbRykco++wS#<LPYAel*{u~1ze$B?-PYOqNv5bRI_39d0P@tO<nr;C_4j(R_Z#Jor(4rL{crFEpH`J5YZVj&@}03EATBKKt9C$qIVV}{~JL;yOWlnav4-Y0I01n|VYC_vMsWqx|;dfO25al=a54DjI}?)+-)HRmqo5}m>2X;qEqR9ujRmXTCFNWM5A7ALKEwA@Lj=};-sj&Cm9F?RZO-tO(=TGNqRYi}e06XG@L5`)izlXq&xI9lx_Mir5O+Dtqmg^ip<8h904k`Gjo$y#Ghm3eFoHkuRtPK)smcR%{W-QUcY6TCNMgO+$scWTVpm;Ac*YMIt*_<d`lF*szjCl6B}H}p4rJkf=me6zI_7dXV{T5!*(gP~T<icZWbQUy%0R?AbyG%Akk6dZ7av$*8~`8qO;L1V*l5sM3?aeZ|G=Zd>Yd4cr`6q>ClFBoKXTbGqiFLSDnVB-7u+0euC=(A*^=6KqCdE62OBwZ{Zu1xfSJcbYlkbsdIB4VIN(&d&^6GDo@CxHi~1g8}u=M?$bbW)rD-J8te`Nh-v)hvDL*{*#p3jjm}V2t*xwF5ZTK7g9DWUNIZWezDgYw#Hlw3Dci)!9Uy>oH5x;IHDME&x3&04M!ab&Y2np>p-~|L4}9qxY2h#iSyt?@0W`bc=XzD0FvfWpqt$^z5%SP3hBRRd~;T{n{xD>Q*}q9%-^X_CiU31>h`{UpR)Dc?+~;qLa9V)BvDWF_xvojUh__L#GxY;&CdqGH3FB8!qwQBv{a``>1mYU4MEl0aLe@D0Gm?StMG3CgnjK$m|5LoqGLPup$=HS1}A3*RvRqeq9P@5sB`uvV}{-r275Zv682j04tmo+yniR3AELQYX;I}YY{0|D}41_3;ZMlZ+fV6<e6r|m$Oz&pIoa&ZT?e7?exNBl0HqJml`X7{ywi&QWC%ziYjkQC50Yof~|_NVhS^?;3k`*!8{FDRcP?Z0@_O=eo|*aM^R(PYEfjA0yxY=5#;p%($ttZU^UAO5)he)SqtZ!R3z7Fbs)|V9N5CxD{yS&&scc^Ffv&EMrKks%UB~GMAs`0nw^5i9WRMa_Kl-Wd<p34;y8&yyI)GAiucd4QPIj-7UUJdA~l#(6ACBiz=-b}@g-#j+Oh#SOO`R_jxmXo>r4gqs!C)AzCsu7A}+-U#4c?>`#runfLl^nwfNeJdm#j0EKc_rwaX6UcA0w;E;97T3W*IR#)glLA6qeYtK?!mA}2<Id|$xqH5XJ)UZwW9T<(o#<OH2btiX9AE61c&2&_eM6_8fNn3^jzBC8~m0NE_V7?YPEy9u{Pl5~^IplT}0igy5U%~)lE8kBPqYK(M^q+F=--C<tjMkT=&DFXgayIhO}Sal|lYP@VXMF<G#PJtH7EXSp3+SSzOmk<LZs3X%-q>wbye8fERoTti|z+xjkTtUV+9)s&in4^v}Rx~LpdK$H$8S`K-LT8IhtVyX{3ms%<Nm76K%0hCFX`>x&gW+o|uPKo7Ak$WKqrlYq!w55ajZOtGHQN_-{;#_I>Xj7;{XWIc&93_lOK|c=B)vN&a{ZTsGfgU^SJvYO!=Fa6{<(Kkat2eVHe&2qj=%&U&t&fslOzKoiIpaRuLGtbmSNx)>mVncBPm89@Q(j9#HOBMdlt3FC=7ok`JRN(s|xW`6-mR*M65OugN-><$0)R@8o+5a;%LVGuBY`QOmF?Z-~9Z3FU>GVcKXwy%kvfHgYZM=*Et9mv_`G>{$ez4F;iF3LvQf&7e*(4HeS;y?Obj&p?Yxr;;8?e`*V+VAoTu=trI~Dp?}|bi{APD%)>oaYn>vN{&L|pzn>B%`Lr#}pO&=x;__)9Flb)%^kH~0{-hs1=_$g!7YAlA`$W3Y@AOZ(@~!_c`fva3q<6kQ`<_UTFRr-lcRpVJoBJz&Gc!$2^4RFgJMyHyW8h=dxpy*il~Hq`7;iWtQF2t)7or!yhd?7pQb>wYBipJ1L_l3^E^}86(1<JK`EZ)M#{3s*|2cE~c6J^+9h-)$ZdIu!*WO;?GDbTSQZ6fYdEZ|C0nqT}J-&j<KH4&TO!7#@xE!%26U_jfpqwPFi<Sohmzu;uUCM!^?^I3Hk52doe?`zJyY{Lor5CHJnl8n~6S1H6A>@NW)#Ib~RP5vatZd-<_xf6zTPsSj+{G!yQu-p5Vnt&C(@Ua)EPz~6FyMujvfvR2nR6Q?FW!)29B2?L5(t<23CXA3*3*gj{i)rD?o{(b8o#?AOz%(L8Dw;BHRCv4nSXnRwXbP|B9iMR=&K^0izUPjE|)ekBPX+Hk`7>6XzHgy#H)A;o|_T?n>4nDi=?0(Nrjz-L~;U3N$|vA8Q`mp2@6|TWiv1)pko9Obv6+-LX74vFlCB8(-8k~))4&tCKKZEch)~}HIsEG5>Q|Ov5f|!OQb&wx?NcU%?26cF(MDunh?0a$T>MWkdXUnelrq*O@i?kOMKbZ<h7#SZ!RM~NRUV#*uJ)jgHDq8Tp~apJTpboF8B};!;R7wkN)Nlckj3QnGXh|toG3Br>)-Fu<!UqANd~i-tV1r*Svw#vclC(Hs^!h8J6RBbC0*4<#UftOw0SV`J=v_<<6X0{e!krVKm#=a&Ksd#q<ArzqEheFZ;W{ZT9JZ)0&dq>MoYvsJ++^hL_*&)53$vo^P+m)z`%I&z-518Lj%}q!)0%o3^bt)15C~n0Pq-dI(KRjLljc`0Hs!u$4Dv_IC7=c@JCH`_{OA)D+O&ZEnZ?lWxSNQTIR>M(+&`$IHLk{HNS&&j0S7a<A_X9%7^R`72+txw$zvzb4(omRwVlD1aqJ3KE?5!Inx|Za&#acEynj5rJQ*s4->$#*_dZi&3oK-0SxOyJ^d2A8hCB?1PgjJNpj%$8o2r=m|&13*|YJuf4R=dtq=k+dD9gE?y5uN24pPJNoD-99%2phI<;u;E#YYrtH@nY;hy~U~9h3XE2_3V%FS#+LPW44)Fc279U|7vwcU?uh$C>@{7jzlMW<x>U-mN!?VA+@M)9JU+}BHqBbbVnG<&fo|<cYGM<G<lKfT#*&O7QAR_gR@+dR1sVPyC|937~Gfa0@0Jl=p)g#QG)DHjjwtl<WU)xbwxw|i($Oiw}y423S+kWo-{&R2FTk1Q*aA6WeAq7=cY1+pW-ns(b4W6ijB$`+;tt8kS0|g_kVxZY<-T1s`9`o|q%`Shl;oofaD}g`g5a87P{#61s=iO>Q_h$dO_v&R7t`VX`C{W;AB<z*4PaLb8dyp;62}ean_=O{KPbI5BlU7n;V<6x8^Bs6B-|T;seSAuvjahs+8gVnRhTopq%m1G{YxifkjeqV<^mA|HpLZ92jA@4n6{*0zG~#6%lScthi6{%gh9Rj)%#x@uz&3NnEmuZr%$7*r!(U>}xC?R)kPv~Q26HA?am}(u5D#F15%)P~jawhJl1*6yCQSzMW;IiJ6hbfn?>qu{47J_1CjIiDr2e3@c<5_prg{{5Wt#i#K6%g8`*N214Tbrnq4EvSSowwzrgxbisqyWT!It>p^xHO%c5nW1OS8_V;Crp?mV@43;E4<<Q5EYgQ*^+CIG!z53<7dQcIM1P<H@v~Wv@v4k<z+OF<7iyj~M>E>F&A5ZCjY8g8wu)L$>t)=VU+MZQyoiA5XxWHghQfU_vxSdB(i*fToDUjs<~`hP3ICwUx+;P>Btmv_47OCOIH*!{&7W+^Wh0vq~Wk**gGfk(ZNHh{-lMD?CR8P?8~uBkLOct_IpFbrEqKfmcx~wvn40o=1VZGunsP>aYZaB5VW-Un<#z&~Yandoquq0c1wBltm|^6i7xS$X^-KG6v5)@AdyFX@@-Rl;pq?0RUM<00qvJwn*@r7jno%ng;UINB|BAJu|X6w7DvA+E%MzUtRltYQjV(hufyP4j|Pymnw-X&~B=w_I%J3Fa(J-2k325Mu5tqhUgu_)w!GNmXvAEG}jr&C8!Hx0T2wcezCW1u0`01h2FZ4`t<XlwQ0ZAYyP*}tY0kj)^RiQ--YO{!*n9`N6<gQu+@?BKOk+Qa3pLL6$y>InllDXX%3jDG)u@M!AZ;GMe@f5(*=-9A#0s5i>Qi1Ay9xtQHf*-psChLPX^W=an^(TFfj2zKFem_WqH|&MSDK}e@&mjc36h?NI#uvsaexH%ce4Ihya!#jO2(J0b~%QLR<V~ily^ha0g>;susjP*BD92oy(BTSx#vNORuSN>Spr)XxPPE^EEq=TF=NxC{wfy(o4h|L`KbMmm!j`2Zod!ogI(J;pG3$Jm$8j^NzdgZFR?8d$9S3yLZ2OQm6a3cgv`SrnCi2eqA@hKDGVz?(sY2=!i&jFV4Pm^Tlqbn2hPSi0gDYA5XS!lYe#dwC?$IeeWP$51YH?Xwp8va(DR3tNkM{FQ`fl&yfrAQhat#d1eQAVVii^@Aa?Fzfre2j7MJTpY!tkF7ZU&YH{EA%9Z(D=9gYwy@4#3?HkYTd=J~yOLx%C#`oX;TleaJ6yN!79h3+F53g-rFFQMX02R%C{PNa*wZ-_BlmKE)1pxK;zn->f?hd{E8?Se@9UGV{i}C|+wiB!E@V`OY{m13SQ9igi3P(5g4)?E@gVNXm>;w3{y^UyGc5b%QX0%I9rkIPX)d}WL6~4b0H~LzWtwYSToic0+{RRDSl{8eD;7>!1Krd4x=vdz9u)S>;NP(aL!U<n%E2QFU3AU1cmk$6-<h7%j^8xXV>H-BPYpTjv4u~>@fY*NL{NXwdorPL;=xo@do;go7-%wjz>sEccJ==WH*qD7h$n@NA?VNtme^si#`ophpH`n%FC%Sm|CvD>n{j@<nVw=+{%!66wr^DOby|>wFHa2I(cmMYj4m?xFJYY!8L{dcrVHe&xDU%aGUOAA@cOjwbg8$`;<i@knNh8aXK%Tr%2cE4x*abS^ys{#P3{f4wk_)l?)x|@|joCzjhhAPdDSdT#>y3pJDmLan*S|WO=;#fe&52|IQbu$wS|Bn3OX&bc5FKco&^lV@g{1QZuNqg3XpBe>V2Nesgmq^C&>szETV1zsS6=Y(v#8aoB>0e&v7Xb(q|cW70(_VQb*DE%NijxWnN!7hlXGAJM9G897TS+F<!WDhee}w=Utj-KC0{hDTz~*Yp;-h`Gt32%i?mrP4f-ZYqEIjrTK7J9vLPy!JcSZ_ZUuCI-tv@<;ion0q18gV&rEwTkxVQh70Cs-f~%CrK@q&w)fZ+NQ>-!*$=X>vu+j`VecEi|jj-UFr&7&@WKcAtmh0>kE1BC83?21)(k`t<7Gy0K0bpoW8B11V<Xp|?tAwBF)&Ra7W_&G5dW0<}NHkR^hqcaP4J?s~mM&2kDvHJ$FNGL%@_?z{7h9jHI@;*+sFiqFNf^JkeRLg{Is%E?PNh{>ZBANAh>MP8Z)_hO;+F5;9PNkggTq#ob&`7f*WCzjl##eB+c((TDXr1hXs_{0CU|~Su7P;co1DrhsigVZLAU|Va2Ruwki*W-OX;wuEnCVUdyOAW=1k&MC&-*FVL;!#PL5`WqxQ~jP)8a~uH%DFXg-XZd*7Z(w|DMfdw(C7+ZvY9YiSx5{6PnS;6|-&^a+Y>Hwn{jgLX80`pUsIe7!B=4O^H7j!tofd<y)BZp{)kw&BYkPD!zGBQFjUBCOG<=L;#T_AFUfc=QrWMutpH1Db?!60sMoAtQn%D8h{*sG|6Do3+}K&yS|n-tq4rb~eas&0vV~T!UHQ1GV=$IxRIPM;xnilvFnmmB=FUT0upnH#J%?5d=2U^?U|WyGkt~k75ZQdHuETerJV=Bxe_>!$V!54i5lFhLOnR{M$$>OR7W*Jc5?$U?pY&6}w^80^n2(9g<#gB<`wnWS%{5N7;NVSyMM?&v2%_SBG|@kA|w}&xf6bpKtcvDzw*u#r*D29o?Gwv|1V;Pn{tAa_SAfaj7k@Xn0Zc{NJ`aE%eIGp(U)hFK#w3@c3W-?YBl>f0gtGF*#o7zs6@%*6LGQko2o}o51svsm7>J+6s^6h3^j%k=<5E_a>k9Pk7Lu+jpK*?{yEhpGbG)mxI2}Vg<l2hEuZUv$|@IcLtk_pG}9{QGdkpN&Ebt^*86;w$7w0RmY{gH>FrL9zD^IS@s!!`1S3_`k2$-xp~|BLu0|v;q-BTgCC6gqMRSjnMie7O`lB~QQLVpek4c36?R4Aw-Dz5YL*;>FDtunP0gcV6(mQJ@tP+p3gFAhe^_d&ztlPy9ypI!`x*(s$m|5SE^y(;7TLifF5nSPYN#c0Sxg}J3H;3%=}snV3sAgM!TKsFfB+d}$D}i1td!7}Bikrr04)%wBo*IiEfu)}8*+|Ncq_0RCNh3d-Wb;i$vb>*8HdCTtF%u`wbERRsuj1^35Bmnosyso$ZVMmm~F;;CA_hBp|NSqcHW6ea1yd$mmL+ia*lwr%9+WSv6W-Ls-QJgA3)|!>%@Zm2|`m-N|igr<rK(J%{1BBCzXvy6sHCX+~z=LpJFVNVBo7(X;7n-gEE0jADE@1Lq!G3HQ*N1NVTm2gfA*`o0d9}fd;Q*W&veGh*deqB2s|T6@#2aP(oFZMvNnMW@y~Oke!#(koTyS)H4QTF%CMoh?kM^9!Ew2F~-1<>8j77(nnidrC?blQAxoHUsk57qLUhgPJKx{AJsC;;Q)e|i%Yok7#fyTHL8{>y!BoJ9M#-ZhXW+h^72+-1!1@Vp(GZN9+wco$$jFqk;k1iye4BvWQFQU6Ui!)6;I{dW1%FZZahEwDkIj5;Q9Ootgd2t7tXom_~I&hUV-dEk4X(Rq(bganTJaFAuI2a2goSa7=hWU8@<%$*>w^K%2PRi)>Sz2QH#=P0GePea4O>pL#!o_oB{WcJbo0_M-dq?wvOctJ+iG}Uz<p#q`DeBv2ML8%}v6Ts>M)21+ZICMI&%$EN=vyY?2fJh&FOeY-ciNmo`=EfJlMxvET<ahZz3uD~*rBz5Zlr=#y)jMgNyeUzrBrgTx%J*xFcZQw5)qAULx!BGM-*i`JYPZj359%~)=P|3cLL2r%;c_8V^jTn+>d*!%~Fr?9f-);>`r4mcdFqf#)(g^&UfK9;1&o5tr{v6cbDk<Jt#HC2pDnK-7M1-+_NwEuh|ac&>8sFYgl?u_QU5&zv|>0aY@3jh>GO0~E*$#LfiZ*bgSz!X`zl_d(0Ws@T&6lo)Y3Nr?b!AnYu?u#odmBlJ;Dnsob61ScDFtlhqPJF>^&N2kUyd;B@U_qUc8WB9W(lR=ZR7u;H!F-9UMW28N<MUiCs=3oD+W;KxQY=U$IU5U9Bn<GUEk+>qR|kv-Y-_m2g$HnTg43cT_o#biXYVCAw-1hb#Lcs|-txoN{$^lVZEL`XqwYL@#u$;WVH=Hunt&A`HjwKf&=ufYG}OuA$&jT0pw;B7rJk{xBxeb0w#kNwG{v7m!S=VUD$#pG5k;$3`@5#R?6>oBwm+Tj>%KkeTQb^mjZR2o>g6pv9jZ7!ohrre2XxSUrk>`8e9wKGuJ_|kTT$^@FUtLJm<mIs#Dl4yZ%=Qf{C?O?xjoITKkL49$BQRQAT2WZe2`Y1RiWljNrV^p7qlr!uw1M&Psd9NxjEIy6Ck!DM(c~l2y1dJMM`EROJVWyl7-=?Ng1s{aM!CrZ`as~-mYg_8@9^Lx>YCm*{JuQ>J<?Bou7xz7(5tyhB{%~+d~;py9Kgr+#XZX2+NMRP|UoD<h(E5SRffwu+Zp~)wmS0#QKYwWp88z`^6AtuwtRWNY)S)&wh&VsDbePz9PAO^Trtruz1aJuPC5=PQH`QdPR<uQ3Mk}49TT4s!)t`U_**3n2dE&8c<wnZJ|@xb65QTw7KwZpY%bSgl0<q#a#WpT&($*RxecfuXx{?%6?;3R`&C$%6`7i%6<$8(QaXkrvsYk6JlLOQUX3?2u)EL0tU<!u4)AvXJiIX9;mW-1gt+9xA$;Nj$qqIp+$V#d)E%a{`H$z)Af?xSaR$nPjKTcyPNHhHnDee@8H@N!gG0hbN~AO)=L=juZowizP^2M^{>J;tZHD+m#^b1Hp_WIm*5P_12q=vJ1$B%+H2F<M%q(h#m6E9&q*5#q5^Cj$ef4KMo$D*^pVL}%)H93l9&04&B#@*Ix>R7gMu1YYOvmsSW?7LU{+DbmQ~c($~^{Msw`+nNRv4O*WR*Z@)DQ$$^9F3JYdw{tx<QkMszE`b_hOmaCCitC!|7Zu&>>`fnbXias63!jYq7Uf?J_fl+?!NG&#`46m$eN%uFKna3zJT!0B0wKurle%jTT5ntDRVH2ZVMG{v+k!2WP25f`}2ihH8xnOjigm=g$)ub5j@80($S0NhzJS3HxvGVx5Dz)~vvO?tnHJAd=W)xFwjQ6BE&xtQ{cTOek+BUm3~Xnus9!_DpEwA+N3o$Yu4UcW_v<#6~{-QK~!@?JWoGN8k)zxnG8aF9*8!I<25b5nG6GN*UzE}S=xheOCuh6hjHXBj_o!|u7yZE}=`TMQRD1ps@Kq)EWq16Bw21}L;tbRekqvudppC+!a)MUT%Ch1Dh2V|>ibWwSg$TD@9m93fIr8zfcUW3JM+dh!`FvZ#vSO@03)?;=4VbaTQIf&+6v@|KJM>NwLm0M{WRtRS2ka!M)!sL3W|a_;aOk9Y&z<!m-WauWslmtpb?70eRhj;R5p^I<A-qRpO0*>pq||D(OyJn}7sWAgt-Zaf&}y`BB*+q;)WhkK(xzVq`28GYQJPv86FJOAADk@Qt#`(?M22VV1m%2P2Z3ycb6YnA~)+&jTsrfouxsU$F=Y!%4FHQ)nsKnRfn)T8+78{<6HxWYfz#~DJJ;IW2$F^nARZRK7fqb0Ic0*T_(f#^jFMcP!EQh?il23g{!6vj_K&bZYm{Czw$>#q0wLMVep2XL&BkaMJCRqDM!oR_3K49FC;5L5s|mSxIX6N;r%QtCal{NhUDzP|l!u|nRK{aN02)nteaBm(z7a&3U6Q*gYnToIo&b+0pOpwdozL7T5O8nUJbqVssE4PP3k@%w%C(7Tgexl#X#os4qV>omL3x{i890pe!>u%#jiBjE(rsLnbVhtO<<(vd{Sv;@i~X|jNT9JmaUbh)(1XX#eyg*(?CjU=-|M90ZZ)e5d9KuWeDfz$&ZQ~=+Qsb&VTM}o+MYiU4JFo?(<QrNjSQ|?bo{|v=PgI4G}GaD<K&zKM|(Na*eJ1=%&lciOE{k(OmoYtm)xB1i$homqos306+#Qx6<A*g`)<~>iGe5$N0igD#^ae<L$hfT}^M1U}m0+4o81i)y=qLBIt74V$F<nq0K%e6h}_+X|5e_iv_P!&T3?g3Jgi8-fb5itSDB69+{*T~3}DuxhsC<#-;Wngd6<50~iacD8)KK7G-WAj0GRM6DSwSnl?`mY$##Mo+)vTZB{^xZ_CkgsI|8j7@~A~D_tz=%0XVaO_NfhQG_!_%0j0eFg5I8t(pASF^_orB|KDK*d*b4(=sirR<}6kOugb854xlDddIU&IGeo{8JtnG_0k+og|sR_=qOu43&gcPau!GZwv234mBDaPOKTl_y1NdU3dmE)jw(wyBQ`%?{9<(Gy>>2j$hOO(<<6s6NQJ#J#}n3f_?fJtuYbVk`zOgC25nQLmbM`;^o=8e2**0Pw5XNy~>iW->b1duyY8qy<OAfIj9)_7f1;Tci~DlAS?C3nUtf(Uo3wv99UkAqxJLA!1%_(a4M-D*35V>Bfz)fAxrjSDQAV-J`2*|4a9#hmUX?PT%dxt;mv#i^4G=WQp+>K;vGjAR?>5BM|{|EAj`FwSs3;{S2uq0oarO_o*FsZ9&Piimw&dB2@9<?*N>EdE*dKNGLRu0x0bm(5`o=AXgmN#`zdijhQ-ig}*>P0=ki^w4lX+&nC%O9P#Wbl0ibG7-xZ@E0{7y$vVy$l>^ogusB6HKa^>D(fb3kw&S=zguFByeB*z2=FW;RrL+;ubE5KQ5z_RkP@PJYn};-wOY(2~8nqV8oc!st&_>PvcOjq+kc!Jx%gs{uG>*aonz}@$AG9ZxGhj_txfq_nos%a!W?s2W8F(mvX=u^E^z{ipY|gO(R35IG`=}~PWX>3}I4!msg+vy8NfB{^d-6<*v{xPJM2-s%;8Mx2C1i5xsLx;@eCSjhBqS)k^!=09ajA3;F&W<Z2Az{O#JTr&LomiSo37W*YkP0fmc!OTxkd?Us~jI}#l6Gb?LdE}>j!(go7azS?93yiuhE8P*YI+0)K8~D0QC!M^q92M&?A3*=l`7Z>KgpP9xk^~Z`b>8M_0btG-NC-W9*+ZZ6*9z*i0doYK#(RMFy9g$7&6riev$uV=$6C?PX9F>lUapBw%;|IfXP5v4rD%?}A#1&CJRJ!dj4}edIQ47ZBi8ssXbx0~C@kF`d|OlbiruK(vS?XfAD#&VXK6VHdZq35hksw$Z3d_jZzhyZQR&=ye+Z(JN&?92_;x&&^=(?d)B<If{G8d+lK%J^z((^Js6kt#)jly1hHXMAB>Ct6qi+JtwJYowMY~{vvaU^$Z+iV$~e!D3h>TVPvjUyjIi~z#XDNiZW7d=<~kSOoz(3kSZ$ER68F@Mxkx`Wl?cJ9xIa<;1`Hzn$f(fTyumb5km4`nmqHM^J1Ytf8Lr7t~t*j-wU33Fqn5daqtxe+600tEz*3TihqYmD#3<r$+Uq1LX)V>G@BaJGu2IK99Mob`a0+T-+My7(02PbkKaE<PvJCagF{kV5NeY&+`yHNn!Rjl>eympi(*OG*$8oa2MlXNy2Xv{Ci#rdq@XsfVdNI{g|4jDxSPlty)^&&DV?FmQ!^SoT?UOiZx+P<US05A86^4qs~et!*iNZ52qWq2knlixafDEs!fvC^&<to3u<_50=J)t@xv^S(;jGGfVvF;w?i*e4{ST9GtA=~CsN(eMg1b%sh@1Rs<8<+<mlp5a@G>tK?)MUPLc3^qLn0U(ygS~1;u?Eysm4Sz=||n9pOTqye2Y83xIuDK7<;Np6XZWGGT?C}J2wDYR>0vv;54#~cg{1Js^OVw2KrYrpfu7@6<lP&a{800ilfxtS?x4?K--6{SHKd>oio|&JaPfZ6E=`v?G^9jDt~xjEJs3#5dfM_g<185{1^l{R8fKS_Y|4JP$Op4r8*~S(I!icgqY=WyOiD*9ntPv2jNC3yJUUxMw_1g@^B~I*lr$JM-3ZW0zRLj`}YNM{b0|ohE4ktgeCi@=wj9isV@R6xj?Q2xV<Vl>g^&0?kuV5N7zz~Q>J1VocK~kAaQa8qOF3dor7zotI=I5<e}O5eZN-3q{0At!M~6baE&~WKVbl`&46!oZ*}s3w5k9kdkiIsqxgdRgd1J5wPh@+>Z2k+L{M+*iU9Ghxp0=jh`prhB3WoJ%}Zlcp@u)OcV4KL-Z7OH#RM#__p)74lxD=3Te~8>Re=>9t*R!nS_2rC5i+|JD*l~46$taH5!7VW3Cx|^?;U1U!KyOe=rQM@0zqfQibP?rN}e4F#%EQy*FreFP(~YLGF2(YgcVK~TzT>2fwnw83VwLh*hyh#jN0+vx!DL?kG{1Rb}$vfrGfNQ5IoU)>M-1FE8G4m+&JD104n9Eec(GG-w31K5)Ov<r7eKJ3Ez*%PS2`>`22nUEQF|xwwksI8rL@rhD+qvCNU}TgA@iqhP)#j0stLQ7o=K-73&&JrB&fu{J8{~!grP&EHt`QNUZ`3JJU&K!6?z~w&WQXJ&PdC0(TWiGa%9`@&@F(h>LVIV_2}510hHW@WL_aq-L5+OhTY8Q@!IXDlI|;tSL%VSCeK1!HqGk<CmwCv=@f%<#u|_Xximna~qXGA~n8bqz$H(Bdbp`S_BfNX2Wc<&JlbCGb<`UVwtn&p=Kr>HMgz5&OovGUGzpu>UO_H{;(B@Y6}c5kEH%8$y?1lV`vC%wSlQ6uzBJ{+-cA@YU|DK{kwK+{PXseT_IXKH`@BqpN(Gr58oZVdgaP@Mzdb>KWoLR+9uYYbxo#!P4$kiz5Jg?-}=w3F4jLcuXwLH2>5uqU-jN-Jo;WZ*d?*R%OlRI^olY2$9LY}nC`T_HN5Oz*FQUP(D<PJ#fSYgYb88BXuD|NZ4VyqOdTjZY`<|Qt2K4N(4H>b{dDw=(ck^m9#YqTLc;vF|9A9qdp-D0uO+&k+n2h#)oOpuYqT#n=z5`DW_Md%s!osavC2c7@PdWn$?P<0a<u}xz@)R}oLNJ3n9N9F!3A%l0~-f@%OVB}qb|*thbVTrISN`P=I2%+9Wo#-3QH*$s-=)#Vs2-kV3VwJk$MWU3Qj8N0#76fTd@do^1=zLrF`P`A!j@XkeqfWoC)|!3YXwqY6N)kpAIoao`naLWCc((kgbifCJ3$)afQ+Vzo%Ir(!M_Jlt9UU=HUJ8nfi&Agcw~S*9yb~tS5?KJsl5NM6fEEaRFeR>7bJ)&YF}nEJx}Ba4{MZ%{XtZ!O%wkJ1oC{!s2|eP*^Ziv8e50VZrS0xmqYJSoI+@^#gMs!!JZfDs}RJ#DE=wFh~m$SxKQIdUA0WE;8jYPK8+jSm!J!+vG&f7n-bz_35RNzv#UKf7z(--I?ztSdVbFHb7NgMV4Gd_&_uX#b|*w8CCR{p_L4jnJLVt1PJDk=~=EC%n)!rs|y7Y9b<@=T2qB_bkx$VtVNink~^+AI3l4Hof2Y_X>fy1&VkN?mAG_qMPXYjlUq$<e!op_OLGeUvq0>Mnc<_WmDB?*W(WytWyi6^qt_Uw%n^mH0U5G^ww5yHdbT!{C?ybqJWVdxBk3u?0szw_DrOu=RCHFxOp%<_*}XK4^|nwZ6?`NM99&B&RWOU<QdHBTt@nFc@p)UwJ|!1MX4jlkO-mu|L@gAUv5I8dEi4drF2*6b$VY5KD+k-58Efa5cYu&y3O8xKnLEF_etZqd;u^>=O@J+s=x|t$4z~^ixa{}$Zi=m!c3*q#um5%nbf;`R%2z&)-!EX)_^j8q4^!D8A(Xx2!_l|?<LK*$#KI0QjY9M1{@(s59q$|+ACwnPCP2GQ=;Ttp@#LnK<=F>rkr=>Iz&OFD3l*#=MSn9Ml^4lmL{-GrWMf1y4T!Ibyt%o4cBjtf?eiQf$9yCGpjq>+Zx-0}(U#)fqVE^;csgKUN@U>C!*qRbZ>M?b!cnDPaA;dNvz+*;y|U8s&0kr3`C}e%&bzQp+;v`AWz}Y<9Ns3^v2~tvSvz^};ShD|%y-5?fCSOnSg-os$#&@J#JXU8zuDFl)dM`I_gVu;roQWDuaXXY%-b*0;cMA1_+ytgHQ)ZX|L5PEZ<x2Kowi5SBlR{N4OIk;6C4IO!vex`nHu>1guD5MG+{8AHhutn^TyWGuo}RpoY`tX$t~Wjn~QP?3<$xvVA^w`s7jKA>@%=NAk|c<9mp1bu!t_mTuR`1G7`z+RmdnW6hO0y1V8BPplHp!h@|`5w5B2*<|64NFNErxQ-(Q(WDN*zQw)(dbo@m!X|V~oFJLfK_2flIg7d@+YStJRUAUDg0UMSanzBY)O5&Ug&a15R!gkMiR4Q_fC;{jQ#4@L1NRyBoFj3)ImZXnjy!J*&lT--g?G|_frxzaC=($#-y3@(CHrrO8H+bv8nm19wzZ@B9YvHJ0pUFz?+7kRBYv*0Sf(9yO)O!c;M0_GRm4X4##6*&(x2JtR>4&`AYW&}8R57L=@3tlBS8FGrj)c)c%Q!VH#}WgevJld1g5woviH`1_(XkeuqqJPdOqNJ2sF)mE|0c3IQ?uG&v-m~V720fxUK%=*@Ade;`_1Bp<?-%z+RKGbsS+VDB+mVC$_^&yF$>zRkUrApp!10f+MYKmC2D7-Qlj82DR~s~lcN(9twSvdlr@(59ZYkXuWHC&sRj!&9jqh^YlJ{oDsFHDjP-#u;4xzXX0yfz(>?-IUS%@-6g*f%hRFPxHMntsuP!4O5|e?p5R7p;<5Ik&2}IiYE=A?4#S@g|%xdkW<0M&T+@E-Z8_1#t%aNWTFNEO0dw9mel2)@+st3!)3a=JZPjO8OQc8g82`UW7bH6lY(Er$+mfUI+%0KnCBfp=rxLC8<pLQ64<f7D>bh)EQNn6^;6e>nrR7gA|iBM>z1p@PdTy}$#nf4|^cBgM-ella*(x>V5=Mg6W!m38*U@@eO7>@|V5!56Ay{_4qY>-b#VU9B;@JmsIudZldov`-ZKS7%-!Y^(CvgQ|z`K5WK>9V1Xz+X;;Et<yj6;e+OZ>Ied-WU{*zcDD5)R^NdKEfOLi?p+dVNbs5DiPO;WlT!IIAph4g?BO{Ggy~Bk)32%Rd}jo0DLuPR@|u~Z(qt1-FUE_fn0Xl2)kIL>z`UlTu1~!s4liBhs9e9ZLK*MT?KIBIS{SGSSlVhw<dc*jz7RABp4gR{P`~Tm&K~B_!Z!YBL)%DBCRcs5TvNRESjhtmEvTcoHSg78UXH-jO21#Rq=%<D=_eCc^Mk_U1ix~8JgO3qc^s9_HGoUw8K%|JD`HyYXKoRUq2qb)QrWY)=QF<)Y`!r9TcDove7)+Belw|Agz5iRi4rF@%cy3s`RXi8jNAN=!9rrQ|7X0s)@JMGs?L05%|bqd1jZGBdrgz9Wve^A1JS%kHX()SUCnJ6u1UO!t^W=%c2&IghvjGB$gs%B*TITUZtr0AZUhlA|sd@%Rn2aqlSMNG}x!)a~gNIrtf_H@F7J<8)%XC&8wGO<qD*qg=JK+U=baP$3zo>gb+<kS{iSiL|kCmp03bz_oHVd&w$7xc?51z#HlrR)zRS>Lk1!uoNPh#smMzRb4WK{YbTIzqA=bSYfc}@d!4jT+grZw%kvE1Jc+<<5E&ucn~F^AEeo71#0&!n%b+JD*ifnyi2BCTxju=j)N)a`58iJU!~N;anbEADz+gmh@9}D#QYDpnZY0-=yMj3pk<bG!F(ThoDQb<{L{+^b2N#AU#Y6QbJL#jhOLhuYv|Bs9A{pk2_i|R?-FR@(6flrNjv>{N-4yPk1#H6KA_D6~MgkYXlN~%s8O<sJ{&}b1dD$uO98gtp3^W~GFlJp~F=kqpyl7diKqj|{Q>Kwc2E!sLAwXzd1xVdR+Bx@9c$KtKQhB2TH&l_#d8$UGm=mvbAY*|-gpmw$mT~Y+238=7^R+&2zO!m9*%=J4{&44Ko1;E0ygQZPe=<0=wnd%~yExX=H%tm^d(+l`7;Iddkhl7!?Q4cw(V;ZDtzo^}T6zqXssHPgQOp;UPhA>?*ZvjG+lY8>3sTYpbfHv6XFmZiEh6F7mb)Zr_lS(QnMCxHiej4gtVR}*alsmaC+0Vh8}m?-;#g4!Yxvl$G7XgxP~OlvQv`NLN+pL+8K_~u8tPTjw7kg7G$5DN#FHtRIm4u32pnmw0b+8LSq9n>su8IgFbGIa0m}IrT|};n<c1@HR|FvvxI%Tr|0;O!m&BKBnZoqB9XBtO?8h@OHY0!;Ga1{NYPkde^xo!T3~p=4iL~(=3vx`n$berwIH)6PA!{zJywnNKS)F#z!`?=Dve7p<az7k){iYqXsNnWNE6Vc-=2-Vy<uOvN9DThBb4?EdNnE!t``1<v=x(P2_8StLYibhjPxT*6-k1vVCxW#7#}ImLdm?_(pNdVrynoVwZ0}B+GW8bYz3bLL_Hsk_nt=L~wuj+CtG79=|2levdpUS?-x~GM`=2{aqb5wZ{!yPb|DJrc<sM(}_p$%M)Q8~9{c5IxWYinOz28d7-s`kdo3GlocYM}g_Jg&!gjGg+<e8|VbyepiYEc=>TPAOXX*6qWstB#wGhYN99&#(S)LMfa*9+hh1|r+rgn`leX`e9UwCUQO8K1DT4@6sw`;&J3mdv2Djs4@AhXt-8Vya9^Av+cx6m3#kh4h^QL*_1wyd^y#p|`|@b1WdiN@Y*2ET{ANUv`xO^!1;WLvIkq61E_|=c}s*m!qo0l_?W>b;cssdtOMd6_b!!MUmlg<<u+T%|#*!z+&DA5+k64CaD;;?G%ZzstFK{nVe{U-*S)!g|(+~Z8aqkc|eLuUz(HAyM00!8r0rj^GRq^_n-_QPK||ws*^6X3NhAHickU=##Ctp(z*lh^v(t!fLbIkG33+43`l`rdQv##QCC~3s)59YsT_|K_=_|t)8Q)DK;i}fl^i>QXQs&Pr%njNjn?(#?uTuTz1@bKw%4;IB0udtF=yN3dPCuRgK+1l@8|3GT9-5&sLhtt?IVe_@h4qxe#yO`dwlPB_oxX0(~p+7Q#sf_YQ}idIM{!Kd#y%c8x`Lhj50s($ziLO{&1tefEEs5iu`iaM4~>KPxmB#{rXL6q2JlweFKoWHw}7W)XR;3+-^k<_pF`ZoxSIeW@B>a7chNN-%iffr~H?5zk7G~p++%v^!4xU9qi=M_b@d^-z)Kj4m}K?*LOkQZLSxe_r33ZZ{kGIR$_gC?B0TpjuWoDl><_0jZ3GWPUbcb`@8i?&s6D6K5Qq!sDX9w_V%;)+KX@Bav-`NY;JDO&4^)=V^Z_LJ!USbqtQYbgW2vuqI6VL))DBy+aP>trkv%qfGA+eNgpegTAtY}=qDPReXyNnvky)t+3Y*)A4hNpBFsm}t>5w??R0JLUcE*E>*|7X=HybA5PpoyG&OrY9373Wv}$Lgqi}Ey5hg#;KAL$swJ8O)K_+cbA{#2sCz@#AoG)RYJlPIU9yGf8%5!fQirMW(v!>c5wb3uBjoBAW)w6esYg3AC3u<oDUr#K2D?M9tKcfk@pbBRfh2q$v&e`NcPRuJ)_x#D00tDwHGn()h7@$uQ{Fqdj0vWQaN{R(C=1`d|$yu44&Q*~@L!?I83sv15qEV!)B18nXXFig5chE><RXg3d*O9wjG*o&XlO$CR(As3kwY!9c-qZ!pjutQfc!+LOeW$y}J3IYGSjPlf6w|ep&Uc}<&tHb<X9DD{-kYIqW}kfYXcAcw*9Xz-HVqAouzR1szVe?&lkWMS_N?LV0GryXoR9is@!7IB`QFT%eAWqPHqUAk#_y(;duKLAf7+t*B_A{)CGDsD#a9-LOP1X9sXko3$sVim7J=h>*}eLvdJ+>a+QJAKBd1uOvk{#3E~}zS696)mu&Sh!!jMfFRnuFNF$2}f;0ByMVl8?)*gHJDx}CT8ZgfcbMhPUBaPB(vM+e94h);fK_cp#*Gek~2tHihtv~?G=%#PqG{?A1!kfBq!z!`}}2`W+H0c=Z>$6?l56!Bd50<@U4z5BmxtGCzO3XrJhQgNqBHLgY<BsD}jE{Mk)X*8E~&TXQIC4kQtQlP-Pz`!J}PqpL0YFT?(1=2y`K7pe<rMP8C7Sb4>Du}R=Ie^fsLVA40rskv}VQp930bC_tY>#|?WqW?ybD4X?=HkE1Y9y~^hIukH4nb5b^XQyWAiR<^p(-X3{&wV1N(Fv0#K1iHds<RWEKGTmPMcw7?HfgnU|8TLkcgY1IzA~`)sgDHkS>&WCfWopQZk+UaPqLyRt4e62fCiV8#c%-Jri|?Q~;?M6>M8yFd$?Q!xHuVWt~KAcFCewLLn<UFtZFI7Gg{s&vG_gWJv>nuHXFI!}c)YkMDfkHpWZ~2K{RK4EWP#0Dz4?S<#N-&Qs#d*QtHL*IS{S!Nap4u$~jrwyTy~g`AC9lc=rKgusj=$8T){7KI2($zpRNrP`3zzU%dy_Ux+J_4@i`3x^Q;r!PFIeK30GX(x>K=U=x6U!#7~wK7~S7I^r=;>_l5)!rFq{iE)k{jNp$ElB@#x-@cH7VQ4yyEeP@?ODYaeYUzYeb(3Vi3jbo{$a`-<Y9kEK|6B4n$`@v-4rL>ntRvw9{pl437Lq0eK<K@q|bi0FO6tlMRepB?TO!LY9T-9<LSd*xU0#!gJEO0Bi7O#eD;rqf{R<j%RZdhs_PG5W;M(n4##W%GU{4AsR5GS|JS|$$-ATe%iNl);dsASg6nVFWGot!lXk=Il&25hWAS!=-hPw$yRv)=oUkRQWM%?ONr$HOO3O$no+m8sfS;{USa>R1u3|8**l0O&hR{^#o{W`GdrR)@+iz2M^KMUo9xU9@cP`8p-3d|>enFFBBG<66pSMg?VzpVq<4Iynq&1efE({-Q=e7HXZ#{8FShBshWb2QuzDT+9vah_nu;X{aJ?>l^rgkM4ujq=}n7;x3)n~Ikn60RuoJXj$#%E`hR}~b<lXiA+GI0gwqZ}w+P)Sn-S493*f+X8@d6xDhjk)06JEO~^;pfd$Y)P(+u9e+#5NJ8RR={F<cZ{9w-7>ndeYkNW?33QesNJBtG>XSZqwS-%)Ts+{jU3683`w`Jo5q&IaC5{NyNv(el$)o?)nL_<x=75;8Yb6`Eom1~+8`{lflnTcO$HU48+0M`4x2G=mEGoj4o`lg!>uWCmcQwBQ~ED%NIoTuh7#TGW^BC~c8*U_R9P95>?FEeF|&d(^uvNj#|b<1lSS?NE&3~F!P>(t2;F>h<=MnUgU9KGSjZQxScdg_!k+&(=gnOvRxsb3{npc$W){PYTk*B&V;4dZUw96mjiAf#;|W)NWieem4L=mS%n#+IhPdW(czr|`o!bY~dN`V0ASs+|snSKN=zwrqCPPh%iJS#R<;DfUfOZlyV5ywCF>T@Gr_Me2%!1ce3U8J$0GR!fXosg+dcD)e{=wdUIXJr6(39p+rZ;QrFj03=+jL5ikFQRCe|Q4um|e`oC-d=UNEB4wDmz%v<toU~VYpUK%v47=_Y1iac3*L?*6p<2)KQ)PM!Tv~6w>3KR-tOhku-5tOsP6xN-0_7wdA5qVM_W~SknMTVs1H|LHzQ^9d+p8@n;Pxo<6>RckO#EIoncYHXJP0go@<rZPL;3BE0rM&q~A~1f}PqHMgmVfa@+&1^oNeMtT0#^<PoVj5c$qu*gI&tVr1rUw0TKA*waXT0uV)A%Hu$tR)slO6tMQEBid#Dyo8e&vnf}E>@EX1l6kt9i0Y(0kttv%0m4(jyj@?mx_6w$(&cq63?Jw&{jlX2EO0z&Hn%Fy=#wL*OevsR{{cRY@zr#?+1-W4<$=8HQnxVmut`;Mh)UT$+VN1iOi%_8sq?y3l_(4yJ`RrC6}yjwM#UarYMTw2buaJ0sTu1|B_z&zT(C!Gx-W8X{%%w6&V?E?>*=2z1BH<uSI|Pwo`0a_rMj^Na7^na7)_y5Sw#k;}P4b=8uk*vnbKfv2qr4B7misB(nkQ)4XVnCFTTOB}@6@=cD4wp>n9*0qW04Cw&>`Cv<i+D{cO(R^9!2Uq<}P!A<q!(e^^S@A2R%bZ2Nw9^cg=vLIeM<us<W%`~GWZ{}-tnYu7J!j4h`lQ~6}qsWOYi{-qdin$<G^xSc=%6~pODtJRcCwFdgs{|v#+%lpVgR3Yku;gtC5!a~&<DRPfwJ62=BC?P#VpGIPCCo9yOHzm*E+!H}^M+T!j3>upEuA9;cB~mZ2OS0SqGyU+yb|}aO8LTDnU)yHsy#}ZOYkL0BTKe1k^X^>+Sn)rs3psq+{P2f-=rZ=dr^QC!tqLE54#=OrrKidM@1$15~B|;1@PWNqv(uGl?&!b)<0771_-j_ic>pc!7>2~k{lTlEd00>@;6&M*Y?++DM06gQiYYoB{F~w&hwZoLqg|9`daZ)v=XDr1dm=ij3L#DdxQ0yPE*lQNeE2J$ylp5P9pZ`rss_5Dt(ejQ?i)oYLSd8z)MipAyd^{qQu-iO=zW(){BbXBcK{2QbccZrK=i1*<|Lma-l?pFw0m{u4owmRZNwFv`J2YRz7I(@~s{(51Zg$uYCem&#2zh`zmyGK^Ylj&QuNp5RGx6y(Wo3ngC}MGlG-CAqgGg-NnCXJ+R80r~&rt;V5b{I<I*!Y6-p7z?QBy?QVa#`Jj`dZ0mQYXBb3?em7_Vw5KNHlOV#W|CZi)ZZa|j`sQTHgOjdd_t(wW{I)N3{&g$saOCfObZ+wKN{4-a@14Hh=Hua_>CZFm;Z$34eW!PlX}^R(+rRc-_syg&n;Re9{B(52(iL;GKU?y^=1ZtY=W1O=;9Pm$f4X?_@8*u4em5KyPTc?a*Z*;U=K6*YxHpX3y{5uzI9mL=GwNx!RSz$8+pKMYUi)Mpbaj6V*VkNi{|z%2{@(CyGhh96b0T;BEKQR$gY{2a7*F5(<-*s0IGs&1@BID3JDYj%aN6$N-(mayyVLhK1K&=~o2$Nd+FRAew{Cdu^kw-oMeswxWNZC1U2Bt3d-mP-z4!Y1$0-B&=V_zj_;mQ_1vYSqISIDIj#6B2zT|#ig+Kq64^Q$L;{vz#F#D7KIUo0{AB=d>Fm#9g!(aZvC%EjN4`;c<LFL`%wD?xvuKDefg4VwIpN8)j<Cd~vYTum|puImCX$juz7yiR>$7Oq7d#8Wt=0F0wC3ChLsN0{PYkbvjwQHnpAbmJ{XAjT!q3c0^f!~fxgK4mC-+l7Wp^>}&#gl&yjp6NkPyRWFv(9mU7>A9U?edw4t@9>m?P>e{@sRx&GPLRI=WcbexcjZU)1H{0|MB)eHKTXf2fuE`cZCSUI_r9wyWr3a#P2Z{!z;}r{c<L>^nY&y$KAd-y6e>$svzDU`kIGKFdW~1+sBk&Pgbe$-0dqK+NXVEsI&NG|6Cq)y*1x$?bFUS)lT<bZJ*mu+sFHPbJOkWq#~Bzw4>Ee-lh|^|L(SJ`rf4P&A!{WC4V!F<Xn-+*W*fw`=gZ8UGsE>jry@3c9kH_P<%LZ5qFzz@Q?c~v0n}!I>g|y-P>o&p3-(b)3#21($}d!oVC^Kez}{4hocESTLQ5B<Gc8Fub-9EI~l49Zj8d>v<H02y?@e2nBn<Sko;;W6P<jgH@Q3PVb-_bnG7ZU)8f(gzTTU|W?zkl??w|tZw`a_ugwJdc|QOT``VWG`#^htSYy+-esr#%tkaU4-wZRWLtFHto1^J;f7HY_`_1iOf7MU5?(2TuubR(B6KM*RhOTFt692Wi;P?9&G)88hwPV|j@@&sS|Cp8xVh=_RN8n<>9W&3Tz1e@9uJQf~Pmsa-r}ag1!T0(E+I5aB-GlsjuOdHa-`;u(TZK<5<b9wjQPAv)lZ8`-EGtzsmE+9V3bvFa^C4vJ!K1lgbYhle5?X+a3ZIuLfgc>D2d)n|Tjfrlb(9W7-Z2riGRzvUtSzL)5>p`h%UEz+rb^Dgk%2nLL`G^t2-<r(a487|{zqJsv`AHv8V^XY=(z#WEYSyMZK@W$63?b;Avu*LhQJjb+e@-*Db}S%_5$r%)=&yL2Pkf{Vwfle6Fa*gNspo!vOL8MvJo#OE4h+Z=PHV3jPya%@p~<!5<mLOe|sM4l$BI<gq3tZ($`o^@K&p2NQYO@|D5Kj(oeK4rtuo7M$mpP7ttTDPI+fsY~5-(-hB>ojs^h&GggpDJh>2tT2+;ycvDJsInyR4{-SFUUWA~r#)D#tO8+hQLcU<zNt{3K$yc)h*sGJ=>(SKvgF4RFJrsQ!$*6cu&nvKO5LLD0vR32>#|4R7loS$H6sr>Nvz*lwghZ&(ctgf?3~XLEnXlAMUUh2a90tTDRYDFaIuVmF_);beIZg|$vQjEUEt%srYg2T=)UiezpPXUtC6V%4c4~!C&y^8)Jlcie9J#wEW=W4!Di*PJ1j)BKuB9o;Mk}zG6sdmV@4yN#xDuDaT4q@=&Pd{4XfiNxWK)zJqm!x>vdW>jLQQCu91j48@H^ytU`os{r2@NHrNDlT=3aX#jTU=X<hkTX{y^Agro*l)`Ldegf@K*YB3kRC!m_qV;~I~;TIIQS0HYCkFOdhbvw|!WfmUTYu$C-F5XS@kOjd*cC^Y(v)C`p{x>PP1o+L6ws)>+bNzyS==J%E_9;DO#UXUxIl0A^CL<#|}5R8-)K-tP<z>kn{L&(1x_uwV2$|}c$6_ubhP`0C<O@@)Hn4|W<TOfOtvsVtY5kt%g0Y|`mVZ5-q5f+a4x}dx+$y$oET6%}aBk*fW6ye|5p3pFCZXgOZ)qY_IH`4g}{Q&WE>+1HEur>Mnu=|s+8Mb~BcAuAu`;#8HAB1e8W|3=052{3C10X^!5(%9S4!tl|;z?pXDgi=NX!!`rQ(n-C)e$oA;x97r6kHSLD<-x@(#@6QSqr1^h--1e7>T7>Q}UVX;v&Wjvl8P#)pH0?@dT>qGXjDmz0_u(tvvzza8Ta+q&p+(Vb1*y<}|xB>xSfp=mQvQsoqJl#g^c$fuz1NW5^0Mctul>Wu9$xHu9{Zt(l4Bi#Y)#AIh4#4W@7A_2budpOx}hiOE^4pX5SB@?#Q<ENki?l0qW7l9QJNCM&7r5a~eJDg0UxEACT{`Bk^oe82C~qA`AN?bL%1gs;GlLXjzD)jCSV=!z_lI20pbRF&Xd0*H#rvYb|U20*<?GpW2h7FKuKt+7x0oxAI8J=}xYZuK>&Q5X&k@5phUq==)`;1g7}Lb3<Q=K#JHxE~V~IJbJmF)B7;^(C2se-|oto@a53;l&5*NjSOkWNHk+hze{TEVg=?m1Q0j-+-Sc8J#0H3J$y~gU*<j;zYX?@6+Z)Bdqrvd?t{&kq|7Cnp%P}7OViH<$zqlO-O*VNE1R;S1~rozsUAXRA5^GSp)QQng$EJpAfetxE~Y1D8?=7z=Tprh+c3<CY69kvMTaT<p4C1+?|xuK<tmOrF5@7Blx&EJ-git)xNj(ATNZ8%96T>!XQ#hIJNjk9g*BHqFIxYs>Fk-Z3CGExW%0_Qk#ulJcn(UvI+jOyPd9-tGF#dszw8r-huyaILeah9r?K1EjM0mBJK1T?XWABS*F^40{0Dz++m2&09_?nD<&DUK=>qJ&ZCu4Wkfrq8Z0OPU%)KkltvQLKH~8&xNpb{d=+_-tL!%f!!vhDbDJ5N6-ToE1^3FL2u23X$mKxa+)1iMCb>TkmPPX2(Fa`9BJ=p$e;^c0ZA05BT?6F^`^}z7*xJs8wpsS+mAH+?+znqYd#7LvB^IBHHcqLC;8l~;+A$7Ff>lkz2r`41C;**mi^*DGb-2p21eIW=n);$|p$NsLH;x(0b52OIHW!;J0Bgl`B;#RkZRW}cO?%7UYip2TBRR)}e3~pd45Eby4(XLS#taEq1CCZ_v`N)yj4R+c;{~@%{E}ssC4OKfW`fKyovK@LCfD3}F1QkTu@d}?PV1WSl1`8*G(s!sBFhTMT0ld{t3kV<bv{+MQot62HjGZ`c=Fs)i-A)`Q9Y=w9DoM^D1k{OaCR_>8gxk7NiTz4=x3RW&;M78jSuHk*n9PNdX)-2Uir^$97K4z+G%dIzw+^1rRCwcFq4k!+q3_hZ86MHRdI8!$m-vAakk$PYrf&;v~H$5O~2b4c;4=+Bb#);TXw?6?xm4xB3?qoDrCaPCmUD67WOvcW|_R!>Y-0AUi|AfXdHI~!XN@ae(Sq?uQoT?ein^d^K|0YrR_`2J1;e#NbzR+wn3<2FI0+?^kTpl4Hwlm89uv5)_=*d&yBul=AjK~BmCc(er13A+TN9opS%Nv(DNHYNv|wT0*e8AbpMz3Nz$<8WJ69RU#0F;hX!e%jF|qP69Tp2CZ8T+BMXm6+ujIZLRk-f3g#e%GD$=ySh6CAWFi<VBWH>ncmWU>n$Bj}@fh{tR4}tE<+?E~)ml5dkUB&VryK!8VgdV3iMp0D**aQb0a`#dDDD|;Nik9j@Wxc}FqA1Vo+fsIyc3EJJS%bn(Up$M!1jc~>rQiEVl4rP4P!AfYemHEAzy)Kc%H(kN>MUuU0f~%9(XOzSR|fmanWAXDsq+~Xa+cCi&xTt12Ry(WL?QZ3vsG$C1+lv=fcR!yhn~ISU(jY2%L^l)j&#?I)@yc?iiqE8;a8)?xtd}79?*w&o)Hnjkm)ly?p6%pbOvMy;jaFD&o#seKgWQZ7Tb$x1Q=stA_gJ!$z#OtDn3+Z7gU)>_||TozsTT{4*!7i)?b4Q0fYSU3QMlJ;wjoRSp{CRPd+we%)02ebC?Aw|{!?U-0U$#$|HD-fvq&IIU!FRe3*MqGJ2OSjej1?igw}Z%mdx$`AUTMZ9`*R`-^j^W4n*Gm^n&BN<$_HW^$@L0dsEYABIB5(61b`=|lRfu2M(8qu8@uUzC+r%2sJS$G#PV?iFn6HmswP1)!DY4v}<T`@DQByb!DO4Z7Py=t!)XU^QjF$<$PeACD~jb6F2Fi6UREdwtyEy%O-u?DGo4$Xvr8snrCXa?<UC4@g4@j#U4ys!kY9+TtLpACwrtxO<N3b7pg$^{iO**n)0^?BWI`!>dg24D0b=$AvEMAtAhX(iqIHStkfKDqYoB&Cs<xeQ*`QXL12juMmC0k!CCYQ#u#Lu9(pUXxHDVV2bpf)hd1<H5=8&$|-z&XDTM=HJiPf1kn9@|22aFPW!X!dmoIGlC-$07TkqrAE5LlFr;j+B38)^4J)dKIuCBb=TN2sLD>OV`me9QdV0sr&kUW(JwAk#W|@#FzuX>TKI*9Iuw1sKROziDe8ULSGP~=aKGwHs}C2JGk}`l7Ar>5uAn;|%ehqH3KQJWX<jyp_ACqVfN+)ztZZedWYAE_f{aqJVq@iA@LHMDrwU|h^6Aq6ja2mOBJL{%8L}`OHx&Y1cMh4<$10+j15vQSgu+=6)KYwmKOMC*d_Htk-R~tHyQTOA{xauFN4fsah?x)OBr`G#4+hIn>YTXVwH46&=|tng<ZpLKQH=H%e^f{-72|$|d|yW2JACMavAf-pYn^Rx`D*ZpYa0T7HEs~NGiq=gS{A+;>{YJ!T>!nVW6w~R3u_LNnI8|{a%~IBoe?vBFtRGRIRy2u`(;QA{LG}^i=Otj4V`!Q#>$p$Yt-yR`seYlO&`jyrazkNrW`p=H+MGeEgU-+{JIYgH<w%J^-!(vHtH|&Zr}LYfwck4{-SH9=?Ci8V1RIAa2K5P`w>GsL)S|)<Cw>ukz<}>h+qm4Rs_WUTC`NDAm}?=y#R-^zy)BdB`Xjl4H(Cx_grVEkt0;+GP>xPhw!|=8z>?YGT)^=b}ai{<b`f;<{7)_@u`a*odHZ;^c;|wmTZNK-lMMbW*qM>UxYNAkLi+U$oAg*5|Ht#HHKB^1MqPP&ZVpi0=85p7ny)en2e6mk*rF<)5+zUPqA<|nLKrEOW%&B{NYhTFpn*a&DJT5&DJW6ZL}vE@0d+sPt{b91uKytGb`D%Y(ukw0diGo8(P_@uwo;#0&~iyxOQP|Y9V~)c5v)zN?L!=T*s%={@k;@G3@P6yU^ad!`&a696JesyF<h3vY)n}{P~KXel*lf&Hr@V(%7=5ua`Hl4c&-0`;>ftaA+HhbI0xcO$7U(&)koG&t398^3rei9=3hg;q6v4;>Oe#rSEc`Iub43$v30j+$oW~*~Z<6J>6_lDqZvSV?N~TX~$ttK$qRZ5fau^_bfy7#(+%$S9?{-TU=CXNyB<Gf@Q_DQNXVSah(TnDDHKNE>v#CX_ByBJ%Fc1x7xqHAZ_~kNj%~f<(^+(t!e${g2w8gIy?Qgi?g3S7JIcPk*QYmQrOGHsfQn(T~MSo3zz-If>`bJ`(Imds2jv;pO2=r(INR#L^4KtU<>K8k4_lUMAbPUl@-rw23{)$AXkMX16X&C<QC$%Kd$$Imk_CiP=}0<#TqRsBW-3WHNXjTOeFk>)uu+>C2l=uIz~FJ@Z9r7lq1E7s%?HVYB2t$Z_@3K10Sp0CPrS0Ceu0v)~<c<BuoVGMjZn(@ls>v!e&5qa)+S`*PNrNQ73J=oEP4c;(}iex0-g!WyLnO0$9eP1i-YYAlii_YfUxU%3K5D9*K0*7ALR)bBzbLuqH^Sak)vXE|MYCzS$SZOm?^5IorO{29sfUA5#;LaTP?>TGwh6w>)5inW_^vxW15qUt$Hrp%zS|BoH+9E;zaq>5`YryNhs<4cXZ2XjfAz(;cmLdpa9i`{goqroA%k3|(q#XJ?vD+TTx`B@oG(*&jQ3{XAXH(_4~AP~5T*Y_OteorPp3XdXchuri2rV6{d(rlhMfTpK(_;6A5#=7Ar-C49K{vsYcfXNz@`E3SOS_=C-eph$id;XotA2<r+FA5*F+S_NOIU&#mrqf>QYi6>91-AXDIPQdq;Y{oD$!TJnpsC32pj7~>gWC=B}EQEGI9LzGWPt~m$Oswjw!W6=l+CYtA07D^g5^k)q*>hJp2PU1>08;=_0yo4lC;4eo{-awlI!AKr^4v)qnuBFkjY%L|nS38z3CV(hN%B^;#v|cHrUR=o$(P3@?t@;Ur#s8}O{XSv`}%r);*4=%x?rrxhO)9k-D(VDBb7uVqm~~-M_N3SJh&7B3#ArAQlU(6!Tj2}@$FXk=UH-N?mb&+m9EvJa^pLr{nYmx+R&-Ye9^i8Eb7~R&^oBkDAD}!);|tA)VBtUoHLV=M$pYEXCChM;h90?`0fm)ZnvBd&kP^>^YNiiN3OpepS+(LeAn9(_IuM0yEi;saOWkrOhkN6Sf`f=n_6LgIP7uH`T0`G{8!Wff1&@ZaXG;KC68t1o_l@5hT^VdrcyEuEGB@asj(?&Ycz|g8Wx>Tiu2+zX(cJ<GtOLeiu$XAani)XPPyOtqtC)8=dk=K*`u4|oI8kc4%QoQ&Qnh(&d!q1vG>O73nXx`DW6&t4V2NGU2q!Viv;n}yi106e}Ccb?4l!+UFnh3xgOzz<PjcCgDd-M%PiFS`uMD&c2mVjv?luGh<p`LS)efvJWw=1CRi<P#ikr|204)oIMfNJ$z1*A0RlIg<B@y46~RzJJC(ctY~6?eh`DePgy$tP=c<d!gGT32Wl@%4P=EzMp<T4z=qM~JzR(`JPR7LJ#lc6Nz_d+zr$06Ns69(sJL|EUG3vE<)R=A<_>7V&0z4}K5>|a8L|Ps40G><1?6Y{Wa)JLO(VEkQuR(HTDvS%os&-l6PO=F>N-EXFSn2A6hytLj*cg%r?8=S=nze~OtCvzDgRhXt=(Zt-PW|2T<C40%$a#^BlPY9%$}*R>w>IBh)e!cS6)jeaS*()SF(MVORAeD5hSA}e&k8&sFu|fE4`S-|OU#m;x0lWr&v}+vR4fbM$imGFQCumL=2>()6o|xETl2OSc>Hzy|Nc#1`tYzXdbmFcdy}mVO!=@?&hBkw#Io-Vke1FO{<=q-hySL3`J>^b-c$|Y7qMV__Y9U=(}(tM*_-rRNA#u|HuuUp#4p&AowaAEUBOR}b_IKW+7;Gh!simHOLFgW#3QU(l_D4`#%KfvNlsQCtk!9yv1oG04iQfxc%3HqBV)bIq?V8HshoArlYTyypj0L`UIO?}rCrpxN#&E!u>{)V545*j%&E#n-jT>-ilF(VDdBYH<k1Fduf0rEp_sGPNDRX9Y<W`2Rf)S)v<vB8G=VpEAmxE{kK1=VuWh;@)_Ty^p#5|t@;iHKyzqvj4^R#sN6#sHpRAxA?^G&+lZ^>To(r2yv9=^l$1fHzQ6#dML9&jO)cc$+9MHrcq(Xg3Dq~QjGY)fsOPQnKvI>!FG(1YdBxXEG1!;-QV|-0qTYyp@&4Gx@NFxgE-=@H;)Wtf6q*^i0f-@o~p-RR&!W%XDo(uA+09NrvT4&}Dc-#f9F>)6$ew?j3?6K^?sROQ7Hdsd`a&eXc>DWSjBTCaT9S>CStd(FnQ`4~s+*RsBt%BsDB+`HeUdsm(YsHlkN@dwbkRDc;inO(m0v*OUOC3TfMdzz8HJK`N0!jc%qHPLwppasOrQ&2SoFjWM?#a9-;{Pj$VTq2~M+H|g((W7c3?IV)ag{}w)Gvu_s$7zOsmPb=*fbr=1?osmsYFB!azH}<_2iNlYRDO(h`AyQ&}szdjUb)D6f9~RjgeF}W-_(kilk*JnM{KW5~8(W1D?<+i#Z{xIgP+qTn3=RLSYBR35F>j1ZF(OgG%77RmxXg6lYdcDl<;PV`Q9Bh*g+4LdpW<Ck<<yNM2;&R%8N9Tt_OAO)rZ{0hrRMS|yK~n1}R*`B(#;*$5!wN{T6XIE+_i(-RiY|1ezJ{NKaw?#A9srMxxVgUb$oE>_CV{v3bKmCC<{IkNS`?RU!NV$C~pGMoN+Dr(+s4hRFZh2k+`t!F7mj__)0BvCtwQL0|(6c9(EKv+l^IC3jO_)X~qEi7MlLf%8m)`GeH`l)?o+EoJFS$pMOw3Q<cXz(8c@Q?#3^)Z9h7!YTTWeDN{g9a=@$>=n}DG$1aT&M#HdWIx~s|txiWCn(;sja>t8L=|NGeeGGQACnQ%~m0Mo<yo<WViD{+XA-U(egqjQkCVB6lV}+EAE7JX{zyFAc!NY0)7|+;#!13_7C1mL~h0%58?P`qT7GG{rjO`y)|O`YVEB=-qvKRCkW+31ZGe^3tb#3xGF@MrXs`w%?Hpw(qs`B&{R11ic~edH0opjwf)@T=xE)VV?G!<!sJ}J$fS@Yx-WvZ8U$S<4<qn-;Ha!EmBBlS@Q6?u4X7BxXnn*j=(M}Nw|8kHZ){)f2H|Q6TOjl+2H}fQx5U$Z?q7_WHCl<q8|!}ZfUzNzM9zd!Xe{sm$U_1lqykHjjJ7N@h`fqe4d56Gp)a_QWXa`Jsn8%E)qo|NNWG40s=_nGLzo31R6!Y1sRK|4J!c%i+c0LcYPKr{z~FHq>pzu19Br~)CqD>#dz0^D@OQ8EyOisVTY-EhoC4{#fN{VNp!AYNlZvch_GA~%18^>wTD7z<ODvvMIxlsAxu<E=0w=i)lX+6bgO`D_Apj&YmS+Tc$duPBt?+l?aBGET8r)2fVR%+2fI^>iI_;EUJ{<d)^kz2H(0^mniJ=m~|3mAxGMVX*?%vz_7d;p4(;&T-fKJs);!p^GtSBdy0=2XO_)yePokOI=!y!78$Rs}6%!2mE)}20!O`Pr)MhD}KoJ+0Xl(e~dZFiRr(rf#G=67-QfnZCvaK@c4J7IToBV5IYoV&dJ_PLnPhluBOeq)Pd&aO?55XMH4WN3A(?MWanp)*aB$E^mD%yu#k4R-BJxjko6c&Ar>{J6=@Ukz{nsu#^^eF0Z@3LaxYIp4Q8nJP9k?_KHw)wKcM8~I^&Ao5{rdPp8-qk@i+hg<s{xBu{;C;!`aSIn|fCb(_EygA4M-)h{8+U)r2Q9}ngi+#71^1FTKT-y{nxK@69j(+xG|Fe6GKl|v_N%@CS&qVj~uV3v9;tpqi<;;@py!J!7K|kp3|CsH#et-OG`0@7fMh%Q7D@ilV>+Dz|8*!Ih0Z_I`KgDF9d@c&1IwfXg@sd|AvdStIq#$=dCMiPLKw1TY^optax6%fQO$dg-qO4dOl|oX^B67nE5R!DliV{rYvomuH#V&Al!1V>Lmv5mdf8bY^Ok<ThQa<y@qd5W6M_n@_FpttE{1jPMrW_dEY8HizW`$+a8Spok)q$jK#1LwUGAipyHzSd9T2uykmUG5pE;1q|7puV{Y>6c>59Fs1xt5tm(23UeW^Sa^_rkk-6Uoj@;<f$B#y+y~J6n@xd7q`_J?Y|qD}ac&kSR7y_O~aReQ)x^AzFNg(s&z%CIl2GX?v?~>|Wi&`{=N$?7m&{WUqw%y;EQ|<fzqACOpdu_@#}k)nwY8HObIil4Rh5IpKiG5dIwkKeB3(n1)925)Miv8S!d0$1qyOwL}0%s*oVCrnqdX0HR>2L>B@{=O?E~j5U((eQYM!3%%(T43$q*fL!4U&0*S&1+Ng&sZ4v?*&3Z~))F^gtV-Z_)oG1{mrY5>_2nZZe$iIS%@jNy)tY~F^E`dvA8%b>`kC}n7y0i`d%{M&adWnP-;R3R|1fT&Ym!KJ{QKpQ23i-oTm7qtovmnNPdW8M{_ad``p6}8Rx_S{ohnIQZHfM!R(JQK{?fi4>?iMc4gKqA9jB5(UyO@3CaVKuh`HVrUrOOvS1Bf-nuxWOY)OvvAC0nDBOZUJL*UceI?npeaj(6W<JNIue3J^6wT>?zu=)8l*D-#~NIq`+cpxn{BOQVl1I%?Ac|!nQR;?*InZixB4D>UDlM0=c@I2CqtWAdb6Nvl&s=@l-^iI1!ZD7#)fqW!M)FQ2`A&A5SLc3K$f}tWZ1*;92Y9X3eUNr!J4hG2I`wVzNg2Bh#r#aUgj;;BgJmM+yBrGnI6H(<<!5?@~T*w@dbbPGUGG$^cDr3nd&aI?FGF6#9hMc6pO$B1W;xrB@0=MQF`7vfNAuk<Bufk$x+)Evcpu|UxS;%V%v`MoiWkl*kN5Mbqso%Wo@qox%ZqNrzET-H4G$xLNJjo}$is#M2Dz`_=%`EJDS6k{Za?iZgt8!fLkC?w~EAwVBpSsZ07%l6LTjdW47FVsPm|MkxEOJM9R7$D}Ky)Ob;fRsMe2j{!uzkcI1#6PKLV6Btiq!g%8+~}9GKoxWBq9hVymgKNxKiPm@rjJWw2IXQWrIOL&PFi}yctLdBJeUQY#IPZ534Z$HZHeVgwS|b5i<v|j_KGNAR2wd6AD>-ZiE2#&yoYqip-?~5KhuW@(+{((aMwp1eaN%?v7e2cg&d-N!HuTM=0Fs4|Jd6x~++%$Ta{zq9~O=A=3dT8PE@tRiNELFpo%{hCGtw3I;K|LOZcWsu!#>M4HE|3Hh5h-gsl;m^jyX2R^B@U-pRa0Gz3^z(MIu_25g2m{;}|gAnnssuz_R$8@ep6PZl@s-;>*Gw<vt&C0b8M!90~N3d2z$Lz9Ag}R4oidi_vEykh<0G@*NN@caQ%`32CH3m?s6r?er5ihZ~pWu-x@c4lA;fyCM4XT=DwSY4vgewJ#n~IYl>g0W@oiB740d^lei2sW&OrMx!49n4T>A7cM_yOD#%qg2p)&;D=EQKn4I9VqoftW3%h@g+k>EmJ8M>7YSYXT@q#l)I(C91@(Nt09&@{N)fpgxL1HRlTX$5)B_vW>+n?KPW9N^_^__!&cBW6gA^Om$r(KL9w%gY_1KCdCR|tf(`P?nfD;twThYf|ZJ(tzXK@a<e2;OmPCDWd{=|ThNrmP`$gna?l(ocA(@zCk}O7q510Qe{iB2bVr7TiiD2=RkN){YAS)U7AVX3XfwcsiC{g6S4B9(12+h8QvlweqCH5YMNtEpu~uz$reix@2q)uHJmPw4#H|5_l^{bjPM*zR7s3eTWC;g=XkM^TJn6th-L#dKl&C1KNLj&!HIN$*!3A%lV?}_UWD!Z089X^3q+EdIGE<qF@~Xs$n!qjKxzR2bBdN&&5T5p4xRkif(yTcLXk@Li2NfAF1{W(UM#};;i1E*|WRLJGBNiql!(^ycLScsI!X>#e*MiB1=dx(bLc@1KVI(gIMa6`?`zvx~bEvQ+SBe8J4}?45&f<Ug|K6<3odOSto+2P$CR0A-1@DV2m3GuDE;bW-3f1F-LMRG14px#h&=cSlhRm0wg-{sE;K6Hwh%vzpq7XPCxQ#an>pE!<8kK|qQb^Q9UK%5L9}Wy|8!U=9Pd+nHATEHLfu+g35&Tjr!fsS!O=LP^NOi~=VcZEYCV5aQ<H9h;am5y|sq8C>YHJeUcA>}sFvu`?)E-VT1!#?D&y=fFUFi=h8OTdgTLUG-=aU&c=ALDwTB4YhK{za5+@mfQi6Y|p5DWx+%M8=tz$3PDfKNmvV;ut{WHE|Vq!O6lWCFvKw^GtsbjkQtqrp}11TxSwS8Vw{v68#_Y-_&>Tu(b7ar52D&hEz6{$A5ydAmK%IizL>LLcw!Zf{+l?Cq4z&5f<g-<$mH>wh=NyW2ZE037u5P5SZMR9H6I*lJ#Bup;g9P(FKJCZ11fo^wDofI(sf=4vG-h>?tHa}5X^neeg1wZvo#7RXO(p|GfH@Qk#fBvPk70NFt#aKJ?oSUxqBklPaQj}9?K(#8fEXT@^bQx*lOCkW7VWhxzKJaHxF!BfC#r?i*jtyhNE&i9%6{5%tE_Yb@8I^TVzOrCm&5~hSW-^9a43GkAVEm>xa$#Xkfm*0Bj!sM-2tT$9ZP$OczBA!m&P6d<%_<~e>5@W$gFaqwm1xLVYR2W8*8KLCD-+JYgIHsnUigh(R8!`5U8mBm-9U~PbA23gm;Q&n$nzSW*Kg<Kf9WC*ssxbV|rq>R^X{YQ0q3k96WiV~)I^5em+PC*ErR}TzHJ;GVtv{6iF-8V2pKBrfxqR(hNX@Z4=5yLC=bO38wLZ_iemb_+K*7`t*Bl}$p>4u*M<~R|M=N;XNPmD0E=tU4A$T%cRRAXoP~;txY7sk#Rfg0n;6Py#amzBHGg3J)cCk4yTt`ez#I2Pi!gR&o0L7@)5INk52%y9Ph0O9yHVo&4G74yO-PqmRpNvL|5)wkAZVy8#HQ~4Sk=TB+H+knu*_w=oaq<rG9sTzv=>?pd{BZg)XGv>gkgTHbCCj*oPumS!d$sJ68PfiC)7)3_pLC!akybahFYkt(E67{@ta@HeeNVwjZdD>+e`)GNc+F&QMjP$4i>sAXS_SQrBi**5volTG58@QSCGG6`ET|POCFd)i`5F`gPF|w<sG<P38Xdg8biPqxne-L8U-pWizgV*_0BOi0*&v7n@{R;Pa>f8m1JwiL(BXarx_4BoQ9RWQ00&jWsLoL%%fGPovb`rtYXeG3g?HKmnl)F=W+df+1k8bW32~ActpQ&rYxY38bG*?pvxRoYfJZ%XBjkrHILT_PU8(8!ZQ<3kRWauJDXJKMP0Q}~KlR#3pEY0l+j%{rJN>CxD=z$XE7d#ea?}e5&sycRg`Sh9Oz-}<s_$pbfBe5wdELI~e)?Th{?J@sBb_vT{nbf-T6cG30YKZG|Gho=Yp<pI)j{UB{rTb8jlKKI86m2l_3ybcC^Y|nM_uJm5ONdCr7iI5iW>V1Sa|b|(USdgBslzS+oUlshBEx#_0e^-CQ?(isxR8NG`?BQ?YE!$$#J(kf00Kh8GEUi1#;GgXHY#5l|rR*9-!G}?P<%2SFVFbY(!e}OmWQ(P`Z;O+80i{Uex1F*;Au@+{q!?zjCtw%OLu6ab8RKqFj)A;zx-B51LJOUWI8YkG!a%eDUJK&&;A|)S^)F>#Jq2=7pi~Znki9>iH-Z=UOB4739b0>}zy9Dp1oJG)pW|1!i)=8V;lkc8~XHfI63}rY%={0(sP*H*xxlCNF=q_FlEZty5UYoT^B<5Ta(FA_=C<4AVk^=K}|kcf3RDlnMEo&P+0Gm}=qwl8uke#zO=ex#{+je|;%C{p`UA7T=ik{S-qH`GZL>GyT!1+_94f1dRTqO&>R!G(Ljdw?FMO=~R#5x08_~|MfYGpJ}q4eqB>d(;<qc*uGahTwaAgnZN7K@}{qOZ*gr4E%3&lF2A*V{bxTs>g(sfW!zbCuf31i#M{l7-YT}iA%p+Gt^VCN$CaU9_rElCEI7LTpfPuw@6epR%cc<Z?h%<;1un^)Cs$FT^&l+1Fzsyi#6wc`GS<qCWSKb@NQ}97!Bh=YsVnQL@)$VTtUNVZjUzq9)x!I)uNFx3t8(4gCg5bciVOMFE?m))6uqGN*nQ%n-@*8*_2TajLW_R>_lH#d+EkeGwS|wg3*W!QlwncN@{QhI;q^s`pclc0TloQ_$9-`DcD%W80dFq)G&~<&$zrQZIu~Y@FxAK4vA_+hJ~$6l?u>I>A{vm24B2Uy5_D10dfH3(X6<PiLG$i2Ps>8Blry&e5z{jD3EXWj^*7^C|FDTv*PAVp&KCYgpJaY9i1LoZ)~AEy?EB5_jQ4Innv5Fc?#(yI4O#c@T-u!WRStH1_j*wz%FvTHcQ-P1^VIF#$(z^qukDtp4c_p!!yXZgS_GzVTi~m`W9~qAe=+ID;>NHy@YTp(_YdPus)o<rT5e7BaFkTPX$_1fZS%&h-p*&*7C8SzmOp{E+hJ+m)Kq~LO*Eq9OncQl(5b89q_LodwY4&`J1wy~PeOrMX;koItTN=AeVX#-{PQ^8ckub=(Avz+vzVG^Ycn;UJ_ViEQl1evp8t{|nO=0yDVBR@3M*8U5=2T`x@Zy=Z%a;_EshRDt;pVjZh=-sU^?(H7LCl*26&P#?PmS--6%sl)cOkr$*oW&@LyH}x1|WArx=arDOPVI?$J6R*jPz9(s~<MbL<$o;v$RVl~1~sn-7Qm!TU{I_-wGBZ};PFJSP|FLJ|oK0i_s8Z4|{?6AD>p8^bMi(&eTnpJii9)G1O;cWiup6iezf)k&5_zGi?NHc|}~a7Z#-rx21$8B{|VUZoF(DMrFepeMXolQEjM=EYNzms^3CDj26HhdW_ocS(8V$x0$0t$i9P&1tK2(u(X`BWJqQq+DTCFBeUSR@dMnr~W8v(lo&bs)m<=6<y}5;kml7q8MJS65-7;Yb<824W4p{8YEOx*`4r7C;@~SjL;$mD<Zde=o)YLPS}X`4(^(kAK=qr{CDWkeRucT4kfLfE4yK@G%qxD4%DXHO5{=^+b`)#(_ngj2kB;G>uvmc)4ZIvw+9F2{;tln!DRA({)g}X@OhP>@e*^v?6>3(W3|`TD<(Mdfn)`1p~RF7V2p5_yv#(QD$rV8s9FQV@E1c$_c<0dm6CN*HX$7q#FUsw@g#UD1NG$>83Zzv1-ZiFh|w$0ffBtT^9(Bbeqx{dmOXqjSc10bxn)Ejja~2CAB;J}+-Xsp+M8rH75&p>)a>a%8u>;jxhVQ4vC5Tj!=HLi^zRv6;;eii9;+5w32CW2MvdeH`A9yHuT?&<G6a046qa795&uDthbp1Tk&47~3g!gTHc2ouz%)jc7>o#TF)p2|UIwH$9dUt{Bo4@33XCQr0y`GCZe$A_6^u&baP<!8qXKAXM+K{usYN}?g;{*Jrys49+Jn&?+`6{8*(a~A`?@V}=w{gbjBMY(tc!ZQJNt11-wr2#dhgdAh3(i)`;PvAnBH{fPw)N9*(se}-csLfa{uYQU%j$0|Cij{X!i1R?6Icnm3Mcpl&wo|tEV!+rkRJ}3nGe&tVG4Y1`EGRM+Fp~dEo<NCX#q@^d@ANkvn1lR`SQ5Jqf+`<ECn&ja5>SYyn;j<o@OmsRJ`bfRrl%%;Ht$<kndoumTAT7Z)``iOQ$7)%ki)xo(V{ln*<H<U6Cnn@`8P9ZPhz9uBkj5t@`&VYA|Z$Wj2N;6efC2Fx940Tr`jS{C3i<Uj`^S`i=&fV<I}ksy}Oz|H#TCS}G0**V~IR2Jkn1rnf|;>;jKX4C{L$c03PLLDy{&%j)EoIBxaH2N=1%Ij%TCQE9PFSn{WL}EDy3^uO1kd36J&NiuvM0g3pF-F5F14wF+Rn|wAhdjj&MZfDL$fld3Gg6)FC_St-ky}P|%nHq|G(h#BNEHFAlG2l*R%NRl9qoBzF&Hs9(z=R)B>P_ITF}p%_;;f}Fm3bwz4Z=&Wlcw%;8+ONy2>rf)sab!l!>J5Qx=j^kI+aU46jrrHK`g&(qD+{x1Y8rHC_0oNkX5FPFZfW;HU-kzg=>K{#eG;n2L;NPa>x>TWgF|ngLBRBK6eYoC~iN@t%_dTDCl-#5@y~MagZNZ#R_oky{3gtCjmmdP0b^baI7Ug{Oy%V>PlEBjHAESVX*woHb?a5`TA6`QQV8>c-S3%p%GtJPdM2$f@Zu5=xUhQ7M2n%&W|JMNXFFA)x?Ga>SGLNTZ4;+>3a|#?+)PNe#YN!_KAs?Mp+y!li3l8yKGML-!sfmpw(7YOGvx>}Hmf0%=K7`8*3DF{6x*HsdLQ$Y~awv{3~B9*5NhC~0|gcz7>z=p2sQK5M>ZFrA}Pnso=zSZooAcCJd!VA6%mL&<5qs+betP{<n5HA;s!+UNk?!BjBmt20ubBKB-S?7^B8X}72vBhR@cPkJhbdLBWol`Oo~(XIh;4a_P?+K)yok@4m9n3Kzt0D8&U%cbvHx|FVz^pj<K+y}xQD04)UTLrt`+kXfDJNJ_-*EY98FFxBgT+%7&MmpR62gn3x*;8M5+c{GKWdW2PvH9&}S^_pAUloC9;fxBTUdOmj$(a(#vd01<)9Ht|vQi&E&~YphiN|R=ykY);00c>siUdfZsAVAlY-eBq4}t^&>jHRf;#6`G3ou;~L1ui14Uufp4)BvKinIa=jLuw~Pn<+I5aLoYL}R0o!C6cp43aWdlajR#fRCwQVJ<rAadE)a1%nZ}z*nSM!w#%DSlO;m$8!St8#m}_y<-+@a6K&kW^V%U)zU$pGMIV1A}k0CropIIttiaSgCIyxlT1>zih`v~+N@F0S^_wV)XAPTT|XXTM0!NvRyHy6^{8Ny<^Z_M1uH|ymYSjpxVpyNPL#{Fa-38aFu{Q~faRE?u750DaDQNvqCH=J!0=iVZFAYXGI_I^k(2M2ov_=U8tiO_RN5BzA6?tZVHdG^k2(hbT{C0P;I}WA-78^#g5@*$QP}^<m5tr-yv`Dy)RynyNe?3q0GTmLl4dbr4Pqzes?!;x=4s~_!N>y7L<P)^HvmgBtF@Poj=wc<_KVR&`20>Yz*5M|8_$e%Du5sbqEWz-!T&@|w5(iqi4+Dz#4<D<!w*u!EFL+J;m_tUHeB1kLZ_yo)rZ>H-M`YT<yHCW$ad_q)pcSw5~Am}cQ2pc*vjP}&h1>;Isf+o6R7-^c>U7fg@~7TsSN4LV_K;q9U!(Q1We}yS+m=yGY_nI<OndWL~d*tWem8GiBzLph^Z<cL1&BMi*49^MX+BH<+Y4~1XZvYOASPoDoYWgP`D!1G7FBE!Lcyj$ZSFvS+wN|$;(@kmRXFk<EPT}Psi=urs?|s*xn6$Ve2Q`XD0t{d-Eq@Kak?t{^a#5*RV`B!e9EJU(P;g%L=OzE8CkG0&Dg^k}Ng_1LS3CiaVq)iTQ0jb$i-ey#m++g=dK$7sEmho)?1wjLAn$qTgVE;D4?PVMHkoAc7QkY*8L)0}Bb+(MwF=lI^IXb++PRz5rFMsM4F+;QzMWTUoOHydBT(|Jc3Z;%>Tvc=c3U%E;iMzktwwep8@FH?(&iLw#N;tS0;g>#J;TTy4Y$kDMuUANN#i<SYQ63v?K-nhg>sk#ww-R1}Bwq&QYvae#=3kj@7Xe?^8KzJO&`8%|Io*LpY%rf%P^yEe>nkz52I1)GpYDo3t#+C=U$fLn@06AaHRAkwSM!99RzeKknU9NDluVMW5f-@Z%>dlGU6+MRU}lt5&_X<B)P)ydpqF@X`30Z8)03$BfyHxS%LOy0Tj?q2&9TeOol|8=v40hVc`I;n-HHgIzqd<88CM9gGtT967YiT%jptVN_(GI~TWRf-XaM}mA}$;B*IQgYDBc)g%3{am*;xCQlOp5)X3>^&G72rIcv@Br8!Fsn3YMti0$Gk_OSMKYBIK`Y#}26RVg_IEX=hs_4mmyMFZP{QA7^HXvCoN7O9JU{3g-QGJpI_ms{dcub)fE&HYc>9+J7bZu1;hC9l?4<JVH~ZZ4)dOt<+NZsZ<1hN1@Y|z0mrj=Y{?G?E>Dx2M)i4idr1tqUlG^uP7mQBISSujPq%mj`PztvZ)HjhK2wo#^LBx?<!HzMa+N(n6u0q!pQu`f9DR~XD`qG+SS0DHvOnoY>uN!YivqD(^i3hPeV*k$EtQ$|d@2nWxk-N{$#fFXfN!m@4&H(5!#rrKWp{i<1Rg-YPB9U0ZI75WwnUm6z_Q@$;9Sa$xUKc9*0o~`GvFJ{bB^lC)$Uc~C1@3|YV`*}O$QoSLW6X6CmGMvsnBhJ^F|y&+g6gRpt$GxD&u{zgl$%4j&#xx^Hayvmt-H9z3gffoVCKOZ@+yrnut;pdJ68j_wMmlX1j7PJuYySm!bT^O2*lW~9pZ{55SPhS1uj4D4i=l%%@zDoJPE$#PZi&aqwy_YFTNEa)W91@PAxO9$jP8ODr5E=<y|JBaw&pU%!n$ZwasBy&KZlJ*YMJqrQYoiq`N3R>6^xWHDvbn$`N5?$oMjN8>Ht*tkNohsFi|5ATCb1U`+6$Xsba)D<W92&a)hXpcQ_aV4sNEknnIyidsd5b%NQ(J54(dCPCJh$Q=REB2aIqLF~whjL676n;jo}zv|^_x>L@zlPD=TZ9IC=5UDV7g+Na&&M0LgLl(tqQXCJ0NnJTvssv>~5)-!;<F)p-*v_c=S+>P;JdBlXv0gr^TE)X&aOp;ik8X}^&^u(*cZYue+)yw6pzq`S<)7aBFH~ASDnl99kpnqD>aR?``}(_^w3kR^<kbsPJ*vUV>uyswe0|V_YNzOj4O@7>#b>{o9C>HIyE)uRTb+5Y1A|@jE8$FB$HrE<8h%9TpRdj|$^LWe9X2npH8>_wH}Tog#|<}n*wxjKC&E_)tm*q<yO8`1K5ofnj(_)7|MstjGSedjCN$M}T1Vn984NL2LmSMg<SMBS)Cmq!XhS54>}Va&(&(nf*l1z3HK$$r>CxudsS%%e%`K=AEVRO2d}BdiVxbduNxSXEMGdF#&-dN>MePT@@X>FcK;Po&wfioLM=bXJzUEeoUkp+gQ^}1vSuIk0I6MC{|9rF`EZANM0DU=o(rwQS1VTB+LI4P{MmXz{eUS95U?C=|!cHl17gP>epQ;)gkPnxH_lW^#`A|rUA8DB+<uWTVA-l%H3owW=HD)?%WIiZXkh(*%&qH16*~?gPvnpR}^Qh^%Z{ngad@#)>AC0#(8XSMSy{M_W(pdlDZkvjFafi{nmRS?O&gHD&;1M~d_}pwQ2Ck|Sz@yHPJ3`1cCd-p1sWEP=meq2hD_Pr_WJ6IbTkE1bO=1i)&-&D)i=)D<fBgh1e2ZE`Zc$<9;_R2$WfDhXAqBkVR({*X*A`RA@NIlS8L1EaPOnT#i(fWP7ksto(p2haF_qc{!HLn&o>Ou#9LtHKU2&1hnG?B#BrG39R_;XNR45{aE77qWsai@K3;L6(#z&o|twI@Gv6<rv6A}WI6=>--_^C>QHVuQ-nSrnRYKxD)Mpl)RHXwpngzr<8=Q4&NFmlMyHkKTimq{0wxJSAPh6OEHbk3Qit2RO)CMM){<`4nrYe`q@UL5wDoeu~3*w06i{;=0q>)x?;daf8F^Yz3EC4Ge{DMFSxN2?Xbi?qigbyS*40HdwhAumfKFtiak;Mjh}weRREL6QnS;tqJJ(H2cjm&_Gy0tsMd3s%GoD}m0W1NFZt>aAnAQ&;n`9k#7oHT_E}gddK;#lz89cDr)Vr2BSG$MAmJdi$GUqj!cUFF0l2omyt!x=#(cBNZx|v!70;=XeVpxZTa)J?YJ_?9M&;neo%?&b7(}YB%vEPzA39umt9m(Pad+sCmj*=_;~N9g|oRMkXn^Ryw-iWQnIK6szKbadZ+4iZ5jpgxc_bo@?z$MuBAJsv<W(?}J2sVO2XS84TpKb^?=k62!!x(-XD%<KgMK%@PCco86&Lv%m96KP}%IPHcZRdjxo=-Q-y!+j-b+K2tQkF>JF;CDJBy7dxJne^{KA^h@hF3{Ea0;+V=}_9=r%Wy+Z{D@{G%h(g9><H*n{lXa7qm~ysMu98nRFn$^igB2eRgAp>PN5Mcd2DER*QSgz|;$a`1zU&0<X4C>&PWHP=ck0j$`SlfNyN^7p=o{x|6&-2(qEY?pPBot7P*~9+X{I_LQqWP2>9a_Qoco9~k%wJn$UiQ5XM~i>g6&eX0pOFjCn$;fbR^$#IGIM>PHFD2(W|9>)t2)7rghF<FQU*$y@?sFffF>)ACp{GaCsgwujH}ii=a~*E26cACu;%0L;|F698c@dPKGLPV4(N=*t>Y*%b(Dfb5&QUt<_l)OyOFXsxmj!IHS1LVD*tX34CM(9wkyr6rz?&4FbmSEQ*Tr70iNP;47RaS9YqDgrF#*jv<s{MZxd^kShd9P9g+~@gl0wmUD25oMqF8;sr~wGF)d_;Usl!w=r!Z8jq(x$z?C?ZbWio-nzW8y|Wqi_TMEx#27X=x3{;>ZtoOwcbSAqbAcsKu3(k#Pu?zjdr#)j{3JahA|%Xe@DW3sgrqfB4apQBP<StckTqLvgO#cZq;_vb;5@iwV*m`u`Y(D@$HXaw=tAvC#nFajWu$@U@IQ&^UzILc+MZlYE0}Bwj#;v;rK0Cb@(U;Meg4@fK2HxY)*g?I%Ep4@krA}OO2f>x*)i(Q<tcGll%)z*r#Q)wOHX#YQd&ukUZ$qCH_$6S-$4K78=yG<KmMJ({x@&DG5KF7_3d&sg|vNj^1n=&Wt>{tgqXO+^UqryM7Z_Mg^ByCGxKkl)!J=9!G(!(e>I?i51V&ipZt5OhWWwk|FH$O9k^(%VE(1i-ukrN%pIg}k%}9Xxwt3!;j66-?EMioz2BBMKYDLJ@C5Bxm_#*O9NCl}voyew8ptz&1QYBm(UY$%r;>^2I3fyYvB`qXs|hN;vg^?K6;T|(B{jC&)MKfvkR0&VAgME^qH`oxg2;{#WukH3WJWS>v0){yuw0*4bAM3Y6p;bbn2TUUS;<m$6ix@GfE<8gqGs9R!5V6Zvw{R<6!SWj5`_QrZtf@Dp`1yQD2iYK5TsEKJXA)^ELz|s2^CZ<v1Y(DjD1qV>YAw#C+gG0q)kXvo7dO@i-h|j!I4-1b9hBCQz92kZ|(plJ?BZu%#bH?6|%6P*?fRZ;q4NJV~m|b-pQBV4x86XQ~Kd(eGSH>bdE)L)LK>2ETKcE{;oC}_7-+GHv)CJUYfq`h)EBKI9GKy{e-@A``S)3EzXbZHO~u1=X{LNmipoJJ73$~PFJq&mP?$SL#Q}%zU{vYUU_M_xxNkksSHx)0OJNphDDGFK=T}OaF#LBVa8lTia|mztk4{z>VH{^6Ceb}n#hjHVo?hZIzo=!iu)W$<kzbNrcqVl;B^5QpM(aG*la;8MYIbk0DU1til(k&0pDN4*v5^^*S4><%a0Z_?yL(2J)_*avaz$IE8i{${m1Ld-0cW*-!b!~Gy&_{8R@IYSzyfzMf#gmS7NX>g7wt^aw_~w9rTJ!mq<AV=sl}U`^EEewf2Hpo?o<FkzZ1zHZ(ATK<Yo0T5&K%y{xHPglt2W&B_tA^u>TBXcwF(QGk>s=1N>-2^TjvC$Xt6rAEfBvqQ&clWPXV3&L=Dx0J~{8wf|#S>7vsO_~MpB*ft4?T$yMFhtcDF*@FwT5U2~T3)dx64)!!Z{@|4RjB|vP~`tyV@yc_)<sa+tX?^e2$X-g60Ys-_qDYP6o5ZM`gsNr-hzr3f*818lY$T;>!>woktfx(lrSGLQ8?03MzoHG8RwcFH6>#er|CG#l9?Anh$}I=2>20<WkO}Ai!aJ4p|y{Uhp4O6z;V@5dnQ63ew2)SiK>m8O~1Blr1d=Q{<2g8ZXM;u)~6~Qa16p(D77#kR|`bLz+{+I01Up`D2*qU6{Po(djuq>LUO*8V~Jw7ef`BBZ{L~O;aodYw8^C?OsoXpV=`b>fk&=V@yz*98@bO?lBq}p2~^Q2l^oSO39Kr1nlPQMh=)#9#+mUFpi7aSm~}Z7gu_~FM1-V5E$=+;3wZ@ANeIv<t%J^|=~kSyH4<lX$a+NLV&oh+NH~CPDNM>>Tgqi%mJ|`-Qk7OmmB<3#SodeOpW5xEHoR$Lufaa6HyEC(tUtgI!!kjZ2?2T5s!&ZTh5=OCg7846=h+A>B!<UV$k|2)P)X_WPG`dsBJNhG>`AeTi8C)$;<VSJfC_-eq|?y~+OJJgRJ<g>!ZIKV_o5VXE744PtnuQQS*5!J2!jVY+1Q)>D1f|9en4A#TdfAfk1$QIO#VJ>VutT;JTE1Pr$FUjfW6Y^s04utN*g;~ffUtjb5smyD!G_!p=nP_P&{HrC304!-d3q{QpyHI+<2U~8OtVN^~)M~kyLEx5TZ@C1_3!b1y&~<=Lk4p=ZsU~39+ukfsZhIEy38d88)sG@K|<`{l%nj324jw`<l@QgH9iP!uD3(q`es2e~3UaTit=?4+`+aclIWi%NF@kU@o_xKH1#ZDwCaUEQ!Y1g8OsEP15f6E-1s5jm<LI*l*`|S7JHYBmXaM&q?#W3F*@86#Rh&)$V7}8k@f3TysggyW4S_%rG!5=jscu(q&O)A@TWlcFrc6Ut9YFwI@Z?^Gu!R8Ve6=?(7ezO?7AcK0jjg2ji^*M7r(G(tqp@Iis{*9Mqt9wm<Bo550OeBK{V<cs1ZkXh&jDlA(DssoR^ISV9eRYVUA-&vdsWJ3v3h@+ZHL-R&PEcsF(*6k6yfSHouA-o>p>TH1^kcChX)pKJC|Hm>4cmh5+2-VImrqrI4$%FD31`C8iRZyDE)TNwwcxP5JlLKH4<<L%}Iy}jPEeej*UQ({{YI>A#lXPG_A<uou+dZD<iyWlqJUL4&NW<$BUc=B`9Q3$@ahvq#QYzgR7S<=yxb5-z_iqu852Jlu;rCxAYd(`uqqqjxSnRcSR4#AW(+(##cJHGynO!V$FK6<ytTK^yQojP4b?s|*E9N03rVb)v;sbc{6L(~x#$q_*)XD*#jh&jnHr#T@hsjXvzSEls5I{%BTto=db>G#XA7jAnh@o97B(W!2)S*g@BCM=-4az%K7zXFj1yHFQ`4@@(wWmGOut!8jl7Z#{=LI)!Lc=a@acFzGVDtJ;3DHl<sbT$fr;Ak0uUve%29O?i(43Gw(QY8qbO?1{sXChA~9m8FNX{LHHTuZ0S^iRDg7&HDc{b}ZE|GLkd5rvuil~dssW5Ldqvssa{Bp5mft>sclS-Hzr@lu20%1~TG@N-I3&P`SZu`ec`-IMwV-M(|K^$jJxgL7@O$+_9V;1(csDyO3s2V*3()4X$`4s*qq%z63De3P6Roii`%jXk$%mx3uefhBuZ0|SRe1mIa4usa)c4Hn4R3tL@qNH~ccU?jqtsosSX$pws43t;Bn-ldJav3<2mD_2XPDwZc!x_;CRyRZ6^wD+3K@mceqzg_bdoloH34rxIoq6oTZ2E^hb_;8I{)|`lztBBa3GoCmit;~vyWVEhM9AC@PKDqaCyKCQ-3p_dM1#v=W1(t53%$hoVoeG@PmYt~;AVv!;3$952N^=30FLROl_&OEZd;EP{B>7EWQ%QvpZN+U{nt6LvH-2kqWN4f1J|2GKXT9K4TXy^32UnbRKQQ8?iN`Dh8L23rglRT2lgd7lHX9K~YS$!*0h$u{ndKzz0%Z7z=ea^IInNF6fY&b`<?a!gy7c34nOco@aSxZ8H=g(~_eg!y*RSAHw|79s-VN`P%;}G=G+&3;N!Igu9oz;Dbe&irH$Tir;ZmcM5)Dn@%uB2!O&k?>Sg_nm4y5O-3c3J<n?E~Sbivr8Hk8iwgWMW?xLdQT_7-jric7=wf7Y~%9Ja~!UT-S+ZQn1_?3c8ka;v|UFDG52N!M?B*lwoWY2CwHHHGWTKk%RXSBL(TX$Wa`IB&O>z_&W#uiGd1@~%c4H}l_`w|p|ldp#W9(z-5mZsw$0Hx>`Ob7r{LHu8Pm{Coe^_KI<NKV5n4{eU-DecXlK@1twiq%+tvHW9N?v$)w$qi>tW-p(NU`ebf|>3SwZUHs?GhvIwQKcb~KBJ;)ML{tDMOlB+zk-<Qu%@NerVSPH#T*)HHzUN3Z&bU%+Va6FgXQ=j^TY8UtkWb#?%bre)?_$%c6RW#-&4Q4EIYL^&&41$b%idfNYj|T($YHgR#6>yiC@eN6>jfc;Hy5-e-dyy3e{=Qk&-L@pT+!Ku&gMnkip58A(X7<gxcKIx%EeUp;<bf4SR!C?WD0#|?Zbgi^rEWBoLT9bEYcQpE=uM#PzzXxH#L*+HFJSZpy<FZb5CwcmZ#&f?MEXw&wEF@#?FW+-e@)D*1Lg6b{;u!9!-wbR~?d;3FO`bR<KeFb(s|Mf&~VMB531{h#>ri#iT-cY04yS^`5`@_mee~tx%#W5xJvi37qXiw8>;20#{z};;1&jGUn-MlUb4ttqcILR1=gA@`;U_?~eSGS~=0SX5yo9zw3;w(}yEqG|k-iX8oBph=JE+-691z!;&2`!I>5YoY6Y3tX%YG{?}gU@|!6F={Hjc3ATsBW!!15zf+_dufm`AAmz79E_qItS<L)%DrA`>k2c#_IU`ACs{Sut2`b2vE;`}^h*#7Y=0z|L6x_z*Wl^B%VCV7FE>iJ`i}b^OJH1mmYhmuKH7{3yKFy1E<R<Mvh*I^Sx!OspB4MF&6X{$B=>t41drra!Iz+s|*dpX{jlzfR?!o7Mb7i|3v{Ggo?>n6DBc53{_nHs-ZLjuqedw!bvGgayiulbKj<+jk=37qI3QQk$;(O@m%8fynXK2C1RsUm;;2%y+t?4gc4Be*pySLH@e%50A_uFrp_Mr?^;pXVty9WN-L*s1MU;U@|{$=Rf|K&&+`Eb<NGVXO|&#2cKOBjJZ6=6vNO0$wRk1_+XV0MM#awhv5FOA?N8EhC8k41xUYNW(ZSM)m5k@jw&XoiJ>slBzmkNLifED<-$TKb<w-~TL>@zefir=;duuIPWBPUoR5c>qwqn}t|YafX|17Wk?;mwOMBb27KD&<^QL_wxLFy3wzKCl$A*3Nst5CF@%kO|>;r2V%~&>kh(bqLJEpAth;q30*mP)O+a)IA%5LZyA#(s*V4kIdz_zd4JXGhd-XLD!0)nDo>$qL6sQ!?*wYLN-`FzJSQ7GU4;Qqin)01N|MY*mc4Px<s}|KenrWb*DFfCfK-&UJOg|su58s-6qY!7yjmT>16+(gkk*i>N+`=RSU+GJk4zbmho|0r#ZOkb{uI1>o@Mh@`}=9L1PUB82=wH=t53Dx3JaFgfhIZZWEEtQB?n~!=33@XSLHQ1&-y^dn9K-Kl1-6Q6{6ANRF%3Xra}PxQ#w{EFU9qWD3U@*CBZq6S)qKvoY$DMg#o-MQ<c+R0kSCn@)Tj*7=`e|o>{NE=pqRnlVO~>S_K(~AtLikW~za#=A941nxsq#7#qnr5W6smb1YSsMSUK6kH3HRdXFphndKwvJ&q4ZAN16<0lQ}>NBp$kHf}es8{d>J)!*;m)45hH-SW+udT}|K(xo7)A=|FVt2J3j5gnbF=Bz^vfgD1;7oNn@4O3W;QYC%5wtTxC{P#|e;;&h;iAvj&^BcY1uMqlOE~WTJ|8BmpKfJJGyX^NDc1$n6v9O8z+H(@hO~7~@S&S=9CL6+4BMigD)3mjhnMd5A?x*Oy<VG`UvH=;5wWg@^2^4a^=;Uw4dy{KAcTUJsq_RasTaaWPaxdTu$Wb&NKb48QBv`6IyOK#4$u`Z1kcv-A@E5iMq%9n{Iq1e6j%n<N>+jb}uErWI16}~qU?802>I_&^6M(Nbh<Op*#u$XaC}`8K+Mtbyo;fGeX@W{v5Ye$f{R*|gP2{ER403h0A~_$ypjj}~g$&LOqU|aOfb=A2lGLfvtIHk_SBpqGdzkyRI_We3zM_^)SC}YRIAAASW05$9q?Jo_THv;Qvi{G<yM3o@H6`Ucn`PQ4a<w#E@XE#x^45==8k9vEEZKon2Z|lxkW1z9WutL$KTrJ(m)~vB!v<*1`FE~_{j(c;XWQ4$w<EZBcDr`3P?gESNj_-n8ZoEc^H17SMYTGtNUF~%l{B9W9Y&`j8F-^q)P(4)Raqr3jKk|rdPlwCEE}dpIZ)(<&J`nh5)2*1V6VU=G-jP3A1ALkwU9?ZACd7F<&DAGio!ZNu{e;R(4bJ#frVh10g%RMo$^`YovHvRk>4dSvyFjOo)Fcol!cnR6h}Ikr}*pw*BGM<oF7PbVFlZ_>cFc5-prU~oWGK7l+&qbXE7yHC>i%*4PL}u8r+S-C6nKJbv0=V?jN%wn@I1FXxstUNn!dYGM{4=Pe<lJ<p99vPDhzA!<C5^Nek=62v%6~R7Q)bPUUm<fZX7t!rf}_s3e<{)~{n&N?@p7MX)wU#^jZmBq9){x7kUuG1sIRu%gjyXLtMR&i;f_(#9@z-{Y@a<sI7W>t^7jTNmdh|Dl~?le)1-t7md;55G_mfA-7dDow)4wOwSxYgd~lla0Me+6;JS-JtfJJ?i=&KKpx3r^&cCWkv;nkj364Zfsu;d*>#9|IYU0Z(skrZgEVSfyM8KwxOk)_>=Bx!!BtC>^F;ozI9`3uPq;&G|;*1p6w&rBwpL6ucW@0Z3N_9ygca_|75!OpI3vO#3Cx7=E$a?<kWMYGPfn*AE2`oc^0G+%s}%(Fi1hlnjp9WZ>)IH9x#Qu>9FX6sDie9dGaKlQLwP-#2VAt7AZNnA(xebfQm+qOB!m&b@`x#m^Cs12Td^?6Q5Vg;ELSqq)!oX-b9}(iTz4)+s=4DW~Gb>0fOh#+v4}uE5mE&`}uHwb|<L`j_n8RHGXNABx_#;boS?Vwl2T*%7w{WuM{PHuv8j=3~GQseKJ($128HX*@`BfK?$RD2y;wv3`I5^E0S@`@YlCqIWu{@%TfL1a6Y{p#7iYe6{<?%F`hw}SY41ZbAzl|vh~0KOdxJWykr>(SQTS|n2YfyzV*r}X@AVkL>EB!$TN=Pi84uh-$r2~YjSi)#_?5iMu9a&Cy@`7Ae8{1k7LbQ+Y{kCBQ4K+y;Sr4;YjP&A8&s#e{goUmq=(gsp!vRjaiTSvNJQ^_DsB4XHG(n^5;Cx#PwO7Wg+d6yi;TuVFc_H5l<JJKq9iNGGPWJiJ4FZj}!?52f#3#`jSGUL~~ltvaIdTvN#6_7#(I&dyG@h-q&@>Y&}hmuDRF<f|6v&mLYE-F@Z`M(!j_<tN+Q~yY|?1UDtwtWnTljw$LipdOs-K4oKNb`^HY3$nHktCLq>REW236R2Ajuf&oNd*o@)0)qtC#BdgnSiz3SuMG^WyQC}pWe@Wn9(qrz|*>&n1R*@nPi;^9MELNR5XYak%nsbb`<{Se7H6sZXV+@!Dl8lPUCfPPSlQPlKTN9(6@s)8WZ&oI<^4*F;-S0`_rYaSRF;lE|4D-IoL`wLijStWlLLtrG7*(xIl5&uUBBv8&<xnM7Bx0`jk-X()u0lH@s%4fc<b+k$X{u|8w9Uzkk>rM^$$AOw;TYW??+r)lvyRDr6LxM>_Wlq}KYm8N4}{K=2eSz&F(X4|p?YdUWD-;*d6js|ti@M=TZnXkP(W2Na0#SvoYEWqCC^rlzYp~o$0@USUhyePC4FMAEI1NAG;;4<smx0O7{)#5sx@GRIY`Aojk1<tvd$|YXw22_#QX>~{y!cNENF&Omc9ye3Jdxa{>M)EU=}-PwJu$Gc~?j9(ILPw5pW-m%Kz{7TV42`b$N|HzTVd((4SrD&W-K^`hI}!nCAZQA$Nx2hxf;N`8$JC)32$Pzu!je`;6Xezq&IV;oKQ+iq_<BdlYl)(`HtM)0fFy8OtWVXtmI<45_DYgZSCZOlvHu>3-!G(+}+)V%K0W0+~TqV#$y!tl2qY7-y>Yi=W5T#pCAiz^@ICyu(W-dbadTW(84q7s%&dAKJO@H-|I)v`*s2c;fcn@T$2|;ZKLssJFVDesi#I9F_R@$7DZy0Mm-5+rtcbcR011ReBzB68QUX{MWO8{n`&-`}=RqiX+D<H>>=-Gno-Lx^GRbH~VMSs_{<BwI<JS6_B$c4JpHDFIqBYz{?W%+2`m0K7yuZR~r?QHGY_ls;qkIWpqK?z;u=;6><taKu*a8n;}lmxbu8j`PAY?ykNN=n7<1F>TEgPm>N*bE#=9<{?uyk##3FKkBd^Pb1n-bl_WLP#H$Boj>&;OC*Ylm%nmpti*sP~f|5Pa818}h6gyH4>}e|Tw5#;xv1jY*kU#dDzg)8mI;i$LIR7wdo})B!Xp=f_?Cx&Q&TpO3PLI3ToBxFGjX&dz>xqobYkOg@kOpo;8Z@+tIF_jIk>Jlv*oJo8@LOwPt=p<=wO*EM)EtoRDO>AtbNlt(PN;`eb|(5UqXf<F`ic~5UTVhou)cW%Q!#Aqogmqo8p)#~CZTz@(fK5u0?1-XFegh@K{%7a0cy}vYu(nIT5cRy9JJm(Z#uyKadm*ZTJyVyk6^}3Q8&tF9>Dq$;&#zSRy8ROx>$4~Pa7+Wb&4G7BccSgc*G!i6%L82h@b-=Q_3+GRpPt|us_Y0ScNP*@=%cp$%7ObqZvarUeIX|sg0*7DguzVmGoXi1w&_=ikTEpHWd|h!X?k02JkT}DXYMwG%VvmXp$DpTozq~cNV0cQ>Klz7N?OmTw*S1Iw&RwFaVCC+QQN&rvc1YuK`SL<&Jrpj3T+J!Ks*tF4*F%kF>ev9p)h&5h$BH1NcZb7s0ZmQ><qY5XOf<xvLWgI~QIL8(X`3?ZS)O+do-wxND9&)}`M=W;j*yB}y!3@z34$AL3Wm|K>IPclPX=_3hml!#Qb~TL013#@-rJt)}~N0?LDdymfngZ%wQ%`+o7ZKe^xhxYKuZw-D%X5b6`yIfAIT@m6U5p%Dl~+LEFv7PBl;sFJ8Aoh<Gh<}qnH2n3<RYphvV#33ysV<~CLQ`N1f#WVs*bDA&FAqNIu5`?76g&bT}0gx)eYbJq|SWFR_B~Xk+_!Gr=9i(J|S5CFpCPi;Jp7?@O&$I1jMlQSOo4>hOwi=Ad8@P&%y*FtK4Nx87h$`lWGscVf;Wmc9Y2_z&flB~}cBU|(B3R|^3onx=<nA_UIP|ZjR*{X}QCP1V*%KgPE~pZn6&?&yhw-f9Q2}4FT10gzkTo8Lu#gC@QWU<NrwvP!um*i{<$0Gdueo4JV(O9`OB6aFaglhhH3l&&l(r(C<&YFIE;kAh-U&=W%M~6_d^E8V*B-ski<QKUoxSsi#Y(VNw|DT+rawKt=LD!8cxJ0WvT-V;QyW|Be;1H&^A44&p8sL{=BLi@UD$k9!X(Iwg~~V(t9hh-I^<8yW%5b`G58XJ9iu}W3En3~9f~%C9?=PS1pRr=kfc(_K$2_CkN*0VGp{t?qU3-e@YmfNHm_`Nz0r}b=1iBU@U5MlOYH~0IB{a^i?NXje%rOQ{*S?Y@kVE<bY=8MYEwwRCDZGFjLiDeq4Vxqw}taBlfA>f82{R2^K!p4KO_~6wQd8b*LmIV^e4MrxA~3GKmNz9TOIkoHaWz)Gi*5?x}kAz;6a}Q`%TjKyRECC%W%+AoWJ?c#`DloSA_QFNb>y;6MM=7C%@zMe>lH)cWtus_HnnbF#C+*K`a)AY2WFy?$hs*?EJ7-HWQ9{dn(-W^Y$(7cU@_HzQ5dUseCYe{)0CA_brR3mrjV^OpXM{UEHf(1^!{gas9<^Iz9mx6IrncD=8#M&#k6{LLZcnk?K*Lin4k}YLZcsz(7bW$D9+lvXxYCtKkpSajDaCnL4*uOW*?oQz6%+NWEg5%N$+eLMxdPS1M)>G|C;PMriE{d38~HlnzMf(G{(~!W_mq?Sph86LUv^rSj<3jtMNL;*lk;sm3B$uop>n7*f~ZNev%grS$xBm|D4a#RaKBSxQX2w)ZTei3yN^9E7O?o>xgzC2D6BW~I>90R=yAsfhE(+|;;r_X#iw#VlAu5>5p{wZ<lq2rvktWNlxnN(y5DN{weLT#<MJHc{1S;3nBJ-?hp4vb_wxOs2m|RwclPB^;5J)q*Kk3oA@lWrfm)n%&5LBZTa=W4IWMlMoh=PvrO&4g@*}u8hdmYLbI<x~Q6%r9NP(SPaDrX}NQpwsw?_R(Sj^xu&mEVb6IWa10^Pw?)drFiK3uFwczVGL$R>e(I6}7a)5xM;gc#34`qF14q*GC#WccDyGR-l$*+fz%_#C30p(NC?Wu9P2Nq(3oAmdv}p@UQ(4gt!w;e#PSmXgWZ{r>v_6n4v9eiZL<TNN6Fn$f3(r(o4xSnjTZ}IzGF_OoDQ16EeaS1|{kw0U`N68ClpKIa$xO8U;m#CaMPZ_57r-rf<x-pKqb{U6YC(b$fJs|+>!^*UbNe}>WZ}b3O7Yqth4-t8DqvfqF;~vGKb+>xz~{{t{C7C_x5LrUao+o-t}5XVbNUP2F4hNwY~dhJf8bXSNdGOkrbz+XAo?*JPxmSuUks3Yfa$9^MP`Ae5)lXl?GEtPUNvXS3MQj<t{AIRwmi^=u~i=FM`l-$^jJ<Nw~m7+@f4X`KZ<Y|XC`%v;OAYF#FA6wt8I12es_4gKNYTBIID&;-=$OPF^_D?@JCxjL9V%p1|i%E%aj1zcUqDIwR4Ii=`-L!PdXb;A+Q%IE%heJm8)q;eXI@PyEFMps^hpiR?fV8m=?b~xKi#PSQc%q$bp@J-!!u~Cbl`qNk5K0hX<cCj09&D834Idl}D;1bp&O6&4LxD1j4nlEM})jtBO=Hf}w>W=Pj9}evCb1Cy=aT3Q`zPr!j%cR<hY;o}<qxliip=%JUg%AN<FI2Ko%#!;A&hc~<7gil>|;u7}rRO^&RHk{z?5xazFeWF1u_13T8v(G$#En8282_zj@hueCX{W4Tg+j@8_UjBy5d15h(MfMW>d!N)4C(v*-(bH*G1lf#d=C_Gi6k67^@4_c9UmjpSfU^4)MR4Nqlsjb@`M`T3!NXP^x6&(Z%+~ug0!nCS!;cD)RxBtB}s2@Y}>2HwDB<I8niy*BwBo}5>bY5s-r2`H!4)aw$;zVoDHriDMkng^+WFz_frPp6aT&*FICDA%KOxP`ZyX(6F$ot3JZ;JJ^Ti^fw_kOg#2U@fKD3^LEQWIr+2Gb>^=6k?cm$1~E|J=OxCb;B3J-lmqs|90Rs-HZ;X;Vlak8|x=@j#G4I)rGJnGeh*%MjH8kF)iP++K@}WWK0?IU~busjrSON~>tx*H?W!q?upOHIVeWSF0`bbs(S2I;*PzfzhRsYC#o9-dJQs721_Tz5#J3)gt5*BMfIA3s2i5Vmd(`*^UQ=Ye!P5mLala5dj&1@mclW5SbwrS!x9hlC~kD)^y;8X_;jKoH$9hBFMTNSBV6aC4z2BSU}W?RXjPAA}IJMQ=|i0vUg9MwqZFPA}S$N{nDN<x9Go?A{;bCL3Eq`Z-F6-J)|PqRqPILc0ldZ&He9y+UHAO=CYe7dfm+<UHCJSDRl^)I-0^l%p?X-Lbkx)ZIQmsDS{+XN2x5J!l4x8ZYBaSx!1C-hdPP?cLej+qY&UF2R1{g{_)81F+R6b!EfKM8JKKW!u|PFy#M|vUH`=(R@;5p@5i$#>Lz%xdFcqQeXC>H%dhA8WZZ&$FzH}|TT_srLy`&!9N%Kp-W+umStn$R6qJ*JD<u-BAoD^<Rq9Et+vduQb=%VD3FCH1dd4nc+_#6y>$}Hh+z*@YPm7HUYSi@%na}1dN$D18E=vIDfSWZgkPkElFM_J4>`3522M}~>0|fUux>@Y*c++<Wz|_gpe%PMS(v3W$o<H1s@HY6s4LMy`a>CA*^q6^0;3FmaD69vFHprUZCl;l#TKcM(lk*~iC;yzuJZ|yqhg~&5GYH7N(~jCyq5H#5jO@#vOJ_V_Zt+lpOr+_<C67UoskbXx1Wt0Eg#`Hw0*6+KI}?n?9Q8nx0AcfoQReiL9SUogY)~XFlE(ija!1!3!-4g|l9Sa7u&3<2%H9yQ_dvcR;1II2nY?}lzLTEE!>D8Kfdm%tBx;rcPXJMY>c^eJ91kGsSy)8(OlKD+<bYW(MFA=EfDoosue^WGOVgYvq`VfL0BF`xWM-9$i4{O*9>}CK7CNXSU1!i!j#L*ppmwoH6VaU{$x$bHby!(}bVM^9djunE%9h#WG1^SJN>WfH{X;dwh*q2q){)X9CIEYuT{9ic{jAL^SB4s@PbQtnpN$Z0R4cy;fJZoRzw7$zLe~C^U#6Aj!kZLbH@3>osjafNerD%&1etZ3YU>Uh(K~7R<wx_Tmg~wq#G85zE4!~w9zx4^+P%%OL8bMN#si}|#;rZX69g3EJ6&X1-hVT%>-gjCIJ36i#m#S)v6JR6=!<iJLP}z$*r(8iLl5C_7f!mgv(w&Gtzlzl_dr|D$!W)-h2;e&Apzu!i0BYFsqh5hO{k<M$|-w9mTUpEDqvh8OPh*jm8@=$MpJKhTTE@OInDh2Npaa=683Hjr*4cAuPet@QdU8FZ#i?}0~U6&oEPP5Mr4#yB5r!cN&?L+PGx23#6cs6BIhF8ALA(X<IYceRtC7Z=evK+(Y^=Goa@5j2JJ5fIREi*cGH*cH8TUCo-|$?8Ud~iC5ms2ZPRzBzc=fkygdnEgXZn5_ma@o7|ix3H*mKZn$q7J_~o_9y>`EIW8fc?bET<t!-L7y_czz{^kpJSu%spZKn>_;y~@eN^lj{t|GevbAF1n&o<(;=XW9)q-?MzZ>u_x!?fY{EOrs0utx;HKe`M|VMz^^cn{K-*i8S9Ii)KGxdJAK_>iy}fel|>~kGoI0-d@)Jl)BF@{@pfF{U6gV_+JjU-PgbVa+q186vY1Od#;74Ru|*|rj$3;1rts3#^R?Wjb9@j5jrP9f+~Ow@U|mav!JY%phM}zT2`=uyDP~ir{w_mMzdw7<qswe*H`Cd0Q#Z+;J^Z2LG)NI{`2R!@#VV6ozz;ASY<IuYF>k{6%a0~ltmOeGJv^|6UeAGDLDpmUYJNe0J1us>~Z{(>H0qF!D=5i`sez2yYgE__FKPSeXcfDI~i>-AifbOy(HXsjlPsbHo#UpWbs5kHCQi!WP^0o0nomXD%go)HHc7(DQ%16g1Y{dv>p+v=<G>IQIiD;b+~}YBr9_80Arfu_eE5dEg3~IjG}&F<uz$CRl`-*l_M+zGiRYJ#{hYqwNX_7Rgo(VP?RkU&<Wf$Vcy0lL{2QN-~{yP$rtSpVWSTfeS~eMKj{L))h?{9_~cV&3{Q^6MY08}MKVkxm#eE>>tMYSAzHB9R)|_=+8`)3a%yQaw(94tUy2=L^RUG-qqxi{DMK;tbi@4%FEw+oIEZuFfyE2W3p1spxBpW!H0aPv$99=(&5(xPX<qqXe%QQv=FAV80nvXvO0AD=Jn!@pOyAphqcq=t<!_t6{M)AOwHd4vX$pPPOiW1jo68F~Lx1~XGg0E{_T%qP)&K52ME&ov3bz<Yq9^gW!UATTl9e1Ryq6L2*{GyIIwh0Z7?rJJq`c^bB~6@4Rr>z0>hh9i9lfkX-yqZ!!)s+)N{uL^WWmP>qydXWS#s66)FPaBS{a!!^L3(b0u~Eag8K@W=zm!$;OD(WUy)EQH`ydbO&W~g1X)`q6N=!}mlTCkGRKNhA-g;3V<693JT@i{>kZIpHf>+Ebq6znnTMU(e@tC_x9)snfkml7(s#2z&EiA)_JDZ)jDGG{)%+R#q$n&|i+N67sXj9j0kJYh&61QwYo2onMwOfnR4<1APP>PDh{fkOe)3a9=rI?j-L-$*J;EcR#TqoCPr3*|am8DoqmV1(LHZ(rmRNLwb1K$zAc)p-;HSic*H&t>MXwrZhon!wGKRwY6AIFqC5;5FZ|hvGi-%Lng-F}<fPI)u2q{ro<M3}ABcM5!ox>;rw;LUR2!#@f7wIFNR5*V$MY%b-#V@*g<zKW1dm}Au3F$MSD5`CIv@w**Q&O2#Q?09|h=kaq6LOJ>(nJl6WHWi|My*AWBGqEbNir~T3TfGdoEbTAx2A+Cq{I{5gh+}Bz&Q*xh*l;g<wDysGNzoAOdu)G+ljM17HGUX*)C}-<rf35Znti<k^cQem9Z1<{=+DJcy;#UAwvA78TJ`ErL_^L4O<#rInVI&w}<_WUexmm9B$KLX_S?LH)z5~u2ua5tjeW;I>^ecEK!sk3>aJ_>y!jM!I&`>q%X}MZrjsxF^m(-+%c1ul&wJ!OZ8SJ5c!N;oRQyHaESS}3YlWig1c;kXJnzsQZ5WsUzhS#KHQEbA)<Ob7b+%BMoUYDpe%Xs86Dgd@0hg$pjH(;#~LGI5eYy`jF)=e54V-iJZ0p1gd86<3sHI8q6*0g18xC;!<6P;W-T!(KrBKOn4-*Z?sPeh5!ona(Nm(Cn(m!`|FYj2{bKbQ24f0#t}L}MW+qf3kEAT=v|H2=volr&&uh_)l)@|d>v*gQL!y#m<t<5^9jk8V?TM-P%#5s8D(+Cz{>Y~9OeE?%18&-^{P#hlgeoIH==Y@;Nmw0&$4YX>sR0o$ASJ2FzzsZUl?fLSW|DQ7c5OQ>5_qLh6<AhDa-NJU>5yz5JKa8q1rg9K6P*BQ77^qmTJM4|n7~}RYD=^VE88>RC`JvQB!!j{CxLlYs0JexvdV1*)idcoDo!w!Jv9aciF--frdDBz>OyowR50+EgeUV`qc~|i27!5o*-GkW=Ih;m1iZ|p3RZyKSm}^p9UXLq%!2~#@**SZ7waS*U?RUAT_YxHq)sm^`;%1JuQ|q@8k8fz0S}}ui8LKt7Qr&mZvloN0W2~AUzNst?il$Y7paw&@+)Z@#}~lXaB1&6S*nFD<8H*wjqTTW!o~A%o=fLTS|ljwjt~@FAmP<FOLe+k3#3JRFm!HvYxB)zs4z>?&|<EH493E2aw=$Bk%2hCIcQwrniV_}JS8Q3t`rP`6{#+Z9V&U0ZYe5s%TS>z#zn4#1lkd#A&c>inv=pzRY*rgka>?r&;&+|3?Zgs5`bf5CcRXLQQ`kePipOK_o$jp%;w$ZufiJ}Ykw8aV{$ijV`q2orDkKR;m*2Mn0PIg+B3`o$?1saisbGHZbz^tgnnvh@X|X#Vz4z#ZHy0~dH`R_1<wj|wLHW^i#=Nlt$%)mEy1T}6Xi7WJ@RTy8n8Koc}4(c8yMkBq<UgO9(c$JOeRZ}wuoS!N#wGic{m29yx;5G_Y&Io#@fEm#;)m=LrJZR446jACE9<%!Umqmf=6;`6TV=zc@nn4|2%@ZkvwoCjY$Cr$ke3*hU5s?yX*mKT}peGb_yw2?wu!Mg{ee+pq+Z^?n}L{CH-W#T%=`KHs5U80=?#q?afOU%I>K}iimRVjTWdQ(hGX4>!~^-Ju={hi(xCBrwWbseU7b=7`IMQNk{jyQ`<YQuhT!*XO3KBp+n2|Pq)V_y*mB5;jHD#tgrsoThB;E(P<}2G(JZpVQT%h2o#`@#3(#)hb~reCoQ53kZ5()Bx}fq(h5LOYfsW%mG**{SQWR_eOrx@S@G(Ul#+R_oRumWo=K040VZL&gC;UiDMj_Y=IG`7?$8W!y<NWVuYRot3{oLXMOp;KRu59*jqw$jO7c>D&8Y%6f$=eARcf^pkeUS@dDyQcm=X*305{ThCW>+~Z0wxtthRU4&c?;PX>;X*Q)7Nv$;Eqsb69Ek2qeWs=`?L$xEKaLdHGmK3d9lXsaVpo@YLCkC`_FQB35fy^)zUb85^n~i!7B*;f8xqaE?Kb{xVooj0Q_JL`-&SDIuMoobV?d@>6S6CGTmVfdU54nJ5D6AJ3A-DD%9ivXZB{x~%x=)<t2FOa^|;)iS9Jt*M4S>B>lExnSU|jFddc)fQ<I86$JxLhUUeStOgqQjkP3y1_A*FH59XPPYMKXao`xqy7YH`<KetEI=N!FkUM#pr{a;gvgk9Ra(4)=S8h7mE@JXS%5)mHZEXJ?gCpEWR;(m&9w_*=O+!8dD(fX-R2vMw8<=jMzS-P9W(|mp5NXo6xoo7KsbjN9&nZTyO1}wn^)U2ovrj{^Ifu}-9W%@_nn^=c8ta6qDhc4;&Ii4v=5#YTXkY%fQz#?7+N7K9%j}8pa<z7OAe4-s;QTQ@}^KdPhc)OBn%}BD7-*S0x>m_6k0UP7U9+yWer)LW6hFD9Y(oT@LvW##h<|^#da*3z{v9gizJgwG~<SB7y{OMV!{=qE-V6JSd=ZNVaURG`d)in{L$I##^iwEqc(xPxAOWh%rVWG)(J^hf<Tf9aPXLN!EuUTQ6O0{ADOV418$Qorw?t(eJ)lXeUnqA;d{$Gh31M|yPD4H&!X|bNX-GF1G#V4fM{PLV*<~_Su7$%R3I)HwIlKLf}~q)kcqkx5Js_rpKH6nw}+#w8wZ=8R&4#}3^Hs82yz62y%JnnGKXavu`m$GLV_wDYppQuToH^;9zq2!wt&F?YqLW|@sExVM_`aQ$o*odZ8?OcggX|HdK^ES%%#*p0`?T;8S~2H29di!(nTSitKwxf3K&9KY6A4+P7qmi2&zC-Y!3JcciJv#yIr;~?|32ICxmE?&k!L<Z8Qi)Hb|l>B84(A4@06QCTWTE>H*4lA~|GTg*=Q^CVY9gkZ<lb7;&uL(h!*?tnoLm{M}mWq}xk=skNQo*xT*xAewGOz?`S16CrMG?_3C*O)8t4&FdIoK#UXsK6YDhGkgSf<F#Yce;;!q;XQ#*E?nGuR=CAh&^EFHs)-I<fLs7CbhS3b3hIM#PFPSW$by`LNkCdYZ3i^KGG>A~xwi2q1a=q`I8VwXpM0(*`ogjkm3Fm~17!3v8qzu>-2qN@8Cpf5kg2TWcD{aUK}M#QwvYbwFYk65{&$8Xa=mv>KGjuIoLYV6rHu?=50DL8NM*?C4C+Nmmuip@^d%mER+&!LBBG?!nI!#$OCTc0C{Mb6u=51}{MG8=mXn#(F+j44#6S|p5Q1R3@*b?q7>u&O(lJ`Onh>`NfIBH0I#mys<IknGkTE3+X>LRiGOZNDLUkO!Tm=Xm@T?0qhKkIod@6{|QdR3E2w<VMjRo{u58ordk~TE6HdY=!D*pWhz+GFY%(KXz?7(^G(5QT7C!MD%>zNne(0}#_1-1*$Uxo_^-TSw#%5-zP>sZt4YJpwgLqo#c+qHq=iOm!Xrl;sACz@D;;G$v$XqPnr1z?b=Bw&xM=AA}7W{J$z1@+hizR19i!#-$qemOKlJXraHw-zBQrJ8vJ&l0(E%*#yf=m2wt%7_4-s><0|tc^tb7!eFuNUGgaX5}Rlf*-47;#w>E*n1-XeCY7|^-%lJw`@-BM*4ouKa8zvql3!m=ka)Yt&XJlaj(?^cO}cno*Yw&0Zb!>OI@T0K&OTy<3!3N2o)sND!6rsOY>#_f}i$FmN8#~$~ecTRx6bg1;H;G1VBvUngf~1WB!t0oU|HXpIb!xDhM#>s4QJ~>!tThswG57f^6_W{8T0dVO6PQsS4m)FA+CenV*0gtMq(504L|cB}7+=C#b*P)e_c6Qp_ed$(SVzt@0r;fsn!IL=N0%MaIh^fY}9*XTbCzY&6JE=BUOdi70`YZ1D*Zp|B<oP$0npWyAmpU!+MnYS1c8W|f{`6Nr~*rU3O%vK$cn?x?KZjDFmA29KRDx+2bL@8}nUfYIHdsJlCV_@Wt3n!2_A`}wrVagV3EnyZVOWY|T6p3b*FBsIVT-)_&wXkJdgTmA{)|KeU7W7^N@q0HT};`Eop-U_vWb}?nz&GfT2>a-5Z-}>&i&%QWxHnoZmONGO_J(_oi^Rb&_6Z&WUS8Kf--KnYYd;5@vi&%WMHxE3)wnKe-yW?=DKrk=m>;s8vy6U3Cqyk$;(kmK-9bqdHxItF%v<c10>xZZ6428V|L3?=Q?BF-M2Y&FIv*&&A9WGvqSkDxP_bwf(MxI^vLc5hO{hX%p@U^hF*PQ88SDL-B^LinNv7;~(dj!lhFY7{<&M$n2y>Q07;B5cJSvPzkhA+O;e{p8`;_R#a7hbOozwd_^b-!<4aNnN)j(>Ih&hTqzUODvYSLZ+0iSg48eu3YzYa2O^Z)V1CkDoaAL5tpY!Hv$n5q@;|J8XDsf9B9z^Mh|R`O)mx&ids)8}IatdT7j_4CoU^s{vG>X^;+z#f&u0bS;I%8%gtzlkX1_YE2R#178S+;LKwviT0XLl(Lmg8gQgkFDqxvniQE$C6}O#^HK??C2)4OEP=Oa7QCoDSu3@r@}InVa{+fOSo+Cm7|;x5FP|Yf{OPTKeJp6Xx}vjX8-6ku!!^k%t5}STA}G*>2<BUAq8%u1lu_16@>tQ9F~D8LsPf)Moiu-D(DKU{jeU)wrDG-lJAxVl%7Fx9AxU1cXIfG$A;W|yVhYT%P^u~bodyijP=_IN^=ngbK@UD(qW#|PQrd%oc3d}h+tt3vMeedN$2uWc-nevm*ciPKe!P8jLlZNVI(kiUS&NvNPI#D_%G5RaVhF>Hv_2uL6kvN^84w^96G`b3!$@5&!cpIFb%6?`@R$l$iGe}d0Ol4scx8%Uv{@k4WJVJO)Ie}>T%CoUFSroqv{08hKdqLux7*8HE#tZ|@(?tp1xTQ<MMYc~H^Q5UghAbRiMKApmM3ka#5HhGVFW~tiGW#snMAQuFgh0%oI`k{+aTp6O(yfnbFFdxp?JJa;SPD+q$)FF8RKB1d4V1Fc7F&!6NPqu8r}TYOIxKY?-RWFKJ8tt;wXVFYJ3NOE(r_FI;_}C3iG@QFg6fFqoo6#R!&J(MAlJMp*&L};z@GkW+~<6Ctz;(9F#z?XLlCiIMy1*Lpnpdj53-?V(y$XOUBKv4OmL(Dje3XG-UcLGt0EqQ@A`yZ4ZtXQqhhpT{HfNf8o9ax#NZGWsw>oU8kZIO)~}@Q|LgEG0CF%idO0^J5=6kML%=PaZ76e3Us-(Teo*EEH}40@>Ka41$drHltM+IOo`lga)?T4W`VIwPF#}|Zc=1l?<<JDk4#mBfT$edX1B;-N-pyfT~;**FfOR;8}SFoEHYrRWCPGGhBH>fe9`1GiYHx#WybMnq#=w;^D6nY_#n9t!p%lsd{%awzYed5O*-QZyG`g0h`aXIt?lM)yWyV8?)fIR)o^Xu+fKm{l=dtb@4VNH<I|RL?_4U+>QMLzm|Ok%Q23$I3lm?E^;PAPiZQ{SJyjlK?pxNjnSvwG9)84+x^xG?QE1j8=;T9oEb}5LwBK~E>)QONKOpV3iSA8&Otvm<ZVn5(msRZOMlXjlFQ3xAevv$4cR;v5id=R+>-Rc+!cLd@v%$LkZYRHd^TCT<SAFLuPttdtcIR8oL`lBe0R7-EzA>-U^6)#H=(g}vY$DXcCuS-SKO;W>=Ed`6>)adaNsPBwl_VahqoV>pN+C$T!rAI8Qm>EPdZWqxHFHaP*e(V})wm`bpv4oVspkQO8Q5B)ct8?U;B8Dv;Wd+`h~N<LJ(Wo-+72NL>59mYt70xEn~y%od@zjd)lQSOm(*PO*}jin^GL<eNYDmcl%%Z$vR4Y$N1a)<SdQLtQv^_z!V(pL7u-pid^k~iU_N=0UPoTF6*(|PPdXw@$N~-}+sX_i$rO;11F%fcfb4>a63NnM-u-FGn0+xwpZ#(;S?TO)-fJe6hz|xW>i?(x=X>*JIx9c4##71B8Q>J_ttgB~iz(}o+LVbYXHXTAaw;K8Cms3ZkvK~B)FiH0IZ1;YrH<6;5MtJ)*5rtVssO$W3W~w^Ir;>atR%=1ueu1ZP>&ds+8(77bt{HZpS4Rls1%KGAmI+6w+7})s^BP&VL{eNViZYNt|Ji_>cS~HyW`bNtvsM$1i-y~7DbC_b4bZ$OUh_Q;SMk~-eXV-*hS4+WI=L~u5c=pkW7k4)b8whayIhD_Jw{E@j?k(z++#!4*5})v#t)i)$grXAu9&%oV3Cj<ky&#A=_AFs}MOSg-+5z5+N`xTt*=9CAdPHK7d<P&vmHta(i>LBUdAE4V#-y+nBUN^3_dO2fFd!fzQXyx?bs~HEHO6I!Iyv=zn7V>~|Xa>lNFq{m<Q3XQ>N%Y?JB9)wlz_0PD3DW1}l@H+1hdgOJ^wP6zLaK)!T-dm|AA`fuH*lEnIlZ8*O($R}NCPd~>)^!_67qT6`u_LugTUMgO7cX-v!E^8b(sGr``{_w&h?))DX<JEb6nZb^Vfa<hnzE`9sr>^EtZ~c~psN3W9o2>=X=YM+Z-$q&O{btZ^zBgDRUF{UpMhEC$x1~g%4<%aHW{a+U>2KTjnQ7Wz>zbE4Q>?BU;dkAWI1Gm$XE>IEWcJVh@XyCg#I$SthJF%@HDfU{1~OiqLb@v47MZp29594|kW)`MZMTA92H+T1y~)*BCOFSghG$_3Jgw^*m5KMKw%&JI-m^a&%qQd>%#Y_C?5W4)S$YS5e`A-5;=@+izO>u?<zF@5+NE8A-Itos{&{ixVv{ayQtSGQ5ARBO8g|0ua@1C+fsBEcRUMV*ReD7bnFp(Ym<k!f8DpYqH|zyaGu6WvXI5XasLU~rhX0Z@O=#bu`4J{U^Ih5{p)r2EV_~ovvTNejO75%G!7(7>%DHmLlL4!$I^h8+tF%~L%oK9O^Ez>A1d;O%u;z4Ib-8E*&$DF*!v)u1DybF{c=8!3y?|rpn3*m37r;8WG%i!SLaySXC;+F3l?*;D^AkT9KDG|9Fg(b}326wWK$VTt8T7#uCJEHU>A*8wwRnM&oLQ~Cblexb*0HC2Fx>rq+5{*p?Gf<|R2+Yc?1`QQXKBHmNn<uC<w$3iv8s6`*JUB_4V{Za$dBYrX`BK6CmXp?O>1s^EaS<}bKz<mjBgK0W>;5EJE?ibksenJNOq~#TB=S;7X%-MiC|pGsahT)F&ia{MEupNT0@4Z{<)<BzZ|@&_NOXzS69pnz`7WdU{(fvUn6`0YLtwBXsZPC%1~hjVw@#ei)5YS3ZTGxOgZYQJ<1kh0t`NF@mD9DcXuqcyg4e~H<z2aS2quP`ReDXlP=Ph2%ZfmF9RVn@;-vC5OiK*nFAB|;ILd8fPz%1iN}PL8L5Om19jTI+3wuMOPly}d38jqQuSEgr|<N2!2_&*5-z7z02ve|R18(9tTdjnb{sh(k^8QS0HK~$<XU6s%s_iaeI&>gMvAuf>A*ReI?ctB%Kl|Gry)tMvLQW73^0OhrpD>0Gc`}=L|yWgWd1QmyfA@_B2@{)FoxRtw4hkwdzg1`?qC4uG%M`xZl?`8h3wA64nM{whs~~zp<NN3&^EyHy%V4X%ak+Kk+-YB3UE1-do?B@`F9|@E2k4eh~(rcBP-*G*uX`ZNuz_#E<f*DutlVlY|d3{iz!FACIQNjOCRP2D-!d;F+gHNMo7rlUSZB+{({XF&yKZ!{If|E#KgEga{^g;)GIB(jXqgH;5Xntsw8DljnxL236yD5QkV>dD=rj|t{@D?SZJe?AH%H>1pRKe$+6h2uss*+cx1Eq7Y47J-t+EWziH8)o^|49cLqnSK8@U;$hnS<4ZWq;J6%6OGk~J~K{u^$Z+d%bHs9@D+#9XTJ8Af?1M0o@gL+(b<$#;ZOQTu&{&2BZMhnExhrON~3mjk$_}%fR<aftNO+`sjat5j8%9HAG0{5dbO)C;GSVCU&!BU5(14UCL97L%uN>IWjes}#jcxNX1w7u>0=xv7ta=@3~Ipf?CZ@c47j74U1V-a#U*RD5%+uYQC_c&8#@vK`iksjC6t+>c!%T5im)>3U+Wj5e%{F^D}GFW&j*|u7`nW~$aqkUo?qp$3F^Jmh>lG;C7&!y91>sRM#>JNI<Uuq_OsrS1`bYtpEIPUU(aBwg8*3|p+?%-+r>ybCy>52_L9;ey0q1&sUz4u1#|86I?-=4cUT>Iza4r`Zy?j3ZzxY2y;2ccjR?d<M0KfuCje8aTwX6DtVHvyTML=DTL|D=KFT<KQE*t>P0gS2~>UYn(>oM^29Z++T~+r49V)+ZA~$^OlHx2m;+x;yuV$KOBUkNRiv!Hsd3`pSYwHf?9^mhkO^hN!c?*Zs~wVRnW4{rj&q!{+&ZH$44I#?8@Rv;%u(vI{=?T3#LY;b%|o)5Tfiz$YAj#~Y(A`E)lH&*Nvq+<Jd*^L?y1`D~0$_j-r0K~8e!9@{4+-0L@osnciM>1_UJFzvke&F(#J%v+}%oc%W@YWV*?`lJmhvLB5urC)pWMc9q~+hd8&feyj3issYlEEun>yV(2vqL^E--Gcg;@#!zQ*9B99fX&_3hyB)kW;w<~ZEJ|8$YBtArB8fu@l`zRanD=q4xNzQ6@EOs!e&q!y*9?D{+S-<ISdc?v%W30Kd<TI#*HCG&OiIf)eQqVXNvlp1<T@I_w8;0{cdoUUi4TF&#}X{$!muO1~Kua_S?6I*)z_$w}wl5|D^@LI6TZ-eTp9nLRy!-=I_5TxXgAD@7~{kV{~%4JDL-GF|}PDk>i4Bc6aP<9RkZ+&0w*5tvA7)nZ1lUS=Yv?c(-3?^rBny*3@@~(Dl3SF`aHN|HiP1^F<%f=A!D25!n2quXh@T;Z8S6yQ9F~PP%>RH-{zo&UBQ!Z~bgs(N`wM4!;@!*`03QzB7cpUv&)pSKYU}XZG&c!rC52b_4ps2hFKdr{*uSlec@&z4Z0~`L1?y2k*2$ygya~-s#RZuXJg3r2e0bk8hOfy)(-r?TG)fHOhW)`k~>fxAJ)~S^hUh<KXuWSq;<ch>HB-RF~2Ib*Mj@L5ur?LFByyPT_Ms)wg>$pvjzhX+A73KteN5@?q+{wbozFV0c6m|Ng|B<Hj&k=eGsBpnTvHmOE5FV!n8+^P(VunX9suxk`HR<g-h5wI=eU0Hn1FoY!cBv;|mO(qUSW(Q^`!FReS&f%780Orc__qY>wtOQ38?Rct~UMIN~|7GQCmsXxbNE_JLS*Bk{S_lHy**%`-DoT^`8W6Bz==GiMy5`)im$@#(!$zIkJgO{24Oueve)px0eWk>og-rGd~j?cR>WhIWMF&jxTRe4*Th}6*~T*Pw;S?8Q=iK$wXT;$YV<gEwbx77!&R^JJ1j<p{>MYVUD;echzwf7h9QXgYE(~+cNV84Nk1T{HkmsC?G&thvtsHQmUtqwe^V99YLFv+~MCbP)vyx@#9O9f|^9sVLx1n;~H4kX+~!C6kh1));TJd>$)q>V@+c`%NIBa6U<jn&xXaE0mFbp=Pvw!*2vz!}dfH`E)%Y7y0?V3;b@_TnT7k204u)?$s(q{M<x?rYu^Tn$e`Die|vu`tf0!Gm;IUa308wILq`TdZ|9*C2c`WC1CBB>hk?j<EsTImF&xz5(0sO?6Olzg<-wss7}!vPTCL7ryNj<IJnEj2jf64j6Q_FT6Ld;~Om+n9=e(lWMhok$*gH^p57%znR-oX#cT_UDT|!jk&Zs;y(B)BDHQ-O@2#-#QhNNypt~1qN+?9OO>2sf^bZx5cmn$MdeBjyQo|XKift1OnBA`mM$+TTrkFuK9e4E=ojtKc9%i#@wmID2v;8Y$yPcN57dbpsa~9AOe72<BN6S~8bFXWVu%&VPgdrW;oMhus8y*CR1=qgbJU%<%+7gc+j)EH?R%#gsN2UwihD3h>I}uM?3CcvAo&h|l4tY0Bed0K`ky=0Xmq>_xP6dP@6Ve@d#faZ{P^d*dGxVPxSq7PbERAwQ{l>~;B}yVaEay1wJn8_`DI}U#2FD+@sd?S@`~AGA(vv0VdycdH{CxlF;{(}=b17Zki_^}ll9&kDzfn)2(Btesn>E9aevB@QvwG#bby2vuNFGTBW2OQMJ|yqy4By7R>k-dWyqwskPVsqkw69jKFRjz>Q#u|VWnaWc_B+^TZ<=oSq$_EYWeWM$JxjLa>8rS)CEm`i%9sOh|EUIkwB<5rzS?S3DO8d=Qkt;LoE`=or<@f{ZvfOfe15U13(u-Bo7infho^?P*wri25w9i)`6Q?{2cLu6cU)y`baUouU6xFuXEB>t}U<SIi{n`qiLDcSTYD<QNlS<BuGZonFm&UB$Jn%kS<CTNP0oYl!4(Xlvd><R#q{5f%nUjt&Anq2dlYQ2bxe!#B3niSe=b2QAIK*%Npcc46t?J4B8?vW7(^myp~p39hWz*_tk$?m(%Vpem?A7-fIRE-Ydfn<|osFv@ga@&v!cK_m}&fjt;Tmi=X1b(pE^biN68Qnz%y-VCy@;UTb^d0wImu?546+o|R0=mvw4+KAPK~oyv%3Sv#$O10&*C$^v2Bs>rpI!FpO8)mnofhp=Qp*p=dmG_*o7vT{`)vAFedJU8wS+u*+*^cZ_o_hbz?-km4{cV`mQ@4UZS8x9N$qOla&I+J2i*_PlXlUmlS$p57XD(I`(Q9G>AqH-Sy9tO{&7sqKoRFh$u{rG8*%A?jG-=xFe&hxtO_<Db|z#h-U?~O-_tF2A;Tm&831re1aNj_38w&Y7Dke;KV4K`$8nG{euq8D=xe2lbqEk`y#Nsx{!mK}*4Aa!OTMHD7gQu5N$JLM&*ECYaJR(fuH#GEVSxg?|1xY=|xr2FH4+`8T2xjXG?=*sPdaEn~&7;27AlA)AxR1Fpd=K?nzn3huNit%)$t;B_GiFA>L@l@iGg^`-Qu<9;l&#my%Sq)+og(?n^lo{cZXl<jS2j;_ANn0he*-9l?LI?@DfIEU_oFI)T+)DDapITHcDpjcs8R6BKlWC1tE%mX1AqiuZ2v!4QwGPG!sUP^@MBNHq6NUz4HITZV#EpPK0PO+oJ3*az#sey9pqT7=2ID2W!>TyGnm@Z(RDnCz-d<^sxQA2Ko0E>#v9zc^TfNyGJRE3lT<!A1o#$LkwZSVHv;xU38T`>}QCh&viUv!;`osWptUIJN2FeAz4pw{&kCFMamD@x0$rsZ)$!R;w&s%cYZn@Iw`t`8|xI1b6Rj1jy`X`k{FhZHajHXIg>&WUpn;0w)B+&vukV7>AjE>|Q72`7L1XijF0P{&oE0Q@N8HdGS9EP??334f{42g8-RSE@URzzNH1)`DFfSySY&Xx_2I;>oc{*PUg`qNaS$D|g^)aZe`A61V2PFu!wZLD_sY`DOi<B`SvLFjxe4eTo|?+o2_2Y!6>U}5E0e0RNLlXJEjeU({P_t}?LJ(yKAwoOvECyn&?4qW4cLboqwr4Pe8z1|kFyfsvU-97vk{f-eS2^^^Xx;iNHU!93gwe>V(k$it1^ikJoKR@>O$7u3@4y7HRjWKmWRoFsVfVK>5nZ$ks<&vUGPs>7s$tJ|fREzfv_zLqm08}!(DABHH5Sjc#6yk$ZK#@tDAc_5hbMeH;<)1k-=)ue3J9=Imf2M8E4hy^wGV*7>efUMA_WP*2Kc^W#k;|XG+N>Q?$3J^ef#1v&_|LxjOmyYFlvH!C0n30dBQ^J?Y_nDt*v00EIm=}x#dZU-z+5oko)(eTCh|lIU-~fV_E2c}Znq+Dv^pMLH2&nk$q~2I%SFZf*3qSy1t&+{54<*7aH|d-93Ayx2hWGPcNsp^J(gzO6FwgT4<P)g9jKeIN>!Typ%~XOQQu#tIt&11Kx!mF%?$8Y)v0jjeL(U|C$(Z`Y+LC&z+W1ROZz&|$10ZbhMM~$1L*^(Qd$~I8?i-30IS+MDp=+5xH4cCCb}rC^r@m<V_$MPTrAV>l`pE6S*CUBeX=<LpWxGVNJc3wr1vI9*KX1Sv{)lFSAc%i;F2<-994KsqG0%*l~lqBua-G+QVCv$P=I=M<t#c&3Uxek#!4XXQ|lBK4k`Ghz@o(9M$9>cT`z5N8c_H-Yh_v{V}FcZ?HA{c={tP=VPEdunJY~Sc%MKoHx`+vDXSx*S@6{;toDplLB3-d&rS+zlaF}7OqR-|;dDIa5#5{vNm(FaZ<a`==w(MoRDdBhCu(er0WB$pjFh7Eu2>eFtypYXDD5hF`Wk%KlOS{PN+E1LmqoD0EX@0|BK6OCD|f5RYJ~+2h%A%5Aa%6{$*S|(W!$+9!BNGla!Fy`tLjWavJKUwD*VdX^ZwKKaubZHW$aln`H1YfH3e&i#(<8=-JLW9wCZ;4f1P3+bvTR-aBmIzVf{Vb8d&b#N&j2lBhU<V_Imeh-kEgK^%Y_7jWm2b=<Sr2XVN#r241h<Jm|jMdT;27xH|R^-5;XnTVp%V{evwrUF+1OsbKtoUk=~8wV;Wn@3(r0Yu)@cG}FGO1J|{{G;hImwfh*Ore4d||JV#W2E+bAUuk!%Pao~8_QzK(x}dg?ZPAZUojSFQtUIAV)+Hlb=mP9)vMjO4N<q08E<uF=BcnIDD3MLbJnLM2t|8I>ym{JWJ;Yrq4Bpr}_d2DZbLaOiY(5HsUp%!=4wHX+#&GV{Lw0W8IrNU|Frz=@0yl0c%s%bxLH<ws2W#^VezO}n^lNAAVJp5f%l8)4p$oBxeEZm;TU)T}@Q^(Ueen0E=6y~d8mz-EgNOHD)HB-Ss7^~oHs3WW4|pDh<ld>mN|GSB)_Q<)W(4<^S5KxB;DXxtgiJiI!@Nk_e<GIOXNNwv&6>iVg2;a>e|n%Zd(y9c>wk<*cU@ECXJZBOjW$hP=>qoHk8)?eiT2~JplDQ7y?UTk?qDBk8=bF?rPXt$sP`9`1sv~@oiNt`uQ(Y)wIY<<vCJ@`3TR)@90bFGD1#?U=~w}7H_lY8MXV(tp*S1FF^;!7IQ7v%&D)QsD!5&P>7DUm(IDO$s?)Em{1{7BD+6$&0QcciWef5IoQi*dr$)zkk(_JoNJp@`N~b<57U3Bw=UMrj@!qx{^09dD)v?#PJEy(U!owR+fZl>URhgNs0^}830mLYWYB5)-=}Q;Qb+AEaPChY7J5UU=C1*AJ43ax<>BH@^mM~|QkXy%MG~|qutdXQc=2&3bgF53%l#wxI3?ex%24Rx38LyHd8&KO&cwk@Q4$pvx5~*z8TT`n|Ahbv^mKrm>2g|8o_>NgCv?FsptdST~Qq0sSgURYS@9^|mYj0dmvFPNL*R1O7jG(<`Ydrae`UJ2os?xR8g4y8^J{=Q^_C(#W6h!~EWhWmE3FY&#;d14bIy%XsXj>v<K?<^7LHP1f6c<UUk}D<{$bMpxldG#%)NJj94qy$L*OPRnZ<u5@I<Bmi(U@YD4%!)VMM}nLLxt?NBr9njodUeAwn#&&!^J1Lf|AkWYob1CmFR>1BpfqnwFRMuIbTaO1t68W??XhY%uyH@q|PBSLvaIPMEh29y&#I(7!gX80<e9qANsg*zZjLJR{ivW*Ams++Dtw|bUY-3bAd^bs;Rar$;po*ij+8$)H=yrR)NMxn*5cjl5S7lx;FI{4}#SyWq`Zvy{iJr!CMeD()=$$l3IjhB4f7OV2X(hX~JQ6M8z;6j2@jZu2aIe)zXi)Y-P$-x~jSNR}P{9pPl7K0Id}jIH_ot;IO6>`FClR3zw8R$-kEwqrGLmklbueBITnBs2I7}`Yg$CN*dD7w4xdybs<tDZ?hsCFuDmM1;!(Rxk_HT)aOM{pW1^q!^Q=gQ-?aYZ=U*(TX$b*zC}Q}0c(AO+!wdEUZh{z-m05)fQMTS5p!~@XMgR_c6PSo_D*P9Hs&^me)`j&p8D~n*MUX0ciP>dz3m*{++DjEE|#73y-WBu*(t1Z>9~}K_l;hi-**|m<7Pja+hsYhadT)-<l6^#Stffbv(2H0Y}YK<92s|+_~9*`%fo}f^a8NEkp8iK3i(YimyNWBu(G$_|N5Ds{^ZOnYs;%tzTBf(ct_^Y9gV-{<)MwQ1>0<Py*1$J@c#F=GbmM)civnJ8#@hv{-x526i2L)hMn^Ag|4xwKi#Iu))BUK@?6F%*(vt|;r)v{8y7$u=A1R=w;O-F9oL4Q>h)br!OgNZFrL4lFV2x-r!s8HAaE^P#2q%kKHg|M+`G#ef3x{%iQV34BbYF2%&zf?QwUvkTSm~cj*H-$8y9+(^Lp88yR&xF#^Q7PhlOE$?vOtQPSWw9Clj2M5C{q~W*v-(Iq8}!>Er_jCXf_ub3s&L4uOlaA}#((nFtMwiaoc=_}9B#uK#z)42!fJR@{n0LJkTKon+l@B^@UR;(!cbS-}uxFbv^TOi@Z{0oBO>Q8LhM3`msxsN~3<@hi`LHX+v_<+xlVrjc{PC2w6-$cdU-y-lP&B}!q{h~$yPxc4~(B_#0Lb347RjcTJG_Tp<RpH5(>$okpf2}=X^ok%zv%h<@`nL%nz37CsT7jl$d7)?d00aQ?i5+8@IKD}-XC#jpD=m=t(E!h-?B4iRWDpf(D7|SI|t(lh&E5xd*xdG0U<TMqILQo+G&x_H<#pI(_idCT;kAzT#A%j2{Yz!5QSou^~5K>j^C4gGVc5;Y5Ymiol&%A|LR&P56sK1|;bB~HLr<*}_e4M7g@5-lX-*`#MnwjRdzwKXeW+$Bo<T&!Ik3Yg$AHlid0z&uxZ4##2-(8PscU`Hy^K1BGZITin$~)%X?#Uh(Ih$lGv}qZNF&KK}a@v#vFH~B|Qk`Os8ktH!U`m>Ej>tseHrlM8g(~&9an~wHE$zzZ{&9D&Kbo8}b+sNp!}MwfRot!}{oWJU%nnJb)^AA`<*>qPA>u+Em6h|zbI^G3y5iAAP5awkfFEfkNJ~S-Fkh?1{plsP8=NmQx5DU_#7SMH=o-8csy!T(7>kgK4LlWaQ!hLfX%<FSpk6scUJOyE1x^0_+H6Zn4wPc_XI~$T$9BIPes*`%R&AC&{FgiX!#R!VU$v|Lt(iR0aC~=c#iL1m$n9PbcF_BO#)V=;@{K_t*GaQTqIa_BB0QB_GS0a3vDV_WbJ-=TNi7-N7MKP(0dZMB?}cJ6+T?tQ7Qrc7$*uF0-Y6SYpC~n3V(zUyjA0spFd)U}s1hl$g|CfH2}Qk&n`I``aG|uT{`|(pbLo6ZKUwbM_eAwd9F-8rVi6%cV22PT1Ytw0Tnd$}1VsYg&0OaUyiOI@GFyCy@e&dA&0(uPzT{*MQ(;LoW3-FolxILLRunJDT${<rbTybig`5>CG8%It1_P+#fW>W!K0B6O_qK;6OS-Aq3=f$2ciE+)qzy3GfJwaFgtU>t-?uVxi(O*$8(Zb&=Jm}t6TYC|pmV+Pw_d_~HV75%_xJF7+hF_cM!-f}ctivGtmH|afNkKP<w+hM*aGcIW<(f=wSoVUzCth|WS@-l9x<aBA)=xw3|Ig(j%?eulu90+1>hQXbjFf9((l}^6_c%f(j`o`^5<cgyet-3fed&#*6kfE@l&sFzi}$&bz-yY6a?0B^+qDatsz42H6=A)i%!Qi8ENx{ldywbU%UK}_t{;~rG`tJd+T*$6aVV|99~G58#ag6V+oSC^g7S9^JDBejF3O47&qrU^1(R;!mbU85pcEsovk-EcDA=JU^TyhM6{Lh=WFCIwuvwB-}l27?x6oM{(E&NTqr-?-ucP7*M9WsneUxKpgZh@^79sg-0xaDy9(ETA5T4gIePF7hBmF*c~>j?pf{iB4@&MeeW}p^ZhkRvryFzSy@N#kYuls6$9rSX?C9!reXJ(E+xt!oRaI9XRx>;HqyKFB@40IsW2u{M&GJVN{m4Z)y{&Y9X{cttx%l@#X;ETVl0EbTUv4_H?*+d%{n&?hVO{PwNY5n%qp;CVhKFZJK#8P*0V)|jM~ws|lnsE<wi3uDLGl3hj`a^X9s)BZ1AuVv1BtOYaXI-Qz0gt>l|-%pEduaI6f%^IiYp}E1Rg=EwK;LHam$cksXit;X<D#;ptVDv=I%0=r`6&-tL$a(mvwjuO}d3L#TpZ6i2^xMbX<;bW&xQpf*f_`seninarKqF1Q{tj=7{KX8S6J7_tQ~Ls+O$F09qZntp;jzAdBaqxrjN7Qgz`uhs+awltbX;YH_Nj=CF2It<kb-6Y|-1Q!1C;^UbU6!f(D^kYe^|ylCI1HLGs=@YQd@o0k~3%}*};f1fl67VTF2<ZWKVxQCsHnYiMT8?hi=lTB@@m`A<{%rBSQ`U{_;q!JV*04yUn64S!OlB+BUBZmy1fOiWb|7FlhOi4?w(<)Xeql+X;9~Fr=TCyT{GLyo!p$#)_f!mZNEhlL(zp8xRYj#{lj!j?`-T<{mEJ{`^uhmHbxNjIk(8pXx%0#j*!K<u{v|gwHlAz16iX8u~Z`1E1%L40zcL$x<t^;)7qT^oJt>3S_yvHLjMXbL9gy$HjC>ILroD~wM0rhh2a<{FVT3bDgMXYHHDU-~9{WAeuPQVtLzYaj=#~28F`&2imr}Cv!ArV3$K$|wpI!*j_FV}hdE^`@bEFC<zy|ww~GP_p*9E98Gl8ut_Y)vwmx-~6JOegKZ)_lfesF`6AfMgru;1+>K1Nhs+cCX8}o#fIQO5-u8BrJ)Q#huki7(~x3W7bwAf{dxBSt2)B<+H+YrQjlBjt|<YhZ41xf(=_j_nJU*kl(|@yhILa-`{?t1!joG;mwx^A&<Tlrw1s%8Kgtr8-+f8Gqy4H%dtJLZN9gGJ8dgyoBu8BwD0+>c-s^3i2XBfduTk&G9}Y)l4e)5FF<WbxE2r5Vp(L&Q(lk|rFI}&4$CPUQZJOTNM_SlM!)d4qwUwYTl;UcG``h4-L(tp#T6g!1@K)YjuHzn50IqD!P}~Yl&xn2+1+P_xI&wyF@QO0?!d$oRqI==Pwe3E$2-Qq-Vw!>4?;;LC`V#wQaU<B%2e2l=WbLj)R~0W2n2hQQV0>yKLe^t0uJIh)oec>dPMFIB1{Lo^gbG-q55Xlm7nehN6zIwaL)t-gvP9-yj4kj(K91RdqXJPzR9degb+tqGbZ4ZR6CWthxmpJGEJRf*-LYlo}W#W@~*8<q@ntAkNcC3*!SC4-I>^0e%yPLb#hp11V}q)vc7Tn+Ppu>&ku+i%=DS}C(2uQo5SzZWz?I~5`(61mR*U<GHJU2_lN{s0Xoh!h9!7#SWHL>oP$uID`8Wn!D&H%ya!-ZMJiz05}%+yNOu!0GOV*=rbVV8KP~a}q^9dMX2^x``c}a-Tnn3*Xi8qXu*4SRxZ^ZGC}SV-qRFt}^^a-qfhA*&SKt=l`do;}lqtfKsWnK314W3@Mk<ftNUc`H6og<*a*>3U)g8CT?7jA0XKm*z4-KHrHtUc`?j`~+02gpANu?+`;07f!wOM1<Ay~#(TP=6|KL_h{h@2;>`Xo9MPI4*h3I>&x_X#8!a8hE7e71}O1Oa=3HY8)oYX@03Q$C2=OL5eYQ8D4l$!>JP;?Wx!r3WGqR1WWbu+h=(8Bxd(a=_eE*&(H=Sd8+rkj^I<(Xv`IF`I@D27etAAzc0Ji-tIr)PS&(?nB{D@SqcIC{|MyXEEwX&1{AlLQz<3Bt^nB$ni1cTRY{8->m+|3VC$^ng<?*WxmG3h{|#YR+$Y`7{N+|RyblvTh(F#%_C<VG?%Zw5=t+qgpv!ZQ0~ro9S<hR15tf2aS^}O*IRUo;fuMru_vemN}r?>C@f}HD`GKdvH=nRk#nimU^9X+uU=A-6!>-9VT$M$LXf;97HY}>1}7;tCx+;3FxDjGZb*hpdlG!(Jm{K@&%%04POXAYoEqN1WSJ!|8?1jGk6!3gkLY&?ym@2g>z?G0BDIiIq!q8Zt{4j?D&q{vh<2@Uw>pEVr<{>+jnj-#DWokNS?am&9s6B>m_>o0Qxdz<oo%hSc_|^A#LQ~pCKROkK-x})Cszb!9WqEk{?`^Dhr)2pj74skPfkc(f;=i`p8f9kU;E*AURgO5z?uf6QBh$@u+~-7>5u^ec;*SLPG@pSlF9q5eYC|yRcaM3V{yeJi)){}=%hL%I@um=et4>B|Nf}|`)>dD$MpBd)XLQTeXIX_zyJG>r<R?ycIJ}%kLToCF$u5|@zAPaNTbF2;IS41ZBi;cSY{kE2&{@4-wfiC;k8z;t4hJ>1=T@I#3|j<J?CPJ$Hy{j0Z<XgO{Vh%fLRA#;e!MTA(4KN_rRof8nGV}m#pa&HzhwSE7vDbkn{Dma>bg=%<u&I<%JI!n9muGh06*k3Imugk}3=}#ysbYj3Gh=?Q}e_XmdGMLj3Tc_hK)5G2K|XKUBDOc2-mpaecBM^I+xqGXpfIf+<5M#2)ZK$sD<kg1Ak>JgejsiP?`?m!dP$#Y{@bmg<P=u|$G8k(ORB4>$bV!y9&;@$A13!k150Fz9~|n|q;oWqYfgkEyJUqG|_(-9MRlPxK<_U4s4O^kE&JPMc;_lDpbWbmhBS8isXd&v0|9ayNbHVD519{ppkGc2)KtCI@tU!_Frv2b3MRT436c@oIap)Sf6+Bb=qKIpZ9d5ic1DUKQ4<wCHTD;Kf#R)b8dd?IzP_hmDYX{kDF*I)AwLCL5f+9^t6Vzi4yuFPgJd`f+A!_jEIU)Jk?7DRgI~KITH#uEway#gPMJt&B-Jv@<f9QjvKo%LrBArc$_qKxO+vmnXU7DA?ZlHqMU8^Y}ZD^t&x@+W|=3ZX0&H1hHuA@$p8gTc3AFovqnQo#{ReHE_e`<(<CHs~ykB*@es*45(Hl^Qb}~Oy^~?fk$ph&BTjjKsbO2024||0@MlIN%5c<A+xVdt;0WJT5Z$~M49u56WcrOQXc2gaFTnd0`yqr=Ho2K`)}rfT|K;{&O~CbiHb7u%oy@Ktuf(&n?or!tI9QvY_9{0;VX)RpG0Okb^FaX(8$wzF{(6bS;@Jp-WUaZ2;g1-D~bWeEy_BCjpPlf9>oC8O-eWgt6Q>csc;WJz)#KLN`@bVpWxYsEvhhR(`rMd91Rs4cBN?D6#nL<Zh(8|)^;!Lyiwk4h9TN{nJUaNs8nGdO9OtewTo1^gE!pUZr|wA);ONjo=wH+4e4&S1rpDyj^PQ^qy4##;h|}~7^8Kej<OKQdbC!o3S=E)D`~(GBNC+EEvV0r6uPU-0L4k0n0PtXR`<__osXNNRPX4n^Z9}3y>fbwsF-!oT42di(S}i=N!D7&tcjc?+d%MzWR;qzd=ZverojhQ(nS<~9M7(q?Z%Bc+2qyVt;k#>y+x-rLLqI>D2`#a(K43<ex#GNNKp7jCU=7jl$Th@CMpxkQJAgsB0AWfJ19!sn@4iLXm_u#wABgS6MC=h2cNX@4l3H6m(#2k{p|^t>K^DHMq%Z<gXhugoO4u^?v5-6uco#iaH`SW?V<glSy;9{^0!|5dvdQsE!czc0<R6H@9zu(pPwH<m+UE_OBF!fST)g!LOME`pg51zb)_)6CAml@WF>f^YN$XoCQ1)@Zc}C~q$TK*YT60nCEBUPyh0AaWZT(zBgD<}2#9m+5lPz@E{3f)&vi`~%b;ecDGePNr-$EN>!8Lml=&rQ)z6>EtQuJ4p;`5<7Qendaorh(<3D+E$((8N?=M?X_S=0w#b8JFV7R5L^YE7niy!y=y{JTW4}^ZBk(zlU4Ch4wn;HO8;V`w32ciu!i>L))6;r_#!-^N$pQN*o$_!Uk1Nv!SrS*wr9m%Jhb4+&0BJ&Q2f&-*gjx^{%PE~BJ)q%n7S4Yw8dOOLlwRIc6ZnraTulQ)7Ixw0nQ?www7Nek|P02GSm8*n~D>4w~oy#&)uR@9-F3};;Qp1`)BJK2>(I0lrI}b*0wrRFm_?l7H`C<L(^L_r~f_txh$@R8;iumbTJ9M`uZVz1wyw<(;<E}#Tv+hGb8W#JF>HBqj_O0>74~DJ5x27+=JG!0@A>s382xuRT_NSkW{aE9DT^%K_Z?rF=z%@EC>>soXJaC1xD;oEu#tpqWABczDO#i}P9e(-ZTbdUfYI-MA{9_f#lq1F=-bqcwUntE$FL()nM`S|^Mp84_dW=Y^)sU>Zk}5gvyepMVil1^a9Y_kI0$AI*pKknQ<6_Ahq2&g<{nqGwTHVP1MQZ;tF<g8J%Bf!-UjAQj;mWqnnTrzZQ4{G;Z~g21T<YfHjrj$4yt#jU+&N!hLV3NFc>k<h7_({dxLL1;C_{}(2^)lTNl}}FDoObgsx`jms>tfth+vf}-d1A@=70e0M7B7Z1iXE!TUV#L-K<kxW5E#lhaIeYEu{Q#<-7ktf?mm5G&7Y~>I&x7rsy0HKq<D^T3D(fYn$*7<x(zGEbSQSwfcH2o#JtAUmv35wS$=K^OcjK8fnMXdBv1;m6H@#=7|tZDBMZ>po$c<DV$d=oG#=^)b1*9Cdwg|-xS^`fecJuxwJzkUd<1}&gMpFyGwe^+PuE;M%isRP}R%L_9pG%46wF|JKO2}rJd4tDemD~H+C-!EiCkF^jrK?^TSr?yNfEc(IaSIh~V^QY4!YY@0VV`)HR{Bx&pMb(;pSkOD}DOS_{RV-U6om`dlkI*lNZYzE0QIstImiEIWH|w)RI|-~tG_l)6pgy|A;_RxJ%fvC#&l@t)5ee)&T-d=4rWQpM2hora$5wi$DMfd3EMJlT5_k?`Dj{lhbU0+S|SzL0;SlUNrLXwro0td=U)2wrWS#Z+Q~>Q`-vk)^1t0e@B|l9T2!Ob|XgSfP7$esQcOCIqYj8JQK-^(bQnz_*;)05BZ{`EE*|J?=|q7i17**B0sA9C!|a(uU`{qUN2Uil)aMU3*KH&}JPjtHscPk?Nnd%GjW(L?U`2Y2!MXs<hS_xrq5hZfa3h$Ehq8c}oe-av^4a1cv^2q_bV-CZF&RM_BpJNQ6nFy7|^jjkK@8>vT#QjPQjQ=Qnz$L-grLbKdO$-5j}{sTABEHj+A^@qU-(2MW`ofQfJcS+)FM&6yXQ1$Xk%@;iCCIlJ&z{`jz8d1c|R^weVUuY7yquiRPwDI}|KrX8KrkA8_OLB|5ozyk&$AZp1i12piSlbVT<B*_^~VucqgjNmSWP;AzzW@{5yjF+d4jwkID<M~aev$1fi=|aJBJ<Y}5`L(mJytej(Z@+qauLL^d$Xy0oqp7_R8vnOtws{W*QL$eis&u$EoxNQv&i>1tj>hMmm(cuA$92&Kpif5^qA%Lz(3kvmxcm>jtLX4v9U<)K)5K-&AC9oN8)YzVOq`Zx8Xr%4L9aa?m=7H2OpBtUD?zd9$WW6@=b~iB=4f?DB?|=6nzL3J2^%<;j^|2{K)EN#hk~|egD)&r9!rwO1i^!%9$f%cCUNy9(QXzHt0^gFmt3ZVO&cdlCwGE&<8UjkkgB@(F_CTuz)*@I18Q2bsOH9a&myBzED~B3s+yoU@J0yS2zwN6`qRI>D_4$@@Z_oWBRi%__CXaHsJNALkH;T4vai8taFI+vaHMHj6Jr31)oQKuu^Jr5BB7NZ7XAP1y=#wM$CV}cSI!utDSJjq#`{4}b%By>SI?APRb$u8heCshCwc9BurrgA>COTqyDU|~aC-oQl1n9w+BPlH5=9aGV3TM-0{WL2{w1^adEIwrlA<n?l4O}mX5PH_oH!ACuNATPT7l;fxu^y_NP!6>kW)pbwoXRc^o`aBi6}3@4RPa{JX!(%s*+?heukT9A49CTvugfT*D$x)Hd@T1q>XXO2&qYnN-MHQ@-m5n%xYuMI!Iv>(lQIyQYTymJQUt|BbR+qJ{QF(YDF;$$lmUy<J31=J-yJ?NG)1=yCbQN6gyJ#7%7exBv9)LvC&tqZV^@iKRti#YI{<D9$`zLXX?Cg1B{rqvuq6=Hoe-{>BJR}P9@;sqo`n@NQ=xXp>3i%W(L%yqeaF*hG1D1N`N*B!4pj(3o?Oq?H$97e_92Qqe`d6WY~@Hrg^A{f6P#iA1HFEqztTFpcExcG@iscQ@eS}Sryesd&G3eUPWPutK_I;4HA*ZOH6fZRC`JdB8nM@jI9^}N-wM@-d&I&{9-}4ydaK9h~!8({Gr$dx26qB9C3G~*Ol{lZIX{tf@+cV3f2JW-GKWe3L=m*t;vH0m#bOyw2fF|8v}!JSTsr=As-YQBv3GhO&6o+s0sh416tg^<&iOlA5t?o)k_9ew2@gMQ6j)vcSMZO7qSA!_+qsy#dWcg+}v%&Bj%*?$^xq?1nPD}qG2f_7(~O!X8ov;B5HMb8zC6JMCO$)10eB)Y~)=krc_~V)q)#a6v!op-1=IDtN=!<wN=I~sL26YLKLh$M3}?v>rJb113uX5taPid*{i)lT_21GQOi+8^Z9Pqi>jFr@wcf0X;?&5K;PsEvLn5FFXqzi7eAais+}HxK5UP5Z;nv)lfJO<*1MaiVOmwPAB)d;tPWs7S`rD-RA&<t)+aLtcrJUx$*rK;V(@6Nj9uj!$U$PdnR@hT#axaoCV(WZ0X*`Q=o)|~b+shOTA44%CW!kf$f6Sk<6QAshU!mEDIkhRWZzbVGXY>!l~kMq1hoarFz66d<XL#6C(<tnq-0l=CA5l#Yp`UjeKyLl_){yxqBeGu$m}QvXRK7kNnMz*$dS%7j3JgY6BQv_MryBOIq8fi=M0cnN+)7TIspA(LECUn265x?QYb_&!qAw{2;ng%&nqw`sDMT=4I=XxE$-UmzGcb9mSSo($?}9KyNGL+wz7A^4<|d<b{f%iy|ta;PItA_4ePucE(ySmO9zr)y0#mx9bQ2K{t-R;a2S9s%B7G}IXF0GrBAmKUi!|(=3*iT9Il#!((W8Il7HoxeeQhq>afAx;fPD5OM3ot_)$3|%~HG-XP1Q6oVC#5pH6&Eq)Ld3eC3Jmdy5bd9)gsC1y^f9+zCkVUMGS4P21uC6**-O;zPwn2rgiy+JWjYA<t4_2%YLiGA}?51sZXN1Gq3smjEaR$y(8_e$q0==!>*m07Yp*eX_tyJf8rlJz^v%AO=TL+l+M>u88Q1WhzPO({n0K#SgIn+RjvDW$rR*nFA5mxU${u_8^P;L7$Efy1gfh^}#tQ_kx}|PO>=+klJ}%u$2zs#5lle`H0PCtq5k@BuCni>t^qjYnLx0nj@T2KlgcBF9+rD;QT>Abo~C_b#eZM-EV*UZ~oytm{&RfD3^LIejjUxF5p|+dogS`-=vV;{4dRLw05!9Fxfx8i!AQ8@rq0JgC`fXWMLAJD^*iOawg@>QpT!K)b)vAMk=L9y$MgBi=e4>$|5<l9Et)nTWXZ*o9h6KCWu?~?Ms&f-NoVlwetLOAJy#v&vZeY8p)n-hHu{;whDhX8fH&_cyHK^e0TAC4_3T-zDJ|2ah(0(kV9trwAeYJ7kjbTIiO~1Xt$^TbE$>u6j#3HZr5M@s#`t%4bAjfv2)~4Jb(R<(RS58WYEvcyH8>gDkmj%sUl9%M65uCB{?Mk-(B{>O3keW$x9Z9t?~-qg|7kBctO+)C+|CrCEK42Z8mc{(Z6jgUT<%FzfF0oc#*LNaM22!Sq*uw3nzjDL$pdO>lFB4_Em$xfV`>zgj<;Q+NEdAZe8(H7}oqvTN$7jYcAqd*D-PjfBxs+TsT%Qd}y?y!-fluQqMi#9xDvWV8^_c+xE4K>f<+tQm)(Gg1XR1``lQ+^=41T+DrZRi+|sI_j_IW^uM*(^Zxh-KX0emYH{!Pe)YAkW~LMHzR{hP^vAM)y7gb1xwSn#xV^Vva{A`@^@=9T`O?1jg-^iMCs*bY_dxe9Np7gl#Bs75<GQk#GVP8eQ=-R|%G3ZKNV9_srZsKCu*Izneu1yiS8b(OS8ZbuyWl`0ffE{w3Q1R$h^ftzi%w+g$ulg87)`|F7MRxb3L}^@QStD3_NZ4LsXPk>2pgU8nLMs^6ix%i1&}L{Tg}K-4U8G2)(XT@sThE7DM9#WT<;pBpILHXrxUZ%Lem7k!6gf%Zi6@*EYgdkX&@@TsT8@nClX;ZS({UxVZ1f$(0nqK+Wz(_)O!UDMwwzM4C#Rv(xVq8gI=>#WUw`|8jvib3D(xYtl%W|pj<0x4=#R|4hFSy*?ByHEUoh3$ktKQpWrd+$^#Ql0&zyI3<8Nny7H1r+I+>0;+dYjTkw0b)w}h6`{xIne^TCLWN<)co5|?PglfQ(qN1`x7JNZ!_f+P>z!<Wy)f*RBbeN!;s}l<nfB2*3e{Zh;Xydnq_(6i!f%H1Cc&H<XcGPk$JR(FJ)hYn8q8(#~&Jt}TcS=^WQetVXJyi~E=m?qvf#r~#ccCykZ}8-uRJHLWi=}WsWIu+0Aw^9AfhBs7?yQ7;3Q&v)Roi#8w@cNDd-c-Jwe7>n0M2u%xZhq;Vo{&n9CIjith@3~`p=_7_h^IdmQAlm&+WlH+L_^77Z=^9FUqyX))z;I?r(0Ty<E2T>Q*_t0#LYYhAo!Nxb4D=X8gJROY;`i_BsE}Q+X%;>#PcNGnBuKy$18fHjgDGBn7k7Ms1vsMFO=Xgz^v(5S?K$a!z^_rjV~ZB8B7?i;S_9v?P(;7D~;j*P9F6tdq<W>ueK)*V1rg5KmqRsxm9H6hcW`MK<7J#@R~DL@3b69DG|PWw_Ldvmj1BW5KeH;0ikpW^Y4NxtUJbW_!{es>JJ^H3{(2aP?nr!frzcqfPfvn%#mIQVaM&xK1950XXjP3LqL;B_8e}jQjxae<jRxezzZY29O$g#8F#6f(~e{XO$Z|gl4q>pQK<&m6vvtCH1Y61ENEo0HRHlF_J6KI#M%;fT3n?KzjpO`4oc~8xb81Ff1JcRVk4n1O9ccrb_Z!GDgH^8C`y6y`t~5jIX77Llk+t@A2AfA%-EQAegh1dI*-#QCHN3&StG5feG$I5IK_ULo{G>PGW^eU+6eYe=Z9##EMKy^S?W=z0|(4-@aU7dvc`or?H+qdK`@qN3Hx|-Jk=c3L=oXqe2eRHh|p$EQ1F|kmRiCNM0&B=d>D}Am`vz6i2xVFlL3J$6DkZukjw7sM?Ij=MA1Co|eE!B1)-(7lKYo1b!^u03gSf?NS`Z;X>=lyCUmIezZ34I3_|(8uNrZp<`h_(W$<795b;*ha^bpP~pjHOdOUX<2f6V>3or~a1=N_;Yyq?%Mx^q$~uex6ZXWla^?(pE=VuOL;))b0(0GiKqZY~tE&@ND;wM8o4(=2Hs7W~@rHZBp4KV+r#?ilft-xI!ZJ0uS_j};xN(K&nn=Jqlkl+M6(q9QQVq+lfQ5OiB>H>&@(lch0Bc#zh-Xm<YB-P4*J>&=MmuE~)4>&hw;WxNbXcr4qGU>7>KJiw3VEu1?YlFQZ9O2pzpVH0VXMV9H2n`X+#4s1*5El2k@DbE$!KJs!n7iPP#+Om$j7aKGe#L`Pa_m-F>=(SAD9yP3zczp%jM8wH*hXep1fZ0y}e7F&e7if&Z-Ue)#ZMVX`=;}>Z&PiyQrt5%o33@tQN6a17-jsz^Md1kOl#eS_UaG98?*P!JFuhf(x}I<Q6N*3#B0L;HfAk@KOUno0Q5PaHDvErGPY*MXDs#Y%CT)(91P&92u5h;A=#z5ZsTjaMmeV*}yyyw6q9m79hqrA2CS+)}hmZo>bC_cE*q!kRiC@I#)`6K}xUHQEEMQL{=`4f;RF>&Q%J@k~K)SR1V0U_F6FEN}*#bkTR-&s{xC=RY70KEidf#nSo{kE$ZfP_e0t)RBP55CzAl(J`%>kUOjvz?3b|va^zC2Z#)%?5#ecDC-(`dJhP=?PhkIgT~ZQXCMAl&)v@TM0^<a3Vo9{pC(cS=U95mrj+Ewv=SkIAgI7ueE%;hI4?3l1x)Ji>c=p#F&TqKokp#$+kJc-RbVoHi2(eyqZxn%2<EY@4wJx0zXiRgC<QTD*vI4l5r#<H$O9Ot=?N4p|T-&KYg>b;>4B6bXs40i&Xe6Sf!78P?H{JHt3KbP?g9{nB(wQUrx-X2}<P5P~&nZup&>#5_SCN_|@E8)i1s-!Lg9_1R6OWPstO_!P;FdgKTmaiw$@{8AEs~d5idZInYv|rOhh?%&P8eE2eV0H2ASx1}2TUeJlcQ`4a%o~z&Lh>!0}%UY3_?<%)92MptsNHRxS`IvEw4t1Lb{}pr!B<^p3CJu^0K5Le(<caw=c=pNkK?E;8vlGOGtSYBk35RL;#4zWXs7@xtpP?aIEei9TUD_${Tz^r0jFk?e%`IxiHs0&=rle-}>DsUb5lsM`tYeR1}>`a>+~N2qO~n5s!)rHjoMk+GRlPfLIPvNPR1YA&3L*l@qcA&?lA-vt&T1s~seAGnWxT2G5$BPNPQb*2OU3%tVVTcxDPP>WsHh*eRn$#0SkNmw0Dh@$SKF`OPQuf_b+`>%wMG&+Ah3yNhN?+<SSBbvh=9PhEF#y3+oTd$#cXy=%LN?Gl^)Zuwy<`&SQ#EB<{mC=I<o2-A!{clVxe1|6}Fx>w$uF<E?bXki{S{r>dcFPp8d(&@oallVqg@AUrgdk+Tbz<c9_9(<grOk#>j+G?tPlzJJZS;MDkm*`A6*6(h1>*GOJHPyK<zBblNcJI2_f<1e+9}Ro%_dD%veAbW0=QgWQ@?aG4Y2V=XsDOB9Mx^i$-4A{`(||C@9CkX4<BPt!FpOVoMy>w4QycX5avv_}gZ*M?*Ss~;mp*^fgI!kpV9%X9w=jLCxG7X$c_dn@$elWSOeq4G+HfsvL5^1zfe{p!B<w=S){_u8wZSM|)!g~$Fe};oH@hi0|IO))oPUR_*CKKgVVJ{fOQnF*%U);K@(43~HpPE893D0oyYublFzjD0egD=Ivj<x|KwmyHpFJVJIG$FI=i*!A>G|-p7slV;S~_e0#^QPU^u5jUN%+OZ7rAdP9-#Z-JbrpfZs{|=vGf|T{29xaHr5QhwDe-OzIb58uZ<`AY-zCMayi0yX=d?eUs$;P*}+=-u`{<e{n)XhVC{n(Z<0RbSdcU`g`TJt9B5etyOh$}s4PQ<rPfdInD*3_Ar8dE$)Jf;#2nzhZFgQublfSa%n3O+1#YTQU}niUDTs|Bp8~UItEHE~V_YSx8F>vv>Y7M~v{p<dMFFYV3i_i@Rh2Pj-n*T^WoPJmYY@16b&;*;3vxAE5$hUfZ3ZBrjzY3qtX423gVioq1_omos7dsox<=;&-bk=A3ld8RAdp6SJ*6wd_|0yb39s-*cs~vfD|G=b_15a#y7`zG673X#OzVhXO~<meX5j&9z{FTvQQN=y2=@n$f>|)R+%W-Sfjwwn;3+3KK8<{b-r{7f`(uk96bDb2Ik#s@;CZYF(wnfMxaz>Ejm)x=%ox*PfJWKOL6I0Ekvfo~`Y1k2eQE9!Ts=wJs<{;vSkzYOf&&{4+>^DUP&v91Q;O}rxQeQJmPASbmNApJ@2PYJTj`p?si)h|`KT=zc&CFKH|AO!H?^Td{Ry%?iV@s-jkzAI2~lg@7bq)ftyAxlQJT}ntWzYbVZBl|v3yc>oaY!k4peT8td@mp$;m7m<-y+sDd$$BD_#urPkTOr%+|P)RN9(5=Q7OPovSl%2^<pxlhp(~6598vKGCT!%N}@!R9YgpR8wPYE01hE9xX0`n<Qiz&!_vhjk`0tUn{hQMoslUj!0x)1GJUB9yaF&!O`CNr|G}fgU*k}CY}yJzOnH4sT%vg53qT7+ZFG$LTdC6Eg0?XXb9Z?Re!T%@j-vP?+pCqUe_Y=-k|O|m4g~>Tkp-PU(MtL<;d>Z3CBvLqWo9|BdrChDq#f+21}oWlfnJ8_91bEK|`wU#uvd`d8}93zE&^f7)@r{$6C?RUUBq(Gi>m7m;K(#7}b|l<h`9RA*Z*0{K^7e!*5X=Yn&V$?9F~-Z+CmLTR`~pWcTv^-ri2k?N9&HaS<uxok_d%bm{wh@zN_L4gzHW%2_;-rHKIQTuuh*v@Hu$Oezr!6~U8J9#_cWQKnEZ`Aw7&;WQ!Zd%RARbe3vxJ^9rIM0=?+?RkqFEaMDNLJ939vV#;=Y0q2*OtDz2h{#T9CRjwmJk9oa&zxQxM7w&GS`t&(_+hx`NmE@)YswK2Q?0-dlvF{)03uj&0fCpQv1Bln6bUdAoSfFBQ?Z^9RPnG=^+=|H7l}u193r*J)(DXz1`Wu;agNX^QeZr?7^&2l6s<5Zyd@lzw!>tH(CE^#yEnN0O<R-b^?SQ@o61{pw;>*oluXwZ+5T*Qe=qKlj_dB_Rffvb_0|$5Z4ncWW?yjGP7cHN^>dTM^JTZwPTjg}&-ZQc^mhN(H7vD2hoE$LN`<ApPRBucx$HKn@3rj}bk4PDW3m(WuP^igpIR12oh*wp0<1_Y9?*)mpI>ez{yNgRHK5Q2(V=6X4Ozr*F}uZQ`kV<Ew*a-@j-GLrets;EZJb%AuE!CLX+#>qTL+q+%mhdNj^F^2X`KSHx-3}AS&$kIb1EC^EoVA%ge8fYDGA_Kt;m-l%cnQNy+0?P`!o}}bpkK@lf7#P?Tl7t*$;e+UNk=aX^H*(q~8xan4t$<t~npJw_ivH!xY2Y;yq^;z<0tSiAXm1V|Q|GhgL$nP8!^hu+KIBuqC8>zS-}mVsDexnQ*oDe9W4-PM1pw@Z4@WJb!WjGA`mg&6V>QWF_gWvR#Z%dcxDtEl>E%1y_Q1JV=w3-m_0}(hC(bcbV*nd^T0;nu#dNT2u6j%*g?wQv_3FEHNfo(1pnasvuJpU#z0qYeX#Rb#!!j!b#}qtDCOvPN-Rgj<^yn&eoh~=(;$a0vQDtP=Sc&LuNzX`e-vi2$FQN89_!V-u7)-`WDf_q)g$7ldc!b{D^TUZFDu@8<wn#m=jc-YdDqrNsoN5G}Trr$w@{mM_&_Yl#ApyYVwNOC)w6Mt=YKn?1;;C*3d$Tkd)IJC?YaU5=U~IRSVE3CO8$(Dk^I#)f)|*7)kIj=!9#u6*KfGV+tr)I5m7ZAsFVEuiUwkL}3E6OSYO244evaz*zE4ry6q5D$Ak@Yf7Jnj69Y=wLKbLy9TJ&VVSfpfjhwFO$8NeC<5&@c&ho}%L1n_3!K(OV1-p@g{Mkojqr!%h!7mJCDD3BvQPrG(iASMB;SLesxP1uQag;cI|ByyEB82Wcig*a1qg1>cK6@y4~Y6xth?jE-PFHpJdwN8a^#-AkEd*RyQ9BOTi_>C2k61_<lbOm{iw}sE@QUG5>>?A+fr=Ol2wdGwkg+Qkid|-geirRp|%Qa&#~gEU0FM|Xke<$+5pJex~*H}!2m`nGxB9YkS3#VD@DSG3sU>$oc1V?Q88bfMgUZpL5@mYV&p8z({7Vu4Q_Sc=}wczN%WXor3e{@+?5eIFqxD>{-#FTU?sRPCLjd|m&l>O;swYfMEAXP+!wsov6cN{w4c&mWRHN_=+Mh|^qr#<to_UTo$qlk4s)ribC|QSMf_;xKX$@ZDo@Un+8U7`UPAiC^XcF8;88YgacxhGT?a$5e@T&XEfAB4Eix0Drwj-~iV&Cv$nvb}kT_4$f@`PbVn_+O@(f3T?s)m#nPP*!Zlqhz4~AF#`mt6laAK4$WNEz$+$uWBajosIX2E%DF|d_vzIg$R<urx>qPaRPtI@)*Sh0T1inV;riZv`^@go3W8gm3?!6KDavq*xXD*KRgjmapJrN=5Qh~UBiSxKNMx|k=vV)^G{#nL_oZF*jsQ{g&&&VI7%6G7f<&qbu}&%mnif}j;CrK`fZ%~E(43LS`nUK*1AwF$pvkWX~j{QBgJ>+@J;A$Lk)A>)q`7HWU5t4n!*lo)zzjml6{zt?5;JL6B^?f&!rjELA9-FIIfOLvBfmO()1wP9Buu;7o)R{_Bbp!190z`qxp7yD~Vfv)ZO_TwIQj%AC*Pb;I>jjuH?{_RW6e`sF#+m{|IIx&^VtV={!B+|52B-;Xbf!OVXaCo>OQpI#Zgz&tTP%xP+K8vVc=CVS2XdR(iN2bV#rYaDdM7{1lx<qbBo&!&4I_)dQ2&*L$2^o&kE;}y8M*Jzrb6GvD;PjaY13kIm)ct<iE`f;3S%^?kgF-l}LZr%D2~w9-(jI`7lb8$7L0CF*QPoD3vd3g&#UMRZs%REZLlUI532Te6F6~SX+U2#rrs7x~wv|;^zkfVL5e>uar+z3b5d7KhJ~*G4twn=5r|b}gfz3V5xq_T-Naes|l`+f0a`6UNYY;`-jJ#^m=4eDofO@7ffEi9b2u^7aG6;^&XwhaXvm%iV7p)4B>rjmZ`pKjwy}mD)y0#_<o#sSSu1*fGlx=c#KAapJTq_Mdl7p|$w`?7R>-dFN_C8AuicOVcJV^}-ws{ST=$r;M%^DdZ2dAwCf)n^Aok{}>Z~$Eg+JGfTMyY_fl}->MbuajOI;Y6jI&bzt>0zldX|tz}3De(Up6gg7D_S{fD9Bto2@*_OdpWs4bxRh+9PvndA6cN9c^?s{zU<AJDHb!QY_@%LQHO*$$ec)q(J+!ZUNBy2hv#8HY^6z70k&cyYOWDIEqN!<dQ}WRl64_w3UfiQH8HbQ2U-;~b<{c+ECIunt^nl8Q<HWAV8Dea8>0aH5<phmR0-&u9Tn3^Z<z~Rq(|t`Ik*;g%Kl}VRCH#$P0;81WcPBjjVJZAc6uc;R!Nw?)*(q@277YeV#XW5B~1-VR#8<XKowwA706afM4$xjWtCEX*`ae%C;|@?hIF%)rjbU@A<176oB$?@qyk&W_lG3_Qh<2dW+s*aQ;in#ES0V#MT)!*E45e>o7BjoWHE{<kvRg7M7l6c0zZ<ck#peuLLg8H4;;kn#@Vdor`m10+h(h~eYx3C3)ZO>{qeEDV9lkqeycS&ud^wkZOq{$00n4~5zVPF$0j;R3K?-FI-f;gKqr#xMNI(7*BRlMb&ZG47b^;)Rniezf@EuDyY<f$08p}cAQ{m%RcV0?t)Lb>3#4eMQ9|YLwD=oUXd9BMA!#_$CX5!Lo&~2!9><nQLIk$5AXPLqdg(|KSJwiLnyS-ZIR2J^JG4@XlW=3OsbaRZ#)(N_5Dm$3@`T5X%vQ(@9skn2aDve>tvyrWCHQKk;H0Tp@Q?w=6k=ixUX~ci-~o|Qf`*pBBJ-@+Le5tPv7)#Vzo110J$!35GP^4}h82HVUhoAdF9_F9tSflFEgsyxhDe5WPT$%D{$yWP8ayt5Wls^nR+d4@u!1uO2}mIz32O|u;W^R_2ZxikQZgowKO2QbqGouR7ml%KR>89KlfS?DmT^E_#c~E~0L-by2do5)uJZ(of~b}w5OEHq#xUz^R^)d`0wYM2h$qjOVFovkxl!9A^w-8#lRGo*&t2_%X9w}??hNSm=&arQbw2@t5g9VlfOR=UokYSDt2|KiJ@FzXJ?JzCqjtt{BT1l<f$b%PGfO4@t$5^YhT`VJ8j}dIPVfsHiO1}t^wv1bIc?BqI##wkWhYapr7}P$4HmC~Zb*CH0f=ZTL9~4NJex$YP#FL*$?O>q#Osg+6lOm-SRws+k#nvf9DxMis8Iwt##hiH<Z$Jmd6S6DOLo=Q9Ha$nO0>_L5`eR$QeA{?U!p68E5J`-9M-#|nqKQm0rvJA(7<|3agaw9#JWE%0;50gj{DTzN$|~4R`SDMI<hYq`*<undkYa}!!lYx$_}B~k*}myfHI^M4FD;MRaDNEtc)NoSogwnQa4};%e}~yybac#%Hre0p_=aY^nkxL`j|cH?$j4PXoqq0?ooi$)gxlL1afAH!YT5>Rp3|TGLej_nvS}ZqRdq_;+iYk5xkCwhiA$DBO5X$MgRh5Wm?qs?1?%!GFSw{Pc_sOT9HFYDjl&20a|<HV$T`!#pgz=XI?9%2W~%`75msU;BwU5fshzzgO;c}vUr=KGYM1~L8Q+LPzs5qRym#`BLf$tdHkI}(_(@D-iE+;yGGqjhqwqfZ(2yiEH6<e&^#wep__Jpc+ttR94%SM7A-+508^`SG3Vk=cEqzAKX%M?P-F-U3Pt7yp#lp6poL1+hLPZ40<<Rc2&rYZRFnzK9l@{w*u=Aj-1QOnN0OZazrQkh`CObJqD6q}zTJ3b+2@R_<cys2j8`T#i2uB8+y100$$HrJd_M}9KU$+qzXs$3C<>Z+P9QW$V8_^*8wE-KWy~UQBuvt_-H$eP1U(PN4Qa7C6ONo>EJ~FFz$NuU388C6h#^T{<ad_96++3j2CB&*JD$L?1T3DVh+LVYSb*U&HChQJC7r$*lX=#bWXemxD#26&T41~t*~^7@9RN~kXH!;-=LzhB!=t-^=cm`&(rI>N#L?R7Rfh4fMKHw9)+bNJ8adCjW2!k)7S?br2&>f<8Wd|1V2VkkPHbXQ`KbDoNCMoj2nJ)AqpFJ%4S6ZyvVl>vD9P!<$<fTEgt2cO8ET~BNsUXh2EmwRj8_y|<b>r_FltN~ud8)cim0O?Nd?-9-~rc>vd8r@Zu2u~kGG=WMXq2zCB&@BRsg}mP;D{kkej5)5gDX3rozcGu!1gPh9-wdoOSe7MhZh54@V%F7?u%%HN!I{-waI}2i8)vI(I(ST4?st){slOWB?gQ#!;~-ua8R1<dkSJ7kIRaYl(nVog=9jQyk4?@MM<2K&XgaQHwj(vPUd77N0QpdI}{sxtc^mmRKM>(;lWX8j}xe6<E&{3ou_~z_Jxp@Jg23Hi>MU_Z7LKd}_zx*N5(z4$6_M?#OyxA8h1r4{>ww0=8=IpB2u)V#x~l-VAX2ct+pu-e5v|ua^gGyL#?+JvQ@hz4M={&V*ZUv0QScq(v?TG;s(<B~fTD4)dMFGj;W}#VkP1gvSG<Xq%Liq>ir767MKM70YnBfs;gN#^lVdNDCGjRFXltLIGGJi{XbRTEt)Bgp<AiZK#uN2XD{n)ZHG`VMb-hcSrG*%~Hpbxs@KbA*)7yq5~Yt+reQxuqbhpI!4}FFD5)>ys`wqOJ9vwF`Ni5TTS}+hc>cqpI|I4$Jf98^Q-^%6z(}9vL+JJ&|Y|EfEGls(NXVT&WcE8Dne0>uRtO-nBW0<7IeoJsZYA+7dL-HRcFduo?Ns5D{K~y2^HyZJAk!ig&bspt(e|S00SA)RVGxBB7*9vY8T(_qhT*}3efp*<J1AH(unM95CKSq8RnynnUR8%t6l{uNH85w7nD@w$aS$*R^t+Kc4TMD=Cn%i8_Y3{NyuprJPJl)6txmKBbTCrAPKT9tdk5A70@V#s+Bc4<rB?mdo8%lPQ%)sXvvycAWNI%aUaHV?P}1)7#`@TyO0M}C6JF>vIsCiB(MZ8PcaQ&Z^Oo&j_h=Fzu$rUT^m+!bh&!NcMUEn+FeD@NIxBG3nT>myT(ZVX@G1jk**m)J@TSbbVduf=!22U+ovhs^KQSTL<{<t$Elo9xn}q4Q-K)_<`0^ky$nt}`4MnC*)eY8Z^L2x7Dgsa_79rLVcF^G-rJ`B+<b5E2iNzu1aMzNhq-hB-#(3VCyA+?R*r}36XbYCWtF|{>pNE`sncee?4|QH{O9vm@UZvr8{jfqooFJtjp7EjE|UcG`B$!#{j&86{XyTP8`<jLsC&iMm9m{@)L6bn-vnKuo;y-o!BDhg(6nPfy3$S9KD@Gb?edj|ezlqG(vSFh5tg~@F1zRFKK}gSIKZEQ49hUM<4o9^n+fe+NdL^IH+hC+S!%6dX`J>7K({MPnocHd1<lh~L5^$W4ZuH)KUlL4NFPjAOOkX_mGZgSVeU4I-XA&@Z;u*+qb&Tb&2BMA#w8)^BBV3qGlQ|unrfMlS*ZVBgCrK!+B1P>lX^e$NH8Jd+X#Fo=a2Ews#6HR^}(NRz4zRvQA}jA)vvTFlg*S($|@<XmMSsFT`kGP;$y<CTSxw;7y%KZizXvoEK!Z8iDDuZVTY_;aXd+~jI`1Kd6B0V^9+QrDh|#NYjs#$r0WQJVGD2#Q#Qn>u$PO0nbZkAS}Yq7;Y$Zs%l0<qn)L;N{4{NH|1<NZQRH4CTj;g2g!;$L(PPz0r&4lp;AOrRmAw<0&R&X9xJ$=kvUn0!MN=wwIk#u?m|Vcf0r1$`Mez*U(OgsBXWg2;OfEp;Bd<AdJiweI9)T1&1Hdv-Jd+^D27nfXBUylO5J}qMXGR$+?Z~TpKzfg|W_SvO1zOJ7bxS=Qx}$zR`yFt`g+4ke;hP+^Z&LCZXr}vu@9agE1bJ)(>pkb$74l?NtO(i##lSGje2ob80*hBMo-B7DI}RB(21NKZkz7tA`Mu$6blAP?&*8iJC6cuLZ-aZpYjfKw8;bE9x5gDiEUYZD%&74rlazWj;5v>MP4erdTBE@QEcx$xka19_5)tE0MTKs3TZo&#H%wmJ))OE@nV5_z1#{UTEC{Q!<CTh+NlzXRnGyge4Oa>38Z|5V3{}&Eb5ltgc{*Aq1g`;ewlG#ya7AnDNlbuQ0RTu7mXQOdx4@0ubL~^h!c*AY+wICq7x`81JAprMHG>+<eaaityI*+Cn={f0f1Gh18Ngr*(MYl3TxT_Nt^;hN#?v3>lqF^hRDRbfQ9qnopZxCT{0NspcM`E~cRJR_3_`{IIVUn_*r$X!_+!lcW;ao1*(=Ds@F4^qypUE>VP9_Tn6T1iCMkO-SWdp$<b?|^M<<CzTV-S?zCE;+44+R|2Hv+HX!jd)0nZf~1rK|F@^XmVZLwZITXqYHV3M45OLwNx{0HDb`1w$w+(Us^CO?>5EqM~!NJy|@CjRfG08)rwn&iKf(tn+DDZW&anXeBo|4U1%QEZ0o3>mhs>FCR-KAPDCUJ<(Pbo#{Iv%EQ-I>$f$dxNh|UzgQA>D6<fRr>Ci%bKS<rn%wab_=FrO(0zf>y5|L%GLt_N)-yJS_;y72mJAxVUg&Ps8bmKP|1!n+S@a=_G|G~hROr@Y4t*=C^Qv=ijq0Nr1u8vTt{l2b4b+WPX`djQf0=ve3oFiq>IHO%1Y=8w1ovlTBB6JS-s?;7%HswHDd(=I3|PT$ud`}X2X;|A&m70RUdaev=^US2xBxWAM}drgBacWGnF`Z#;M;?+5fNo%iCSi@pSX9`HR_4`-^5)>u$v-^osk_BZW1x`2Bgq-EOb&m(73+=47i!>6_c*r!<40{JY&M>zMhQvx}=U|N3w|-<yxI?Imm^(pvo2Q-Pcg2&O#eDlC~Y7*OZT@xOvukia{o6I4qh@drqR2K~gcP4<jNS}yevX$2<sG}1aRK{wY$+HQ*UcF5CNG#D7%`mdc$+thcVf5GF1MM*kU!eWRyDaBBGp0*reLsm&bWC(=_kG2Bo*QAjEQ;x!L#xnJdoM`9D&$~@;Ye%U~0CTazASC)ob{UpDH>8V2P`eorpsG~%W2nCrv&z%4eKCkU*@RDfIDhNIv97p_Aa75NyguxXQm0_)+}r^@vmhI8`iELRp?8ZX^z_bmnzjvMbjeudA%>5>z4Xy@2la0L)BRaEsE^WBa&X7Y(_p2NMNkb8cnucH9nIB9o10mJquSzJh)$?P8wZa3Rnn)fv^a$;(@cSdP*Rf_ZU5(r0t>xVVR5%z$`6O9cDFM@xlf-!tNnw~;Njsgt!^)sSlGqSJqu6Hv%+0fS&%+Y0dz(l_r$q);Dok9%qm6!2F57md%=rWNh8FRDD_zaT;y~u0H0K`fm(aBv&m|usE3^naSa%7;t}^Goy<`?<6Qz*HYP;`)-!c02{_mk4Z4UGr97GTMn@(dj5amE?6nn+yu6c3RQ{)G$=;9+Hra-4+_3chF1me|hNXfDwa&0qutzm4?RAoF4obx{R<&(He>fU{J{Z>XJ7a^}gPB6`2Q%gKuZ}6@b{l#AIO4_ETP%97YwCC~1d;cL+;d|rq`fovFa4Y_Z2MNtz~|l^-ujKXcN{<DLAxpSNyq92jpg<~KVAOe^WzD3f1h{8R-2wTb~2DRXB<o4o2df6_3n5b_uJ6;_6UY<j7aw(y{z?8J;ku}7x4d1T~&vpsT<SDLuDc#H(&j~0giSL!~XRKXatY;Yb)$n=brE}W~HT-O-u>+&q;(gr<DuU156=6a7pd93JWZU!eR)ah7_IiUfLuCUZ+26v(gX`Nb7jAdx?xN4li9f+}VBv1JlzC=$UaXJ<f;p1+(OT$rc?=FUlpe%@rP}FRd{Gwa@wS*S@heZ=^co|2gCKxn$P)&3XIJSwqinE_;;@pW)1sHRuaVuC3EI+FJXuy(8#&BP-2bonE@NmzHjgE&b@F;X8cofL>Hj#cMPwUL}*+_=2n{LX3ebUo_8JdPRDW$qTJ`%&d}TjX*9#&dk9&5_LXPyGF_gRt<N~*$hGl7@av+2~jBs#cE=qC@oBqiYmrah=LoEuHx3vdGE&MorHe$Hu2F*pvw%D6W)A0jo!*hao<{%yiHQKF7J`{)eY#PFxficm1eobKxXcWk;WHESZEs9w5VOT)~uDUQ~Xc43Z@xYcMPU@<{1b13EW7*lq!h>V^yU!>5LQJa*UkjV_}I!CyRbsivO7Rp;TpqC$dexp`tkIdXYMLr38jEQn$QwCYD?=O@wyB@Dy-YF{CfF;(w$*bzpRFE%LvN(c`g#-+BzD;W{ZUG5e~jlKUhgb;U-*8J3c>v2gWF_bUEkShBxv_aJ|^)!N9<eQUEFj8P-@Ap+Lgnm2mogcAY7k6|Z?tqSTwWy|=vz&w`DMVCDpwwAU^_LOCz?G|#=GX0MS<J`A8$<>XYkkTC|YE@HS_yqP|4HfuCnMv#48S-`Jz6Pq~NXn6=Gj+Kk5Z1`D*?5v@eHl(p5;tbl|L(K{{>Pa<hMOIx|5+<dfA6vKMMwnYJt;sTvhi38Rld@MsGLUt0&re(SO!#}Atp~?@NXr!=0(M@sAa|dI;9Z4P7+WtN0kV4WNnxRXhGtL+DIm-hmhmbQ^lBKa(a*Cj^S2{E5@tN82RwjBmqBdFWMJP5-@5y_>->MtU|ULxP&RAF{>gTKU2XQ!7FvOXYH8xxo~MhjYt;;Ynh+8W3So9qmE&&04W$;Hx=73M4b#M<04lAxPXglgTj)cT0^WO1R%kTtaOlKV?IUs<h|i6?f&de$h#XqnlM6DW~=k0Ca;pA?#xs@9-FEhkesO~jjrS-q7}d%HPAX%v@Efdl%83!tsaWcA@mKG=o$<3Zk@>j$=NIJcuj!P+*k=%Rf|A0MCP+Tnn)56hM6dNPRs`K!dM*o^24j8Jz9J@Y+s`b-w8jQ>|EPvIBU9%#jc3HyWIBb$pZW<c{g0br>aW_k^zMN@X}=G>iz`JYX>0o7|cCATYTv|7n_TT3f;MC4obUA)ySFi#QA!Aa$wC@ub#WS_cHY!bPWfW>SVi|zZ`y44i5`;A8^ht39mV8Er0*%#C4Gi0_rx>&|a;RG-=OGKm8+e)fo`0g6jZO5QtWpnuemRj0xpAH8cW-B^4?be^3@&LR6<oVbI!Vc6Iz~yMsE4^mL1*+xzUR5k>=^uJrw8w-5RDV8iqArq`0McvKodILMMSf;LuC!4mTT9aLn)Bvuy(7ETf*MGH-Po+*%cPfY176`&x}CgueaswF$M2$B_z0!X5=uMC*sUCN2{JV-rRfn-r9u;7MHN}@Zdj_!RrejSQ19`t+j(|zyTi^V8&4X0Z}`CNPW*ME0T)N1R5f`Om56SkYP4~I?qX$c74<zL1#lJ4mE=1kSdiB~@gcl1>wP1}0jod(=|epzMe*mt@=b8bCeA{|~+ufw_atJ}>&({c0F_98#RkMN2QS`yV>KW*FpqJ66$cVgea>^bDjTi>4k*3z4Pi1(d&x4A+X{uvd*WEVma&O4V>;VHM$>LrlQsY{@fS<vxXDc*70R>{*ElibA25EC<BQwY=3`K|@&vo0?bW-Ko>o@p^#_5ze@?3%Kdxp}`Z8T)A0os;)=U^h>gO!pQlnI5-1#xfGGEj56zWngSP7A0~3_KZ9f9$2Q_84M^4c$f3ulm1ssm2_>H_|)YG_Xp{$5jYJ%YVj}QQhR$i2faC#lRX?xL_c}HTcR{d^d*Mooo>Ka9LRMeKUd(Oy*E;FlCPmXJilDvAW0qpg9hqutrkn=Dre)4=Bp>P=S~6#V#*>7*qQ1HK2trxH(EVmCCU7O-K2nQDmd-ki)@LoJ33?MNK3(CTIT8l_>e`C(%x!JRchv4Ly6Rs65Vc#5P$xZD)FtcQR=`drU@tyjXYy1B~OPD5h$I6?wv}Bhv=&!dpeOoj{pdhw(KmOtF+2cz&MP0r$Nqj0#jun0<b2{vto!9j@)N-uCb)Z7(i|Us{`><mOay3w!dkY@vuMp-h&PA&pN=eDo6&4YzYB6HI_ocKvT;(Gfqe{-w=@+7Xvx`ATZ%eFKCzU86WyP?SAIHu}GvDLfq|5KLlN=9-aH3e2&^X#RWOUB*ttNflMo06hh-QMn%qeu~8VWLSYu9;B=iR00-}_?L_Hw#O-k)9?S@~{CohP-8kJJ!NvRi4)A=zWHbAT&o@H_%|OoD-i9$Tb#M8H*^llYOEc3R^WjF&WAp*kiW@_1K`cgPsLI3@q=;6q0(^y>0a-yP3Qhy2)R4i+v{BaQ%+3OO)OJ9R)O%rP!6Q(X7Hhy`ck+8@Ft2^knby8PJhhJo<*GNvQp&lKqNP5S#pyGxjDK7h=>VHrNl{$`QXVNq3XSX_sl!-UYPbwcb7Z^P$`o<!Nwr$#qAVAmA8pvXGi?at*BjP@jSirSyb?9=g&<dONwJD~zzeYcOnu+Q2ri9}sRSzu3Bx9OC@v;{;;A0rynan|g&a|+gFOjy^HoBa8nbhTW|@u1jkJj-=?pJ=K@Gcf8U;L}sxsy*ZOC|{R7lbi*a+dNNUN(rSOa1NHqVMwhk_)>6fLB1!c#j4_%IkS@#D|UQh7MYTZ|6b?`(LTcuX8Yrb!wpg<BPYr4Ymv$dU$x!ivz0G?5|$b*e7MwvRt#vT+Gqo=T$pX6LfKS@Pk*^g;}!C@q63m^mS7FKd-x8&P@VwY0H41+Y$K4>0M2kYo!K&7yka>KmKpop(or^KQT5opIB4IQzf9`2he#i!xLX9$A=_8QIH6A}%ZjV^j$|8*LRwa6ste4x{Bt3RKvp;*teP3NM#1mYXh|i*PSZD+aW8wd@~WZ%e!Bcw%pNZ=ckGav`<VwitWvpyU8^`S4P{b}4U9jyo7xaoTb1TuAL%$L<6{3(NcZ`M;)*n;g6tt{sG9mr-XwJznjTJl^m|kf}PEzBm&skF>lHgn5n#U_fyqlcHaaBzu<`sH3D)Ts$0UtW9T$^OD6zE8HKXL-x#|UlmC;gdhQAM6#yDXt1nYH3=Y;S_{Y)D@pPlXHQW%tOex935GqAl2_S1AGh}|ADmYt&k@tX*3}u^t^uSToF7%Y4)W!LbJupy?d@Mae|ax#AIxZNk*bJbk~J4Em)#tiE8(Eww)yT$-|7v$xLz--by|V3_Ck*n82b!#x6~k8GBY^OPvzLWTwdPCEz!?XV+%7o6)Sxf3Nb&DGElWt>1oLPk0Tc{Y9zFnmXfUAX}3N&skHWf=r=nRo4s?Au0sEsQ|T)FVxfiO7b7Ma`yzfmp3Af(mCk+sTi>}nY`Z=f8)&-zvseH0-doFOIOBUwx%O0cYq*Goe&S8=-y#qTNUPB!E?8C>DWD>`bD2!2N?H;+6sq+`6~RPC8%Kz{2*pJyQ_lqdRb$Jrb8Jd~J=lqAow`@i-`CrLelVVW+#Q<K-|3s8+GqUKC&MbHJ>PKNUhU&j>>v!1Y4MSqWtAsWS;GV&(Tv3FNS=ciQh>3^M6z-Op{T}?3AwK36g5P5s64Q%&7`ky_cz(@2zBlJZ;qq)&QNLi+UoAj&9XxZT69DeGS&DCMU0n}d%&uwXp^8YVi}o<X}}HPeX>HC0<ar&Ia6ZS5!VU^;sbdBrNWX^s_3Awlntj^+-ytGp*nBLBr*czf=L<ypwCUzr`bs1pN<`Vc2y!D_Pv7i=k)6hbHb3hO-?6pB!g55#HA}46mtjGAWC~cV@W)W6+@<DKypqNuq(@fY}_-w<Ni)plin%2-RMmShfT%zD3Nb=<2@9wIp>vg#t8)zH&>zxfe^?oH{OG{7i*Fx)k0)L&P=H?*^*r>=_yDU0`-R7O9$8SAa;&XEtHS#8Tp5SY0o-E-kjOK|M{r3@>pfwU^zlZXQT2S^oJVR$*|n(O#b9C=IXu7o*NaspoVv88B$J(js+WAg~B<f7PD5(khg=ds?a--^$h4&s$gJ<c2OJh!!3c?U>SlUw=Ckak}58fJH1hWt%W*pMAoIQmNSg|VY1s`?(XegK4=d2T9v}xvj6hlwS%U6VjT&g@8`+o(n|RCKRE-sg#s%8L5CwT2xmO2+>o_5t3_1uYQj{Nfc9i%s}z-yJCC^=O~Q=!$(3haw=htLKB+Ornz0&fLSO(GmMX@QnXV$&gajBWS#ptO4rT%%5|f4Yeby<Mivj-Vjf0kmkP<0xTSa#9n9zmvKM>eB!cy@$J03;yRQl;WD<SDvjm*qQ9;Ez~l{d5JcfX%2f#F&@fQ|9b6)q~XKc+v;LJ`d3o6l9FEc~uI6V)h5G7*KWl@m4;x`g04xNjjR;}Wy0wK8Q@;u5P6agkM0Y^qUOlNYywpJlJn58Aq-cjn92H<9d<Nm)k4BIbM$%Ecf}_AZd)i{Libm<^-!>q-?`IZwFP*O)Y`YEiHF>24k-%dI0_QO^}QCv)6J^2vanRj>|fqm_UcMS`zznK8|nv$6t@w0or|GqqD__G~tmfu>|6?1<nM1&OWzL?d8Z6>XL|BPg3dM{=G{BUth^WmjU<S&-Qqd&Z0dm3&P2cI)?Bf4X(&xn*Q_ZFiFPGQ!2xkZ1=W?d@C*Cv|XV^6Vh<6d5Dpmh~8Hp*<<QTJS^z(t^~Fq(a|9q>>4omnW`x^(-%XcPn`eR9n}>O}z@#by5nVXT}xj;OTHoW*wL`DqCTcsZn4N7mF$B@fSt-Jp#dG^Er0!#=T~0+fWKQqEv87?HG><c$OKH8JVhag&<bRbOeLOyu<33Tu4@GyIuDzyLZO~lE8v?IVM%f<XBslg$HV@&S&J&!tlS8t*-@PU!_WwAvL605ZBqp8*LR|T)}@s_#Z@{H5YNq*Kwl<fBxs+T$qy!>MC(=PVJ}r5*^YBy3k1WTqhPZbmkrNTG76A?gev-!QJ`lg~r(DnxP`-&AxWLz0z;L`1j3szt`0T|699@e!rPHT)*9PyVT2^^3AU`bDMuuzWn~MkNH-20goq*AEV+tkeUb<ekinqQUQAd74cG~mQg@jJb2`NJWUtC-Ux3+$;dULE=DMRL8vtm%IP%`su)swWMoUBi*-m3t}v2#2dT}~CIh~pIXHtBiCMrvGE=7BIH55~b9h=R6G3@XL>4MA7s2<k!r~wrOrCwgL@X<C;A~xz2K?7pYPXb%d7VlL!hf+<CX$yWJLg5^ynrwv5CfB0GW3wihkyY~fN0SLrnTX*0<mx}gwzVD>P)HHr06vB$d@s3;i&*JRNClL0LaK6y2^lf7fYpXV%!|V4{8;x7O$m0g>zTC;BUR#pIQtOW*aZrNC|mD10ZV5;LM9@E!ZqpqQ-NItgY<?*T)SfJSgycu0l!V0f=N;&(b5yl1ffwb%v2dg04VDz{62xP17-FBPcZbDiMd6E<W;PN)k*b@61x0XWHA|?!fD~kONRsixKxp>=!_3e+N{UtiJL!f-zHs)E>WgKOIi|{^5UP{)KNe^q&`Pv-KakuU;T$tMSNsZ+H883(oow^^gB$aRWiQnO%e3YunpHhUm`-y9m>72`p|Qbg%CRE?wE1B#Q9=_?6a;s(t5H97&3={HN7Ng5CD_kJ}mh@sJMxFe5VENv+>{ZB%jZBrtw5B)H*VZ)hZZ(7xN7V_nnMR~Gg(+N+w`$l&ML{LaNcopQyHrT5GFiOjR&klivE0MEvf{;0?nUt3LL$A#Kf0eDD6ffPu|m&t;tg`$f@S}q%xy1d;vDSVbvm)v{*1f?!-%<H7z?j_29I?S*KgVI9h8PLzZk6JqZR<rW=?+#Kd<2!uZLbhKwUw!d!`>xI#WAopg;kSR=fAm|^gIPMLd9{1fx8~}=TiM7z%(R+5=)WN4$=AmDKH|MMI{oy{rQqJs=Gs>7b$_`zo`2sR)z}B1dbdNt?+?%Iw_WXXclLdAXmlHj0qOq6XaDx>w;yS!+IN0+>>eK9@sG{c;%GF_H={J^Af?%r($ft->^^+P1bTRg^o);@zSO`JObO=$9YzG6q)3S8UV?0B>Ku&RR~x`Y$aBk7rJNH?AX$5XB%VHN>B~ovxqOtNKQnrlQ<+FHui!W<C8?J69>1}q*Eotwy7j`3$Ej<2dkVoyH@tZK(Nv4_8%x*b7B72xx8t?>@jF|KqMWPbB~Kx0xkBdCEe@%FaY;CI9Egh-*QkTej=x_XzvYj$i$gQ>w8mdE>jgJoTzKy}!PBQBv`OhVY3Y0gd9#{gg>a0kVyi{$Pz8xQ7;8LGeF%&Sx>WJLAkR{P!3oJa<lupswV@7<sFec81!9igkqdw*q#SSQ49ynhQ(-|!MW?sgBg@;FWIf#v(JN%6<%;-m?oYRVkHo%)m|H&^F@`$Ki~H#c?OB`@E7&8%3aX4At{`;p-&wHY9uCgqS)NyVxrX&vhPv=}Pk2J;Vi~h6n3}0}t4d4&;dM=w$*djE&ep<>Rz*<LNCiQy47tQ5>KHlM*8UFZJly`;#t#FES2gJX7?^4)RHX>OiP@ZtrBY@Km}os?qB_!bEP$pA=~Ej3Oa`KKmY5uExi~>xoE9M=W$$G59?aWn6LPk$-O|Q)orsDyc8g6iyTG-^4Vjb4_uyeeskSd-{KE`uxwY|!S2Z`v(m`DI(!0RgVjHCuv2LbFGBG6WOte1m>VmgUku7RSvgFe!LOvL@#YY@@cjJ2v)hH^O)Xb9>jtiOdn(+k!Bh?j@R1i0lrTQ+VbE-L5BO<_F5mlVV0`Tt8-t^&cpL0kU&-d+7ME;FV5WYj59r(X7%6^O$J+E$lkbFu=4LN4gq_WB2OP-T3w(_cLWhq(9sHI3?Ny{k6UQl_R6&Gg7-eUDkn3_Lr{`R6ph)OnQ>1?V3X*ghoYr{jWnk6IJ#uO^5t=UDU$rK1{okXLJU?<<+?be2%Kb^-j-TYDp5+D@~I#|gPPfADd^&A-Hca;*Ur?w6mQmg85Q&NyJU(^vKKbA*qf|4s*pb#mpZR4*Uk(m>TyLPGj;iMcqUhC5<WgE%8T-qrI2jOygS}jjCGvF%{9t{W$(%9_CIg}DM2x5v}7hb)MbS@sF$768LNS3J{fE9hxDVN|ZC{a>WJ|q@{3Q7e*ZZjM=>)O%>8#rUGmSAFJh2ybHDhxyVq0PVX#Pm|09JEC#v@bsc9I5R?drx)^TxuOWk~S|SHW7@9*~nypNvhA9jA0U4d;_#<L<gyC>+&`Ii2c1+XrE=paXTTW{-9x}wBYc{<SJ-yNk3Sjc`J{UT7AU6a0s<se%SV*?1b;{(V<0uet9&WQ`AY>&O>F*^mKF8X;qYg+8>eM0hB!%4FP9K>H{m*isYC<fO8liQhlH{x5$l6fCglw3YLE)1+6bj0nj3N7#1z!2Nwt+rUVIKkR=9Bay(WbIpMO6g&rwG_;FS*O61B57UOV1t~r%=><Hj+&9RrrLHU9VhBQuKjP}b7=ti|nD)0`2n4owl9?x4;?!1tgREjw~WT9O=vNn^kD#C>i)oJde=g5pYBGU_R0*TXN3K&7JN@O5Oc9NJ)3@MdmxvB^=Otvvq$Pqg1HW$D9jf>xDy2;R4MiUL7BmeGu6i4Z$-^E(H()^!$`z3suQ~iFXoIiav=5#Qv$(3sJNSHAqaz+;ck?@BmftyrxK!D8_%RdRlHBc_~c4q)6;6cRQBj>aaz=%1w4qp`!Kn9tqzR4m6v&6NGAZ0cIDQlsm)TDCcIUQ%G)JolVQ+T#Lq+M+Mq%wi_c>gLGT)8w|s7u8H5|3$jd?PwPxV*Q2{nF)YVV@d<UY;CW!}kL&0CD^9;L`WILbm_%PmgfIH!RlApxA)Kl&li572qdNPyjUoao`!4mNHsf=@Cj&Xu*ud_*X!5F38}jrgAE*@-OWn*?ZkN!jId`KH~IFkZ|K_=85=H4MC91RKnurB7r3$rnu;7YhF@=nDM2iAepk>A;rg}K*<meqBuj1FNt|A6#{Bh#&|B6RWhVPUXoZ#2&@4?n1Ou(j3d>7P70(eP{*iJUaJ3l3{%6dVD<Oi&ih<JtXNS2t5>-c&i(#e&8yXCp~@9wT#MvBGsKoCZ04YESO`cKm?_C+6~To_=v0w5IfZ{#jk8Q~<f&{<bC%ruv;MrTdU<ugzaMQH{3ZsNYq92aO&P&}BL@m*KwL=_!DVg{c4<>Wa*LENW3?n!!<dbTB)NPpff-)|%o44S9QKdtSmB=vkF7ymCRsz)GoaoE$VuS_LN$?4v{zKM4-jRNV)W>M#8S>8Y2XtF=B@#r76Fr5=Rvc!)g(G&D$hW49*H@(n^wW$*@E?}5}gD`WP(MbGzPl>*o910ObA9icDfHMT1^Oo^yuY<u0^(0%x*a5TGwOp4b1q-!Ig$PYn~RW;U(sR8EmzX-fE9KR7{XIkYok(nS_iD5{z)1s_aA&B};HgYND)K@Gn8hNg=5CT$R8e>g-}nRKk)F6dWQ(PF#UDh^#Y5$~InA@<Ng#kV~bfJ<*dlcyDzYzN?LKHy?4aUcAH1(7;;Sc*}76!C0Y?w-Jr06uc%xNg^F47NZN9rvknd<o@!Ank(L}*l>7dZy#*$N<rwMa5LFGNc)qHbuP=VT%(YOkvO<?ZMPs(4=*|8X2g|ruKT@K>Z0tP$AUgQ4~TXCDeXlhU|Po{brj?rGGhgK-eo38Mb#n0MPorlF$-Nx$nrr|TcR<+N~vN9r>Yd2o~v&j>((q06m{g56_FCI-b?cQh+KhoR%$9xQV2`#Gl0WIDyOKAiY3z7KdAxik$k2%zPNbFIohf~Y%<bQtUBsSzD3D!ptY=$$f_i*7?zwc)K;eGw6Snzmu~)Z4ri#qY6Jm!f+v00)vB&XH|$jC=E~&qmG<2aCOebua5BKDCMBPIuy|0}IfQ!`EEXI>qtm{UX==p~7{tFCoHC$oNqWrN8a+bdqRAd-YfY2nI@2V>@o9#XAq=34wJMX`nV<xtb7aUzv=w2HrDU-(b1I5<)HN9}lSEz(a1O5T9)=%o<*<V*%ME^TNMMIHad<Dk(+t$OE6lk!^py8%rDLUGxAoJbrQv+!Mpv#g_m|hP8mA4q!v^2e0#C&Wrm{z1H7;<E$;&Xd3XBOLky$}0Fp^YZCgA`sxaC1eS+E+VA^GYr!PD^p514@)FjHXfY%#h%f+%3JBC>m(vr}LT0+AyCUO1gK$vwItg(c@Y?M_Z+AGAkL-MoCW8AN6}y@U5Q9^ZwGSawE`q%HVRtz0Ck1W`IfsY39YHU*8Q28Bo+RkZ@1lm7^qNC_vXGW8W+Z;!`}bG{hA>d@kx;P6{Bm6*4O!_0sE(|hmqg28_UW`jk4arfXtv*y)rbg!P;fuDF0U+>BmyW`=p7WCGr`!lWb9VPMqeTWkGyC`vQMt}8QD@HhI<TgXK;74stdf3(iuhVd1pOJ=>lC3JdtWgn*5<yuI-omXwMpQ;y5(CvCU<z>OD|4dQh_6i4BIve>f0c$4X`$^0+Fio@#PmXx#GKooyiC>MkD%++-B_C($0XB7*A;?JuatOUrd;X*m7DKg*+UfGnio-9BkOeH9D6*Wq$CuAZc@M|NhhnE1ij6wa-bHA5Mn^MtcqYF$P(0kM_NhVVVOlH@q^8Gp0UNwvtt{@)F~Uqgnq6oD44UF=mkXnr>+F}VfUM(MAeOEsHlIldGW8m*L?Hh#lM?fIQ`mM-+=b5zaJXVdLgOhHngMW{|WoM<U@6#;T!-(F=qe#>N{KQ#rCzyH^yJPJ)>gYf8G7y{T^8KvF6<_hQBd}il0wKat6nh*Jdhx|2XJ_-TGtmm(4%^AA6JC<~w_@H2>qjH!pVggKPSBqWih|d^cF#)Lbl5U*A*ra45d+<JaqhZq_?vL*T=%FyPU9>r^$RV48E)DquJUUZ)D+MpYIGg9mAui`ml-j&eKz#0N>sBAD!$THHHR>_Q8IS71vk)@`0KI!}o#5E25>s2)sSvBX)<<T;n3D;5ziX#87QBu*A{Yn!56PW)-L)=3g?0&oi~f>5R6T*};*fPau}Q{-8AsVQ3UK>HU(0kaG6Ddb%WAz$d$x+JuX3XC%wa5)PC;plK&av3ymiXp))=D-n^#su_&q-la@#TTapkNKGv$a;H#8gI8JfIY718b99HtX@VEByDGnDlCb%bQGZgsYV;xb*qdxHfTNaUt~5R$w{*BHIjPKG)v*^))Nnu@%HHA_N%UAYjt=M{L>T|^Gy~Q^Nki51Amyp90_0N>@hcL4Opkuk`%K59p$1b>B0OB7O5zj_6MQ{(eP4cmCgVTngWFZVEp-?f73iIpzz*L|DxL-{J8lazgdC^&2bpv_||{rPYLOi8zJ4dzHu^w`<w54uPZ73-S9T=j;aNxhjcIg$L4$A{MK+i6#7~Px;0_~zxvuJ#oDv38|~`7J=5aVHnY9n`PMxa_;ImBt|E^?FeaSS5iFT`?V{HtLc$8>aEP9pDs_t5R8p{$rh-y2&Ezup5$iVAEMF%qpgo|EEXCK5L=hR`kX!AXEY@U2@fKlHvLvZ(5>UxW&nk(Oa>tFePs@({jH913XGfl-bhc8@1=pDyg_xPD=01f|i|6F2qGiQDrsV1vNxCPeqQl4{l=5(f+8L4rLtw>NS2^dp05`B|0n$k)3G>(Jpj^=c6T&MXhbqY7ijL!RAq~M(8D2aX!tf9PTWNufZ>UKZ1rn0WwQ@y8>YjroRe&O+0%pF6HZ$X#0e~@?)I@brI$ABkt73k}6saUTlIn3aMc*H~Vy7m#Z$E*<kLZOeB^1N8FqyUGXu<|bjp@RNB$=Ti3m{i2$WvAe#f+$lDk2e=(vgh8hH9i^ng@g5OVm81fH0H??v*M@I_NVFk(X7T7|GVl;sIY2BEMSHB2^c-IFjswU&nKg{JL^lnTiQTlavGPNwZw>8CGzHYd~gkoD^<S;;woE<{X4rWmd>~3?Y-dqpY+h2TQs-o+FEN0%Y0>EGk(v#qr7|nKM&aK8fLY!fw8;E3YL!i$Yl|=(t#skeG6&IwMS5LGN<r$<sP1SUy5l<#e(|V$`JPvXX0vV3ni>5Uw!v`3N{PQ!5X$FEdn8!)v0tAuFh|CB;;@iC{&XdrXn!fLF7Qo{SSUo?(8*l~I!f{KlPPSb{Tv-6X}9Ls8^e$P#TQtBCJ~Py?<p1jZPa5~*(n?G!)pV4SR_?~P`1pG=YIUo}U~`k!vRK!Dl}Q>cJpQMKyOwR)q;F(nD*y{TG(BZGy|NjDIpCb3-aZH^)dp`W5>ORt{L$rJn*^y4`HmW$2hvRn28_1b@%Oi@~xxHCD}+6h;?L)b27kjQbHlIb0rSFTL9OEWp_7I3d;(Bx7lyXBx+zJzlp`ovtPB!u7IKet;B&;Q*%9^(WgIBaj97giswdELeR%ZR<_Y4V>JLSn*FOVzAP%qx53$5O5q{IPquXvoqH-a&h091iz(8fqzMbJfzp@M<F&&Tui9GY9F)-rjc0;0G@m&U^koeA*T!>JQF;297cGNz0Bg-Gkk34$@>b#6J!Kf0nK>$h1_Z;au?GJc$HY!M2zdq=Fw~3O0i#=v=I}Oc@Ru!AXrOi>itFpQc%Pu%+zy;%_&P7>hsx&XH6LwLsos$k<xd%o&#^CFCd*rL<XOXUR>1E8?6C`G#{@PC9~}c7o2IwSIGNf2Zs>e}}mWzTI9R{`%c3W%7e7&0hl>?uMprQ<cSD^B*zk@VC9~?Y&p9BAY9C<sM}we2tlR@!DnlrO{2O^d6x(IJ`znXp4S*XYb{*gTI{n?44>9Aht2NaG>zwL6nPX0a?z#g5bf-is(F9ey5y(&pAS{LY}hJCns}$l1??9EK1x*0#1R3_ENgG-)^Uum+8;rk2iDPEw3D$yS(@ExykN@EA7+RqNN|!oN4waJKbJW7kxkMcUn57{=OUZ<5|6+1ylI$*NHaWo@UcI50>#@>600o*I}1s?v(uX;lvN`@$R@+efx2bRuxu%ltMVNc;kSswHS|w$P9qGsmx|TWTz#bamw1#whR(w1=n#raPu@2r#||O6{o<Fk5QbGkE%HJVCXICchBB$>l8+{vVO+QZWWS^@9dp?+&<}Bcb@Moo;zRZuBdT(;yyCHVb^r_$G18;s^KN~`!VY^0w45pp&xe@k?)=KIUme8LCzM3yxVPM;+=lnEZq03j@x*R?+tJIVJ}QNtt08IjC=5Ma~y~jqk;?mV0yvhW#1TX^uzAyH=Tyo>WliY`_Ix_pKHrQKWhHheqdT%l;4jI+xOcQ`quQfXil~t^ZIb<?~R|)J@(%nBR~}$V>Oa(rwmdW;SH#0z&gW<j4BDhVEj-Eo^sKa_>sY~GG58RX{-LT7N7!VrT9A8y+nBL@Y0pTo$W^urMh+J(@ozqqf<3il@jaqioUrZp@g5wc`c;bSF>ZZocxmD(@SdwuP$C(l1aL#o{BmW5(^kvVS*gnbxbvZ#6?aW)`D{dpyg_chN^)JNrQsZmRPAGkU8Z|K6z*I#=Kej;BM51rj0A+C#As{Bh#@M>#H!1JJ41zMzBUnUq$YKJHcqVjvV7*BNOyLfnj0tEOl-wZCsEF|7Ar-m$OZ(vJ!%!-L4?;C_`e3c`m__Rh6UxUx2H5md1orI7;3gqVzP5c4@lHNcUD4OrG>Bq>Th*_Q9wMNb&z`&Ctr*#r$7&DXQ0E>|^raeD~_$i#7B55o*jN2bya?8T<rqG{a%z{aGo-y9-LCZM*ScDEQ0%cyC(IaKHWY*GoRk_j}31_GkU+=iOd7@)vFP9fkaE3~%(a;efP@vHd5FCxrKA2JLot3*C=LyTy-Y%&1=N2IOC+12UgFAI|hv-02YNotcrE`Z+G8Si0uvwY@i6Q99t@_FKI?=lm%|pWHv09mr+?dDD~8xw&xCgPFO=QQ2}<w)2yz1m{e_>rV#d%D%K=akd@*k((${wIK;<NjH{d_5ZW?uFG{@*O}<6>`EP-NC`pC`vtC?6h+BS+)<LYl=L~3a<S%>garZ&07_<pR7vi(6d!luA8J~Tth8dACTWVISo|SVf268BNy<;s-&nW3_6-0j5(Mqo5=CIIz4uykjyc9RZr@;5hs29#Il?b1o<>eo3FS*OmO}7=`!PD9e9Q@M0r0qe!X>JYJbF}_e)OpK+(<bcKI$CY_?5#)uZ=h`?~EoC-aE2@EIAO}w9rR(!@Pn^MI9P65)U^ogN}m6Qp0tkO5uioI>DAEM`3PjMXuFSnFbM&p5)q}cc(0^guRElhWo>Dzcu(_OY`;WKluh%m}wkj_yk-ticg244O3L3>557|TaVoas1mHu@X<geWyvTbqboS&ULCvdezUdUPna#5t@7QMb49aNp8F~{?IiL14hwd_p`80+4<zmn9i1n`WttMwF8V%h1+y%!=}BkL%0;eNz2N(tTS?lx_^kt<?OC_*F1<*@+u!5h@lC&896(WgcJ<c|?%CoKO__DSzUb&ZC1)<%m;`anSYb@HNt<jurRE(EG1KU8$E0z^5j9n=i3q5ev?;;iS!jHJsueSu<qY$OhfvhHqb#>v1zDWXEahTdWxz^A-xabV$}GpU5+O22lwbB4D<k3J2sI@C7xQMKl$uB>CyX7eX|DHb`(B&UnJX&kvT5fD9fpl9%1<0pRFas<RAnBq4FpCV%t%N{`>F!RUU>i)m@}CbRR{<;oyl7mI^ng)63=(kigmN18e~=7pHDPkg)+#2Xqu(Ren=)6wV6k2VLYYfl3Em{uNe-pr~potLmmoqss5H!ax5z?<iyoN{8}OkQ8Lx(WTk4N{YWWPL$hnVtvPX{Xi78lE*KyZYU+$RKI?@T8c<u6#GJs<Fea<wOsXQ#ta)AyV=NY$bH+U&wvr{;T%ipkV3><C9UjDbr>h0-W`N#p%y<`vc&BN%YI*zLJ@#;xmI;z>z0k5M57lIe%Zrje(R%6-qJhP(YN~@`RIAGYL2)8Q))kJ!NeX9at&7&!ITekfiOL*)6>&x->|((V#*#Ix0Di%Mm|di&1XxxybGapIKO~&>@xz6)Zv5asZvJx<XBcumU-pppmqQ!b^&tka(&s$NchAHao>YR^S@<E`#zGu?QWBelk--!y1q(V;Nw>p^kU_#}&4tJ)jC8p`mq3#z_Ba#7j~GvIwht%ZsH<r(!84c|DOUs&KJ5brUDo#25CJ+cPH@nMJ!%X=dl<)1EOajxoBE9dO=Q*H(@275BrWII8z^r>RH&ISje>QJuBsG{5^--Iv{P1N#np!Ejz8#5!7NU^Ot9LvTH%w_CF$%O9O29pKmv~{BnBDtuvjc)3UnpzRgT3Tp?6J<de^2eKj|UQ>>>NuZuEO`z1Jfjw><O*+Rw+|!=?dwO-fnt3@k--HIO{uBpSgeQzZVEoCGxx1*u3h2@EG8Y=SAx>bGV;t_>6hLovuP?}TAlz~Is7I$yXiRJWmwc0$$SjquD-F(p*Aja~|`3_uK{aG<0b()!yo|9spR^)HjAD(8nSVSgWd{NXV-dK#Hi@CA?~@M>#>H3ZHg2A?Fj0?v*h(@<Rr^Z~6E6!{=pDflOFA=9(wg0e@Qt=+P7-EMPw5BeP2yJK|xU}y`x+hvX(u>q|x6Pz@XbHzht6*{YAm9jJp+dwgMVU>foSAsM!oY~TlXKO2Ec$JBZ#7=Ix8@iI<n>Vyv$_Oe!mMI9Km=#2$<qU_P5oml+4q7m-aUrWTY+5ttIXuE7`ERis`WU#Go@qDK&<J4T;D{N?Y)Bq{crl5n#GyViPH&_EsyOaZVFj#o)B{L7@d{{2KGPlgKknzNygQluq3vG%;+U1UmCQsQ6f=@o6GB7K0X=TGVLamxoL0r9&05FGpi&{c<Y`(Ffx6`(${QA$(S+?RWfIj|7hH|b%9ty@7buq#1-&HopbYWU3l_M`xD!<rt8r=UZhr3f;0Ue%p0#k*IN5<h6)Re(LW77M+<pNuQW;|Bvz8Cpn_6}@LZj!sa!m+wDdXnFD;F-nov9&Fp6v{DQP?eeyJvO-PSAI@uZlA-ZoTr#pZ(Pt2v0flFt>Uoey5~8x`9_Vc2n6TM;Eqh@{Rv8d489~c=yyKG=E;+zC59^C0BOJ3kOTmR^z#Fsa}7K!W0}gG?#|vxflVkEU~SXn2J>ux+W)dsj_ORT=C?s`9giEo(5JJd6>(_o9Vm9+p}Y<1E7Gib~zZOIc!%V?sp-5h@5;)$?5MoWToIQBx;u<8Z;X@li?WyH-hg=xi9s;wEK$hH>Tz=tn*m7pj5*JwW7hldEci)A#uarw4Az$%2n%YlfD+Bd-X7yh5M5^f`!{tf(2q%`1R!ZvawZg;Lg2_f6n&&ORuCbL%wu<8L`6t!PE*uAC)ajjM%a$h;{^qtoFiTT<oDN3;o{@-iqlvrn@n-6>qmJT8BJ2mq&bZuAZ?cUuw15!1r5Csr*~b+T0ylF`6HL{KDkrt#lR2tcmY6ziY@U9`p<#?G4=NukJqc#((eRv~?Z!Cuh?|z~ZKtrnz(&w0wKI&ij*v9N4E_`FBGm(%)2jW^Ok(et%kBH@*G4qY88yXWs35J0tLeQ6Tv4@WBT=H2V)n(Ql>YRWNTgjiaP&MoQ43j@#@~37+ztMYu`{4B?{CTxwH|S0Z`1t$N=T5nYJZ)10KH=Le8fnj9h%+5wBF8Z!Ro$Y;g(h)6>r6_1XnYqZ%Xq=L_Y)z;ABUKEw^FU2K`wM2Ck;UCisCIUUB$Veo}+!5go?!v}>mzYje0q+<!YgR4_#{(!+aMN(0G}qL4Wbpwk25u<MEO4V5ZWKPv??>XZ3Cd{+#B7a{l8A2<o`B239gDsi7Ndxvh>W#?uSZ?r)|f0D)(vY=WHUg|DlDX-Ne6%?&d6edFhC~~K*YhS!+&>@Zkvk=ya;<IZBlL@PY%e=G9U~b%mRxgHPP&NT67y2z++$>*qs+=X~I-8CS*&ihGp^&_D0dg@D|}Vde}qHQ?4+F<c>yVBY<brH2*^fnN=w~luYfhHUuvm-`7IuScrIA!XW4B1J6;ycml!_p>Rk>cx|XfKSZsKH$YjsVtX|W#;p4aV&MRLg~B5hOT|W5!jc$P6gZ4}W^JheY?R#PFxK2gqEfLj8@O&_zL{UIll@%h3Wju@lQ~^C^asqVUFUdWT;TgZ<+|7xy#0f(ppn{l`+lFxeR{9j#~ti*VfFzM_POWycNGg>)+$0^I8lCJM>(URzSwJKa8QIPkQ<{|W<?5toz*qd@S2*5kllIf_YHNs(`ZeOt%^=zV**?R>457BE(U(XmDsGDX?TkX72c|5R`QxW*J?k(sdrfuopkn8;TN%#_-Y6Yga=DrATL4y2P?S#-0>tswKVXrV85N-_cFWc7;}Or9xQ-m5&%m~1daVrR{9LAsRB_KcqxVUQUqW!7AlTN#^Q@e{gEO@!zkjRWRjXFP_lwxDV%Zd6m~oU@1;s|M6J}cI)JB2mVIE(6iYNXJ+c{p04by=5}BEhoK?02Mx!~fs1@>0GZ~YJaA)wVYQ;X57|RZ@M_#Xsk&dIH<SHO?#qvP~iI5WoOgxEH*0c(^8)a-Z!tH04Bn~UnbSswn=sC2Aa4JYz8iqTd)h8NsLqx+qX6#rHxNX4l!uBQnZsOG9xLfBb>lxw9RH7admKfS_ZUvEhn+hikUz}(Tyvw2{$c;<;H2X+bKwQhMfO-RNH7-F_89H$?;{hEN&Q&U?^Ey%~P4LiT$}`QNDMK)t2Pr&`AXmW1bf~FFMae)DZfHKCq4BMO5U_-H;gp)q_*7cl^el#imN}NoVnPb~jt1~mUWuM5GMZ)rgAfWN-$HZ2qpFdm$Y6?+Q8wfB2O7_ecPLl)u?Gf=8{v(R&V)L{TIvi}!AoL6!&|Uh5tmw;av?dh8t25}P*Z;d+Yibc*as**=Ay$sLhh@Mun_R!gNZp}`&qUwVVCe>R%DeV)HX$Pg7DCGisLNc2iUtN4FbiQ803;_I*D%t&xVZ_yg36YM;bk8l)$q`7_5EiY%~UEAL<FpAJ!^qSWHqk$v`Zu^%eT1v4f;JsF#z80?!PZz$UgY0%R6uA(@~ujV)9-y_KZVHjN&#W~(ih1m37%6~;Vp9$c~!IsrEBu?E)o@dwtNhH<SldqiUupx3E{1D4K7!FZ(UE|r<gRCP)Frj(8FN$|!O9MycJ`i}c^f$N|2c)iOI58cK0kKI>yKsHWg7;H!7!P+|5D=T!O-hg1Jc^i-csyAi$>6MD$bR;Fu0ePZ7tS|!b{qD-~8<V!9r}^{Ne7nzq?>sg0)|u2m>&Dz|*8SelyK#T$3ZX9ADcRthul2<e=c@ICt_`i%@AqEs`+X#`Yfk~D|7`Gzzt?}~v~zoCFZiPGAGtPJq(tw%o|!q(2R4ciPtwyKl23ATwilhQy4m|;ciU(A`7nuL=v--;@`nza9}ays?W2FM$=G$y;0Hs;)0A(0ru+ALe}`Wl^vR2TLBDK^!{6z9k@`OAQMb~h_bIQBnu+esY{O4SZAZ6GO?p?g?;by*T+M9$hof$*n*;6P@Bkn5TmHfLv1Z8Xr_Dq^?oNI4!L5wEH*SVHP6B%&h%pV`jgn3%2b&2;OE3*AsBrp$IZdn@Dv!e~W)JwOi5$biuL6=u;iPs4N!^$8z;wc(kGg|A`LudPLC?>}BT5bz+$W$zt__`ZOOEw4?=h8=M=cm~wRqKRYK3cBD3%^YGQg5v02kEAO|7*^TO5!<F2VyT(ELuDfi=T3omRRt$Jp`Va2`MD`%Aj*dT{(XCJ&fLlY%)1xM$RZ1t%03Ula5)0KPV}Ok))RLb#%aTvMZ_xDLFyhaP_WZqL&*?UVexk+-fPf708VX-ns^BNaHbK-I=-Ul_4lGG-Kwg($ibV>x9UaK@{397wljg_m<pOlF$4QPgii2K2^=LSx($b@RAs$~|B(+*>80f%^}qgFIQHUz5y-s)e*nYRbM7SEP#t{}q<lRCUQ}o;d0lNT{WMs055SVvt?RA;{h&8V-%G&kyVvvz-j#_AkZ}_s5alIcS!}m^`7!Py^)#%+l>gs-%%ru9B)!ENE<)r)UT@1~B3fvs4JkMXWhy!A*&PA9!C-4eo&Nn*4lR(693qRL*YI5Lm@n($&zQJVSHRB)m2Y-N}j4*|}7RI+mIjE2RhV{{*j|t3<<O6m$M*&9Q*9Rf@$e1tL3SzKG0h!->IV=EhJ;*cR`7w#nxhGly36F3?~<ptth7J;%ZuE%1LC1>lSVEH1vnE~czCHe6LemjJC~lus1QOIAWUWXX=kLtAyY)%?MzWdGCl&>X)Zv>d9pi8w|Y^=BvzlBB}(q$=>5D;$?F%K*=lFPU4LO60-eEruUuirUnvYG?jj>oZxmCcS3(u!l+ahc?_}UIrg!;XndJZZ3$I*WtLQ=&~hJy68+~A`^KqhfD+)qyru(5-UQ(Lyx`n`F6{P_+)9zgj!8o#vYco?DM{Y`Nmvh!{EbqoLwJwW83I2Uv_*QAM{SsM+^K|M5y;}Zx`>iCiL3ysol2>F8+a?FE`pnB|LPxU9ZzE{nlKf_D&NJxHmcAp{Bhy*A^*1-ld9$K7af9D1Y0%>%Aox-|jc_&X~OK+6W%rp7g{M4S&eZX%TD7D$@S_r{l-@U|^VOBN%tvP{$Xv_V||O*%x0KE%duf?({*|cGi2S4`vw+-#UP?%rmDroI%b^rV;9rvK&NZp{<5FN?yP<@i4$FIi)KQftNM~3RP?B89qs^^fRJyVjAbCz2Gm;x1diu&xa0pvp^$e7JAwJ(BA&y&;q|U--|!}B(}QYehKT@+5X~%+n%zQ_3!diinN#Kul6slCPaI2fh?_~2f%Om!qo@$;M;|pbaSk0c-wGq?&-AN`%(e+gp1h{6F806OE_DhcqNKf#<B|Y4*18Sj|3BguvAOItWicBZt$Dp`c6Y0yBQD6IT4kGIjz|R$}!X&s?`ZPLz+mPTr5!hNqGFkDc)hC<W5pTH#`mY(PLmRajVTZe^@i#CZJlIJ00F}YKYMPpbx8mHuK`${&)9I4fB&)FS!Y)-|y{e_m<WT9oKu^XdkstfDj6Jdzh~_?_Iw$f}JybWdFPQ?FU0_?)%M`4i@UmmO^W~6P-~W2)Jl0(T|pY)SU)Zv>A<GkGRE}6bS|P|Eag7*9S_luBvQcOC7<LuKn|uQ+K*I=q+OYq=60XvfgVc;BNQurd`D@%P?KSp#ZE42R!JQUb;v`H{)p-kJZ`kw_4;d>G$>KctjJ$1`e?2sph6rjEWg(5}?#ntgD6Rndw-Aq=dJPrldm6DM8WUam86p!!&F@N%s8UmF$;kyqG74E$1kdpHqTR)4%rON>ev5$t<(tYlp#yF@X-BMIR#=G5VmF`q$1ay#K|Oyg6R4EfB1-l?Y)G1$cA7qYd0ZZvN4R?_Z>4^<Q62>pES?uRk4J5G40xvy1>Ts1c6730mT?3Lgv$zzC8X?>rL*$5WO<gRWGH)r!>R?w_S5Li?C_Y~GuVpHI`Sk6}I%&Au?wSCO+YS_`E_a1A9YjWMyl8kcM{hRaFnUgJP%qE;EQ$|~AA-$Q6uQQ2s<BJioHy3OaPg!PeHl_K>jWk+L8sKYc06`<m?3R)X&GttpzuQQ9uMP}u1OTpNZ*lLI-#faVU)>J@hqp1+uGC*bT8B>)P@0hhhQ<<igtj3s<Vx~qJOjgf&sZ&s<(s(K!XhKy2e5PJ1E1?UlbyaAHq>kF@Qkbk>@hF^PP%|{rLHVQP6gwRJar^h<cP5PIQUu1N&faShdJ`j{e6-R-K~+~|DA`3kVAPpv?wmD5T^4X)*t|we+nbxM0vonF;4r{eUk*D|#enm$b17_Y{2in?Um<Co9)xE4P5WwJ5%E`l$oz{hPv}4A>}2i#Y5(@(WMgYGOq$u=+PvDF!A_9>=I>`zSfj!IanzvDl?UDK5=R|9$6Ku{cL}g%1K+%LWplGvZe3MHn+a$&FC8d+-xh2YzP0q%cgveDZf_*g1b@@!IQzW$l*0er_S^al=-W*&s&yBx4^guInj_zQPhc_d8FpnpzxVBBWgEYF>n9`9{jSpL)@Yu>tw~Q>d~1Z#`}&)=oBF7C`$xFm2Gf4?*1t@nW^G;85BmjZy@{U=3A<@*uPMgZzT7YFL+u#7uiFHEnhm)a8WZ&Ge>mcCB7^mYek|vSS~FOx(20^O#h6vD#HgfqmZt*mAQR(XWYrPTU$SuI1%OBL5pH&R$eerH+1}kfzmYe#FLjBkOC`{#;IEu-e^~C0VNUS%#$UWPwCWr)gGUI|EE#n183r2|l6dlgy$rw}b$%=A;wggRg`A7?*#)n?5!q+0<)M+EKQC`iUMg>t&F#xg=hKT}>-EX=fBw>oFC06I4<62H8#MrBp?Nekc2SF}71dwo<bq>LP*+7QhANd{%V~lDvyri??2(v@+S;uy&i?6$<N4<CKW+tHmP}DPnyRm`trZvq=p;mug45nA>kU&H{<y0t(a<UygJ?p`&K%nHYoEdsrKqWc$u%_}LelO<>X2|X1m)m8<mv%2l2F1)Uqk~qi)1=0<&U4fS}n<*iKPz1Kq)`20&L@Hw^o!TxDzzLUY(@*^~5J=c!1)f32V7BUNTdm)Fe~#IuxIb6uDBCJ9;=KC2~h?c(6AnrK4u(@z0H8KLS*8U{o|t<LMGSP~tF!tcD2x3a+zoR>y3r56(FyWT5#Va6Sabp^1k-8;u^i@f1V#cz~RpEgl}e(i!KMnhXQg9T2@Zs}_z^@+DPj3!?#<(I$y^&?HWOXxW$GEYK9ssC-B)21OKs0<K5lp&`bp)TqT7bF~D(dSV!a*f4E^Hiq!_;ESHm8@mnEplHHKizujB+zFfnPW8XRr&@Rc#Gyfmk%)Gz*R{fWLMhwCO=84XR(+DnHY`Tduh`jsvptv#QI0_&(0I?itAyt+6O1nD^=0${OTrrc>8=}_WeulrZ!cWf-nwv_iVwqPGrBJ3=2fD~$YJMnx`-RQd}XJ9jUotVuu`Nj-FdBIIIl7f=UwLNeC14NPHcfioe7R6yFf>08Hr}s6`qt6m18lu@$vNKTo)rFLq$;CD4V%2j666vQdlklhg&(~@8O{&Q9>pX&X8$?iR0J6^JdWyVIel!`GU>DZWkh3nhTByzHV0#EUoJ6tOE8XT2&1oYsE1%MnFJmz$~lT!znDhHcgjR1CT&+y>*bsQbRJn<!zb`j(lS~Y+~iZWTTzZ`j;VJ3TxjCm#%DuNi**C3d#G|MlZfnE?k^>k=`5jCg=Js{0fnht`Ypq(OH*9v8%7%3D$|hxZb=uCG4=sNfYsaQw@Qdp0I`ZiNvyR4If0l_J!@8tLHCV2|JW{cw=Mt3Vt7O1K6*<-SgjR{ffW;hli*~>s#rcK>Q^E$tw;wNl^nS^?I6ShBK2Is~$bgR0>p-XdIK(873?#TrG^TCYd9(nh_>h!++MApRNrK=dl~ZWCXkyB}sZTo{R(9Ld`(0ddQFS%2Ta+lrS2GR)j~Gd#1Ui0BOwsS)O%hUiY)666vj>gRm_o`h3VQ&U8REKND_k%CWkb%IzU!K^fiu)%;>SQ0>8xE*sBu>_}dw5B>l5?)S|p>&9rz=|6Q7fQO1HytS!QnrRT5X`37R<Ua4xcKyVup<t?g<~u#(U;DXt`dAN50_)nkJ{=9Ax;B|+z^CJ*@Y>xMhT&K3TjnQz-5K5ewPB8A`>@vrpRx@|{JIZId^xw8GwXP-D>}RJ<=Ti?<xXFPbZ=%#>~zQ4mAXF~chybs#9jPP$ixAuT_J+R*{F@m;Tl?o8(p*yN_!wznoMUxQl7FB8ql;7%^<Gyz<Hc0dy-PS2g?45$>JVVXyxrf8g;RTN*|u{oMt!=|HN?-SeH*-HY;UXgLUqJ4DDQE*^Y9hc6EmL+``w}hbq`o3xiP>(#fMWb_;1-yI5ZKbdtBJRHB@qm}ww-oGMvSFjlD=;R+lqV(z8mS7l;2pITHJ@l<rN0I>_@%Y=3T6E8LA?7I^&`ORDJP3(6ke>maz`wy;9*px^9!L%c|4~(~cULVoBI_TU3e0<#VYm?vn<NrQ2k>76c@8z(ubAI~We?z{&-u4cTRoJ{x;!XhU{O*~n1WLEJD69x<8@49!fp(}lbJENQp&-P!hfkx`Y!VFJCy+mZqn>s)F7KTe=Ov}irdJOdV>eVfq4dkXr?mM2`!-#Rf4~3L)}sH!RSVJ%F=!rvB;e}hlGpzBO2~NFjkLD0Rq(?mHCFt?A){^1_g^(9UI%>SqX=H68J-!=t}{NR>U{~sHrPm;sh)|(4}_Tjda|ZwnsOl(oC98?Nb)IE4eV`PDliJUe7DK5HjXCAK>Li-p1k>j0eygH9WPuef%Ns@EWg}-@L!glN@=^chi9g^;1B-b4|<lo4;mTi+Po&y67L4e+#At`eK7!>_eXh(yR&nSTX@-3oP9bO<{ft^vmM=i@Fz?=-07tswp4w~Z{(d(e;RQx;QHIAn;{Z+pIbS#`MJkX96{xsLfF8S6@scAR1md7w7gAK64e$XZYZTPgiu6k+>+3BPC(-Y<LBGLxL$9zcv<|_uLuCJq^-%!2vgwPXW?=~P_Mm{%7@~Nb6!@%;567Mahq_=cq$Ax>$#74F#z!5H(q)9EZppdQgrgISH3~&&dJltKR4N~lfCV|rZ=p8d1Gtx_5TmB%C2#DYVzjAjdXF+$!{Az*lA}c(Ff?~CcVm;Wc+1guQX~jLgC<$H|Pa>bpq-tO3anR3)5WjXcgDgazjBBm^Z9*HdO>JSt`*%_gLisC=I+qEUI9!+1F>ft|BK7&pqb^?~`bkyrT%Cagq8Wf@8+SYKn#Lo2!QRsWsPHE6HsRQoG+Cp8ab*o52UY4@a8l*70}G7v1!kNEbEiO0fn@pES*~rE#}3kETlEud$Wb(u#@FIgyJCoJol0Nt#?#j1aL1rm8kPbL))u1(HU9k;;<-)1HEINdzJaR&=(A-;Zdmn9|Q|3jV{^tGh9s@Z5X<BkI|u6yV2^n6Ip4E+J(+t6-IpiQ0B8C!EgUI>)3mB&k5GgfI@6-a^{Eg|y*nb&>?E^6DLBLzGY27#D~v)5h!-m!;w{m2OgXIWR{R-zFuIow3$?94>ZvRPs(AYx!_!763>&_Uu*FrKFZEv;-PD1xsb3kdcVmEmSc82hP8bl~})&%sECHA{`6GBgNs_xpr0sAla{G2&s<$L96zF>cBPCTOxI3u1a7p>YvHZ<?M;75`L!QO&LX~Tw1SaYLAvdAHJbY#PjCAwpsDx4lDy3R<0taz2wyC%e^f$t}BaFlp)?n%Y{vnM-vniF;TBK@$lyir$(Njqn8^BNMCXKjKFO|zgOz($1eeVrBjXu2m}utp&_!|DP2^^hIz`+NcTIFPz_i1E)uVCXd&Rn_Jy5r`Qp`8DOJf1aQw>F=~gTo4yn`II~UG0uI1^=7cc+u-p1Z$`J;0e!`|NZ<&ZC44OgEN=gz-M#$)e%lb~y?)SM^uaP-RnBalY}yrF|rszhOxNCL}mQ}j6-kH2v0gM;IN*J=0*O{Q^$aw^r$vtzK8^tY?HF)~%3<Es?BeeP&txFl9D;2-k(W1uO0sXF31jx=J0hQ)DQtBg^#nk7!qbVVnf!kD~DYc}cD?Du!!!_eW{-kNYT`DWO<5_YamxR>rY(w$4amJqXMK5?s=q0%Z|lQA$0LJJ9<k%;2Z!?VGYOSmi{<ND~Uj94SV^Rtoe<WDW#34COHC6>67VR;>o%0Pr16n0|7H3=1F;4_ko2tOMJD5?S_0lYzdE$IYJj0Of`yha*gSE(u0+GZFkuQV|&g_5e*M2gIU^jf&Aip&Oh&;cglWUpf567$#rf8Knm*dfC)WyvvFT$IFE9esg)l2Qa>8nHMLWPlPeEDLxhT8Ri<1++-C<AohNvlCy!?DQG&{r9!nAFWRdxLcN`HqrLp(1|+76me_na7^1iyH?lhLlfbYdGFNZPurN<Z3@`k?1p4aziOaY_@_||>9uhx^VYy3(dG<qjB<vppYyZkZs<!t?()k7s^2Cr{@;JmKEk_0hwtP!Z++O@(1-2!^z52Fzts#K;qSNa{^7I%cD7A-yc%5#<>a+LSuq}b+F*P4(P*RKZs5E?GRQ-Rwt$PP7`zRZJmf>OqAit{Q7cD*Si}Oo>^1aqj2c(W#|QnoT5cHSD%Plo!^>EO->j2%1e0?xCBd&IAum>EsqyW&(MpC;P>qc6g8S?l@RWO6tWj0eU<?DINSXn|LH8D21dT1Q4d*$$nhL}R^zRaLcpxCk;*2*mz1r*FY1SwhBo4>O!byrnW2r)uaKkF$S1}P^xYDG81dlpU-ExVMC*tvE#>)CkGYekpTVc9b$}f*SSeI#Jnv1#+?v*UUOC`d7U@eg<4xJR1fUnV1pb+S+MJE}93nMu-J{(~-_uY1i<kYmzlTFWD^W#yW&!81Y;Zsu8jh*h*H|fi+g><NiS;8K8zYmaJA5ra%D1YztGupa5<Pye$2R$2C%Mke4K*QV$#RvTtwdP<HIcu#_|Ms4Gcx)P4dZM29_|_~>_iX>tWZF*Otkh(vGV1SgRIxK^6Khwst(W@cq~~#XdyXgh?y*cO$EmIW6+ry(fB<lqGR0=9^$Y-}6yS;+P5}*RfjJ7wF?g~{x|&qZnb#9Atz61sT)C3WOxcaC(?&PV&&$}2?{q~^wATN7pm=)tZJUlUpCMl#?c<d9>+>o1MElu~$XA>Ni@a2&;Rk!*ZVv-SlRjbEtSzvD5sEy-s*U3*8E2WPE&-t`?I>h&2tH+iwiRf!!e^mcD13mRzG))}l_*V9kvP{T2pVk@S&WplP8@CxRl5_~NSk5f5^3alJ=7M?4JBi<0|CabebzUbygN{z56a<RdWI@$t!)a3$>Kt(h6p>b)$MWf+Rn~)+@^v7tmTrPkx(~Imwbh?-xs$pmEAMM)=efE`Z{cE?w*O_uXuR*yWjfOOW)j!FMszNuWtCi_}16IdG4?O<gZ`)udgw`Kh!^MZ(Z04m&y(W1LF4f>rE9@*F3U&hUVF7*EGX+v4R)RO?I#Byiu-Bns5tMKjXDoDzu(WzAa$=@@vynDAh`DO|EP;0s~;xd<Ml3F7Iqy3UH5S2Tanl?v;=(0{Qi|@!g}I=^quw-byjXZm~<l;(%_v(VW^AsI3icx^I@)){adC6K0LsH9ne<NnQ$_d^{Ej8yxhFlD{^gM{h$Vco$QX%?;YMveT`{BKkWB5aAAb?9d;Tw%`77*kl3UJ%nnlwvI}3T`Zy<T~%nV6}5x{iCY?aV_9OF>r|aj)G3ZLXr4y(?u{p2z03Wo>RoFdu6Xw~=!nVM1PelCQ-AR){ytPBjRpqnLTvMuO&sA#i^?Z6^(Q@JP>T-l<3nD(NXBP(Z}Oe(IH?;uyL+pOK^-s3z_W4^>N7Btg!T#uxca8q6k0FkkRsfj0w*75!Bt&rz|TZ8UQSRk=(p&su0h*vY(4$LPbyVvcXe%hccAC_`eLo-xmmuwg9v73%SU&6kyvX*VvaFq#24t#&K5n+RA|l-^^8kQ#ZxJW%x#(!-e}2*hT*{t;~7ph&Yh700xDBxnU`2-AYH611z~B@RW(c`b&|q`A_6+%=0X7+i6(fNMX6Ayp>kCt3x(92u}1rlvc|Jj*vJ%c)?!-7z&+IzmB<0NYoU+}%Ss3ar0D|j>K87Y%|)e{sjesUP#G6({}9lYbwEVsDO3w?Y`p;(4Ll8{_xkxRNIqco%i$_T5tj4=A5<+!C}MkOV|%j<C4*F~xzi|B>atzyQR*3`_b#nP+)}|WF9rC@TZ<vefl8ut?C0W)L#x4*A^wxzPCe-8%=+Ns-M;gInj$98zew$9sZ3tGShjbrQU%)CJwT)@x$HK%m<#2F^~Ty9`4(0LFne0O*4BP9>UOx*8sh7t8Af-ee0ekXJDyN<bKK-{#K&!%<lB8T^Pl?Oi)#&g?)C9h;xC4Wd2e~E$LxLm@_K8MSKjcrGi0EO=mxS(TW-dbA<Lbv&->o;pAA8Q_a<L|`Q_&8m%Mnbk3!!bF!CsXP00D$<oQ>xUW(hBFH8<^Ecxj;M!bAX)bg!pzV>@FB#T39%c^&@9UHe6NgSt4EN%Dp_=Vf!R-$*CE~dsgZF+EiI!7G+W&drrn#*y2x7R1#)_f~ZUVfvkD7rrB<2d~ooln}~P#?F^k1knoee&<W?4s+FH?B<n=4I<ze0IA@GTa&j>Fxee?+qN4{Uy)i4qrHU72j^xZ+=aNC%QMH+4*d!c>3u;_k5=(oWvsB{ME1=-yh%SSJ;xH-9#JHdUuEh{M%4vI1>q5UtFEPnS677@@m>>I$>Yh0PfnGd=q8}YWURBL<83I|9zzVg5x@8>ossiQZQHCX(Gl0{)M?_(ot(>lu`KVBIhQHS`=qM9IO^^OrFaxD7`LMu(R=|t9x07LBRF{3}^}8i70YRGSY<L0#nfvWnZz33S{X%suU_XWpc<qn@D3Gmdx{nrf+J9H{oKAD~8iH>HNLzNto0to0|Z^J6J^+Eo#Yl^{>A{4ao3+skjgR_oQ86z>A&NcTZ264{fjjr6|eiZe#Sc^QR{9%3f2g45$GI_Y&5lb948u5+5A7y|GNFOJR@PUb+<aau-+N7CQ?($=juM{JU?atNNvUl10r&O{oMLkL?w<$&mY#oG;wC!gEcL5FPVb3SOJxqXif=yOKlko+cp!u{_qIW_IjF%>?`+_gqxf$Y$Vxfs7d`JrxcWU%bfF%HOcCh~%heiDuL0B!gfJ{kwC=n5e`)XcR+hp4%}K@3&1p(_I-Npm=WEB;XptOX8eoR>r7Vl9KAFb1tEoggg_*34e!2EPLBmb}w$c{wDC@kO=`|``g_^YDSy%`r~zR3Uzm>;GtQV71ReU3>*T@1JP#qWE4%q$v!%oSQCJ`g`<(OPHSP~;U$Q7COv7|vCrcm52rBfi^wSpWrEZ+lwIPHg|bcq;HS!M7WfTuc;iK4r!d*$`FM2CY@zb)QA70o<1ItZQo+k&u|!IaI~O1;tWZgYNEWs$8Ky)a5=UXV^$J$UMwMU(nGw>5Bc#qgZDMOT4@{jcTr;rh4J_P0ne@C#?>9u)&0j_dsMRU5+YL$g^)Wa2WZL-M@e#Iu(z}!K8}GJGK<`z&y<({5lS(idVQB`MkCKZdf+>YZ!Qo1Co@EI-ISsEr@+v9Rs(h^(M#RLNW3GkE6Odryc1SRBE!=o~rsY5>D-ukNQ`Er9RT-pz>+$8=Q>SKLdgiyU^GY969!VPOg)_uRBcwITOO|m?VB{IJlP{7pnWOin`kJsQObdWDm0hOrw|vAIN*{OT?x_q+BCyG_vW7-T82k`2Y>bO4NJpVvDuvIG#0pUCeHO{{MxVRnL$*~Pdu^9u4>z@s9dT?QF6?W<zVY(mWBdN>&tGeumSIBQM<X>fE!FDC_*YEqJNKuP*S`LZ!EySa^HNt#=ljVEb4kHLir%06VVjP>^++y|XDVb3%)xyqF-1-tX+;D$MFGB%NkUZMSrtTNwnShOVrtUFQtfKu3oZ~_MZu(2bztHIK}g_UITKx}LMw&ubsVSA0p9UyEtE?MK$ohdq7=)<MsFZTl>aG@y~OH>O^%W)m{w;3i<NH7xs<sr0sqsXSxMogrXp@mO^%}Aqy@nhk^N9x>k}@qYTutW0fI_XY&B&JK6)|mW38%^u%CfbOxI>sv~pM=BQ(c(SHy0hZMHCp5pGrm2=vUt%P59s3#X*z@v9p?uWp*di*09<vv}mIc+lP7{Ewfnw?}hDfMZT4wWCBo8AwVFxK<xgec5K#=n6IS>TAO|(4)@+egB0?kK^#qCy%69m`W+bC&ZbS3P@Aol$a8+>%{|sQmb54(ZNN_)$=BXPu$$D7E$3$&vUyezTQBfC7Q*rV!me)AQ#TLbMVS7tQhw|T*~A00gKV}Lyvo8CAk2v9ZotW5wdqO8UbBnpO$6;_%nsnU0^OcxVPl-=_s6r;~fA#pj|Y})+K2Sw2Fn>oup#W*#&yi|L$p)3U65l988mykpj}Djdu{i;BmHnQ4lMe3e*bBT5vu^_&b3Jg_&BM1btFj-kU-IQZ?V}YZ5zxhxhxQ%3l*_L^rqMaTC2y=Nc{fx`(^DzSiR3`hflnDaPW}cc!jGUnMzv(YKd79=E#0NJsQcAbaxspa1D!{>8UmdtuV@bWIwfmwV&P+We<4wghxlSnYi;A$Nb6wK!4y?pdiK;mk$pxB^;{DOC79x^gH+L!5+$yaQixq+o4}F}hkwlbbDvHu6V{A9N+92gVP&YsRIw4_1`kc0aTiP3vH1c<F{fz_n?Xr`e|)BL?T5Tdse{_xo5*n*<vV>+E7jgG2k)Y(}arDmsp+Lb8SW(y}D_j9e9bRzM$>cte&>p=T3i(jriQrE}i!1kmb$mb5IP1?5k>Y?b>_RKZW93h*%q1g?lGT<f;5AGP*jKUVF2i+CRN9^2hv*IUZ>Tl0~HqYA?(T>*6;tU8K3^eJHSBp)?4p;V2TsFmTT#gtQu73e8Y*E;-nLo)$#azG1K7RiI=!DQ}&F;?l~&C{%L{jk}D)?aO%>9<$b6&zH-`s((jGPyvdkxh-KqMXl=?zyzFyLKsDZW2aaR%&k>*7XfY3ym@O+JvfKOPbbu!L(A<XrBDFV3V#+v`=rfLyPbkX%oNSE0fezNt`}P5$zjmf;An)Z(d&Oi%NkH2=$yHHu*E0oxxYGz0&C5va=C3*UoKUy41D3aI^LoX>S|2c8#uH3mK17@){?~<)OYdY=%aM<3Fb_l`~xh*T9i?)~wBQ<DGqZZDDro*XPT`hTlx9)7%jV_@*B;<@P(u+lIk%FP$0H0@HqU#K{#V3lh6_saq>-V#^rUz@^%6&)OSTHc2c9mJyQKzO+$xSC{`@x=LQ&2HtUctL&XQyK@0L<qYM>&IoFM@=O~41S)@3pP(F(?w^{%Kd(+ut*`P9PeOmdnV=Y9^}&RalQz|PS8at`u9!(dMc{6qbFo@`vNljX<>XFocy0Six;S|?;ec<g$kQ|XpMx5fONI85m8zU#vBpwkV3G+=4e*u0de{VU5HdyyYXlgQT@r>F#~kcPHAje$f6`6g>j+`)ccocvCD_~3(ybn+H}-ZkMD^H-M|5O@I2nqtxe}LmvQouL%p;L2?yF?PWX-km%6Uj|7F0Qqvy4J={)~-V6n=jxkH&nEY-vR4;~tz}>xze(?fR<xfwoqNTO#xebzz`O3>1panx*Po;!$|ZGnUXf=7q|DQQG0(iq*^^x&2HNu_2^YYn@b;!ZOdfrw&6=fsjp9)XdAJO)6<Ev?Gc_s4O0{5e@d4W8QH4!mqm(QwZF?T@?i{Sx6CZlQ=X)d=3GT*#dSk?@f^q1gPMki9aPq>}z3;Yimj{JcO+Dz6=2RDjK|8s<TceR`5g&r&LOj@j<|C2CB8{Hh?d^Q?jZmc!K<>XK~EqlFA+N1c*KM8I8uIQ4!t<;ya)wYF#~}Ar-;kfa6p;7)W5`QgZQ<H#W_i;7#({JnN-DI?>1htdVe#F2*I#B1@TSmBeF+<0p;DM%R!$l?7!6fmH<}2yr$is(f9;*Wp42I8LDkSt}Qeapr6A*}DQ;1@+7lIaDS?zx(Feni5SPs`V-8?8_Ig?rx0t74A->IG6AonMc*n_lH=_!$<nF$08Y>Q)PjweQ2IhEecQSvbP2cLblM0N@zgmN|R*(7J0SJft+V)kY=5)plRT06@!rgK*ZPLz;ft$tCVFzTKs8ZfeY*#hGTECPlAxDVATY!a6Dl%6p?@}Xuofjd}^xGHn-sd?(R(}GC`iko8gVJ+0+M}B7X&XEl=w9PIKK_hWIr#_s!KCM9lm(;&$B|S{vyjn@#9y9Zx`cqkwC#7y<Z7ju5Nd5bqbGe2h!MKx`#X#gVC1#2Ddxq0YVp>GsK$XI{ruqWYwn_E6!{;c*MGUFmFCW!|Ptz1{)`o9t;YE*bbJg0kSjF;<`9YBdUHwX<1o_l4uf%$}Kz3KZEAK>}W7V#KV9!HmK*)0FbeO{M`A7Jt+TbzLeBybBJ9F3GxzPD#!Wftwk=aqQ(b#(Cxn6J!EZ;Jm^B#L5c%V4yu8VT!6D*O=omxr5gvpztDABEv?BlVnA$&VWZwWCx#yIc*xlSFwW}zYE-y!ZgEuTctw@7RQIGNCMuHvQfW11Sy|1PR$2>DEgmTmW^ZPOR=_C8+6gd6&jufw_BE^iOt_LA{VsU80idvDUsCVNR$;;@+uOux}2m&Nt;~Cj3Pe5c(`&7@&m;yizdJ=xdDiX6x4>ba$G5&d~`Ym>PpW>9-+wmZXewrx?b+Kjbon=X-K%MP2Kr;g2sQ``1zn5Cw(jVbjtI+t{bk)G`7_Lw-%eje%bbxjaz718U0}BpSwGqB+wOn^o?25;Z1+MIFqB#Y1|m<)tU$D`q#Vmvmf`ry4EHim(K2h?kv=WE+!k7a4uWG^J1ITnVdcQhcA;*>;wRhzip<~J-_?HvdJBB>-_fl=F0QUCw_&I9Vc+{mka?}wgYra<K?HHyR|>ODS^z-?=3}di%4s<jOT$aw>l*`fM{ae>cTvIXU<W&9U2QbJnZAsSU^t3y2^yDu@=}92YBw_O?yKj8R97;sy7Sho~eSUFq-=8oL2ZZA0A7**A3okGsfNi-){5w$8Pv-ld0p;N-8Q&@GO%As5VY8OPoSfgOVb*+y%vDwUI^vDHzxqLvi`Ar24IvrRwf1?^BmYzuVStjaKW`i{bT+I61pH`IE34wkE%M>wis{JR{T@j%9CtY$-`{*}ZuB$!D-j5I}K7W3OcrPQo(_aFzk`B8`f`e!FUkk*PRcINUs3tcqn`!FjVhKgJC9ZZmh|{Do~;I|Ae9<95DEYKR7_F3VoOS>pOeN^1#D^6O=f;`i%2VGAF3VfT!*POQjXKNvbR+g9jz+Hm*heG|hMLyzk*(<uScai+Y6rA_ei95=oIW@~3~E?IS~rV<R*Y8obmHC&n821*E{9))`ClU61Vtp8ZJrf)p%L;cO8_8sWWYi)b-+ig?9WL&NEpy?YJc-uxyozn?9Lj}NR-H^fNoiluET{dFI5Y1okP9I3QF_ZVFuGR<5y-)ekZcj@FnoB?IIq^GS`EFlp)XBsP$5Kt=Xy|sZ96k!y$!O1+4UWcis_Z=#HT#@m4DfzMl$vp#H57LgMM@QOJ_$W4T=dl>L;^pqCQ-smx#C2wmY-#Z@E}QCEpLx}C@H$?DM1mMR^^(7bX@X~4FxS+$lPk+dqKn^JZYV5HKl>~QXheAcuJKpwjrM%yPYI%7n74uL3pQf4AxVysn#sHOAdcGt_1D9WWvLvG!l+ywX#58Wy4RBgG?@Qp&2a27M=nT)r6{T<;0`Taqo&LL)FMU`OI*d2;9&ERn0QbM{tnU$U&aI{G<l}J#2L8UmM-}udQ7==Ae=To)j_$qg)Q5HiW!B@)By!HIY;SYMGQR1)rieOv9qV+e|*BV#47d^Yg(LG>YNv)}kN(u?dn9F)b@NGvw9>SG1<e4dNizj%CGitlVXxDT8Q<gdJ$KKpVQ&XFkAYPu-`_dLoUF#`EibF;HMPkKe~#dfJP&2zJbK(mGf4o->V({a&Ma=;yDsbfX^+Q+{qu;5Tl((bV=d-%RhF!wb{<yD@Z1OqSerTdub@H|Md10S#4L_mwSs#ZX+>`H%xu%FUZ+X5ZWMyR<Leym9zre{F^xW7+Lr-YJ(hcFPPg_u|^jda}P5QzUkEho27LJn6bE#~^||ArO(PgqoAEK19ZuOwx+Ng1lscm#Ml)gIJ`4ZK7Er+)@U=T1&&ys&5*jmVpSWr`ris8;yg6bFj5d+?1VoJOr?qJy6YP$4$cc+jSYDXkTA{b9?9YHL5*a--MpU=db-;+1Xx$6mHWorE8btiXSZ!(hL)@zi3hWAOY9LL!G-J+t+$QnUT?+RrhXJK;KLKwVw9p-2q~f;yu-3)9H+?AJ2Vw$5ljhH(%jv+N1M>MhQ#@bp5E~DC$W}r>ASuoX)nU=k4KR@An_Ouqxx2&-EkR=I5;<dSuvDfxG~>GEu+8ES*r+L#j<k)*8uTHt<0W1+=wiSyJJXweaL&P|EzciiK3MH-RY*3@3pSApI`58l9CfQ`wN09<HtdtOgiT2JUeXEI@@vrBqR@p%Yn5<GwR0hW}!Sa!z`N%kMQyckB0Ncr>hK@U|%(U4(I53ssm=(RrbTmB3-eIG?qA$Zn>lxV34|@=WsI8&|=Nir_9@xo`nHP+@W5JfF#7FYK1R-7~x4QrP;=_EmA_#jRIf`Ln+|0|ZjeJj|_LiQg${k8a>kHfS30)ye-oId^4q@5)XgQnWt6apu2m?`-A?CEL^P3kL`8w;H?6rF#AG8E*(qN${}=roh#T9-_n|(?G#gHUKafwbq$=C!7VMp=6NIIz(fo%<<THRe#m`<xiMb)vV%G6^uO`v1;$!eA>4HjGIM193EldGGCmyalfTi9l8K+j$XUn>gl^fi%P5iT2KC37c!Xk5Zq|P1oHvu;pb6or~kr{cS&FHgT4Xg-sI(%UOjtm^8DoGP00BU!MDjvP;6HxuR_|1f$8j(z3odhAbN5xY^E!lWTjqMX@y38rHvY&!ldtYYPj<ee=+dJ_lyj^kNagG>%P-7HcY+AYvb+z*$_@2+UlDK`SY)ooztgJucoMF+{us=DNY<2DR5j|t~9UOnVR56)u^-0QYq$Zrq*oukPSg4k%&HNirNr~(F(7fr=s$`^B4CnZ62Mr7SEopKuxuq_-7kiIB`#%zP9Aukml%Tsh?nhqxS4V7lH5J_4NfpSvDoY9a;tW{8gj(El}DzG2c%xrN$jv4Q&6>a+db?QE7K)Ivec#4NN~^@mr?rWfy+);$lO^T(ic^JuUQc_|>aByw){>(6>DuDqk;HP87}Ya8HRkx)8fWQI<x|GaGEOFhUssM`y<v<#>4&k^xW!ZqiAL?07+J1Tevj!#E4&M4{<eP(nfLYBYTDl0#)S(<n!6Y*H3#iv<U56%MED>a^xybv_$rq#k>HyBD!o;&_iN(D<dAb%p{iCR&=mglFYe2l#U`%Mz(u#Aza1D7ZATxkp{r(A?DePp#>_KE?XrAO_OM)!aOG)5x{HGoM2{H6+nK!C%IagK6gYX2)yPlfC|=8=ciR4&NDQT1NeWH(Ph-!~V-=nn!P~E%p<)J;NQ2IMZL*TG96oTBw!H#q1xxeMD)xc=a-+liyH}MWSr1K6=Y&)~E6`NwOFvcoI5t<yBT(L<SEkL<NZJRgo$(uX!ZqICX?f=#5D)KDYbi@4K_3Yo^2&x0bgae>&5i+>7GTKAb$T&&MgeyM2nNoxXE_*!LTa=lhd!L(4lockE1m`IuJZM)y@DWH+bm#Y4Y&BLdw~`o#>LS_f-8*`BG77|j=GzGQ%)Zx2kyGZevXeZza>Nkz+e)Sgs$3Am2agf63L^jj!27fKspAy!)1n4q*cEKR{*Ed(mM7%2i(Tu3oHW1et$>5;pjweIs)!Z|k1TQRSMSrK14c6qxt!27S5)$$IWXOcR>N(SewDb)B9v&h<~qJRiOs@{Ygg%Fhy+pUwp_*n;Ac{x#-fcik37d})fdE~NK%K%=edd&n=3}mr-TPU(qosdaJO-=hDdaAKL!X)uFL;UvOEp_^+QCS}!d&a>NrKGmnE1K_YyyFqtIa-%fO)3>YqlA|OGc-@wGh8AwF3KVlXY1h@=)q{<)JL;s`J*BFJ>?hw(IC&Cb=tWfVAj6yqki44&A_N*TCE_xrK#F9bc$JFIOi?|4rN87m5XrIWv*n+%GsPsgx9J)abYFtz0k+sPOF!+)5@js#t|R=lsTczn%r8|V9UNrgDrc!4YvBg23vN#4Yq;msU6eUKS&4E91*0k;3ZSsNW*v*uyhitu?PeErk$x;C2n(JjN9Lt;qP%HZgvyjv65{NrVkP>9sl06<FA_J5w}j0jc#|N<!-=_XBOfbFVB-Iy!i6^LhM1##vaC=yIcq+yu3Q}pcZ2Z=jY=p)7QiTyMy=hA32I)%!VL_u|P|2#4HjxoS%Q&bcoPP3%_}B^*Y*>!HwxP!*9-=&fj_C%D{DTZn05q=uSP~QE4q}Rns_L(MT(rBOqfbf=ibxuRc(By9C-$)%zG-N&$+JF_Em5A|*+4bLR&N%XPk7&FK|=YSNWZeeqNiYT!0IoNE~XcLK{&^THejE3;r(<KzSP5m8>`lsp9vY}?;v`_%r+&c^P=t?+uF!SIv+wz0W6c{A*`v%V(H;h+Ig#5G2v_^3(k68*rPbzi0HtQ!-5=CJOwWvyn`<XWngUL_SP<;|k95`RDwC`-96Cv&z!nj{p5kz`O0EN4oHSytQhPE*`6MwgAgV25>~HQi&EVb;p|sjgY*MUz<Gt>J@+-R;8m&eii5u7n+`Ve4iH1>8V|^4~rGowg3=@BiT;81T9kd=kBEfYT-jU{I=?C2D4AZK_;A>_RZy8Tj9lNuf+hzG;28#Y&}Pwvi|POt<IU=rKq;War*7_u_;IM<znLVQp!Wo;H!_wrb)v2nMhW5ii`S>J-EuE6$2EFo!ti%qI7&aYEQfZ8&Zl@}}(;kAn>Y!tJP~I8cCAR)Zf(HUg_vICY}yQn{pVRb2}l70Qd2Bw-RP6p!U|%3av?6@;UF)wFmIonCWd$=jf;f~}Ddhd>POs-^^YsNhIKEF$-U11;$QDC7js3Wg`uXOV;9sGO$RM6zVsS)D|Q#)d3eE)t>(1+Tp@&_GE5a4LBFwGciORN4u*+E?*d+l%?}w--m^Sywz7Rv`&K0|R>&AtFR&9XZ@KRx=cV3_<XqfIf=B`Gp^CViD=EBE9>4vi*AN;m%aZz29|4HKDpCov@QWIyA1kX(O9`ET|1Ww7&Pf;rgx|XDGe7HDWUUVm8;^1xlwBo6}0CTjROp?Tam!^7Zw}+|5s2{&r6I2YqCd9{bdg&A&Aiy-nXRvqHy2%@|A444Q>goeF$=;FszQEUXdmgLs6bJ8LzJK{CFYD9LsNBdL~?g_A(dXk;Ih1awwXjz>N)a`IM$B@Yw1p15<gTMPYqBMBTTC~gfEO+?=MsG&z3ISI_4X=qTg5>`0|UL`SBLsKiL=}0BOH42l0Eecy^tf{UzqB=R8%IfCb=cwo7PE#Loqs#f+`0|Bg&)_A&q>eB*EaC|z)K6x>A<(Uc*tW@B$^>iawXK%gw4tJs4$;$qdsm(!b7U9ZyfNuzpf7uQz5$Q-hOU_(4tWUt_5~^-zP;j|8@rRu?KdZT8x6VmBM(b2ZNQ*8Kht=@x$W;hC+zO|7sKC$o%1i!fboH_@%1YgujJ5<LBG5S$vTs^zq<?cic8ma3veu5KBhtZRzJ0E_swv*r9%vFl%}eOV9{E`U5STTqlbaj&wCelzuL%i_z3W{)G^FF$4dSF?xb0#$%WF!ajs5kEAc0V0+Gi4!9TZ#qV+VRN{33MZ4rP4E;@OnEL&haD-^CJs)Gc50>x5@gyt}F-M5(yxtVSfD%rFpt?BPH7v7x=Z=ARAKWc6FwXTVb-g<vXKzBn(hS{9cIZG=m!Io9$PMhY2>94Js|ERsg_r`PE`Y7<7;k$+wyX$?r^xC+i>Uuxpk}?{1+7`f>B<!iSp8dzeIN2{6){=X}OxI~|-B(@RVVu<W#vOSN`k6yt^kW+Cjlj!nvh?#M8H&|4;ERdcam{nrkTz4F_vEvz5Nv5JgTV_WLc|$^XPW`U;ZhZ0gLj^=O3W!ZYW@o{I~j<5$$P_HJh53;J^&hK^a#%2f5xqvVAauM_1pjclCyTS6E#!%IYS}6e6VVJ$+B}eW*UD~9F($fMkoz)&nxpeX_bqu+$eaiqN*r!F0NMZXsWsZJ{3B8X;f4Y4t*F7I=hv3wm0&zC)`qjOI3@N9!Q2;1B9KMp;lRGg5osl%0?jqRJPT)JX=o_*OaqS=Y{ZBg6R!I_2=b=NBTTm@GJBj+<Va&Ug9R+yV_Jr(^`Z}yTAb%@_V5xW2IzT|8yHow>ivf*ue_yp6@R2E?JfGnziaXyBoWw+c!2a<d5rf`bGAfe(8@!-%6i&e&acLwz<!P%<10M%Pr+b`yAV~!u@Yf-dEm_tc&%u?%Wv25*#5{bkI6E<|MW$7hOb&o?^nLFlCDtDL1g3ns!eTKTVE2K`~}Z%_UwSX^I97CAdP>#~N$O@bGegcfo8l!ZNB1V9<^tS*H%qe3RqUO!WHxP;x!_qkeKkH^6{?dF+R(DC$nZCPtcv$KW&6R5?5h$(&XNx>NC7gJqfrZL#06YR*dW<malxCQA-Z;3%fhp_O^8(-Qc6G&$%!WB2ltv0-jQ5I^Z1$s5f@HwP_zXQ+evd^ErQMyq)5&USoGuf5mz77T5E)A8N+NA+R1hH(QO;Y^2V+CHZDriBc5r<HvD#O*HzATV?&bj6`7?`(h-KW%#z1`P9{hZcR4`f&gXsic5|i63|+UPc)U$&txCLGQ|1lHw*sXn|-$RS+$joeUQ<zz^4&pHx6$2SDM4l6-_H;2cQ4s&4CUUpdj?h%wQ1%N22>#gNms&~86}|EN;@`;+lux%QBU<xw$3anKYo2byCHlmIu$Cs&~~m<}pA*xQ`FF-oYM6x3ykP#U~wr&+No9)UsRZquiDcO`=eq51dwX|ta-hizu?=%gQP|Mq10=M5#FziPt>Ez9_PrRVagGoSS3m)A!;<u~6QIY#qQ(YAZ)XQPTI`b`@@noh^KJ5>11xh9<=cwwH7WL!esZv5T8O0?|+{QP0~WUv^&9)#ptf7#tdQubCC{=9E?{_zk9et+aK-W`0tFXkMdJJV*;S?;cDZLiUI(r>HAemN}6K+^T=`Op4xsL!75j%*&gp8z$5?6YdSDbN07WP$0s4w5PQuFg&UcvN{izfwQz3yG(J^t+=F>4!5jkEY`C(L5blw>3W+7VLXda<hJ!zwGC0P38Sp^WEjkPC~ym-EG|(+0hk3H5n>Td#<*&`>U;8T-er|!-{@360KkNYO!mYn`xhKA8lwnzCV<u_QRLPt8#>W>tZK|+~|C(-s&4)u|MNoIP6Sb8_DCncENu+$2)XqR{p!l%#7wcOiZUIe{#^#KOW*CAACs?;8~VZs?OE3#`#Q1r((0tA;N9TF%cguHCt<H>0>on8>BoDW30rKc!-Oc0&ea+Uqw9Bs?;%4#I084JcJc{3PfSm#^QwX!a1QusXVx*<i0TDjFwdfcs{DY-b-7VhR2jbQ2398&+U#-0{s2{u5WdQf!5&{PosXfI<Bdn*+(q}=PHEEn~4BYb6&WjKJ08I#8DWyuZblAB<LbQ4oglxg_4dpAh?-In96t4=9OK_iNme$=X@TKH-4CyRPx0@xeH1gET3zo&oFgHnsz4F+FuUEw0moPeJ>5LI%-`=A5-h**?u|5_ML%C<1ti=l$wQxOB<;T$y*+1li|1O>|?4xu+@8+JvS;q$f;u5GG0ZA@aAo_ZV7Ass&bKr{$S~V+N^0hA7@e;$Ww^Had_3f;vS6kMR*(|5+$6ps0&V&$LlgD+%6UHE@Q1v+ZVQ)!d%LC^;I03S0@k#nwmwaq^*+5?k<fyyS#ffUEJQ>zCdUjzc0BRi-xyOO9w-jfNFM$CadjT#OqWpvawe#oq*bs6GdJB6lzOqNN}cUU`;esLgTfFJX1pyeixF5p3>2#lHq4_rB#BShJ10JE4SI6r1)Bm#3xxDP5~=P=ft>AHpDEFb2QZ^=2Rpc%4!7FuS-`V(cSS(BpPg?<+Ph@T>kdT29zTG`uv4JH@>%XrJP#qNPTBCTJ-JSaUrAnar4`c2e0CT|G4=-dw-o;I7eNn?6fw@Tdy4BG#tj+`94&3T5)cTvP{E#N?3q}md~$pBcrZQcG_EGm1=4}4eXVfoiKYuTiPun%*utW$Fj{iIvul0S%!vW68KJWs9S8Ku)NAdW34R?t5uHEBjju%1s|2j)*gZ8dps=mUc)%j5k%kbDB2eZzNgbc+uq2J#!Zd`k467k9g+6^-ru=CG{unzIeGr#mH5JB8jB<2HIRrc9}T)d=sge@f4Tfi4<qpA4?*C4>Ji`fq=~;DZc|1Z{8)BmDa-fbwv6TKcbnL|R)Iro0c2^gj#APSu=Ork9CcNIive^bjS12=$>bgBL8&O`Q!7tLatrSCjb=AT!Jsx+|JJcrSk<Zv3p@v=YbupNMP;25ph>pCjfy4#nubfcaAJ5;7Rn50PH`);;zyx_ApL&72|>QqKX6AKaNMp?VLZYj%UYt9EQG?P0N{v@=VB9_mF%c;MsgP662=608+;5hGE?Bzm?J&yKQ=Lqjy>wQ_p=a@wfDjsng_(G43N7B_ciijyp`BzMq3$}uvz(l&G(FkE{l>J?D<J<)|{ob0$#ZJ&oc~Fx0_PUoB!NL)^4nB;`IJMn>bg<4>n5Byh-D5DtyAfDhKSOWZdHHt4w1gEo){35^=gpP6q%*LOME~AS_;RfAFlFkwKQK1h94tA?W~&27fqu!ILssSW#Hhwgx|>D8VfLE{wE?WpX}Vm09UZKk?_U<_S7hyyw)+o7$M}-3CM78)EBInz6N(>t)(B{(hgcY5(%xTx6U+s*T?M>%gh-{U%)d*}y&j#fahZtx1=G=|+p(Usb_#9D!8<VGIH0hWe7E&!S30e3IT!AhkdOIZm^)@T;K56BTMC!=ntb3Tccy34v8<`oO9(aN@<d2whX7@ruyu3|If%4n9u%)bI5bT<@-ugtwoJFia^lmWjn9Mq~@;lBk|}mbT@gU<_uO!M;?_l{d_rRBf^Xc0`yYW3kTW%7UgqyE;L98ZfVrMO|fuYh065E-9yySG=;y9aA)k8V;?@6}mMCSs7F(1U7Ogp{$QO9_C+qEOqB8uxkMeJBmhWY#1<c&{jKcqcF~3f0C^UUb~=_IJVOQVt9kw!}%?ahrsg@66K!`!P3!c^u~I9Wh!}`xc={rSY99cdS)h7bcu)dTmE>mKQq#DeKNiWyx2M^yWQ{G!D(N9t=HVcPj{$*fB0$UDGpMVoiWc-Yd&?6a9al}NgEsm*`@Rq;FV^nCF)lo16NZ5uoo`N#4=Arnl3y9GZJMr;rPN+I!sB<-%$JGOz`RGYIGc!9G;pK=H}?x+b3M6P-mi%>(Z=W_UnJ^`T90rouGPrLh^{Z4Q@t=#0V%e;IcEJnTB0`aXGuB6?9+n#nvJX97$4pK-i)hZ51!vKVx0BiXWk?_Rk1ubH&)?EUvl=EwlTZ|MBznHdFF%V-lz9qprLgoprkL#rlNX=T5b=Ou7K$0oMxAT>FE$Z60Nbn>^k6#IWZEim)GdF_`97Upe<*TTkPY#?pQ|xExJw3`xiRapwEhgfDese)7V&;qb%9kO)%n*qNlkHUV%qE332frUnusP4g<1$Z*-MckpU+f_J5@k(vPpXDEkxP6)-<8?vsIZqu?vP=*B%&N8Q5$PU7tWO3ecBB>y0Hy)r(8tN~m(I>>S$f~7w(aLdS<+r(U7|^A_DOZ6xI3qqQcq2Lr@|sjIF~=0K8P+AhRf-ymtN<C9*Qt~s{4<{1DH1LfY=5=7QtHYBz}u{oN}QhE%fJ)&0f>M}17EY5h*=Y4hy&p3Bk+yRRt$|8E4$#pJAGZj-D3|oNv8}efyd}9O%W$XRnALgRskJ(IGip<sSP9c_kz<I0}lmmHO;H%hX&WUXX^N$DRPx+rkN}z0fD<J6c3qa2HlgbG1Vz6xUHU|hABv36I%ffs}#l_p@Mu|;`X37!w-%f-YO*|v#`De(7X_@l@}s}B5cah2<>%DT*QFw^+j3FUDd>k<{gplh~G*!E4hr#3Th*7U?2CIQ2v-+ywO&DQ^-6<ko!vQE3+Ia%-O_U=ze^Dv$y*LQmq?hGxt0#2ir&y6pUg>iU=E|QJO&s0U{S{T7`Rt?Tq*f;828{l8DksGA>{;xh>e#eOZC5YD~;Y&#m^U>9`Sq(Lq4qqpY5>0M#W!p!rKI$D9+l(x=Ew())0YjG~_6ihqI4Sc)kmQzALCMCAlcD0LMkL+RxLaVoe48(H=SRyEICX|wLJfa}ujE4j}7a)WL8KKJymSYfwCu+Bl(7*{+q+`yrOnr1NA)^Opx78%aB#|juF08p_SD#!#eP0Xmw_l;+<(s;{|sO}vyCQ8i!9wjhKJlWKPZX2c&d^F8{r~tYUL<dC(Dcct`6C}`dk^qb7g>=Mm0`Vk#ki8~{g*ux6oi#4P>Iq(Bb_K>EC}$Uu!)rhRO73K;<sqdqxd79+JAoAvr<&prf(c5s#XghEzyw1-@Kkuml_$I5YFk0l&egnBw)PqtDw;@8+K>D51`Pw8{OQ(Sxj>k0^6DOq+qrO+$g1eOpB7n_<S-=xT3bkfeWjp1jMEw5-%BeTjqNZFYRFZ9v7{s=v$dCww3;@K?L$C34L;9s7$tTa5Duc6u{q9Gty$ulL^#-_i_`*MsII{?&#A}}0CPVU521z><Cws^Nlt4=iI|{qGz_pSM4T&$`?JMlEdqlM#k+CG492h?K$j){lP;?<RVr5MQ;=Rk<5#a4(@|5}PI0HmEl85wPLW(Oq4HuAw$4B(0wBmyOmfTNKAM;G{-{HnQZdA2aQrsfaTB<KXc~AK3?l7|rn*~IV5>_?FxW2B!{RqS;8$d-4J>#8!a7L<7x5@HCs&C?*vbNr0Jb@BI<!VZw<a$foej>ChucQ}6)!+m00mH=CZDKB@s=x~mRPxgYKWe4fy@GF2ooHE4hDQ50U~t_u4EmmjPL@ZNFI*iSFVBjS>FLc^JY%B(tEmb)Vq-cAGLULTJK~ybR2wTt%UJPFP1w=A)vE>8#v6OXL6z3>8Rg1TiP^RMI}!{X;V?yCgO8*unlm*aBsxt#2J}v3-k}e4(y~1KtIgBvJzmx2F+l<W7hSkvFp35D#GWc7j+HmZ58?PE0_xZjAPe*=9HgEsU{dc6BL1*Qi##k>XfaF6=(+jP30*FB71~u#7<H+R6~3Vlzt2hE_Vc2&B0+K6_x1%>4pTW4dM30QkiJjtF;N(&io(t-n_T3>pU0zDjNlI8=<7M=0Sw-fitwH=cH+GnxH6(f>@)(n1f4Fc9g3FNXfO`z)gFATw1N-)^*KDtSrldE+EVQNWeK?B=9G>@4MGtd+*;wQnE~qMJoTe$F<k+uIHVeS9>XWIF`8{Subub&TE^n#P*kcIZBK9ywllB)uukB<(JiGwcqEL-0G>kN<_`|ntJfwWRVM;H{&Pt#V_u(W`mo}{HFIQbf5R-p6<`4&`mh#j!}qf#Ok#-zT-X2k{+1_$BA<XQ=hyyDi=7Ud>z!R%WLJ)*yPhyAbr$xSe5z;Lq~o#E?+vbhn_@2ZAus~f>in0f&K)_Y7`o_ax6oWJtz`3GGa42_(1=K4IOquMtBewc63k8H=IyAfc@1JFLVUW2tvo#4sMCJlRrDfQFIoxt#ds=k+y(w^>IACk1&zu<<0b^UoV$ngAa>?w%DgjzUJL3f=^;HJ+9A0vJ?a860|LI?2(UvnPfC|RvU`C1eQ{a-h05jA%fiHmSsVx)l%D+h)#Skz>K-X7nxlS$&~pii~}4#UAEHG@xo~rFlh!ApRk_JpuTd3N_mD_Igh~l(d{zUv&@A@P+<jzR6+-XZfHO?)Brmu4J7hpJX%5+qisd#mYOsRYX}$9RuO)dEpT1v1s0G5Xhr}8V=^ijoGg-#jzBwnXjRahxrs=2$THTiM~8{Bh9T`Ubup<U><=((O&Of^i->@B8c@R|rlfrD!ZtMe{3ed@=Gq^2jMvMfA#896S30jo$EW6EEVhWSwE<|Ljd0!oiFf{KnHRo6<(|G$?okYKtcgyLn+3~Hq5X>xm6bQ39b#=W)*2oOdeuY({e>hJoCvU8CO~)3l42-5Y6ylajg>=CSC`09=cD7sm;}qoaafHR5I2Z;8<Y>ifpVK7rYo|i$L9FMt+(78&G&w~^#&}da2WX_E`DSk++M#3^SRktkIUi0_BNbfb~!s(;%(AdOL4!k1=nq^{TxXJaY<ul0W4Q^m$h_)B>@h8oIZ-u5ra(Eb~7{sdcTC1Zl2jpZ|^L_vRK~0AhjyQch1D_Gmy1$cV{^#Oq$qUUfWzwZ^F^H*ET??#I4QcjKg~7(xc{4QaO`Hd2u?AvdYT8Tm#|dn*R!F{hQ6($jG<_hzo?_4tymv7Ja|F&BUR@X(Id97qZHj&%FNqAN=^ctFQm?hd+7!2dh8&{$Kq0kA4h)eQ))T|8n)OzrV`qfBVS~{u;ZyYuS@#kF34X)Z3O}cYUKD`y%MEJ66iT|5g2fXN*xYDef|Z4cKT$;J?`ii&VgoP!K{=k`&?pP0XlM27<K_A~B3OB7!uI9bHUuZ^S}$w;75cwV`cKx?<v2mjM?4&t~8~=(sF80){Cw=fhf{a8yb8`c!j&+ec%5b@^YL9$I(o42!S+FSF$mkB{J=y%LEqPe876ja5n#qH986&q`EDXS_)QfHlp^W}~dqCC^?f!gv7GP7OJ#T9m@N$>_F_E_%Z#l9xp#E&3V=#tOsSgi{1V-Y7GHg4#u3-^tu6OD&y)h{<$h(xHJw{Fre>lznTQtM#Bz$s871^x?QJcR$dxaYdJ!uP*=dU%73N^_%C;_D}fg|M`z*T;sFl;4F6@jEX&Wzsrf69bsb^S>6$$zE_ExhBBdddNPzNlf|@)s`tBn<aleVPTpyQA(8l@`5TN=r}XzXZmSaLRctUzZR@_A;*%WmWoBch_D+p?T=v^W6pP@8rD=zIj2Gx0#8L%NNm9v_0~}?DN2*0Dy%eCDiHJmSN(C=zjD`sV&?KpBs4-L;pQm*Xa&|%36knu!a8P9|39w2u2gruTe-kyAHVYORQeS5P4HCVj0RAKuGUlL&0)%9lH)=8DZ^J1o+(#v3xg05(E&cj>q{No7OHlyGM30LHeJLbkK>gOq!T(9;gQt;M4){NTuI`j(=yP$5iXc>mgL-aA6Ew1m0xbf)1^QjMVQ1rBtIpLw=*8S>sp2xBe_|K{hAVEkH&=JJKsU}gAp%>yu(<}q(>-+O&>Wjdc#my+Vm4*y5RUaD2!M1;9dwt?Uhp8J%L=34(aRt~(=~0{IC@@##Qb-wL`Ot;L~$E2DsD5JK6Y)Jdm|FI>wO^RJ4Y3|A&*?>rrTg29Fgz&giG8c<+)>m5-u{6#yFBlMow&<BureHFp9&(1PZDQ-jXO&%sydIC0uFDqtUCHHfC~~Ue&K{@1CD!RNZtxhd)*)&K{3UHEniQn}OCH&{6u?={L{6dFD@d*4@Up*mqa|yZiat>JQtOJS8qw6re$2`49`M0E7aOuo%2cXjnsmp;Q)@8k&hxnro9}ve5AqfMnoSzA&@@gJQKk;i<2=3R*`R0GN4(wS@tJVR=sqJ6=Y7#$nsZm=nvH$MIpXEqbms2<|M7!CO##Erpo#lgdZd*FU{n66uG%rB9v0awFO&*nbI34`Q&i1@~UUsPiOo3r<dn6aza}hC1d`{@u=T63L%v_M{fsgEAyl)Ixbmr3=zhjkR!uMogL<4_GmeQpGX`GdWC=p%E!0XP8XpM`9F9|A}Xc!@jLykqQPVDQk?`6)7o30ReE({e)u9IIA+%KEjSha-uv$k9O^Br8~0d2tC8ti3hXlkdfEeVzm?z0r;iN0NyFaV}QA+g?8Bj6r>?X%qD~$h|JqyL<0GhAF<F}*xuUN0ZG5Mwb6;9jpPu;f9)c3k1BmW@2Mz1E{>#9T;PLdTzhDL`l7wU)_1}(%Lv%62}@A`Vn?28%>f1*n`2NqIHM);mYd84Xqs4aSO(`s%no^ieEizsBzpsQKIZn{>y$ZA&cFS`t?jpQD7H^cgp)9(jnq`I@~@o}Pom`lt^ervOoWiA`jY-t&p-%?aXcc|7-on|RBt@@%y8pF<S_-VJo7HFz#Mf3#*;d9^@bo|eVOT4re{9esSUn;tQvy%N9uQBODy!abFzXuYvd{#UENx7OCRc4<0&VNhHuo<KsuAmi6DVR609Ulk|cHZ=raW^SbFahH*hBU2%D~*$qJ(OteCIeibb)-SrLt1vH(xV`m_MHz2t!9l!i?(iVzg;5f+pn8nm#=d+jwzFd2->&OVPhLB~{}GlNO0NK7(-@XDY)b1WfX4eYT1bwO$HptBlKPNE5zH3J)LN@y}8K8e=(`s8wg$jRpfnJ8iTx`?<S!Sn+u0U8*_t_I}j*P!$Ql<+D?N0|d@0cw|{sx3+Y;NxW}GTKEZ9sK^rXHOzeFfdef17nIxW8s5|4;-WEod8tBBIsg@BKe0D*1_bXuz0P8TQbr&#@X_s3u;G;Mac#jVFp?rk}82Z0qEGnz79%=1LZ^IoRPXAxhN@I7j>4&z^=}5pim)9ud0i_sMST2Nth={LEjGvHVWss#^iNQh=U{EX#jsBatYg6($0z)ostRU1EC-@(sNz#`PxwT^h8TNL#JnBVZKQ&u&!^mmSFiH0h((xpzed>4_XcggW!S<Oc8|KG<l?)1vz=Nvbs0Q<m*eTo;)R%$vzQ@Q4tu(%RvfaihWR6`z|}Cd@>po2x&QNv1oE7(MbbK4LK9|G14PGs4>cg*6g?KkB1X10>8iU(TRVz6_}dY5>Nv2gB1lr1WhOk;JNfksgNYrN3#_3ashbo#9_Sw_HdHPW3Z6i9l1|S@Zcv+jg##naZg;Mhe_-JG=u<y4CAdu0!@TEkRj5lNHeJR60jCepjE<k8UtrIBh=*(v~ov(^t+WYzjFe^AA2YNl|g|D2`UvwU#h0YYtdr(2yk2EEYSzcV0(zLo?#w)nAd=R9EX$PI8%B~;fIr-zo0Qtn5WbLCidE>B%Bh#0Bq4Hri_IZ3<AvIGEEd#t09Im7-1Oh<7AHan%9HkU+jyI+-|DVz0<}{_QNqgpGIm8e3*vmN<u^ee(k*CGH2(MH$?lE(c8yheRKf{&7ebw*8XW$y}i*Eow?b3yZQ{8jnv{oK*lT0&elfDuD8(u^4;7C+iS%Y*SxvCwz=Ce>@@W$pPo7_((3#8m~ux6sR6jtxQ23ZhH^Rif+m8*54$Uk-vTJtlFG+XP$dB0V})3q)4KJnw1HZvf}c5;cKbij)$(7~;%kf(OXUzY75=UUO~J+*b}C@5vCiYNxj#(j9AdPlGvZ(b*xKe>0PX>B73T7UMCi*GHW7Qay+8G(wZ9gJNo`!j5m?(y+ow0v?wQxO&q0eyY%eETXpsH%o2a8JR0pF+JcmmS`gFH9Yx-;#Blir2Jvj;5qZVk-=L(tTK->_lv<)h1Mkl?JDykH{0|1+;m!RMn#<ew|Pi20r{iFo*{Hlgr6tB}k%czDtZ})2*mQnTG6BbAY5FyBnl!yVXJqnS$380N*3cd4$1tDBWgjnni5h-U1$v**IP}qZB#!&o4r79>n7URJqYxf*sUPecggU%G0I13vBu^`MnpueCEao(dXgn5}^3P(qfXogbQ0HEuq?S15I+tNAey&((qd*eB`!ZQg_8gq0s0XP|PtI>eUcqY~vC71@~RYxRI#^4dmN7qtQ==S-c^fijU4W2)@?cRc}cv`{k$uLkuElg}>9JMfMV80c@W|`1m$#Mkb6&P&D6uy))s;Og~BdID!{`yzJ?neolq*InydC&@Ur7}hV+fv~*2SLCk&lq1AxH14|T?UXF1t%eKxJ8gOSI|HUHr949dU&<_{E}nQqn9pKt*&Mi7a^oKcULij;?1;)tmNw#R|j~|6XLaNdt#-xNhsTKcArCpfE%1&TQ31jILA90v_Jdq|FzWo8HQK?;ZC#R)>;!k%w=4Rg|gy1-@_QeV#Pv)bbar1^XHqXaqDIGOy%~bB)YW_0t3HV<_dJ{bv048Q}^T+D($$9w44D7tY7S5!%DOJ_ErNZDQ!2S&u`)uwyRr9h(boqEU(7KHr?!Q8#t_}u`_M)I@=j}=lW|$;(Q<^#N%yyO5WY+b@<*6W(^$+zox>&C$Bssc!?6IaoH7+Eo0O;%*ID=%Q!;*kcLbI(x=Tcm?pS$rcD`|qJi!kxAS-n^6}_y>909c0_@?5yU>`9le!O?o_hTUGq0~R=;#g^ChfTdD6~y_7){*zpq*l`U9c_=@6|)n6WUK(l%B9m+cW_5oYwrcZsxazAx7Of^=Z@7@NW+jr}M?p6GLmBM5;jvj@mN-Xg+$uNX8P*uuo|jC1FW&lR$I=C@@q3ZiqBWPy~ZY^jS(;_aX@56#4;TY-x%B26`VT8iZ{i*#LB8F$g9xl*R<BZAPwL=}hAGX!QNwh=bt$YPGu8(Vf)V8voSS?JALAeJRvxKxhNEPaKtdW2km6FdvA}#CY`KD1jMOXbYlkfE^qOa7_-;YX?KGjFl$G$08^A<}$;)O-ec#K><G;ntix}#(IM_>jb8TN|>M&5Tz5K%fldNfb&9GsU%P0xwij*P{Z#(D5bSe*b4xQi38rEpua^aab<adwNF?AB|?*kg2ckq5e#rImbsQeG95|E7_&}Kpz9wqBJ)wl;oX*Y_N1LKKI@PjCL3fHvI1Q?QOdAVShS$D$z)Yh-hu9>NQmg4)lz0ju>^-y{N(F(h&lB-*xc}z-;AnB-<?WsspVjAythEfR9~k+2Ys>yI>ey`I_Sw3=m-iHp|(_F@jv0Eu^cny6^x~~AUKsU+-8_1gi$0kO597>3w%^b_^6+QYMyiHqw8?(xZPcNVN4qr+hO|YmhX{!hhTfq`d{$J>RNnFGgwbgo`9O0;}cMeW9E%XbqN3TU;g+IzUe~RvVd{=S2cmEr^ms)>}stR{r;>4s%rTg?x$Nvo1Z)(0VYGRFi9j8iD{m#X)}Wo7CbgBaf4%&%g|;wQOq3$=)x3<CW$#D)N&CC^`iI^tzj-w44=Xr)aW!(lsNztG!ve*rc&Dks0rSVq-sPffEAf%7Go4rky-oGWMTIaB@eQ*Ty71wZC!RW)9>!2^$c2d+zJ;^*++x<M#~oi>T|$NBfyL<=xQrBZGy;7lio_3&D)@DV|~(gjFGW2oZkXLvV)0!Zhif$%m4C<$NH&<sD#8NWg=MRF>XL{s<|{6yhu}u*24B`62+`HP*5lmn@8Buk-C*7kk&`4A~*U3$KRFM>rZQd=-m_7EYSDpDMb#CY$?x3+>tH(%RRC@Cm*eMrVZE%pnoWnX(BQwMiM8Ay*QY#<7F!DVT^(w*T@;_IG!+OjGpTlL6Xwoh!+XwA_z1^C5srr0YFZ+6``#~;*1Q!jY`e~fpwV-nbDSJlGZldddFhXFxnKHH#Q&*32T6F)=>-G-d@8^fm&u)@$S`fqS$E`g~$%%S>MI0n?LIw)BFgAZ5z!!&FcX3@Sz~YLhET{-*%_|U_lO?kL&L3ozu{D@ZXMu>kRrSTkrBR754P`-Hr8QBn8C;VhJAdL^IzKwaPk0l4o9V&V<S?B7ZWB7O^}4?y}r)tR@K}1UYsMY*}7mK4|pE5FjO5^meHZHc*0{xb#ReMgw!vNDb11c@qJDG0ABl03uVefC2~&Ou7(6<Sqz@TyRWlWpj*2eW+|XtWGxGCK*tFK_6D$CJ%ZSpbQ53FlER)z%`7ZH<r4D$wVOe2_uFZrGqgnxD;NXtVINjSBf#L1ws$H>P9D<6O=F{o@0k+127~AVTjTh@JGynz*O@wgBIkR)jHEGB@A57S||(~rT$qmai+upMn%kTG{}`skQNI$;2lv`fO4J)k~m|&c)-$U5rg+IEs`RSZaS`5xYdj>?A10{?_pnG`ou*<_V5Al6)+Z9g3$v|($-3qGjgpttO$?ZY+?X(shD(1NRrD851?qwT0JHTVgIaot@-NjuZiZ5@ISm1rb9uv)cmaBaFm|;86WB=c9e%5nc|!}))Jqxs8sb1Y+&e5TR)dPY+0fC*_n$7y+PT9?aS>Vs>dv9+~}G%kOqwhbBa!PPrzjkdQKb8v&vTa*Qh;xSU>!)|F84|{P(bqm{UHw_C{GCZToTPF=2e!?d+_b11P`T@^8BJWtfd?&>s95F8H-@sFc73)mL<$nq<d){$tI>XgrZjdd@hYSHrPniS@=P<F&C^-T}nA15MHe!*pgUhYXvTQVOfyJq3fxhJ#+f&k5_$CFo9Sb@yYkJsvAH?c;Vb^kvGgb!prkBh2385w1iP6sY`IW7=nGwe>n;<qfL?;90GNpnzV{ybp)62goW^qJavUEs1oN?6Hk9n`a~7YJrVF3fLVoXl_{T7|?Ri3JHsZ0?m%dNRs35I-am}p+K_%HK5P}vZjtk<@uZ(l;&**9o%eet)~D}7L5UM&DAj~3}&c|=Cln<vpf6@B%wV4=Emnb2`vLo6<Fg!D+xf<@TjGUpwtqWl|pLJX0@|13-mw=-j*5gA$sEs;hdg0Q57Y}TDf~87uBuG<kUGiKkn$YuU8)I{ocp9*OqDj=yA!En8)}?SThpcLNs%kv>{r%>=}lAu^5<0Q6qyTbT<Qmi$z2{T9vqNt^HHsXNyYOELzz~2oSQ_dFMIIY_CC;z_1`#x(RTGHG+F?g2@p8It7GCsifR6%L!sG^2EuMj%inW0x@OZB$-mVKddK*OL`(X{`ls&C&`}#LRpiXF?&3N(DGDAgV_M6M<)ZKi2+Phuz?d7UDldKO!jH>iluQ}zBIw8U06xYR>knOVniw47nCk<qNS$B2o8bztF6bzmFn1!;+#2l3=ZAdJ&GdqYq=vpV@eiua*(uISipB62w;&b3EE+DSy(Qqj0Vdwhm5&~USb}U)k1R$8si*y!~$`&KpIhuV6<LEBYgsehQKs}C9I>?ifGO?$UMymoJT4o*aLDxJ6=vIVEVuUTIp8$qp7ZwN$4v8xPe1^;mzI3BU=W(sh!+2kCqU<HjcJ+EWKYbe@9jMPV29ISfd=Sm(63$(AVL?p^VnsTib7xB!y9=WE0M(@J5X?Z0Ay!1%i%`l7HKoBARX%Hgy(0t;pUd;N5r?*?XwecA&x&PN+upDoCRv*EC3}Llz+Gtg~8i*w;LVZjQl($Up-y-r{oA$0$E?tF7U2dvW=Zj*4}!Qm_7cRKluKbN*s5kyKadLC4ZD)JOPasIt%|WfW4(8&`^o-LdENH5E#bSjVmToA&QMfUG2l-s<1gXQb`-s@Ax)!+r^zXy$)!tJ=5k|DY`0g6^)AN0%(V{l3)By;^+b!@AlZ^gKXcc5JoXCtojZ|F9&+bvMqIU%%5cg!TgD!MF=QZ_kyC(f!g_2gUpV*Z-pY>VoX%^m*S$y-Rklj^fQ?iJbJOk4u=)Z>WSx_BeMz%BP?#2ni)yvI>y=45Kaqfl85aCzoD2gLMv6Rsh=oh=9^B-BESdAE8`Hl_hEO90=Uix7XfSYpDQNdxo_+ewVfQ8W(m2_XYSViIOYz^4E+(3y4`zu;lJQR@CM`&skT;TRrnCn9N&Ik6A?|Q>-W)J>j0n1P}nkxYmHAlyfvu3Q!DGgUC4OcAjw=qsSgD`GUeeY>tsc`ep5j>qU1uQA27*#t1+y(F%_Iur>$@YGzanLnJ9$l7mlZbxIPx7p5yfS;ZU@(*bZ!hZ<7r;P$HGt-dnT2SYg5jgOj3>}SjT(pSIQzeEd+D20snd+$IWb8@>@jX_J}>gwvOwUsvToaSfkzig4!bz|!-00b8{Ta1|Ene7zOvWga0-;RD-Zhe`JciWgRLC(<jQRTkcy-i+QUteC^Z2UUrk479@*RhEAdkxa_H$Q=&Td@#`K-Xh9$L2ZY_uod$g9hMA<TlT;+cgQfn|3f71ZKiET0cg(plt#AQQt`2sSD+mtD8A|4IA+BhOlf|{U|=tATBTyPY>ffbMgAxCM@0Me#qd@zXQ>RD(U|?BIfvTJMke8eyla7G9jRdB}!z(lzNX<HYCAP-8viE+6Gy21Qr&-lJ*9WjS)hDkOAfX#0^DUAInhmPwoHiwgF|oYQFvY4}Nm$u}nW%i$sl<D%+H(PM!q~f|Pp4jKPYeAc~;5kvmYxKzA2WG82W;3DjWP7Umi2T6`g|5M7{adE}0Y3^iz;gEGiFiA6wJ#-dTtach+#0a{{djxbJu=Us@x1miVncj2dE`Dtq?1_Sy-3?aezG9i7yQpuJ|01?QcmmG$7@+=epVxMTXN-#l#hbi;YS$-%?Cs2{O@>U{=6wd&;ShU1Q3jOAV*D$xDvw}tonsm^Jvoc~Z9b$NK!je9%!2bG)2Ol9PN9KVtAsEMseE>N^vPPu9i1o-_M`2y5fNbx<2%AOloI2rT1SRkotU)*0(8@cbY6aJ;9#!HEepf8mT_EjA$AV;JM6z7Dn9!O-0u{jkcE}+22$4!~EO_gj^I03KGE+el&;wwKBB>ocW+4<zKe|%`bDwnWlYjm}a_0aZZRO^501wSyB8SD!SC{`~C?GM8cO5*#_uHQDm~U=QMlz#4>31C>tW`M+?)Y6{xSCVxq}1g6RZYCQ-zQ!@dxcFhCaw#gQ__x72b;B&f?7_POi~LpQy8?_!$!vwN82zFR2j=iqA57{gmaS*OF*7Ww6Q1w`7cWZLr;dXw$uD!9nOP_{N8Idxf>t-AzBzQ33*;_@VPdio|w6rzi6?5b$olVw{klFcXY=$l6kw(KF!Zy4=&CT1uRgqM3)}7xl>6DhqU;y(j8!L1Q_`>+P4vZ>Na4(Y^=E>+BbFOsOSJvJEE<h>C&C^bFIVH`g$9DdS+45?QC17279_Q|Mt)zlW#Qj7BD%{G+@p77y*lNG%5oE6`(i>Z$eIhKN(;ap<x@b8d@Dx0R2J=0myk#Fc%v($8k}f>k}2RPiXhULQO6^MH!%Y4T3O55~K$mAaKP!LB^4ge2ZK&hmX%93*nMa9I${oI$Z6<^Jp;x1cr+U0OBS)4;Ur|$|cqy1tb#_1uU6i$H!VdpcP<I!kp3m4}+P!eunaB;ZsafHWO%l1O+E68?rJ+%0vTfN`gn^#nYgiC8G_)5(Qaig(NTu^onBz6W(g&$o1v+kK5&s38VYTPDp^EjB2KXFwU3=YG24$*ePqRQVJT`|7ZZLYGI#7EjncwP?1E4ntJ)1aS3yAsf}%Vte~zBN6d^Lb=t|LX72G;%-nwhyhPDj#+8f#z&|A*`;ZAdfoecZv|2&uLceCqGEoMkB(mE@D`C_$g<e<WlEAvaR1i>sK~93qRNM*yQ<@9Kuuc+adx9tA&xOas4*}Y~$<Z@yV4Tvlc|o0}(1+qIU0`J}<`U&*5ycT#=+;fqaX|M}DWV)q1I*yfRDf!luo*c;O`O%qz-nX;_F@OheGt@NiZGXgf?Yzvd@?*L<Dj+PI42}0&^I7&GN`N`Ig$g)qmbb$ok5d@ui-BU2Qr&u4q?<20btpI&SapAU=?Ff1@tD%CLo+n9Hf}x8bh+sg=k^jfu@Oxu)(BDL>QF~ObW*`;~)ws>{YN16I=%CALdNZiE|ne9%Vu!(qfoMIL2{C6QA=8HI#(eW$HNycF;7H^RVb73!si9<V#XsVR4pZmB@^F*d(BnJP1Zlm|^rONWyWw^42SbKMo0JrN@wbhi2hYa?CVbX1+BP@9aY$?v+&Am9pJ^S?>J2q=gkzPjP&G&_z|3r@w<5aQE~FgXEpA4q1O6RwHtMTpe<57VxFHlK<+drq*KmqALbDis$Gx_))2*j+UaUH1u_=Q816?)?)NT3yhV@QS-!<7A8?-P)r$7STL}UOT{b*cxMU5x_M``CU!O+XF+x_Ut}a!$z=i}fkmW4P7+Hia##e~=Zyc4FSHf}2rLQ!P*}%^g2>JRn&4no&r1%zI`Z$|o+Jm^ucWfw6??UqZSh+_XbV8M(|_kY$k@ESwbA_d3+Mnk@7C9r;P=k?-8EODhkv{cSGi}QaK$r~1WcZSh&33qsvT1)0_-%3nuu&DFnNjLQWHU`VX$F23~zT%3BsujJ}aey3{RjI(I7GI*8Jdp72j@V!+#5?n<!5t3AId0Apuu0A=zwsA^}3%2rSOW6sJ<FImL<|yCD<xi!7{n(*HxBh4Mu($X>v}#UO~6KFBPi5=Ki<kfH!VMWO!yr-Am)B^GM%Jc61iSa3n>M4qAKGSIyp4+eJkXe?;!0+5aZl|6(+vbTw5tiY62h#bJH<s6`q2f!M&&fXj&xoi+{52`WuX7BkpaaB+f&!VSJXh0wW^V2zF0LN_vyln+46-jy{QxHg%o*XwB%MB_<xJl1<6^WaZnVGk1C4w$!s_?mgI#gqrNZC9IzY$0q5S)%%rzKVd=8TAJbz~94a|GAIF~F4q)Iby9km9f(YtZvR!{geXJkgPyLv+OFpsOFf{$~sP)#PiD7?CGSVk8eGF(M~QVg!=`27rR_#F`Y6@hB;WSs_6NC7NLK44}K^AVEbASfs>*29ble(MrQee1u&1PiNvIJ5GnEQ<MY263kIlX#s2o20EJ(6w68`(3uj<M37n8c#^C~k*%cmS^!`rm;ub}33`#pqj-^iywEFD%nleZzMTNC#tTSUe?B$(e_y49ye9wm)t@yRTlm~2xb<+MSictfkU-8|KR$=-=#HbwQlSgCj`GLgLK`kba{n_R$0W6Lye-FlF(||N!uoj!EAEK5PhENDTi++&dR>3(_1EhUwcP$v-+EotAK&i}eYgAZoj<NW%=8g6oa%3@4>#Yd=Scmbrcb@z-)8yp^jlv4emzouyspoDjedKgPCt1DC;Z__DGNMQv*R~0l_r|`6nSB>xV-UN@qkfK!5Kqb@>)xsK>jGoF=E3Z8>GrUS&F$&5o4C&3KBFw`8YOq+&H$Iy<XjFBiDLXgGY~MONK<`l8~8N$vl(6GJp?Qks3)ElB8NG42<TC2cC?~8m=cwQoztmyJ>SXJ8R~n10@$&Du@=)Zy21G6O08YXbJ}jdMhIrL<$B;){wjxqu25M8GsXL+0qHFe12NtY=i=WgPmRGj9VIYu!E^A1680ZyC^lN!C2xuC87k@80X$A0OOit<xx&vx^On715yYODU8j`ff%w$5Kz@XjdX^2WX6jy`v_++w^9b+YPh!mYeCD==7{yk0obBDI-!><1A4Mjz=24)Pk@PS;-L0<7!@j$!kX&=P2mpoAyAF%2obxDV(vn@e5=?XZ<hd}hgI>1C3t@-RqSq6LwcH+J0-Q_qb|~?wL;E;*y41_bAEH(m9j9aNS+5LEt@*LbzyV29kTJ!^s|t*yV&P<3+Ss2gMZ&|MpbtwIGzh3;x)Yd#z!R$;nweOl>8oy@xSx?-|bJNsLX%Ngk^cTl}z_)RjIZ=YNp+EaMV=p&8Iak`$3<*^Xtx<T^OooGKOx{GJBUx>dUxOhT2a^*`|d|E-R}UfN5nYU~2-9+XfY|;-<7jcmkq1a}q>03=}8O!~lD1KOvCoz>vv<WmqN;4x=*pgo;yZOuO?tXiHw(T*ab9u-uWaYvPXnQx31FJcN<;vhXk6E)t5hfg;VW+dh}N1jHw%g-r+RAUdqTN>%K`DrV9P0oHKjJIj+$3x3#X6lQzO_QyT=lKlf_fZ6M6x&Z@>)Ab@FXF#KwX{DS-kFSeEHOf)<JXvj0IcK91mPxE*776^K(c5I4i-2=z1VsasM~S;l(8);-KqW?pT)LWz;}nX+m3gxx>aH{EWEDr1t>J_i3L7iB;#wLC*bIcq0t1c$pc9iA0f?pB#fnMB08Bzk$Z&D=M;=te)U93?6%*CHS_p8jql<0jU-1uJIq`nbR2ZXp4k24cJm|<IA*a3fNPZucr!HyhIW-|W3V2CyjM!8@L5CS~bU)0e%IyjOoc!l%*5c+&Y>-M(A+=0GJIiUzfD{;r=R}D`ZVkZNu%VkAF&G;m6;VfWJ%V-*YDa3aTi;1%o3ORJyS2G|(QR)wKYpW;ON5Z71(0YsXyD7YHqY%~DY_qT<4ShhZ?q(hjI>U=Y7M|NoCXCezIhk0S+fPuY#aW<Z1uIx&>nZ|ZRMGkm!NSQxN)4Gq?ZKbc3}fB9zK3`N6Xc>x3@6W5BG~UqY7KQimg2!&t&(!=0|Ji&hPFtf3XeiEyR-bBYLJuEmxpz8*jSpH0N7Kmpv{WCE(?!$w!ISPn&TT!CV3)j!sw`eZ(9ct!#*vCLf45#B)IfNLA1bVbNvcU;;XJ^g_B{V~7j*yIPYLI*0j*FC{}UMN*=nO?i(D>V{;)k=08j0*F{s8J)~QQOQB$g1rOO<19gjdKSV<RmBcNypE`3$JDVZ+4b>At6Q7v7Z*{3AqiKYSdtF@Ya{d{BPL;(fM8exhN*}HX9W#LP^GzqMQ9k+a3yr-bZvUID2Hqjdk~fq4^(sJlX99{fC;%Rtxl7Sk~$EngeyvOG+FqN;kpF-4L}M|#}zg6`8vA%)X5rl-+O!Qr)e{8yLC{#f3)?~C@(mCXTrjb+IS2^PVgO>p*u`CCAopk1<(a3o-9hwG83Q|FfVwZmS@iSE6R%_<&6SR3UdZ?5RK3}8P7bWio-PLFxh#+%+aVFXzDH@@1<utCcveH(<24!Pk#8lzc_gr3hU8R6pG&Y7(8GrB9uruM+ed}D9-?D(SXdpRx2!|jvR_S#su9#kNT!orkt$P*f@~&o=MOa09^%cw8e0w>;)*p)`Kjt9+tbN83Y6@9p#agIav@2C<jiS29920{1TrbgX;saf75K8Kh<=xW3M%T-q!N_Z(IJW%m0FiC&997tpd&ey1mGrx>w@fI_ibTvZLzEm=46jH^WC$mOSMkTC?N$UqWzq#c&BEc?A(<Gs7@KLRAE{#wd3Q-NafMhf^*E2iqj_ClRbKt0nrvj89x@EHjFM3&}+mlmaM|NPwHjB?tQgCMl|jQ99n}!~|y~U`b!?m<^2Z09e^GmOTME?nGt^UB&^mR5ESJC1(xv{8@d8(nU61K`$wB3AXqJyG%%0!h5ygL}Vl#%q(~ru}1&_1W-#cN+TtVm`czp5fai;X&Io`V5VHk7}ct{kyut0CllPPvoaBS$4J9qDH>s*uZ;z;23nAH-Y7``4`fh@&7}y#Ksl0AWAzQdAxO3vqzE({0ZM~o88AyS3}a}i&Cuyi22e2+wJ~Xa62@gdk0lSJjRX~_4GO15!&U{O9fW3*!EWsVWGQdqBdI0G@&~I8b}hiSu(CCy@sf8DmXjwMpb(SBC1)M<sex-^A_7yvSu%icC6eozz?jLJ1Evz%AY0%ifQHBx619fAoiUi|7l_7X3&Z7I0x74d0g%ZRC}{9FkRu@GCH@ZS<uh+XA8RQh>JQ08&H_k+0Bcs0mi>&w1-O=}ga2c}WEeRukhKwy$79s)yWOf4UzV`dLN)VYwbk4$A+q;tMDFdLuILt=Kk;qT5}2i#z`o*CP!eFPjs_OD$COeHq6<?QwDyvL0xzv(&`p$pnjbKGg+5CuMws!6rOu561u7|%5+MxK4x)Xem;(!%2u)EC*e`wLm@I2x!C}202Eyy)y@>o6UPJ`z?!xYQq!wIDXy4d+qdC8}frgG<xAVs7$MYqEuIF`z#<?f~Q#(OXX|oDzSu#P8vXWz>umv3guu@2%etUzG3*-)$(>oAf81=k}hLFL2p#<!}UJ|Fh%7F^2f<z+G41)-ZF(x=fVU>b(1Kmbs*b69huoqdSr{zlo6y}xU4jQ9kW>{DOdMd^2;6O&;0GgRaotFR#6($;)3CQp02y*~CUOHiGxv<Ve%80d?Yy~|LIDi7103o4OClf%kG6NcdO_E_I2!k$+a9<$l*%B}&Nt+C2@&u!_6`+3t;`361s+1Du^Z690ggGiGx6HNn8q0_S$wZIA*tOr;wmS#h-pTI<p#JfI{}1bt@kBsA3LLb6JT^?BTR&L9&Vb=S{or08^?*f}Tu9P9O994?f*j+Z4=ET91L|KOfLDXgurxs-G=Rr^01QAg=mVhUavwBi2UDdPH|7yTZo~H04raKoZEbXGu<~ww-Edp-$l<n~=k;GYW;>ixADnCz6f7H>J-{qu38lyyt=k&OSfxPtl(OXBQcSkQBG3+9CemaMP%9`drF~(l(p;<Yoqhh&-IE>(*o%}?+M$Lb&^rgvjW~dGPD5W)>p;?S$zAf6c`iUFL6t?LDh8tXr7|raVOFN{SZ3u%PDCadfVIbbh!#aeSQLQ6we(m5A_Zhk%n@`y5lQrksFX#NA!~!#lOpnIhI)BcFERowN*;!%Tfl)t8raYP4r9c-Ho;{E+A9e%*#wF?oJPU~3X7Hs9zZz&<=uGa9>K=^r{8=1X_=J)!)7WoRsp461LEZgu%^MXDpUbNQyDX&31D4NqcH<8U~wZW0Vtz8_Z5@!k+KH5j({AYA%dDrd4}a5xgi-V#~}@s@I({9b})rtW(i^dLOG`aV8BZD&u$_GR-G9<hIjjl6(3EcS+<c}3lc6Qf0b}Cf6{oS0gWZD45*?JS<AAd3J|E$)DzD*lMDb^b_($VK=6PRgrFoO638)VTM%U3t%dBb_4IUoB*_Qu$lmOjLr3JXCvNZeJShfSqj%Kcc8mr-B0(4T1nc(sB)D>(7(oN~5w_R>>j^e=rC`O0IK6y=dn>s<ux%>1g2G+6Z){@;Fxa3lmJ%5=*S2?dSI5ckxH2XpAKq|py7f1Z)D}8mbmbwFb}IV7ZN9O#vApe&tJZn9zIJ+ZeRuox*7mtGn_DrR;fftP9a}2Z)xPOZGV;Jf7o)`$eT@<nRD+ff0~;feh+cw?Xi^1`(Hc%TAS6_Qm<9ZUe5*%i|J*9P=pPn-+7q5%%IH8qr3(RonxF}qodV3CNC{=F_aOkNpv*u&P%5-y5q)K7vX7Y`{c*^{_i-0|i!n-lRjxZrO-+1)|9$opzT9ChJ?46&2N0CKj1^Ad`!ccG2p$c{8(3Q!cBbeO;~<BK<tC|w8~^jg#6$TdDjt`K=YCC<Z7KL~TyAN2W)7}A0obbm-Ube!9Tb!Z`#HdNmzC!VV7%}EaAQP23amVabu$C<C&`HNP7>HC{3QkefNPFrO_R|{3oL@lWmbxk5CC7mIGO|<Y`sAUB6WbJ&Cz4W9OrQwH;c2-ar~@UV97L><Rb4GM?@1%XcrW;hieZy3rr!FqW98(SO=wt!RJv>u554|u_k;xB?e*iC5~~C@U0+l9A+sv4vK)MCI<K-ku=kpw#EvPJxnYL_{pQ^QYYqx<m!1O-yz2%-=P586DXLNjA_vWp{*c{QqV|90(CR_5I{ahW+<BGSwdq@&c<aavt;y=2K6ar&0$FjpE>sigf)kMKt&WfB<T!~Q8SK7k5p<;KyM@fV7W?wR{8VJ4B;&b{eTSo280w6k!gV7iboeLPuu~ZU@8WwK+}Xjh{rO|63bXoFY?g<>x%+}n?C|c@qRI4p16buFob~;R>BrXBi1bf^`9{<g@vsQ;35bXfKZ@3D}(YTM*`BqW%zurX1Uhp7@X)cn_z#nJobdM<|CofWC?-|lsu7>1R=*=23;8`Rscj$q$Ei!Cv6Rr)Jut8#^j=oK0R-HDyNUTq3)e3@hW}h(ZEIWSgt$nsbkJaB>}S}*dbuXOIXUVqrrw?Wky@JGH?mtH=;A~I#^iGoJ80*gqe2TEesh^i>T&6{b=K>9Oh|J(8guZSWJe)7-h^GgX4)}8D;G<C0Z)ZK)J~|20)qIz95^`JQEX4RHbk)$8x=(#aXYBWE-TaH54lla1u0(iQ~d6kj&iR{sm)3tPk19=Y6k<)+Jc%*@Ai$y~fg6u;CJlCBdVi5w{it$F)S?u7fSuc&Al1LBJLyj;N)D_Lt%!JLq#Du67LI-)@zAG|e<~ew~`GxOKhq)ZQs{So=K@LwRh+uT~0@Ty1&Z_iKV(r^$R)6rTH&cjC$aEH{5x^WLtE`OHd=$~*N$AwB!|KGklja@BNF9en;jj!JJ`F68I8D<6GVM5{e<zwS8`S$e;#5hpkPzJ2Xp%_uzdwQ%Dn%TwLai?#cGsj#W4O1HWS&9!uQ*VkR}-b5WN@(tC&ipdCpN}owsaW#@mu@I-Qo{BbCSqWIAB3o2B1cvJ|!ZDIGXC%Ee3+iBDrhZ%M8+UG42q4H{b!@M_RqNe+!?m$gUmA;HT%H~k$Kul3QuS|6ajdSo6{>JYL?!h@-|?-NNpNjU6?nX9(>}D8m1`%?;}*<HXtDRmZwecUjDb23FB()i3|jFSb|QfONo0Vq8I-V)9fl2JQEku;U3_jmx_$oaugmWf_8=yPff<;&D1!&g5mf*!%3;-vXr%;{VNvn2Msno7g=GUG9<<g>qEv~zG#<o~PPW?CbINN-`?eW-<78_`U2V&c9@DWnw&!2iY^|d)y_MFt-b6+k_`Tzh(|+w-+5|xRX4-mlJ?)&?Om8D^2(@PjJ+pRkZT-v>u+*?9V7#Lf+!6$xi3dRyruBINmq=KN0qaR1X^<oU{RwR8AaV)nWWlFd2jzki+Hz46jPrzeW0X~x^c0=c7*?cVY7*};fI2f}CAS{XhRn=E`yyvCV;G|~!OwS4!mr)N?Yyzmg2_MKYzLrqbgVajom^WK_D}mRZhinu^a6gvQuFrtwNN1FPkz|IP^_KX#G=GlR~9>e1^?Apu%b;~JPC&<Ia!A%d3GG0<mov)kyX2;L}8YIxHATXlZN?dDf;szY{iKLl`L;d#Y({c5wtP@{g}KI;!$w7lq|hgi7WO?a`yy|z1r^19kBXUS8#M7z`S25d#_Bg1sB_V4SM;oJwMdZyWamjE~<Ah;w^qq@WSNdMu5DleL5<y+NM>OlO5DO+e|fnD7HH1vzK~WkAuF9?Qp~0s#>=w{v^t)8p3#H8S5-y83r^q+8hE1M~8L~Q2mGu8Ps6Qsqn$V;<7<U=aOub2A_rUs*3gHRSglGk1!-gZE?8(L*8kz{^dhTVEuco-F;=qR{z8CaR1@_$y(<~y=+o-8O_>p4cDI3^~UAL29`2<7lM#p#%PtIkvRaCfq9_`2~rB8(bY%cKbCr{q+%>!hD*+{za4EM{cQA=AB{fY<b8li1=v1SL?}cE2HsJ@qld48-CZye6~Q_P_;EaCr(siv3#6kFy$<%dCL+DpPOMH7U+#!*PTX#qKpP>LyDI=UAs@a$_R;8w^%8)LAx4`Z0TLU4-O*V*2BW4UNEns$(S>-*!n)s=3vasOXI0g|Us=HqI@-zJHn87K)3!|7#65+^?6>8f)+PAOyam^I_m&fneNKq>k={y3jr3_8X{V$L6qQYdYPAQ!zynPg#!s-U05D}t#R4sKHd6Arjal8RqjINL@UOI?F;`~Tc~3<Bj1I)GhtWtxIIQDDICU6!ky11SGy>Rvxep4RqYltNXsI5gm=Z)#!WTxx{6$Z34=~<AZ#Czk5r95dn$4}|&F!@fw|x=7xVz}r*4Nvt66n{{^Mh&H^uy_3nz-#i^IH4)aRO;3u03`r4eExVcwBUEAP@iUTB{}g5Hy_howd!a2;F+-hw#MZ`WO7zI=AiKJb!WdM{67FYu{qzJLg;OBN(%buB#pQR3dM(Nj^BsVW+_=P*&I!D6z<mjV0<c&CE(`+7MJw{jlT?68%dOEhRx80j1j#b4|;;cD1K?YigY+kVXGF1*}JBnpy(mPPKNHDix!(II{g-FH3U#mVpVa#8Ld>_t(E1%2{3;(mvW%+*_Ty9FSr~TU4=O|Ag(S0+4SMQ5#rd%<=>X*il%cIm^U!jzK6X1E`qXnv^L`#1p45XIByKWO_#Jsa%r^hN-Xww&gJzW|lFv7!6O4?Lu(-hy%V?hIeo<9AgqZE(FGKlAMvF3mKkaU3rh?t2hW;4gyzV^0iLR1}mer8LMP-FT5k!VEu?AjC@u(tUuu*=K;%~8E3LLn8chpQJ@}E47PxijtPYk8PE{Sj2A*ALD~{0vyRo$^R;M9*DW^-*?(v6=_sYOCK#Wi6HUk}D49T_Pf*jDWE!;bn3S<`6ry{)oIseLt~J$Zp&4I9wsii&xpUC(7#w{%Y@La2*X^X;oijTi@Hc<Db&;L<&gP%}*$;kl2DYGd=24n@!T%KYA#A{R*WOwSJI(*m{0Wj-Z8{u<Ls9K!3!NwzZ#=n;I(x>9W*+89Q0Vey!FWdjQCV#>U|A0uGYqXIjG|>YGs+TUNih)eRy;F_e()_V5%bS>JIhuV<%$g?Z3;iu-vBsHyun~J2~f~tnuJv>fwoU6W}IQzJAiu)&^?1pL_!<bukabY;TXBXu<;%y1Z=7aCe&N-5u0197dAHlCtldb8Nc3HCEr3hh*mJPh(QB$-&H6q-J57H?4)JAYJF?#jSFwq$@~ZHnT@q|G}R4~(XVaqo?qQf8vu27@ooSJ8-RWKkMH}jo+#rmnDxrp^IJQ8hv5Lg(I4Q88``Yp+}cVJ-mb3!;_CXh>F{o`zqSo&@PTI#D>Zp*yFFfl;{kC1a7Jq_fUxF!TW$FvSOPGHNVwL-<hr|;8o&W9wMe;VseuQ%&E~>pZ(eDtvAw~rLQMD5t+h>PzT4SB`*&}eUdM0TJO|(M$7sl%*k2I@)%~k<VzF_qZ{c#<+FpZYKepywt;wB((W2Wk(0Tn~-7aqT4rciGvr|o_J^8RW;XbQg<o(tL)7irIMt<LqtE;&f%n*?LwhKQgrM_C6Fwws8R@`sTIIZA{x+YN(+3hl?Yd>`ZirD}uT-e3wSH~W&DwAQD?^O%Q+gm?Ro2#&3BzjNMaBWYw3X1>+x1}9N4*@{^Vidcuxx2QJ2Al41J*<J%or{|=@VKh`3(lwY5lrZ$FWm5*0uh?Oste%{reu9fXakE}q7S%>%c*i<JoU+7C5JoDt+mj<n(v40Hy0_}$)Xy#vN|Pc-jbDgwk>3aCU9tEERUF^oevkbw~>*yl^$(e%nk^P_0_I(I|XKF>JAENLIQKf+o-%IoO!Q(QpysHMNwGR5E)=1cR67+rGYJ)B>;0a6B#7zK+lZ)8nX{Vr|+y+#p#=qUVR;MtR`MW7RIv${IoM+YhwdHr>z!&k8R2~tY2)}Zi>A)gMB|W>+bE9CT+qsc*9aN%A@GSRgqxrlh$@xVTTL=+n6(PE@3}C0jUK&hm-^LBU<@2S9HBIiDkRrO?6Ed3;vCRUi;arpzQr2{QscPK!IAdGJWIi?)=S`BdY!RDN4cKInm|sOg!fEV+pa6gSgYLGmQ7EF4;cM^F<J6{d#Er#Qi+&!n8Tl6rRsHf~_SPMFx>S-Gphm;U;)SmwE<K`fYaVg%W;I5t(dsl4DSKTOyoNG`mtzX9Gbxde2P6`aT>B24M0J=pa(atf`wT{7ShR_?Rid;|kPS-Q9v6G(lU6=vv)K=<I%yqHvfcRqgGXvQQd!N0;kBEq4oG87KR4t9PfkOb-{DbiUO5lkVBr@TRo-?0oRIabc&j@aBtyXlw=4X548D^Ef!R8PBgZz1UhQv7Ynf@>&tfm9}?MgXrh^>E?BOk=sEzdttMkqoY>ALqH8c&;L$<mv-fWO2MG=X41fbZo=zl-@bhTD-FTVt)W)XqsnMHDWl8xulCKKC!cQ|oGx~%`=c<Shadn(wGcWp{q!~81~Gf8HA=MMlDCR4_P6lYSC{{x>G`Hdmime$i!tICqgcONLy6HgzHH{*18;R#Uf@CJQ271zdlPo7-gErRHmLxEIGfk{gMHb8yDR0`XQK=6)C)fwU3h0@_K6SMRHn8v?VLvsXIFZc`1sa?p_b}v{RQK@hYMDkuFT}{fU-Y+0XMaGw_cLr?&0i8Q%yW~hPw-wL@i#~KA^v5I6nW@UyRg)+ih^(!^sESn|sZihm`_;*C+43*1o<wRmQQ{z20jl!7s)^0sZIVm@oB*>mv%Wd#lwf%^jYugO_G|zlII;H(&1p3dW6V5zJCqGuJD&8Na+d|6pl$HVxx5`*M7EU&r`C=l8lYD$#tci;6s^5(b+ArlzAJ)@|Z#Xw>mQ%a1`G9R^(L{@Y5rIAs^9qQBd_rx6VjkgsE%id76~ssTM!3#d5g$pM^H5Y7+Q{5JY!s`}Qo62x#$+mpOrJ=*C+Xjq3HTMK8~4m?ZS6v;!N4V?#V=6BI9Y*;&E3H=NJ9hP+_T1aNHVwfl|wc=R{)aR6B7%}Str&y#wX`wt91XJz}Ny@))K&R{f<%&U;%anKpb)Gk<(_3v|WfwO~k=>tRZIF60rVL%tYA}D(UjuNaTvmSnE!x~})lH=r+Ml0lpNcsaH%Il7+6{29(BaPZ)dwHXKH~Z(6Zans2%_YI?BgxhM-Q%bOHffBeMyV6@X^anjlX-ZL`js$yW7<tf4|*SuNTwc@ZeL}!vFi*HC^QXz%n}HY`XDFyuNGoA2nb7{k1k+=@)+liu|`*Wga)hQ~#s+E}Gcee_<?PKENb+du=mry$%0aUvH3daJvD~gwy20`tD9Qopvv5Z#EaVE;Jx+-e^fHarP&q5~+ITqi%%gCT!VAF0PK)9FMUd*6H^`Y=~oxzxDdGrl;3IO2aM(AKiSf@1{9>;?&xCk+(vFEY#E~mlStexIiT(3>ilmlhkD)lVDm$86^{1^5`g$LS&|CNmG60uz3L-HuTXQHV0j#CJN`feK_^I(}AyT(ZG^Yccm+Y*TspvKhLpq&~$}CM->U1QCa94UA}PVt(o@!yj#`j4?Ay8r)xayD(qE{(5E$u`&QjDChWKG)h+4HL?rdi5uw#y-J}ls(7by!YqGoXgR&!y<0^~t*tCC_aP5b^7I-iUohiX(L!0*IrxVsUv^4`B!FdI15k2lNy90|WNn6&EN0si+tPPj47_e%bG{QI}C5#FS4R9@mA_{aE5a(JTS0Sc)!Ih5am?+axguwGlV#CD-!QtIya`-ktUTHSph-O2cuo1VDlmFI9eiW-ScHIB0cHCmQ-K+6vU(2R@a}?e<xB>TPOT_N?kMGrt86)rY{+bC>Z-+#yar8kK0QXusb*-Asy5lS5+5gxZe8<+l_NM!zn<maq-}<obWe5G9-th=LtXI7|K6z^9@pGQg*-Gz>-ZOz0s>Srq=rNNQ{LO4)bsy$2HZ9+{QW|)>&!L)}6D885PrV<gM}`*8d3$fE?Q_818Jj%Oc8R|~>El{Iz^ndtr6*e$ckg&uu5}gAhprn9%hVj$>*m$;9O#a(lvYiR<&;hSotdGUw8@WOIksm1YOWo7e+2kCJNB)Sk^1wIiTPSPBzsGf&l=g3t7Uw_G(CEbjVp#stxGKByxfj&8SHN9jm^^2Y!UO<*#u?7Z+m1{XH=m>ZOc|4*3tDT70b4mx~}?(ZXVuOn)Gh<7<KiY_G-%pd33YpB#h}M?pI?7UiF*tod2@VMC@Mr$_D>*ZSaOZMs&e*Vt1e-`K*lMNSZ*sk&M~4(o;wzr>G4wPjZp1Glm3XELDsfqIKX*aP`XW@bU(V7HN64Y`!1WY}he#*ujip;ou0v!e_s~@##z~>FrUHWd|*anCy>!Tmi_PS5c$Kj~WFvRFD@baEeNeAC)PO2CK?RlT1pC3#8PQS~_jH$|Oma6H;@a7Os@h(t8p;e<jRbUSW2#^)UUeMu=dpRIfQ;Ec=4k_fO&9Qw0UypCY815Mh1#)j{QGdpHh!A1s^iw6A+ujh_Qk#4C|@43Q={ku6Kn3nS97xRpk_b4d)yu?P|?`ba5vGDpP|wUKZuJqsYulhPb37l9P?QfTAo%Mc>5;dY*OgaktU)_R9A_str}%t^1o{i^j91I6c$X;^II^t(^(?24_XQp}9ScKLinP(agHciqpnHn%n|&b2cB=jm3&-WYTH+;6Rn`(4;Vmzs*)l#>^IyTg$06#G)2s=7S?-m^zr6DJ-sQt|h?i@v)2&mhCUy8JJ=OCb6^wbUIe4k0`<LcE`qXojAe@L?<S_eQ3XiO9~rCB=btvtZx%dQ<2zzjMl(d87U0%16{b=<djF_U}DWQB#9B?pFKjoQsOdvdjAFtbpyFSg1c=8=vkDuG9?l>YgmTOP_Q}8?Q~C9-Ue#7gnyu(Hrk~9aeOY*JiJqxL~EfV}iw{XA-%CqggigZfvSyb@rb2`jsgg*PTi+GB!cKHgnPB1?B68G`TNodA#?U8V7T)`?T4M>IExJKZ$$eaC!C3k(F+LWyU%<>dar&n4DTn;k8*?;`>wAl$4oC4{kFsz`izfx}I7Y1^jgV`d<IK0-mncHcVbOb4@X9ce7^bw_(zDmhJwE$+IiH8L?MmhuL+duHw>}Q`b#gurlSQ?$+36qxXK&jmY?-$qTx8@P7JFu%mT2vWA7<nLV}A3?}hTJWhN|Z>BAl@2N|tuURSX`Cr1ABQ3<ZDeq6t@8POBS9C)?wuSbztMAQeS_#r$aL2r>S9+7d{w%L>Cv_*}Z4GWOqx8$Ev)zLHWjn(sxD9TVV1v3E+bd?yt~6B^{CZ4N^id6}ADyemyN<pyUit5~53k*Oy`C!zuXO6WBb&&W;<$wKjp4}68cz4x<ZYu{X1=;mlI)+J{MrZQXeVzT;FeJLq=On~GG4p&qJeRGa^c<@#n0aw%YgRKjk1^R&s;e1XLnKO;QD-4vfUe&ZP%=+>n1Mf^CCtB)c5;biSF;oi>?;uMU|Kb{gLSfxqrxhIeXn`<Gk1Yyi?s>^S7DFi)Oc(9>)xAc&DbvT&Yt2M%S3((&-CUs<Jzpbv<2p@o^Phc;?i%=ei0Hc~%{uFUK)K(@*~D^8ed5vedP|HIj)KxU<xE)j&eNu!W~qnkv1n_oF*n;fMx%N8aeVdDl$Ky@eZZoy*maR5!hacP`J~(=Ff2Q&a3$y<n@$xV-|~y*F%I2X)@OSA@-ny*t0#Ys%@#Q(gYiuWMUA?DwoPP`8Jze<bDsQ+`%6lg76WXUlUtcCY=qop*xdqY_gvHDM<%7`=bL_XFM<`~BPbGJ3`2<5zm#`uCc_hdrc2bw^g3sf~WGBAhxox(#VKzS4`6kH%y4S?|pppZsii&DCn`?%s4`q*?CO_w^S|pIw;lG!O&z{>ib_SsIJ|PU<c7LgsF}(0<vEK}Uqz%doq{L-t0ySHCIUAF`xwRr>Z@{muYyS?X!p3#sRwrA3>-2cs3$K5%~|27lLYSxf!6zCSg8mc}rx#{~CB%Gj4n<M4y&t!$|oAo1WX|9lpnE!CifLY#!p>c(kl7AW5zZTAnm_7AykQ__8@>EJnj>YeVuZ^n{s=!6@!;S;;w(jvBw+kN!@(1MOgx9x|&uX^VvL+%<h<sa%!*f*<-1f8A|dtIK|^}FqvV_80tPH+wWTHSD)EfMqj+POB)98Fn|?hAjR;_ttfGaSt{lfLjm>GWd8;Ikkp;ocZz#kNvt5)8FS&7MV4oKZ=l5QgU{BTHJi=yY~YFp7k7`IY<V<&DAbL)Q+*Tleo4c#Up#QIHcZq7O#~rH;$Ih{oGhx;|_H+&C`1pd~#0Z*?EKJmF57yF-pK=iO1T(Va;nWudKu7QBjOe-s~-qLLm(#`Ju}vUB1D$<ce_thORLB7<WF{;^i@EV%W;DrQu&A$!gVm7dzbh%kvs_DU|kyl(Y(X6Tqc?rk(pFZ%kr=Lf}qvp{BEpOKljMnOa+x#(Uq`O@CUvDb^#URS>P%CmlKsYWbg)V@fbb#9eUI>#gdycr*vqmofdiBwVtL6eSL1S%4G+*9y`rIeYl)Uc#HIoz}UwLI!r_-L_2qhHUF{x~x2zrvwf;b*xt2c!vpDt!FeW_KIpQ0KKSGH!S4jf8TmJ=opwM!I;q`N1J>Z`V7$-RADr=EV(kSnr;PM;B=b4_HAz_6~GZ({&EIr$Gd6xQmU8KfMr<oGz8D?{>a7FmetQt@dSge`)X&oAQZG+)9PFu2UkLZbdgRqukm$YLQO>Lv~-(jpbZXd8-yLzcaGi^;V0$qHK3e-UDUQaQ(yuy{Y}q$SiQH*c;}b=z-sjUQoU>d-R(RL_bVg_OfE`(l|`lE(;SW;r*fNbs?ihZ54z(GWCMkeZpFGoeKRmmFY!2PAq2qg03e}_}5MzCh!xLwlMcEzynG=NQ!6c_q65OMrE>?DPq<q!X*zn141M&Tc^F?novWDcM(nilGGSWPAyf6X|4lSxFA>voxcK)FEM!Bs|Lrekk<$Eg38bKM4X4FtABoYy83*&;^~T@)Ar6**`7)uA8hW$T-`Y<?^gBw<HJ(S#tx>tHNSi!_x$Q8<X}NAdS5)XQ|Uh(GojWPg|3h4+g}0UW2^UE8Yx~RAS|p+SsN}@#QIspML@*HDM<*WSObb$<E3^vM#@FBD&?qS&{`=Ut$al{`ig_=d7~TcL-_8@CL_PwX4?LJiCbu+Lo3fniCCBjC8_~+C9k&n)+Yl>KPlO+eF67}1$Lh*?<ppWp30<#4Bak%*#4SPOylhmlhYrZ@Vr0htF8>U)^DA3w2wnrhig`vQE1)uxsMse;f*t0CoWi+jQp#bB5`l2TDQBM?%u#t_C?okH?ExO+_Vp?s&aR@dDIW}%}c!>yYuDFwrujw&aFIZ(yh8j&fJS^YL`Fk1EN0bL%f!n{@}gV;<Pt3<<>YN>-xCN=<l!p77qP$dGc46|MjcOzd5pKW#5MTMQQuIMlIYaP@o33^v>OycTxi{CZ@wwH0Y=q|FewWt%`s4>J+{<k5#CE3~cqSQQpeKK9XSE&@V@X>Rlwp)v{#Xttr+A#c4n9h7U))KnJD0{YTy&argB7HWMB=4C3_~2hsE~1>HPs+wx&gDsV6xqI16n;k`HNv^i{d{p+sRLG4~J79u4e@4XRwY>g1<t1eYMH!)Fq9C)?!w|7!%_{e{%E9}8IiT-x&)X#h8QTxbVDBkGZP|G7kdr&<2y}xO4GPO@Qm<lK>1hTcs2gBT+nnyQ|owoUeJ^in;-WO-RbMxYQ`jJ=`AS_AAC!Vv+fdk+*+y!ry)!s3ZS=8E^jMiVSxK7$g!?Ux@yH`eum)9uqVH>|x6z-4ujq2_=`LhoJ=r{Yh411qXrM6#N8mE=so^f`4+5w;rbw2EgP5Qm}7y$wQRP6&FPnB2iccgnY{BLi{xY5-#>9VqZ*WO<Ob%!%IheCiogHzc&4|;4o^&Oq(ufMT}N9B(DeX9Qu<lNEG_F%8px@^B_C|Oyf*$2S;W$BKw`J)Wp0#nNM(N@-5X>Rq#^}BtH#+`w!<yGaXM+=tFg1$)B6wQoLn!vw~h-e8D$$>=Y(NfN()5cf<I}2f6JBR!OM8W<OeZp#2#ymB*B01c6vKH62HlB`Cv3)ff5bA#OD1OCK1Pe2A;qJH4s<MR}25iimup_KY`2>AB@7`L&ji&AKo!)BFy{mW|-(K52-&#YqVOMFZFFz5N;nZf-e)ekD(8C;u<1ERWqxuaGet+X1yRs0yKGD_VK9+cj(3!CH;(XtjDpmX(dh#4#47av9Pf{IHZG2i_S^Ze_0w?dKdraZQnLw#T6G!?0@0<=6L9b$xt0<`V%28@y5ImN;h8sk5jv0@91LPGhet9Ec_s4FqcdD&pulk=RJae5D<#uHXz1B;qk48C?Gw}$YwRV-;qj0VvYps%|j3_8yVX~t!S;GjU%Y-j8((G)yH+Rmr!#iUj0C5=4va{P;Z=a>oO8X*(w`#3|Q(71<GZ)DlmlDeoOVbb@BnKF$siE5Y7+g>}G7JTyl<I_0W3Q;1Ur{wbcT~-Nes-U}S1biLZuR+rzZlz$$Ex)8-ikekuy^8B)gm#EK(*st8BEA<!|kO$DfNq*^iUJ!`sBHP`2F?&=t{Bm38oWP{~;@J97DIi)T`~cx(vH9Vemgx{r{7R<hL(6SJ2dR#-y;no=Arr(-YhtipK5N9K)VwcOZ`F9@rP#>C^i^naWWt{nOLp-Ws!rbU^=Bd+C1+55C=T%?#wfJvn!6Tj7uCT}KB?)icvxw=`)l=~{oiYxKy%Gm0zeo>0E;{*Zb)TMVz<{Lk&if6z6#?B+eI#+`{&!Woa@{gJ2j^HbeSX{p>g4uP6IdcUXU80Ex2?9vmj)Cv2=h}h=N(%6T2V@R#MQCCmTRs1ibk8E%I_v+B?^Fr_T6S<p%2Sex24EdMsjQpa@MlET^_ov@+Z@N#%%rrNKjN1|a@KA64PA#f~M~0bub;O{2d%~~Ur+wF}?^W9N>r1mVCS6MDQsH#oAI&fL-!HrI__z7B6$gKNUl<V&kNf!Q(iF}3L3e4-dG!87rtH=BKrd9TFO6M$xB9YJOOvj-!S6RtDn3}6GHu?frqEvxPq{jN?6AluBw@D<eo-~|0-0-HjA9fzu9GTy`uW(UH)E6EAEwm6sf~YqQd8<j^JGkO@M)K6doZ%3?$_0SWBTg7p1Nv0CEASMPV8P8`QAG^gr%m^0UV6@mwq=0vsT`hKI&7Z|9+zDJ|AVR-5eIuoe61+@BUz^XX5A<P`}7KlXbUdm!^a17LNVBt_=s1#^Al~aBqFTUcJoqhWistNwv1x#9F#PQF?P`g%5SnmZs?AM|;cd>f^pVXCoYogc62$yBCbZxZEBIgF3iVse=!y=C!}5+UZp1e$jq(ktFxJk13to`9!+DD($#B$&mNOQXeZ*Wc^-gQpZ1Ve;OV9$2fjwI^5%Wjb8YqCktL$#N*Wq)EnbVa5mmq>Ivodd#14R&S5|MWsOpu1PlA)#}DG;@<?h_Np-F8pp5pGVVpZ8aC;=(4kkicCa3)VoXOIEbidLcPWQ#t9xP348y$ts#8&ZeCg9@X@Ty<cK-s$kJX%=3cxjx`a4=fXdt>xAwL05w8fNkJ(U$dKss%VyA1w7$U?0{}i~m{^$Lez4s}&3Sb$fkU_KyZ)2rYRyY|4F!;w!Tyio1<xQoi)n6?&L+b0R8ul&CxI{vO&Kj2hW5?<-4(MH)|SC9PB+rtz2{=kpE(-Woc6rtW0dsxK!;_s)3LcaIt`-g2;lvq5bbzHz+u%)D^AobpmLVi^4O5QX(lJIUT3(RuuO01Lk!MSb^c@xeHBY6>j&qPw=Yf7<7@U+#2OCx)g>%G7Zwj*0o%Z#3O2_U1sl4nfS<LhjGu2Dvv4IqubRC=ce~kzaM;j02_JBxtyE#G=35w*7X^y}Uo{oexJeCAa3p?Y%deaJMSq(R&NmeP1kfB$w`tD3-fvHAM3DZNAbGyu4TayZcp0j>N#b!{M%kNdNoVAj5kP_Wfz0f3K1%y}wXZESU7>5JPyc72tQ~Z{#12B+~w(p_o{QM-A1x{Y-ondi9JI{Wj#qtvI5sFf7aL1!GR5Gf9%{l*x`T7hI-9+8k0aECq*o2HBaIoG>Iuw;&<x6{GblM(gL7(R$2%ceM~BjIL<6n88ohZ-ynL_+}r0JthcyP^0~Cj*KkRcdqmps*61yDsdh1#OqZn=&oKFYj!OeU*`!tRd(4cqp0wO58WSaPs2Sc6IQzp$#td4+cylMUUQ#1w)o+xmFd2m0EA`FJWfYW+&1<7lZLPfD(GW=|NQ%#|FfyZWp7TM>PUibVbApA@==>%@<^X}P%~mCKknv~m$uBFxlftwrh|@$vYTUloxZCh%Ds7y=_#8Bvb-6J$g3SqZm%u>va5*rU`z*cYSPGm)F|{1yEDG0BZMA&))U0S_s3P_YKG1i?e9-UR8HM<r<&ZhZXiECX4ASGR+^F1z3bY&s&GCWGejL!`$rdAIk$KFM~$Abh$?bCow@{urK!U+y4JN8+;;ItOH)&&c=q?F4?pPr7lYxv^`T(d=-#+=Fm2s0sQ=Eu!?@qN7#?(nss7mgYGW9o=P&A0`^9*r(;<6%OqU@vefUN{&`aaRv_IH4ifQm#-}}R{Vw~Hbuz25_uqEzyp~anMdA&3HjON8)d>ka`(DBy>3&r)3QEOs$4pzIn?KHUCz57~qkM$PLhtnf<uWELmPoKHk<A&)od*jrarCAv}>bZkzV7)Rjhd(&nt25b%8&|%0>WWvqk&AgFDYY+dI*E-=Sz?WjjE2aWR!K7L4Wo=$O-RU!QE8IX$_5=$_R<GusPWNg^X!DpolBblY1eE2;njcVzR{SuxuJ4sX#o6<fovGDTY~1wOm1Uu2Q8&Nztq%~yi~M-r4l33uJUI0;+yGo^ZgnTS0Z?!p@s5iCx9;Cnl|KW^Bn3LCF&JlTc7#_qBYufmH@h~?QT&n4yF62R^WX+Vm14$Bw+Mx7Pp5qi^`_m8QgA-!+d-E*Vl)6&`c4&_td+E<#?}|NpK!}#P=6cWt8NLyM1fhE#aU~-u-wgOk}S*GJZYa*%oj-`$2kxAYR*q=^NHB?7%Yo1|{*=3))j{(4MjOh2pGmj--%+7A#6HsiIy;uWa&EvuKQnghiT2<jQ(!2-DR1EIH3isBFzEiTnIZ+|Gdfgaj@KuWP4UoA`UDtGw+*-FsDJ-tYCl9*@3PnFQ|i1t6}@iqLZ6#+;%asI|UReMNhtszn7B<0YpG{_XUPA+)(K9$xMFpZ2@Kknz!%OWGS*6>it9p*Ht7<*9w<UAJ-UwW*5L!Z)%yOw?$4ck<ywp5K>ag|mBE$-kQpO6!9dJ}4LeNB<=^J9X^%RnTAU4aeUbcnRoc>V>KT>?u)%SZRW>nM#*c=GHr>yjPT2YXE#pEtv_48YYtv(FsC>kJeMAUnQ};yvg0;@PUpZ<ywnve_bn_->kuvlawGclytYJle-TuaXueZ*I?orK03cX@X|C1-E`_qPeRx8AwM__vEP`i_0*Rf+3R+SBiQTq`s&Gj4$iCWn@1{PtSEi4<gq831NzQfc|xK!*=NS0up~wAIY4zQ1)Qgr`e>pO%<=40q{bN@U&-T_H>vQT^$UEtTtfMlTd;Yzpwf=Wv&g-9^ctTX#s~^WKbylQKa!9GtywEm4=dn49q-+`H~NsSDmDuazuKBoUe(liMD}^XnSP;aYJ|W{&Vgr7q9!RT!Bup`M}>totg?w~New52YZ^!vIWwCr6JC2s?MrOH=tDW`m+p*29)_{(UR>VYTE^C#d3$U7jWh1T?)k0l-ncL_cA(LsC7AEeA#Z#y@%+WPBKR?^6H}tEv~snbaQ&*EC;x7jBHc}Oi9H<W<b62;p?&N@Ef3TiC2kcBy6EG-8Ru1%>$`f_g&eS>6&&)h>K`Q#b8vg#>kP^@V0KIZxKy3k6Jfqp433L)d|cx&#~$Ay-LSVu46t$tdM`5hT5p#Aa!joFyx0q#@q}+%>;vY7lDv_d6euU$d7YS}R6CociCh`Mm{QzFz^ft>-e!Yy$#mjA`<OX5EWN~%ck{wVf=O_u@ZPLX1l0{va}t9N!geqjw-)6Z4)}K$Tq_L`a>lg11ekMecW0?-M}ManmRlS16M&rE2cSu~`-a8P^VZ-P>y5sxr2MCzh<@yP+wW=0J{$|+BJ*yIk>uSH8D6PoM-`>_dP>n(*zgHRFVjk|7YsK>`Tw)`Zq0QZSGwR|Dccbv!UD+5%1dPt!RfHO`<&?ryT>!#{Uj?CS(g-Hfq=}5)f!6?fD|OGplNlq3{s{jLplV3Ef56v-~j?6CV-cj2X~ww;ozfx#pFG+YS#t<5)#Rc9o-hO>zb97wQ}WJ-}*l31_GK^C8wn3f-|0xx+a3xHG@!~I%R2AWmSQZfB=-N%y}V5&WfTI)oDOaE7b6quKksxYxf@F?eB%1Y^1<iYADS8YX1H7+)w9z{_~&xTZc_V1TZ+d5Ufg(^P&Rsv_z1w6cf3WRy>TfM5yF)Uzx(9W`N`cP{k&i&j%C20ReQ(Ty~0sj9;IPv~=zIcsnA1F@=8%k-ATt&-c8ED}G~Eu8%CUZb%{pR^Ov$emn^LUPB$-fPN2)MK;KOx1vl{T>W9`dtwUT4zk>^wvI~@mOW35k7@3j`)axa2Lo0e)D$03`_;ovjQ!i=QRJq8Ks-`DEGsZcD^gV%R)8(Y3W{LPQ&A9zI0HDXbC%Weuqv0UZcGVf(vDXM0*3j^N|8HO-h3sMH!oaBe*4=<?|lD*V(!Ba6S?`qg$r?&y{|61ulM}+xBv8upU7iBILFV*Y36M4U;j5R&dD=hd+jw{F=Q*R<<I2Ff^uwcOvU6&Q=WavoUs|v^!5f%nu;l#)Hd6^!)O_7JEb?O+NrZ=;r<^y`JJu2k!^l*<3E#M{;+k#FMrt3J2Da29i?03PgMsr;>r`IXsCTs9Hxze(E8YAS7CoMu|hPu!q5{uec>y`c3c6-qB>Le+9$4vf6u%cx$(RYt+>i87Yo19Nq*HGJXU>o!8-=KQH@vMKg_D58GG!G9(sqybL}=6cQ$>jy%F~|m)mVNXSWF}8Bm4|$0>-NFHSmW@O?b8Bwp>eG?_HgsEr;tnlP<9r-6G6%qg3W(PuF#+c_|lYS!`i$g(5sm~5&Y<({w5;@j=!C4Ib!wJE`#^K!Ib{h*%vz|`?@Y1KdcaMEq8FV&aduJyL#76VE1rkTU#MB;pr=XXJzQtDfVtJ=8>a%NDsI}sM_A-7PB*iUgt?BjNLBm~j?)gZz4i#7A~ZYnTG$aVPJiV$3TBo$-=W>wCy$f=x1V8ODgOmT&>6qOk%N>~+HRUyJkxn|@Q;6;uqC5ujqF=X%*Lk1mA&LOdlA-^5fRrVs47{h2~r>wr5J)L&#mJw1h3q`qYGnq;ZhUH<T18a(=V`|Kx%d%!gZd;I+fPOKhL_0&*oLervy6x22LZ<eC1GKD+S8peooq&X(5yJXGND!~9l7W;8kQSJy3T~ZClSxZ1Qu+UcB2v*3WBKaJT(0_9s`9KEuL?X)uAZLcYA^Q6jYTs-rS)$n<^POfJgW$)HD$`DfzR!RIqg%gX1MO<EfXDNTQfRT_7ukl>a?mn3j3RO5#?r7kRFG;?wjb}P+sD0?WoT!<*eo~^<>%m(Q0FCNl(O*Vr@wg$>|6M$spv6A(hLe()+A%QK`yBK>^EwG^{iO7_+qGpsq{lx>m5RbIi)3B9No%R9n(fttp4kQ{uScIJH~Bj9zTKe3eT-g7-c4a(=s3`p_0du2{Y&!-HsRDD@w^!4LJEKgtZQHkgZoXSy6BAr4aGq0;RXM5`!$KM2$A$BCA;wsC(-Rr{-ElV5zLVOOq#V1~&^nPHq#%s`&AinD?VQGgnjHHHAyWmyuyVGYwdD+tdZLdwa_D5MA>X|<)jJ~eoou{_Ku$-^|ZOhp0f&4~ASDJF^6vD%0c+e-*MPU75(cioo~$Ny_$8mo?@Y~QJ<_ao8zzGrc_+3Hq){;|)B-;Rk!)NRh_>18wGDzo8^FCuPSdyCti0N(|b4SG`p;3pD)B`=G<7eVqLyQxggb&iKu*S5DSFGIL|(sYLEwJ9?@H%R{*$2xlZ&pK6hERlDk>e4d-7~%NOv?jb4Vp*Ju!s?#lz3)p-Llz}gfZ2=$-R}lQeHBJpv&FMceZDp02CBQnRao8etabbG%SaACcgwmcp&?Dh<AH`;cab>d#QaB7Aiysi_04>w&L*5pU94yHUvSGw-?-3*eo(h>5H4G>3*VSfCovUeWFE(e;#Kp2rYht2(eicCaocUhVhyGTS3NCe;u^E`cdl_;?xS*_f@{rQ{6#pv?MIfG+`&rh4b0Me$A879L3?`V7sk)}fxF><_&a>!zR~+_O>Fr#br<N-K4lrd<^J_Qn6JGb$R2HDR+ET6?^}}?b9Y>JmgIiK9PbLCqsQ$#&1|i65SPgq*1|ngWf-YGg?ogjdZ=%;V)`_3XWTZOwPt%xzgSnas6pRT&B~a4+k$Y1K^T(py`!OQpEAC#eSbUd75hBpe&U!Vw>=fpcB8iD_yhQ^3w>CAjx|?wX}&WxLT$8UBZ4Zmt*>Ex{MNr#$E6b5vl}slEIqYVuFrK0CUIID0o!CFFc19by&hAQ+;zpIChi^mj_ttL1AX?`h0wcO=h-uRhJr98JzcY%>C@KWhWX1=U3%lX=sq|T@qe<gcJ{1URNe6)GGF1UANmL3%a4?h{DAWJHwdzAGZlDcFx8klo<n}heNB&&Sa;Lp?*~7Pmb}Bd%i%AXt_`?2(;z=oypWHw3QK_()y}X>iMJK<^r>MQZ0bPOKW#|g)Sw>8vkm`U50aa53TXb=P4j*4nQ>%ar%jBTth+HrgPqn^NA=fkaN;_{?A!qBFt`a(AM>>t_<#4bA?gQ4u#OGEm8nRpW7RV|^u2`(nyiuR*ZRj(5k<wfF}`ru5&SG9ZaUsF7u&RQ7olI;z2Sw%OvkRdFzbQ=kOVU7sli@v+$u1m$|HZ*k_u{ll&<I3te?A|c;lkqd4Mcf+KpW=;^AlV8?H*zf_pkfcDIb8iKcADB)UnM{3EI)Z_`aoCS1Xq{is)W^ke?Pax=$+!9BS3%dbaWxax1~m2UcO`dt6Rl)LJJAAb!dl1N3v(b!t=YA?3WUx(mSV=DBDZ)RyQwB4<%_G4Rr$2&8oTUWL9)bHivNbBTIG>-SoAbsq;u|MdM(wO!31Iu&kQtsn?6k%mO9kbeb=P>CO6N+lQ$xl?As^#khaF2I)!jRu>Xm;s={UQ`T#&o!cNqk^DvJxLqi700o0K6t5t4f|hN-4@R!pp3#GEmh;NmG>OtYS2UX@)aa;VKol%ve2Ud^yE%nq`+yFbRjSkW%7>12ivNW{$bzvllVJ%htqDsLMS!u`a|!xF3aSK$9UDM`zow5wAA|9jqwl(F!^atJxS+6`UAV0Uj+bLr5sWS(ay{$_c}H32K2*&N<3)ftdV<p-Lz~B0?FgbAedNOPEzKD^r$=u_ZmdmNe#ix#y+VP0VP)t2+2t`@$fHmQkHs^vjDlzp9~zd!B~t`Hf(saBZ_G1n43T)-4YU3=4f+S9nRE?`>DMxiqi`&f7&~Lwoi9uo5`SHS?+Kh2@AmrxBy}5l5d9*z`x6Yus>c+49i|&(OH*`)r_+E+puq6uIv!`$`_0E{GpI;6j_76H8ZzMc(sFl(BYAqv4^IzAe2&TiM;VM#t?&6$^ppXw?TsU0`hecg=iM2r2l!wolZ0kpt(LZ=*w8-A6*aD71a?6WuSZ`(VGI-1b#3{ZNTACeJ(;d1fl7%aL*bN(K?aM5L6bB&UpsI>RZFlRd9co@UY$$Z|kAWV}X%mNhB}Ac#RBw}G)*;#e(l#MBZ^Tzo4)zx)OpKl?24T4g3V7ZuP4a^rTP{cgY|gl$oJaQMq~xe3z>R|jGn(rB%}TobhXWZ`q);wKA#ildh+E<|qE!i}ilqbTCo0fjb!cq?<f5~C+J!c3?Rx5>-~1C{TFU7rrv@E2;cV0oNGz=GIzJXmccM`IDW4UgAIF0>8`yQbY79-U}9^GkQ4*&Q}c$GimUT~Ea)n}_Ph=_aSz*FLHH(CMTN4sSe@%LGO65Dg!{FnHR~gQr`W|8V^V2H-G!z;r{a(Vu@W8jTu^av?shy?5Gmw#~ZreMUQ6(Y*9;-J571>kpW2X=j+X_HpaI`W(Z76FwX`$LIz%j~psg=#oxO@^ms801Cv{@(9_UW!mudxx0~;_PUk$^BRgbBR_sSh_&}@O1o2MFu*`!L34Sb$++3k+Q%u@hb(R1h04N~$z&k0(HIt!v2BT@6}K8$U6pda>93~l@KJDN_1m0GTBwFEeXeveB=e`NI7n1md<gc%${8L+1?(GEv2o2Mxz<{lvKEeI)#t=^z0gP_RTefc=>iQaHnQ)vv38sdThxE=nC!`ug)d(ec;}CYfOmGAnbqcKF|Xlwe(mAGA^wnUU#IO~{BAry?3=|@P&tTu;<`fuHCAcMk+BTquu~0V@2u*9gWr7NUyms?H1(6JwTm8e@${kZGh_}AVw154&b|h@nvQwK%N|6}DJMD8Z#N^H#&&0%U?Wc6ZN@aVPMl{e9<A4-K3a^S8?Bt7$&BBNGmah;vg8?jLn<<Ub-h-HY`8}Iz{|3`+KO?L_L^hCIx!CE6ha4K`dYM@5*L?w;4vZQo=#3fpL@dxVEjbpWk&6Hxf6X(TY87j05`^V>(Q3|AYX00jZc%9#>nT{Hxv75I|RzAn^yKCq)nJ3qQrCDukM*_L}}F^%c5vPKrbzO9^;{gKMJC#&$egZQG}y=;ENePGb!Q2Ok+%}aVtO<?z&WYK)tT`@$v&_ayFw_R?w^MHaz{<|LmAWJi5~85bk@YVbhur-I-%`))%VI3bK)<280<b%aUcZz`PU)S49SEoTdc9T6R;7d0umlab9HzfvVJ+t*lIIigGNnF}cGjrUu+|Q_(?`J<;0JhP|PgFzLB?g))d6uAHvH1vnWesMn4{*Dz&g8A~U9;C&IZz>k%vY`9~odAKp}y8|jwn-P!CvY%NRHGlf>ht7=OYVZ?9hTNm*Tt2&TiAIQ`*Yq*^rn#>4Ff^!_G`AZ4*n2S((6>>sfQ{*d#KWLIL4Rm}ec?!njuUJjlM`BU!j$D;lnVY79if1TP>>Tqup-XJ0HkGJi%OInQ!Muil2@=w(-bQG7F^VTBTOsK%kgY+dS{CO1-{;tW>NXwf!X5mk*A8yuBl?HX}%BqjL?(^bQ#RCT|1s#lU`$vM#`>9&zDpJVnk}e2^1+QATJ0jE3QC8nNmzu0s<9fNSZf}D?#gu*BOS=#=kUnt<Vdfs=jhjzUsGUu6XsN??G{@{<UnJv1OO|zt3E{b}5;;B>P+1AO(|lw(A#ijanzKIUxPIRnPqA*Ay?SJMo`&7Vfowp7KmKJ&QhYv17AI$xE9P<w8rJIBgnWe_GlJ%S^%j^__(~0ko0qXZcR-h2M&wxYG<apIVg|pYa%(3FGA29Kp15n$_Ml?a?hY_qLqIu8ujrQh@vH<fFeeFNFM570~?{x&be}dGp^#^56OYRbQ1e=eeY)X8C(pIt<-E{PRzepUBBK1rSdrS8C02nWXtBK`3(02(39`phAd~6qm4ug<!Pcg{X3>EFxtjM+|dFIjsQ&Sw#h<KtNi&FuG|trkfr?y6NNb>7MK0-)$1^+T@Ykv%X^7!tldidNJlIix>Xue#M_Sm|x7RaNjS(9~dRV7g0xan8o!*rSls)2?kw59jgMJnl<+=Wx7u|BJ0MCKgLC6;j2NjD?gHJiik)fgzD<Ndmd%p<o)Ly331A_J?15@%UrOO6OghJ6mk*Giku2k)+I@EELdLESyoi6$O;ZJlvV%`xw<AitH&$sQ;cahc~4u7pSTER&*nrO41LSw96eC!q2Y`))MN(B(G|OW{1o{XX>m7>NU+;)CM<j7QlDp8f)kdFlp|WgjM6MEiz4H^N+Dz-E2}E2QeI)4qY9U4CJ3w<%?L(Cou(<qWtjrO2+CeoM|7r|nZNX6+)#aMg-&(W@Hn2R!s=epU|o5(B2!e1_p{1*oRjuyy5&U{^+o=h8D$1_k6-aVti6}<PZgD<hT3WdT<HgVCr+k*?6+(~4^uYj5H>NoCK0qPIG-1+6w0O}XYycKtBLZXq?zOCJ5H0ODFdMKI#o-aH)X$3)DZZt5@%_b!+kUe#6xYD8v+qq4o9(&L?_&G*}3K9N9T3py9F}3AL#=ZmoPw-We2(4PonbqMT<>ab&!?afZ&MOgXF=VPliAJt~*ir(voN7+H$?WZIOnXfnmmxR{Ha|V?55PMUv?=W3*=HRV<3m)vkC{<AJFO>o~<^<W{2qwt|=(Q87c!1VS0lv1Ud~Gb2MedbKn&jFB18%7D^E$0V!e<Wyyt<wYt$UBg0!O)`d9QSh1;A{TX8$+->#2<ueIEkeODBI($DKfMVu&pEX39PBL<!q|$Kh1Vid#^pxE>O(K0G2p&GG80Qo9JGCS{;WaD=v5KTjUr^P0e?Ea<VKNd<8n_aZEYhj3}JH5!!nb&QqVm2A=-*#%))ocApo$nQX`c4D;Z~iFk0kTE~pB?F{voaYba`zr<`&@QX)WJXBEt8O+`^v7(!9XBLRnFgMHc!b}x2eJBqLetnoL7e?9iY@5aElhex%>+FTxHT5UngwyzG~cDR41CVt1+)Nc)%dTc3Az*16UDN~eZqhu+oJYx`7oKu#Thzo@3l9P<4CC@8dF{N6RRlMR<KDl&lQ&d6e?Un-98Kz>KDI8}CPcT#HKT-MRMGVEd+|cKjs#?s<=~17&(K<%j-6spbe^XiKj?&x-%kW~X=iYQ&mZ2=L+VG|3(YWW7S~YjaPJ}93v?)ft^P8ky(4seQ__Cou4BIIx8njY3T?9te!9<6|3Cy@dqFYAodFa8yzl-Yb^|noiDY43{rhc^9d2#|7=(x4}6CKC-;MMdXmq^5yZ)$l4PT%)sQcrnN*Q1cuezQ1gKO)O^=lT~Giop*(-H~@bh=XSX!_Jc=JAUPejrK<1v|by0&`_`z%A*#14dk9mSfB9?n6fO)AsM}x#MHYnDeB!A4)06-@B5O+Ud$l8;dxjizMBDJQRj55uQ>kD`|ff36P<5SSC|bMFRdZ5Y%=M7LD?S7;evy@Ve2b##iWg-P7j{_IA-$gYOtcTnQ}Mk8A~xS<N@^PJG^;MX-@MK>G*EV=yA;A(eXp<&U9=++!*mzsN)9H!{5V@*QuP%GfBjN?0~d3>1Y`NCX%7!LG{njPkONV`h@A>7WTJiTG9}nli0)wY(1$e>2}wDUF&q4%h>7Qb@`c`QSoVAI;-CLhe>Oqxqh^{=KA1t^XAh{SYwU}Z+AB;#Fut|W!fk;EbGmyflT7b?XQn$miH{_wSGI-pE~}B4gOH<yI^O@M%i;j3uWi8>A?cw=)OZ*V`WNw@M?V}s<t1}o?GqCTwe;)+kUug*WH?YSb-HgCiLD)Z1%u*Xm2V|s%vyngVBEC8HT7Dh&?k9Cth#0aykh+<tX($%o{-hUJK3yu9nV$+~VtWdJ^^#_mas&4%={K@tcPp_SCP$npaN^;C_b2=%05YIte*=Z$|eVe)fj@85%49IMS)M-B0^w@96in*Wn$;<lfec3?H;V?Qz=-`?bAi>|f}nud$%uFpO5+ldMOwaIqSQtBCJ6RS1UA<Fp>AocYbnZ1Ntz{4686H8zshlfl6@nGBAs$z)*MHyUb5cMQ?ShKWvh(HZFvGJFe1{uB-wyib<?^Jw2t>sCXJ-{hf|(U9z2cV3$vFcU*Qt8ksc5#7Ac;C}h{n@5~PcC?Ss3yX)L)_3~pQ^oOzmAT-@uXl`*A-jT$wP%fb@LTqrOhFhWe?L1f7sBelN~uz>k>8t{tLER6|HzecIje%2d{(OwUz@#p#WIIoy>@vnd4K*|Qt<K}xw1ErQh987(P7AsThITUB#U1jbZCsqpZ`sL+0={tMBe4fl^OXo$v^n#?+CiJUh*h{LMy>(32Ol}K_L);5&2goP+BsK<?po43zBCkBpB3yWug`tVUP*}YD~z;@f^As>6Cp<Q);-6RTf`8mBkKq#bE+}5sPkghUG#ue28Yrq&$I-qG@O&K0AmpdWF0Dp0@9?f%7CrWNp*y>D&quxQ!h5hT~dV8w}t)@Kaa_xon5)Sh<%?Pmj3lnWB{5)LJx$1|r<3$xWuh`)!$C(-EII-nKmxJ4nv!M|)kh<bv0GveHp{v&j#sosk}t*=BoU@Z{BKk3j^e{j1UWH*Kj~cvI^q^`^8hXl40s*`(Iq$-S}FVQ=iPmao39oW=_c(z-{shgL7dVV%&hJ-dJXg7Y5N9bcAaRQm22fc{mQfH47(vs#ztE9r-L0w6lNax^tTJf~%u0Yd5`%@M|FS(Y3M%<HVmGFkyDas~l_gkePtgfd0;!0I|H@-YD7^cI}#Mn>?lNALEd+K&5vo!5>Od+5SeF!ci(3tq2&;%E{I%U<v^T3fDo_`}e`lZGkozO)bS1ufmy(ln@Nb|2g3=4nko`w^6`mpD?NF~9u@05XsPWDr6|O8!O=M(Q%Haf%DTa>TPjWE2#sTm?c7DvB`5A%P`>Manp*X#t^JHnN(PRaTyoRpARJzd!AZLU`;&;XuUHuyo56(BaWey;AJPbc9T8tQc>TVADqKM`vqnzI-DJA~w$P5q!sSA75mU`F7a^tQFI7FA`*R59Ds=L{z$BuHkHT#GWgY>AHjom#YW)GS2E-OoHcuJ7`lsFME}up66Qk^We8rM--bjbgQdT!mHWNr(}G)dfO*`dl4W1s@osoSq@2jy8iPeUC$F}R!=e#ap6%~>^*PvI1JL!WqB2h)|AO58>v%9nRJ9oP?Z^zYdkDUkf#~1ScYp@7HNhGxz?)-oMj}-ILav-2av}B<d-vmyxu!&U`>O8<V`0r=G91>$DA=S;|a^_YH$>{{a+jYuZQL@D;RGJZ4Gz$O}{P;S~MYg__Sza>u@7&z3pdhHR)I_iOuk9({a&%Jn3NoA6t0{Q~vsKe4j^V9y6`$2&cDuX{x!LwTxmRvK*SG)A;^ArU2rmU7%m^hka|I23?0uw=VX`cKOh=AGOa2RP+NUg<Tyd>3~4T8T&yUCOX{r;NTf<*L34OMYriY{hrN%JN2g<uQR@xKFRND+-y3$^{(cvIW3KW=EWy%Smu5cxoMx**gM@ETK4QUeUjhq-jhl74<7D8vpYf2-C$@Nm)VML-+lbFsc-KMT>if6fb|e&u@}1^-j5l6i_KTG?Ou`J8^5LP=@|F5-+nX$LNvhG3!O*0K8=nX2lQXGz&wH@8_`bzsQ{?z8X{Iyd0tW?eLlzwUf`li(;}-&#_JLjL`6<3xxLIX2tlo=PiyqT3U$CsSE$o)(QMRSj<uP8SFTQ6zw@4oRVaUH_9%QRmKsG4>1*;EK4<KJ6Dzcd05e|gt>3#iFV`{pCMxouyvr|Noslh`Q(mJScjd(saJS5py=ASEy}zmtI#QTs*Oe3KskJ*&vlAcl)B+qY-4w_++*IZrf*r5EWn|&jcB~j})D~S4aL)pVUGi3y?K}%AS6%vbf9)3@3pkWaaEux|blu+=C9ec(>=~zn90F7mTv9+q${?+2Rp&Vpa!SZ~Di`{g^Aa+eW5hWuS;_%rl&}IYQSfns_w-h{_nVqky~Od8^kYYTNZwX@!jEi)mzPLcF?rC3N#qtU1qvx&bkVasS{gmv$j+|lt2xFhrzY6Q4X;JyU0++5eq7pqET!;N@qCO>@wzmEs$9Lu2qmzp7{nP(Symxb!m<Dqk-8G1gqRY9FklpL0y9#RJm*w!KK6;n#p_4+q@PZ6z5ej5uN)0p*)D9^P|t3~4Shkg6|Yud%x@-=MuA+g*zJs1NP|g7_<UQb+PKiSwfQwQK+5m^)}d$)nqbH+2WHuhDs9@qLQiX=BZo%x^^OxiXg&RTZ^-u}g#^o{vK?IWdyY(YduVqV@7)ko-7&4Yqf{YXWmNO5wWX)EO*LN?Ax@ap#FB%>`lxTC%>E$KG{2QZvdIPkwP+?mO_pe`Ym2%3_3R_-vdh~Oo|JGkF4(dP!~Pxi5(hcj^g-lLj1s14bI?v+bu;@%{yCRAF71ExhX;BM>Ez!%?IG^ihv++5TPrxW<4bvCJlNfV9P@Hqly=uu88>80w<o)M+Bhj+c*L|xjDH+wwN2Y>&2k$#yc9V%kG&{x&*#Y7kJNKDn3fWt?`3pbb@u(3r7Q_t3ujs>g)sg<YiAq6m!Nz3^NGM{5Ju^sa_&~pl)e*HjNWr<=V2J^IEi!*X;@2Qg;Pg2pc3~>iK7~N6e`~*!w6dS__$c*bwjh~+YfNb3(N0%IQ*v9MxBc4!o3Rifw@6w(vo{9GlV}g81V&fof<#Z9D;E{8`O{5IQp(}J>2|Ej0FI`Ljizh1ef9wUb4fQlnj9y*EMHc<SDKQgH=%!YRR70Weqc0W&ou$&nt!K6Nuy>%>{rIg7K35^e)-&d3jK6{*?=0=a)MXw{m|n=90ELe7^jA(=Ez7bO1J=q66@a{O=KU0G9QY?lm1+v-?piZMfozK%d5U*DTDjlY%<sp{ql(+51>Pb-U#_cJhjzaU?FviI`}Uo!*YaBcCY3A|L74cJ2Avy_bIb;^X+l@!}I06Ff>Zq-k1XoThmp|6f2>Rw-kca|9qy31=v)L0#7=2doe|DFiF!EFkKV2o5k6Vyu2JRzEnF`hhdhceO3N9U=t*Z?fV65^7~~d@(`$)t;1m;LGF7Ug7Ut<kd`Z&Zuop75we$BEQ^asdAQ&1h0LHYCs4ekY;ICvw|R8P!2L$VUpnrNY^wgs!ZgJ<xC(dxJc6iaz-g3X<ELl@zb$=`p)eWct&-W1}kGCfB)En-8M`Gyz!$}TqVYS^2Qr_pNW1dukXz_)f)bdHv;<3?(}h|C4H%d^DUC)Zc&ss-+W1QKE8ij$mJ49*@9=4_+S~xM|i=GY60s4!8&6_N=ph6r2>`|WdPv3gk^>a&tRoeP?!}Zkv~D6R-(vjP_Z#L@;JS8g51bfsj8)Il^Ag=lNK=GUROJ&;M3Z|s?%;bf;DLYH<b?5xp)4n|Cc|Wo%_olubjR5KR5oX&(Hp9=A69XkCWG4drix2*l@l4nLJrgA=m3u=6^Tj|9<BV-1Oe0{BJnca@5Hq?>ErrQ!d%CVB75i7VC?a?Jy1l8+|Tm>5O!hCl0;;K@-t#FDaY<<i>v{zx-i*=r4b$R{18ewHE~vRwt6z^`RXXGUq&Q6929**$TK%O<df>kZl0@x}Iis5~t3!?UEbH75_5GSxxC%cC^ONeT~F~Cmr6Gr^%bUK5J$G_2wTXzQxxaPs@g9Kl54!1GngS{17gmPU5cg@adrpgMr{>FO~p%9P4>m)?D>CKkgv3Ip0hsM2U-@ACs+Cbv(kPPfjci65+1I1(S`pVA`{x16Maq;wZ2Y>rc$2e9Wu3kaVlBUAPcjn|9Ytwl;?(mC4#zCd(m@6rXwEZ`Glg?3-}cRniW*;*ia2QIhEM?m=?F`ExzTrm9m(?#4^{i!kn(LTW0px~m`5b027I^^7!|AAUILwu#wVPEvDkN1Ie!d0r(Jj;O@;T~q75qB?9TLK%6r)~n{}-HbT6ZpA5OLV&suB?TzU0Zpk85L1XVDEkTsfk=(3GDn0kDj1b3S5OL$Yc51FUbl|dttX`_+KtgkoUUlcKC3GlW(_^Hm<jT;fbxmv`jF5Gu=ymIcN6Q2zzSAxB0%M@NArf6C010Td&*3KA4fz0i)P}s&WpWv0O<_5kef3D*ksM|&0E0Aw$015oPp`qB<(I|IFGjjS6`5A)tlO_j<6(_DPcck`+pox=+;{GB+4t{Nj(dnk#PuoK@G~INF!C>-orvx8=$FapSxQMH)=eibF0FD+4888u0Gj|9J((IncOo*L^?+3n{SSzD=$HGC1nJTaC8NuJgsYyQ<@<;Gs8-hWezG<AXxGe<~Tz|&K2DeB56(wHNi8G3!L(-dK&%hcoH8^;@^if_+3AJTpy6hIGB9O>ML;yAU7kBr<JzWSHB)Dc~{NRo(BJMgowQ5RAlzY#`am&s&O{j*0uyWU_g|qG^~u&qJ{vmD#rkoStSU?l#>#rC9XK;DHEtD<bsMZNQz95mj-<Hc0DJ*{8`t3>gZqd?-Y6FEu?$WD-pCz*~zQ#PhFj#Qmr}n-u&fv&T(n_<`w$Ue0e%-EKOQ%YKe8Y+V4|QXpbx_-tUCEoq&ln&3sFbIByZU@B<sp0BxFZX5`Q}xT^z&riX6rCy|f1U^pARwX)X@U^G?X@aV7W(#u_+(*H<fwr+Jrd~L#(lZLWvW)XvQ_e#S$?}h`kd3Fffnupl)z~xmfmuETDOt9~nw5G>oox{mGnI`MRc!ZcbrAdVuNz0;wKslyZu#!TKF~9{VSy9)ubYgjpK$VIrO$AM>Do2c=niK@!F`oDIaxCAC9L;S5E8Vlk#u=hb$I7VSvqOq;H*A$y1C|>nBW&eN;zX6=EJX*OaY&myh%00ELrN}80#0oDh7R<bZJ&BZ<^u=N^-|akRqFfBr`~T>D77#3&=tJg99!Wd18FIe-Ygm|Q=6v<Kq1OH!yvCygh*MXwC0S76z3vUl!h4as!Z!tAR#c3)|O?cB&c9xAnh1PdrTnhwslx#5B=lZ+p{xQE+(?)?4MHxBYIXYTayoxaPVNxvx#^6*}S3z`ak98CYJ-3{e}M*Upe)^&&|%v#T+McpCzD7{<rx{b@KP}<-a{Ix4W%BtFOp;?Mm{mDyJ$xTG`FNxU7g%iYxxAZr_YLu;UT*m7{yQc3qu!mpJo}=}&_$o!o8@IrG!>9~;}<@jq3^yEghpy9>KUhn7n3P>!pg%+6}&p(}~GVf0(ysPS9wH-FEI<iAN1D6e`|_Ue_e^<OVa^Wr!E<lT2K&rAEF;6J~(Tsxc8=(>MW!$Y>~;$>-`hIHWnac%yp{nM1cCI9;VyNV}BHgQ-ln&WIQnjfJP?Zbub?r{B#|1u-n`!9w>D2%i>X65(xK{EUQF3$2dlXJZKmEsrrjdXbCqy;{gbPr&6RB{GR$gjDYPD*KSuP74IcduQ3ceYMkU9O{Kc0`ubi7UM{<b|5B&@BD_q$!}&iA2m^lk--kzhvoTy)8$$?9%$uyrDUjTV$`ev+gt&Y&V>5aL4CS21{EXC1=otwq~=l(i%)BS7eKoL7YyMy-wbqpO=5m${{uTzKOhTX+%-P%Xr%W6+92NuC>-Ww2{Zt*LaYPiR28Pkhbd5rE7{CZaTTl6?kkmaT>h`I<pnCSg%a`B#2q__N=od*Td?)=zcerR5`x0YShNoLq7JObXQ)HqwY6cKH)5vj~0%xlvsi2+rxRbV~|UumO7^1HV_j76Z_(vwoQ+0t<^wQtkJSf&(zf{NSAN#$-*DipnZEjktXGx%h%qOT_t}P@+q%LV^-N}+a;R^bW*=2iS?QJG2N0n-;JW7A^aJRpJ+7hp%<U_v5MbEY>l^EN!w%HR@L`c`MWb$E=?=n|8gyjUm<5JeyzOB%lB1tZ|L{8?VfpmAk=F8-%vDcj+<!YOhW@5g2;8%EWfOIb@fN0drK~!-%HfOOTO(h{u}Y(63^yvb2>`dHPyt|TD%TE3AySaWabjh&fD_@YAywBUXN{SA-zi3LGRUa@p(~p&J{&)H=*0CzxViS*Q9TXgw3miwO(=K(;xDU4}Z;il&faV&iwk?yzDnVTh3pcpLLBgC_hb}(fvVB3_)wf-$>ieX&vNlo9K2q8#w7F`$e}me70Q>t=4^aUHbZ7GU*E(*W8ttgTZQk(}AP7L75TRgY^}UWm8#Y^TPd!Em{IYpJuP!Z@d7V-&nV9jZgQx^z)RQZ<T9!c~(U=<$$@YCPgks5WjYHMqHcKU)~kD`K*U2Y{e`k;RCkzZD5=}e<glgl0Gt}OC2s<br?&Y^d02->*~x~zj}|)^3f<|!gU5{i7+a%f?=AHlt5UPWx;t#6<&-81OO@#luJlbW|)$y%*zaMo~3y{ULlTGh{s$ZY8oC(ptr8&APq5(;*d|yTQ_3cfqR!Db{I`K6lX`ggRyQQzKd<D(jdWRoi}$z&xYY1Ik@hk^$<RC2KK(KQde7+4Ss#uS^1kuxZ!>43lHj%pHCtI`2a@yQ=*Z<A%15{@Iw*TofzBhr>Nb)Bhx6i?>O5xeZkY(2Ua6LC$8BnyViHrO<P}iYA3v4ir3SYf`3j{*P*X+L?$nhRzUtA%*u+ESw%C>7|n}}*K&>$B;!aiz(6@$>3UpJD#$Yi8B7I(W3|pxoJm@B%YsF}&ba5v$NJ7*s%7_7s7z`AoqEFvVGLq*!y|V)nPKFnhXX07MiexO!VD&z74$;KNuA>)bpVE=l++=ZSA}%kK?xbjN`M3dX<in&$dN)M$&M~XE~pUF-sLRKFct`MhIme3LB>|^^jbZmykqJww>?4Ame>7Vb)Y02sOwIi_iXa~ixy*J!;;<5(ZxHSDEpx+`P_7|iGIV0z2}0C0U*^l#Q>JJpN}tUYz-l%nBaV*tf5kI5dc;dq!rHVLi%}-^OC?^TFF$}Lm^EeDJzlXby?-SW(6j>D9g02VK%mgry5Ok5a$e*t?ZUFPa9UNJuWY6yM1zvuUP)p<03E5%HMGJdPkIPKaA6_VKaw{r4P7`x-4#NB9BU^m#0so(|g5CWW`~g@|0zyE~Jl$<?<L*b%t^xFhFuaEKMdM9D*zp8O<>$YR*YPQ6Wz(#wPOg5~yxRawW^&o3r+7B}$NX>Yl6}bu}#ie{-V4hg@=@-<?*)<+97f4S14<;??e-igJ+O%0H%>k=PZR*9=>FWA4%88B3(B$S84-W=Oh_1d6Jr(mVnHOHN8y)@jZPQOPxRE~+%Iswyu9OL0{cs1`-4^rZ8ej}v0!gxCp|S9%xEa>SV0%3^M6aF^jmxb7<O_YJruB#e5_*=_9K^;Jm5j^2%A2Mzk<N1j&4L&EMQ4vD|vnFf}cUh)80(hav-f9Rkcy=lw5<Ulm54$QXW0ie4kIkoG74((%htsa6E^00S)<Yy}T{DBo>2<uStBUQP7*Xlp$ORpNpNZ1zCr}upj;=s{G-1CPWoo2C^J8vNVYk@_;kwGnbA&|YeXOqDfKV}Me=HgYDri`-D$c1HG)I3$eUy`f8A<0Tu0~P+waRDG<MO7fmiwsbagMvaJNLr<!z&LvvXLf7Wc((cLSzP|(voPANQ_jJdvM**zjo>NOAi4SK6gYg!1Ko3;MP^T_mfJbyeD^Bh(lt39H57BND0x~vLCjSqEzT1o$DH_SV|ut`?#va-18F9-hWc{i1((kzKT$W5Gpw8hugz&g5VcBPL{EGo(K^>QIe$rxp)2;zAdzo$PTqd-?w7A=@a2r`E_1KvkJ4B<>R2y&P#vsy_OI*T7<1e6qEcEN%)?##?Y+`cYf19oX&LfQH58*CxN*-6Fx#0rFVBf%k+ZRW!hH|7^twk6wDy6%k6EJlV2qlf4B}YF*!?kfV!GgV*{N(_Plp4PJ<E+#{i^9g*OBH*?wIK$6p|S%<iKS#6J7JL<~2;7(LAivp3-mvv&z~shfjK3w~5a01P$YFZNIHAPE2}!#|Kf`U}v(~5AM#BesXmxVgoH}o@o)KQIj?uCUf9i`AAO|?wM1Kudo?T+jA4}+DFEDm~;TsNYbq_UH^6M3y0@<u=-@-%O?w;+1Xh|?^?=T##eaY+pudu^=9q4XuOKOz?GyDD65lEOfcxWMZ+N~QtTOuMc0h~Fz;_O0#hAyYl;O#e1ku7H7fbujT^cbvr1jpXWTKL%nhfu)eB6L?Gx^K_^MfP%l1Z!XSRMb?zc9)HhPKrrn??xwcws+$FwCB6ZN8Hz74hRI%<%msMEHbKe?+%`<i9k)VEu6FV)^4HX{2*zty|iI}yL@gUHox+$`3Vy<_2I)=Gtk{M3D6`!Rvmmh05Vz8Y0u(i_Eql$#2UvUx4xl-6L<SLj8sIn%WJQImUNY`4bIdjF`=lM`0``d2wM9p;%jw@XiY)mx3Z)B83y{xIB2y>>-EjdJCD<kLlGPA=^79NJ)ZHf>6Sp=a@LyL(9UkH&NLi;mSWSQK?xGm;9Pk*wwd6NJkm6_xy73L$|R!&O!TS}{g(f$9{KvVbhF3XqPR&ujaC%6X$x+7*1^POlnE+SOFS939JPhGBPGRT$6c1<UdBzT9$I-lVclu>KT`Pi!3RSHmfmO@Heu|M&U1`AhG2hULF?&ed0jGG^6MfCu@?Z@(9b8uuV0VR==*&;4rt{q)>V=YIb4pZ!}GwP1?VbcF3GPDx!=sKhlSc~yftC8!3XNO8q!hNJ<dq|8`RVwe-ktE?<gNdc%TmUDDk?5VXeQ}A$e3diR3g_u)#Y;$_Ok(4~&MOb^rFN4g2Vo;UY7Z)!yDvc9ItGnm~m$&_@Dcp3bfd(14x?3HsuOx%(1TU_j>lKqppE~_a|NW^gOf}ohG3Uea9(xzo?K@EtK+9Kd*<ww@wr^{smy=Ln6V|0BDX<-qu8z<5AbO-lvm9FYF>Js7{6tS=c3IqL&2-%4aDrJIg;mj`pvHaX>91)c0KEf@vDGd9%^dk14Shw?DRDc_CvW(DPZoZ!uXr;Yv3Bt0;JaD7Kk>B3J7d;+#wjq1)sG{;VaeAQ>D<NoJ;z<$*+#vdRb8-WD`8#dY&i)EbMV@Pi#e!cREbw_(*N8yC0_bkJD#p-#jH4=<r$~LwJfsLU+SUFcj<@Pvh>zb_~xsq99NI%=A~S#yv+7mWMMZ<WW+wIKdb#pcl2v)?(2M+{zDsl*6>i%N9c>Y&eE*KLL)YlwdEogRx+efF#OmTmzz?d1sg=X9_Yd&MuCBNj2<B_ZeYe<&jw@cvu-=4Lo1`+7+cHEL~J`&cPosc5F5+xt6Fxry8STD;>)b}OurxD!&!B9W~*5h9+l+Xa%_pd?84UK?b*ipVhr8(IJda#gFxopZp2lGpF~1EeH*)r2I=j=^7fcQLEX;SUoU>L)anLZ@~s~Hu+cK{5)|{Dgz(5ntw5f)-0!d#{tOtMS7g<aqeA3WS)_~=D8;1^Iml_v2^BeHFwKctw-k)=5^<bo70powDwvim2QN^KbBx$|RZG~HwR-vg(@17ahe>o-&j|ONWsVRy`$5I)6iNsm0@pZ(im>{iM+35vQ&fstm8Jj;X+=>%GR_NzSPF|uWNA^;GACJ?GeQ}HLQskU2W1X{sK6MPeX0xjwjB%ZzDFf&xeV%}Tld)_zvY3V6=jVc4TEDp>f*<3h}HzOfcAVn0B|!}6&#p~=G~}~cE#nW2sBfarGHv)uCJCSpHd1l*`SCVddQB0iyH*^eq-mhBMH-Yi-~rZCzA*pzTx;XmM5K{qIclcANM@Vn+c0_JoRQ&qqA?hNBnx>UNVHYWO-b-^ZXp@P;;n5LbH*gQRg|&OH^P`3pslbh(Ja}CFhc?hSJrAu$I<^VmXzNEUS24!AejBQ+YH7jE!fH(>r_Iv-U}&I6T(sBM(wBgz5*KB|Sz9;;>prHGv$l*3qD*zc#doUeNEr-u|!an}$(iIpzia%wMw&sP+NP7Pil0<>JNhrYFCQVB%xZieuFNK#B<+DQ1lV2*zNJiV7E$l8jWSMrB%yR7hirIAtlw3nZ8j(oyH4Vm0O|p+(B8vLL4+|EcnRV-)pBp{O^V>_p!s_}W<N<tTl(YSMQ-m({qX|284%+nzjmIl{*Vmej+d>WwC}BvV%#V)9QG{%9QKCkww1RM_A84W_kU=&2<V6LkrS2dh4)-q#b4oC7}A@aPQ%)V$EK*X){3A_(qHH@Ub|)!qJVRs;GP9Y?kfn|3)FdyQpZc;)cnJ$+Kx-acbPPkJsC)i|lKYuYb_`om8R2Xvg>x~u0MxIb{$uxs%7?AyPo_=;PP810&lk^QR!cioIMY*(Y#wHe-3*OC6cFF7Ve3uV6^E$93za%%F;J&vQMW2d32zCYRO$YsBToa}twMYkBf|AL^iBb7fnY3wXSBMkN`U`AL`=4qZ5sQ_t?X#pxIC<8!b0?E~6S?5A7C@YK!N$a9wOkjaH%1baVbRMToj$7!QbW}PgUJiaUsFE8nYe<tH=tIn5?ef<o(wl8VDuUkZb?pW;DTQsW#uOzxX8FCi4)-%lRqU&W{msoWnjBv3IdZk{?zr0bg;o3FJVR(SGDijFxJauyud%2A1_+cn$@3gxz#vEgPZ8pI!Q|d50}x58UX(R0D|tW)#<9Os9Q(T+Q9kY(<j~DP1Zj!Y*G!sYdn&Lc7+Qn@?z2rF#WU%SrI_+;W5QUpAtaH@kx+~!U1*a&8vFS$kid__b4Mn=#SmvB6`m_;RbmQr*_~Mh5RyLtqGiEV*#clCM->wY6w*=URhs2>oo9mLGOrOV%5rS=POsJ5wd&bAsNrQJb_X*YiOWmx9>Uzc-jN&+++-`BvCP=L%@(0#Ka!g2Kb`Qgkr3&r42){Xh7O6XFFBEJyQk+!*PQ&%Mu6u>_}y*y<7>|tX@XLBW0KtlfsfH4GcmD^IES;*)Z}fIaIe^_opHdYB~u15(I)ZrRRZ=^#Yz?T;uOsn@nq7V4>pj8z(Hg|!LDO^jgE;s&!WH&EWFpkqzAg7(QE#YuN}eNqK|W$4loVrcR$!#R9$`G6@rsUNxm1Us*h7s$K~*(85=1j--@O+k$!!FSytdI%_zzl5P~9B!VEFNDj+98n3p+CshlEnR#ABn7ciy#w9xy!;L5zyo|l5h&PyE{6w))!T{kb86~KKJZuzl^vfq2MaLeJZlwmgK92!SbGrYNi3fVL<S2|hn=9__yUf)Mi(7o<BPhF&HAkuV%D3raSrK^+oF+rmN{ZP5*D9_P5)^j}h-CcdyJ=<UFw_dk7jOTi$#x*UT!e&Cu1K)^3WvT~{7`{3*$??qLtWlx~IU>JZMzef$D?e&N<(!8oB57F`CFUd(6(VJt6S+hLMXgX7Rmun|6x2E|<XW((ax79_WH}m7isMP~Wt<cb2W!{~<IiE{YaMn*u2Cm!zTIed-!c`Cw`@Kk{PWQKv+FjxtcU|!II6Z572c*VTQuvNI2XR_0NEA^vepJ_tosfA&{)e>wnRR_A10z)j3Y5~G=1`r*ZoS#KmgocJ5%mkrS;Y5a9!+qp;7xeZOSqIA*-#~SRWqMYqs{D2~ykkzd=(tt~h;Uap(F8<I;G$uGAIc(@7H+u<oAmAQGZ5NR8+)`@D@~rcDXx$I*iY7=pfl-fQ`M?e<|O$+hy}8Ee}ehi^D!<yP~IuzlJ@l{cFA*mw81;a;HKINkZycS9P_6gaeBfBS&;IPikd3yz6#HNLCa-ae^8trMNp*gklYqlRmo;WkeDLh4pe?wTRvuq}&GXWCuuJGt<|1;>&c#_MLschGaNtpjZ`+~?Q$IJ%E9EDY`IHST9hK@9I^biIyahC*_zENQEICzEDJY(;9T{npM#)%BWXy=xER$<{DE@Rcc(aU<d~kTp-rcOzDc=@{&ZE7G~ypy6I~75ZPh&`(GN-)IH5tQ@?f=v%N%iG#O~hrb`jOL16o&t-PD+zl5zDfg0(H64U1#m25RS0{52H=$$g5?=5CL*#D9K@G`TyL@O5Uyn6(c*3X!%PvT$`sqQ;MSk1SI4l`*&XxH0s-r;Ij?stTnwQowlbBnMi49*aa28{uys+T~uB%jQuww-33d!{isBtA~S}Kq_XOxf{r$q*E4hoE5PARQw02rxWnoQT*^_=v4XI=lPqkql6(==CSc1wMg(oI*nCxxssc`h@OHAX5dR`mYK#C_V6X3;Qd)-L+9Pp-cIZhbcSN0-DhNGsV;l@pV_TgoK^Pjmo$?rpUsF*V%kw(hDYkn_^*G&!&N%l4xjp8bk>aoe0l%g$H5>9SGkFP}uYXRPX>Vvj>ALme}rPmKB5e4$bN=%z<KLFs)Vv2$2%?3Kue1S>l;C&y+vd+mxs@_a9Wq2c+F6aoZBN(NGXV>r+1jF+gSs7!fUl9JH^Fq{>PGtBZ7R!pg#QC!fnE?8C-0Ax7;6$jrTn)Q-=3D+(Sx0m4gVn5$0qSaZ`tMl*FbH{?S&RmgJ&_!<e+|~KH_b;g+%+-tX(gsx^FEFiQGgo9+CAM>{IlaW+Pk8mKYn5!_)!Hx<=@Fk4b;+f_qHorO#F|G;s`>g#Vr;Q?#ZCzG+Oog)i4G$P@k`gfGhI#9nV9D3Trzlkn+NKKFRH&Dr}A&OGtH$R=|$A4mni5vc-MhHd^fI3e`t-frDt0+OhMnfqx*6pP1Wpu!Y-^EF#Z@v&kbG)MCvmcWds-$h~_lSNluCsb1pEdIYfDpqZE}GWen6@X{rIrr8Ap@6pFeKw8#l1V?Nn2pX@8hC+j^h`Q<xnZ3Dm`N7`x!-hNcIkbR^suh<J)$L}~&(3PifM{c?);9JK$YU9q9#xX{M?YqJPOV!zEmkM$H_DsvIp9oL#`k5Agq7CeKWT7VpuRd+KA$9)3qyai2sSPs4i4R{iQQ>!!jviANM%0Bvzmky+@*q%yy2=PpyoH!Bxpd)WDl#H!$_RzHVs%~>Eag>!1gMdy0Yg<mvzk*t(92qYH(s#36blx3%mvHS%JVv)SQa4kPrvwyJobZgSO4e6%;mGyfBolQ{`k&0dFpGgy`}?4HsU0ICQlYV3oDZ;6Z<j8H4>@KL#O=2HRaSg2hmad>eU(B9%=eid%_B6?c)zL{wYE^uiugld~)MIyQi(>mp_cp{^bwtwCfe9Lq=E0zYSho9dD-DP|Rluiyynm(s7KN1cy3k(#O&kZ5~ISp?l{OmpIxpJuf$&Ph!Q>)o2O$*sLzs+U`axY4pe?if$$@a(Bl@2#pVLAx4Jz^o=XTacj&60W5Z*yCD(1vTMk_$Pw9cnV*9gB){YJWh1KasB41T;ayeFPw!>2woC&enmCi2j858g+1J%3;NZe=x!A_KJap1>NJKVTYANHswl8?Yt;uZ`zuC=43m1gexj|&<>iRaTdD9eSO*XlZ8tJ-k?T2*T;pXFdTAh#m$c2m*wC}w9u&#bk&wVg2r$lLeKKyXfZBgcMc7E>d4?q0gSf3B`n#|lUr!BLpcb<!!Xw>aaBwjG&7Akh|%vilN1NhY{PBjnjCYFIBRRkHyOImQ8m1$K0E?||GwSZ81IGj<r2{5XTst`aWq@*U4m6V{YCKaqtaT06p+9kP8F3*{|{n>_-66&a$HCiFZ6_Qsvx8;1AUtg22YUb+usw7J;!zblZTdwsd6aB)km=u>@X`Fz!VIM%{rtC8ePfo@4Ltco`FJ@QcY|*MI-7zsv8^CPU@dW-EQ=YW`&0p8o&YFL`2I-2Q1V7Pc`mWUvUtjWSmTef@hMRu(9du*YO|d&IDBFeyQFPS(eCp@G6~|0!br*u3s6ja#$(YK4FGp}a=KdVQ{mC@<C&tgPihNm9T`0|@?JRS_Fv=0ZX-&!^lV%pBbt<Zy<5U;HBEkwmQB|`X0{}707^^AaW7Wg4>ftM?dbs}0SzY?N-jrH7XUzD9+oGX<euFjNpCuk&=#{G+<iNrk`z={utK-r|`EuUC&kSh7BG;zeyz?i&ELrl}JuPvoMmg|TM&+i7hj25|BYp9Z)LV9|FAvag$czXUch@m7sMX!?^)zH=s{di0Z_5RCca3DR0c%V;;_AS7ZK`>`G6l$fmg|MmRkOs{bPGDYGFGb>6@vDPLpLp(gM;kF%0Y(v=8;FE^z<Us++;bb)LwN8J2ptX)FqS@n8!V_>#9CsE6SV3LS*hhFG0N<={!A(-{w6}v+mlpVf=swY~DgylTi0$J8Mht``Gk~mup}0LUT)rTYS3;Y6mlQ&>hFo>m<+nNn?{Gom1WUgQQEcYS%9C*ZMyQLfJp?LuOY$)^>~Z+<Ht~CYlh#{|z1bw)5B?`&Roax(3-G%BGVZ-;TejM{cohpXj04zvGwV``#_v^)gEaL4Dhw^hIm<f94^fu7DwcVl5aqXn~tCa<ds2@!~n=T}9{K(Eef@3G1K7<XyH}7qw>Pb3Gb{c9=#BhX_j;dmGXB<{w#IJuY^)_vlAx+-2Y25B1?I9z8oI8hnvhRyEJHcm@F+S)M_ZD(<O@!aS{NSVF=xMhaeHfU^=XQCA|XD#p_cBQ9{kAWZ8bP2~YB%{V{BVF2%PxhK66t@mran4h{jKc!l8?!EcT@0{b;u3ntK>;nK%Oh9>eI+Nk|H9`r+`<<@s2}A-+GvCr9&RdzTFo1W$?njjDZWV2<o6=XBOybG}j}P6{6XA}xko_3Fy4h4q*^V!vJj$L~#P2xs>(u}QrF>Yox5(w4p5`<1$d4EBhcDoBBi@m+QCh(xrS=3+PIyJr9OM8OnaCKe<eI)9Iib=Va>S`9aSbrfU<L`yat1Kw1z}@c)EF1_O6qL)eLAn@xqdHVcq8S3s@|Ua)%^SExu4Gc{O3RWw=R@FNJ%=nMN)`BifI8k!>AI1pqgeWFY^i(X-2r5RI>`@d6f!AXo@H%7*kReFhdoGxE`}VjM*PvIZC7>0deML{<3dyICF>MaC$MgM%Qv32E}nlE7;CdOdRwS+U^LD)AQhv56lfM2<N{^i=O<8?16VK&s^2un{cE8-&h`Vg|+#mk+15h&F|Lp(&kQ_O@3~;$K_7>s%EaLCAliRQ?^$QhF|{htPgq`sKd80b47!lp%=kdqUttZ@n&ZGg#q#k7qaa>f4i}eIRdK36Fv^Ppk>R5H4_6};A%6T=$0eThUe+Hzh3}Zv{XRnMBQwIlgon%RUiBg{OGYf<Aw=TJq@EX8K6`<gK^@|+AnOd<~+v5EEuIl(}O`6%&$8;7Jr_+amm+AZbj`_@lS7uSF_)=8x*Y<j}JI%Jl$l+i!zT8_Iw;MW88Tj(9u0-8<t&F=`AaKVv-d)D*kNPBKP%1;3D@Ce~;jzQ%4g<Ak6Uy`-z0sHN&YWE0&AAP`dAe0{LQ@sB%$e(i_YOsVIab2b^X=;Ht(<<h;PR9Iv*=tL;~EwcSXH4@r5iuN=v&%5S;6C_lFGs{5&{?i8n4Hp0e~A<XLx$yGOpSt&q)GEwA!z&fosDhbUqkzvNDG^#}<Fh!s)t0L!BT_AjFBr?a#?pKX;F%kl3E-3oR`Q(tj!Mhz^o4I8(TVe6xAR`+V4`bF8>nll9kU%YbR*btJ=+U+;yWtJDs_FSM62EY7tZFx7rZD@LVQJY*;>=rvsq9m;wlT0<`-s!c7DwF6`alN^dEnQQvR&uRME-k+M_ISmY(IYcV*Qx5pz-|iQpeo#esg8ksBLX?lo?vVKpnU4`eHLV<6EtvG69!jR=OMh6?u~W11sye=epbT>~x*x3sKhHFuJ1`@e3ONwf;NJ!eBr*s(tr-*LyO#1zSaA-`?OrePeK+t{z@Bx0Zh&RzrL}g2vd(OgA<+22Mwp+x{K?S|6;2-IMW=AM1e<pV$#lnjAR9zC=0PHzyCf@$iS%c)~+JL)GXPT!DNN>)LU>aa3<SbfAo|>sFQw0+i;2fdb^LL>WK|kyF96P7yB{rlm+hnF>(nWeQ75r0-UzX`N*>8{@{uxbacLjho2wVjxOY^Y@;^8b>H_4ONcHXnST!{Wz)$>ifg??fx@va+J5ZKQY~4!%G~!>)q}v#k_gZfoCFb!+I_H!gn3i`D5=r+#a~<h4?^ExTg<2)5R6LuP^<r4#?j+<hd`HqrP?-efRLoQ8;<kS38-&`CxbSJY>4_262Z(hn{hGg3XD^B&0RM+ce7bt~S_{j1OK)xmjm)ulnA(V<YH+8oCKycIl4^OXa*|J;(#cEbWC0cg?vWnm*Ae-yVEigHLluoC&_QH3UJ<%QQzfVa;pauf@e>dvV1<-vP0$XFYi9wMY@NZ^@?Zm0K6SZi(cLirV(Xu*=<YuJK0lMmTNCxb<p7HyOP2hyT#X_~@VJYuIQ2qbDtPyxA;u1^3rpoXoN0lHm)F{b<`Boze7E@BRXBHhN>l58drfYoJ=%v-!U>jT!}g@rh}l$+1`Z+5;5Z7c+-_?H3+FB=~=g_PUf`xbRxD%uuzc+cJFW7JoO5NT{_nA!#}><f3<nT+|(G_sjoIdmR6+ulP8`3SMX*-Zq8pDv*iR4b}rW_PSX`J@p3m=;=5QYv0YPPvmx5r=@U&|6DD}_k>-oulQ@-b?<S%qub1p(IIDa@j&AnkC*Zf9m?N*0Nn%eVYr$hN%$fxCOa;<Yp&Wv-#YzR&lm46y=x7mu?1W1sfU=P^+QabxulO!FHD;n4>R0t|9w8}WPjl27C(H(nY>5(@$DnpNpt=8YQG0ktXN^fzO@Qbirm)@Z@ZD&(HVDjn4^8Rwt34@;nAv*`ZngLLC|615?`A7E&n>ZFZ*QSkHNUAi{DX~eRC+O{zjznXea5(L5JXxi9AetXXcR$Md+Du)v98uD?A9@+i2KC<FeoSVxn+<RMFz8EB0M9?N4@_F&?2FX3`IhVV_x`-q@W?jMw7v*c0LIt|rM$*IN#)VA|bt>L#(1u<XrYD1y@SemG+-ibnOVAECk|6LBTi$DXDBju&ugdyU>jv2|;ckqI-~w8z$x1NhXH`-Jmv#x^)ED+-AAFz-C}1#FA1w#*Kom6&|xUMQB+x4o`61!LZAOwDlUN4N?PF$ge=__9MBH0?rl`Da^kAy+>t^$eAA%KO-g>#3G}*z~r(q8{W;d4cw*w4LTJaog9U?Y96U9h<(6br~8zg2<=ev+RboQ{fqxZM}?TTR0!xvMq`r$|=lqMnzt52mrts<Q2+cj%tykyrdwbDFX$D8HaTZK$#(u)k1ErUl=>xQ|Pp3s+svqFNT{=-&%z~Tqxx@^mHm1bVWyl<k^b+U&S_O6{DO&K2Eo&<%;?u|ILh&-qt;S#fO0Gy$orcg;geoqD6iX-w9Q=e(bkwL!X_0ZX=phJeKUozR28?WIu8|W_p!;41oZlzM}7plj{4Yu;zu_H8c0@bW|`y*$7ipQ;rH)mw+q&X;?yv>x`BeLx2dLVTd><so;<c0tq4*5x^_P3tWkUpfQQcI2is)2E$w2)LRkBf{pnOVWn;y=`zZS<gg>1ymsHERj<1!iave)4_<K5u137@-O6;_nx16^G2GRDbDVF!uh+Tj77xD`1@Im_{_3N0R5$MYdIYQ3(!tL&!;o`hlIqizH17CZMtA{G<cJB3^1Q655|XZ1mFGFdf>tHZX-RWTQ^EntKq)YRDZmWCGAnTgsd#C48{Ttu=Wi7=m#<!wK9U@CsyW{!b;H2?*3cu<4KXTY?<=FwWO9`D{K~u{uYRYl&L;mbugRgWT$2vn%++gGYiXFdJzJ4B&dU=o+jA7PqueAXszB{fJE1bwRVOgTm`>EnV?!w&AG!$3Jy#SIGD=(S)3KU1+fzD_VyyZ@qhH*Bz^T;j<G-#;<G(r4Rvt5KQl1*eR{!)}=*DOEEFV%M`?KTX(Z2NPRnisv{%IS!DvYW8zZ8#<ju^@i!_vvmDoy|tmB`B+)T~gninJ^W1X+%=9M-kSQfUZl$g={Gnx{ERGf<4J-s!b^jk?WYnusAF?b8M8MsnJ7))^o5PUP%v4Jxfa)`xFJhif9ivF&<(${D07XL6d2lwD&5hM>q06@pNfp^9)w6cY_ErB_!A3Q5W<n&nK?InMxRS&@q>RrU`RFKamW1&EC_f)v(vU6yO*e^-n7ScBv}wZ>L!yxF4!b!~iBu28PYukW1sgk2QEnnRO*u(y8i;=Ek3=$k0}m%Pg_U!9RHo>OrXU9fcmcF#Q7Th^f1`zwu;?jnj-c%kRk{+^9%nCO)|(#96T@x#IkF_vDJkjI=~4ugIx#&hpRzUQhHp=!2H=NEfELEDe7I2HZaak>A#C-7WU7c}T9#lbl-;erqm0?c3wjF9ct>*|CXK1&)<2a0uUL}LlBhREad1_?R_{Rqd=<-Af?d6vRbI;RA1N=Z@i5@r}vj6s<TSTj}@4A-f8Q*s|>MF}ztOH}f)h}Brc>ZKB~(zH4M-_5gs`T8$^ymTmgj{LAcn$i=+)}g#PiY&*%kEhhJ$ED5D>QZ;i<n@7^_YM`ViVyA-p=u~*G!&`sy0yw(1N>`{$#{{DrKhpi#5mqfGez3o^7>46)w1bOabH+S&)QO({bnnIyDmR7hfVw59fDLh3?Rrhb}i_U7(mr)m*#(?X0q2BnFQNWn-T1M(IXr;b<FF_7;$ydqEogBlZmZXv`*53-DX~BM2dgC<Lx}3`c&gufoo>RePi2`t>f-n%J^7brT>g<G4`EAWHFcKiy5_HtV8weI#f7^5RH})RmLht%Cu&rE^1r=Mp8s*&I(dV--MMw6g-7R##ol+RbEt(5LDMi$p|E}E5|;{*he`zALV$)_FRW?Zd&80s%P2pTyOZ<@cQAq+ni+0^M>q;^YC{~@G@SIZ^k=b1ZMTsSZ6*57}bq2e~bg967ot|ZuwDNlNT_^m%2Ldo=2HOdH*>#ucb`eZ&qSZ(G>Br1|kP3%BoD@BF{xh%M7MiR2-&N$)H?XW_3*=6eT2>RR~l(9oz4*{eG$Kw-Y|MYn^3Z5YyP?SWiNIK;Iu2e)klOh$4H87CDZ&V>jM9Hq76rVa9M24KpVYRawbO#)W{=UB(bfml_B}DM?d?QfZN;eFhw*gkVmEs8YqoNGXMgoMLI0&IEaoYf+fdt0q-$hOd!w^{`mIe=<^f)#y8;;c7nT<m&tH)@PG{ba`-NePu(90liXQtECy&IoP?kHCwV~*H^c7S3QCBk#46+o6ldiU*gaNx&``d{dN5tQ<ddwz<iBltj%IF;_9SF#yUQ-wTadT?I+FH(Cvnhi;vUXa}k;aU#fM_C1MtB8L7*@Z$--HMgeW#6C$~4>^PwFYAC=2OFA<r$7MNt?TT7He=nr8&yb$abb3C`MhbSs3nY|yD*!1$qz0&{5h?4OK}G<`N>Qdnh_ooG0)mnjfMuB|GL$k*D*(pj2;*{uSFs%7!iA1>jmg#o3d6{|zF-qPn}Iyf)is+;&yQN-mxd!w6#`EsDjKhTZM$NDn~5uLU-UKPak9h{G#J*2WlJ!pi=uU;KaOOtf8>r+*@H1+|NG=G$0Q@83&bzVJZD0XDn&)c5n!mKA_bX{<{tp{>>w>@!E25)%IY)|5a$5WEXS;%`52NqB9K&gd`RjIr(<g=z_%L2K@Fh&dE-m3nMm%k^|-BgbKk@8rM-MK0K<0*AL?eqcv!pPunS3$pmr{`W1vwccE#(bc*2*E9lqh(X`b<ZvovSfKkwJ3u)syXzxG$#b7|xyPnCPSrJ6DpaXfl80G1{v%|}V7u>xfQQ^Z(WSBL{DZEnqL$RVsTRZ3$;jyd3jWiT%?R+bR*9MCGSYgQG-SUO@X9q}qkN1Oy>=Hr-RJCGPTY)2=LVw^=(C?!n^-B5vV{CMpD33t*wTze7yj{jSOf|q?AM*GK&sLc1<hzmWQFb}!qax{tgzby{g5<uQ^e>Z3uHQx9_<7f2uf#Fb7xS3mq-*4denRtU<tgJ0H@LogP99_rGz5eIHbYR!=FssdmV4B5B-H1{j2R?c6Q3OfqPaM_@E6rY5iF#qB(F^S>B#y>z+YPKEzP#o@ybxb!KfdOWo344^VP7vCmtE7>0)$)N`p}oZ?4QsMpYXu`?_n)??4e}_;upVHFPuH0ulqPcp?Pw-?v_PCPNys9J`B4y)>pg{5xs_8Ls{om-D_D6WLoy;fdz~?5<ShQHJP{<@m2IvzOsAz5t09X^QYc_+p&7@M}O;zQuoVJQQAEdf!)<V)N-3r?sLmyr#Ab)4-6@zWw^T;@0l=wJR4V7b&NFr=Ye5h^c&RKjs2EDFT^VR-?}|t#v`x!^Wdl+>zF0%dt?bqN0(E+CL~WwUO|k2U=@Z{Mhlddc`gt@2o)j~D9;Pb>Wos53FS1yf&xJ59G>P3;M}!K(v>XF8S<^!hSwK@2F-v(NDzk1pH2)qfA!bbxcmyPzOUjB(o>z3-l%kkClmelubBLx{sd3JjkFJ-5_|R;4$bUsnef4pGJ6eHF9&<DNbN^@YT0stwk<n>1<4f?FmR}KW5>6B?yql5>qn8nzhkYdiw7@hxOm{veWj9lFL`aPN!~&7G>Gx56(*<&1)_vVNCSu!DI%rXCJLAVk{3k=8G|LkyebOHD*#vqD^c(&D_}aL2x=UmJrxn!M!C!XXo0zISUTM(eEVl7S99R4S}dA%Cx|%2a$d)XWsaEvo@TumP1#6UFGg`K-LpC^Dp=tP0#>k6po-&?R;n}tr<`SkbEa6HvMehEE`XpU7ism<mPd@4UcYmuSNN<<uW_7DRcHQKSBo9^0E?l@w=UhiSbfe%4V!u)4_1>m-f*FvZ(Y!5Ct`rV|JU_}^Q}Zw(upK^^G%I0*oes9H=Ck^>v74UyT%uW^h)02x-GW5)6}DR>^(})Z@&MM)T+Hi)%Qn9_%n*gt5u?AK8?ZBG(l@x7N}$yhY*3frUD|lQkMOm=cK{~<D_CV%cRf9D=rGmdCDP?=OA2;bNi>*b6ob6DQ3x=SOEj&2JR+7+DJ!R2Xp;Blk4BLf*dy8-vZEvq<?Q9>Ax0{=N%Yt^M+2_H)J#9u~UX}Pn+YLL9kO<*VUw1>TFd~TD8&<nV$v>1P(uA+P#sz=_2rphh1U3mOmcHA0hyaR90W)fD>3|l;<Rad0iC<6j1gqs*AcVMMg==c~#~_Fairo$~uD-(7IswWeQr3jqdkubbBtr*GMcnmcF1Rw>3aydOp-DZ0U8owXM=jtxuSIqD38fyY`9mIzQ9-9Zz%GZ=2$WL4?T>pzgbSFSr=PG1*PW)=yhM!VqO>w5*?ywlBq;<}}SvUDhH)1yeO!tWqR^q?Z4KykIHFQ~83VW#rN)tx8c60LRwv^fH|8IikEJE3W5~_lEhVH)!#`QD)9TZ0<!V?WM_>*?ACBdKh5zB+T15%sD*FiFKHh!jWP?#UNv#pdbaMKwK`jD^$wWG!g_OD9W5O3|Y=p3093bE(8J~<teT*iUrEDF^j;MMc}1ks(a_Z`ZH&LHGlRJpF5PN&Mnf{7p9C4Ipti{`-k$@4LfhjHD!>*j$gex(?MPr!UP}qVuYu&*oCu<1dIJ;YjpRTtah&_zx<)U_Ah^E!HZt0)9#pP5Z=YL13O9QEzB$U;s~^NJCR)UwniTK{<9vtqpx+)Q`@=jA>1o*Z+y|9WY@gTY#Wd8Y+QJU-ME7nZ`n#%YivY=t<N0agx}qWlb4&0NZ0ChF1on)X5x?=0b>yt-E|P?JtIM0<Bdjm6URcg7uBllHmLIAu`&E8L#}jppfh{VL#wwLWf_LFXviU-81~>+TP9oAZ1lo2Qt!Q|Q#=v#UNxh!Y8Um^G5y|i(C=lKkddO_!!U&?BLwH9$aq<)m1m7>#u=!2&0v~AB`hRzp!~87g0ujH7NDqM4$}&a7o4Ye!MW=s*B`YaZa&kYb<cw)#(qG2x6&QQ&2=>5agC+vk~8GnF=@zhR1f2!h2ur(i7iTrK@Lj$I7-AHI0dlcjHQTG6i|{gT2oPz6mTTp03;xj3sK0^oFZI;R2s>wV0B6uCz*H}x80cdV_ahRauR>|^`<fBEvnI@(PtROsw?UPyX0zC><vy|(TeKvxnIq{pPu{a+|Pghvw!Q7bWTYMN0uf7r)kAVRx*&$EEfzzQQ$ITCBxEq<`qa0A|<9+x|KOCYE-6BKukGPE+@icH}h1}WNcee1WW$bYcWOK7|3oU5ntW<M2C;)%2#XvWyx{#TYs_FtgW@BvjN&-eA^zMdpkKCKjWxEoeI*HUdZpdy4bC8!uObxmMlkjgc80u6#|nglNJy&%#{^PYr?9kV6@CBAVryHC9gT4NTey}(!Hz;K}cTloK)r5FrHq+*tY_eHmba3;>2rStVWr{U0;i0qQ>WwzkT)RzkPN8Z(rTf|CN7#`M0k=ms@ZD?W;fj?W-H|zq-OXu3OnP=FH=@lOx<)yN!#H+i1cA78pJ$+h*$yzfp{$Q{G_9|2{lC-|VgGo`rZ@NB_WczS*WN`-;N@6J}N9%YSxxn@xY+PplNmdIS{OHWBuoWxvq3nDRA>-Xo6F7|UZ@ev%TO2(eMZHDW|kC9YQGby<oG^BR+^;*^&amL?sfCCm{JBCo45O({+(6(o~y5pbH5aV%jROL#?N35U=XeWJ5(%^cZ=bFNn{g^gt^@#ehmW+gxQSf9N?aCF!!Gc(w3a|XMwZTQDgBOGtEDOC?yS8x0un4>0jD%a}7+3B(IWQ~=>ZdA3@NILI2DAmWVH0p6sYiVU8eOoqd++1hQmrcc$$rx_AnQO_BZQD9}lWz`(`$MOrIa{-(B+h;KNT*qsyoSN8IN7bQ)i^IcZQ3lR`aW5{IiyAAFn`pvNwxQlU$Nm9R$GTR<F0#^;RM?N<9+m8a@%@DLoXE0@Fz_B)!S`zr(PuB&_Dcg_PoK<y88U4?!rgM>Tq>}b@zHcm2Aj*Jf^^Qv-*T4hB5;6tV(W3g|8X%+{wY4nx9_0Z*$U15w68=VA6j;`yr>J&b&A236VD3K-jj|d1Uns18vl!zy{=%5G@kJCehd8*EJbK5!>Bv;Q{O2_x5jV$3_KI{i6AZ9UGGSV(loOOosMZY=}M&cfT2)sojl3+1DDfByshmdAVPC2uEx;RO8<qvktvvtU~~eZm1U#Ay*V>DIHymAYg))Fcp-g9HwcS(KHuI>yH<Cj^tK>3&aV_D-OyW=3~B%Q;aBBiHzE=qnwPu0|r~*e=oc8!C}$g&8}d=4ev(8b&v)m@R9bcV&wXAo7cDlxoD<?m><OYjn^an*$3XM?XIa@vq6?XtH0}AfV<Hd_mWsOD(;?#vDAq>e7BoPc`b^`9K_ha4L8H~{N((_n4{#lbxAbOXg*3v2vp)ML#V7sLDL$DvaYHcmZZX<O!G=qgydOGAWB76r3GSjiPAjJDpr(SjA=;5G$hBPA#t>teTQ_`&is+=k935}i*isu1L)w@VBt6dngF2`<Rd}CfibQHO}W5OAdW}@08J_8MDQ%9999+2MNTS~%YQ3@X@xk03{t@n6d4=4lc&|4yn6B4rQ!nwz`Otcf!uZR>IY5N>yG<drN)vF9gM>inz-YR3zOXQ4w+w|uR66lw@i1rGanyt3wxtnq1{Vk&^8PdR@a>e>5k`YVYuKoe_h|0a!EGnqHd_Ge-WhvdmgFVi|(T?_aKg>c07H|o75k3cAQw^i<-n210&@+;z~M&fM;0>gf!fRDATGgV3vszl(qbc@)QeL=R{g=Uh<UH(t8#{Zp{R##s>WKA{^GD<!sMWz!@O|8zo4L(1NzDcPu&l`pr&8rLAuDq{+CtVB_%W0u!qZTpym>C>abVTz7>L245D;<zyRJVB{!V7<$7Va;LdOR~I(XY4^Z3K_;#7?;}3uzE5%)W8HT#3VqQylwqIxv5|D$byX8VY2d*0Z%lUbs1;ghO@xmvU%utuGsPe_z_8Sse#W!L@t|*jPy$8?5rL};&>R*StqM^TI%-k_3~-g>s(=t?MGeX-BXwTXG|eGT1rkc|i6B~m7Zy_h<Cww=jwyf>qc%16bJ4>G8Z=IuA)6}A{t2^~It}ZOTelv#uWm|1*~iaM*8DF%(<Nf4vr7jr9-$vh3HWQS{p<0JVVG7AUP~G@T$b+N&lD=UzTBlKHBf7^6JkHd*X~w!SSZsqLd>jOdT%N+^33F{S^8?kg`RCnAuum!Rpva!v;e56Xr1O&iV!VIE;m&Kp;FXx@d1RKQc_S+Qb9Szuv}?)kJlQfcdZf8_d6miqeik6@FB*OwsVaSrHh;xVK$d#>nL4*XmgQ`#f72K@SvC=Jh$Z7J^E2WbR+V#AMtzi;vMUL1!ATu&r!2sz4Wo)eUfS^X$I*idF6zlnhOXxs8XB@!eCxvT1k@%smLoP3i-sm<{793s#7GbD#<}EP)-@ii!l?|n2GBpVB-3*Jgu0U`lny~L?0dKes&_n++$0JB@gV}El>TC_W73`iIxfJuS)k_ZMLw<H{U!FVZX6<fYJ^i7`zhxsT}IpRMt{s>Fese8_CK@$@|Ya`k|B=t12KXG2mH|iAp-8ET<U(jDxH$7|JlH5R;B6m6kWBii4-5fKgs>Do|RFgW#u|W@q2Piw&8y@eUu_>gTpI`sgWCI*n-e`q9C08(!s98^U|O_}Z8G?M6zneQ$5?#o7R#$t=dHo7n!AflPaiViQ5xj2OL_JH!Qy&;6ONxQKOUHjYt}H^!_%3Mo#(C?y^#%2-)~tP&{%Wsa*_)m2lZ;sdm-NKH~v)pbq;R|FJ!jS*l37dc=sE5^ZqQydKVR2x%GETLE70aR;<DQ`L>JBU2C(k(Y4g{hU8P5be&*tDGyRhRj5Ba^r7xJTP|ajY)zJ+Yku>BNoF%3rSNNsaPSI-MZR>l~nhlr@k>5#>NyQ=S4*qB1K5MqHtOYf^GRF=T}xLxfVt7~e6*_b`mFCsRJ(;8ynGP7@7Xv6Rv61^Pr{>WX*VuEx2Wjx0jUA4K?wh4?6AiY+=LE+gKwcKE?+^7pgz@`b4X$_FxGl=o)ls`>ZiKeMw*IjiPC`GKjIcWqX7zA4MTdhPOD^8Wm_q~PT{$<=u?vC3o1i}Ih+)@zLuS#9I2YwfD~%Ejcr&&w}TM@W*N$Q#M|Kt53N555aJO!Q&HXfzPf8O=$?5l!VB4YD*BP}Di(1qMZp2!=G{HRNf90TCq-jOBSPU{$0b165w01||tbpfF|W6{g_fHXeRQOcL56&Q-=3aL?-C?Bp{NM<l+~(@;ey=7^V*DFU{cBGmK06<4aZCj{@6$uHUP<KJ_F+D>EE9a7aoc8I!8@`gY1$-?jTABNM-{8aw?n`-*F9w{QMOxP%iFDciw0qaZCK6<j}KzoZ}TWE)x6n=Zg@jmRD)q<aI1~d~UEJeiY@ijhdKcNmSPluL;COp^elmgI)FM0|)2}1diswjWnNv`ALc!sseD^gOP)ft6yenTa%)qfb^X`SaesdM>nmSf0SNf;BQs6<+E2oQw;U_@wLi}JK^C;k8Iy<2k~$9*sQRo00gL_~np_vsb_@A8TFNu?@DIcNJk*(%labdSYtirnEPvW}%HP%08qXho^WAQg%Ztq=s3AP`)I2MBxto|D=i!lh5L`#wFh1|YzTL@ta)lfX<*Pft(JzyH_Y?+4!S2I!;{Zir6-owVLCpEtNBTU`{SJ6Q_(en=M&Ss)HYrggqKmG+kP7&(Y%yt6AE+xG?S9S2rOEd0!83dYj`8R?qS&v)jI>x;@Zjsb(@Sz?e7GQ0Z)m$cLctc@K+q%oERrdkSElM=`pD*(BwD~4$$skW(r&VQva$BY71kpXLOz}lNNti7~h9C+5R?pAT+`Gj}1or*sfI;eQ7E*3HNd(8EP%&tm+-D~r$b_;2n`MAG%E8~0AR8YD5eUsNJrI9`gL7Jsj^U!*I-@~m%3|PN&-95UO%^Wh;wO&3HJT1TDjx$47L$D@4<B`$gVIQ2o;chw(C0@5(MOW7Dc0|kjHtxL58QV1L!5`j>;r}(~99;{C(@uONd$sP2d6BydA>ifVa(5drnxJ7UH;1>fGO%xB3H0(aNrM`8`DcnW_Cz85mt~u;NsUX1MTw1FDDC!62^JM7u-VPax~c?Ii~~`kvZ4a33K1fytV#sru+0zK{L4zu<49fYHtmscCt0X}lA~>|#TKib9e$D#qT1^QN}uOI=|cdde0HU;kXUn25=^nMQWa28fD$QKNv?6t%TfT0U{xtq*rI|8m|qGh2xL@|fzsy`r?jr6T&lecAZOX@{CC_>TRt~xH$e{PMXN5!%Kh1DAs6_d$wv)vL{#k?Xzbh05=b=z2RW~WZ+k1{yxwX66&=R*kV^VYm>me^#VqBSmZXL?#0Ar=kVX(n0jrwoTEL<blrSbN&l#zJE`%%!Oaw$63Jo<z1uch)si9)(8CFc4jMx129v<{A)VPl@-|}a!b$)M@_fd>tCw_zdn9c8T0wNsDepFp87{CwVFnBo%rk=tR5En^m6TATk@>wjdDvN?3Kq0L`Q5dsd7>TPCrHYqW!Wuy&2*ZL>1Q;e%RRAlfjrs~O6Gb%$;6VVt&;t0T(?i$21{|EO{&4c4tR*;=?$JTOevbl{qnVVibtyp&QAJ7$kojXw`?3TCA<bo>H3b?`40WkWXpEEEl37ZEEuxAf91oWM>9zF7v0n8~!Ys@YZto`ggOyn1KMvS}J&X?PUd4QH1jNdNp6aS5^yY6DK&8g1iZyoWPd`X32>BQ%kEyLX6kBoJcqm35<|-g)CGIPC^53%R+1++p&LqcjJV>Cw53;Lw(imepXw=--W5o8Mozc(S^jqZNxq%nr`>(Mj1PMfE$@d{yfrY|K0mur@WFfUMFGwp4079~?C}pJ(ij-PIqjoDI3sw`(Sf$Hi*cwhXu7OWli2K5ICy=&-u6v01|0W8nPh$`(6O$dm+sFI<VHBr#l33o3QvG0xW74NQ916L$5}DlhqgWkBEW_niQSR+XlH6_5rGxK|*r%kdM#T5!9=gCDAwG^p!wEAi>>Rf<M$5x+7Mn|ahDt_#<O%S`9&O=P#(})=x0@7Oyy0LWV<yVD1sC6n`1W1ZrOWJjMSFphGMYW(4%Vc9s5|Lp4%^fTp&$ekXqL9AN+~S>GA0RD$Rg5P7zn^?RZ&&Rik1aIbzSHRRkg$wsB29tVBJcp*7oJSAluZh^m!v>&&Bz(t^c8Xu6xw^EAMEt2wwLU!{=IVo@W1O?@iX5U^WbG^C<i}n0J-O$t&~^SB%AZ_VRmnD^mL6yxEJ+UcNLxa&Q0Ayi37wNdV8lUlShStqT|53eRxpmK635BWS-#?v+Utuig^K^Wj!9Y;@097izF>coB9tCdRGg=wm^j{+B3QAGNuDA9)Fw1KEr_#2Z@yKsTcXP2zk#W3%&;ZxDM=9k%2?Y)NJb?7_%6NmZexs1USru%NQSB`9njd{I|KK|qz|d8-*?07<C}ZWMT6fgiyD_Ar1woD}TgT1ruiaxo-%`&8)%S(166!T2z0;`c@kk98DzAsOU*$y|F`c;SBicB#(F*QN7oFUay&%E+%?6%Q7(frX$Hu@yeGw2E4ZjnxYPU^Uj<l28I{R6bQ$sYW1{6`-Y%7_+(*2y^q(slh_{p>u#wI~W2uF&M(1HpZ_dW}XC=c;Je8_LA?rd-hN3*GI3KzyHY<uYB%cb;hC4#vOM&#jYH5DW0;@Hym8_{&`yf<1YOt&jaMzZb#%Zdt}jH933T2?~FG{2M^*svSzV^Ske>HlA9u$TUkcnjl8o!U$h<Je6o(dTfP(Z=R3&@^3eZt-8%x(=p&YUR@k(b6~s)M8E}qxKbD*3F5fNx;k)I}gUiEe;CbooB!)Wfr|~?D*p(mUm9^2&vHp>NoIEYD{k(qcU&KW26+eWL8^d<w;@ETZw(k8K{=T1kr&<s4k$<#ppV|0PYIfbbR+6L0RTBlEiLDg&2+;8-orSbiA4F#BHUI3_(z=bXqtTV)E3ghmQ@D3Gn#9g#ARi~IDyDDRhmO@xaTwR~fKjSM_FlAmdC$S=wG+c39`$tkm#ySVTMK1z?me!hn3kEGg`9nB6kwzSjX?8r{ymS=dv?#Z6)eOM1zL{lcE_)n)yxH#sK|yYtRK1x%Zc_*u{ECEIsjE09SoMVq?I%VBcz~`m@r70sxj78Wlc)NN(M2Yl}&l2NZ_*KlnbTtkbN~|U%k-TS02fAyTh{5a?tr!TX83Pm;x355=#xfOoy&JAs%D$4T0GD_K>M{1SsJ1$cAOJn5<RvT36fxUXVh;T4?|gt3XxJq9(ZDxFmH!3&<)W7*eI?b-_fbD<zSro+p3HJ6fH+^3EWTPn$qq3k6*bQ|x}CuJs`QF68fY|J|tN+E#;a?<D1>s1N|YloBnDzzc}CtiuIcSw?}-8VblOpe0mX5l&TIR5s;Q)6%kr6Nw?wiUXlgQDG*qtT{7UrofO746y@4?6YBrol+sU1IUZ_yF5c)*Z~mwiJ46So?T{BiV#N-3S1*v7D7-+5oDl%Kykse#&s#AfQ3+yP=UCr2$38Tq<AGt#bz$h5d-TB;cYG0MLmQ$PEDBOnu|89xCp~%E>;m*`{SA2?IRs2b*6R71A^s&=BC%y@sL9pT-{d}Qys%NyXBe#4O?oR3mDGplf_ZU+z*Xge)CZvDBNk^$L%18p^fcj<Fm1j`2?5np!1aTDGJl=e13YYQ|C8Z$Ybq!=-#t^W-q3y_II0r)ux|0rgM5|vY6W1(@WnZtj7bpF?Z_l&hNxBha+1hm0+XiDf2kO)X7Oq-R&EEC%o|lO<JS;BFnJ7cSPGCC_wOYBVdO3K!1Dy)0vGAP*idb5Qn-TxTIBCR8>*af`eKkTor=wQVJ#oSF%E~;zCuGBBH7YMTFG@O51?acG|#LTNtHTZm;!?Z5|)5w0q@Z3Sx&U5pSxFkM40jWD&&Hi)|>@GpR^q&KD(KZ6evn17~fnS!_X<`n~m$y~BFA!-7k!KD5^y_SBx5!h4-$uy~WaX7cEXY+bh4g`&;tgJP{qR<!xp=kLFM^y35&yV*Gvl5e)`%%t~nHxzZA;1S1Bxo$sPYsSWL_4M}Ao+jHiTVn|uIA!u&?e8WQ?F5gB3&Jh+d?t%I6z)g%=@aYsK{El@vRu)}yoJ=+B)`Vq=F61cu^qEt8pN`3w#LS;qi>_q^=Wf~>$fvxH@cMGbljnZ3l{TwtOj=Zt5~etknZ|=sO@%1RCjh$S6Ls&K{oc!z1f8|ZY`q*nmBOxb~Jwb&3_mWq4UFDSN^Cu<B7a(yTZR4ah&5W#Bne*=98~(EhbLgZ5`W->0_CO{cY?ueT-47|27tVZ23EM4{xLwJ)eI}ehS7BRXNYpX^y2-sQuRAaI(RMo&+|dX(;GPOzYasC1^A89^5YU9c<r^JFC+_zt4)Z%V4JkK>a9QxgN6b@;9}cV%Xg?x02q_J+s>IC;9tyWNbEbem2*$C%qM}$=`beKs`N(jjKZz)C-114aMwQIk;ps5gZZDN>vxIU|3d6*F*zY+X5R{VTu8Ok|0fG4K!f{@j`)$mQ)Oh=!1v$CG^nxB;Ko8+}@|UZfqr0;=`O=H3tX`JlCo6{2*s4YTYWTHPNFP(Xi}w&h@*Q<p7%tw&zC?Q1;uLz9=u|w;SAl1jpH}UVmBeZFW+~GR0&o=rx(b8Me&>AM=~f%9bJoVymA|vS&(=iF*xG!@os^GRsW%YINh=O|up{T#K9Ic5+?^FYh?0;eLL$<Ku4Srp0_uQGjplt4)KS^=jN|@3fqx8;KqyY~zV^@y>i3hail8Jq&s?^Wk1i4MPw9`Svn^oQzK-R}sE&YFT8_xaM-sb9jh*jastmCSWHl!8094@?2OcA330gRC2p}wEpohzT|_r?*7>5V5!o<pLb*5h+j`qDRUGcciA7K%jIg+uy-15XTQ<;{Nzu)@<(n(ngMQv7rWh_$gs#ebGnzU#ton3-vk22aUfgSw;YoD=kuucepIZR#CtyoXH7P(cS918Uktf1ycU0O6TcEweLwfMj3B&^r1iz6IL<7R)s~KZ(4OO4VJ#fQ=O4F%9+Swc65p7&WE&Qi`Mi)}D^OGW+23sPof;=v4$e1DpF}DE57<zjHoe@+yKO%5mLv?Hfm^--VPvJ5i!6v1)aWRB(t2DA3v|;xU0hPT$>U>xyB9D;@^v!KmbzuwX^y40<iGxnZtJFaJzo;`?zftdQXF5X&KQSy&8_TR*;126m@Ruld_JL~tT~e>o4SXw*M}YjtC32Xw6*uX85ZX_eLzhG(*2Br{O){04EK|@dg!~~Z6a|w#MWv!oM6w_Gs7-gJaN!=Dgvcz>s#8(*X;Bc-tfi-6Edy$Y;Q@0Rm=Txwj>f<oHBjYew(`!lB6495MOQ!_#mRW9JD+I<E+W;?&N-(8UDL1r-OHH+z4~!$8oFerT(_Tu<^2f)ep&6KZU2tw1fYM*RJGMMGZ%i_p11P<g2ja^QoD%VH@Jo-^&c}Nze*z#3yMW;hO#IBZsszr7W!F-8bT6Ig=VuQgtY!?&j|~RnIljEfPny_KYwoYdwo4J#_lg54v-4D=AB@Z6oB;x3-Fdqr#e>bZYjdS4;z=F`my8`p4M^y_#!pov+hPQ?rWKvn^}CGXndnZqBD#uEaY1hX(G$U$dHFg**(av?Keo7ts{y9W<M=-@?2s{3@z3+g=ohji)VNk|!niz8tinmv)+ojnki0n{hVC^m$*MKZ-S>vy~**-FmD}{4^Wno%t-LWX@Zmw3%tFQC>W1zuigx)rYR~k2eJS=euS@^Njh{Gj+rDXuxE5dl1V*y&Ro{>3$S7=(eXbUr_nCxIdqRMz)p3B)a2xQ*RIByd>Y>e)lsAn3LJ^ZLQC=(S%#9_NaD2gXp(H_8K{oufw*=u^TiR`}dSS@<~&wnGL_Cn;rSlZ@aN?$BfCnX74<3(lJlx*=^~UE7^kck1`3OwfDPUk6&;mmnIp==k9!lU_YJT@gU}|=WI)v7`WaXj<odg=i7}8KZxr5URUVvMjF8zhnk9BC-q^$-EhtC;=XW5LdnCh{~YovTHoR0Ue)!q5(kIt3zk*_IhbLOtCB*ivMx0NnimBvB&aIL1Z6}qEV<%<R~5ySNvUM5ONJ_4S5%W)ilOdwD06z2Wll>=qu>5EDh_t<e_x*e-~*cty5urKKgram;Ei%;e*4?M{@G7V+xK5T|DV7Aua92)uaExo_ZMF`SHAMfE6zn01h4tkLO+aRmDHyTZU?qog>RZP9s76IdLeC)3qCz{=`l)J7?zQJc39tid^_FpeZR|^!F_lAUo))f%%UIt^3U=se)(tCx0Zs^!4~oM`R7SLm>cE6)C~Xgm%f~LV!G)wszZd{Z)UfAocmVvx;g9zH#48a4f>IvcRM+6_`9q$^O*o7qTkQ^oxT4zhS2>Mej_=JzD(IPx8q)!7Y2I^bvaA6F=JYaub|~9NofoL7Ug~@5Du-ogWL*5aslbY`OK5v>8CaKq-}N?mFM$Z0q>j$F8ooVW!?@>(|Bfn6amBouKEpU_v}wS+Fl#veX8m@wrgBnz0)>5#$=;Tjq5>{R&h0~^QF<rioxSG*RtETb<5q%(Ao+b+O^Ka&p&Q4q??Rm)P6tQybDomdU?ZqiI?Bk=ih(FY|-Vp4?dWW2aIq!d+BYf3S|e~CVke3ybnG&mNI7cH(g%5^xVpbVr}wUzS<`$KxWsnw|gdaM2FnM))YN)Es-b$V=os#nt%GZO-%u%Q9|pYhGkhorX@hIsAyR!Ug5HWh){?NzzPK@t2wVZC;?Fd<C-z3rIvUor5s8rPl)%p8pAJZQ`TeVAzO_`eRHx=8Fn*g@WZ*0&!xTQHu(p6Qeq0lt@<*Et9Fwx`_W&XyI|(p{%ywUwaRzTo>v#%HUAjhvN~sR^^K)r-^-PAmoJ5I{N*bb&yU`_aAhRR>eogW_SplbZS}VKr|OH7pPDkw!wS^B<*M|hw?}_<!F*}Xq#ymn+{M_V=4nPhP0!yWaMNlSO)rNTF6tx_)TM%DEeTSBAu4K3i&EEAB32bf`W8ZQQ5Q%{SeFF^n3kZX)E2+1;<S`TgmQrec(go+LuJGdl2QauSQ(L2F!Svv=W3hQA~sRFHnU?p9{r-P<*xVMzIphT5iV-g1a?_897^ELuckX$zc<$dqiCD|_DmjCZEU6<lQ?$qYUi$t;eL;T8PMcs2lT_HBxLS`A9^c0(rf#~TiZ-IZ=ZGt?D-C!0C4^#ao!SHKqf_j5hqkY&Lk}qSBMlqD=KkiHaQHbP?{*sIYEpQRTHg1WwuH%5OSOfLJlh)Z{_iC?}K^d75^5OUheJkenDG%_jT~pz2@U`uXR!+Q{_=`E8fUcP+OHxo9%3a?NJQ&PNLh<(OCIkf@5(v^0kIKpN;4@`Y;Ciw$fIU>fCSon?KB4M&Aw`9#7BVfgOhjByfgEJSa6%RA`A+fop+lSR#N?Eg(Xbl!6wFLLmzyKCu><0>B`mmExq7RV@G**!NB^`(BIKttkp#cbS|GI@O0U?W)PvzY`-h%P9iWkdAh!iZ(++Oe~&^WOX4o>V%A=w(o&-^qGl7;zS}AVuqOBDAyR*wHXZ3OcYAi02F{jDNAJJq*hEKUP7d6Y_kG*Q3xoh`A6!)VxSJZMgy<WNg)HDDDMB~<}!JH;je%86W4laH0^r`fq2tu`kcAwN2$hTKUT+D974~E8IL+Q$eVA5^tT&XBI#-)VNb>qcWOM^$3h*1f`OT!Qlh0S1T|JS5R_9?R*ZuhKqWw_Nu_kn>QeHeuBEZvIpnp3##T2bx=;WRq6cO2^eU6zw3(RKQ%y{ONo}(^zu|53ZAZ5GV2Y)AJBYgA3=Jx#zs22{`4aTWFSF8)K_`3$Iw1>YGAW8#(h5s038cs<dJsTeC{h`X5JN1Oz#1ybG{nG2HCEM}bEr{OR2&sG5y;5C;?z{<|G<&9(_WkpPh6b;7w?$4{qbMCW4_uS|HbS6pLfpx7$ft`d3(g4KKtYUVSYaQ<1DxGkSihSUimY3<@bWcKU49bV?g#Z!LgX#Adf5v<C>^~RH8C_3NeZ$lblFQtTfhUm%*i|1Rz|H8VIba0;^gY6J5~9AwLc|XG6}}lr7$4ihD~u;~#7ifR}>t-UGz%t@s~}A~e-@1o{ode5)asxR0rLy(z^?)#7a*Ky;Eur|OSvn(vk0*nE4Z|2Eig+wJPUt3E|)GgbIUpXiNLSib6Ax({2a;VXHG-YDilH?c44i~VoM%o!fY2c`u%C7c&CoTyd_q=jGv8WGMABemJNZ0a!92y>%dn_o);jEsl4qBXA-q5uI&6y!oEIwYzMiE1xlqS{dqzSpv;4V4{fk7>95^HHoL@;M57A;mf2B7BtpJm*T2)?>}>O8b|kb{&*YlyzqpyV#;VzqhVj$j^uIbn2_($yB~Hx7e=Q$&f|k?>!@I8Z>!wsr;omx!C0;&VK3XHJn`>Mc2|w_ES&$I}L)&xN|Z;9ILvTWBUt!Q(a8ZsFOsm9DnJ#UpPCTH4K~We(PB4GLN3?_yV$@mPWC5%^q1yEYf$fpL#~nGzydM`v`{{3X8yko&D0&*>+QK{*7Nf;mBfwU0BI}T58(uh1J>WFx~HuqlPczk3O4cq51tVJlb0k88);Jzt3l1OFR{R55;6(MnHs>h4gy|G+;kmb4+hyRzfvYuRl|`P|9Z4G_Fblfh?3loIwg%RTdDHRG_LXVO=p<2(0UpvyyU-1(BeTLX-#q(@?Tgod1}#g7~DDvQUb4S&R#pFD%&6y#DTmi@$!oG<NERiz)LlONO)xh^tRe=UO?G^1WW|>lxUyewc5$6_;d6Utne1NxU6jy1SWUWO@#vZknPmUEPY+&Id8~CInG@>drT=#}8BURHPcJriUQPGs!i?7?W9wp=ewqr3r%^vyzFT(6G>e2%wC3DiHyyHa}^iO$+PT!*vO2T_ep)EX8R~+3t-AH)6=zJePfR*soZho1M%$cYZ!g-FC`=Ps;XR=;-o$|4Y9%`tLFH>wR-(pdJ_FQe}*%a^$|3^S`pjnRnFN+s0m>g;cZS^pWBV7sJPQc+B{EbeY}uL5aj!aF7DRcVit)pp*E@M}|j{P-`b<v;=DPuiVl5>HJ-v-_o&yeH8ku<Q%gG-b43iD{%$vbipJV<b;<(KsKu1+4E-gRp+i;vbi45vH0bw1SWR^69GO`sffBnvaTsrTvNcA)RY%mD$FrbL<3qXtZ0pRsTrYCVpJ+o$(j@u;y{rhp=L%BYT(HeYPQ;{Ue2>}$3w2|L~m2T3Q3}?!NzT8-FQLwAlcKK0T<N%yq!VI1aW92=Jr%-&L^=l?N*cU(?lYgWSvPIOeC0!h=p1FIdcfajPis-9^v9U*s*Qc!wk)`p@rHCzMhOVG50ooo2RI6w)hgK2Az6Kxs)qO$==AuuxV%Ln45z6PR%OPr%7|}5&F=WyrdAbREeWCmb8#saIFNZ7&Zdl2!DYT5)e>kHi6P;#wDx@2{qOgq+C*J1iw+lARN-phqUu&mv-KpbsmG=*P_P!`9canFZA`a<U`|!1I5|c`qJn}v1ua7z3)$i$a=RXKP9s#pt9Xq`o7|6Gn&}@e<V<ltr&Rsj*qRB0lJ>xxZM@wtcd8L6X+=dvg}3oeQ$9@4DgMA?!bO~nvzDmhb?U@(S(iK-L51kn?EKNwry`QeCh0yyJlcEG2f%U?gihB<1~KrO>-`vz!oG}<SYBov~Q-g+iD@=papzkGj+DMK$|&y<FE@nwC!!v)He9U;}}dYf*~E?Nvi??r87|!zyMKYO=St7Pz+&-NNqHRLQ*6^QC5JflA{{vs#FTF(zIn-Q3z{Xa}D@l!<^otfJpA?1K!@p^IFE|9*mXcDKo|V^uTW#qc)J_?PQ&N6ReX*ZRl~Mvu|wsyru-pcsG)oOnlet?$2fyKOh=Yk#}(vjhs8tM7Z0k=8C}A!3t@vY8gzCw<`3#I(XNgStUOJV3yqL%*aehjmXrB2}CecngS{X7q%?5#wBMKnz5`(%!S5P4N6uO5W-TFSR4Y-I7r%4;Vs$@f-K;pG(b`<wzNJu`}Ls`w%7lY3(3aY<BL!aBJtE(LNI9n23m2<wdlORnzNs$>$h{Ydo}3&QM#rFc-ZEXJ5lU%@WMDR4CVk~K0{s@1`ENeT8pw`C8Z3MiYP*jh6*T>vLqN0UDk{v4kRr=U1LlL=DI*yz={qs^z>@0iHs?LOC@d;ryur+&UNo?c+hd6&*wVo*K_@vWzJN+?jxEzf#}7G!Us9o{CXnV9+bzYR~}(V%>Z<!bjK3oN^w>KW2kc?0WNf{m1Gi?6al4bN(zgyS3#~AVGJl;!?K`&F)8$ENzebGxu=Kx{1b9hKSh4NCl%gq*!I$akk0Y>C?yno7xQXVg156mum<2NcAnt<_VSeJ7=1kaWqLP597T+phJmz3n-zbhK%=?%yX7y^h`(F@H$P%eEfErB5_4MvV)#izg52M0P~QWKS>!Mb<pM#y-U|$`4uK&9;|kf?4XN<jf~UnzBl;{7#QBH%eAF8f@Eu)u3Eeki>UzlZ-i$1B2QE=OZ`7PxjBJY8g?=*s5HH-0^vB`gVh&qo-}OR%@m^#Z*>61k#sXUExv0PCVqbi@d8Tk8Pt?5UxIv~bJGAkluk!m{;Hk~}UybC8SAF4P-l}zOv6VlXuD_G5MvJvETFTpX&!$3xfQCFtah|5f5b<ep{o8F*gQh6K=L~Ge0-4r(ZINYW^#j>8eYBc~RulNyxEUXCb3U!{tDZWz-Pe9b8NJ-FMQ&wgtY>t*+7Q6|qKl0<%#1kJe;JQru;*%*t#3KLxraF^@_at*JZ&KmQa1GLlbAT%9PtFnHvW}38e7X^K9?=q4XNjLPB-t;?QW#r>>YAk(}O^%dOfMO3+}_4Gz9J55&USuHCk<mzMF?$$jC)A;?$gh)X`RZrr`Ae2IE1x^8^};@4b8{x}?{VXL-*@1q0=C&*2qEC$W-+2qOmVL1bo)%t`mxzFYo-zs->k5DReK+#U6C=0wV)SaBSxSw>COuDR=X%fF9e-Ic`n`Hv61m#UR|R*QcPA15@%-PlEHvXhLVoXvA3C(gdt2CU6y;#!AYew4lI$ip7VQWsjnK1$VtXg7VC#{=&B_i)pVHo^nPK^?X9{kE0HmoRq^veNLCL-(1)Q<goa+2k^w&iH+2M5hg1=VqQu+JvyjEpFy!wO-5S=xUsd9WQTIQxrZH7v4#m<+lw(b!Kpez^;E=>D7%p6S5fxz1&pqh2HLZHR(}%Ep*|Tc3KM~RyM9VDX<(X;@v~0dq%QdckT7$kW3rdjvK^I`Yv~~&uQD8h||<w+nq;SO^bZ3^EN(fWJ;=tKk#*&yKyFV(np&QueC?q&i06GUgI-O{F+_!#ro4^v@0<KVdW^np98FZ92ePYS~H)x=?&@2pWB>-tc0{jH0HE+pN`U}eLL3IHQ91^(|Y;8qR8=Lej~bI<4)HrcqY02*|UrIwtm@cBm~-nfRP;+%)l`Fg4O7OVrB_u8DgBGqCgsPRTibfwGu)@$*LMtQ39!|QVW2Ut=&UXYN~6(2vZf|Qlb|XzGfhpeIW>D!3hawPsU~WdJp$yo9yyW>;Dw5PQ(OjDP4Laqb42qwv7OfF|TeTu&`!F|H-Kwlax3fT;xB55Pxa%%br3&$0d~71R+hy>>|@JRJf!wR8%EGB?5vMg0or`gg{XX1cBs0Nz5sr3aV1GQc9``gc@lj2iN!D`kpn{_gYBR&I#38jm{W#eJ&mfQaL!S7WyxLW7LK@+X{Z(#u%CEDRYX4BAVQ0_2$+i?*R+0ZR^xN<12aQ=KZm72k-7qCes7&lJW+*_bxTA;)aIj_>b3d@Hds7-Whr;kKKk;nC7;tU6aa<++SuJCttlcZa(+sl^x^UVwMh>RTqt8@=owErb*n64xz?#&{&+KrKxsynp$lZM;V6QexxPJGcm8n4{<L82H2T%ou@0COS}_0xbr7&UcAzgkdBs8iyPBt7n6Osk;da4M3dlbZ_e3+i;mwlg-FiiPCC16{OP;#zOHFJvY66|c5+&$_0dx2ggv;}$)vK4KemzjrOt`n(Cu~(OPyNA6CUR8OvLD$WG3)Ew+gn>$B>E$9(bId$OAv!sMTO`)UA*6ewPetFd<N_#h}Ra1?!Qhx79ef<sY?k7_u-$bizN1A6ItjOtzGLb>G$0-F|XBw6BTV!qt>UoL|yv8$A*IG>UVJ$<e;GkeBZ*H}RatK^KnqJ?=(q?@z3I;>mFQh(~nwB_Yz*>BD<E#_j<Q4L*j|(LC^@*z8^!TU-c92K2Vho1Mw-=*2{jupC9O^%m;LNd&7Kd@HV(B<Ex*oEMHSrp)VmImNRVX&&b)o(^0>@zj|nt33RDH*OKh%^Xt)?bFZsAYF2ndS9~@+xK#3)g$ZtTk35Z&UU}Otxg;_r1VUEa$7MB_8^F-TM_DWvy+4#lH9}7r$+6wiwS6Ml7FQidSM@z#&Sd*i^ub|u)WaxcIRST&g^AD?sn#M+jndQ@P<8|e^vI}Typ%G)x8x)dbLp#&K&(F!sY4wP7IdDUZ&12jxy7?2{I%R5bfp^bnR=J+AR97EPYmMS^ZgfQ0E9=nri2xY+0Md6sS+aqFE+4vt^RCaHsr>3Sou=w<-T()GK&e7iS$usoIFtp4)^avF#OxxueDF=*`7Z2yrE^dsyNVuQ2RJ0pxd_{-&*FjU%}wcrpnYC(LI}G%GP|xsSQ{M(3)<Aeqys1G1p&EuF~V2D@uJy}gVgI8hBRO<fSAR!`%_1oE_&Z+c<9#dUfRmsN9WaWt(PZfENHT9yT|-(22q5b_DG=AkK{?C?cU;=;LbaB&o(ubHCn1?5^K8`=uOF^f&MqW~$pAdg<QQwzp13-fAJq*~i?`?NdWN+LMOZ#peZ+wIy7j>;GC?=wljgdaEwID<_NK<D$HA7%Bgcd~`ClcM5&xy7$CP&fT9mQp<k?QUi1CSI&g<`3In!VWp#SpiMC=X@*n683$MIlca1eiTiTNeY68yRAn3APr^Gy49nYFtOV_{bnqdZ9+!#ZJBIegMZ$w5blp3OOr~m-kr{b645GBXz~np^ZJDFym#V*?S>gmvCrM~9z$qzXFl3yA(Ccg<$Rxtzn6Vi59jikO|amTtloInPnEZ5-8Dmy>%|IkF*Lg|F02u^^<`49Qc_d^Tp(J}LW7D5%O<O94oD4&W_7^<hXp7rBp9W(Qf-KF4Kc14JI1x1K_h%IiuUj%?cs_1c9OQc<aP?btOwkNs7<fOv@e?zIK;9JqfdMefh>sF3`epE<Q7LkT83^`a3U&PYgp^55}=kvT~$aDEdV6~@)9ZrD?n+jFslfusuvYPXb{COgeZnTxG27pQPNKQ`7dIoYp``*kD_-oN!yeN7qat$raZ{q?^Esq{!++Y^A%jNST-&YtJNFvd93_0_doA7^70(wUf`BO@ye*Rw5TAcLB#+OwUxpIluDPN01!Y~1Ff+Zn4<~^dzRHgL&gVzI|$roC~%*YNc^|I{p+9o#4?Aye)+#X`p@5=f6e`K?saqRE3drb7%qbM#QZw?ZaKuCZE1oHL;FUpxJlM-Ar&ewIOpEd(&%VRxG*Fm-aMI^k92?9{A7mp-SvMR{qoP!kzf8<2#h9b^4sa>xzVe~ZtC52*V=gYq39>jpzS%2)SZ-<+%#*V{*g${T<50dljy?o>?!xm!8bZ3Ad|crVav~Ns_))uCloB@T<<BxZa>vJc=zf381?n7yZvn^=>IiO9F*b!U-}5Zo%lZ2@_g>j81~xOA5)upY}cLA;FH@7PO(p#+;r=$JbT`JMf!64L64_JGz070tyM!ZKZ=3cN$}eFaMgq0icf=Xq27hVE6#TvFmuffEJWVr_x1VrjU2jsPG7q8!3Xp4P<^)g^*Mb^7`%)I=DC32#l7n-uLs1<)rfC<yJw<d9C8b*Z9F+K9%3&W6rO+jD6gj>@rbd}c!@C<0zhM|F;wsp5e|{AWTiBvNI|J!Ny=Jrr6}eF6#|vons;o-I2<w#U&@TbPz%+>;MRf^oyrJ`^{^W&18ekiV2y?d6%5R7OoDL*Gz5Z)vLaen0u?ppvSNs#+E`zl0*t6GNQqGi0h6#qni>O5ONJ>JxUo(LH`Z#X#tf-Y`zcrUc8pSQd%b+p!H|ndC+(`rQD#})O&rQilMddTkJyi631v_zzYNjg0jcS!k(z{u`6M$1@>!4y<Dvpe6NQvEKk`BXQZvdpw@PEpfGA*v6s2V?D<TWQv@BIsRfv~^%V}^qz0f}XBq3vpd{!X1<BKbjmKYE}awwv4V5W)wj}C3|{+~APdbGUN=#+7FBpd(y{Fz!$uk)W4v*`RU3L+{{=|U1Nl+wtU>XojbtW^ah#f;VvK_Y?Hq$pttjUs9q3yxI*-~hLLilIWUO_K@6#Kfzkm|gvOmVG=)jDw$NSZsR{RVB3;_c3;D_d_&qyFcH|jiN=~B<*fyt?wrHb_bWfNs_*sSQS%!*Eqv}X_&imMCX8{Z~h*)NE4gH?1erAoZA^ucmU^5)Gv2ks^yJ*x*D#qPqO-$G;!m8Sm=b%w%$ch4dsQ~K972CbjT1Jke>0N$sZd|nh;VjHdETVC@Uap!V%PfRF&j}a0O}s9mFb73YAqYB;;jXLZio9PH!qCuS&`*J|ssD+WHHf9J!LQr(E?Nqzw1&iIu2v`@LbDgM+Lcan55Kt|lhl%}n>-8M1i3M@1gbG7N}Nq7qsVP?VxZxZu1lnc{$G$q}+FyIg8wgt^2O2GG)#%MxH!mxL%O2A}Z@?lb=R#VgOiWxN@aylwpN<guv>7vBmT>Q54;deN1<pTGChvw#1Sg}=IRPW>zjoF4HKw1|a;f?`l`KtTZ*ED9z#TmZAA{u!BKCb=jvg3_4&Qdw_6AtB>+4NwVaSqVyN$#{XetV<=!nyLyZh6SYc;37D^j7&YC5*ablRN!eXFSy;0XwU}{^mo6<$n|}H7^PXLaU2Cy?O3Ck`VNrpTN(06gFm|LRlJ+2Jkn>er6%i%NV`FPAFSf1jBg>&M8Y_pB?hL_n7@TWR1su@SEaF-L9L*qywJL?l`w`gsA{BZ%oQp)Ww?}>6%<0Gh#q9->6MwQQ91jn15mt~yriQuneTIBw2-)#9{g!zeTuCv_nNLI-1Lt3cBM%-^*=jmY&VWOS&r`AQ4`#EOrrzW+{v7SKokX?B}peCmlzjy!I-8<h>~egK`ccr02jCbLZZ5a$ky!<Y>iJQC_+RR49NP0bq+cr=pV*4=x%NE8l(6-iFeRud0uj9p62R*v=-q7E5FP>aoz~1#g1oB>bJ{(JZt5ioAFD@uNdy-eTMIzHI}YT(!6MdoOQ!I7ssEc-VOsj%Kdk(2Q@xmYaGOSIv*Bk^VaWszh0^d*v}zILbFE>kz?P(m36BJ0>n$A3raU{AcpdKhV49p(!{~Omt4@72Iu!u`Rlh!bymJEouh9-mcLR)FYFVZn2H(p+({y;ngPWz0|ix>A#Nlp(%e|ywnUEu!mEN%EeoKDrlca)7RM4-NQ!FE7*B5?el-(<lMG*U>@~r?*G2)7DB!~N#@9Vl=W(^iTM>6~?XbwWy-^;1_x)I2j)<4!vgQ{&&~3ZRj7M%IW35TBVPlu1>h(`|uWJpLTc9)<DCUo^lrr-lz_aACl$2Pa3RM<}OC^wilxs~DBal&Jh8X*s2ySfjit<_&HLj&mM*(34A(~f%QaUK5CsIn6mU?QIeaUi0i`PV{mqL~ER=|R2ux>-S^N$kxPJ@rw4{~`e3p9U}2pL^M>4$AcX^@W%{P^fcZ@5wP7r^$9?fr-v=@sq_6fPk)Hu+Dhk(JZ;z1(2F{aZwh{Llm5&Zko1cr149W9PY^Z;=6yS&eGtWo<>wbE`*vp_J+4m7-FB=qxMJqEJStlZsarE4buD$$}d#4&u5Hq7qE83P7_vDD9BlO3Kaum}$sr#7{$Asx8O-!3$HJm_4s9xo~-zibSLEeZo4_QL5ncm8Z54^(84hwa@=nUVu1Fb}@eKPa9w7AiI`y&XHZDNZX`))?$^v>i)Kc{~tRJivT(7k9S~2BgMg1mc$fJEk>H;c<E><5hpK42<*FyN8I45Bi%@>dUJ=|?CKls&|K3~g7EDG?$!d;XUM_|=Qa~wcK1={lM81#z^mr4r&CS0<6hi%a<zA|8q(Wuqgm6}B<Tf5j&QB@TN?&8MG1~+@jO-i#{iHT0aD%P+D1r4cy_CiEz2KQ5CK$CN@$HM$ZCRtB&fs;)HUV=lNumh)x3~JQJ|VI%y~f&!Qw?N*&9r%=WbHLCp4*!#({b)`jT(fp^aY;z)bS|!t}XdclpHMI~lD=sB&(|&qrAR!EOwH$XTN|BJD{an)XfZWMt8?=e+77p>uEhn8D3Z3=-L%L%vD(v3wbgWgK*8e({S=Oe16IS@9d;8}j@c-ojgNwvKH0P5I8}?dWrRvfg}iu)v<s0wY3j@p6Fy`HyxQ1gPQ|7XU-aH~^9Y0kL33B`Kqru!<6B%&}4^2Fp?jS|MX^6{>(jiBhWi6z3bCzw(YYum5!~Y|pjqizb}V4plTM#_>~|cS1Xf^MANfx*T8|nNa%Tys>-C7BN3^FZ$BFGjg0__6+2BcmSIX9G>Cu2t{H;x-rVdXcT{)>gEuPwNZ(0=3RZA{nJDWwiA2!-4KxtseLyB=UMbQ?RfiVl2sdiIi&49leE3U!%|Q<Q)zpeRYU`=3ROZPv|trbfYPeq97yxBj3$c_DHM{hQk*Mbgc?(d8c{|89fGi@IBjn;tAidVCL8a--tx};cG})|%b#a~*R8p3X5DnHvIms#%3VyxyB_>2fkWrtTh+o_8A{LO1ToPeS<l5qjj=0`m?imELna8+C94Fg7-j;48cA8zLZMn$uz(zt7*Ir&Fh6k43TUi)N`yIAfdO|Xnei=uc93Bgb$#|X)}YS|eBU*T$?09^j$_IPtGqn_t1IRUd)<Gk5lJzKMelP8w|@8S3+J@IiIs!KWWRX%tQqln>tcVWyj;Eg4CJ4GvbTcZ2=})I#D^7_c}BXuXxrWzdAGIAzDs8BS`KFW`lw@G`$Ro2R1?avr}(-W;0soByoU&!Gu5-F>g?}mZa&>9XMwvppA8_;wORG`MJv;N@vv~8!l4@0*gF@D-Eux+?iA={#LNwkGHor_Ox6`)fK`wytp$Rhkg_Oh#f^qpQCMkRb6nO0RiyytN5*Seml#q|@!5mSjr-(6cs~nv(JurP*#&sC*oH$~^0~w%;fdptNtv>{P$Y)>D9BO&TlQlji@ciTgEHUCdG|4mevvnG%+fbQdPwNVA8vOS+g*J#kRKmJD)zv1JZHTLK2nzjDcy$C0-xVpOwQI%13u2i@}D*$KXCJ83vo;N++nfp2ZHDAtOzlH4F!z)sb_Z_Tz7M{%u(9Lb|Q|@)rFWubI0Syd0P7Qlp6DS{AJza{yP{V?>b&z%mR?sHS(bS2(gSYQU<skIzeT)t8+@Qbq_zkn1F>=tpU>NF*$PXG&}^^S$Dhn&M}vUy1v>yygqdx!rra+o^b)x(V2x94;}keG1qF`6mm{4ll6OZqo(#MxZ=a$JR@wAcI7*{7n6}z7h5YUOzYIsrzbcF2(4$Dwb!r-w`fGWbGVTyHXC70_OEuoYBugJbe^d*RQJMiYN6FsCNYUwx8vzYiLsQ!Z*Ru<adgZ@Xv26}i<j22(We2}#rwH0ytw5{F!H-@W-8h}?+sju-oV)R^jM8zclWT@eQ57})6{A(eV<1?i~G@0OY@LHb>x)q@x#2BIjp$Wmps+R%xIDaA7`wvlOU;fG6;xA{ZEsKc&8u8j{4q{t>K%wa(=`LYKavuO+%ZG2uig*jqFNST#H1_`yuQ7vxw3YvdA}+%)OpN#~m_SHvua?FMYxNbH|JF+YRfwod7SA_WB=va@eC!tw}eo)yQW(S)5+6es6Ln&(y6B4!?<8W1X&&^F45mHOr^jwBl2ayBo&0>C9pl)(oF`boTd-Fi5czFQV{C-8@9v-wccM@el8r>RkXe)%7<!_u{B;gA0&=*&MBBA4Z~Q4w<)M=d^Mtzj!fLWjJ)f){R0Xu3NC9j4RHk%=%RkaSZNE%}e|M?Dsm#(vJ_ks<P0chMr=T9%T@-x57J*)bVq@?{M7kOJB{8J=uF*nfd|`eUl+^*;k3IyR{a$85YyYRzHjRHGWUJA0=hqAoilOU~+_@^0Yw9u8=0{B;?A4=A(}Cm~AfR=D$7T<|vSqHxex(IQrTfNBGx>2z&U|?>@fmraXKBubO+BzP_6KVAN5{y>sP8YvuQ=-<_5%J^Tp$+8($Owx;bIld?S}hwI$_ruoa<Z)pkv3FL5HEKKBHL>sx90mM0wOMmZ=UhRx&V{4S=KLskU`j?hc<M*bgbT0y!v_)?JtnOxwvndUUfqfI}mw11f>68>b`69BTe|5U2#Zk^Z=|&lUUElv;^q1!@7*o^!ZEQ!69{BFr^XkI8=AU!tM%6hR+9}VuaQ2mRmoEkL_41XA=SS~dxH6Ju_3P2)3*Oo`ZL7D<KUJSy`P|s00)*B~Brtk+o6?uw9{tsYf7BP9qjB_;s?wJ(oi)!q`e~Y_9+zAG_Mafi@scDi)|yucL98%SnwABkOq4~%V1=MmfFa2gft(3k0)$B|DU?8ISRe^m1&5f&fL{7Sa}BI|tmcu+LcE%Bv4qUXy#zzhTTo^<gxye+<#aNyLMAYn1g+Ja9#{sR8#pNwvq&;fYb9YV5w~cgSl5Vf3Mm0ajWFdvl@K9S<C4`B)<#$tf@{SUF-MR&H2_GTlCU#)_F-r54|f{;sH0&wXPlD%DAcj0W8Tp@X&qaQC<M_vmd>na$<dPm(D^yY%)SUQf-{T>Q-HEYH6=D;%nDhfst}acLW&ZSQkD{Hf(fY!AwtL+^O{qQL=9M>OT_7rj(B?05j_r!Pi*li(g$78KkxM3YhYdaQDHXC-_EnzLU-x!zujL<oC#^z<K)2fQJ$Pe@F0{pwlW^QANiArK)BP#H?y3eyB;`Qb*-rb3K5S(;Ay7WYwos@Ztf<3(`IzO-ip12WXlAeXs2#7rJP4!R=`f28%?=(;`Kv5=E-nss1bBCl#f{esY-+-(#Yt(2*MKLN=SwaV1z!dYKo<xh!IHYN|wAtvQPpQ1%*aS#;3V_rdOyp^)WU>=CT*>eo^{5nUv(?&d+CMG|t=S3uOF<IJ*4a|I)9G{(F?&-T`3->U{{8sw>*)y#WW`3VjdRXKx#OeHJXRcAQ?+U$_{)uS11?5ZIOv%V&Y4^6;EvYb%u-`ilBb+W4=lzOv@CBmiTisi8BppxFNs<5-W{(fvnl_d$a$JIO!_9MZOTMSEfVl0$4a>hjt1W?@$6u3WNtf6vkH`&6|%+|};P6r*ebR4FXj7ghuqCE5CWr6dJms8k99Tmh|xr9a`AFd=~vxRlk98r2%Gk`LwSLwWj>z|dOV)#{(el2OhtVpj7@kLP-raeo3ZSB4Oni4y1na>YuCP+9OAYool?q=Kx#0+A8|WOX-IbBrpA0F<z<6b7=^990!Z13k)V25$YTWrJp542dHy8KgVb#9VQqd=KSdEz37XKQ&^?Oz6+d?3|lx8}M&<3qBOntj1Kv8_BB~)J3C?eSJ*SS4U#!^BA1kwin4)wh}PkfV}!lR5aqts|A@QSk?jxA*rYesvy-wVp$o1X~ichDs2R5A*41NgXEwrV6E#~NM%H%QiRE(E(TF~di|OA10{0w$*pAM&7LXBVuwz<Fh>qZG;3|eSH?)ae$bMH59gBb+leWW-d^3zP^n`MZ^}-~y_s8yg370|Hw;)iNzmujj#+Z#;0`=lc{ai`1wT<Bsv#GWRhUCuDPzY1W2Dv+7XT6hN-Ya!A$Lllh{iRPh}8@jebXq@rGSGGI~cJiGGZ;jnk@`^|8E2_+6g|zsLF?#w8sMUGUi@U%!oOQB@s~K8Z$s?p%5_Ag^;?gxnNS3HA6s%q86HBW5^Q0j8R*a3Kg}~5{W@aJH6)YW(*UowM$7{y~(t3`n6ZjBFoa0VwSVoZB1FISq)1T1itOGEKk~gyEB%6TpR-uYrbY+Fjb$@RK?y@wcN+EWUFFhby8WA0v9zc5F<*_nhK&3*PK=uRkA2Gs6=U0wz90Hq8j5`W6d}xaFBziR}T7uq?YkG$~<QsQTV6zk8GZz_taf?NhXPFGZ%pKg9Lm*B_MDTkODqa&N`z4*4!$7g`%7x%`^tcNH(MUab>i`5|M(Jij_hja~AUoRj|ND3FOphiGu(<y#jF0p+9(KE?{MU9<$#j34dUDS|#j;9&QKc+_n>g>yGKzG=Jbl;bxZMvNNAc%kh-Dm~axCbif8DaS|X_0>Q&t@Z<Sn?m<94wSa_9Kr#&JED1;oq~w?=hKMvXP7%|b6dD&$GoU#XHtZ{m4P8}0p}K}zAj&GqIKco9-nzkC_X3d2T}wH5*Rm?a-3UOn+|L|0-?}of+-%2k#^YSS<?$FhFjE}~Y{jytsk`zU&px*n&;ep8e3!IfC&9S{xbbUI-Ca$R{tJ&gwB=$-$>(50?i;>|kLP%}NE^=?9HURjHD}G|+3<@}g){;+R7TlF*a&pRDuQcJ76q&%LQFx-6rkoO8=S)mqN1!6A`DZA#3{jU97yG6iBv90?tgE<;5yAbq`9Mwwwd59NUi%3IOrgW<s3w9YZ|O8VfLo{@ld+H=KBtiChPvUZjyO8^WF0ATH5`ibgbtBM;pB{@q0&LQ5(6Wcz2Um{q;6e-LcMn68$<G{@~(NTMr@j6m8wX#r`ow^1E)==6Fx<Q$zM+SZlXBhz-}Wh7k&8v(sb4zL#|x?jBM5y8XK2?XK-4f46+cZPRPf4K|Kwfh(?7%PxzR7A9AuNQw)lJ86ENY{|fP%YU%&`gSjmumueqf4p~VVtjd609V*gSZ#@x_GTKz#O3YXaDRJ~@~dcvju#?kmiNDX*&fPPA9<gD>ihYOY&Z3C38b_6gjN$ST$67S6rTP$2{?&i_WoNOQ1nkWWLKS$MHJFw8o<xEoOzaDdY(1Upa3(=GgvTVOjm#kSYp5^5V|H+iButpCN)L6Kt>T`rBqb$N&`}qh+-`it94B)`NFszWrAih>*H5vFP)1M{QR7bNt-ZVqEQPx7Ebw_v+rCnM)f;pzAs(y08zoU__Ok&Y5Ypqd4p_T9sS)a*6xlM{+&}|UirI*erX(ex<5_3=-5kMf1z1o?!rYsfbeUmoD0QuW$HibG|#|}HuG%WN$B{k{D}`ErSgLAzQ;?umR0(0Wbkzjv}c28mn3I^NNhDcmbmogf|u;q@J{;3%!9slp{|WyZ3XBvst<V~{D&D(S^59#%a^rLFr_+IzI&-h=>4UcA<Hf{`hU;<>g~&yMnAiF;j%}G8~w!S<_G{WA93SG1(~H3KrU+YA7w(7MySz>30g=a|7>xvtN~W2#<)V78e0=n$+X~DKurrOO9+Q*l2e=_y%uS;Ctg91@VZVL{ap@;>7y*E)+E=3*S(I~t@up{I$esYJ6>IF5O9ZZ$y0|*m<yL+MrI1@K+qaXQC4Lw0LN5Ggb^<YFBumEa7D4?Mvj5fD1bneh|#iSMl;qNDk#dCbD)L!Ux-3$!TFVbX!a9pBRGxEhErTLC36FLq^qq+w`mVPp-0Uum-j~DQG-J*ppE?`IqhB!b`&%8w=S2zxp4l%JMZ<n*S>zuxyUwJXu`q07LtOtGJNbuskCl830;pMem7z4jJ@oA7)xIgYJG5Cj02j_BnE}<%`J{Hxa0%JF=BlfxBGiq%OmSe!!r>mEs>mTxsGqUE3UdsrpS9e)$3B;FcPfzrf0HZo=4FgwC}q9))g&&dc<=t`BsVBD?a>cj<%nmEc~`NJssm12<pE<J|S7tGtoWd)3Y9fcJ6JZ{;paKGn44Eau9Q?c`z**Ef13CZ!TaquXq@1U#z+03ZFu}Z_i)bz=qC^67Xp6<L)*2VF~+uKdJ~1rxCd48u<E^QBw%&p0`<5sYGf!`E{pnz3!G<BAVFGdhT?5&Cf>2BH5Wcth+1=?-$KwjXis<-!uHNXCMh~VVhp^&*`(-Ox3yJoSo~P_OKl9M)>JbO3!QD(#;p%;h4BK3cU6)VA^!>W@|R?rlq^nD-G;O$KH<@-eL0a+0NZd`z&Eyjodf1-`|;U_EiRP6`A3GoSk)r%jWN<<JJrFm|^{r;j`%Iz2SYYLDr<q@Nef1>Gbv?oyNxDy!ktz|38XvdvdfCT<SLOio&kPbgpxQjY-m?H@qJH*c*h_!AtK2M0huA;YA)IomQF_>2viW(aVSHuA|V`?NpA~Gi|<Q*5r%}n76+1oGaxUzhasXpnoBL)AZ)q7r$(gKXkj?^Sw47&cn!Q5x>x-yz9MIM%i|J>O(}*#u1{=ce#;hbQZnQ<ClAwzb_8EI5qqt<Pn^SBJY8F{P2ij)7<!-J)h`#?*^>6NstN+MM5AwcbjoosXde7t-S8lPUfv9niRhQC-IXws}8)Lch#au+YVOJ>+J9LyXD_cJHke9`d8|=4+jfq!+k+o{(H<x4x9A+v_kN}{d{LWDKp_f*a`hQ-wqLzG`+2X&9%Dq&$k%}#r*Y+;11pHC5j!pRr_OJc&w#uXg)2C!#=Z+(_#lR@2TlOjaUuix#sS>4e9-vh7Kc)dmN*pxstG&#9<RVb^CsE#d>C?Pl6s{!5ZdLSZT|t3-|pee=9|FyINyXQLg$GH=l_xe=M@=_|TT`+)vhae*KMH1~=2_XQ8|AJoLvrKKUeHKxt)$1kL^Yww|K35$65E)Gvog#tfU%oYtC^kac^Af)<%|ru4IIKdH?_hgF}$o-L<loAL7s;k}!#$EzU*mkp!4D#N6uE^bEXw9srniOQZsap!jP-1^1G5x>mOtj1*>qxWu9&mZ+Ynp#85&l;8OkRN;^XV6O$^Ai=qTv1&^SZjz%SXP2SN-J)TOTx>_0)^DJ8dXA~G!ayTQdR|EN=Y_2y-#nc!LC#P0w~B{l&#6Ny?zL&aV#m#iSJT+>fyyzyU{0pb*QF#zQqC%6nv(t1xoWujNl@=0z?CisrgxGgt5e$D<EWr7?uzi(N=P-RgDXcgak!FOTyr3aDQ93@4){3lCyu0n-mHoZ~MvcvNl#{-u4NaZyrsuxE9ac^?80Sm97!s4W4kammB9%GS(+;Yam(<pL@TDKXdO;+P!gRMz4ojg^ed+aI_Qfs=JGwCC@?lcnZRqpx>J*CI4yp_gWvz?CSR=sR6^ZHvchKAzUb4TD)c;G2@^tkthV`yuh%eTGw0?4W&@R=D-j=Q1A^Dd@nQw-(Ci|(A`?R$7uJk(|ei6Wzyl!ZjSDSWXs@a{nWd)CRvc^+ktZGd!*?@n9LwgUr`FIK?O9IO4&r~LRYBfCFVq9LX}cdDpoioLPog8LTOfFRnZF6nBzfC4|4j2meV^nFz0+XxALlCpUrq@)Cw)7B*#>87JeDnPoJMy1z;A%s^qeOOcVuyC6fv&tV%)&(hzGTHKSDVT0qLR5(HGl$XTMR0w7(95)V*fr`WxHCq+xguld03Bow1``TE|-3H!F^8ZZR!&+dZZG_-wD&m6b>+c6jC)}iEe^X+W3fJ(92%5mC>c?tt5zzJyX07tXs`>v>^wu6#UNjZi9*0{h@VO`gCNr9HAB#aoLSd;)z1)wScR~4>RUC08R8sGPTo-j-336bs2XW-^%ZH}j{lJ|LApN-yVTKnTo0l85=mvZL3)~Uq~qQbqb*VgfNdH2=D%s<}Ta?NG2Ew#?Y$fg4%*s?i!p>Zqo(cN&X(U|ix^1~9s_OkKW{Ga)R<@2EPl=CsFFxC0|^jN3PZ?^Kl+Vjx8XZy@v+y?u*P5Ep+rSk_(V(@xopXlqQZxYtyf!!EcKD_fg(VKK+tAy;d(esqoq95sA&&f$f**i7(E(z|*o3uvxMHY&`cSPGC2#;V$PC5dR0Mpq8Bv3?2S}9l)BB2IYF`*f(u%;5$1+T1&jEU0Nbr@i#AOS+vs1QX-OAgp6=IBlPb9~ZEnQdlukIU~ZT)wbiNAvo-7cTz#_43N)w=Y~w{J9yQuT=${-z%N7?yO{*$@hBooM+&r^22<~t+*s*`Yg3n7}YK%u_jx9x!a0tX**c~RfA^J%7{%^NDn>DP$yOCcB;~i(c+N6wA`wA8f>m-g>FyaES2Ia#$?5W#6kgGV+ISwkr8o(Q^rwUG6-$4E)^OVh#KpSn<*=)E(r!q4YcF4M>`Imf_6Np{Z08!<Q&g%oPBQjL7UU^Zd<jyYaLpjIq`Tsz-;+~m`A=AXeT!ODis=BbGPk+#kC~qURD~hk;G<Ro$*!1gWgoOg(T1vtX)f4rnjOx|E<IM@22_O4<?JR(FvnC#{JSpkQc5x)YPzHY_?o$X*;-`6<0in9s0dzHP%(w-6_RQ61#eQU&)F1(ASyWPrnSj?9ai!N0`8rznJ?;lGy1MPC}Wj#q2G=vhu*(>n*b-dU^bob>cW-s{F=c^x;A(hHMh6*+dlLKoG#BE_9))qJ+w(^Rq&gK)@2vqL!kx#3_htQVx-h(;Vq&C^B~vE!1NZ`OOCM;OJn-c1#cTDVR56iK*3_?*^w<6wTk{tf2Ph2a&JS5w!T5rmUSEhjHqlrM;l_l?dPXJ>hx=XF7@asr&gx(!FOY%HPStGNJO|+Ei~3Iwj(hY_`4(+v-3ldZNG!foQf!tW;Sd1|^e%!=kPkskETA(c-D9EjYELh@uMG+N;W__NV|9R1zWx1ubR}w0gkYIqjG`pEwg_)8_*{F5&*9?K6t#!{1&El{5?olhjEN_P$b}MRVW)mz_0{ApmzA<LwgQhz{I>z*)<YgVz#3i@nNK^!Gf=Cz%B_NmKDCv+)7~@k$%(CDfa*My&3pm;DB=I6{O+6w!SjFZ$jmVC6`J@^IUx#yoTso;X=WRx_DTH@G*SKt=a+3o4X`TuX?nsp=$Je(S3X;Z=lDj8pi=PhBec-{=CD-*Pkes13lL5Vc*PE;fnwYKR7I_++>SZaZDD9jpIWn?Rrqj#-s#!=f8R;*9oXTQP)|BA*e@=aw^^!zA9xqeCsNYbkG;f=P=heQz_q*9ZABygfaVPV?L0G4^tZl9dKoBahlGN0^rVWCt(i(ca+2wNv4p00Aj!F5-LTMsq%<ZV$fsqWcUsKC=D!;t)Rr_8rcp(*B*cDc+ej*ZtVN<jn|46UNs7Q7v?SpBuF}2|GY88CK1m#S|KS=re>eoP*|7?;r8t`L$CQ1;~|aG0{C-K0UV3UDnW#ZQ=#@;mgWh=40~Nk_y-g+55ca-skRII(oBn>xf1a-d`5D_0O=E$&n`G|CUF}ICNEXFNt_L+o9L(+_keE(Xw*UHud$6W0~CzUpePj!!fYRrmhY3Z_9JxE4*>Ynv9>_jkE@>RBa|9Q4!sH%sOse*rG>Rb+|N}$)(rVWj+tB_FaUZ&r%j#4r`t79f!QIsSERq;tMmM;>JG*2$zpK4n+(7GKvWZ>xaF<upNe0%{LMdr_(03_V}+QqFMW|<xYn4Gj-oPflJg+_-+enC%(zYd}louwLUw~_Y}|z4;LDtnVq3S@Ql1KH407(0}!y3);%6hY~2qN*Yxkypu>?F-&`DJC^(02vaw0N$&N!FWu`H79x|S9w%VjTv><+RD49cMEM2%6cI@6c?tN!l&Ac5;N^GUAac(iHaWQY+Q_y-)$K!mZ&BdK-yI<^P)cHO-Mz67PfcBakwuo+Jd?CIna}p4cKJ;RJ`k3!8etb#mU(aAX+e+L*uN|@Yc&p)2KuOzJaci_Q)*w|UK`9G*<Xm_7sagMs-wZGCLHLxeNAb2BbcEZzuk1ly;TGP`TE2aEf4|-Pj@Gi6#$>UDwBLYZ=I|aZH2&~m%suYQ$WhBSZNHDQb-vt%25m4kL&x`hcSmmHIfRtEWc}9hz9#o(d#;Y(R$4PFqa2mHGlcj7O``sqvIMr`;yP$lEsrdBKP`nOm#?Env|44Dpv#LXN&HcUtrj6kuMg>3FBX>66EP;dEZJI`N=ymUs6aKMlmaFZ!yMugV<{o$f(R&3O#mtsrV3X|K_CHCP)j^`JO_{G%joe8855~!e0*{anjkq&!a6fkLe@_5`Tu$0J@oqjeg1F%_HX{b9$|_F#cXzEr$$U-ZP~Z5EGnr5C={TS)wL3(0J0#Omzr`yw5)WgNmVFB3JejVOoJhMe|ouRJofjtPx-&v#R!jST?f%!;Pa>Mx0*ZeM6(6NRIi{bM_AymqEr9C7P?I31b7$6YDh1$=AO5FL$<)zf7<wFuA3mR;>@E-DgvLv^^5BI{3se*{=tI#W<D=;%#LlR5|KO}k$|;V1Mbu~^9enSC73(0US)ZR;Xj*1UgQ#ana#+bp-f)QA(b2}LW~B>sZzLrydotoDaJ%$0*a!-m|;{`f-^#h;*26z@TxWv{uF2Op0@;ZXRFuE8nLDNEgi0rrENLgB>FZPye`2vYM=jbrF8WIRxBI6bKb~hqukDq+}pq8^L(8y^bEvucmSKA8=m2C@%&jzL|u*2aj%7{9bCDdz#;yAH{>=2Z^Ac`131P>Q#pc<xw_1#)r>R7yA1B;{4@cq$ep<7JNi5o@jc8aT>Ozq!_wgc@vANzu#=BuWBW0;-OQX*2gxPgJkvO`o=x(A%D9t#)y*>=<nptT%iPIjRPdP+%SPBzs)?>?QB_Q;(g<f!S6ES^iK>awvX!bh;X>K!cg&5lZDh1CN3<FMtxvHgZG_CnQ53GH)B|7WyPg7Sk9yN#`?cTs9QArLB+JI~>fV&u%?A3v#X~ZVlmi)Y?n0EVUV*-sB<X64!Xnf{oP;8B$VzotshBcB!7NF~8Us`bQW1{pN|%738rDivSiu@_p$l5q%t%gLNLeFWP-_%iAz29^E2tQx<LRvu*a`_{F4<}n-JXuL#cGH?72i4(rsU%|!BA{Q`M8n=XInmkttlUY5Av_1da}(HM@%SY8~ATeB-kKhg@Ch^ZeNSSnAWuoGC`x)l?pIpnJY;ltAPd0<VqNkXVgM#qeG%0sS6;>f<lA{9_nge{<_+qU%c{+%4&l+`X;5Z2hOHriI;cE%hlUv%|1a;=Vhz8{P}x7J^S}RS@^39=hV+)XuL<<guv;-0yD-bgtTCSK~ZpHmo3nl(ylbbq^bzQgyEVALM!u(B^Odq$yvcsRY^cO)|KQ$lqFSEP|7LP6{-0U0UsjZGgkBCbBJ%pe5xjQ)zbps%d;u(<%J>sRa|W|V>dOS+4Zf<?*|ZqjGyzn<u49v{@wDwr6C7ndm&O*b}pN4`Y-{Y^_Q-MF1pY5n2vf%enLk_&d6hW-D{_Zt+rrpVujBD?)9Q&i=o-g7UKZa0u>N3CZ$%eq}JO20pN&&q9jZ~T2czAV6v!y1@t0J)SMgL0U%Heo`}<p%<Ltd7hT=_g`HR+Ii7MwJn)W-TTx2hoq}%e0vIGbn2@s`MQ5WROrkI2Va$%`U@tBYo{8gnCK#uPk{NPLa8S__3!DAQDx>owE_8`ijmw%-%`Hfa76d7!C9I9PfNeHxg_u;-98~2%>@g5~JVRoSrKQntf16>(@4qk4fAE2Y9$s?r!=GeyO0leG$zAl<Kl_Pk`~K_de{Wv7_?kTX+WE7uUH;#j7hX4)zw*i}4lg)V)tX<8%KCZ3*}dQe<ARrlz6$4tziKNhIvTexT*wsa{Ms<oXtJ3*_>!KixG#*^sDF0MId}N`QAYc+vo$mN@2>xAKV9JHmw%RD`pZAF+sVxg^J_Cyoc}Rasl4hgY2CnIHI93PD`0w%pmpcG#d0mf2-`Q0GoLfFyVBp3IkBCYP|kR2I%^8jTNV8eW64TFHA~afj5QSYERWJJi+(w*#|mMacH!UIC$XCCkzXO^;2Rx+_SO`186F=*huV$iZnpEK_+q=k&q*q>%^=c3>!a@4|H#Qiy_1maHuL2hd*}>nudNPDvEamZtvQ=dNsSpa^R<|2=%KOi`a-ddl`IWs-GYO+=Hqx&Jnd_m*V<ln^P}#HY>g*;VAmptXQ(y`hg&(D=CU`;hj#gWeg6Hk=gut{oAA6banuJN%%{WW%?4n^+Xo+fZ$y_b7K=}5RO|BW`Nyb4hjUdjhADk1$a%sD<6SPCH#$gWcA265PqXiuTS%*$1iP%IH-?*f&8Y^wMzWGzR*)Md3iA?ZCCdtGT~MYuH>ZHABqF6K0g|N<#%4sAmbDUt<}_$dFEl0EW&#5BeRL30o1-1);(jd5zLz$yN#|$>u(}H=y5<HHP7I`K&y7?K(pluH<d8zZY}6cK2BoMgqKsVD6qbN#QAh|YDX6NoG!1~+z7(Svh{D*wv?|3AYo9p`Yw+~J_HEDQb2H2C`8Lb$*~$Q$Lc-942!PPdCTdv?HxrFp3&1P_f<{F@9rE_X7g58Vj4}l0gx1i|QZ{u~k50ysa>~(}Pn4)$ghch3atuPsP*E1B7FaWc6=YCI#0WMjEU8NKFQd$eS&VARtc<M@EES-zR)kc12!On-0gxl}l#UbsSj+2cQfcEsa&T<>N}U{$Kg2U8F{6DPRoGO~nIlUk+N>dN@r;UWpeWD`Wjb-lDo8YeS{0(uMJ39ztdY>>u+jRokx_u~Qq>p(Rv^w<NpV@!Ocw-{Sk7FgQ+!biVZATdwf#dUI-Pb-NqAyT$tMku{JFVIo?rOupZ&zOUYcU=O)hKSu=ljN^hX|!@J6nEws63DhzlL2+ApG4-m0K)z8UbUe%Fxth3fe2kYLrmAVR+Fy!mF_R2m@6bc5@jPRl#*UCR~%f8V$w{M9%P#6NaK(f3E5Mc&a9yNbhRJQLYR$MPma`1sWSmpQKpD#ZllHA0ACC@ZWLWmO3f78PX$s1YqeEos33gGSK?sHjUJPz6L;QU0QOjnAsH7vA|{e8%q9Rv1UYeiFB_&9S}Yl5EY@)-7Nc<~i%>J#S?8V#nnx_1on?p0!H0oAFC&2MYJ{?hNY`aXX8vy}$V0^YI9~$9^je^r(n>&yL|Yj7G#|{@kUo$M}P{GQ`NR+5E_Rb=M*yY0g3t_KZ|ezcyVUzu$BaTow(Gdx8MC*XBxWV!zX!+3?<#-}o@PAM;nfwfD><Z%cz(-}RlWC5+t`bm?Z?PIk-=koLZ&%~jt)cSQ5KXZ=njcNwRo*7oo{{_M_)#@g}|T@RJI_t)l*Z3@*^vJC;uwB=9+o`~D!`*(yVUy1+t{TZ`W|H?kT-CYnV$`aMav|**Lw1PrPRn?WoHIbzx%xox_GpTCxHfr<oG@u13bVVWIg8g8-3mA5nA9!~GPq@1rVUYCNBhegj!q7u@eQu@q-~7$O-wOu8=!|*ka|=H|d-;O8a`|$(5P%CJtJZ2p+P)nb!7UuvYwwo-_<r826|N8Beb?r^<<o1~WP4usRNV8dfzgcPh+({c%ywR?J2RzK>d;C!8$|jOigeC6dMPCOlNk|+NQGHR83R>WTfi>CA!Y<HN)#+ODFFmnB480_I5P%B4MBw&vO>PlgL`XmZ@q}_t@LS|k9ZV#)2)-}Tgzq?PbyuTsc=N^zld#axUI$^?r(hh-SY3=w6ecPn$I=qrk}fmS3OVCCoV<Tr`5(J!<>yNC2GcWn)_>`-=2wR{d*qXa%-CKHzexLjS>m`tAASmy?>xZJI00_DkCwt+p1HX`(X5!=PsD{V*j>LahII*?%DI|!n@|5bLU3YIU7VWQp3K(E9Wj>3h(;zm5b*`?_Ibul4bSl(d7$%X_~gx+vXp)PmKP~{3_pbGk$6Gex6l1asIN+R#s4CE%?yF3?22h0!?Q7n6coF^AZCS!@&_R(Uo#eB2UmbCrtB(R{sl4sjnWybKT`1Z#03YwZ!nWoN+>VV7R*;yZ<9+obCim&0YVszoeZ4^)w6Q^E8Kxqvkh@rT+q~nC~q~cBVX^E_F-kOK*?<>VnySoL_kK6SL!9x^&h&^XR8(mU@(#&UZAszEP}2g$d<A8@&dTibG+HRZU8QpcX`-5}+CZsAy4?1=W>#y#y3SuYp{b)!-W)e4{_SZ}gMEDb%K5-VP|F>toj_@#6)xV$|W>n`)8~>2<Su^R9Dm=)+is+Tivz_y%i-p7f5^Puv8w*mSnDINg2%uuj~FcCcBRS>&2NW?98x#P;|li2!&u{F0<BFw~SIQ7fcsX`PS?Q&^~~EEQBnv<s!RL=q!X5Lsa=Bx6Pm0hj`L8jxqKb!3m%rv2pwaP+nS2AlP{*sS2hHmkQe1B;uLj<=G)-;d<DljPm|%u}v)N+}*>xd*G?EpPQ_a<uC$n`((ICzscfagjJv4-!1~dW%lh?<n`zI<0-W@|sVG=oW*lIqca<>eQ$0CFF3co9_Bh+4qo<H&d7_PnW(YT>2ix;z@FkUlOx`R1AO`OTlqb)D;&+L39lzfDCe^3Mop7NL3?L07kihy3&{nOiN=lyg*Pa1Czmt1@K9j419gWH%=A)aDEiI3_btKT`$lcMQ-^fF>GVYyMT`33232SEp#g0Jbgh3$<L4Spi`zh+n>IOQK67CvF993+Zn0Nnv40{wmxdKGKv-bdpT{+f=ey<xS=S`V9Fk4Nf0T?tzn8?Nr`$r%mwqUEvk)EHr;;!+3oTqT$`Ss#C4k4Dmj(pXC6TJvq&+qnz$zSowIBwmcFb+U*Mz_H*-_4-QAWpv$~kO0U9_w>`qTMd0)QzkYN7QI}~<16vlX_s$+=4Nb0&OY{Iy)U$C)OM4?KhWB~}*wIqT=1-a%$MTIU^O_VX{h}p@MBtwni%+(k^4WMsGs!sl&AW8jZwCd#NgUAi;Xc?kD?=65J?|Sx~wZPkxXSZb(+_B!K<ztGqi=o$J4kF&MC4tjkvSAvpyFU+Gh{Y#=$gD`bslohX$~HV<mE#XiB=M3}Is#uUL<F2A+6hjn1X7U#Y9p@^G5?nYle$zKb6J;VNmzv|BF!-)#Q`RQ5oj|h7zez>L#^ZK1$%tda6MR|c0W~KZ~1cXl=(G<_4u`shl%ELoRds^9hi7_`Z;*+HozUG+l47o`9SEtp9|!0X}(1?mE}j3UCxhMF=rPXXiS7jh@Ax53_pl*4&^(j(8p%IAP7H}?Cj;zlBr<2#-hL#BZAUWYm6u_l(g*YN<(SXVQl{6wc=XSx<UmjWQ7YTD@_Knv(qhC4-(^K4+rols)jbwn(2ib9qQjVyz=>t7h5?w&rXi5JAN$p?^eV&-_6pE)_n-J>(<{6#JwR}dOxFIA83G%FAy1IMnKGxKrE=jMF|wC5u*i0L{eo0Bds8;g)(X=5rV6_s;SX7iKr!@0@4}^4vZFx27!2b1>#;N{k}=6)-<&ESDA>r)5`_vT;AJ9>zG~X_f2+_-2^`Fs~C3XN7;}PMR`uvzL%{24>QquFfBb3^l4k^9E9fag(lzv;h7Senq#R1qRfa!E+MOhEJckmLA4Z$@T#b&EOAu|%i2_wWkpC?7NDv%BWR#48fc4VjJ9Y!gOk4H_1~<;W6q~Mk||c9oC^M?v*Rt+`SpcHcg-S|^R0FZDZTyU{^l*;Y|CXa3PIZ}#c?}}G)_iK9S?^O@$P#Nk(h_*HT1hj_p+I#E&A5Wel$J`xRO^rthF~Z=cD#O)opK!i+s*!DU}&Zyghu|CC6m#ZpS2>Zv$UwcE&c%dhmz$vSNf?@4A^EWvQt*vRCWQm>0Rb;DWQ`P>8V`OUn~Z;c|00&vE@WMzW8+<vtbk>cFmaIJ*)=V0Kgec_Epmj46UJ!)1X~Srv*}iV==r0jSbUmu1cCl2bwyRftQjv>>cN6&)%VhYH470_d_AUC&ZUw;Y$rJOBCn*WUim-(P&~zdky9B#+6qk@k-kav6Rkdr438lXI4YdNuRY4c|wgD*1mdUU+BpKg^8$`r_Hkj&CHiv`izvvXv>3ha_BQkJ}eA_uA-heZ_`3>l%*i;n&Rhqan-E9lTu{x@`*Xdg!i385c&z6!J!^zifIU0mizIdE8DB^wnr2f0-qljUPkHvD6uL81a11Sm#Gs2)qFtK8$-K9>3P#(_TnQPHQjAKC=qR6EbTwFDv8Kc&B{3ym<D~+h@N=VvW;>mIefPBIM4OL`@Jz7tkfw2w))u0k|fB6{Mz&l9Dl`q^xuSkS<ChHFiXdMAnsQSs;oBj*Wq1;~8>nbSmEWoQyQ4=z~M*+ILffLtzQl@5OTVSlE$j()U^<Q>oPaTJ*1OW>Cs8fh(mNH;J{|VL$rQ#^zCFC(W<!!Y{ln?YoFZ@tGL>@(<QLf1nA=xvq9>=I@!4v69R^LD{?=#H-(ZeB0mA&-1HhmOK^Ojz2K>K^Lvy+A&_=wayzn*6WLneZ9}v*XFlg!_>Pyq}|ukGx}gL9gEI`hDc>I751ET4zGCzIO`gjdo`b!#nG8jI%Vc@6uXT{CSGf3fe$Q(S$Y&Sh*(tWL%K5vRPb`p$2=_rL9-}kkus)46)0#`$f6Xwf)Wx4Fo2TS5JQD)j1ktTX0%ig0m>Msm{f>L!MUu51i(`qm;ELhkOyAWUXOsEnFX5itgnt@0p=t#WB)jvva;b*R{FW1jZod1^BkNZ^YCDl`G7yoOP5-=_MU+4RR6P)VzAxHm-*vm4ql5`I%D@ZYr*H;8rP%Pz#|A9jG~*~^I$)SK(h8O8(H9BE1Ot5#LvC&d?$0E9Au~kEA8RL5heqDJj9TXpNR+yj>Ig*kjtvXs$zu^>-GO(?`@mwIIc9ozfxgGjDR*kW@csO3lW$JwcXRR6YbvacK1eX*a}5vWtBvW1loLQS!*eRl!k=0v{oA}gOn+{DJ~)43Ist6CV~RK01@-e6Z-?~{p8Re*!>Bc@0C@zE(nmINM77-o5rom%Bsq$I(hP(=RA+frCuarfrMU2*16y%#|l?O)LfN}(vnkAmAT$187iu>r0k^RkY6<az$A%$M1F#2NFqPUlh;4~&oEnUB&C!OJK@N=&a!eXA&K#t&r6Y8Z6nFR`jVZI#}2oii)qizZpUu+acSO-IL*0@b1~JqnbPt^U9_f`)|}XXNNh0|b%}3=iSeon`uE(o<A(Ui-mTT{o<Hd5;r4T}{?-!oOof`-tBq85_mo~jE+oFA;+)^!Jhp+S|FJjS>Y`QmoXz(fVYAOoo4Ft#9qI_WJ+6n2TCLpn<{rQ7d2ROHFeSJ1Wx9RNc}u%tjNG8xO1f}VvOC!|H&TeZ=K*fn4`wM+)o1hlM~`Yh7yY!BFkf?;*4viNcbj(_%w=tOU%RjNhh^?}&}!e?Vy<B`?Dluvlbg&ZgomC8D|tV^wC}aUBJTA({_W&`2JNTW95*}M)D4}(RVTRbQSYmJyiRa-zUR>1mi}Fc4OhA+xSbx~ZQR-2#A+Og*V;iJXscoC8(oe4JB*DU_QwbJ)LhRU50kkQE8bgAb0rjoF54sTgb{DH&!yEtLr>E)gxP~*E5C6rezm^O?!gZnR3}_^t8Kd1UB6u~bi242-xo7o%{L7@yGCMs9O$X_)MnBZ+LhtiZYC{R_4hi{`+g-mciNNat)6qHpJhZ%tu<;Af&3tRa5FB`dB0;3X0j&c)@UwoI{F}Iqi0e+iq7n|?^bVSe7@?x_tc0+y^c02Y$HjX-anv~8Q%LLL*1I~S|S=@ScWfjBO!oD2<$A&y2n9n+sy1<l!ltglA#>hvy&3W+Vc-JSxVxkE+E#ts=mohAC5KJt~cCEIFlvA_C+B0NV>fpN<`c``*(ixS7DX>`}8`FMUrX));dnGR)NNse@Q3~R0VL(MUBcDPyhhsl9V|u8KHT>2p~)#3JXLER#t$QIVzMAWljn;;kuY`U8Dh1XE-BC=t_sz;Z~z9WhRL2%mZ0HZuN$z51Ml1LvEfpll^h+PygkQ5C8aI|Mbxxzji;?|McM>AN}dtpZ<5h;ZGm^>DT(ly4`Tp116H$PZvmp9CJKY%CZU<bpfShun;w*xnQ{hiqpI<D{Uziq<SB&l&nZzsZ!LnLM3C0^FkF>Jt;<+6r;R?#V9Xcc-9Co?SzbwDRi*XNuSy6kK3Uazk<>Y%+6MT^NlxNteoq(GM65NP$U?Rv7W0cN=2nWBG<Qxs#<%##8fOoLLldo7Q9rLQHljXh)F7yz}f{XWKF9=56sCZWSH<;zjD0RYlBk`gVrXAKHUyy4N*v6&wgP4HUB-6CCI-f4CGn_D!N#i*KHOX%%wpvKZS=r8Zy;xi@0yDf!c?oc0D-pQIbGeo6cIGeA8ym0SLB3q28c%W*Y>w2X))Ty>l^h5JCL0gN=m1?yl{o+p^PUE@DpCl5W;w!l~J3qf2X%Z=^r?8AiG}$mO86eCRBu9JW#-zL_l0yWU9xjPA;pXL@hKc?Pj=WiZpZ)@9%A<4}7TUjTOrEB=`4A^y4B#M-`SXGcOjKGorEzY^xL+-|Bflh?Otwbg_e<LmIaiRRu;04*!Sk7;DLLqioi-;TH73Q1G$!?mr5%hgvgy#Ihf?r<fh6(Jk(wfA+rn;}}h-Q1|d-h3Dwuayu>ZeP61);{+A>?K*DJ^x^LLYd7VUU~oHX+Q7Hp{VX!@`gP6P6|}VsiNeWLY{WQXvaJe7~hzulDNtxkrd>th9$-X%95C05EXT)7y5v6!fIM5fb&w;ESIPfyskk>IiJKl$1dgxo-yWmV`xUb8xlu8vAP{Ixm&GFV9?x1>-4uH>aw57z01VA6(+I$<T3`c&63^{J6;Quz{6CvKDc}4YU4Ds*HEVi5}$2X78nfs-3a&p(9Q;52B|1t@!(sgawV?@;rh!NYApx?1dbDCQ&p%SCFdN}JSRCW>l)>?REPp8ivmMgkO~q}<g%i;(&kv2|0`&PVSy)PY2yOQherqH+vSzb&u57kyT=Va_CPAX2#MlIS6p7nE_w$R{jd~tD>foC`ldzTKQy$vYwKCNxchcQ=s)xXYptsAXp`4ckXj36-_MaY^8}9GE{9r#%?CJ&^O|23PV|>t(1JNZ3!Kxj%4^jHt4aa^DRM4oMFrFrwFF8DX>?W<Qh-VcVTCEKs{+>o2*erHZ6OmAk)@L%vJ_YS(c5$`>a8dV^aRnKmRH)N;IrP2X<P{kr46b1a)Q{jW^hv1b4(&V5RaArSRqlX3gSAaRZ$g!=OxN>NK3s;2cU!^r=VappiI&-$F-ED)a!e}DdnQV6aVq#`j49dy`L_ZHzUBfhjw@0Rm?a>+=u>mmwk88qxYa)#Rt<fQO9|+ub)ok<`auGHxh<Ihhu&0{NO<oVetN~+x;*tFNxsSttb1PX{DzZ`B`g;{B+dz{9WFQ4R(_9h@F(Z(S@B?WBAzfADc0CGzt28_oovbw}ic7HHjuiH+Uo7qt~BXPSCi6iFbUo3Q>#^#pC1vS5yJQV2v=Vlqvg^1lJr_HHWz<5L6XsgfS$!o>F*Oqe{^#r>u|#5OShbae^rT?k0L=gA~-d>q7|M=(xP(RIKhF8DwxWKo4w?!TmH7#c;|0c0wSN%nuu(z+r#HWYMD(cdebV+)7>Xejnb?HBn;jYUCPr@s68KE$76x9*MynAxg(e_M%dN<t5Fl8d6Y|N+3cDCIQ8@A!(;IFL1$Gk@GqyxKL$PaV#MM03ZeN3!`xZ6B@TyfW{3Rk;d(4q-3Z2Xg1RZ4*vROTFG!H{)iQ+O<rz2c+o3ve_G#mGoH-<ei3zEuLhyjGoTs58xq@iKm*3F1h>`G+29%}3^)g(LXtvJ=Ku)G1jA*C1gAwwii#A9K>;DCF|BbS6#|IW(o}s-{M(Zo4S8%0-~SQU?CENKFeGhxrjZctr>}bO=e5=8r{<UH`oLZPvq6q+C$XFDs1NVcWGf*F=#=0NWR{9Ci9Hw0=qI)MF1V>C9&+HPLOC0!=tE6Kt+cC~16Ws3mY88utE!SEVI|_Q5ZXMeT2!DEMagPVK*4jYs#@e2sYzMcq^xYrG>)D~Po*cF+tXo_kDvGu#%%xLXCiieD;%&Mq8nTNJ;i++fvhmCG;?O&<GQnNuY9iCetYG2K1RJYJ>$0dGnhR|ROMk535}E@>oG*#P&gm{_B!NUExDyTzOXvnj-Pd#(=K<?9q!8AVWzRW_~nogh-8dwOVe|S=5;%qWa~NnM0q7y+IW%|U$-zwCelGd1{3k$A9dQaS!rA+Yf<RHmxgbIFEVDTx2M}oHO(P&&Z%f3T-!lP`x7_s+XG#T8^e3&It2||&h=SNNV8hS4s(uRd*Dwu)$p;{4by>t+n?7q4O`|CK$Fdbdm5WihZ1*~jdL9`8`JpLCK)u}+3|R;Qw7j!8FttxU^W$^jax~U<^Hs<(n*UPwya-gM(j=~^c-%~YA{^t<C&WLb6NGb)`*hy9*{!1n#N5bP}5-wH@M#v0QrwA*)^5C%8oYbYD;)TZyo9`Obb^V6|&@zR@-l{{PqCW?CFGKZ#I5lfzxX~8_DF=z1_9Y-tAa*wBCOE0k>$Ke@`&q>BE&}4D1agnw<@mt9e1D=MLs2O5c=d1A1Gh&8CtwIy2E@EZqyT-A?5m$*@j)Zn>)=ro9y{=T;z=i~Ia`bI2`w`2G0hHR`7JsIc2-FT<{>j%-f^%WNklTb^Ft-2JqtUb}C%-t!9&W44>2-SqhK`QD~}Z}-~^SCPz)4}(c+xAR2v550N1)l|cUW8J-a@VLMGrev<QcP?^6ZKYqSFE|UY#+AUW7q+MGg3&kXdAieqeNWG2-tKnBbklWg4Dg!cSZoHu?3<2j%QasexZzEEwB!dp34eIpZwz&~AvZyE_s)$fjr1lU#4LS3{Y_(=xh&!IYE8lMiAO%Sc02QJwc7ZXWhXHf&1-Tqc}<cQc9(llo<D8hPRc*cdJRwO;H2ZGoGx5_&uRQCvFRL!>4VK{_QqTkgK{QfD^g23hv9KBKyJqAZ`x`Wo<RpglByc}0sA;`9j2U;PdxZW=cqYvnd4R8vJY+7_7->eja|yt&D2IkKg*r#ao-fP8LU=MWB<<Xr|(ze@`d+wKfG4WuDNX3H{49z^*G_Dj$yZfbLATGss>H+xp(6HKEH9!#e&vTM}ON(j+(7E+!lXrYk`GsDZQ`la9>G_WS1`j-fB4GhNk28Zo7RhYgYg*A7D(|?b;1S*8j%m87<&a3|Y0<_0|%fYNM$R4MnfsciV{f>+6T@>y|S9ty_81tvFZAxM|>bffLc0)vcLP^NXJ$XF6{{LrT)%<kv!x>voh*nMny@W7+D~Oeo(rCybY})jHK~KoliKz>&1{cKldp+BVrd26w+3zG}E#wQF9j)$VXcy5&|pw8dK3rrO_1z*&#|i0((XK0ZgoQh3!%wO<Rb%C@tWGkq@pPV!wnnodVH)*(kYZrIS`z?_K3zDPboGQOg490JM}#bqVxlGG){yvQXpv@5jcoXUy{f=X2}R$<9<{X_p`MDjuiK*dB9Zh|H_Mp3xqM-w~>sIkGU*OT$Otv<ZtZMaL6t<1XQSxQ0i$WOSA&3<#X!{oj>6ZrC*$hFI1y%=8E6N@)sV<@G?5zSf7QHf<uSV=_5YE!2x02VnHrlJ>1No6ieE+8gAsgki0*8t%fLomS^oLn63cJNADBn4Rr82Hr2Y>2JICU=GkEli<qthI0v?WLK60SIX!ua8D#UXIZ(0{4eSm0OU47+<&(7P0$YF~Q|a()h!#hcE)p6mLdwKn!3dFx(h#Z<c~S=km=F3VbI5SCHucFN!gO*mA^S$xCYV2`xqd)VzQQ&{8|W1p)%+l@fJcmb@kq2wDq5OT?hmn@Lff1OS0aoKKhtPZ$&7$2LIrna%w#yMkr6-Fo0|%OLFYQB}XyXNA*48r7;tp?zDLYM7+(<921;opz2pW-V_FH|r7?#!HT@i7q#HBi%mNQ4PM;9PPts8;RgdXrucGdq>6#hc9?F4Trf*OmiPQDaqbYa?(zGE$OJ4<bJ}nxOkRZRJoKv3?@EGp4X$U0!lhAdu<ZFo(oRTwi9wvTgiR4RY#JLr5}Xli6F<z$4-aGSCK=nu9T=)p63c+3LvU&fsLSwOTGS56az>L1uI&Xkd~mL6(b6iNMfQFWyNHTb#k%X%Mm!e98x2Dk%aDgQvcmkz5eI5jYv^9XjD9cv|wAGxThw4$#cX-fcKQmOMDU%4=$_tcsh%CAU{WVb*S~Q28zM8-c{LBijawhe|<uM^c1Tkz{d$@&2uRf6-;RBssFBdSr87HUKH^hFi{u4FnUsj^8)5r5{RpU<N`~o^wuY>k|wExN$TKese`o#ryTaVg&NQLk^~1O6^tFdH676s+pV1`nMW(jeZ-T*DwdQt{JvM%3`|hVo*rav$s%sjZ1|o75stpZNb~ez?}hF~?|dbJ?{pjPM(Fg+lQJz~U|a(Gvu3fEpl>4K%JE#+N-Q1<U}?YDa;TH3c&2HS@o6v8x-I-d?}jIDP4?KK5Bjr>Y}Pkvo7Ul`<^3ot+`g5Oy?KPyP<t-{yx!*Q9vrIs)={IoUk?-dAv^viEb81*{T;hUk><&^!#lj;e)>CeNR)pI%k&-F%0axlyIh|=*tXR`qvMWlKpX-7nYJ-%xLa*5bW42GeaHGgO!u-yN9_yu);)i8#)--5wU?a`gI6M}=dUoFWpP1!D9I{MU)s<0tgKHVjpyQ*fi1VvpVawPHAbWmwd6qNu$DF9wBktcszkiPij)8&y;?;omq65@#Hu2dlmMU_SL9{&tIn#ki|1dAS7lFaqOHNbI*Mau%F`~|`ZRsCAu2cWOy`U*w4fcAxsFpLtGC6!oHff+JK`6kOXknz+$}>-X+y`BB2s@V&&6l5xBaHS(1Sr2q59atSEDY+m-^pBSvN-uf6=TM&IkMEPcTLX?xJ|-ZK2M})6yEcSt;I9Z;K0k>pjX|4gP~@j@--Q_Gpv!mN>iMtr06(C18P!Tm#Oj<fI0Oqf%BC%q5o;2uL}Gb*ZhEqKXU31*)L7TD6>n$4)K;;kxyf9=heogW#@vNcT=3&fsA`kynE@&@T7dA<+i6LOsw*Keu;;0xVl-@nc^?(qTjxg#63itY30W{q5^Nd3`z}$nK=v14raPoPY-mbO7djoWKK^Fv*}4QVA%L1f-yfNs2KqNC8EKYpE1gN=X3;U`qcgRZfv&oR)PtF_<S>X<;iN1Yfb-wJlAyt*8a&D%(12G1g3ObirDYO?I^pt2Ic9kT%*_u&qw<)N)|C^;&q7`2XPz9VjpvEA?(}R9FFnRaHX3ks_6nQZS6TU_{}9F;HtOhk-0%k;{TWC`g`jsh74C6>4JWPOhN~YyvhcelK7`I~AJSJyci`O|zb;e)TZES|aeLrp20=wxcmf5eAB}GHper@|<ZCCZH<Ix+>~I=oPQgKEPG2w`rS(qy$Q0s4%N3B}|E;poJt%@QG<V$!Z5)h;YB7YcNPPT|c%<<3ZmV*zr}bc{cv8+|_|gUbo;L1X##M7gQheG>uTqdc3r2jUsI0`ZXo;S0*;`XzN;YiQ~lS&ndzsrwUS0l$ExMis!VXHPWA?;*`LO0o?%SnA1FmykfAXiXdL|TKk5RO3?}2<2!_VG+Sl^JH>L8EAKXOk;#S#r5zl!Tm=nXTb6>K{qi~Fw$sf1Mqnwpl2XUD&+99q8jRhoaE)87IAoGLPo={L_STwX(o@{BIq#T$_o}#`HI#Jr*p(!4zmY&U?0_sM^Tb|C+~>})+tM7%Iy_G)g-t|wFEvbhX674xo`*qz=1LE_N<)s|>vF$}Y5u+*pVZy#5b@P=TkgHNw42G_W>BY_2OsCt79-+b`bb={+0~TO&mrv%7G^ft6drRpc0W~(NZ>)iGZ=K4e8XkMn+W`Ub{dbJ4kA!BuDN!4z6GyhJmM2b-u=B+N0ulgs>`R$VGWs#!=&DduTVqh+gw5N^2geNjds7LleNJ7w3^nf3?f5Wi|456(;YfIC>voh*nFQp-k%)%Yu_SBf@X<!v<Ihb`4^{s-|Hy?#iEn0HzaS##UzwkCYj;uM*^3n(~$Obao5u=>45L`<5s$3JZqiOFv7`eyBo`N8r)TX7q*Ap#J26Fm*V(s+%+Win2n9v??St-CvTXm*NNY5Fj=8<KT6TqZzTh_ocZX2HJhQgtuAcIQU}gB*phxGzmG51QX}MThJQ1LLIZVx|Lsw3J^Y`^aCtD_#|a^i?LQgt*89^M5Lb3|)4e?1e%gn|tG^nr^<khVOPNU{^JwSlw)~eFtYjD%y{-Av8}^|M=8)uuR#MaQaVN^$oll(}*`V+4oB_Sk{^}FvuhoQPe7Q->^^CjaX^eJbx#x39W_M>l+&ptTHQk%(x~|VRXqnOE+epUd-+~6bOH@8^aL(0s&+#7IeSZ{R?m!e7+5cL5<3Pnp>+=o``|b8;v7B%Yc{R|i&4zJ5PGHW#*bB$~CJ?EudQs87iEqabtvC8h9SCVy9S3?_$M*R+oZ^b-hYEM~>*k^ovfOyP8`0Xw>1Cxcs{Y~K4YTDfU*hJ^((vQP8wr*x5D71*<YOC7vQ>9tdy5qpOW_*7jF)}IF5(?s@}NUp*DAh?ESLUmvh{Vzux_>fio{{tp%i|kmA>Oid*cBkEcVbjh8SZLhUJ4!=)9No=>D_q#wXRDFB+odq&6i>Lnp0IWU0#z)|i{roNP7Y(Jgryad`3w?<Zu*kr~s!OTQ0>aJe(1yOVy{8PcnVd1kBr<MNz$y+i10FSmkjqiMU8p6r41%iLevtueb3CbpGeD%~0VAq{WZji&7pPTM}2n^wMPpJqF8mo2fP!?>@9^~5L1ylMc};pnJe{CTR8*1OhPOV@je({Jr_6Yc7!jO?cykKTM+!#N<`KZ&fp&gg}Ub^^?Ulm;+0wvIgeo5Y;yBOI4*kj5K=WV<)A>7$tzp5O;}R;{f=`(OEupp)m_(W~j4GW<OZ5)Wy^^pkj2yYZ^K%hP@n;LbMiC2Wo7))*z{%MQY@+zlVjCqo{#GfMtMjhh?i_h7k)=i))L+-zg+Z#87xqgHR}ixO2;AEt_{8;SVAPK#we+Q&z22)nZp>2l9hj8FKdzB{BkBIC$E#gqWN#s!B!sk)@OQn|tfRuBV@OI(#0AeL8(RRxp~R|w{`B9zvImrTB_T7wfP`g4b(gQtX|U+XSqS2s@Etxkwx`E*k@WcELF`=eIFCw_{w6HYfA<EbwE3U4FT%rJ&{0(5^GpgW|z7+<A7&JFKrPV!R05@T)DXwC@AYlyL|>k{OgGQsO2FX}4CsLUDX3PVDP!i-OfoF`eY<IH-^&u72-Rm#%+zFhd=gG_IppPvt`N?tS1EjI7lJ%3>jyPi1b8~VuiPoMw8Z!iDfk4`Ntp4PqmMfTcjui03XU;gT!o-EafKztS&v=lX3ZPtD>YZ;m5=X+N!Nu+n)-ibC~<}$1?|BXIUU&Oc9|26xGzTbB)oHcWjx&4_i!DT=D-o>-$^ia?I?0fp}^OrAGr}SUpbjiXg-1qkrqN)(<Nc?vnYn)5j%xz5hFQP7iTPvA?JmVIw4<o(*<G^RM6hlkCXu#i2q8sIxy<T0>*9e$PoKLk(l4|jJ_t?@4bENAHvH5MgNLhD$==RUh;ZUh|?9Dy9+YR@vm&R?8w{Q9IRFlfO?+3Y)qFT*Dze_&qZf;F4RQAv4r+exBYT<qNz&`k3CTzGQ%Cna)U(m1ZQOjoaXKNiEZ6E`(WSr*Q1Xkj2vv0PA0CPHyOATHty_BGk<djxSFi;~xF{<kt6gAc}YgJ=XU`#+IxnLLzJ>!<Fs*w^@_c4Loo!C0F<q%0)4#E<95#_Pt(rC`oe`;;5`FjX>_aGKc42H@jM#J!yld16DFxBpQ2qA8GEeMCNhzcVc38}*~!tN%eWTu!3@v$l;69Qsw?x@0u=+y_z85D$Q`&XAZmjFr0h@vHyO0Pf!l#JFTWjU`97Zt=4n|N|<Vi3Xf#%a~%|L+Duy1izJ;JtJkSnksvAG9oEB>f&tcPc7kfIhmXUUzgOMS!_RhbYT%w()mC2x;Hpp7xV;z}-~5ty{t|lbYY#p_k<r9D3kWz&=5=e6+f@iN%E=Cy>Fhvbd7+0#+On!X+@ZY5<W!8(bv_0<5GkhXu*&iV#2rrWhK#iwLapJTJ&gV{zZA7PP%M7sk)D{)Kq1<~GJ?!ku1B?5PP=UwUu$(&DVSn$zztUU=uUxP0mDMU!iBZ-9-gXhW#&rQy-G5uLct1Ztu=mGAZ1_hX2^xXXOgzH#xB@3XGjhCJ}=?2fwXy+|Xlk=WO}(Q>Z3`rMrmu=cToy`YZRXfrq~yyMK<hBaL6Vv?m0+rFEe>3)`^yd$xftCQPKt9_TVkgbe~=&#)6I55J^l+a){`}WG61Vy)!f}b`MxQxq!ItIGnMZ2;1qN70?GvGJ8SKQoA3gNidNx=v8iiU)&QEB|t-PLYF5w;rL|2qk=<F4CuH)2Hh6Rq97Fo^zoO{C`AE5CP#EcM-QBB;C+VaESgD(KcQtB2sx^#H|K+kE0EnLG9awGosj-E-TGH?bUWO)bU8<pif@pGyFnxVP%x!bH8DVmG!D4Z$Ri*t~5GG~#Z&)<yuk`(xxl^eR*i4Q-YnD3*>!xpo3ekun4joCWzT0@=j{@y{3Ew*KJ|$@kmhf;z*`Tv&Yf3`AqaN-L3`iwlKCT@`u;FCd4qqEs?2R4Md~ju5K=*J~>wOO<1wc#avu)YPPxYLc*t37eMQhF=$5sOq!7H0kR!kE5q^H))?cOyb(-pvA&lm-W|o+I_~Eg;T_(mIGJ#Q@{K6;<?J5#H_8de-;-moz+*oU?f1!i%aF($1wG~JA2a`6@Pw{C+c?r=I49zFfI*qaONWf%&$kW<E#O<H>S6Rk-g=J*i0bbob@XQaMu5R@PR|u?FG$_{VctxEN$z~dCRJ6{6+RN$2PM0v+rfIc4FI#PKosbe)c^tL>$~0-EbJ5+GIC;HlsHK)g+*<h57t5llJ+WznT4eP9Vsp^h-WH`(vf8(mQA6*<VQgaNXr~F5=X?;$PlRyUC`0`qQxY!vNFwXsb8gn6|00`EG8=KAd@<O6%F>a%$htOxIq!Hp=Y2@rK{Akt{5?!t&A#Mr_eNVm-9-*w8oLcy@f&Bqd@}BAgZQl}w3vqtnfZ<ZdFvxu75iWu+II$Or)Hb!Y)8DfK^9L2(TvC<q3;&T-CoC31vut$2kbsnmo9bV37qB<$6JQ(C~MCRyslg1UOswP*~31x@Oswh|2RN|uD=HiCuS4+ji73PT-qtcxA4$10bYK#$q;LFnmGSi_{q-Zm$@1HFHhR*NP;@dm7GLE_(!*!3o<zaA<-meTdiMxv#qZ_K2_FaOeE9afSxzMq&M1bwBgSo^b`20oW#j5R>lB8ovzyl$3z65_%wvU?!uO=`b2T%A!r@V1kOHdKbQmuPXhAu?yR$F^>-xg#H%@I&AfBe#ZY2;<n<ft&3eI1f~R()bQleg&>^b2lHi{(4~lGjzI{^epD)`Wu+N+oO-YdbkEdXCEa&+^VkntiL#_HU6#2VmAowXiBHoX8fDMUXL4^NPElMZ5_TEV4KaYjP^2{(+v+mOI#uNv!+TTlmP@t{->dWau9+ZqdH0l>~~2}WGju9wTIM}g|t265VC0LH5_>Lv2gSEX))n)QoH9%Rz6FLJ=!~CIAzcD5^U25v({5(xEuw9T#T<%1D=E}m*d?%d90e4n@g$RIF!LZ(A1|@gYGh(WT|%jAla8J;pDMbvz{>jFLg{ni|sXp%y!~<xmm1Zl@4!j4xMwVjh8t_cqk$DuCohI?f(}PQ8FG}+a+cttpueQ*15nrf;j^<0}`_e)H$uns?h$u6riLyFLNxg0E#h*F=9nNarq}M{}EmOEY%gtY@o)(Dpq{nz@`h{cq2hNxtxrJ$eM-@4QueWwpE|kr`t9GljyxYP9`i<&3UF1#>sjY7Ke<PjH8)+rWgq$uSkUCi6&ANw7^1^vOu(uuw=ZDTyUW(NhJU*$Et#upb7y_tDH(mC4yLp8YvF*NrL_*PtgD4h0Di~pLeF?tE{a9efO|xCN`;{ZNf3vcfAZ5!+(75M`!>3_1V8(Jg5FH)V=l?3_o?x&XU~VDk-N3F@|&ffq_{%PWeJrBg&vCIjX40X<k&YMwsFv*EXgAP%0r&mGesR0@S*dfRq)qBo$MH;e&my&e!~OzSq;3m!l^T{nlAO{<GohoxLc|i;HG+t(#w(1qa(A&~J?X+V^fQ-8)2Q`cazaFKb__m3q8r@_c3@w06fhW!5KadmB3)zK6Nd$;h)8efHA1XDyj({l2`Xw_MOCQJKB-kePfvbIRBz2YGqHzQxH^>prblS7%lp++OL0>b#z{F8T`rnSSKjQ1)DmHhIQHb!h?p9vuTj)L#0Vm&7j@7Z%UI*JC52+hEeFHt4sKfIlAEzwgAtcvaPUfUKqesZH?RNZVUBC4Y$f`G%L?YgJ@Tdjjv6G;6pYS|K01jJL4U+qRC%h{eohNq%&r38K2<rL`;F{kMW}pRc>R8t5wq3C+dIT&AD*>;+?3%t^n!d2jf-?c3D3^pvpXYN}l60t4Qyy_qt}B~d7Q#|@WQb9@tdXs8|HSoENp0^TS@afXPR^ufQHvVA#<md)0+ei-6Fp7v=veGfkRL42fheo%IO<yRxjx$7=J*01eMe~|s~+@gN1&A+B%&(>JJdv-xBzN`P|+&TR<8ML(?cJm2bKBvE9_o<kYu<X6X%b67AJK3d0_vP!h<=gsy%D$QYJ4}~`k8R}C9%e23^{Y2qHYLsie#RTUHn@Dz-j$gNqdyuvg<}K_*_v)uU3@$H>qTw9?eJu;>p4q%9r~uTAH`wn@eD@Uil<m!^pDb|3;OZaix<uZ_wGe|2o*#i&kKQxgo2e&GAU6`6t4j+YMv9l$S6<+D~>CeLp`z_*9@W@i>fZkqzdEYlEH`Yh=-PXgPnOIyN0bNvd>-_F2U%lztBebp|4DH>}w7d8^OkYDtPE4D7|GD{ivCUT~5(0%M--qQR*TnMcQPH(;}&&93WNHA}@1USG9nkfDA%X@-i=xQ8pvI&P!5Dg+z(;5=kM-2{OuO0!<c4lSR_2xJdf*+SjM7$z5xCxt3)9S~9eOm^S1s+pYg1^VHjROV-K6x47%FGcMHb6vw{|HEh1AS3S?>T}HiWHy6$Mc4UZ4h=YCmR*N~eZMBDX`C4<P1JA$O*`8iSD8y)bb)=4-hp)+94+7d`@8FYdG<cz;pH3HZStEVY6kzvv*tDC8e0m#!3$)oe=XzP~-p*#iP_=yUnbrd$qjk(&q+%WnCr*-*m-(ihqj2_5?s2k^eZsA_nl@*@6*DcPj!i%~J{7YUxvCv8;Zh_ekEi;zLo>|8m(&*~`3;*1$M7Rh%5~i*^<$$)cHti$&UNdG+z<6Y0TOdPFxd`5H?yIiz2;;eHm!F;#9%e4pV<!|>g{-I@pbc@DCUr>-e(*BT$<1D;KA(n(>-!(*rr(Eg=Kbe{Y<;-TS=&{o1u@d+2e#0^_!XSTs#PG-Lo36-kgbu@V!8P>`+;Itv&0WzucU4`v}@-$IL4^Rx9vlpr3SeR)>7D;A6Y`77#NdG>!#P+F_)m1Tu#bK_R8A6jFM@jTn+OMp6Nu*P>uK2e2#{qZr7E>p5{f$IbOTTr4*8lR<#oUAJa`(4~Nm&G!8MsZh{jsG6tevV>-M-%Uu3j8eC!m83Q_X*gvsYBz~9Y}k&QXmi6vHq`6<aC#y%ol0u2cVqpo?fM&6sCvMC&%0LlB4Lle(&Z#$ypqTsn+bV%I0nH}>pwv{AW74yFe7hF&tzR867w`&4R%X7slO}lIdm&`XR?G8^-4c=`Se`CFHS}NI(yMcMVT+>4dB%2R<-)@?e4Sys8f+AE^%rG>Ag<RHH`7m&s~l<_eL#eD>!QVAz{=#wd0ZBjZmSvl#70~quF9c!$;b-r$=@7NgzI7&wkn*)|AU#%X(Sgf9-hdJ+f0j2>a804eJq+;P)lm_~Eil58<!fR2C(QIEF7gNocM3>luNaexo%SP2Kx;uQhP>=5BE?<(F6??W8-m56y)VeNXoi>~^~|*X&*dzc%Zdf1m6Pw1`c|tq0~R;QKAZr(ub4!zbLP68;3k;XXa0)7v9D4UPSN{jbk0?uMtmb+8qj?>6t6!mhr=bms(XO|ipTDk^ip34H6hl!($lo0YiA>$OfxO^f)sI+N(^!*$z{?<*GV?DY(rYne2eviw7>Z@jAt(GiE}dl4Try?NOPzY01~-(ok>;0d`{|Ao$H!`c?!7Ho#Q9$}K!4lsSIK~2bm(M>mBgxB4V-{+$ajtxKeLPAr)<ULG^q_+H|x%L~g|8~RbpszYgjl~E<Qv={MZw8|V!)4jKM_Ze>oFtvSAeA#eO**?%tgsXxXMeXr0OON{%i?PG#>kk$dcsoSC+V4yc!inh%mg%on+aFEOt>p=j>Fr!u<2H}-vok@)#mv1RN-zW)=RH=<tw+%8IP4X4K=DMqdv2lM)=G8c{c&Z4yK#4KW-Q?rV{l>cin>^D3m%0%h4P*M6Pz;Z;oim;dnp7P$}9^%u~J?cfk|r-U<)nPS<OUE{dZY2{&a;SOX=#>54QAn~LrH(@<#A9DgIV!7E95^-5a(=U==>u?FL6toT_<r?^;LxR;*xny=^%#=PH{26?OyTuiD?R+={<>2`#JW?iU0|I9~&o11Rr{o>s;LE|X2<hhG)hH%mySAY{Qqx)7i#5n6wi!hNOOe&0eHr-2kidsKh70egup5^db?|I8%uj;5YzmwR|vBrk*7{`WCK{zo*F%$z>QweIW7|e4fl;U~LnGi$}C5xN_o=eQ2$VG{Af(RD$dDhFEuhiMg=bu`jWJ1sXQqc2*ql!`wF|T)N-P6VN={~0RPW|al<NmDO4U5F^V<#!f_bIo*@TOGnc+H-rp(buNO|+t(qgGUG<Bc-P+4xGP2*m43)tJ^0=af}-t`NyFphzhRm?)sO${5Ll3tTZ)R=9#h76mK-#e%^JUhd@L<s47IXA#?cGvMA@;O>6HV5i-FKdrJ~w)nPN$%<;n`Q4i-W;zYVJPI5Rdr7tYUUH6xDrCo=W+kp7o?PH~C~D-$I5h|}$bOU&Jk(i^Yow0tiacgfpX{Lto@Q<4{X^VQL}Tm+eapz$oY0#*xu1Sg(MYE130DJ3-~Ics!&7<+)^$%+kr1)kRv+3X=~jE1EHITmO0;F~1aj^{pwn#VJ|m56yt6?Tmi|%CJY`k5a@YspssuJ;=yEv{l*KjhYVZ{v9pBsBJ=s&8X(xe5tvVTn<GP8}WL)<h%jxZey=UK@ZQW~ohPC&dL==9j|Am~uWsiZNlv;w)BFAWqz_bF_B`G;X2-QGv4yj;FVO1htLSBj-l(j-o`$F1<s;W{iUMZ7Q<^`d-P~!)t4G(J8zwfi=-JbPH{#kgiR|TIFp22!93(A!ziHI!-{>)Rt7qS>VaexJbUwAq33b2b;F!%pv8vJ=eMPY|kdwV9?W;d8{^LpZUjUZE+9#MqY^l!@#H{;(!Cptt)Vq7CT?$R&jA_~yuPFB3Ika3p+TT!gc+YG7AL2tu3=LOsi6UMGvaL+lfB01Q0_;uD+^>(i4x^KOl5{S-qM%kqqzA^XN9sG7I+hrZy`PLm>_f?xR`NU@ddJa$dKWq}g9sVE-LU959yC3@V5M2+T-AK<p7lX~qBTly)P`-D4wy)c!$)t21=+qP86eD|F>FqQWfDXMR4d%FefBro<dikZhCR2N9u4wMiSf3BFCwzTAB{Q^HJCKL`yv)00hjG-`6vwHd<xjC2-}rkCB}GjJcq7)yUa{VByW6K7#MZ0w?_YRb>T&28dTxwK5$gAbAM#T;h_6VJqW~N;A!wcpj)2Mq;I!tL)2c+0Aw?)EDFa+EsEASm^Ac4`C@3MRA)SQNPOb=80{Zi5ft46Cu)>H_R-|Ra@ry1clg!m5JmQd!CWx?UD^DL~vrgRQQ5HfeJ04&WNN;1Is=k7+ryam$f24|Hf0QcDZg;wxh%GqWLD8v6+3L~cybxeds#|@f<h(?|YbhnD2<NEasKiQd!nCBc$jbsTsQ)8)fvbwubxjF1$q33Zu4P>+JP`w#q=%1Dz5ThQhd;Ew@0C>lE1LbsaV->rqsB>o96N4p+6kz{oq!(mDUyew`>Sr#^9S^t+{7?MW2o&d2(NfWAf~zg7chWwP-yQ}yR;Qm+LOhI<ARHd0?d%au)>hynkuR)CdzzL!!@bldL?VPObl@)EKzrYgS?p}1Gf?aa!XfsDC4U1c<Y6RaemG}4N2u#0Vd63Sorzkd+79EF8uh%fAdd0<~Gj}XX6WlWHkeXRF%Y)ekKIxb)9n}DpBW(3WW(MOJ0bam4!CEf-*r-#VP<m&PY*DyxfW8-616J^kV*$%NKOz>auUId^TK@@l{)a+l9=H&0EbH*S5tvHPo+@f3qbc*8=Qfy+v7W(ne>pAQR^-siRo%l^1?<dqXP8FlaWXf;j_+U!TAtpM&@7U%WO@ofKmy;U+kNii=VTsY+EVCJ81v)7F_HRTFK2Rjwc}d5tKSHLiHhWmOqIVv-lycYaZZ8BFvXUa+16cq%=I*L+ps+MT3?I{dipL1(LbubBXDpS(f<y=}E+AzU@jf6zz$r8WkO@A>~ZrHxDWH;ZN}`_W=~Y4O6v?8lWlTb^Ceqx|C^{{6-5@BiKF+3OeI(VH)vy_Ef<zUl1m{jFcjeyFX&Mcwj0E$TU~+5DXE;W>BzpU$4Ytd}8wrS$#gs6d%9I2R4*sh9<D7T^rvGZ^W&^L6{sF1jNUQKQXHqTTk~)kBJt>fNj%L0a9*aGq!X__kPh=VE$xcPjl}BLK0!hFJtNh|eI9{X=E!sk_`Cn&js{_%&DSc`UmC(pp;OzTA@Wd+n(Ps~H4DG0FZHd$GYwxN4q?jX&M?s^#{)HxnJSMkM5_JL}A|UIx#xzl?9Md>-O+n<?qD$z*?f<#$hYqCB1`2N0wuT8`DAaXt8D7Fe0=leNvB>E0ND@sxK$iMzuwtasL;o9=R)>ix5v6eImYhUYp6FYQuFq5ZUXQxvh|hoKTYh~%x^d5Z4s7e5F->>zSrZMq+WSk~BfKm(qqQD*!P8EPIh;UHe;m*L&L>s6Wk741FM)dps9XTf|ot*hJ#2%=c69n?LoyB#k#{SNbHSX-uXBdHDDiGpyqp6=q)Q|1t%{!<2)3;W|vQpMB=LuSYJO*9m`mM|uowq?Jf@n9AoSdR#!s~z}xq}q$in;1RYcGmh~T8f<oEcm^IXl6V6$zQ(ia-2){GCsA7A}{alezZ0@IC8hua)7h%Inaz{GFZDCRXCSA^b+oVV`;DZ2>$b;%>M2|rPrTM=Vb~)R7_{DE1#>(rf7ku`z~t=g6R=AF{OKOe#1Ywm6OW;*Tu`3NqZWE&AG+5>~ee2I(cUoE?1{~^B)TBy`IzF#=8sW7KQ3RT!p>H_tW07*>+yMQ<>b8p5jh5hx_jAGudx6#XZa}OmknwcYNDdp?fc=4f$V>WzZjc_7uyg*B(y$K7N%*WyVB}`D5VLz;O+_)s=qTwlBI*p(ow`e|JCoull7{7Z%T|5ts4*-rvv8{?9)dd6DU@JP5GI{Y1s`ZuTEv|NHE9X`J1Ex~N}@v;QK>_p-lp8Mc4?cR$R2^y44Cp8fZ){qW3KZ$leZ?fZ~tUq+|v$??(dvLn^ALVjCZsNS^yly6rHVvLIg1u$@URM!=eLSfDc0jgvi=Tt9`0oDIu#N_8yP7qMYFad%JKvF@TD==9iPnO6>S|VFxp9-Zs3jADF;5Is^*8{tWM<ab~@uv^MQsk3JGxS+RN_{7(n^(iIT@>%UE!0_gTG}AttQ2o4ZO%k*ZU*GzY~KV^q^J~N5*D>oT;d9=0uV{+n%5Q9D+J0bC<x~mF;v#5&SB1|5Ku@wk#wC%x*pNKE!!pE-5KZCde&_B%da{clCu8V#kq(ET8sKt+e5LZP>p+SsPWSkLq1Q-7{c;zIs+Drhki2+82?yI<A_rjGx>}bvcQy$g?Otd3qo>SSBg<>c5<O=?cc$YDh#yehwFk$$w);M;iT3ElID4-ZDp=KLv!c^%6@X0AzT5{N0HF*PQvl&NeXsj@I0HE9@bG>bJc1)p3ASPz3+joYg4)D^_50o)(=uoT24rK?zwo(&A@SbBLUXm3k;RMcyQmr>R0W5Hlh)3LJ@l@vE!#HOXhCst6q!br$Jr7QP^#S!uT*2o)O^FKrzr_M7R}56L&moLo5!sk<tJ<L4oBM0lO3BP+ZJuJG#`@?07xyRs&2x`o~!;k716-saU=&31zBACYO$(EF`TIAvGXHS&FizQpp+?71NUkkg%$Xs^m-{q68sS$P=vh-8)S*8r}vedC5%}?v1s6Yy$?h`)8uU_lyHw&bWGb`;}&w-uvh3RQ6Y4HgMjszEFp>zgS*2F<1M&EWBl=5$EBV(}rG;Ve+uo=_ZfG3;yFfa2i=kI946k&vxK@@nO}?!2B|ZL@<eLt))D1=Intpp7(+Q-$<I=3bH~8r)#kDvCm_?oPp2WnGQee@ziG;yL+4`Zs^cQ4uqStqkPY{BY{1X%{O1#3$ypl8sO}L{+P;hmoFM`_BqzX9;QgEAb``CRDnGj(Rd1LG_<wX=Kd}7iRyE`si!GtBHuI^Z)@L_we@w*0F;stC>1RTREXw?>pycTaY0L5YZEV#V7Xwtu2BtYit4Jwh*53#Um%_5BwF%nMoZ@BfAy;-E55q-zFhd=gZcSvetteFnr&*d^!Cgg(UWwxhi2c8j(W6rE_3~F0i@&M0yDAhb=yms2Oc`-X*(|YfGzm`8GYKzztrb=ztKKV7~WT+F2{C5)nTWe>rp=>FzN;E2Lo^Y;DbY(;o5xs&qQDFa)qviUi_N4#KCrWGzEdS!v%XKA*<mEoKbtzl*Tx?s&=7^S_zs<#dC%j0;;B%NTU7kngF5QZX{VL8833}d^1wz6_lkm#ZW({3Ddv{4$mwl!GkZNCgt^j$8m~6E5kvVhk@6~hhuJZ0x^$#Am#|Mms9}3Il10YPT~y55@+Bx&Hw>_lIq~A6=<kS6EVn(qC|>8jOwC*7;E2?Vt}a<O65R`yreY80_d$Q*FVadR9I0m2{epfpaDEfpyA_y&brl*sf0l2ogiq^kev)-N-o&5295i1LQT8xiSw45ZH`mcVj%~iOn2Yn5(ZJ0Yc-<&f?!C(hrH@hhodgqYstj`!_^QO&knxLmQ>3i%{lIPwqX;R9!z2ZPag|_1tDyl2wQ4ZmIPy1Re6CiC=0#j2bID^ulY#<Ye`E`5m*(N8K;xd0u?36g;b1}1(>*?C)WkN78k!UbGF*Joez>_yfKhxvOli<>A(CxU;l9}`{RH6<Jun|>Hn_Jv>ntep)McxLa*CSZ^eAsy#N+(vwJrTS_x^@dLqen-&){bho7HE_G&9g?u#^Tr}UC3a@Y~pn|h9!PSCee0BO#<oS=L+{WReJPFTt(j`ERMvlu}*A0t;-f)Z5)1hr6g#f9WR0Lc{MNE9U?ij|_KmCPw9Yf*@taL5!?HO5TP5}g>jmKV%WJz;A(B7^l4*;=9+VyXzT(IVPClwq_44f;zv>C7xY)Q-Y}X_Ecr4}X#w%3SAVbxo*)z`ZHUi*Nqo;u!|$*pj(QP+`QN5(rn=NFp)9sE`6fN)XC9A|=C>Hm3yuHLmjtC`5=rtdiQ|;&MXOGNEcY3{^`Hm5Om?A4a}Z&!TM*Xs2U!gX7Tl#iC798_{*?u5C~I?U8G~jIICZ=%R*jM<~f`db3Kj@hq#pf;(<_oU*oecWwj)n7x;cqvhoT8$UZ&%lSp^0xSegQJ%kAcw{$9{#J%y$r4uDiLY>hNyQ)&vLJ<0r6g6!K`v@#saq(-LV;Rxf}voFL({5O7!{MC!^vgX{&c8Lb|>cLKTC3Ap{Cc9Y~Ki#(4BCDLH|bLft_GVeH20;w>}hwSW3Lr8-Z2uahpfmu>hnHv@|sFDuiEOj+H(nP%nDNDBP+oAHt9@R%p*tl_dlfC`&@>QX+s0Ua*o>NCLrlUdUXaoKnR^Ss;ul7@;jNslJ_5-;TyxWj%na%ptB^Iw{6GJtt}wFjv=uBBx>3Fi5D=f4KLD-(Eg-;Saw(b?JY6^oQRroKhFgo_a@|);+%V+H2O)^tMs|^dyD1{=D{^S>3b#@9o+4O5YgASsy{R$8SU+mHu~h7qfjGxcdq5n6<0cMNSC@_x+7&=g+~qOt-p^_wCJhugmqSZ|vLa|C;^B3;GrNi5}E<46WABz88=E+4r)k3qo73g-g@f5A2?ieVMlB7Z(?_AN=;E=~LN{-LumN`5yG^bl$uj`kgtY8;2VjlKk;|LeYy4K<&HGb0nN;6d64r`iei~tCi;2uLcTo^We|;?f%{lxRtC?F7++GN=Sy9h}g~ecc4Sjleatnd>Z9R2-*Fv)@{tRZeV8gN#@frOGw=P_C|fRd-XF871Nx=4@E5Oac}0T-@Ihgsaru^bl-plFTGzayg&RTKlorK?9`L=Th+z07bh6Nr^Wy>UJyFI#p~KV;E-S{n3N@>9Mig>HB_Qvg4Pt|q*6syFfMY%P*KZ5z?x8m2(M|O_+(*pa<j23Nl|00505<O7O=eOgzVZdJ_G%G@aMJ7EWr2fxGo+fHh(8szj)+ISK2pHCaoumeEXw9{GmW{FVZTRL%#~ylciS$rSg&^2Z59G<p7RTwiGeMu;Rps2EjrxMsrx_xX5Wq^I8gC)S#*;6FH)sAOmBO0`QvX#Z*bf#7Lf8BROcqUo9@q`gaZYYm9Xd`T|9VR&8_j>>ER&qX*#(o-^>4walSZ+*-?-PI<bAyToSR%U=(!d-R+`EqGG;(gLESY>C@Zy)_G@OE=SU_NL^0*=xNeK*~W}T67B@Coq;H1-A$+@RNdDua;rX3nDRN4A!tH6e<d7klP9np#%`TR#+5eNy-unRsvbW66CO|u>wf@xoBdTPi|QMP9Or`kH|q6mfsE7=}-Esax}KXar!i}upA3yHUo3VW{MN+F`lnV&GgG+%W%QQ2^TGMCF`Q7G2+^o(Tc*HBPL6vD5R<?P*qmkkSj4n04_xd0nsidst71qS*x*w<b?+{>tE?v^ZLGMM4q)W3(84{i-?X3w&y7Ucv-|RImn$s;JqS<J9GbUrtqs_OZzCSvp+KS^Hf;eZ+Mz*Z(P?rInt>l*6JyUteI{#{B8{8Sc$P8LqTAXb3DGdy}Y7A6=jtpQG+7Sr6LGP$Z*YgDFEfHWK{n**M=Hs4--fvKtWkiu%hNC0upUL_K5|4)huxQ^bLTqD_gL3!=^2RGVt1!Y-%<HZH>MC_R4RjBbrQAwH`0467bWsBi$&VQ&z&y$05<T|At2tcC(7p+05{s)%(*~pR~eOmQ+a&I+6_EMA3tfTXz$xgM1S53HKhNG*elk{_si2>20S-#c5M&p8aadHgR2SHmOfn`?%v50tPa<(sxro=w9yHl{wdJ*Jl8mv+u?zyGN;ikp1x7qJFK+zb2<^wX)wmyPy`|)&FzuTvnblt2QI}qhG<x=Pq6Juj-}C7Z$Sj7B6Q~l<yef1p8%G2D+{Pr|f%&o}~W#)TzR;g2Fohrm8N!o&EKqwuTmZn!T=P0KHPuH=X?`4pYxOd+@tCzOyb?iU5K^sh2sREORr-f?QBS${Ly4dqg2au%=MTywGbN&LP5#=T%wIyqcipdjLvduDZ1NPPK3(bdt6qB2JKA@!QOGm)_Q0yU363V9rc|7xh48esDYv>AZL^6YA%em4U5S&Lr8-C6iSt^jBz4)<!tFTg=pAbun|b{uadjd?=%WhWU2{Ub^%@K00f#OADti>3<HyEZugvqOFKmS}}heh*27G$k9<rE|_jmN!OCh#9$;6$+tJ?qRPeX0urfpyxspxV}2IiYJrdZL8A~yen=yV$Db~w2Danu{8h{#_?fevTX`btD0bXjvzua9V!~nB<|(`kRm?tYB>GNX6fEU<I27Edb*(e$)%4ulOq5nzX#iOV;f`EA;5RmnGort1y?dS#h2$S;3O&*`?VPX=|B&dT<8UZmt(uKP0gx~&a`r0LVgzG5T#Qkei5x-<7)7k&fJjlGs;DKXwFdxdfG9;6m82?jA&JCQp;zFTqJn@N;z`!##HQ0dce<Wu*Sg&4Ei}?{!jia>CYpEca%?p);(Ff1>j|k?qs-b^_EfBJD;3n6V1|w@xQJn1j8kfwP|l^QDCQNeF~&J0v=mi=Ye;}(TtSXXC5b37B^VN@+X{-Lq8w;jH!*T2H#L2|$$SLbu6x#8ZAZdar)2sdC4hDMxDLIuJCwKf+Lil(POL`}<AMIqP*VGDu#-0ejqRN_<g$s!r8RI7a@i)Q+4ATfm+kg8I|M!^ru6X1ZvrS8r{p(LBxf~LP}Bm$k}$;>k~L+tBwR5dA<bn8iVAWH7&lxBWhq%96=4AIiH6#Q75s=AYL5e#ms?nGMvTtLY{itdwA&#GupIF``tNB+vUoKt^&bU#??FT|ZUzf^HC4<RG&v5d9sMY1aSti1xn=!_78So?ds$5q=WXrhTh`6p4L1^2@!_vEuP6T1%8Sk-jAO~sNGafFOc1o**ercz$3i+zk~+^B0u==X!Fv5KP$6n4b1a2KfR?NvN-)U~%tgU!Koul8mw@I3>M3KA3YZY*9ELb&ZT*yUhldDsVqASCvE*O1apz{!7uI^OW~aME<$aynY{J&JSAN@}gmh8Xtys&No$*auP$x&t>o%`DuJdMtn{d|M#ipBGB;=6(bP{)+kmNjxB*(p8rDw@0=va~DP{w76wSz6S*Ug}|#ERoem7G_Kl^j&LVpP$hW`q?0E0{|qX^nV+Bw-T+Jjul#XD)U)yVXslS(nIp%B!dQvSDch{ll{82DM`!@O{~s#a^lk`Dd_~;qk^c!DNzGQcx${jxj{~{0m{?LCjfBZ2~z%JSPMegaS%QAu3vH6N*5M^e@GWy41FlDFBN~AP5z%1QUQ1ctS6Bf`h%AiMX!|%5FMqb=_JoFRtUbM6Q^a!N`iF!L+NF482%O_gSo|d>l4k2`T)oZm{=Z2x+v0D<(nP<56<Jlt462L0eiYRB%xfR8<u$xhyECE69j6<l%xNf<Rf;9232!rvM=!0IusCvO+@6PYQ{+xgy>C(kwXGT7n6S_zOYQgCnzuTalf$y8(XiNkE5h>P7rdhj&{%yl>%tj)uesO5BLxTAqvGudP66vgCq+bKTPY-L!Jvg3m%gcsHoF@3m3wEg_;UPpo1C(LJYZs8d{c;YA0J4|K~Xp!DO2IrqOePcezTB5-sDq}1~jA-SL&N|;lQs<M`PR^t%Lve45UP$VxH=O9<0WUN+@)Dl357@n|YPS`S!OPx4{S)?%BU;Roh%ntAK%EW_6bd}=RoN&_4T=S=mWq;^s-a{VAaK6l$naP(W&4#5zQyVoKmkg_s;owsji}4-Lj_SE-c{2#w*&w@ztb7q?qLn}w=GPyOQ{A~n;(jh(ii27E)=`pfuY0Bz&x}K~=*<siowi<b!A-XI_kpKo5U9pntF@&$x7N}Z@-Q+@HkTMK!Y;v<eLlcGiH3WuGYWSj6`2FLH)4J_<H$S=57+$o(t(2_2b$GEwLhXVl;`AG4r@Ap3x16^mqnh60Nc6hu6@bv`6iIC-L==T?DmK1?MUyLxDSr~`m^Q^1`#u<Z*?x7{&B=?`omWPVfI${fw!LbO3xr2G>**xb_Z-V!0BDK-EN+1?mlLG0_)uBW_KV%IMUwkLpvOv;OKg{;G%U858^dE;Zw|vQkWMRj|#W;%DQ{95J0%vdc*v^To1*VeczL7@>M!$zUZNUXkE9P(dBIYxD{@q^)aj@A64==K8;T>d{yS9viYv-!Gj%sz*+brH`)*ER`A5vv%n3z9B<DLbQU0N%R`@=Lku@BLwLxKLnbhe?CSPx%Y1Z4N&o%T%Jf{U{)m7N<KNG^yWMwR>{a(R?1%GckNT%SZE9LQ(N~xE@QhqI;}VNqtJAvJ-ISfaXD)f;jR`00Q<h}FoM8sWm=u=b9B4PVsA<JOu6<v?sNlI&Ig$YJx~vJ$VI>5{uogu{7=!>3%qz$t&L`pQlN-)nPLgzY9o{$#QHjm?BgUSbZ#1y8ge{bB0?*wz_eK_stn+NlOwk0tA570gH|TLtA6f~n^@cT&2I0J#1q8I0$neUcjs@&)D6>x*OlOHO^3pS~5Kdy?dW}p$K34J?oa?!ONl+G*;*{o;=Y=9rA}aL11my)ofKw<7fuI686C73;Q)L8**o1-bM4QTCzUxrc5y(#yvw16N?hMvfW=G3Ro@oSz26j=>U+JNDtPNFqtS1$4%V~LYtLisvU-z<KcTRMMj#|M1IL6~-2nCaz6_sEW0D{(%!?G^wf|61Xu|Yj3!peeSfT-R_Ap*2!94o=<LjT{y5T0l%)*Hd;dXP2b8a}bM6JfS@gYUOBZ1y&z-?koXY0LOE-lG1`>U5i(!5?`yM%tW3vxHOTZy)|^&X}0vT^p^5e(#lu#r$r|hjUjxjPiV(%xQ^@X_OVKNm&S%8{|YO2ogXkp|4S4qD^RBumT|MR_eRYYrA4cHOv8-nA4N)S3Zj6X|pB(+HCTq)cv00+CkEk4~Kll<%F8(UWysJ7LpaN{w;wVOf24^7t0g_K2}CeA%=K~AW}SMoH8JhEJ|ptm#PuuNYotiniP5m%o!`R^%G1mh(skO+4V_w{n)bWO=K`BDhO0=8zF!g^{hM*-1obPtBs&g)gWrWbgrt~c_&T0xi&XrlGarL5Bk%vjSz}6r!g736};MqhZRPj<Zy<qYk>(SW1PaNNSRl%;F6=Vf)G|5z#IdhJ?m03$T_Nbi2<u9&J`>QSkWAnWeE{d7{iJGcycAHZYNBGo-}#K*rci83R-2V*>pWU>=|B5R>-$P_QVTdZKtB<_nIWO%~Lzcs?S<7SGHY5EpIy%QSdD7Q667$G$FqeC=2xz#9VO9#QZ8zW5}UWj8qucxl$!o0AO0791#Hufdvqy01OI<a1KlqGZ$6Kc#aG8(pbi~ss;UBp9|w>TK_^kSF_k_4Gjoe8zEz>j0@@PqI+YlGuYH2*!?qEcpqjQ-hajs`P;8FyY${aSEsVS3P#Vll=?zlHR{Fka+Qh9elH7e8EdKZrJ3@=OR?8uu#xsUosC?);6J|6*79+JreBX3rJaPab}Qi|3=zSoO4=|AZrCX3?*rWMhQsi5P<#5v?90D?Zrr=G3;Lxg&t1M~3eCSmb(7u#EM``iFQ+Ms#v-1=ejM7WX|wj0c@_1!-qhyUnSwV>8IAppm9=TLwo|pfxjIKmyKqFia2NuHfYLvT0+)#8+M-pI7C9~ffIJth$WcugrJ#CgY?_Sa3i_G9b@t-9P#5t0j9Od`YdfaRf-8#`zdU>X^7*VjyHEvB?(f6}-RQNz+`Q)5?f&*Pv*gyR@$>prr^A7NJC`ZXhoXnXRNEV~wP5zmWM|LonMwP5wiLsxXuRX+d_i9Ta}0}QJ%W0){UyC=?xoj&weRQ8YhPzmkxg|v^F*q)7%6LDlh$G-{3Qm9=<ulb4rQL4;l605>htll(f9S{VqNP`%@p&vD;y6uEUCf~|66rQF9S|zQk@g;UhFYcKl<C(v)A=X0s)9;!uyb+aT2>L0pg{^vSc+=u&!CIM9!+BW_s*lRTCjlMYP!xwdArwg>G!?=?SP|0VlHaCpf*~;N(kg4f*k`Tbw4##ldi5<IzYwjN_AD^~LkMX-uxQoLvcVzI0*gV}L8Zq&^DH+rZtBuJUc$Y~W01?9U+<ml8bw*07|#E9zN`MV^txbt6L7_mhOnC+?cBw{)hxlHOih)VXz<N1p4osUT{yIUibBWh?33sx@dT2qvu9M{!EA_P+=nFVikfRV2s(pg?g_Nnp&lHro`6TmVG%5?e?}Bq$|ULJ1C-@k1+#E6gUQ{X_$S8f19Zpd!7k9JJFlw;Cu=SC}MAvp&v@oyq>QsW$(~rRwa0tS-D&U6}68fIo=UX`8MZsR??nd4iRA1)DLUzTqvTF`%nHJ7a3PRz~VVuZNi;y3i9%=;sy41TK(S@M5eCErx{RJVyjFS;~@D6{fi`viGbg09TYCgz_q{F|Bcxvm8*Q*ZjN`Wg*Lnp*^{V_MY>EQzvmXDKl)cF)h*hJKno%rK@-P2&tM{ok0pZUmr>nZ6wG@S1$DU35-)6x4Kb;dDJMvv`RqzLLo@Ltfw%gTh4_JQdxTm%-&L~W>bIl4}a}1dMgX#=EA&pqg%h!N>-UA!e$3=)#J0THnLkyx@{wddbiWk?Jp*7^-<kwjJ0hXD{HT9I8(S}CVhnqUa?$Mpe``XNd>D4f(i&lnGzLZ43(IdJXb0&5Y=w=1YduG6Z$KjzM-LF*!0Q#4Qo6%XTy(mF9Fj^rw`j~M~vW=mmNy9A+26%DZWHi%#=fDGe#hmeRq?L{joT@#5%e;q2rXXF9E|q5mA-}A`}22NyX|aFRGg5vVxM8SP^KX0U0PTu9&Qgyv`LV5EKQRz(yyq(Idh}4^RQ_6#)*w(Q5(E^x14=&71A(Y2Lw1M~n0toa%h`eM<;(Elt8%NA*!4<Z-R{-gu*-b_(rGi~Ej_B%H8G*tzb1^(&!%Zt_e2Qq3Bg@V6E(yyvdav~F0SI@dn|ojxV#ltGe@FXprasw@!!kaNr-E0t7Au>w{lWf-9xv&te)YgTBtl>#hk%BcjXRM_PCP6{8a=SZD}2b*{}p@V%Pbg=NKbg*k*pGqZZUC3l7sKe}f9qk4swi`%BV$)#Iy`a}jHZ0r~Q*{qLPm1khSV3ktxY<kHACKbyG-t}@lUAPT&u0i7JDj6a@w8yckLw}#_eo?uTMluYCVDjTl*<WEOZ<C(E|ROb!2h}j(Ixj{)2H*TpUL00hX#of11-t#sdR%ww>06K>4UC0>Bl~l<fmgz-0+`|pOiOZ8Qh-x@tJ2f3-SK_kmN9_gIgiYwKaXvDBNw2!rf#P?zTpu+d~%eCEINY`zJoU_KP^5_OqAvIpU<N?teTzorezj;oJ%Xx%H%nzZ~z-zdp?k$G=Bi!J~+)G!Lo$T*Hm)CqLh68YK39KkCX7Rll&l4IhT5z_UGGPiXy`UB)E~;XqeA^r0iJTS@Xnd+ne5GY-TM_$bXEcAI+;UnURbOTT9;brIV??f#oVO>Qswd*6&w_+^Hp*<&e|mVL3Ozp3=6?eufo+~wWk-?r26%XHf8v7I{k+N<G8)1UUxWP_KZZT8y^ZYRW0@o#%Qzfa%1_Rm9y9gbQ5Wvm#cpFtzWO|5N7Er_26q5EpEOP3HJToT0@A`(zZiku>)Kt+Hcs#Hjnh$8JUGtQXMu5ncr6`>q!Cs`3R;nh3QGBIWj?Fz&UocEX}pmtl+cu#Kb;S-2b(z>qUgk8TxjGj+_(~!7Mlwpo3oC|F@x8V6$VSgd1Yh2Vh%4rQyUV=g>sA$ckg!*4r5JUlkN`f3gAQY~uikGY|7~&*{6WN%FY|N`E8?)lWyn&@zJHUrBkya^Je_Gy19Hla1tXxeq#0x)Pd=H)e%Y`5R_;3EH$8I8yF&$fZk`hWTmCQ>aa#{*Taa|OYGJzQe1y-2xoFK_K6jbIFlAI}^Jxfd(#*kGL2lC{C0HW@LlPd590(T+`Y}=IQciqKx=l-oHD6Q=j(DkT?Mbt+H_LET9Udop;5+~b_1sfzF*_{@&zy)D^zAcg7_TjZX`*XQlhT-(K9t40xWEN<|9yS^uwwdj=ov%BqFCa@|pPSwnp5Pab#(rAh@q#EIAP|NW>l>>D;uKS)7x9v?yn+f86+|U1B?G+HE^TdzQ6(y?GFJjHMkhY)#HW2VecDF}ZB<uQ@aUDm6A$;v9xj56jIW1_AZ3c+x-MbC%TneTV5KM&qJpf}e**)66->LpB{AHnxP%-sZDP5kBRt%f0PFQ6yZIc@UM=hAz1Ry?efF2eN~Cly@9G!JnwoQmL40}emRNY}vi>qpyAM2=H}4p`kiI|lyKgU^tK3P9C!GDWxNzyLzTyRAknPg;7(8HiXK#9Q<Ik@LrcX;W^-|1LIb}2J)ET!irZAN5a>2gMM%1;{<^cUx3j??oe93KV54Sy2U^tb<RXNyy7W=eG!L@;lrd|yWW@kFGt-p*#WZ{yh8d<i4YRe{Fy$<b4sA-ElGQRQJ9iQTCWV$Q?Y{>a117m;R%|hbd%1!CR*REL2zb4CmB~&-;OuL)<+5s3o3Tf{RcYQla*x5}YYf*l-Ip;>KnC?t8J_DQ1-#6pmW=L9B@48(#rF9;==hevE$z0!Gx(Ih<<gpOv*;drs9NDUv3H2(r|63k*=NCa+b^v}g6=FX-Zn@=e$z}d`e1_M;I^5v8Te2kO<{o|!Z(=T+>a0EjL*?4t7Nlf4`(QlPc@LWD-2|sk<#2mpw$;|uobFf<fg?^~(1FiGez)}ozY@s77&&qS<n6L+pgpFy41Chf^Hot{#X!Xiz$7NTl%nELFOLeTXenw0D#8FPfuI<(8WKQ}tRWTzkqJNL$xemt1=f#EXGyQSId;=y?E>0mxWL;*uGT$c<!(T7e~~cpw!|@4hABC|ayEQ3q*}LQrDS&@G?NmHJZQ=jL!WMEfYil$$%OQyzhB$Y!<3+75coS9RJgmn2^Z!urDB;)#TvD*aZ1JJsNy+>kYTAx{cwOZT8}v<0$RLwQQ|@ZRmi&53o&hzC7}e0s)n>cM7*%o7JTVeTTN|Eu*sJe->DXkR9K@OG!a)=V&T%_!h7dUFzC|Tx@%{>bq8}MMs!gRROScAm(ZLS?`1;${IW93p33=q_G8FoRSLsFXQ}O4#X0U4GqtFfcaCGz`fSHkebZT^`8yh2cM}{~U%~R7gpXdX0uy8Yo;EIoA;GVNi>P0m%Z3`%1}ZCcCcGEslh@~82tGDq&PGW9^SsDwOaPHkTPj%tN^2niCA`Y1-bEm)7@-wpM3yyWs;GgqfD-kx+NeqO-*d12gU4U}7vdzoUe7JVAGtkiYts#E+}BE6joM*e&!K^w8~<<Gn!GR5T7b<|(Yv+NF$Tr*J_F!8lwNr7h(C<J>aZ60Av@|GINRQ2LL(Dh@uw~iMETR{{k*_czWb(zm-iT;nv_@|oFl~;s;gWOM#>Ua6y!yvut|}Y6mu%EETG^etx;X*f2uMsCXVIAu{>7C(wVmPuzp@?Gp{YT!mg*Y;r6WtUO>%8Z13AT8e^z$I#_iU=q*-~0`I#~`(ot2O^busN>w-ZJ6jKBSoK#?3GEP{U3iMhlvl$Weuk2h@T$Z%q3nbe<LEo<rF}`tnh~M><w{YC1Y$KQ@|=r;2m@7tpwz2*QUS_Hf$ADT{XDB27DP_g_Y*UGoM!k*o^=ngM&@WL{-?itUAKMzv^;xi;Saz4`DuOBYp=a#d5U~4PXEllz4DnioU=A0YJ;P;-9}O;I~Z;{>)X1^IEa?RUyrtJ=6YNY?|=Jin-jFjOnqhFUjNq-$-STXndoQV8_FL~XFtf&eoTWw4C6Me(=Mah$vsCcJ3IaK9cGTQBk^&_&F;DZ)*H{n{su9iyczgMuZFe8Ru-_DmV(5Qip<ADciM%fKDE7KE7QXgY5s{P3iZA_iCoH7Ts(adv``wHsPCoWp>_qL+YRr@d|T<ccg7Fx1|cEsGQE2<E*-qu-PYp1@-?Tn`<WPvcO2QjONx27+Cy+SUnCs!+gP(4-o{5W&Y{?~@7Z#5{?(`tIFtG39S@Av=&sjmVdP_Zx82-j(><f#_)G6s3-7;uS)1gum)`r}gPE}D{G!rk;=MzX!X(bgNyz{oN{kTUSH%D5q<4eK?^ys6LKUEdpqxp{3(6FOQV;+YC9)Dsl!_3B%A8`PL@o1@1HD@02vGp)Nq+a_=6BZ;cuNG{tOr$8W8Uuu%exslP(13Uw}N5QP7RHYwFTC=%&wbaewXlxdx3^PK^v$Y+&pxMQrLsIYbQ0aM<@}X5QA~bwHA57N?B?PS656{dfg>+z(_@Ns?DE}3ZeoMMoFocZFx}(j0>9A6~qNB!36Pgf@`-{1FG{zB-C*b`wh?Is;%Q@<Z*pq?cAm@a*k}p)#X2Sj^#>VTnhHgZQgRBtbspMYJ_`@pR;H;Q)<^NJu3CnZ#VQa6Ew<k3K{@L={QwU6$P)fg@mfCAZI*h2os@bt#ZT^&=!;um6sBh7>j}^MFHlVRk=P!aFtK0s3uiZuVNKdry5};iEli#p6#6xH3>Z}^_J1a2~(-W&Z+4XYIrx25^mWvv0MG$bk1<UTJ2I20`c*lN3m=m=BUFpCSj{@><qKv!%)_*@4M%bB^Kc{y|?cttL}}p&ep>ccP)t$4FW%#2P55Z`S0g(nQl(K8x(@=sfJFSTupou`_Fp-S5uLTCyn0l(5-!#yau0q2P4^i;NfhmOa0f=8t+ue)POrO>q4#mcc=MbI*!VBo_2FR=!-ZGMOPf7?=sxJ<K+A>FfxC)uh3xBH4Kh;ukH+1;>6@Bm*R*2I7lD*ptJEiH;>Ru7^@?c#!R=y;RkUb<{YPI^IhC>uW3>Y^TKOkjw9|lR)wz5S{T#wL5y!rY(c30e_YgD*05BpBB*A(f}$=BESu7@5SCGp7E+e9#<bM!fT{|@YcYu&oZwjMr>7iY@BLu4mt)zCmrfiKK-@Tp^JUX>0*+S7J$h8M%h0m!8GW1V+IB33iHnNmsPq*HfN#gCruS@Hu>j|*l;Sy1;!Z-d#}$=;pn&5PxTUD73PGh%M8F!cx~wEA3dpp*<yh7gqgW9J5aA-{63Sc}jT$8ir3hyeS%ZnJ!K)~15X1{CrNJ4uMoHv(UC4O4D`e1G$ILY=n4P3<b15z_ml}^hN+qnjDWNCk&^pm>TuUTfuYR6JBOGA<mZt+Z>!0O_knu4LPw^i6x)x@`N6lhN|C`0jYOXdQUi>7oly+#@0G!mo<B(MxI3D>1^3hQ4dQi`Lz?#|tPa1n=ftisVJb{kF&ek?%;3mi4b$~VNhQ`5Og7c+s>h>P{a??8iXH2}Q*{l2i_R4Q&(j)fc=AHnQds9+rqBZ}RrqXVmzVcvp)P1&2xE%u%p9eis!$c7z<;))ue{(-p2QP=Bj875`f=#<V_1CwR$Rijh#y*{~BvY6yDU&5-{~I^HWb{;)xFXm9gh{DAmx@rN|HD9`7nKqNpeO}Jp-L<jDzziSE5lnyL@sDfCdrmbvSl2SEv*@PM4DwafvGx&;J{hVb$K_eSp&(?r=H6}PbS~qymvs%cPr%J9eLCJESZn)N9ShMW%Qf=HzE&3#NsZG5^LQ`-k5Jf*2kA8Zh0)ROFSGgKx?k*2SJ)(5M+@Kp@B91!DNaZ`zf~dv5jxMmO)-X4C_LWyoMmBkT3#KjkuDvq!=ScDh>)!S62QYm$a&=&{Hna^D&ylYfmz_k(pt36DM>t798%IV(`Eb)XUiI8IBCy-UEjZ3nuq!q7dVfn2pHG+H}?_5s%un{n)SNBsA=>WOseXlOXCd2T^kyM8z;C;}k>XMTt19v_WLKFeq1vb0cY0ARs7`03ZT*B}GvHfRzx4qZ(FK&IwYWloOM9a>1{U9b13agmt|2*-W%lj3KmG|2v5g;8IHX_H{PvVG1tHw;aT;ZbaED$JM6){n}W>FG5IbDI#8bE#ws!JK9c)7x$9{^>U<D{p^^k112H@wn89hoaJK$f8{k0wUGw|0@sYHoHBwjl)3gBx#Zf3B-%%WQWb>&T<JDKaIE)nE`fY`YNp<*7W5~6E{vaP{R{D2E#$)G^Ob%HPCEjjbFC;tW4+pz(X98hxvI<>(!B>WmWU|EM}Bcx=ok9Zd&Ur|>cWh+gl8{ZoXPCdzc^!;pLW%D42IC(z?+K~-t_k{+90Ody-|U6+XeSOjX9NpH)!8p`Ai!${fBc3uet!hdK6`Gw#of<JEli0^@InD-ia}XuNuMTiLE*oZx35<FE2P5CqpI#%eCKDa7uGQYX%`y+C(8ym82~4lGc*w^{#?kFMmr^$Qof*m$HOH0a8r7y_4(h?Ra!+kfe`@qMd-=8AOVl%@C2@9l|m><F^^e?0ul}ZgdF0Zg5Op2H+IHv=m@NE-~FKdmH%hRwAN0v{Jt2GsV7w^^@QLu6JmDZl{#R&C2{yi!=Di{Y|&@vBDrGFsu;9_Rj2D$dm5HZ184?7kgk=gPL-{U(Zsv8iw@IWF|PCnSlF}E&$j#O$D+<+Rzs~$AFZQ0wD6L!W1FJ^oF_ySa3)YU^&laSy3TDP9#)7Paz!E6Frv`qUQphNzX-F^HXm9A4CAXUB2a+er^Y(jxT+PWi0#c31BsA*xy<!{GonbRN~t!zfCOhN<tT~+%C1y=KG94X4dvIm}!1)*l5F6vSl~gbVuIv>=~>H8QhTJ^)H!wyGqXl_ll4xKb<u$b5=1~N?sa@H&O^#%TiS}5RlhJu5yJr!H^bANl7IuD<V}@7>RDqW%;sN;|Z7YbLUb9Psyd+L--c?-nwkUo?J~X%ntAS;DhY@&J1r`JN|hJCf#fy`R$_Ni$HtOIL|Is^>It;N_X&^78)>MKb^gQXkGkUk+gXwFpwoGS_7-g6qXB{v5euGoe6G)Lb!%RX16WK<<Y4Q&=ZO3R-5||m*nQ7hSSXffj$b#PytP`lJ+>8z2>JOFF3Xw&?Z;2uyrS%VGz{0mh5O4I<(j3vXnosk=?o#0^xCK(S8QuKDS<*izD`Bx`5NWW?kQQ<rPpTgF`)1v+jG%;8PC6TP+vPxaYnaIpf?(9{>OBy=!w_$9X3DS9aMSELs57_lsEkps_Vcs$$3EICbh&tYY<5;+O;-UXHA>R7FW8!YW0hR1}yQnG;$=7=S<!<lzSj{6c;{bIy-&>6e`D)tA+4?+p?JC<_FZ;)vo}z3#o=elO4aJXMN)nlEoIN%lx=y-$8Kh&kOCna_)|u6ibwAW}OBy;5E~qpgKd8p9*0g^&h7D&q3%y$u7A#X>=YF@oNBaOAL*B^+JMUu!W=>+c*&*)3p8HUY;}g*BFCZJ)@=NrfIItYm~vz93M^`21&D<kVb6mLkm<R<&0qQ)s}Rsj^9ywMR6<l=tTD`=HiJCmFu7R=XiSXjCva+iPUZ<P<V8LoSzN9JRbSAjSM$_piQjagZq6%=xtD0@oV#_x=)lKl#U<*VAX+9@jm2T;A2+#A;o2+NW+FZ?<Q#KT~>RcdG{y!qRw~lb5@HpH5f3UWiE_r|r`^mZ$q6F=aR{a+%KdyK=TB=*=hm-GC3_^l-?4DdBg+8|$RhLX<uKuKEGpmg(F2-SD<{QpU5~(1W;rr({7NeESUQzX(6nqXPFlKxmz9OJs+=xRfJn%8xAM(*la9lz?f<0cVOUskK!UQ-X*$ieN&SSHVf(ScFl+QVbFHT1iV49b5E|G32whI1YLp!@{t0GA%@j4Eu30eLFLiu3FN%dt$l(b*&QkHKm@78u~pd0nf(BQMw<FQc5^KLNHAWm;)yEvRkbv;EDqVq6xDe83?Fh20JYvVVW2x2y@hgA=)X)EFo~r503f4XUY#2Bea>}ksGBy_kLS@Pp)v$ILL{t%xa&KV85%P134A;Me%hjI_heV;!I7%S5wDUVMXnM@~cT}d(@sR+LOO}OWCvf72SD*i(0*D-%=OnSwtj{aPh<f4~<ktI<A8?)B*#-DX>5w!L=pQ0LB&b2ny^R5=aB<F;vo1?Wp@Ug(%O573#|QQBLQWshX|dKAHF(1`WWag57UfEbW>JvGhf<gum;`0RMf8XQC!k<}{9W#qMkU_%7+tpC&=t?-z=RC!HY=oj$`L>0icfPti2xz`oBatv4(po0-3~A^W)779(1Ql1eYKlN{xU04Nu&EwqmlYxSIq7D=3LRqBrWm*I$vj@tFmVc8;Idw7m=`LX=}gc8Jakb@gS0N~RgScg;$uG~Rd4$j1!G456_C>1hN14xw#e!Zr#bpQb8Es~mP1E>R1Vuv7r2ytksHr`W6ECtw%4XMW%k2{4=-OTyk;wx`gv)i%pO=l*)s`-o?rE|6K$clZZb4R2;thGo*7?!Tj2DsNrLoE9+x$nIWa|;gMAB=agf?nsS9W*H2srIlD1p7SOAtDGCA0ZH|MZ!4eI8cyzX}ussGeD(Pm?0=RQ;2!2g(bou%b_MwB+D}ExdGri1i_9H_Ajc0{jRX9CXdpq4xY}+oN43rt~{Gl(Mh87`<1b;(7sRi9^9QK#T&sg`xOJ^l*%Ka0&g)w5g7x6`SVfk$~f(ymxLJ}%pFjgNWc`wVT)i|Q>B;{Q5z6YC<Qvk4Xty1&Awwb!bC17uQScV)VW%bF5Uixrm?jvH>Z}>^mCkCe&^Tj<mC0@QYI@FZZzdJE}1KCq9&cAu=Hm1MN_gfUbdX|Ii!iZ`%E6_g^T$owYx8Rzv3Gvg&#T#GCVA@fSsgYXN4*p<C954WttNqt!~xI^zlWjU@bR<ZC4KJT0qO)Qzfa~`<m%5k6X#$bcH!84ckRQ<55nCya<)I-aejP3X{~FyK*W1<iADEsRKEbqH`!IMn_3X&|-!;G>jof715MIOMnR$R2jsDw?+U;D5D$z1Bmbr5za%1A3=vsIWorc#}{p|+A{5LG?4K7MJn)d#RN<mllr(Aw%i)}MPorXs+4NRSa%V7zkj=deSevP{Rhb|7rDWPc3>A_n_X|@qeiCu(+V+fI<~6RP5kRdmFg*(ff4R-U?WRABZ~kzQdkJ0fy4$G35=r<!C+^J3O*1v427-1X$1^RZj9&7dZ(BXI2dN>Sa4!38;$JoHL{iE=~1$j|J*r?JyX@J=*m~SQdiyi^ZM=mD6TxoVBRTkvgL$!i1_9EMeuJWV=;E+v}(b>sl(~UGgPTY+!!Q~VR5u<r4xc!tPRluyWo&QtptKhQ%HmV={W;HA_kF!%nMAYb^=K%wFVw&H`>bMTVVRElC!iG`V?e$ynpl2uD63_oI$&$B7YKnxSnCJjY({?me)JbSn`G+7J`|_4Q2M)!epkif1|WEmgv409b6eSb3AC63coJ-){*0HKL}e!$WbZ<QE$1BRv9C`6IcUch_VK9<HH`wf??EDNQaS=KwuU__z=32;95$c<!C97uUDCplsmOW`dTfWuNIJN2b78-_<VK0^~wsh@dQfft>BTKm{S3t$9~#~Mz1tyN&$A&{jFj;-QBigfANyHpK@1~mUV5Cm{c0QrFl11MQT@Tn|@kXwu`B6s9$nwHzk4Nrkz5Zos${0MWQ;uTRq^R@L|W({o&c50AoU7_k;pUFaQM6Q6j;%3^Pb<1Q27d2q02gtimcF4YAA+96QIXCW?5k!rP*Z0>X@fL>nqCA8r4!Cc%E3EYBZ%`MqGGrk&%&7#I!FX2VO^2*ou*u6Iu?e9L)-(c66{Zx>34_>lrWF@+_C)+2=tBg_SpPB{w(&}h#z5P<uzE@;A(@<K@om7!r96E>Km@35b2Y?wHb4HG#E-z<57)fNPmoWP#Yq9+3E#}5Y7HnL10v{A?tJ}o;|CM|{3YEu%sp7t!K(|lC_G6Ma-1)je$G3uCt^B3TS#QB%!i8d@;e<?8}sB=hUqiekKCT0}80l~L1m{5n5a|$?2rB;$#q>u)bd4e6ogn6f+Qph08M#|OE=sriIo1vvsVAM$^0sOeBF(r5NR=4Me-C@_d%30>Z2}i#B%H{uWQ=L42;pG4R@a(T%2`BpD<Oe_aK?ldr%rN1vne;W9_r;{CwHQlkdFWPy;o6uTy;xJh`3{TP=KA{`^U-Dt?#h&f!R*DG>$YV9#&<jX*h;v=Z?668<d@%T_x<vFlb5Wz_^Ty%>C(dF?2?NNx+f+tcf;aQaX_Jux_sfp$;r>_U&?fhlQhnwUw-cwk#Z^=adv6)*JqdPmC3K)I(zQIr3=4)E6mIC2t%35Zu9pi`JOUK;rm_R-%bNeOK-FjuKuWDZmOA{M?HCN8JmlTHwstZYR;fc_3-^LefhQ0n-@R(tCL=Sw|;_Bd)#O<NZU=e=<Avd+Uy4YVT$7~C4~w}(&_X*s_ta7_Mc9(Ztst}F#DV>d(*9jbgqa;EZ3d!C+bCacB?Baw5mbFiCLj9Eq~?@6N%CXCGi#2J6;V_^zyrI>D@T#-g|GM{I#2TVTLwyZPLxA6VFPN^sQjG%er<Z`3Pm<4W>O_?zwm$yIdlABKrn#OY+nx?yv#4)6*AQJ`3b#I<f<6Wf6ms3m8!$sWn`3?ujJYbE=uL79s$EX9zN{AY{l8V4)hZFvmD^)dB(U=L~z!7xg<T^-;99^OQVp@2PSTix)GMiz1+rU68fX?l(}y__9L3`y%eH+`t3syPoB@@Q)OV!W<2LQmO@|sb>A^!c-L`GX(A=)BSM?CGRv#)K0p!`@23+EtG^;ai{I2&x%hs>f^4IrD1GgJV<4lBS?=@u8sq11qU)jg}`Zmh=E*S=n%#hGXXJV7%&K>MNkRDJ@w2<&4bC8SdGm(W3$e{?k`WFYpPz;r-g^RFU)@2gjDw`PxneovN6?CQ#i`I^jo#I_nTE_Gg;O_z4%6r{3<U<w22Fq)pnXKYONUQ<bGGV(dmNn0PuOE*z>iz7_`WvTjhj*s-Ar|%<baQ@Y|-fu2S9a@($bVV~TBVP09n^nC3dKOTfn1boBho*F2T4Da1#qdM%ux7IWre?ocw}y;M#JfGu@~V`hl35^4asp<GK8;sQXi@>nPVksCSAM~?Hu$#K5EaWVn$XSvRw#9jVT$1(kJM^OEFC+@UTbn?15;pLLzY8s`w!GmF)^FA1Js^X5e`*UO>?6uM_^-0=zsxBcrd2Hrp-|B|nD<UbM)b06*GQ7_?UF})UPZxgJc$ql^daaRe;d;3?TUEW2tAi0G2I_a8s%z`a+ltw(>AmH7GZWWpFOL$_ZJU#_GT)jSp86Up{Gq(Hg~~jCZ+Jj)1~!|(s+^}T;eRS0Ag41ICXHi|`p}x!wzj=6X>`N3G6P$e@XiUu`^~mLw6tUkiM0JuLr+=HoI2Vz7V0G3sJbZiTlaVNbzyCoo@_ZsSs~LkSj}6e&%aulmwSJ0&hrf)ezOUKR|YZg#_*ZS0fikdKT0_4{=$@++9!<&Pm(C^9%ZX57qkz#ny<4p+}vF!xk>*%{mLE;5r6F~f1FSFah)#R{+z1lUiI%6YSMXY_G2X7`l-zyHM|+0HZQAGM7|sTw~@)dWwIYK#O+JW<^`d$mjZJKfiXlwZItmsYKo2IN^3`j#mGa6u=Lzvtf*HWGVVCQ4ml~uUY@bJyoGH?<o2|R@y(+6RpHh%yqrBBT+@(#Z>sOS)+zgz^$~r2@bDFO;j83F3VSWB(ZXSim=e?wiG%UNP&q<~1<(K_tS8hkNK~YFcEAHfh$Eg;NwglB?MG(&XQW_zGVJB4ppqYU(uRGX%^PnF!G|wp<-@)KrW=t(-%n()!*1A*%=R|eQK;VHi9hOYsoZ4tbg#bg#;Fc9@=?ifuFj6>FP!qQCA1hSKUSzhicdaRoEjeD>3AC-7kI@+=MCS=6e8=(Z@lp=mVTs|o8sxm0>UHA)UP6qu|x(b2bH8wgNT;n-kb3DjOK_!YI(5kj40=c^-yzbAc+jgVpL-~&Txh0QV`D_G50g;WmjIBloC@;2hKw(+)0J{mSy;U%X<9P-Zh#kX@9L9Z6`?Fy^1NHvf($nvHvmF@}IGm&%o38I9{w;zGg^LV2KS*u8eX{#u<SDFp60VC{ctm$*dOuVWprD2x|(c2qR-SKYF{PxBJ}_4mU~VoPB;UdVKpP6&4a7VUJHU8%czf21+o>gmX-BK>>@Bbr?yAl=U8PLbWgvx1_MN+Gxk2W5$og>9II{q>9tmiYjt+HoM~Vz0x<{;cKTLted%>H58W~bTcHqjLJ9iIMu^W^g_%1CAyTwsU9Cy$23LoB<Z|dH>QekU;Hds;FqW6lX7)<YGgW78FQ*;obRQY`Rr+8jr#O(*zg=Vl`*&4{BiNvA1tJL{&tBP9>kZs)kt353h(s9LWTJLse#-+SQx&Ca^+l5?KKW(?`(KvZV_p2X{(5x%p4mY>6@!J(xC}ix78n(H$KR5N4=H0b}!e;d~UG$^V%OKv4q;4`J1c%^P8)mXD!<q3GoVhtM{;z68$Ne|9;BvKWwzOuIFlNLD%Js+2DaDW%54C^j!UI{7d?NTP3NqQl{`1(|1-%5I4HJ-fAce2KdE#qYqo9-lR=)J^f^^RUVjy3h0|Y;6_85yOHj9(g%3!%QNq7Cs#r$5WWA1-9>i_8&1Y(WD=ZfsrjevXu8|N4HH1)ULwIQFKxR}RVIMMTDrw_xwZ09^95QXlA51-8s7&^lii={@9f6#V)q()o!Wp_*e%LcKP$hmM=b+og<jsRQEzw3Y82Ni7;>>(eLa7c{*bB3)SY3YL3E~DyVczCMnAWw=F$3|3-zPG{I^t=$KvRg<&A|%BJL1IURx5vb|nFZ6gY)LM31d-S~0FA?s(pfB*btLqIzJ$7B%KHk8fE%xJMuM{z%C#t_&B2R*j-28@F=)a+XQ1q%${9?2R_4BIGv;^Ty)=)8>E_?(Q7HxVtH}+T!`+p06f_KfVVXtnVrcSKdT+1U81KtE;1|+qZ)#kQop!5j;wm0!uj|QZmf6(%i6MpAps!5Cu5~%o(5w<U}Cqu{K5`9ZWS;KwyIi(4$E{zGbEY?d-a7wu`mfZn--PHD<JK3)Kw9e@_;ZwcDO1jbCV4<2Pnl<L5|XRu{fiL|RQ={7UiS?>A%+e@bTZ^9rB+ywVS(Jx?<9qdk2-_LOz@6w9MUjclpc&>-nG<5&eF3Ow`*l8Dn1h+_*lf{=Tykkb}ptsN3vTgx?&b_AXtV-NRQDV}FufRtr<+*zkPB@^*@){X7ehc^lZ+Z?XUJM*~aTmG1oS<1Ei(wI4*bjgML8IH`x^X!QSdOm^~qJV*Wn(TMY7u_r=k&jEeZNoQLta82fy?XS`r*no@*Ltl!`A(BHtAWIbzTGp|Cq#gtBjpWyN|B2wTSSAkHd0Vb35Lvr==5sAF}6k+%Q1A}11SI)Q<Mv4IWc(T`mCsfLE$W$=!?#u{Y@-DwS)t2hso4cvCi!Ud5}Q_YU#}@;TQHw`k79;3|h7NDW8@v{q|cI&bf4vXyzxsR*|Rd+~QKK<(*fT&0EjGTTV}QCL4`>{m6DVzaUnBT=A|N3askL=5j_&CfacNrEX?V{;^vRhT|iUYOAJ!-{{LX1yj`-wiB5=zYr0WTFCTO2TW;U*oU)2x}wB-UO{TR{y@|IR4T=M*!k2q8wl-os%Xr6=)U4~v%<n+-g@%emkPPTPSU&IN(){Bm+j>BD_(5%L_PJ(yFOaQE#K_i`;6^qDb;q~n@oMAwP6q9Q@_(U%m4b6qjJ;eGh8Pfv9OQ3`NwidJjXXz|9Q8$)9s47hW$mJe#D#!w|48qoS9W+2C{tNAFIppSxRM{s#m>Nr%Su7s|@E^l-}M;Pk9&R#&$|~<A)`jyEjXMv|jUyo6S^SO&zQco7&~_VpE&kwMLlgj2RMt;#uc$ee4fke$q^s-t*q%N9QgCqZ$7liBvka&9~1k*$Z!nKhB*C)-gKIVJVCT_R6`-mokLl@|BBAlXos$nP_ExRX7?ig|X(X@Q0aa%0@!H<STUrAnPu@HTkOx;Xb;4)yZq&VqqBwPc`{*eGhZIsM$~WBGT^V6>yeyFGRWodJGJ-j98DPc2W>aEeB9K2Av8H6*SsGYr;G9G8};|q2U0a><D8&#`(p>T^D{AclwlE$`<}^2DhH-Ib9BMQWSu9o}e3X`&-wI=o5@RFW+hl3mOxN&r=?qBPxziBGCchB*Ta!90E(?Aa>9oWhGG<d5o#CkSi1Hs<BoB>yY+g6k-Yqr_AAzN7r?64D;a4vzN}5py=DPnl*6j!aJ9YW#bHYW-<&4kzq7!X#$*kXVSeUTjJAdd&6x9yO+OzWV$xzu0$I*iH?N|7Omr|yO><0&Y8LrE_CTRWDUD3pXu*BQr6WwpJOt;JFkMf5$0pC<?ig8Z(VjS_;k@7i-6H^4|_5=iFof)oS^6Hybrg0=7JAF?b*o6k*;_&j8EgessH9Kho$kAiMHp|+n45;oFBb)HcXnIsEZfR#y*9ghc9|DxH{3@yF7ss$%W!9PtZ&1#1YDkIY3Bo-mqan3T&xSP$4G_)slNCWfW+zgg|I0(waNLA;Ml;jSS_&iz5e5&+mV+7>dOXwY?aAd%=X-MfhY>2EQeEdW>B<BNVe{UDB*+Z+_GuVG~69?mqGAVY<9C$&Z@h>NyGbT7|?PO87dQhOdWPKXY0A=EBm2^Y6?BuYYy2^BdO(`JoQHzM9+K8{VwppqH}l`9~#lk<uTx7Uv)P>J(mmE8~Jokz)EZ4Yhya8U5z!zXwnDo2&mLy2sIL4SsIgmMZ3PPZM$8sR@yqDY?^Y9AtIL$0hmE7hMwF*Jg5|mn$jC9-p&2s;C)sIg}5ISG}4}bibiO`(q`go)lc~{Zc`iGbrQc@3bi4Yh5sKwcz&a?z}Ol1+P~mb;|f;Cbr4x;qGSFsWh;q2dKE@hR>`!!_>IJfKzQJM5E|XBq2H()b1h5w>gm?WcHYD{B#aCKh#dk0X8{3GiW#<Jzc5pl@wb7s^v7*gZyZ-gUWzYlRAmhhuA2W9zfntO?rUE49L4O0(LqZx#?jzx@RNHV@^$q>dM^;RiE^<xTj}!mtCitd|cBpuvIFek1{#n>6vjeV@^#<s_tI<-ug+~ounBt+<Pk3N*^^$@%`+one+5Ar^XEioSGu<T`g@B{dI5UFVYSg4mee)?(+0+D+dK5)14#@n%;G4(nyJK&pu1v7?9~D2ScaFq`sUmzt;czV(bkXCa2r^-H)0x;?z`w=0JCS&aG8W$azQ18+~fhgZ^h|l(W?x*xS-pl)lo3Q+x9d^26HXGL!B;-He!9{aEeG{l&Q4i~z3F!+m9Ye$n^ZU;ee66+b;y{z)P@-AldJcZ)Rb6!Do;si6PeZI*2J>EU6`Au~^XEp@7#96oIV2;|8g|MthaCT*1RLHpY+gL*xN;SYB}<<;7N2f0xq9<n+!aPZeW=u{zOoUv>V`k!sSt*LQ?0r|EzTH)xOvaS4mIOuvRtQ3p6QMXM0AXlHTUe1nd7S0C^tLoRgEi=PGv&+oNRKH4XNlI@p_mDNrv-e6r_F%xNlE!Oh-PgrGc$69mGiZEj*Gm-uQ{dF^Klxnc13j!i>^W!t=IXz7H~BCt9Q8Ia>U8(Kq|+FhId^#CROQTAt6#URo!-hm(&^#2d1I#S-mVMpR?`M`uM=l_%8`q+$K;>y)u}1=ao?nQx*5@M(w?d;s`n~w{Ef<#n%A1Nd$?=RhWWVLme;J>cC}vn@T!@QS|r21%*R=-s_gAajo+_}=t?mpH#0VCIAI3^n(N00t#iEo8Xk4?rMblL^rtFy{&F$B`FnR;%a)!`xxdGv7;w7T*K9vk&B@L78{MhJ+IHui{xa7KYByhLy@@PW%Ju4?>AlJO-ziKreQRoJ9Q~lSEfzmHTb+g0>^C&S4>IL${GokVzd>##tv2j_f0C7E^{R`~PS1$EmD}Sg#X~%@bgJH}&58@VEdm2Pau#T~+IZ@Z`u5Q6OnW?>KMSp!R(*n<CK!8|?xuTlvrE~+q<4vX^@+Q)?pbE=U1^}w`8zGlI?6lEa$m@Vi&D_qhf~%)q5rbD(7n9j^VYKUFJZmh_+ZsrnDqQX@qWK7PVROst}is|8NKY^@1`iV^R4`ISqWRsqm6o@eoCxl{6m5;u6!5##U*($c1p%H&9G6D?nh<(#b}gtfMH9VV1O{dtOwpu<s{$=S<0PuUT~neMow6Uh+^DO044KWdq<cVtLDeHYTgT|ZBA1b_c7>=hVFg2iVv+8hx*f$q}yzih*k$lyAdFLz?wDbYSt8CaFlA+2rH}<HOx_JqJ+0W&@c!r_9&R8DBdlMLyA*Ixe^Xg3be-(c@Kei$5?%B>B@N*Cc!I-2<_aU?9rQ(p{eS{*?QHrPL!<_Z2swr3bXz4JJB*(cX1)rr57gM*T0ms**d@BIasFr0I@=ypJ8`g&u7syZgR34r3f)l4qlkF9fqA}7Q={z$*e5f00+6!mNfc)lxwASuu(;sKc7l@eo-aU2HyN)E*KOsj(igX+Tw<Krv`bgT6C7!97Kl8jX(Y<wcJd46m+Bhe67Dm492>S^&D_E4^Y)TA9T-HyB@pT#`U3qY1SM)ks_I%1}8C_jMn|RXs01gj+Aky3UJ~yW7JvWCBilsZ*2_mmH~kkAV@QSC=iTsY=ne5^3F5F1yE#cjhREpSsc{M7k=fIo{5b!tP83d`mA{!tLm5E3a3u%LpZ^ySZu!(UeqMNoXop<UcEC>_P<`Su}8%vjCl83m_&~05Dj)08!~#PN1WIT?$RVlc6O2ObE52=T26|J_pW4}voiF<BIwG1X#SOXvh@Qd`a%Q0JZXrQ9+gB-Nib{`oc}=XW0!LlI;i_K*VmVI2NI#v3ZHY%+8mr!pw`oY5C6P=GneL4^0nEwRk}iZ9<Z&v(A8rnMM|{L#N`(JK`B42a}SzIz@YInrti{Pn~<&BlOnLSQ8MST11N6(eNBZ9FPsPfKX|6sbhqTt?xtRgr+kc?-R|YKm~^$oHSs-sHv(0Mi=j2HsLwfbQ+~;HOwQi$7oy7{7_KW(v9&`SIs0mPnz-bI>o1i~5hS`b5@YEgh0bX%A#fO4WVsX6Q6_y@3MJ#*Gf15lMk@_M6yzmi9x^GcA%~T+dTbwk&h4Y%K<%SXlV$n1iZj+FEK8OA_P2Uz@o#?iS~q%DS0%%py7PyDb9YyDl{>ZJt<!Aj7Mr-ekCnRZ8?~C0d;AuPHp_#y-}8+(7K&EP`%QiEaiw4>13vB<MdQZjVz!S6zxHm{dr#!r5HNm}*|mdh_DDK}5EBe)?15ugD<}bWQcw$Gu8Kpb84rOBhuBc5y)fPqf{6piI5{;pjoxRUB3$=!N~ou)nX*t7k=*THXp&T2&MBpz(l2&$`JG?8latp=wmbzX;YL%Oamie9!Sr_F{?eP1uHC|S-o)j4*5?p&boZH(>lZHOU)k<Stp8Q1s0R|nox^&s;f+noemMhGQ7Z$Mmxo23<xx^>ncBTw@=KrgV#q10(`u`HUC2+CcfEo6Cpu9J?x_{7$0;N!-TUP%*E_D+Pg4_K(n`Kj`r#f;J$Qf7&&vLlS#Zw~CsdFxIkC0Um5I00U*_NX3z4hp?fTiJ@Jll1u3U-@vfo~rbx+hSfKn;YQOZvdNEvp{BMp>wP$(t|3ijMFLp7Gb7=;}M9tV#c0U(IwUI=0w0KhmumQ0Utqj74`-L6ouoyr5zYg$bTj5ap_&y&r%TfwP2g~cRy9N+%)`q~Wp)>b30m9i5LrjS5S6v!HsHy-3%O$IsDRc39PeCt6u*Za8wvQjq0OD<0Rz4N`-TI5=V0OgcS+k#N+d4sJRwY{E;wilHQ9w}qZp(lz;EE(opIPN($!UzH^#DFo#yyTPu1i23*ZTQEC@D^l*#85FB>*Gt4Sd3%G@X$b9GB@C((kj;$JQKykw8gIc-?PMsn~ej#-7-LIH7;nXn<f|eM#D0+_hj<OAsBTFAv#JNf{sY#38q%kV70XO9B8JLCX7R<l#`qwM-^t+V}-FI!fM=!dMQhlaiebGs9Sg_bPKmTRQbcUvbI)wgR&Lfn0)`$aIP!A2?xI0)ZThOv>NE#xX@z7OR;rIA9kbfhN~6H%1WDAaqoAv#Aw!gn9OE@hW9<h%c+$wAzR*iZ&Y5~uks=xlpo>TFtJ*pV4#)a+9;~Df=*N7iDVXrKfRL}TFrn006;(l;hJk=Ao4~?c}$cSzhh>L<1V}i|6;<6A9YfL*HTYz$4Ze$r|l&<y;>z{+9beWU*AZZk|!(W3>&pmx@qRmV~X#c2juqkp5wEeW2fz@ZYV%&dN?(Tr!J-Ka@76o%b)Idl?Wnk&n>N=H20cr>w3Q<uc_VEQ-glB=GH3J#7ryO?rm>PUstnY=d^dGWL+*VI^Xa@C0O{hZ<kN~ZC?!d2oj2^-S$O^ql!Y|YC}p<q!TwwgXQ3(EKCzPv>4n(yVt*@2l>MHigeS1CiAu0cbIK8bP7G?e7?7x`A4P=_e+9l=E$|?b<R0BPuGoxVE*31G=InR&Ktvy#=I?Mw)z7e)R3@$@3l<ysA)C(I^XK`wnesY8l39B8}>oA6C%}-RtKHM8-vGhZ)mHjq?A*AFg;uD4j-&}2M;UZO8G$)fy!Ksy1LLXgH+*_-QMJbr%vXzuAy(M1zYFu|9)|EhYz^jG>Q-B#ErgfwcK@nV;?k%TaOCO%}V}$Q!Kyf_|08+KJDPm+RZrqA!p`kyK}AJ+aJy8J8d8J(J1Bk9ZNYvadbNriIYS|PnkH%sqv5tWngflwNk-7c8pt$4MspI6`6f0V%~7Yy=MxkQOfZcXK+{Bdempd;-;0V@?m^3DH30wm)@37QbF#MmSs2je>+)}d40Jia#tHNubk;!OX`b*p2!DNDTkUp*l2j}?@xLLv&zz5uANqrJGFQGSu_{NCYq-;(LjvFk;)EJWSRHaDWI$c)*=KvlqhN|0;C1eGImc&VQu&vUSNf(irmI7no-Y=1B+%{NylWvi<WGd(lMFk6uQ@IFV0se=@04jM%L<4-CrY2>}k?Ae`p2PPfdmx<Ga0T&GgO915k9mF_qW-eon6km;FzxPkg+BS$=5_yHn|=&zx~5Ci@TtnZEMN!=9N)61Kjz2XfLQArx|?_)!k3Bl4u5+|dn0D2HBp2aUy4N)^ec31iYBi#)|hF-QoLNW>&QfSg0+o$yS6u&^=>4r^cPSZ@8+%dPNvms@A?Znq{)wRNzgFSUN$oh{}3N;P!tW<KA}P^5FFv$Y$R8gg#bpk^ftF?VERNe-pU&kdRV!e9RVjW@bOhol2NdF67)optrqDYk>j+~V;YZ;bWTeb!fz*jJGA9jZR1AIpr~lVebDfgxv}SSc9<f)F98vjSu19MjN8#Y_aP35Hl`Vt@?ck1@~+g6Rl;8Nn~lWQ|L{Cb<8v?j9UOGg_IQueRFptEH4SI53{+f92X>rq1dUPSFB(YJT6b|9lh;>8>q|2Xu{}_%V+#u!T_!JZD@5PdI8D6K5#^025$=#IaRI6AvuYgj%POrcx+Jjibn50gp2yS2dyKNiQ`yc-Qf2EndE`81Lqlw=Z1$)hjC4#tRo~Lt@h)8GrU2FH7AwVD-}P%*{;g$%f|Y+b#`_-CdsPCR{Zj^GxV+h2G8KJ+B$|jtzGPa#vfmS36HNyO)Ixt5Tp8#nryWZ%mLMFhLGdf}9;G3PH{pKo&ZPXb9H<Rlr#1Ja)tf+og#WF!+^<a4wZ%N*Ur9Mn;D~-&wCm*}Gt|g5e3SRibcGzd3vUN^l>~hxdKyLRvO+>F{UjVi^2`tdLOHekMQtLG&=ofj{l6*AITWm@0LZ34goR?9!$v&5#Z}yOa(*m2M#aH8*qTes5LJUu@cQh~JZ1UOw@e_KD{jz2Q^;dHr^D`Zf#q`WGqHnzOR^rgR6^3a9j!qoCda1@(UX)7LufJOm(VwDXRXmM52%3kQrrh-1hs3mY_3%mc_Na|DGzICjA)Au+K`Vu6%Il6vGFWn`qDIC9hz;2_i!mGbjJBae{VaW@Oc=@f-dqyN{a$z@izhmBlv=D@pMT9Q|~s#<ESc)Tm|!+-&!+xj1cf#`m{;2%nUF*7_YcgMAK7aK(|rJ@9@_`;1IG}sDc^dP^5Nr6W;-FXpjn!t}YvzEdK-Kla}ECU%U$7}^&-k*Bv&uedZ9g<${{qByNXXxJE>`Tgze7!2wy<mLWhm*yE3QPe%O7IYYEEU#CAprqEh~U@S5SDWn++9pO<$`Lioo7*aRTHdeq?f~pGNB=b$9SoDGN!IyGtON|#A;Kpx)dzGl2()yu{SRDTWZ(Sgd1PFa!y?gM;|<7FK6A|_}{PX4SO;gSpvHNdykTz`HOdceD=S*w)j^U&e@+8M9CZ@kyH4Ki&#?0Ax2n05W;K<;Np?uRFF^+gM(v<I3Y?At2Gu@T7<9*cEvjfIWY<wVgxf`R~XHO00>A6IULC+M}p?Pd#v99LG!dIUTwO$+f;l0Sj*pAE!}?dzIJxok>FM747XZU&&e<oy4vm=>4N`~biwclS7{_>))-Dy2>Usq3V5eHWzYdfof3i?0+mzHNa(DU#A)G~@dz-DG4;m*6MSA>`mQ0tao%^HVJxb5HT9vC?M?|*SBg0JzA(aJx&$Mh9yLRRGe7Zb6^8s`K!&4a+hz5e3riQyzw>m!a1s1X?VLr<a~G&^Ke)pN=keWthyVCw4o-OO)`?2t_Audlvxw#WMmsb$-+fS1CUf>D-uT?bJO8G$66@}8gk4Tq#aX9sGb<V<$;@ZO3#VKB@a6WZxh;FXEbZY!34c4p%_DBO0Y}WA#77q~tf0r53vH<df@27{z=&9&w1I$PO$h+l6G$ipl4^;tv;s&$DS`@*EjVKf&SB;I81iU5du?(bS)wClN#k@_`mYz>L9hI$rJwxdZ~tMAx#WaPeuQnKjkqr(FeFw}CM;4yD~FLr)&V57(2{d%xrLZpYY0$=E9t{aA(oNj5{kq-3d1iif+yQXFvcR^iR}aji$$(=UeMJf6uwIca;=blZk3kwPrI;Vr6=#mzWcmDBzf4G9%VAPT>m92qyJ0Ox0EZZ8HoJN@4wtF;F){g!aj0pQvSBp2PAd;^hK*YeC0<?d^@*NZZ6d|H`23pOh4<(i`{>1<?v=efRakJQMc=lWunH4?1^WLa3=8~uf0?h1!6w$#LMknpPH1N`JLYEc4c0~L9BCP40euwurrv_`^YK2AnUk|tSk)ju!4K(JywKb@40|sn=}GZufqqmn1mq5FohTjYrFweAuG7jBR$LUrDxf$c%stk7Z;Q-dvr}A@|(}5eJ8VN-)O2ml{>CyS7`+3dycS&UGb&hEZ472lX8Dl3AUU5#QQBf_nl^0D&5c{70a_wabtRv&@`SQPQ5~20wFjg;lGtfird&EPD89iEWjEP3A{nkA_8r=mXa|ch``4t)AL=2GsivE14e4sk=k{nb{(l*M{3uR+V#*-yB?{$!C#u&^#x8Ic)+CQmRrjuk=7$8nAS=ftsDcEVjuwY!Z?f(Le2w7fu|l&#1W!Qj>&`LE2Y?EziY5w^1|B#X0igj{InJG%8B0LSqe|=oyKvxIm|!2ngkfv6WLL1X4<Zv@IouaxOrkSq*Yp}_jtEkb5@4EOkWSP4wYAS&uNTZd+Hzc-gCUa=mdPwcw3$PZRK3LnaO3#%ldsufPB*U6<<C>N6|5qSA3=pd(?H<QzVa4i@gVuDvyJ;hFUmDycAw3DTMLB18hA<#85^d_Xr4#C=dTg=maOyS&GI!f!Zf9NMvLS9vq3YzljNfwte93SXPW#g>!o$YRa5T!C$!&etxf{-*G9t=0zbnBl*&Azjfi9OBac&z~tBJ;^nj9j+bI?rRxBC4oQjhWM{Hf$=5$?c27`jo(%tAn<00K>AXJq=IZCkab7Ltq@Sir;AXM;Uv!)P(Al@O$tx}EX-6y48P%EdcBc1U*N|Ba=xS3+U6~B2QkT2@$HQ)165GA8P&aH<@$<SZWia+=TSwskvpLy$dJUayCSC~CfA}q3iJh@s3eeeC^Rq;A7B0V(e13Hfcjgyjj;!O7N(j8w6e*;kLdXh@VX(rSVPjRWwE{!qyh28EVVSgqamn2X_8-CiFCf_e&+A{E>^$)G0py~y?F~93_t5VG(3t#iQg~Zd8@lf3Cufw$%;uC=E8+}2)4-z>o$-Ir`1<L<<;gRisLKW4EeWu}2~V{t^?jeao*})3+xh6-QqA>N(vrEJbB1+u?Zz8!9wa%Z4~XmKVB1d4k#=9L?&-T6ZVtBP<$U_A@2Ax1U~~Ug7N%b7qbWM|!T!xAzqhq88L~=j&%U}bdG?n}o<++rl{=$H7eO0>CWUE;G7$y_AQl2FtpOEIBgl*;*mC5A)rK(=M_C{&w8R2~iFX!_$+P2&yjf}N?P{g;xn9`3Yu#j}B2E*x_pZ6HodO~Kv}BL^j95*Gh5v1wNHI>TwQWoTZ%X1Vn%*94Q?)UAHKt^edvT{4hOO6PV&z}G-#u1*xQb@|qHqR19_lx>mnv5u#g_Jg@PU*edX(gpeC$X-ick%ll|b8IP%+0=Q0|yU*c0k4(#S(Awd0WZ5M^<~kPAzemFlpnKa8Xohl2D1e&?hYDJ8ZNcXonA6^_$y8u-3Vsr%|IFWX>WHO@Q5yx*Gi3KVgLiM8!lx&S1h^DC#@#<%Z3A6upiNbw7q^eWFYo%KSZGt?LZ%eWK5LqLtd4qEPjlGZW{UEBx5W-7f6+q5tWIm;QqD(1(~=oTO2n(}r5zbr>Wh+>?wQ#ucaerHm+{-Yzjmi~8Q+N=Gd76mp7Tlu4^I@eKPMw@rDGpaq)<=t8F(&bWgzS9=0S4uy9PGfC$J3s68->HXhrGoF^UXoS5QD8_LZPsJb?`T}{!L25S%<@OAHa~0$DzBI7#=S=KZe>0dXe~28i?e+UOUrX1^;jUrj#Bj)OAs7vi3mgl0k4Dy#v>{b<U}}Y!Z+~(YAZB#3MvmZ(p-8dkRaCiQBmSpC`#<KFv;Rn4)DAsv7f!x0#a{`+DbK8RWqVyL+)ifY(Bbmfp1M_^vu$QIuJKJnY`kJ)=UO_t-%s^7H>~?|5<mUq_LTAf1|=;?-v-+CuzyrX`JWG{#*0i%ag@fnz<eTpELV$96Gtzo9knU`r_)sl5#B@gB(hC3tLUFlGV5;cyE_1av=E<>AQVWR_6Yj-bf46oDY!j_g~j`;x6-(kOJx3-7iJ@?MXR$tHOXM<YZc6R~r=WgIZ3SbIA(ey1(1S7ZYK0rEuP^$+4|g120Xnt@bJN4W|{jA%>>U@-latm3g~;xsQ@4Uw%1>M7zwo?3+BAEVL-PN8PZUrhK`bn=-;kc95HDOQI_!$2cruEl$dZ`?CA@PSeU!2j@3DS(q6d_kLXdTGZcDHIVP|&S&6wANBl>=_PI!Un{-1od(gKej3|Foc@ziMY^A6%KdUof6nrS-p?m5j$k}CJ=<pE8fK7WcaG$qu5?#i8N&K*6)a^c)pb7uadP)@>BpDf*>WXm-)tT(+o9Rp%MaKR#otO+x8?_a-%BTV4;lUJuY!|3$>v5r%cS~fYEsQuhs9x2Q)cPBKloz#W7J>CN;RN=7+Qsu2?Tq+yQ{5U=Dmg<G<KbKw{^c#B78n6w(kDuBtIYgB~1!Ph#xHv+9GEx!i;H;l%viO&Vw&4Iids@1FY}_BZLX@R5`#Tc3MgZHPlj!t@+0|6Y^PGqDwwz&WSuII%_S+eq%ChyqryzY@}q#XN}=ZYg-SN|7s?2G9NkrMd1v1T$o!{1}Tn>ws~_F{T|QyuxUAa(4Dd1RYrsQjE(%^CXbTMBt+Q`PobE{Iv7mr4Q3K5%{Z~#2%#KAj={)EhN&`$Q^Oe(4k1q1w^-&oA3nmB^CP?cF=Mx1zx@m<mIvind@*WadmvAQhziP%E|a^bR9WhRmBPVfg+C0jo-*m66Ifx06l8>1X(VxwYYH6G4gsQ>WJC}+%Ab$&=g%a6e){yFr)3iU{=-QxV?NBW#<+CykN-H69(~YoJH*V=t0lGdL2W&(l5FcW<@UxK$5y3Wg^)p=U3!XtwmS=*-~b(EBgmwcOmdF|(NtQF2nivqgBVbQV@r!b#wrFqG17#XHaK;P5hWm?jMy>2J-!Io+nK?m3uaR}dL@;hKk0cknrOCTK)O+4-<VOZp=%hu6;pKGU#FthFDf!%J%e8p-RMe9?zM!eDJV{T><^~ku*+2x`Ch{aw3hPBDWqM=_}#vs)0Yjh4rMa3e`<JO?oAr%m*rBox!3ZpJSrfEXKq99MU-wf;z!+^?y<ZJZR^dA#cB4{{0dfb=5u@4H~+YF(ifSfrsw^GB7?b-lCzITaKUrVo60V4iU~hT(wLqRN(oieLvEFj!U##M3odlmE2)u-q7EjG!9+4;4ds?lq6No*Q#z_Bk1EQuNZndE+fEtPPa8(;<PqPjWF?h&IxkM2)gFve3oHn=wQkI2NnyTUvZJLY+MPZj8NRHWrZttHev(<4;l4FvYpT*WstoTpoyrM-0v;tx$^kPDDHwAUgahpaa?Wt0f|Jg)0YWHip@qsv77!_^a{z%zmTDl7WOUGvZv|?r;h4OUEX{f)i_VVbOUbRSp7}=g$+u!#TJ*{9?COwjWO}AOG@#up!tfilrMVJ?jIQOugw+X^`;xgiyf%+<>J-4(11F%=9$CwnQ;>11xwM)QZJ_WDae$Eo$iVOx40l#qhRkU5j&=XR#t>z7uVu!+p6Mrs9k{zECT{0ugh9XUAmQ2&9p$ijU=3@j{`ZZ)y83j3_<4=LThxDsfo-&jf=qL~Si33ylu-m-x}ntz+$wCt`80Ey6H4BoyN8`Ffm5J}qfve?@jmO~eJ&w6QWiUyWQdhO2pCDAr6%~Q6dHiF(Mn6AfYg*BBDu2K2u-YV6e@#_<;0J*w6T`<JZfnx9hm#(tZsa)rv0$W&EG6I=G83II~Ll$J*dIns%(tNBV1(@AwZX&;P59`E|^7xgX_u_!q^ezJm=g4W<Ay_YCd@$Y-9L~5ej)F+(_d&(s=HH#xoef-~P50R(e-2z4u<Uji)<%`EkSAQb+_&cL)8=&t3~--+kr(`L~n$fB)ZSPo7u*@4qa)5{~`B4}Q=E=ee90{+b&lK3YtW|HZ0Fk;ZN{b+L_Ihc9-z9H&pW`ibwC{;W!Z@HpRHuubvqDz(hJ^_t-hzq$6WlmC1%ER}yB-ovjhp1nNz<@f5yUw&`$(&Y<p2fypmLYN@I3BBY_OkVDW<Ws~U;a6)zL=)ZAv(aLlI63)AT7kmIx2{}_b1Wj`!k2dEFXY8EhLTH{<M3a8@8o0%%1ukijZ&h!HyN;;b_we3cIQf4<I5dOl}G1p7tdU8S?TUHf+;y7$!k*S91e?G_dPx?UDAmu^;X#U>ZEdAq)Y#)O-Hx=eYuS!-|GBJ-zbY<hgdmXK5EY?zFIlN3!_ksP~tZ)qvt<RCuNZZRvyn6luSE5uK?wT4I9zjbjP0!`w?T4*wyq@y`oHJr+YLloiY%!+%Bz91xwvmN!LGU^fo>%O>cz~YO2fSLDS3E(E@%p+x1sV&dZZV=B9tB+R3%mm7G6JFShyg`Ni%G6L&MDCS_a=npXy#fYUQ@p@yy3haY-x6e{JI`i`%*Dk=F+Gx^k{&Ww$28fMLnym>hX#$5>~oZ;1ABrd<}mfr1ulkdH^P&W2I;P>A94wXHh<gFF{jWfxqDsD|!G2`W)i@UbVC8F!LZ?6pz%#O6%MmSQDN8P9p%p!<9gG6)Ukdn?xY7`3pB>@q}D@VMM!E3VsOJ}qY;uu%4m#&<5VNSe~g2i(KFnXUL93~(8B6nZC?uy&RJ^QCCs_SWwxkl?QE=;a01@CQP(tZ6)sm9%vyPrd@F+V^oPUmOXJ?FTS%NAt;Z*6X`s8QOWquJYhdKR&ArEg6#j;hZA-YUyLZDUQHs(Bf_UTovNX;!Jl83w9HrIm7dekpo-3H?KNshjO;wN=|HhmZJD!v>S+P}aBG?#%0@Bk6wbea*<NM_F2WBnmuWs!?>QM#RNY$}|R>Y@D|m5lWnZ%s|CqEP@Lb3*<H!W+H^OKK|86rmghMF=U8hPLH*r<1Ln)gM<_dblsCL&2uMzqTV{|CVzGI+_@98)uN11ov&TgiYwnfu}^Z=;vY{_SZx*Bh(xlGE&RyGdKi?42$V;PtW|JlkVVi&X$!%q*UWH?q=LfWU}X_<NCGE_#EK}RJrmkd<Fw-dN_}{{ILDgKA+6~YPjb5wbYyv<<$|#+DU};N8gso*k-T5!3Mxy?m&x8G-}?8J^!m|2wdgxHfrqkIvwsQ@!H#m*Xz(pD1OO_e3=}{iM>QZwdQT9e$cHb&wNxG&r95`j0)#CT1X4+{L1U$Rw2IGI>Hh6+V_hm1mb!XQD(6(uoi^ioI#+<^6sA)t==(!ewYalr_k2Gwo2Cw@Sf@#fZVAc$Wkud6I8kvc8;zyz1Sv1S*>y9;T=1z$iSajkQG=B6?T1dj|7v&F^Rw0SpXJ>`K(gBBB*OE>(?p!syMK!T3W4vtsT8%xyzkilI<GJ<@e#J>67k@@qKHp|nqz`3RmOAU6c&b4t~5}JTE-O+Ob8(z_Ly)Fgba&-9t-nhVSW#V`T9$Xchyw2c5)HE?6={pOWlN?{A2Ke%diK9>@OxE-iQf_s!?1;yzZWuBpI1{^6ROtcPcr<51PV%Ow)dI^`GmIt`6Dt0MvtfiLz^{nKj(zsdhk8Kf4tV{q0MoC3k1KXgXc6JyFl_a(((!Lshx;o4ZQNy*K&MxeH-_#DB-ye8)xe_Sq$S;qCCpxpQG6#I#lT)yKJX<y@FRNo()&m5WQ0cP?C+Xk~sixqKnz!opbdR`|or`)XrhaJRx>C!%28rMD)3bs^kG_xzLB!o|XGAUxIN$MrqTSx#mr|509CsPzzVYMkc~IwKWTo&o5oGa49ZkR~Bc&^Sa3jzIvaQB)WwJY)##F;o~sg=Y>GX5e2XT$n1?t)*B&FD=gQ>H>ubiA|WH$8WsR8};Ay?AJwNq2~yE*aC?%93}huk@|QFca-ayt3`sLur$={&Ar;f@F;^?D&l0NGyre*tlQmybmIx7^80lBwK6spS|7F*+U0zA;SqO-%yj3Cr(KyMM9i9+(<iJbnB|4ahp|@RS(%%7c`|jEck4lKys=kEvR-<%O^x-E5f3WydioHbx1Gq-pL@hC?=Lih2nYg>@cN~tWdvZSjI;zB4zxi`dLuAn0%%4&i~V+ja1j1c&N~AH@L0&ON6S(DdQ`tYV(QmxLkWzv3eaiBg=lEy#8bvTYSLLLRrcXzZdBJTg=;M+C|~BAtN&KOtfgL36MLnT`}Cv^X_mCS_f#!M=QT317<FN0J~FfZ=Z*1z^>UJO&6R$`>B0WNu9qtJaFIuPzZ<+>2UppMmElRdo$G^lk>Ns{sXo5aT(Q2=xYnIGRcSfa^UdC=8N<B2-!%UY>#*(eWj>AfT!2yi`3`iG*V6i3$(b6rqGx3OsNP!98DA@q>GhUHI*mD%LB08$Nto*WNyYm1(CoPVR4t(uhb;~IFdsfJAKgh-8jNxU4^LT{B6qO4Fb7wCt?*+tAh(3$UjOIvVy_P_m=SJ+6SXIpdu&2jua;k5E4-dlB=tY~mzVX*w;H!Gft5F#EuC=rT9)3*9^YCCZEy6d<`1%hNjJ9^CcV0D{$^`aaJ(#O@k!I<`$6)DGa~yzw`VIg{Vp_JyjeiLPPDCzk?&c)dR>Hp;z-M^)6dlWd`Wq1t@x7#Y?!k>TN%5ZA@_^rdAB+Wvd+#F9=Ln?{<jkhajo=;#G!X<IMBC~<dZ)r7{}i)c8REb*NfdZTT87Zo8|j<H+?5P&f>5OVY_ZEUUc6b7|*fNO`g@HJQSXJVN$}Gd*y?DvN(k3Z`N14*z{S~w!;kQT<Cv)>9>+m^p)Q&+|Skeo9pF2SEuLWMy+UP{knL<N=lS#D*exLt>I@+7w@k8Kg%b5)cyDS8Bw&y8u)=!Ld&|mtN&m8+s;w2XXb0jXL!u+vA~r}-OWdz`CB9-I)IpH@<3A3*kmD5jP`-iod1F@Uko46U$}U_WTU_HFhVP?88lQ|P95}sDXf8)glOd=(K3PDcndjVlv?EpQvg~(J=GinKFU6iZ19I#_W62QMU&6|$D}m$?^By9Mt@^YPuM8Y{H;keOK=nwnk^B?678hWUPm}4FbX4!v=T^451j|WjA9UB@?3Llu*95kEd-Fvjtb4ko3^>O%H~k)Vds6W&68(dOCzpV@w<hJaNkbsZ@p~u)}%0SZd7Hv6dw#^k%w8SwVFNOo+8Fv%R-M`2)#4tf-Ujuq$X+xz1B@o+%y5-81e4UCQKv=JyKMfR$;}7=a5^+nURcf8&)uCp=QC1Y8(uyl8Sm0d{c%w29a_OT5G+uYUH{<#<X%fIoVdyG&LT-UW%3X=jpZGo|F!u&0-PfcMC<n%@YgzY!~Wor%x4bm*$}R)BQu$?JP;C!W;0oNz^=^l>5A&6Ad|0vC)F<Z|tj$Xk_f{Oe++GAbga%GOcqM86l-|!P%5D#Pw1$&H$uPL+K5N4un0BBH~Si=CWXEy_JRwL-1%?kFRN6Zn-)qMK}6k(aiWX`=+gec(IXP@oJ9KniNGkG0Xw6@U(rs)_KF%E9=bd!Vb69a=Z7Ibq-<X;E+iv{mf$m9k6LmS>}Axo2f(odP9)%byH3o$enyt>~kfi$<0yMOKoJQ_J4LA{zJ8q)_jlT%lkLwgb|AAQ8ML@SqVM&1`ukbzyv@@gvU0x%!&vUwj$;Yg@$q?0j0<UXPE-)D5U=Ic*(i~Rl}!#^X#Q_B_ICn8S<KB?QrHU+3%R3KY!)?<h%>hZt2a5?|4p@%sa0{Lb8kD>rJ}XWE))LsY!WR<?zLDN0+W#T0F0Q)1A$py_kT@&%sKE*Arf4zR;O`pJ(cQ?vg~!CBO=ziza8YA;RiNl%sY2ECV94{oa#x$uF=@{z${(efYgZ)b(>x<2)2Lra@Dy|K=`-75kNmw&&E_m*!ZTAOG~V$!lTJM*xzAf?S_K@e(|7q%08ETFUUMv5Ex2oCysqK|(ukh*lgb?mTyfOAQzfLm~?Zq|_OYw6SAT&R8&dp7O!Zy<oIe&?1lP;5nN%i&MK%WBiRq(4!^FeAGhzzOKAe6)@IsI(VDQSbExX=?U_q+z9Uk=b8v=EHgwxP6TEObL4qgP^nSDb%c~_E4ZUfG9(mL+%S!V2zHx{W&P1gAL_Dx!n}Q$09q-y-^uNm<?m@o&)hq$)>--RewWVw(bT5V8viA&F%m}+3E(XCo>NMK_2oiv5p%4-DeRb#*a?fcAYRK5r$g<n<A?~`sX0@u$*2!^j5Dt{ax+?ygzM6oEq~kq8oZH4w5<fIrIi1u>TY(;b{S)EFXL~+j_D-rSE*`0Eh|@xVYg>Qmj=1i<)*9RlTrwK(B@?aMX)}DyH)7(ddltP@*;Wy*WcH|T6e^Hm0KE<qt6B*00KP?fAwOeCOoi&P=TDFKpF!j$2vH>o-m8OV}?l|rRorb)M}~}_l!bi9kARWLPu+Re68(AiCi>!g3~(lUv_pZ)%))+%yN!yBqY|22F~9K+zm=do$XwGa@~J>8k5z5tTy_Be&4dz*W}hJ`BMu_$hEFHwXr9qD?V;I+iH5^QBD_cbpH+u$7d%-i~DTlC`hq5Qp$3kNbMjB0fzEYd94i-jC(2#K*D>7xD!B8ZIH7BBFCxcT!CPkG3OSJBpM@$#`7c5*e~baP64>~t$zb1zLB18pBxO7;(cLMUz8-PXWDoaJWI};WxzoVA<)<%gQ4+QJI)!?R6-A}0zd<2ltkJXK%}+y7%Py>^)Y*ikw`gkIeD9Dawe&f%Bs-qUucv_x~9aWF_M0xlgsb?+MS%dUR=p!%fgMOWauSx#ZA<ta|o8+j6SFhw-z%}m+M)dgH`M9GkKmDF6Q6a6BXfCN}wD})qf;Hfa;FjD6r2*PuC~8(U`RRos0Q?*-=hXi9(yc)-3!nqbo9f11|(nzqfy9m%@BA=dN4|zngEd>-2f>Gh->(kwNtb8`+qYZgq(^4*(e4Mh(0-hC!_e35L>ZYNhm!aD<E{TmvA8kQyPOAS8ehBee1cgP|UAHcDyf3jW($;hwIz@DyjlEBtk50ZTj?7r^j?RNYJr9@o;F-J#PIdwDtSf6?d8Uhd4JekVqdWqQ2!w|{%)7k~Zp*Iz&LpZ?EZyng0ye*F4RfAKdzdhKt|{QYY`dHu{!Ui;b4UjOl#pTGX2zx(-*zt^AR%4ILkL>pE=f22_fvrxT#rY|a7=;nX-C??!mw}uy7O`bgsBT8-fS>I(={&hUj>k-TJUta5noVD>Wvdg=;$Rq;{LkTb^k{}+|s73JVt9kiYa&0N#+yzs9>;I(A4DOQv08W}q|5u#Nr!k_Uo$%E{q*q-gW3{XG4b*R!51n#Rt}LnZ`s|xmE<|&hZz+GaJmGhr4`EUQK)IDLL}L_%1se#131S?3;32R=6Ys*R?wFxWI_rrh3{mbw$P(f=FU9T?@X|kBQFi9J4mt<ncQ*&8a9xchnl}?uvssvr+w*3kx#aW}cX{dDuU|>;NiXd>8#!d0hL;}xL_&l(5{pH+hy;dnXJIfmmrwrJOCgksgtF$GJ52#U^=ox87%KJaw@!mo|D?jd(yg8bbLV7z^s|@dz}coJ;*&=YX5lWAqML|Oa~N~SETxP~?U@Uo+)D|NRRl!*y^r{+;6ny~4kF7a(<InNDwr_f&`iXi{@pKTrr_WG!_R;8`k9~q-T!gs@BiWF|M|5az5c{0=%XX`9OhsrBvEo=4?4Y|k}JzpfT_fAqZU?zOLNfK$|S;)2_c1q9OLjU*&<jB)@ANw?U&f$ltl!=7h%pCK|S%na%&BcRvAUK2LubMtl<D-<3n6Tyn@OJi4bLiLm6Tr3H4N#{o7u>zeCHJlPOPHxVJhXs<p|Kyxu`p@i6nS<PuUOSTLGYV7f?lTe#o6*d;8w*ys#w7j6#tp<K7EkbpYsfWz7=LV@B!8ORy2R#<G2S6))&DAtz8W~Q)yBju5@&@wqXjzd$K%Mb^dZvrs@83n-tLK5a1i|h0Oy+(u>I*hXqL8b^-%t;-lA7R!Khb_@qy6850DilCr1`16X!U9>0!;=tcn2Rg<X!z>HyVfGMZYSjHzNa7&kl^4jkq|nwVDSis`!RW#)5AFFFw%k|#*kE4Q4uT|a*P?Eb<}-RB67J>YEcZTsSgt^%u*ww-W_4Y>Cxm3XWr+Ph`GR5c2cX}fo32;JghdDFcus@!b2pzhz@E51qO$5J_@NVg$xiGTsve0^ne-@q7oHiXwylrM5#fHWe8kASbZG;l2ZYt54LlhkHh(x*d%Wk)zh9nvzLiJ$jn3-U<Oeziy?##ixLoFd1eQknw%fXqz(yY(gv?L1j&LyX&|%S5)`Inn3}Pj9Xf0rB#MP4-0Ao~#FervD#NKUE*bcRnMC6Ewfj0L093los|Ru#iKIf{uv~Kj!~d}mE)%hjC?&Y`hjAiuZX5`~2Mr#f6h;}0d<c}ar<8}kS>+YA1Zt(E4Ngng!NV^TVMjPHYBZ6<+4#|rK{;vQJvzwz!vaDP7AQ(2A;Gk9ut;8gHP(wRy0_d?eDUu-WpHCapEmUD(hMAa&rSefhjW65DJ)S4`~U(L8%}{l(ij?bH#G^N5Wz?pVW1ICDkdopdmKkVFt5-!!Ds3>x4Yc=)`2oVD6D{C<)acX1~CiY=HQXI5QlRD3TcHx6k_Rp2zGohvr2nJfOke=K{)ePi^zu%+*80|a7s=1IdTpZ5^!AVTc&{J*n*Wb35s-2MBN`?F0v4NVuUf|Av6V$Qc4!VvoG(E!b3Uv98!p*;9+V&sn)@}2F59_JaCrr;D2h5F%LgM0f`}DNl-XAqtY86Rt7#=znNT6BwRm;>EoD5h(vIrFd@OEloCeMx66XQ%jg3cN+`vlOqf$?6yd?WWU;|lnc$cx8-gB%JQRovYl9ySpfijr=%f%lEaqYLCdfgk$XMv!rO>_JVflmcygOO!8U(VI!8CthSkyNUSX`6<B0^Azm^hJ)go!H_gI|e>++!s?%A3CcWyP>)C`ChvB&<Uu+_Mc~Iib)qjYD+gu@M@GFiR;Tn1zsv5Kl2sVRc7Pk7jjthHfl$j%94vJDL8NG%_RCtNM3R!+CBcNCL5tOomtp0z_$u6ZnDcD~9M$I-*R52ndI$9Se*gq80~t6FHA#=R;T%v<vPjjgVi7xpPrf6ltkE@*0MTJ5Jhp_@VH>%?iuE-(XsclbThw!kz=nQxd{biX~wb2RAhY3Um?D15eY#iFad>d`UuWCC1t)&qVkGK|u}YUSR9BfFS%^L&?2o1PDzXvxqqxqAWbdOLd0c?jRdq4W{RTQc;w_U?D?Xz+kYL1cguu!e@__Bs-K7k3(n_#31~@loJqP<HDNF6*SNS;HY81BZsMS$|1;H*bX#wB$!L%@rd-Dnthvzh4erbA;y6qL|}q>m~{X}sM;?1%w5JFAfYJ^XX%I&p(T(~DM-BZkXbCkoHYbH2Q&ecbAqX+R!G9NjHxB>0I?_-QHAX|O@|?Um0SDd05g>`&RCdvD6G&Du?X=vFdjXG{f81+#3_YLBB2oT#A9IuFvuDiBJ>ajDdK(P>LQ#o47p+wQzBzB(_rWg(qovu3l=!(N3Jx&eUJKv_P*90z01#=p8doW^+6X6Wj5#kQF^1&Ic`(|lXP@C&zDW>{+Eey>S5Y<P=K)r-a%~6K*0b}Nibdni@~A8drSdfOkRKtK>QVO7MyY~rH~E)Wf-+g3o8^8PD2W1<Wr)?Do!ls#=x*$g<rS~LDlG`&UOUdIe-C<v{fNQfC7aOi3Ec~(c(TlQ^DBbG`X51h$TW?17o>Y0&3<%$j)Q$59Z8Z7a}300QVL~-Y&{LkYOm0o{UTWOynN3p{ZlEvyw@hxKKYGXdW^UY0eRVBE0+Pk}mRt7B(?FoQ^0b3WOM4McfDj(6M0gth5#dW5+y-DNqWWv5q;(jOI!a2p9-6R~yU5exO+&;Qow;@@i#5d)z~u4!A-S5FAw$Y+VR)0U(egyeLBeCB$B{^nlS9VT~3-GR?hY9AlwFT*##Wju363LK-{B0fW{E9oFfnL}0Lp1yBc(4zc~XMz{0wUiB2Ob)MtHhPosR`W?V-hVR265g-<#gU=W~6k0_40NG)DC@rP)VOa*o3n#5aSXqgQg3f3Nki||g=O{8<86H9@t%(tyNG=I+ng}ya#D-`xvqYqHW+Rc_bq6IA*_Da9rpW_kJ0*#Yy%GTtn<RM%X!qk%VtFWw?LteaB?&Gf3NsI>uvW9+Mn<-6Dm{{xEAAZ^&f;k8IpM|vZbIy04d?bRy?BGezj()N=bc?}anG<gQ%Gp=9x))n?*o(wjADDPlwgmGGs9xvYx7csp@hwm*d(&pOfSBdp14-;<><Yx-NG`D9SIZBI75WvND_c(Sh^YZ+Hyfan5xcO6k;LfAPk$sjOM`EU^(3w^2`L;cXnmTjg`y;p*cg~O)<k*hDpnD415lr{lw%@hT@WI;e5mnhkzVQ=3|eqkEjm|XhacU1Pv!riV%PY=Tceggl14X8jR?Om!A#Aw`;jPE$hn*lZKq*AXq!faWteTd?}77!;GK<&r5VTGelBKs-=@I#5=)}q)s`lgn)`mfTfVyg~<r0R1)K0F~f>aHD@+>p+X=H#-#i%Yx-9S?tajK8wcx%f?W)AkfQJp;frBe55W$8Hqs5i(F+iF$1VyX!f#GOpn_NFwZar=OmU>+aExoholrJR(eQtUOA}UfAFL-<RAW6RhUNl~uV<R%&P*P(MuP$4kR!oBWZ>X1n26|Q0zmfYWpY7Zh?%IA002`ZoUxjaupm>Vh+(ne(okh6fG9*f3`YS<;-PfHKow^s^?nSzt0CM@IJZA_N%EDf$(R@E!n~N}%vX|<!t{S#UF>q~pVU-Rdy<VRDz@(o66wnBGT0qL1&|Q!s{(`$XA@Caz%?=^tn3^j8`cn}6|p|V;^Ds#HQw{!X;Q8|a0);d!uK%B5n>#WWW>fl0ULjJS4{jujgZ2;-FhmF?wK<1j3X=&CLJ9}1k(lc8?5lNk#Ir5UkEE~5%eKa_adw|M#dDVgf_Sb-eSv1u)4rSHZtKb;ZdqHY65emArv4)Q)A3DOGuLp?sgY8CDrj-G|KX0;lCap2pLa7M}!d@goWq=jDvR`K~4^%bydg(=YaD_sR^;hjA<Z^)4?gQJkCjvI8Z+PX_XV+bifZloZ(z3&qg<)jpq+U(M34Kf<%%F37BL|0=S=UOnx}|k(2=(i{Nvj@H5g5QjCyw0vHxc<p^oZ2;&6X5Vm84DKa1;|CPrcVQ9xye3$6xUTO+DXi!8T3+9Ru8f+a6|1A@A4N&%Mt8!S{Uw~CP1UQaS!kDE(ix8A+&AAF2Y;66ZLOE)<$BsJ;xn&_tw9r9LJ!8QS9ovCs=3r*A-`t(1b+6q%M20|I5*Aiqz)++Dcn%a@^iWcCX+||hMsZ|f{ZV106tX%*GEOMsF;+gTu*6vCS+JEx8>($s8&Kp<AR|lWTrlO`1FpgXOU@*L2oND8=9IDn`+gK2()nkCA4g-GMDPMlcwA&Ok1W8<OT<kGQULc#6HB#M5(!Lgh_Maz)R-qhGseLBZdVV#{XFC5MH;*#89Ob4VI+VPMnP;aXKb%|ODO4!z_o4w7Tj{DUF0tc9wJ9TYoJ0v7S>)Id}9&ZM2x(5#5%w<WLR^LC=Pxk6r)+3VUDRXv<dY4V7~U?R_&f>#SW(xYc`8`S5~r}7OHvJU1Gb5%U7$e%>jEbVTj8q2yR#iRKsr}%m;DcnkGA(6vie@c^?~pG3Y`et*B8RLLO#1=8gon!3%1Ej^qo`I<sKOEwBQG-vu0*yLRhWe{K!IJkt0=41*!Y!DGOLkVSdW6cIn1U2ld_999`<xe<~=8^R*znAc$zhCopx&7px<%HSwyXte?ohwaRU8H*tu2|HWOi{2LR6us7996u<9p{*|M9>u-mG<{e`-OENd>&YflTn0YqMYi+)gLH8O&o>rYB7ZO<VXdVM?+1+x!6BW>QZvh~r^LsEymx{}Vq_3iT4F(1ddggIDh#q70fVDXh$V<dhxkb$*YSmZCU5tvT~w`}3H`|sh%a2?5PLx^km$r&$X=y{AdyD^{b{(?Jh=G+X$yss8bd~WFuOFk#8zpbr8d$F0IUr0fFaOW989zjz#A~SzjOGN4veM?k*|c90Y<5iF-Z!^6uIoPul55}^Wm|OmTL&9$DT6~=3kTw@thlPgeFW|=Z)ntX^O*)Boc%`OaZ4tU`C`20ob_M&!z#^7CV{Fd!4{lT&`DYH1(haM(@}CkJJ7G<*)=7BB~tl5HCv+0Z;p>B!wdn<qVZbawW0Wl!bK~5Ujn6Z38e&S;Me#VcB+CLGL3)B1J+8hQstFVOHyrgkiTzQi|D^E5Gccf(N;MV39z|DdjAL1;Nu2!BPOY$bzRy_uAKi$cs=_vSF=uLR-OO>zcAM#OF>JZ!N`82%(4+#Dr<<6{dz*<_UD(3vFENv>Hw4+&Jm$-QuK;uI*?o39~(ypk7c3LQqdAKq0zkKv2G)*e96Y!)P}h02Cuf0kzx_$yo?%pukp22o>UrGiV_-z=r4tAR~hx4!L*OF^GWV<Mgc4bF&G`V~%rDcoo*ZEv`<6DC1#8k5afK%+D}A_p_oCf)3}*3{ee3kTHR|awJSX>8+83JCC_V(P{$8Ttq1o7yMC+b@;=ph?7%#oS9QT=hndpWfn>J7C7qk5XcGOBFwl&xDVZz@Bt6wbn{S1OC@DSDq{pt0EA^ame_S9ga~ty8pNr0%DNEmILj?XN=1QVW6%hB>kZ+<&dMe1@J1ohTE80Nm`Rb$dRPo)OWbv@W-2F%faY44y?j5(4L;1|UKx>mP;D(H5yzN_@YS&pEHY#42OUojCw|jM<}Kq9mEHksNtnJW#P7lD#DoMl)Cw7*Kn*n&P7xF{yF`b;2oNK~bRN@1GY0gl)b7y}a5tgR{R4&NGL|ec2%aTG5Js`RgZE_e!|7~#1Eo}v?}%BBjkPxX{5+6h*#;65V3C*F;NXEO2MKZ@Ojbvop^9m!MyAdwL;G=>eFtk3LIA~bj^H5be8UitgM-7`$>D4o@`iCjSd3w?mf<D=_1uK02RIA7(o$IxETGfjT`0qy(by_Rg{2`%G9yRfv`cw=KG*waO)&I0t=`Xt^nFnfCPI9Qh@@fB#tZ_ouTWeZPU~n1a{_7}Y@-EOS;ajhT!aZ4=4S*lF-0tiSt14{P(p+)l1c2SrFs-+-3|Hq)nfCP1JQ2_AQ*ux0zfgP2y%RI@gXb@XHndH%fj+21ZSEe7M5XWlz`T9h`nP(2}ul>!XrcsbOsU{3z8l~>4g?!LVcHn`t>fN*(|KacXP+ufsvrGwgVXl5p@(6G!O~n#3I7Gq|<lp)qpPolOY6*NP#$a;{Rvw-Fhs&jw`{xlISNJqhstHvE#D)ZGON!O=F%64E7Zk;GqpmjTr-aAZ5q}ffn>2(6nW?X^l%&MY32V2sBN-kbi|}U}pZtM0_XD$x~UGRMq+D#!2-@Qt7;WMKd$wM64B;wWLq)&9Vp!>TIpIntO<?8-jFYB%K}N7&ztx0e4mpTdsV?Gx+J>EN|aF^e$hc+kU!Ubn6n{7efDeFRHX3Z{GJWy!kh*+5y#M1u>$FUM2Lg%KE%4aEWO-hL+Q8h|+X&((EUhGzq~Ha?~jUDkhhq?=u@^>T`~arHveGlDhTl@0i5j0bvhk@SiN7I4nzhratLbt)pG;7?8-9B}Lu(mCh4WB)rHd?Z&S1tf}!V(czM7z7OE&&3h$rx$dTb4UFMKhZIKYK@{4MvRM}0Wai&25%}qnHC=1=*WW(<yL&D>Upw+rjFu(A?Fb2oBJJ=D5_O68t}#Z07ILW}MoWyWd1#r`q!)-2&^~T6xxHIv#|I2Wd~3T-<4_7);GGD|-gNOlxw<jUxR&95zU0hjd$w;M{PgOOCqFwh^X(6QxF3DKIN5D}Z>0)tRE!Xj+)`8hI2rk@H735y_@!C~snvIkLuT=bJQ>GI;Q6u4z>TvT7h|WXSnL%%K~bepGbFIZ?=gNK9Y#zaUI+3uKz+8n{c2Sk05Ig3eDF@1k}>$Si|6#ABwS<+wqOLmv|_KYJei^8iMiI=rK1|rufr|sGbJjq#5kECM(A3K#pb5yc`FcqdB}g)8a4Y7=06^i;hN!*1tJ>@&Pvf|)Pxe&-y2<|SF5`ympKG10hSz+Saix9Ed@i4poSqe8+argy$RyzGt_`(<dK1U99v>M#^bN|Cz{U>OU1uf&t-gSlp+U=k}x2rvjF9(C-wyeUY|lP1s`jT)T}zwY-FU3u^51)Xm1DBI1`S{J-B0r?{<!wN(>v7xyv7T)#8S8X7ZQ2au(yih`V?W0L&NJ-@%abgh`gcpqJT+n$8?FM4ssG)IH^q8kJnhm*8AryVPc(#(4(Xitj5;o+pb|=ZmX*4lQ3gp<HJMoV+|)aZ1dp65)FgsR>yxGD@)qQY*5_d+j{?n6+vsB^T^!ps*TAsR|LMHZBUqUZ)W<YiIitTa><BlD|4QUA}YTNbvS5mMj8Rks&7l(7+IN$18*JB6m6>Qh_8w*>Itu!P!;otSvMn#2tAkX_0QVI8{U|+zJ#@8#9j?X~eBQ-!lIF;2_)e^H;5e{&rcz+;3m58`#90ltiqUTWuo^=QVL8(?y0PH<}|S%%#SzgVCzEB!k8h%NQX=dwGtmy+IiEmO_J?w1!Zq*3F&rzQc;gOCat=YMvY>Gi%bMAxZli6KH~r#1@vT=Z26kv}BLc=1j3dQ?x>zowXU4v5IXx+USjI9ny>uvwI2+vG<21*NQ5vn|REvpnUI-uXTsLx%l4IiEa8XmJq#GMf&`tC4LyJ|MB8Uy*HYEdN``Q=lOM=axgOL*(PW7CHIh5Td!ii$c6WqqM!4oW<jhI2(W6LO_VYv;9+>QmOh{gyRxS}G+fR!7+iPZ__A#)-LtK9$5GoSw_OijT)oCk+Z$=Y3kJ1>%)ZUR0X6O7&d5W4ZArToj&J|{c9@bO)#xst7$T8p-R`a%Qo-_bEVW{dsifRE^)RZDr{GKTsso^7*fR1fn#YU#<@F-R8eE_nG-u{W5P~62*7cc_FVeHxS&?rNxo=>@87S)rW%fE|YsTZbkj|25&}_`L7BT=!b#`AeMA#PTUbP}v<NC`vW{GOth_V=4K*3vq_mkiA=nvnT!+xA?oKf2^Q=Ti5D;&<jgCf|=vW{F5DyM1WX($9~MaUAQCC?T^$5}Gf=0-+5X;YfJOO@;6RlY(Y2dmB&dH@lOHl5{64HtQY5vgjkh)YJ2q6^MeKMPWxD3q%!kgEk@YhALv?5d&8G!0*dHhK+Qw*`zluFhADdwer8CUIe7wKPY9dd{9?3Kv;@IrI=y7!eHYwH7RiTdl1b8V3$DwI~)|JYiV1oihfuByD(5L8lR$t@^%PeSfx4kS`9y?cW_+=~st!^$!+G^XPhyclpAR;yhiRc(#z4XV>=`*NeYvz$6%lkrXqd2ymvwH;De@e{^-gzZ!ow-tynxPk!Ii>v~eaa*^TBqm1ctGs6hGVan*`-GFEa-HDIZ4wVpU0>6FBu~M6<wmB^Q%H~FR6Zu~(0spHdn%`ac&)!h*-DBi`br6-?$Mw6bq(FiNW(uInhR6U7eV#Xizs#&Gk#noD&tRcgV;si0J^S+Fzy{ncIs5U2QFM2{%7wYlnhY+aRBGG8@)ibo_C2a`?cBUNTX7_-fhl|IB%XITA(t6?Mdy^pwbjf9HKUtgDPfdAox$K~4#v{CrrBd3-8gv|4KS0j^ljVdRJhRd-S18m3ty}oS|TR~v|L9NtWqVv*nx9$9$aYPm1amA!t}rFo1~P>sOapw%!!4J5#^{s0A@3it!kh;&pBu!Zb<ay#P@vNetEEBLd-_AnPKsCLAVbw-uw7nU88#+UePy#3mwBtA#uI9+^tZ@mJ|&~dfOsrA?P``(5zx}H&$6`RwxpsmSy^&-*qGWKd$TgSNALmzlrWEt6Mup!%q=71j{OxN5d{hdb2}y;_v{5JT5Qe8>^+qT<jN}Q*#TgY5(<uG#4pi0Mgh@i`jlKK&?r4pO=c=iu&d)a383AdXJXJ8-yZCP6+l>R}t@_qeeHKHU|DikqA*9zQceO_LzhQ6c)E&jkQ+;n3gB8&kpJx_{DAzQ=z~m7zfvhYDCEZXqejs{r+hE^y*}-(=`#UPMOi#e3~tfCQVk_L@1C%H^^`gp4XMpegIiiL-X^WfmMR>ze2u3*NlbhEKu`kK`T!RhEOeQvL#j>v9gu)(jaXkpqI`fj}iuZSGMa9R{&P`WL;W87?q6`h~PPt9QYy+fOK*!ITw%`re9$UMIi%?RZ{E)wbh9%n8Econ&hn2b==FqR?=pOdB3xO6ZB`-oyzZ$)OdqzD8kv-z#&rrF*p?$$7&Avy4g@3!s<{40O5i(av-h<M94G}!<ao{njy%P5pgsv5DVl%soDcdsim>8_1jtp*XeLS|JjG)*GC27N5>ubmAy;*WhG<Z_RE26h$^zg)%Uu3k?QR+C#d$Sn`E)aJvu<C!&Nu^M0djQ#rHWA@u<}(kV+?_8be+PdEYAbX>h*3m&M-eX#M(-G@l%%ldCK%Sr+MCjqYEC_Aa02ZlRYMg}&Yf!F_%RgV7irJXi^h_jR11N(6|h2IXAF%(GQ*g<`@1u(i`q7l98I`Y=%3zyIEG!1$X5N3PO#K_lTl@|HjV&PUT(vz<g2sqK<wDh&;Jiq&Oeqbt>r^8m518(e6XSw~65?=G}!id;%BfntlJZ>PPl>KJ@@IO2W(WKI3WQHOZ{@y`$SeP#8|5z(+^jFv?>g{TVO!?94f$fL@fc@C*o!=-62QD(7dO`LQH6&;zTRr0_(5iF}(JvR_KV5qoU_S{1Co{r@wt5)HJEhl2kB+5}d7}u*@US`>z+WbPNYs<45(Ppa)$(UW9lbh<8$*p*Ut7FX|p|qMx8;#3MDQv^Y`!+GxjK4FeRV;T1pd7#$46T>GmdiYqEV!5s8w%tyh8A~4uoqn?%xr90(ai$l^lgBcV2b;w_=a!^R@Lj+IzIQb!jA$lAKhUZ@V5)ixt8Vly^aBI;vkJK(dKIm%)yd~D(-NX;M9idJ6l2$`2n<qiq&R=lEuDHCAPn4aR(^W7pd&hx=*#edp>)1L&{J?pgMC3wQ=2Y=Iv{;>lNWgdjpG)Pc(qPJnfq=+mtJXW1{g=viM9XiQDtidf}MMRB<bcE#>G#8L3v%CmD3qHmKJjVPFQo_$=7_#5fC#AOa!Tk0R7An;Gg$CTk~JtBbJU$HxQQD_@1b`TBK_{p!#%+ACY%7*c+<e0UsBesy&tkK|u|wko(0Lr_3+KF@*y@{Z2vFI^_3FIuoQ@8}{VuHgIW#^Ptnp+zX&xKlCKXk0O?0ofTxL8JM>KCD$Y)4-Q}roY+0-4pi@P6Kk4UL5l>5H`-_eLr<ujB~ElE;0OS(iWM-s)kj0gPvObRA?f{+Q`+_D-~}sF_cu;%Kz}itD)WUFw3@F^eRUE$cWO_?rM3*g$;hQ#O1NS^v%`L+49hPI{C7H{x>V(YV^HK3fTx&+8YG;K1Yi3y0+*sCi(-|M>0&B!DpKdV~ENISW{Aiu?>a87;Z~cCV@8qcxcK~bB!_^7(*Lji|sox!=pX1TDx<k!A%{k?D-8wwsfPsi#vA_eY4FuQ2av()uIMzht5<|YoRnkDo(us$6&Wi&a<{6wa<}r?wq?C7|Jmu2)fw{-bCmp*CI;Sv-l@R-`~A|`WoAdlqI{>L!`juy=#)QB<JWNHQOyMb_hnNavZ5-Xd}(61S*m=+rOzt!qBm13075NcU7+`Gc*dL=Z*c}QukV1R=zw=INx7_@E@<8e-}&Ym7`8c6u_u~#B8h|k#gMKf8ue^(|2BY-1F=jVGxmvJn(2^Bprs3q6Iy=b1SkjF@piVUEiP#7BV0)wbzM~Lw65siI5{39^U%2_uzy*_xkv-4z)KeIB|M=boEF7-+%qDKe+l+d-k99$M%2vAO862e_Fb>-ZA*V|Ii=(zkmPe|M~R&{bN5nNX`G-dVR16Ryrdx1Nz!NBk2wdHzU5;0SScs0Q$N;xmGPR8gnx?0CiBywfYpI%+@N!it3+r%^NT%#uY@B+LSQLw)%V%*85GqFRoQ?4|9h-viSMoW`5rhTiMbDBXP+f9)Te|BXSVVF+{N6;w5I?Zdn#<s}Znzd!Z5ZIJqd75&PoiRwSljfMd2Qm3!dUUB#}>w5_q+k!+t`9k|(kr|PFmUtvw#)g7S>>;l_cV$sNH^+*BXGNEs8QL6#oOhv8A_nHdzJWJF9Is_~A(Nn8QTvXD+GmP0SGa7uTGByRfm*Vk%`{7&tKmU2T8h`Z{KQ62AZ2*fpR;$kxqeiPqyRbF_kuP%Bx%U|}xFuW27^qpWxl~ok1~*At!yuyu?kyFuJm@XGMq&iC)!BLTCVKh4_^`M4!AlbK&d0tuPPg{U1HavS4L?7z_1O1geNFn&=`z8ilS7l6(d|<Gz1;8hA#MQgJs9<M&!GU5cNttN!6JeK;aLuadXeeY*RpD%sxkEGGX{5mOEY08_JPwaD?DC)F!?@fWZ*HNTJ=}4OuPmAs|LhR*X-|SjFu!9BP8E*i3B@2W4;>FWd_@@967a3&P9X0X_`k(vl4Y9e_D&dY=x4uRHZ;bRkA&<V>WEt5a2sBKtH=#!dg2H5Cp69Ifm%VE0+0|Lwd+SuX}YDLH_^-0Rv}Ol;$>YMX(JR_CB^*xKaiPWQZt$g=<O0A*=noM0zU?M&}KRzIv<t)uHOY*o!s((_!Q47e^pqZ0R3XZeR&VgQ=ECGf`A~YkYmJjb_M~S-w?`rZ>zMf8IGmKcMH_7ID!cMvrDH#*lKdbgMKlB(1qcdvtCF!?!M?j%teUEF*$%_6hZvSDzg$YreXEZq=3j&MkM@_u=e!12GHw(DltrJzr!mf8=K0mOHakkepKr1EVEODt$~g><C>`$!rno=cF{O`e&H#G8bFgQs=8W^p6%j&|fb5d|vgRty5j3Y;-mnp>gl7%#5oWO7<_i$Oh-=)6yRp!O0BNxin0jvc2qllPb?KjnKCQm3*IU3c3htr^Sjrbl7ym?rd`IbL9tDherEnhfBSq$nr1#+uMJ)cGo}Vh>96Kh2#`d3_co)7~gC!B}94vqoE0D4hgf>3|aw_;ZH-F&}NSorG#7}4P)gw&Scbl8Yay#=UxdZ)Gbh74b)SM*nd0hK)-$TKKkhUk!&)SBq<shkYuy~y`uY$-%H+F-_4isA!y%CvRqnV8sfwmA$KD^7%Fj+!Ku~OT-qK(3Ay^@-k4Fd!Vj@07jW6I=BZZBqeajA`WoeD-#+=|>UiqB?|iQEJ8}z7FvuSRM#~WrckzsgWx7nxGdbh5Z;Fg*V`rl}nvCdJ#<ZyP@Pr99k%SifzMBnRW<!<61*bRFwm3Y-;jd1vqVDakf3VK9Fh;+MN5h)QsyXvn5)gzJDepZ?g*k#JJ&h<(rHjlujPy_n(n$bQ!-AF$r3u%Hv)7Q7LyDW_+a0}3OM#z!`}EUQ+=i?GGfK?lW=24gzGD^do~Q3z#k=R(AN{5M4!x-U_^;0y!{bGsLd}-2H>8GEzB6V^ox`u4)bi-$LcT8|-B^_X7{g*k1L)jK?PW;Xq;_5!f_}Zf$UF6feZJpHIy%KJ9>~`=$Y1UUp*Oa%d(E=l!3gKsZs`boelVN8kw|$-x@Yy?QzA%~t3(yV4d_FnbCUOEilRWk7KXGQjb8Q)GK7OA=u#l3GI~q10#=5ehrJYXXf1b|lN$8W(`M^^E6jiMCs#M|xQwCpYf$grTv%N@%=#<h)5A{c<NZSFd;5E<JtzBk|ICvEDEaId-@ja*{P6f^jy|=kmbZyQPzt_i9oXQC=DdDjkTG3k)Eh;M5j%9QzGS5-^b!n58>SXl<Sip{h<>Bx<|S+dqA7qMBh-M~z~Bz~i_cz_zj*39fFB$#3+yZFJaUxS7mN^HWKvyHZ3ky;1*8-&G9+s@*wB)NAWSI(ITlKpV=I(9`reHSVQns`kh7{~Y85ShBq%v-d;j+|zwTJ1Iv|YC*RKd9i9vi9(3rRzd*gD}s`|QTt@HqddBKW)DtQt`7Y^cyadayYIFd9~_wJ@XOZJ;A<8?KRZl7#vFtRF_4I}(}c!)+JgQJ-<VgN&^7KiJ1QH{M^Vs4C_>^qGtP-7U_Y7V1-#aQg4NP-5C4PRH~0X3&#2sNpTqE-M2n|Ix5HuO?_ttq*x8b^-Ca2X}I+ens7=j?;Vi*#86cS%kf8HZ?2)$j3*0{I%0K&%urboY{PqqEiMPS|4W6NigV8A03v?!ig=qy%gvA1tC7qhU*?b0mEXlU-swgQY{QF>4a4jY?upoM;3IqSdV#EyGUoS967Fz_Y{{>tHQdt?+LZ{FMs+(S9NS%l!t_@sM<xtFCQ0saPx~pBXHiC$R3)>MIku$ja`ni)2(z-PppCX!IS25_RMX3at+jcd{bpqDZ6rbqUr)MN6f&X-&PP?e`B0)&5|=RQl<X6W_VsHeEVSzq&d^>pO=<)9aShO&4kp4pwdS1z<!sJ#e`n$_)7&WPlf`;tObWuO%EgP2=dLO#|K)LaJ4PYHAuMg>C@UF_BIq@6!;2t5Z$`uNxFST{V6BH-~+cf7y#szJ2K0KfCVYUmYBE_ILJ=uWtRZB>}&8SVUdZ{`A$jEFlG9=k#oa@tpJK2zZ%MTT-GnipFYuvH9wX>+h#LoyHrojS+De(f5(8AC^)KJ%Q9Q8tB$s`p&zPC;Lw4?e6Du7nQhWlDA&@cNYm2e1n}ak$8ta?cz>Z-%Kb1<pFr)fuU0W2Hn~>wnj2a1d4g&fcAF_M$3%&&l+7CCP{?>R16sS)}U@)CnxFl|2{}geSH{`dS2ZVt)~b1h_{!|zr7zTJzucEo?-p=`!Ji>sUZUUJ$WBH_=JhhNwFzj;y#um9Z3_RWdv=afig<rmOD_kM4Mymelv<8_nrk)>&e|6vw_`6n+ottYyNwYs?V2Bz;%f~E(zS2qI`Zha{hGnsZ?+SZH8pI)qXWbHO@Oifs2ex=GNWW)<t$^NwI|(jZ|3#y0vPltrYucw4p<iMeQt+QzV0{s71EOyjAXfJL{!#zwZ7CS&RWMv6n$8!8u8hf?Q<i6^spLHOgMqJM=NMs>S$G(vCs9^jx$X>TE{+Q4Kz4tL6hMWUO0Kyh)0Wue};quX(6M1eQ!-OqMB!h`X52U^<B|GvX9#h{IBz<ucAxqVaL=G%|V~R0yP80~)PKmi;uBS;gXXV$S12EE@k^Z%1A)&-wo8>R{7stU6j*M9m4kDRkD5$o?G{8ETfM3@Y$cnI3Hh=7Ly^4PoEFpDnpwn$Z!%6azTVKDkM&v6E2WhD5L7dUxHle6bwFe{noResRKGt`|NJ?xBYTIs4R4uTED<591~Wi~EVwvqRr#zjNpR75moclLcm8_elN!|BZd_L1yP)m)q!nKA!(Q+xPx&|F^6I2hqip(fE@6WvMCypY`eq7y0VpV3fWnrI4j*iN0DYbGp%wk^CT{nu?Y%CCohHba;Sjuc=ak@vChlb<ZB&^E*0GAK6!qvj1OQ3#MQ1Uw^)CCy#St3%3}I!&;=J_4bib<3*0Dm<HkK)JF`OI1I#6jU?rvt_&%ome#Y9Z%a{3ol}_2BDuRojBJO0FA@CD4@vmbo7BIyw-klkMM16Rs`04EyE9;IhRX~-9&wSs6hAN_ovIWo^(vr2d`wC+<r>@qMahuv$bpJ|F;fQGjL=@4J?%lpeqjIcak}*UWgqg-mLcl?Fy-mVriz1rRly2u#VjJbHF*fha>b5U=1soHJ5`WSa}_J!kV<Pa4WHi+3{p~M`$FoXL+8j)W|tudF|;5sTA$yUHn-8!C(9P;)3pO7G>XQ4Cmk>mXUHl$+yTV!MiZrs@c|5!1g$nFQYguiX=CsRj5RAjZr!4?b74S`SW_(#27-kjf#uPFb8Lz99^t*G>+Yj1?0ktHlaP|n;aBf6gC*FD%!eekga%cGYm3HhvKCaf<d`|R_^7=A1CH2J$1nzRm;;P1O~E*}mgvc?-QLM$zf}7CWXk@VRg0j;v^hpYTS;?*1bPnXrf`v?AelqEfx?+xUuTX~KwN{gB}A<*DFI=y9|A^2NQ0|eTxNTw7>wLZpzk{}`sE3v{AM}&T+aA^aCOL#R;oW)S*I}?3kYFx!!V+wF2KBloa1ZW8d+dJfck8)Sd!9gX|k^>N0w{|k!Lk3QzO@0*wtK0m|cg}i-0{rm;e^CVQ%&L<-q*e;pF4%RTuE?vYh~pTIWRMGfG)oo(+OsVumyfqP_?)XNBH|m4N6=MiyWUe$AF#Hneb`!9uXRNl3F?uac@e*Q6~&-jg9ux8UBpI@mG2cWA7imhbo1S}8@tj}XZbd&u5>gI(VFc>Q&S$Wq`3kSa5%D>hJ=eiCfFhV4@vdP41O)|86NImS7S$P7)P0u&xNWE9c1+2g%>8n-_V{@NAujw|ukGooiV)2Qdy+j7^VpkE!W7Y^IYYjQ0|moaoBQewqmMPF8T)Lo+I5_4^K6Km2K3a6OFtwtu4hC?k#gAAo)0gjzVUIcUVU>^z*#9W)=CiQry$K&Ehx1W#R_>R1|dTGk~W?juU$h|He>mE%x34p;gpdC8Be?tdddI-D8S&Wg_D7{#jF|<0M&J3vptwT~QF%~~2NEWN&L(Eo)3Z@vRAywIo%I{nR@Y&?Sl609?ULCUWv$e2bF)K*ydq6p{-`6>Tdego6oZ#V`IzTQtBw}!#8wxcCVVF@SlEo(=33sNRs$u;yjE&JO(9mF23x52(VTar5W>1fLkst0)oWD6<IUkRe_pZ;6j%Um7T}$Jv8ho+^wXj45aStw)uwKJFT&4_!!B(Me$u>)+(xz1A*bLU<45SviHDD?ziLo>T7Y=iQbF&nTh5E+LzLT4MvHmv6f-sAw#lP0iJiwS2@4SWchPO#151{^+mO4}eH>0JZp(-^YD=e%Nr{v78DmH53(GBGi%r3K2g3@anVLPh7ZzlissWSQVW8U0%4A!hGE%o1!7nAXhgknHo{g{a_Q`fP~ftCYHMYmKbr5mxV8nY$I2(cJu#Sk0T1orLJV4;kLO9%0sh^231rtjC$&WKUQgnJz$BtpxLXX%T?iyUg!5NNpWc!@pH;>(cc<f$Ww5hQ5rrqMGfH&Ba9<AzNcELFA<+ZJsv5%_lwDh)@=&@~OE94RFR1^aPJK(tD9&Oj3`GtE^)zO|)9J@gjC48A2D186eZov09(*}(#GQXN%dX&@<=2~9EAZJpy*^!=B8&>h8?zp#K@ll9nr7YzRy$IMPmtnlviUKR3to?a(>V*zrJfjgRnVf4kN!wih%VRgDqkOdni(2g8@;5%3VpymP9$~EPf=b-Jd{dRqRw$2MoQ8T$=r;%t<1Bh^zLqp6LDgV)2>Z|9-R-%U?K#|hth%;M39&sRYlo}GEMPIVi8KLTMg}ahQikr&POO2jmyL^9zbX2ByrSJXv$FCPkeWPagZ0WZ@-#@kLPTlQ848EgVY9J-c8|R7CTx5{CtCt00w5YRkjY*n)vw?@eY_U`~y2<ub$%bQ$b~WD-X{Zo6=Nh*ly)#HZ-%D^W7e7lA^5=`S%@?PZ@%!=6-ZlP<<s{>qm4T{Ct|gn3OA1Dp>^@6P5%5wY6;-Ms%1p6vYreQ%N|;uqyS;0PbB16f2i?@OVjCrC$(buq(``rOzCPra2f5Gp_HnxRRJ=a>ydDA_q3OR|fBci5T)p0(T-n=nzksM&iDFEFjL*g$>?{?9*VXz?_z=39-D>~LoT7H<7SMIF#Wqh@y=8kdBOBl9xpjn6TBufs5aNjTTA<Xoq5Pc)_4&Q$msckprnNDeL^WfEY`G@|Ma!kTv;&OtX8m`8hp_+Dag<ru60p>mN|PSCkto8}g1uTpm}Swnqa22hfb8pyL}Hi{v9yi*zk(Y6ZkblT{qhamUS;Xsr49e=D9HHt|4x^25Fm-0<yj27nW*ym{U^N8T~l9Ro}BvZrM6c4sNHQA61%K@pD7qYqlO7)&r%1^DH=*fEeVOY1-`o_a#j(nq#7l={%0a&wP5t<yQg1u&(n9h=H2t`58s-@egkd%-Y3`j>mgla7p!;2RMY5*g@)_%@D!sCWXZbqq&1|e_Qgn=`^@$_$b!))SN4ZS++G29nd~iR%?t9~tDm@{<C^V%N%jMG^G@HL0{dNrU0iDnhT$?-Qb*|63bhhojzOhhESbsDaFxymDU9Q3XR)k3g6sT*B1vRsm=KJuZcT{qU$6`?7!ZJH$W{Ua#c)p3OTtA?PDj+~`yz${avzZcr|J}|j|tt9pw^UXZHY_6*#=9=qw8$cspvL2eaYqCwWA*00<TB)rr+*2px--S+UJMOyvOU(qt)wdkj(fNMbPNP>#WfMU#9TLAopCLMlh`ATJ$AuKoe_&(g#ctt%g@|v`*dq8wN+=EN$kNmA49g*OAGSBVk%~sjRNi04U79X{QudPfH<Qrt89NnTJTOQ8h=bTt|$h)xs8gaTej;#(-G4Pb#D(+jA|nE?6^d>&mwpQlIX{!gkBy^0h49UITp9v|ogRMq@0b3B78|dNnYM$V<$N7(!07x^lyC1+-|<H@H?&3chfM1|bP@XfCTPT}p~`l-Q@Xkt}Mrx_wNE`%%=dZd4YptM|KCHxdsw{iV}0`(LfPvjcZfjB3B7$T6fWXRSqD<UXE9XGum%ydOLyj@~0k8=<<t8A!5HZN=L@8r=nP%Iq49+jUE-XvKWXm$yUodVu<DiMvm(o1#y?TBZE$+ww+k60?t6d;w*>%~!t6aMaY4DH8kIQ*5)ipI2*r&WR(nl)L2VN6e+V7_;Ae7UVJO1a}bMlI6?mp)YRwJl{iu2NPyWF<BYonCzDn@GJs@mnaC15ooZ{mC!~UQzIIW?rBEFskvLGY(}snx}R5xrSbqpk}{{%p}NU-z7%_pPIpH3iiy8nKKtHYLiz4TUb65ydVqcp!y;oodC_Wf-8y6iJxi>KK`t_KS!xJfF=jNL$IVw`OBjN|;AZN`)N^mSQwmKuWO9kcJlT>djl4zf>1olU18DsC1WbOoeE!4L%*Qe&xuJ@Pfq%D=GvafO<>6%}&~Al$aLKL5Jg_7QEp}}=VXUrUmnCse4LXb<R=-=NL{#k&<LX-&-h|<^YeOv$x1Syb<M-QnUmUmfK0e6M|9Ve@_QdGwmNNdjqhXar*eHPrfW_{46<^^J6=%_S4JW~pt%T2Hkdqo0MxMR;*r~-jnOQ8JDH|Op?4cQp*VF+gZ8MzPtMyM;p1uJ~aKjm4u^JsC=Xe&?=8H^^&>%;pI4I^IL1lLC6LT%3-Ej42ISeWhpyk>Fj8YPVu^$M&V82D%X>a`VgB{_E`*${9-&qT+C}%g$2r+9Qrt_3M@gl>nJBwlBgf-34fJ<}=HxV)5$RefHI;uKC#9A!Vm*l5K!5!W<N7wIGpn0}mo><(py;}eLB~zI%_8jKP$yxfZR=c`J)Tpc?O77Ta@9g3(X?(NIO-OqAo12LCjrLkBemjmiSSN(6r3fT@tp>4b4Hz+5*}#~f2F=g|YO7o(Y#PWft<3M)_T!^L>mEg3Z)vVcprjCEG=j~`b)3~W-N_k?7Z*9d!HUH~%a_|Jr66QkG!_*r%qd|v0;ABdAQH)xU@t6?S=D&9JH2ds7l#o1Pk%P9{+IGsKYVMf2sMc0if9a0vL#Z=knMA49Jb*_5)vHPNR3DLP7MSn##}lk;3g<ytT~4UL@^mj5eXf;a;!$s)QWuGYV|#L6_)Kj*)9L)#noPy^Pl#fYljqg+)Q8Zx}G8^aweme!MC)4b;mn8wW@lJ`mB^6LU}e~V%1qkB14WwmeWY+qbkdoMwEFZBg>_0scp7az``1na`syct+bs0y;`S#yTHi()sK4%t_?>C2#fZw@0%I3PVTI^gK+9V{aQ+!@&J@Jk?gy4;jF18j5_?=4yl9BtD^k`DEA7pQLLbi?R8oO3eD1+W!?h3CD41K^3kE4c<rygGAeyT97#PF7{cdhu+f9kK6+i@ZL5Y4AsnZ+!0z2ru!CxXXp{yTNt+c$lmufkHtwxdDcxRVQz1mHCAPj<SKf-k_m`8i$A{&}$Lm`^V$nyGSaKBm?Qo892WxjL+&KCY(IxY;LQz`cl-XV7MkdJuxfZM7ldVqc%%fP5;2OI?n<eFeb0(zi4DzLD{GZ<{Kl<@s{2%}MM^}IGqxQoe+vnFzSuFO*JqTP96OH7gxLR*8BEv;SVH=$#cApa>j3zn92GN_8icmCR87)C)3qC8?S|FxkMO%W=>#>Pp-l8_o*U1^}2+3P#l2gjj7jYr3HkicATn<Nx&B?WuTL@N}yXd5*)FvAApjNFEvx?PgY$F8Ya5GF%OOaSdm$HT4D?`7Jk1ExV4hs_>-jXf2wzhh)|7S(05(O~=<1(c#uwf-hRwqd#3zrzEI{mUW*3pQ$8aULNOU*UU*u>3XW0CF+p~#uVpk<k(qoquqwI$pP1MbX|AN^nv3V!zO(@(GVc7V&m?z>9^=v0M$&vy0>F8;xn_Z%j^iQoogW8yKeX0n8$LD8o+8DH0w4KY7_Hv!COfo;$-)llmMB*U64B@*V=2bbY)ZFSbbGP8nABZ&ne1em?3Y|Zo|7dlG5UN4&;v+%QZo|c;272I-U3c!?Q2WKo`q>C*7St_?Ow97O+30f=EteCKuoWh`<i7OZl57Kil;8L+OX3(Nz<ZTlE9jZm&p?m!%#+AvHuYE5vxt}Trio3`=0K_*NZfi&npgWXHOnzQdC3L{<&gZ?)Aq@!?NNTb`$blT0Qp}oj7ztX3j4U;8RsJ4G`t+l#LrLGC;e5E(ts-!YY}7nvP8vW%*hQ<}MD&`~J2U<vRBvNWrIMC0bDiBAxKpi&cOEIKQS>=`7=RX~Y#G##JXNJ)4Z~j5b4#H2k0+lv%3LlZ{&q36+ADK@c;gFfVg2#iS&(5ZpG*2KZwx8M2tEtq{Dz}R=7*462m6w}74J?)<y~~pLb0N>IRT8=^X#$qNxXMvBUSDOM?(n%BWzs%mBjjNzk>eGVFmrssUYE}YjlIa2SKi34w|FeGv)PeD+w;L`<ac_D35?$f^Yd2fI+#)0Q+C)B`T#*xu==PNWPYaJqHjYNfwoLBS1$2^t)5-;^VE@hsW!PpPhXDMq&F}P~++#1H0${Uo9WLWWn~;i2>%i5fd%HJ~cW^7NPdbq;pOv(uH!W*^*JkfHXmpwn;-Z&(tzW@ar@8y9OO4)={`(aLqPC0})jtZ(7_p@%+ibgmrIX_2g)0wQp3tSa<%IvSTqT|L%B#41Cu4<7FOJl-5Hv5NcIil0np}6ICpgrZJ#aBUgSS?t60GPw6!m%dJ%kOiQr1P1$b+;**n1_`!9OTn^Fp5&6xH4dBiF?z5{y<MfLgpX*hJXFHDBvPJf-;*|VgMsSC;78c~o9L?9*xFv|Kfh!WVJZqBR7Hc{6!r14m$nJhU5o(h#gSN<I4-w?HEz-Rj#vdGG`08cr^)C-+l^?FxdbcQ*5XtW&F(kEcUA%%Lys43;26_Ob$}(t!Hb!V{6}?QY7N1pUgetzZT!<_%HAd<g0b1+>!C0p~JO|FL+8?9yZ<pE9FAm2;H#P*8W3QHdyHjn1Z}u7X_8(eKeHSWr$hhyU0}9DSJ%lwySI=e6cvk;k^F?y6+GB|>jA?IUot2GwC0~IW?z1{LiN<NU*0-2v8F?i3?ty>KNGWdZt9vr-&%S-~Z`W*T@rU{H`uyDCz`yQK*Y~IDU#`ry#L>@-kOLWT^PwT_(#{3W-z3*^eh9;X7VKp>rdW^)8DPa!hta1bEi}?u$beCb*t89>*J&!b$PA^A0<nzkHvOh`_0e9#aLwjPk`%Jl^h`0J+w6t&R68zm*`$e8^D!F1oUJD+O|6{IDPZ)ySID^p5XYVNW0=a($dxpEpQFoPZ`kv-9IBsQeJ_BMR0s@cssv04ID&7^BV&HeNtETyhmidmKn^rY7@9d?>trD};|wkuI%DV<ru3<UXtdl%DgiNyeT8XmeCW>p(rG{Rn&Q8^wDjLSpw0W#joyS2p)3ZO3K)%Td%siswgXk)&{_x|LYa0aAqZGNfEx^NF(G92;b@h`uO%y%+>QQVnmnsYYk8=FP_#(h%Jk`k>7#XJdg*bdlzdum@#j)+yBN=Lu)ahK6zo=ybFM?s)o9fynsTE)+r${WpNZsiq%KC`s$k&YB3s=JglXHIyVV*x^|bu?eqgk3y}vk>M_ARrWQbm#A^3@nUv-^hY)KbsY>8b;Xq+v|EVPQw*}GaI>@Xn1=%R^x?8Wa+wN>|IwIV%B)oB0VHpV>N<Mc>-|FEs}^`RH^o4u&@*C%oM`RbrF;*yZvdP3P(><mnvLu8U%W`fkHlPHWZutLu*2)h+pq%t*SRRtXux`~H-r8FzmZqV71`f|v2S@dd_eE%>FdUiBRxjN-FD?-s&q^jE@#q3v?U}<6X(ud@WB-nk1UTc#QrF&^Ma^r$I#%7@QK2y%#QLtB}Sitt~i^#3{9q=S}Vz=#i-ipot^{?OharwX25*5h8nTW(|Gy1r38Qr8e71E0O2e8^XLg4_Z_LNHW#dt6AQhhJJsA0+4)NrPIp7qgl&3=wls#TL26Sqxh>ojd1F5gxTz^#(`490!YePP3|V0QPkdA=J$@<nPLd5w1Am;j2g5!?Hg5UEm<nOjUwW`bg()UwMUO|sW(EQ5w6#{bT<#D`(Z`_se3`?K%m7%1B>Mhqi@FeG=|g<Yh(O-t{2^ypc7#tAW8ra^_V?_6gnv5@yA1*_Zg7QjZBW-$Jp3K*pE_Cp|hzjBu@7S(Z5l&zNEISSvq?0vJkqDw}H`9vxl646d~o;_c)H6A6yA3`bD9;|NLParhOI_!lRG0dDss>(*jL#x<|<yLIclX}Yt$A|_wW6hi4)KTH%FMs%>t7iY+x5_G4QbYCz?I}nK?7K01*6wAzOzVj$!CrI4!Y!7F&^pyIB(f#Dt~qNg7K0UMH)a#9d<(Z$w2g^P;}&c$TQWc23kk3Bmq0g|0766N0RHno%U^;ol1I;)W0^9eaEj6ZnuQ#u4g}M9$|%8Vtu*azN{cbok}1vUFTE+7SJ~Y}<+Ek{cWt)we8G+HtVwqfSfeKUTeA^INQrkbuN9O*y~sBBoY3M7y*ctV>=%=<n(4O<VWwV?3#8_i7po=39yl=1+=GND#;dpNcjC!><JNdMz40}QBd2-3T$?TXWnV3Wny*fFERI`|dqLO3gI^vr;*MW`d$r5Bp~@6u@+C<_{E|bab5<l1Tw)771uVp+DEM^I{(%fuzb)I=9vc*6-!o$whAP>3dnN6dp&ERwt#d8g-o>3>KF<%U7stJzuMQSxpI`mtCs)7!p(Rhhv-d4u&5{5x$4HqF$vaEI7c>$Bye>~l=>hBzB?G3d)XAN3P%_>g1Ju;5E@k8H$xlIsfuzE@RwdtH4^leD0cA+f8(g^KNPSWAS|u<;%H(335`F5Cfp@r8=um+#GM8o{=?ZB;Ga#Ve8j_@FbVyq@);+_li3}(;j1uPP!Eh|+Cd1IHZn^Z;T>9eW9kws_-p?PO9IU<fJtRn3<N}0%nL{=x8`P5K4kgR$W=MSqYS}3mF}FHS)VTQ}uLZ&!_E8h)luN4=C?Ip<#>%dllr4l^k&YE{(@1zVL%IXnpDap;Pp?jVs@}iWoBNafHP5Q4kd-b}E5Xj~fml#?acu^L7%nn{Hc7G37_I7PeS`__cQin#*>`RntHi(vU27aUgv^+zqjZMeV%iRN?n$Fx9YE&ed(Oq)Xx}#__a8A#B}9^f<mk$VNnUp&J%;!o;ACqdxn``o(M2Tteg~5v5k%rCvm2Vu-h&E4Mu@o9-XR*{=QwoZ`nUDsKKl0Q^F5j0@4OwU`;t0e5}H}F@s?09xl@p#{$#l#p5<76kyqm>#+ED3&=cE#F3RYVv14E;=%?W|<lz$A!N|G2NG&@yqKMcmd=)l5uJ3@Ej~0QMy%6ljODwMSyx3WGh>(K4wKoPaCGsx9&MNWis>nqO55Tl9c^7oFhEi3~?M|EEG_2|S>m4nNhS~%58fOVoI|NsF_3K<N+F{e#{Eo8ycxhq&dMV!jw8ZC=wc3V$r6U0ro9B=;L-6tq@VZ<FQkI7hl<dnXln5FrE@E#67k%}T2TE%Os3+!8iv2Vy7$+~ud;0~4CDgdN@7`_%tvS?*DI2v`G@{K49An~LI-`dk=pw6fW-7_}3gd_~D<j61XhLdPf)!v$GY1~-ZylP2ol-ZBGjl_ov2H6uuV%hqo^-`sk7%_=A4ITU1tLT6HDFC>NYOqm`np-~4`J)qGMr%c1!&1+gt>?5ClN+zTJL$%99eqpl(4x&kR}-R85em4^n3WN0dyC!S!;7)H8`;39Kdb5jc@7>)?P*+>qYj+p@U_}JVSzzs^(I)j?{B)0zjkjfNWoF?nORIQ~RG$o;pXjG&<83fG_9C|FWdV_jk+a`|JIC-W}g9$nh6@jp5y4*7*6M2jt!K-Qi=a?eFex^xszPuqR1b7ta<Eln5cm-C7AZyhtjJM!9?FA@;#6SX1SyJ$LttHm=u(=or*OoxwOx89k&#KDnxvP}9~OIwjxEyx%^xDF26LN9i|*JIMDBy`eRYboII~NjyS*pq(I}bqWcW8I7a%;Tymu+FR;sHQc3KdU11s!aa41Nswyw1uRg@EG48FM$X2;HwJ#Mu8Zf_f85V>4kDPoIQQuitN(aCf;^REd-C;Kuo@+b%VM*o3~+@57koi}UE?4K55X!mp%#p8V%2?wQ%B}R1hZC5sZMEN;XJ}9srj%WF<{GBE6r*T!nUpPsy)!+kh<7c?MIPcFS~D_-xHCqJAXuq3^A*#I~zm|MwwRM?c!xFaQLxL_sd-bjV-rEMKbr&Em#eyj$jWc;7oI1#B517)c{lSDAl5JtLdi-fsYTX8&CH#Lwj?zzd4Z({N}L9@%ZXszIJ_d{@KcyRiAgbZd<ULZ|KZe)$WW#N==s;rW1?!TB_Qs5oSr2B!Q80wQM?MQZJ!0lR+FTL@^k#iaH_;beEPbbKcqBT)F2KQ5jM~Vp?W%Kw$@~)8s{N!5fjuvqe^CIy1G>Yl_lm9+;XUcIMfjcb&daR9t(cFiFX=Tg-l!fZ*n7ahaYLKq?;51d8@H8j(Fm+sLBi62sY7fq{%08EqP=@SHhTZ3cHbA!&AO8hT?M=nj=N`sF%M8%;xoxaHHA^XaGi_4ZG1*#aNUgPyH4uto!-fjMi^$h`Pgu6Ga0a*+#hBm;}L>1Qj=oziogp`nx@hEH=T1uZV28I>P8HE1i@a&73MTxHWtc$n!{A!jmvp_Hx2X1B+Rck3|R{p64S(thX8%YXdW=Xkka<}0CCO75|h(KWf+sEpz2*9L!u#U;NqN5T-HL9`t0dr~789I+Wy>l@}covj|cGM~P?W^N@xvn$pFwqF5SuG-<7V}g8<dWVT>BdtaugH;&{!LqHekN8KiV9PLjPnMLcJ>#<2TvH(X8qG0SwQUX6dm5@gUmB%f+!J|kVjYYCxEw7eMKM&WX%}}<-_SYO_}T;LmAc_-jh6UEr`CvuvPBnoUv2~$lC2JO(B4amG{VpVk_{pBB>vC_16~Tk$6tN>_?vGZzyIyy7vKK&oo^ri^xMZ)<{y9Z?Qeha?c>kCef-(CzkOt%TEmwNl}4ZU5u?T+ta-QkBwperiZmof8iOFsnEfiA1hLyU+ALzdRM(!gGx_14A$7)BId*83UD&>)&9~wvH}2Iy-aAe~KfP}0FCYBvVGO#8J6XXFDaDvkh*Q$Ude#v1GW$|&<+sHd8+S-8O{tPbX;8*!%T0zbQz^YISu@LGQR%{*;`jYqD4vGm(z8FT;JrBR#9#f<$(7sVqk8t@zrIB0o(wJX{P(|o^3O*@whs<wfzFuzc<+sIn5+83`+3ze@?Q*ue!ft%m;ThAVZD3fX0SLEzJDSM@#6T}xmLs56TJOw?D<hsdDSb36hboS!I{MViX!J6m?+^zzJk~iHF4exicRI@8?CX{)F{moW*C8R+B0Kf55WjoG?+m#&nS6Yn!ddn_=&x<CfdMIE`ya+Q6ODplg|+!k;`11W;NKZ86+5wM<edJCc~~GJcBX#2+rsuPmCQ$&pHiC%`{u7RApPyzvb5cgC*QN=>BB?f9^U+K6J149f0f3*Q$Kz!h@bQ90*E^qUWSTOcyyB;3}31!6J~1TiO5Y#aJXv0lysy)e@i)GCzY2>_)D6K!}|R)iCVkx&`RzlHYzu%J%?%xdi4<_hou-hPkRHOVXT^TMuOy7tR>dc`lauGJ|l`3V~zFc@kS*oGRji@}4foF#K9!+Dk9i+59MeNGO!NBhbfpwW9vVoB<py4xa9j;;)yE{jqQV@%7hk1_3L54|XUCF<~?goY8)b!DZ)HOW?>wHUp##zh#J4v6YKu&e1ifbjzH+zt{cDF84lSt1S#ik<ig7*r;i8ja$MzhUph~jjEi}d~m%i_0{#Y@->CB_a9x|Ec!3~{WTYRAzDg>6euj-F+TA>qfwCMB598FHi>eTCZL&x=B#Q+JGy>tKou?1&rw*_yZY5Ka2!I|tjgE9ja6S6t9rz}zvR#7htbltcg%WOdR_MQvq*Na5x|`FmW0?(e$S(;=AnL*b%`;o!H{XPM2elTU@I)4a^_s2YgZZzS<0c$SR#$yVsB+STyXKHu*L9QeEs=ejO0e6WEC|MR9DA!3A`xA2NHQS`Z?!cjXxW2`H%i5zwhb0!u-BxSJ*Fr<criNRi53Ll|l3DYuUYK1Gy-J`zV@HAu;Y*MXN|*qBwz#f4KzDB+PZw)Vnj%Ki=Q1u0B*rzVVt2?j;LIVym1x*nw(#U40Ws`2mbyRS_+f&fX0;M8mITbRi2yZNtw4Xr!q$tE4h#NX^%*S$e@yV1jO`_7%0AcTP3~*9t!Qfr_8ByQ)cwl9|@)<6Y<yloo9f%j}Y}ReFY*M+Em_u-ack%~=8`?!c6(**7836SSbMkF=fd-!X8#Pf+J|@0|lcz~B^Ckqmf3+@<?U{5=ofRpIY>bVY&Bc$xLR`RbTTEYx~+dHvA=N79<90!Wzw2UxK$z4)q6?y)3Gs#U6c&Tb8ZQ@i&^C#E9aXMMDcO5a;$YMetBBUQcwN8sW;XZee0q|1yx-zBYsXY7hyH5f}8Q-(uGQW1<gU7o_7NK){QmZ3`#WuR)W!Y!ZPHun0}(h4{pOz-FS&sRjC5qZ$a(Mp>!dpG*Z>jlKaWm5X=cE&hMuQ;(~t(FQkm4dk?6yIBs3N^A~f>Oe02D!VX&oBmN-gNZ7!*I@%;(i|b-t`uXZ;brOAFmdciU1JZUw}0S%L}lJapzV{uS<dALpavY0Bo?*P^ASJ_g+V9$~-LuGg?(pbv^M6l6!023WX(5gW_FF*D$xus@o>p$4lp@`?!2^xR-vsN-8|~6yDiT@{ly_7|t2S%Viepoa*d4;-nG<rz*FMW7u~|Y&2&KC9BMpn)C@q#OtyLorYxhJKVNKZesA)3+aAm3A<<eb-Vrk+~ai}PopWpa-iQpTLy?iNN1gdT%;SFfsbH3B|*oTE2lE-pE`0+?(|q|>LZSxLJ>=dGZ$4ERb~oluq*Ox@pnsA=kcNj@$q5J^xxJ@D^Ze0-;m2_zYI(wXIXCQWd<IT4vnpNCkNd8aQ3M^S$Zd|IhC2409zc+9F47DlZIJ^0=sqD7I?RYJiaXVy$yLVxtM4Yr4)^(_=3k-QX_hiDtE>~TsF*u%~BeuZ_+ozxqXSY!5XVGvw;O?EwxddX&6)-HATaQ8zZ@gk^I|12IpVyb}d{znNjfb4gnIQi=1<m9U2&f5Z~xLjvs=nQO<@ss0UCQDT+sCa(`k`E<LB-r$w*@t+&VP(1{bMsTQ6hnYXU^t4Z?3;@0u;s=Vk!v%sDiU9SP{x3XR~OfFN@HrGDVm>6On7FuNy2&rJD7*vAg&yFy<v6P&MXwGJsk!rFcU5q|&wng_%sy|(w1`RCPU+l89#35wd!7d8x8z#XAkp;UYv_Mj0RV;znpkfQ%qS2BcPKq?Bm&{SK7@nkBTBsF!9+o3%ZYc5e_|1y(Pfzd8mi2XSWq-JwviVr$tHToUenInJkEdt9+z(&&^k(nh@!3Ji_NRMA=9M=WmRO?_L||jwZoGwj9?!nWefP!6s2FOo8qFoSpiKo$y{1}b33ZODgDt411u-C4frl_^9a4-7jV+trgE{}<;G+4>%Csu%<|-tKQH_LB%yAcXV07~}d)^!$g6M)vEA@=U7|uk?pdN$n7;FTa)6@V3D@}vchBv3b22+EP$K-6WNp*|Ed*bkPvwg1u`fPvf{`JC(t}iZ*$3c&dOT>@-KKSXQlXSY?qAR&SOAaZg9J0In1-Hh=D6j9(^$<ci^=7zq3`%{7-}7qX#4cXTOnJEST^aWMhX55d=UR(eT(@FKwvG)OoQ!X-uP)wMaKO>YaFuo_QAo}dLW+RVe*WpK_ND4YF0FYf=V}nV4h*$rH%Wjdy-1mq681n4ihX)Yps0aWwKE1Vx-|Yf&ut#SlEqgGioEWbFR}OW5BA;67l(78-&#t&{@hPb1QLGtgR8^g?-$ok{PLh)e<RZT@zQ}h!Lo->y}h5E8`NEOSA-#J!l026AYtOwGP$k-b&-Lu)AFY{Eo&5UfLOl85jkh0l(|g9@kY(uB&16&#?-LZMUw;(TipWxu5Hb0MUiW{k3D+$)1zXEzx~tyaCPt!d3=PtC!pbXmb_O)u&cdEQ1GL7BVioAhsIW<%M`>fJ-Yb<SBa@7hL$XV4dpg&W0X=l)n=q?DA?bqGJ~M_t9llRvfUWnir~iwL8q_xdZGL5`r-b%^VuQEetvaOxLE}%G$hJd$e0>1=0GUx8?cutL1vk>i!{P8BZpaytciv}O@ootA*nPXA%c-4hR9JT^g)_hkCvb}czY_4`|gcA=;7^(Fw8NbN21kz^_v9>XSpuraFO~T%bsQ$)3-A7R3rYS)>dl>n5>jDatl-;2go_ZULh8xJX17;-v@5_@s-`U&yRv(zgfcX(<LCkSaTnwF3*A-M6EKhQw;muYJ0_Skp;W*;MiyFP-j)+S2#3Bt`s^L_s7^4`)ze`#_Xy}A}K|qo&_x=w|V_{+)Tb$c1J$@y^AAnwmobF7$j2kn{{MS-o+hcz}Ma$hKJC9syfwBbET3(vKm}!3Ei7bVPt^<_5vj!8Wp&ReO+f8_F;@<&t=~7<n8@UZ$G_rc;`3w+s?f%SWK29!4m-nrpPIAx&;=0^821$ahqnC>>>$#Eymw#2AY;Tt1q}$j5VSkXs{6-!I3L94k?d`NvUVD1UxwptYaIg-ah&L-9e=D7YoLGvX`~}U@7!o^8bEM*H>K&EiluPig%Wxu;4ix@SHD_=Tr+eOf)UvjJ>o;mMOK)7HT47KTy-b5_6rCJZd2+7USwPy1!z#H2R8p@_r5et0e+IKd!;A<UBj(Oo++2r-0h;z<x2So1Ac=H*2*4$gp7ab045e6by2<E~TV7QcY4OBZO?PDH>(?!e~UJ=El-%+&mL+FDDz}`RFB#d9+UY<VcFXx@E1j>SQCN7{KGy2to=u;W^G;8ZNVd_o_a)kHeCr_kyC8?1+6N_fpVmg*uOF+$icOvC^_9R**BdNOfxrot|u550UocqAyp0P5U(j3K%meCzKF*$8R2U1bCS-*J>Vp4C*|ayC7C=S-E7)oO*6<NgC}bsupo2OJ#0Oet7m7wsGgJ!29w#@VqmA(4gP1=v{q>O#K>34-3i2poj=6#?WKB&wcBjM_0+s0r6$l`N8PN9l?USHQ&U`(YU%Awy97^DiY7arBE$3OiQGDZ~3Uk@5iP>d*?~|{@TM8koN`(?;fT@`@;YH>cE+Pby!AyvRbiQF}r;M`N5bKI)kTqwO$e`U1a_5HBROM87ZeC(IQbRKr2jW(A2HVh(&ssrMSxc=)k$e8M7;yZ+)llUW|TnG@Jk6a6SC?ad~f_n!h|?)cuD~{@dHXINARD;(98-b|G5Str3#=G(Iuf@08a&f)ZV155VHl5Csl*J|k@w)TBjVQ0d*ke6$Bf>(~~v`fBJRB*IXui-i)^4bq;b;72E!_0ckk|7x8}GIOxp>WayjNzRjBz>D0<A27R&CY_pv8h77Xf;?-T6tc`XQ*5ETGeQXD7be`33)NMs%CJc#-3q?bE#_+%qjyd}wQ8M|y(4WUIR<r+KuoZM)hkMoE^@ncph{<>SXQ<v8Qz>AQj%YakW7)8W9TtkkQ#aoY)?mbtCL_zfM^@R-^cThYxph}`(*#m_4??lN}Ux+lcZ#I$w<O0bEY#YHC$?Bj+tkW)S|B)^<u9`DTN!&D2?K%jL2M~DmC}<sFo~QlY-{T8)Lc^nfsNvFAokKkB>e37Yp)yca?z*g%B+k#cj_pqWGcc+OXuyJXRh!E#6T23@~_Np9yl#ARTf^_8%P$IQG(vB&FnyjTsX~V|cRx)orMGh{*pm-m=*EiyxJ%AO86dfAnW%Mf@SE0e2OSY{kp3JHt6ka;%q$97ScM+J)`yG+t#XP<$W+4O^YD;A}PZOf~C7w4yvfG$pOQgw4O`_L%6|p&WnoeQDU_=zFr!e)1N40l>3gS_>BqfL>%Us?W>=jZis;?E5P+qKo%*f#Qprt|sT#8pSBJU(82l-{-KpJip7)?cZlXE<6lBfftMQ;PaL1a6lO)FcHV7*{ZPSUD%<9H&Sa5Jl|5_Ef?;#PuD*m|NH;>L(8wXeiA?W&JDTO`O~kvF*j<eExKEa8@)Fe5ogf=F6vG(j_B8_R=lam2*oNobxOGkp-{rGRp}#c+@B0TTi)4ElJ*Y5&rg`n{;25JFW(7$xN@h?yC9CBjPFG88?~5s=^QoAi}cza!}aHky3~;h8V5^Kp_8$@HYnAU3bX;)894($=ORXLg5(ycY1*p#%TfCGFCVslaWmt5dI;NhZrx_Sd<g2H`|-h*+n%;Bt<rCT=z2NAs@WJ3fU2wp1$>#K&6#?W(R&tSfvkfx1kyC2mQrJCezK`4mkbGDrWi3p3#}TE036$O{B3Od^d4)W*FF&rVi-USi6Jxixbhrzn}ipcHw{D*O|U$fa%h+tD<D=kvNE1jTGbI!wThutvQ_lx*2@1sdvDVtS9W9x{*}P2W&nBNn%On8dwV;xoxv=6*_->>U7M;wSIwe<S}3z2WeF-li$W^9RO-&jX^NznOeVpFlnD%!e<jhsWX#?RkMIaGB+?&MaH^gZ8H{juM$p~unVn<T?3`lto`$5lYrH>%LM`;3{NTpM?Z)l;#df)yWEDJT49Snn{U8|+$<kxW?*vO(KLn?%nU$I(oz0!uTdNrS)R2wAyTWG<J>=OfNVP(1k#d-`ilNR*+|z!#cxp3zUAC`9!=4{cyjPpym)pQ)<Y4*Dbpm1#%N49=9W_S2%p6&3O!g0*CHg;Cc@|AQcgYP=N=WvEdy9iz<UUvKMi_(r>FaH2pEbIl8h^8-#m6@}@zu$a@8_H6i4GVt0icUnFlAtb?ThAknHv}!7-usoBdNzw`=H!PpU|d)%veXSRWbH5hiiUFYK9!A-CJ7E)c1knH%z-f*`~gY!D7!3ZQKks%6aSMG4n-6oAzQ!t~MoV!RqT^PPtd`38zk-u*?NWQtF{;h*Pi%jt*R-#NGV#^|HM7dw#W2>z79rnRWE?+f|9~_V9Zna0Fz5gofZl0t1c|*mV+x-+H^(fYs%LMh=BLD=0uUV$<z%G*H;wgFAQadE^Gr*{#z;8W~(7IqPH`&;pFzP5oYoK3*pzuZ{{FKRGU4ytvky^}DyssYTKp)bM>EN@!u3!a=qcY`RDVBZgp45OG+lq!bKRq*|Jg)}Byw)MB|)5v+E2Kc~+aXJn~kWTQ5)$L8x*+tB2T1xk9#{Cwl4B$7ALFp3BTmnah0E~b?PFESL7j8em_N)>^I3n56EoQe_Ys?mtH5g(;od`4=b#>OavUF^8Q?q78uJAS<e;`_02Ng)b~fKZIC(=REV1tc%hI*p-sE62ICD%FZ_%B(Y&;r1ZG_=^!32v)<}$TAxbqjA`qahYYUiFSvK2UN12{B#v@dwFd4tZNP5EMLC|W=-rP<s|5ye%^+3HG}QxF^9{H(V>+igEw)k`cj*-H((55G|)Ip4L-e_3D6B)T$+1gO*1w_F=e03c;lhsdc^(RNl-pp&IVT$`TT0B=U?88_x|W)E$uz18;JtB5nyl+7<*6k0DJ&@Z{MkP?%Qe~{zwPI2$iN3<>>B#y<_#pMJLzjkPhODI@wZkvp6TKN8N>X(CB_`Bj)|&=Yc4Hb~3i{@zuT6ij6VMc@d0D7RBl(KLN<~;iJCW*$;jIeg_QhZ~!vW7)UW#${jMfMYK$7IXDDqP}NG3`l@=f^h;vYzU1hyP1)n~;rP6sFWMN4V%5>!1xz6kf+g1o2iC(ox*0*p@Bu_4swKaWj5S%dOx<EGwVq{YBkXg4gMcz648sT*VqS)e4d)srVA=N^@5bLMbM(tiD-vMnGg<^%0w81RDCZHRi(HCeca{s>ijP=yzp{D_DH#b2nzTc;3lBprF+s~cQp}LVwL1*5;`l?}zW=(%f~^{Vxhff)46)t1w^-r+!PTj_=2pb1<WrXRZd&<@(YRBTrCifHX2_WS4De>O$TimQ6tX0@1db{>P4_fvoTl4t(C~(*V(X=g(W_$Iiwy5n`4`^1xa<u2=-7DwWVOh6xttZP?f9qbY{x3ou|=miMa0Bb(~$#bFZ73WhJ|&!M4@c9gsHWMk^6|4aQJ0fY8)_mP~tAe`Ae*$u{&3T?+Jxy7qLt%Q1&+b>vEbuy`3al6tXuCt7EhZXyKPIvRD@N9iYvsFmjQqFwA20+)KqUHXe`~sgzb{qMAeNsP11eRXf+AJwqSG7`pqtacZ(}K%QE;U7N1lq{mlR$Bm0MMvvND_buP9orZ4CGq)Zj`39G#PE^f^uD(eJJjX&yE;C~1n61{##pP|C{;_eO+;}9Tb22L%1E(fcLrCge9mwvW5TT`A`1SQw^5@5hyb(P3=4N?fi#ZVbtgs8Y8(&B%G4Vq>+7CQ*n|PdFXvS6hfTF|=%P7I64=2oATH-W%sMJa~PTy!Yqw^hRPb9+;s#4uWV()EOES{J@_Cu#*TmJ{|ZJ<uF294~x#WABX20iE*el2*B=b&6T)e5ZYDhNbI_D5}yB2i0fP(Y}((y#{ZXfMOG#F~d@-^HNaujS3>@Q+spwAS&@{zOy8R_0JG9NE~Z;WG}(+jrFIBKOf)AP^LnF=|OVQ*}31AU{Kc&aT!`X9JZ9KEVVv7dD#Y`Wyv%<2_rR8Vap)?w>BwgsX(iuh)3}W}6M5J6R&dY!%A4U{NrfrD6;hX??*GEtFEPR7#bQLoXv6F>KlbO3e_9s`6-^gNQ;+?k<GJnkXo?ebD_j1}*gd^Rl4pl;^i6QpN9|1pyoQcdZ@-if$>C(m5mu6fZL*l?Y~|GVRlF{p{Y`2%z?)Q^nehBN>Iag57(Ljhhi9Z`qGXDebiV4*z+2J>{_&Qm&)Mi@5u?DLN2u^yOfwL4%c7Vm>DT^JTIa;abCR#VkO|or_>&%se&MP-;euHKc)497m2UwRV!$BF7-Q*X8?0s82U1JiD;b$BT@ugd~=?6b|`}L&-oF8E<MdWEw~%!q$<7=xi=_IXJiG(!Mj*S&7_4-54|Ez#KT>vUjn^+kK+){LR)XPgiS|pP%%CzT0+`qnwkol|+y-MK=6&mY&;124&A2s~9MxB$zw&Y~0wlJ}n7aZnP+znG2L?>9R?qh%&XS>%X;sVGpXyPJ%x=X^O8c^mWZ=(`pZEj+k6JD=OQ26!e?|j0Be$dj|7nPzx~2ZXJEI3|vq(bG7h`BN;!5QwF#J6|yub5~?sXw>oJ12E}dceR&+faBqoCS}m61>=WCYkb(u=Swler7deq`Psj>z^Brz9jOYFWkz0+ak=0+Riam{54C4hXDBN0zx!^Do7BlTmKKEj7b=mvuXa>15kT2E|p|$MZm_dB8g@pKSBO(UO>IRSzIUVxWR-ygGiwx1$X{u_43OJ~PR>5jq*vK+_(i_EYt(Y5O@f%~I970!1omQ#$ncW9BCvF+?T9sekIQ0DXI1u#ddJ40Q0(sOv-a5W!nKVMq0^pY17(=9k)$d61PV;L?_yZX5O;ZP#Xo++fMjl;8&o#SMd&|B?&e1uImldHgG>4gU1c|=5W-w>ZuJ=cRJiGq$@#y7NBk<0c+B$pr&G%1!xJ}tIA{pinoLm$uX~NBiF&N<@6<^%OWg73U#4M3&(2;5clvuREsBO+-FNF8Qn+pa~lo-)qXLTjkzK7{+6GPX9fB9llw-MPg0$Z{acOj1o$OnMU!qt~4WZ*(WMAX0)um-!F5GQFE40blBJ;ew%w;8A#EAl!0>8s2ffMbgL(e=GJ{C{r<_l;NCSIbSG-Q><69Ygie&EfXiD_>8u|LJ(s>K%0_=N|u_!-d~%-AqrSi%%a)vYgA3!e)sCN|%{z3sQ{+2kf)<ggvGaeIwuC0hcUcl#ptvY8U=6YERHKLn#IedhU5QnR?By@3x=0j$OK0-&aTO|L7)UUmX4XZWOV9b#>bg^+&Db?Wm=_DA3#(u&kb|kB-;Mv*YFR>-BWW{?3Y>f4E3&-}D{UOXt5ItASsLe}Cc_`10s*{bIfE=gX}hufOk;qn!5_%MbZ#x#i<ip`5E*t*0j+T^*nA&yGU?pB{g~^(*k%a!UK{#%tBkDp=C>;Y&`2RyiMND=x~H*&Hb?QBSckB+9NY(E^oT!T?guZh#zIX`ryNQ#2ZGIjp7LVg^^U-$(jx4=27{TOc1TI-!14@nl_Mz7?h4GLUM7QZPwoSI9$qAM~6Kow3nN48jUDy4pp@TvD+waD+JftdTGnVJv-&KGW<{i(Qoiv>L}8F=Y27412w}@3rk8H`wOWEk|G&Ia(gH|AFU1rEo|aQz2jGs8mgA5T%U~*uY4Oy^Un~-w2{{MK*RgYEL4XQK723&O_vkHO4fj#=QskKo9PVlk>5sw;B(}Et8l3<A?un@`x{vyC&;S%9mSvabUF^S(GS?8i^9u-mzN|-%YzDr4OJNSCgS@=`&1dOueN<hP*Y0sB@6?(x*kNYT+`nn{yF>A_4UXZRWk0-vsUFCm8zqwkd)`ba8J%4?&Ga0#VK>-5~iQvu4ILAl2-~8j$Bm+Ts``4I0AUK1-wIGD3H?yu#6S0x=WVLu%QFTkiN+zgjrYXA8x;=5tTi3Fn*dzdQ1_&sRS5;$%th<z3^WSL+P_+R^ycrVgD(OB4hgql;`>Amcd}NI6_)2wPxsYEgR~5R$77#1dyRjsr8+9JGPhwcC5lF&PO8sbEVZ7$J&vZw=q$L0>O25KrEhq@yn}i6pYZ&6urZ$#z+K!pqzbtsQ4{PwVMp-`&El<w0O^H%iR{j!2`!&M7T8CpY3gW~z2c7AyCAB)-XN_+}yWyKPj6&{Cjra3To-L&O+1uht`kiwwjDkkZVfcQ@D^t!ETSM#V#DLAaGE2CNsS;b@`By@%!woIM#JkX`5D&2!S5jt`KPE&P}vM;5dQJRoj9B)iP>gVwloMmEmv<{J~(C=CYa#cerIE9BHGN=b$zks{cukYhA1Q&hBH2B`d#YB;^s!qV+T>KEI(kEDoXuWZN}j5Y#8Jcq)tUStcr8>sDyzF<^^DNs&jG_WV^8p;t%F`CS6Sjp`atM=lv1KlpcRD{w_VeVN7zj(kx_^roMTo^HfODGu%;-vAcX;caqIsDwY7=Y*jJEqpFg;|V<Mx*B#V|sDTRJ)Un-tx2<6SS&2M6mSP=PptAde}W*mOq~TQO5Bu9@gq+sR+c$_@94v1v+?FjZP!Xe+CM4UEAWip%6z_Z0;l&v;cF+)#KJ>k5Ou`9RkE83XPHhm28<ax4kp|Kxh07$Mx?)t~&=8QKFEtD}j-bL^<HW4{+Ynu1NVq5SASXJYvric}79!D8t~a(kxPP!>O)MBPgR!HJTi0Sn!HWCI~}zigG6wfBoe9Uw{4mufMV{4&Oie`S*`rUM;tMdG-CHAAkSov+o~$^Zl=X_Wh&J?faMCKl=Umj~=ZLcxtyT-`ih!^vwR)vV|h-9$Pub%#t%3*5BUh(aU7m-X8b8!V*i575Z$2XT`Scd!cZpKBQ~TImH6Tb%rHi@HP0XI`{p%`%9s}x$h#gE`?qjX8sZN+6^(Ag9yb{uTHEUW`uJ#^$0F<b$KKiGK}<%#$gy}HTW9yG*;xc@>PrqQuj7wjdtVE!s3&y)N3jf$6om#sQl}o+bXT}a`U>0@z+GOsD5$+%F&@Jg!pb)6|;T_Wxu4Fo3SJv+6|GW*d(}-0Ti7s-XjGD8+A%9>EPl%saK0d!w9+5T~GdwsM5DLv!6dcX2<87cbBte35gh7&H$oB(QuBSk`a!J%!ti+d(3%MAuLVe3<(Xh;IQi_4=(7v$r&A%YEU3BR@A97QeoJSXzms7XHMNWtHuiDWTn@4A!86rf=mbeH~+>d@P8ctVf>Z9+JE=Q9=#1_2D-?z?cvU?2<}<bi{--@>{*vd%5$iaO0*P8F}0ADJ7EJu-Kl{Q{x)f!2)e7){TFMd{Kbis{6`CE|JhNa?s`jaQ~FeA?FdE>oWLWWWrcl_aB_xX@m9O~zMWBZsYoP+Kzz5U$1ulWshq=fQ(KAAN_*jMe2{l5gV$R}FBS~>(Q?-Fc;!5s8t=g;BrQc)Ai1BeRqzAQIl5n$8GUF3x=)<Iwaf^vE;lUrx_Ymy&k(W2nO%6#=!^ks3&K#s<Pg1p@6mS}ecvAEd4GT8(!NIjlau|S7t18?M>l@gn*`}JqMy*AflH+vAnQTS&@)bSk<FkPW38iU8l|FD>YRIsH1lYcRq9ak;q)R!NEB-lVrA5-I62o{q2!I@7Hf(AW?eb^bOo8KLgM3_0n0UhUu+RHCN&U+mNwlUGx|5%dzLX?Xqe^+(W-MygIh6z-6n=Cz9s<{lYle3e&<Y<WHE4V0mm?U4%4pMeJbc)nBIov7NszE95D8w8KU7uakvP)bu8^d=Obsarz%)d;}ROWU?j%WTkWXLb>A)g5XOKuAx+<|*FHLvYnftBdjQ_wEA}$Jsu-`!zgvnl0aFa_u$O@(1&hednGi43dT16W)EPuW%qdfy%?E>~VAqytV3a<+MBA$%s0C;5EpVwR8Ohm=T<!(p<CE6o<8`}z6&L&Vn&w>Zk3ZhjBr?=UXmJ?@BxHNrjNRzg2};bDI6*0)gS)SVffcO$jvm?YW7O0!YRD=%7QhjU48qJ^LUKlqWMDJQxCiKKXN>mo)t0ku&FIZHLrNMcy6o!q>}ag$GJ|heDTqZYcprx&bs8nR@OwfZr))V+`?t5`mcXUv;V!}5z_S-?-h=O9f!XzB`|06C6l5LmeYLWiPp{ot{Jv*5E4SMm1!F`p;<+HrOzu{_nc(D$oCFrbEjo~WAOo=*kEzD&Vl5#l4xwCT4k4kD^g1y2T9J#zE$3bY`YwiXXPMD0guY#)(!bkeS}obYrK{ArcB`e9L<cxSA098VR{Iu43+hkI6%jjGP8?xG9Ri7FQ-UGPY@k}9Fb9~Oqog#Y7@XXr?o<x>yEO-HQhm2vCx{ZWreMi0>*0(hI3&HyU~>>$64h8;gDbg}w8k<khcR5Xw6v~yRHQLRDZ}_Q*J89=+id&T<m+O{A0Jl?SBIcSSO0DuuJDbofA>Faf{YB25!6C4SP~$ShflR1c<?rIV`S+fV>r8-0JY%@*eJpMsFPLV6dFJatr^|U5Flu1vv^)L_AQyAG&B;lC(Ji2?7Xja?L$hzijwh53m#f}a&V4TfXXGFAquu&msmxGUMJU?@`y-GGrHeo=^~>}k(maBF{O6VQjAeI(1|klfyf62B0pOv(N=N8MJ?^9+r6-yB{~=8vvpXKT?fd}eN5RcHYn;L9FS4oayf!EUjCG%g=y@ZlYz#GiH5-$=xkwF=ipYY%tK62U2u0IL&#{T001X(7HNBb@^$aOpRU^5>y*WBSFzb!amwEvC0TBSHm^??zgu^DZj@-hI&%3_N!lN-Kl@f0^G34vsG4>C^&k3W-`q;sE<&B(oU{Q}9nVj1{z>ciPmc5N|NZ8N{PO!JA202M|L^1%Ke{?<a{X@U!d|Q9ufOK`^3wd*n>m0@r~HOY8B#{$IT08$ZSFMq65mDEk%?hI)af{7Qp8a5kV;&wav=2%b#Sn2w}5?_<Fwlt8zdhvwWqXSpm#3NwQziMV!`>#Reba5solfVZJMx_BV!WFlY#7=dk<|efQ!_Gb*1K<;&k&f1@1m66|?cg(Li_@gf6Z2#;$2ri)L;R+l&^xy}K#=j<4?-7r#0J>yM6rW;N{e;QYn4)5F3TH>u2t6?JCA-Y(MMUW4JR9vRC;hNP8*+-glT#5Ou+a4Zv9M{e5es_G+59pG~Llo2@<>6EMeofv8CA$hWTv}kCr2<GEOsQXqu{M#Fok)IsT4}Nh9L^dT*$dLre-W>4b1o89NGngSMTx5i{k;2eii(%?!EHEx9Qd7WbjD!8O3sW1SS}liKw#4db6ztKQWRK9Bro?a7oz1N#+7ON5sRnZGLiI%tyX_3o8obCJ7|jHj&^cV8k!m#vnS8aQCE}3<W-pvl!JG@Yqu9(e+w48N#QWZect}gcD9_`yFZPCQ;^#Mgh-XI*_Q{cEJ>R~`83RP;;_8!atZp85fCK6STE(w7Cjj^Z*x?i_$USl!VOR_gqc<&CTu#F+qp9Ue%b-@+T}+N!FH{DF+GVE7+SA;}o$rV250(x1XKT!UdbAGv=}|BC^Ud@3iaCZ9$kK{&bB@aCpqpm^E>Z^GRSXADE9aeis3rE4EuofB8fU5mTw@lAT~dW6_KgK|io&27+StWz9!jh(&-AN>0<CoDldGe2g9YxhjhQor#WX#tMJKv2f06^p!D=GEW$MHz^dY3qJO>)#=|m&sAZ;><Z-`)*Y@8>sk@V3-gtUt^h90RR25NKiK7#+cR`Ih<Go|i`6D;aNB=@V0uKuxEl#?!UOeI$1+od)Hs){m33vtxU?Dn6rT2dUC>ug<c)HKk*OeY^=@p-?wZ@#}a)przjKlSQ*aiUD~&E}DOaLa2_b+b7Nntdo>vt%(|q=%4jBmuXUueoy>S}izXDjBGrp-XES5n5$?U~t*kQoxmDd#!z_?7|>-WU~!5{$y>SuhIB&vyqUM=ol<*_)vglxDdqZ^BF=(DPCmvhYPslvjp)pSW-$8VzuYmw1-qR$2l}x2+Gvi0}cC=gLl$hGO_N-@ooq{I~i{F2)tUlLSOxbk)5j(bC&OpHbHBYZfZ}upgEzHarZ(-HvFWUb0zRat}j`b7EaMn3ndAdIvQleJb*E_;$@zr4+Kc`DQF{q*sXGenJkNP)4Fq<2c}p)y)&u5_*Y#Uvs^n?-rBR=*wx+`Wv<^ozWXC?tg<$%gs~>0s8;=QaG;t49m0V!g}3c33FQxfAW5iJ<O3N%tzGj>GqUC49A?qNZoXt!_l#g+%!>iIrRssc$*d(x-(kS1;LS}WetPS)x=dK#SY3XxZrK0mI0Jh+m2#Szf4@nWB&KM&S?&KpEQW*dS&}$-naP-qI?l#bOQR}b;IOm|OZzjW)}yO0x&Tg?z0imyQ=d>ZX8Rj+PsR@=<6oXgG`(7<mNqLb+PeVAA%$S?4g2Wa0o0-GJ4r1P^M{a0jqFF9#_}y?dh;k4T5TMAE{2cXfhb$bOchj-LQdUS6cE`B;?qofCVeBC{p4ub_W33&B67J|*Zou`*KZ4lbjFPJMG|=_RNL@OH^H#DeS|T&#mtDo!mwLTgb6JV$#x}X3sFmru{fm+yG+w-)BQ%AHW4Ex<3icsqm^ABF`_8v=y@jA%ZxToqV6lyskJ&vZ&;xPfW=fzE4DO?W%_P}iQAY3y8Cb#c9b${kG6X!=a1LnjrX%lLWF1+wVUDjq&`c`=Y*c-iyX65VncZvF+;2=LLJ6a)PgUkQQYq;1xtf7BRfsEqS{&+P~7oc2kv{Hr(1YW{{F}`9<Tex-yZAi<81*5Xq+($Sq5NMH$FXUhA3ZT0H&g21}b7-;56LU2ZRj4a9wSFoE?0DPDhfgxm8^>L<H1*D9avzcMfx&F449MdU<!}i-rBZI8nCwbh{2>a=WOc%9??NHL#u&j|vw#<m)k(V1%xaI~MYw)1e_vONb<;<cJNeJWp?$ci=u^?ImRU0&w5!yeH6k*XH8tIMDlQTQ|mVQ6P5J!>r&=fP&|&$49-${Z8$6ZMnRcr93OTWAxxloTDY}lj?|CDMy*P6qh+{Q4(>RP+&~L{S@&}B`)WETC2s$Hx)J6pYf~~AcxCDf}^#Npi3~6F^Yec$gKfK9%=<T_@)g}h8n<`f3JkG8*6DHh7xyAz=vFcp1utmc^6y2lvUJC;6t=OBfAjIfgr!_DGI`$gCn3Mwg{bu9S8NCxEc`Y(Xb@NngeI6&@)I6)iP#E(H;moXj)yLwI&XG&-#I=y<UCz#mNEp@mys!^Iew5HU_bgm=G45F=L+MYVY7dob_!D0>SXv2hhMSP`JA4UPWqCQ{V_KatCnlAnGhKpq1>V7rmB7t!LkQq>9t({XUcNT7T!O^{DUD<5AyF*R_Wq+>G*Wwf-fv>TaZ)qfwL?m{B%w*~(>ZfESK~B6A6mf{VKMVKv=}crLR_Ek1P~a|zu@Uoy~HgWIe@k`f4H_c(WuX(apif3X}!`y%c2S>M_KdU3j9yLERl!ZHw(LPX;|2FOA@hj(YWNcdT_fyJ5h9$Js7@yu#z)7_{Gkv7Al!;lXf^p3IDK1a3>3_^AGKcp7&ue_SCIq}H@Iq@ATwx6(aCN%KR8r`4-4&jgkyqm)x<PYK8x#SQyWTV}s2BoB0EgZciSz`-eiL(Xt$sXf?8p;5Y+)Jm1hQ`yr9d<889<ijW%zSgQwfJ(`$GMhxd2zeqcyokfmHnS@%mF{!c9MhW$7vZb`1Y9i4%y~CiOV#}kQykp8miCSvkv1s!aD80){_jJoso<_O-2~CTNc$gV|IbhY8>ivsL|zc`8Q1iKmWl3NPn|Ew>pgnBl2pW1!6G1dPoOI_)fVMK|h4tS|wpxG`9nGik)W~VGI>V+%coHAk&hpCEGZdE#)S|J1%S2DW|yy=NpUsvn6eQ{bu{=FPBwE7x#L0I<~%b6wN2D#7iJDL5`C1vMY}Aj&U0@eh6`!dowE3<`e@4vy2+K#>70rL>zq@mofmWv&4>7VDiXnFd<4aXs{c<+)Jpd@9$5Jk^1@J7@S|99#k&3e6ihD&<`zBM(;O8(wva>Uh)6CKla$a{Hr<a<!|GR9SpflhmP9p(rt4@ZzF3!@VO2L2cVXWYi<RqI9Sq%Q$njS^01djc+Lv@;L&X~|74k}yfbaS&C~4=bG8U14hC664h9PHS#<w0m5OR>S}V4}UF|xnrIl>|S*`aebGA>M(D&#iNVh~AT&4)IgZ9R0AM8Cf8oW)KZg=pHt03mzAHnKp*Y|Hup_UEVqS30oktcVy7d*fj{+AFhauBI@%YJ0QbQaRsjm5W=Mx^FDIs+S9e9l&Nav7;Zm?f3YF78x}MD3mW*UI~;cC4qxbpYwr76xxQ&}}YBfpXT6K@Ree82tkeepAJN;Gs7K|BpTJZ~iZP;~iH1?SDGsXdqwY<<Q7&uc3xA=4=-Fqr{O4aKV<a`BkCbOlg4(6olAIKrU%mRu|~H_hTOv8TS*bXAfTs@6<N17eA=`Ef_;iDG(m`5Q*PfaE1eX0L-gNwr5#<;>H+OFRn03v9v5{CQHiNW*U{dAtvmT3^#QjC`E1FuuXf%;9t0%<xoN0XiWk!BnU)sNC$&<?{-!|=|hOljt$%$+h}4ojx<JLZK8&70qd-jCHK&H#OC^kgc<@(syPUE*(aFqPK>VGX`imUyH`i|_L~j#Z;p!fTc$@{qbmZTB?wjHM^?$tn&YF3JV<qqg(?+C!k{V6W@MeUQ0oR$EktpYoGl6)qz0KSN-cCH;p`T(xQlZBfgbDgb@b%3n?tPYp_3O651qV2)?Ai(V=(R;Q_P00)T%iih~;g4LHsoQLr9p#N9U)j8iwpfdSMvPZ`ipSGS0ox9H2ULGRl==6*BgPV03F3`kZ^hJe_a;{UV_K>e{yWXbiGyzCT(HT-Wmc)=D2+eBgsyu4;DCSa#1<4QFX2T%_^43`8B$?1R8fiu)iM*dsW~i5#0GpX2K-k)RYS^SNlXC({g(&poj|1Vv9f1us{x&Sy7*u~&DkXV*!AFRpGPclC?;<nEHwwa?9$>tWb;$6>)$5yXD*H=A{A1noU$LjWPU5-l&f-UmN6c-!$+{vgIzN9+>1g|pFrp(;`<wd(e>#>Fh=-5*B;b21{?%Pc;SO%rCbm!>Q2O|kpE+SN+_+J|F{mYvIeGLbMVMkL=wiCa|_MKAJ{%E*=WfG$T<TQNLKsTsq^7+TIfR~V(WAgvf|usEbKeRjI#5e@L}r&RZxLa%e<O}bspLpQ0`8SOQ$?#K96^Ac%}=Q$%^WFvG~fKlnO^C)(O^l4lv!VI<eu5#<S6|s2hadxW&p(m=LCLyjyukyZ)cc=AvTZOM~4g`)<{4X~ubWw028rB#wf&D)roV6z(E)soC1h-D4lt%54E#RhTN^`c5s3a}juECHe5WD(OwLs+2tR^Gp&TS9Fdwy8YZce6d+}WNV+fT379Qo`xg0iB<mp{1rKmP5%z53t&`hWfF|NXCja8>^AJ9dfRENXN=`yW62hvmV~PeAKG-8diK|2-R}Nl;vV9kUN8r<gOPe9js)a=g%p&ti-#Vv7Swy`?{VgRe~;9D9^Fk_&*7F|G*(p%jAv(XRnRq<h4l_EpzWtIs#Cpm<(&2`nMf$YN2K&e8lz@gn!`XY@^y5HnKCrG|zb^FRljO}rbGi_~25sG)@9F7<9|pd+W+BJA#2_pLCW9(;O#aZElOGvb<jKVFCZe!4||&MILJ<N^uCfH)${In^H1Me1=*sWG^J26G{8m>WzM1DKW!X>o(es6N&oTCEc!4H~Xe5_1cU!p^&2i^tzDvG?TIMf%~@;>P;z+AR9#<>u9L_`93qrdzA!Q-kA|$41km;}Pm7SFfL^ZUeIHr}`i%BSf-HlkP2`{@ovY?1ouY%onLtrXdssWB952!au|~$|zlXEgA=uSt^Bsxu({LnaS>9s@N;U=~wxFN!+<4*2sRorsq?e`vnwR6#v?y`u{n)Y;ROfiNI^0%7B9}4~8t8SFcbnlBj9O?w$=&u+W&k>(7G4LeC-;VWgRQED<ZV+`Cfe9^EGws0OzP*@smg(h&Xn=zaK}bW|v_W|r)h3xUXJv`<eNta$6`DGR~N@8N5x23ALv913C6rcl~wy}FYjHmr46@eryRmrPolO914kjix<A*kSEy`}Fs>+46cz#Wm=+RC}w~Y8*GjkO`3xFrKBOe4*KJS|vs%a#QxG!aSvy6oiI_BOBqWG3O~UmR2<yrnRg|$u7x~tL{R|x52l9&|h2~>+`pFcTJvd9`I+0OaUMZCST_>R&mbhJ@Q4uM?g|&C6rVkwmEAGh9i?!Zx)BzTA<?INrcp!2AvX0&Ki3$;y+-QH~Rz962Ez$Wf>wQ&fvj_E~`Kitt=z*yDpXDhp%zwT4gQN+A7H8AvMq7HVCdY@1mDl1CCB=1s-L(Ycqso5VO10UZQ<e>W(eaPfrF(Uu`Pcq?(gN!kB;!1*UY4jU3}eW=e~qX}l$t4nD%%M^D{#^9D|AR+qH}=ymk&5hji7|3c8GpGHU+JJEhEOMZE(WU?4!uanMOPTdo6bRlNsD3&oZ#B|`zR&f_8Q>K^_Oz1<Ju$S+cxs+MB%^Y)M2wW@0RuL^q{n&`WS3=lcm0_3X9+3|Tg)be>rB#2^;r!)wM`){CmRLFf%O!>=V@w?0gIT-kMdnL_CWb4gy2UGbJ13(bN&srFc5UV^V_=FPHAUa6?+p!m&R*R@X7AO%u~2`yDwN!$%75Neuj9gM#NshTN0bz-hHnhV%e2+BXq+K&&jYbR9mOg$8sV<FQO*s9&(h~AZc}8O+yYZaO;jwg+I_Msmp!0d=F#}OqYCb`@1J~pa}ad3Y|TA;zpgW!QX(`!@21&CKt<Aat+I=>bXUTvbM#tBMm8X0wL4Xd#6|(T<w<DXGO8o<sA{D@vON*bbIxJxz3&10nq%gFz7>V=gPZ?$Jb1re1bK90rG1>g`=^s{H%UaeQJSAwQwG<Q1$!?K`Jn~<haUaLbIu1IyJ_>@h|Wb`5H$y1Wv+&1S(<M!%PhGCOrSDUX;z>7c7)YR2{B27Mg}K(#J<lxohI*fE6S7iuTK4rpDtI(C#%Z2T_XQ@(n!6L@!4eBWI5Y)&<LV?c?30f7S1s!xyS&WINV`A*Eo|s(LTvt%ZJ4uwUS1{Cao20vAJ{Xh+_ZI7-CFqa^4rA9$18WvBcudBGV_Q&f8aagY@GiazDAASy>|aR9E$jdpcCxyRw#m(IvmpNVO7K(gFAGW_dT)SqPLqfFsiwBqcGFkt=DPqYh+^Q}PIfOE<<>np=D8D6?ZMIZRQ*+%;mdPdg@g%^L5M<A~JfcenLkuJHNk4~{~Wu0Qeh7UUc}POUnU5$==%(q{2~%WW5Vwi+i*tN5k#%F~B(bIpKUe5fH7<}jiqXYIk#ajD3VXEzKbBSD;9_9(sIcsR|Nj~}f$^^>bN-t%bN_TC|>y<bsWOe$r!vjy-teE=yx@aRoZo%j-k0eyofSwI`n8zVqaij8V1YWbCT%xQpX3~ZPz+fp~&t4@%q0Yf#gyn~Ae4r||^x(~bTuhB3DcUnh_$#B2}<Olqx9#*SPAHwt$i{?5f7MBGqg`)N%Jw%Jd9MtXU0c-5N1mk3`cH~CET`^-OdrkIud;skJ>Nv!5vmo`^qE_*8>-Ye$OZ0)gZ4e+POPmKi7+sEUyP1#igNREN_e5z!vl<8(B^L|N#Hlum&jAt!daNSX6q5`X5L*tzA~V#y?*qO+{fSkb`qyjQ;Og$rd$k;9Y}&5&p)8FAO5PEMOiE|1wVsNvT~N8t87D%=Y>+fqN#Q0iYVJm)l50%H$usq-ZBCLZ(e)-sQsidkXWzNHGo1S9uIB3Xpz76eSoOC{j{WG!#2#%rQnM@@0F=G%3r2`%t)9AL@+C%P!%pG@1%7r0fT$BK;yM@mJl%O*ra{zfmERqDJ3t*hYv_iEqV7?7s`9^*rY~=n|8G*}SC%ig#UL2;cU90J5VEfbpOYC=zQk=*HIUg`8ns%|kGWB_B-n$_q$%VYQiV{mmGfdSRE;PF8bf2rSW8NI?(F_eD}XPKCHa@@%*%%G6PAFg$f$`i`oZOP1s=LcO-WS)o1ID-1zNKj+=U|M)~qm>0dCpnC%Dsqr%tXy8jZ*IF$0tmVDB{Di^ZQW9s1LL{>vL&`Qzg^FHTze>j=WfcLXPXd~{A(#m!E4Q@*&tt}m`tQ?GXhwjJ)mVqY8mWGfh-N_t3J<2{Cp95~d}JGhWpM|b&w5w$tYX{8ueO(muvGCP(TG0qTB+BCx1Vi}$CK9lhJ2<+3hw4Zuw314)XABSKX)D3CkJ#qgt+Wh%4r(ccwD7NGWP<8s`HEKhsb+iIfV~fccQt>UaoGnmi@k6#)?a~5>d-8pQQ_rKLiuB8qh`e6)-y%toa!vsPl0(wGjATQ4sHy+Zqqim0aFJLxg=Q3ye7Dc=X%ehdDB3sLvjM0K)Rf)usH^ta34Iew$7~I(;OytAZ(O|cW7AFRej;js)3r?10b4%6)(o01GlM3~*}bD`m{CUy9Z2nZ3$ET9TsMPj$TCBBn-gfn!@#^udr8JfT=pLRpZrV-Gi9}_K66$EK(FWMU3>kz+jw<FWpC{hqc2|>-?E1&rMGPC#}8ukB(;H}JX&<^U96$|{+~Pc=Fo=UQ@jC{Fm<c`VrjIggb+*bL)$Jld7tq=KOKd-TFU?05nVJ*ZbSgLh?`_rf=&VnmbnwF<%by0Se%xN?1lO&ZfSOzp;S_t6>F}o%~oQF7>prNV*{F)TIB)7$JN7hEutg}V-LuCW&8Pk8Iv2A-Y2V+)R#wPp0!{4(VaHIrbsnn5RES}TP6(>z3!iNlysSk1etn)YK1<HncXa^#ZacvbQa@GHIx>Bt*{SPteT5q(8fv1GSXOgt@3;6^VP{=(&y{c^wrX{|GjU*-1<g&m+GT6a@RS(PmcZl+pygbvkIyxfbO+tjMnhl=E)~W`7#4I84C&l60uqWEs|@_(xH?-gPTZ6OQf;PnTSe>kOO0YBDIX(c7usG!_@z?{MFz72m1|IfBTpJ{y+ZZ-(0PM<z)PCHqS&y_uaEfmn8}b7z{XsLu7cTL8!zZfXkWY9NE=RagH#Pt5GJdF_sZI&K!p&(cUr7X2?}i&oNn~8iDG)M4EeceSO&U^Q|MK#)#CoQbb0y1SgD}d!T@NnW@pJx?#^;$>MFQlpz&c$c+9)Q8ZXendWFMsf^*WdkqT1piyuKq4y8uZzk?O*$gT(1VwM*Nro6fgZDAt9aLH@Uu59THmlJYboI(I3-FLsSfdXpsRd-8Vw{L8vDnQu8~u|C-R|gf?#unJSwML7eYCoif4a`geRl2s@Z^V^Oh|&yNkBjZU1q`|ZdROt%QVE%LM2+tF9T+wYW20ZAY*okRgi0iW)BEisYZj4R4jT6+UvvB{c>Ojihoeh>??0<UY&gR;`X~2x2sC)ch9btSiZgC#rxDkJ0wX7QnYC1fYGPy!U5dj`Q1kCjX-?>5VKeYC6|>_OQ(~#u=ndDfn)0dt<4EKYBQKAY(bx)4V=j((e3fD=i1i>Lccq9m_A=TA)kD5bc%erdPJ@&E1%tH1g!nhXRkl@%Z1~=S`JNLZOqis-aUH<Ela1w%Z~K+lP9>)!KtFkJd@$mnZyWU^kLGL>PV=mST>#pYU9Mvhe#cUFEG)f3q6HtA0)lALwd74`fLf?ZBz9?*=i_eRSQH8M&r*}&`)}qQCCywfjd$ZqeT#X`923qtTQAFvM}tfpkX2n3&1puM%Ce}RQ5Oas5_NPd%mQ;|9W#~^!d>;YMYe~1;)s}Lxyf&!a{fsZ6dhLyqEJ#)38goj5n|hY`M&u2K6Xi%j7;F58vn*xlZ6@IMrT`&}wY5cZ==;)8Bmm<jWtd3RCv6MKyA<$72jE;`@nA@gQ4;r7pCcUb!@vgPdv@Qq-7XN@4)A6uZ`>hG;c6&A>s)*wO6KN_2^gJ$B<xy0fGQWlA1DIi3xz3R9omXq0@i0LYgMqkr}u_>Zr6fX3kCNmj|C$RQpu#CPLADf0(Fe~h~q9z}Xjc6~OGL_9~b5?$1f;Cm_5)0~50a^ewNQ~*nvgwsm=UbSE6@h4<_lhs&`N+I}&W@h8#%I7#V;AQT*RgK-sZ|TNDtWJ*+WTZS*QkNcPseR-YI<+(cjY6d}q%tTJtL%_>Vf@!K<f^9o_&Boh>h%TU_uxSfSrZe3AXs{gQPVOpm%>}`xoiG2P{^;Et!no?$>r~cQ65nb-_iz1hg^hV%-*xa#H|Hz&4z(kSsKTEUhWRVzM12=7HC@M<32wTA6vwsH!W>~yIqMRF^<V<T+ZhZZ~G}1nM@~C3#)1TMrRs4m6O5AK!_#OG3|n_YInN7Xynmq>a(H+ZI-HOui^K-mbUDuTbLyl3?aH$oo~aO<q&G2dWn(N+z7Gr$QE&y;N1M#7biH!h6zf|eN3e}#;7V*(<@^#d<n+MM@;)l#l3L*WYJVyy}p07aW6e#PGW3Da}aTU$>{)FSA!KUQf|D1&@73_t(980Z(EGS0&!NNn%(fLb(GQ6S1lyEj!E*__9>&&p72gU<E^vYcdN+U)1?A`x@}KAhGbd6UAwcSNQ$WEnAKRhzR39NF&QBlViY)~nP#gc^ujp=<Hi<`miKB>K`lCC^~GJwm6kBk{$dV${N0e|FaKhp%wJy}{o-GpY)(8~rIXjq^v`Zw88#Xz0s6sqQrCQ^D3ZfjSdcE$u()@%%wG)%R;fkZG70){C|Jkf2+3koAXC%9KDRCaIhLNemzi}R=Y5D>|NJ;T`Q26n*c1bE<P13wL;<D)qA?VGH%((e`~cdEK8TB5Gt^!S*PbIc!^g2D8M&0gJ=bb%Y}jq~Ux_2cQH;qj=_qyY^SquipRSG{KRV8}U7u0k4hVf|Uu^F41X6e7BqYfaEm86z9DE_-oubs(<)1;fr;m~Xr5=M886#SarxBx7{V<2`oN0j!4XwVX*hrwY*4qS}+GEYTx%NXsoKIh0G`o)C+h~3dd%g!jj+#hVHGAeI@r{X`<!!r`oIivu-adPp5?W5d2)J><snhp!p)AMXK=uHzh$TvbFpId%;aHbQ>@wfk`I|=BK3=BRpI-~dZgSlwK}uHH3;?7Q5>t+w{goimMN)KwoV|=#Tkb}S5>m7pPZo<ZYt1mVOk;(ilW{~gm@rxs9n{f~DVF^l{DHNy|J45Kum9?=?Vonl$~=wRbB}XkR$(GdqL_lwb}#c9-qBz~lYRjCaiALTOcSl1TmBPn)Kd2adWDjTm3L{Z5yLRVlgdCV8OYp1@13;lb^MKB^y^g??z0;tdbL2%uO65gdbjXQ5G_->1cx`2RF}dX`K^x|4Z?m1f$1kSWJywDW-3BewInOb6uMwGSXpTfZll#|G^%)faz$vxq7(Mx#M4!uYXO5Fu12%JJ!%>}zL`aTdQ>OeIL!+&CkjgLUn~j%0-w`I!pmGi8^v-Z)(J_v8sp)@xd*~fVnq#Vd9%U@;^E4@_JqR)J1Hy2NM#Snn~?nS<WPI<wSTis5R=u=dIXnzVNM30&e6bTxk%BhF{rST!NNF(F*Rk3b6TEDWQ72O`#MjCP-g-RXqVk+onYjaOB#D;{mwYri-kQudtdgj2jq|`LvSg4_qoXDNngi&ksd%3RdTAFqlKRKk%b`l0;5@)6C5G87Mj+oGN)gEVAtO0aTZIGlXv!fH|@Q**&WLQDS&_<YO_z|GH>79x=aP&*(>B}+ytz;hUU_1Kt_y_I}i6$D=A^Ct<`C1aiBosWVoYut})@B{qD)IJz3S))?M$T3F{FJ{_^`L|F8^|fAk*qz=?AP$r3Ucpc74S&T%(hrYXRjlmlClwcrEyc$ewmBh5i=M(CwsQmvz6hu$3uPY%;U6m!S67v($Cjz7JTMPE+0Z|qdJP5)S)U^Qfj1_|A$CgpfeX-2q2GLk}4E5qWmJx!<78MSk6lciW~NKJ<%_N{G|Dp{jtNvQT$BoJ9K-vjWpU-M!;JpA;y+PbF3m)9pQFBYx9ueNBXv8$v)Oai_><9<D7xlmtZiB7?CeCss?;#!Okr(SZOTAMvprYvO+u`3a|ST39}n_Ed5jo}(Fh8-n7v^4Ydy|EO?;EkEM==Y)3W3R=V@#00QPo_|fu#E2ey#73`*kX=V3p2VVRV+nXYA9_a%rcwBBvP~2C3%sq`zgx<$J3u4g?fMd0C{;Q!9HIdBoenwO6<CQc<`AYd^g2DpbwyxokFMT7CbVj)RHE44PBa_`?-o=iCB9v7};=S>XsHul^RiJrr|>Fd#OIg;FHyvboDNKz8pS(a$~OZUse>krF|{Y7r1CS`y(*O7!G(wozCGByE&E%BPkm_Ng$z@<RO+~Ch9uv5**p0GFbqo7V?yV%6%YPd2eHCD0`iLpiY1Nx{CA?k~h=2-)t=3QOIhT1*W|vhTjz_Z7$y^Q7$t`7s95jt@Jp!mzrW}ToiliflC%C4XX64z4s6;TN>4&G@a+{i(7ZJDjqN5eokjT-$<rC_KZOy+eb>#s<)iQU@r2AB>}a;p#~nnz}>|Qd+ji((ilmYLhmFchdk=A%dhpqQ=2gz4#+#1K5-)1MwAcankW$<<zQ79#p=@m_#7|Mc$tAV>^XIm0D1ONM0%T+&xnTittX~}qmj00cej+9w!mE`wdO<OdkKE61b=l{do)+aeXXafZTQPg>o6=ig=BS?qFvw?h<4j%zATqna5<Hpt+KX4t+<n@fnc!w9@&L?+`)k;x#y%P79+_cm+a!MF^*JsRQS3m&+m_;G@q`a_~^uI@zeGA<;7BzkDkoeE~AeZcJpW(ScD|OinU>Tdy9f^8YVv)e6#i40}tL7=Q+`3#(9q|Qa1Q9QUkJY3fa?7r36u_NiEa5g(V3|>q+h9NZJGD5OUbH`0hyNuMW<uDbA~_V<cZ&-)=gz$*ch|L$=(@$=w(a^6uqdFO=L^LLGu_5GL1oY*ckx<na(gn&?te*(wh=VwR%0g+eYk6-tFU<{pFh$oH4_vB`!>7)31LEOyxbJ5K@SbOG#gVr{Vks6lh85GNbC@s;yYbDyC?EiP=PF@O;p%W^4m<jIWOeGR7XUB}ye_towG^*Su;;>+K?Hyxe?ph^a}f}o`Zw^KZ4W;R|Vvqo1153xi*h?e9EFsnV!Ik`2bJ8sy3(kvU6L}APfsm(HH)`;A8botQP;1=ysm29xa-kL>&R59Ro)W@1GGX7c-4wc7%kzANZvOCZvL#ery)Z&Z=d}GC~KDszdD&h(iVM?5TXijH$Xn3(s({4Np6dy5-;!R%^bSV%$z`<bnI~@cPhY#Q&5IgrWh6d?XTMDB{Y#KBc7}yL3_F|<V<|54=Ra(ZP##Tb;#!m?MV0_?c`bQ_aj$d7`O)vUUTVvBF7lWc0M<y~xa-eh|)weY|rtl#|B^60fEdLq%&0Ml&#h$9KUCHEGB6n`aQpPZHJf({Q3+6m~H`)SoFVS~{@!4^2|C{Tm+|<rV0a4sGozMqqmGMBE8#I@w(qmy*+L$r7uDLT+Ac#;fvRWN3s10Z4?(WW1)V+>pVJjN54VVr49)@?r@F&N<<D>OJ`j{8D_7@NCbsE)Q`k<6UAj0j+^_E{R^1LPnnLs_j97<5a4BR8ah`nMNL6TMlj8Jop({31UV`8Pf7=Y|$w@-LL;=~W7e|Q}F{`Hm5?sz}^cc()XTgGjLeU5;cEpO&zm27j;bTQ-04AAQ9rZW}oOB)ALp)L(-li3Icfqh-v%P7rW$q1je>23=>jJo?8{VB(8HO0lx2<*ekca9j&5`09y$jGB6qjQDXLqwx2kqh{iLEt>0Pm99ZV=q#&CBqbqeY2Y%jXB02T+I5{D`U2X;FC8_l&yCzs^C9gEIn?H6kl!jHUzbzPVQ{T8c;L1;0frvQE(9YGY~ejAgxsM=+izSwCb9z$**4@3@r??L&&95Ph*Ns>;_ROl%_OBjcHG)r;A;Ge^fGiyq<Y~bYw-3TmxV^3IFkCA471}342q70|#(}p|ke9EJI&pn$5wuH%W6P=MY=fTuaDtLdn{VF2$IVL`bvLsG|;{q*RDCS*-55&bRTn%IJQv@}N!I*W?r8oRyLUh^idUQp>$a=Y$lSkK9fx(E*$ERMQyAW3QK7sst=ixaP@jnwdM%ECh2o+Qy!C#k_k__x#w8{Mk`v=kpsI*Uy)`UtJwH4_BMl=UXKylKMc4Ri(_rU}fE%y0(ss=L;q1v-GB6*z*;KqJ}^VQ^uTn$2f8ib0ABdg?Z{EbjQt}sAFkSWpAlJ=q-2KQvc?dI5)3-fy99BI1^n)E+C+bml^1MHy61Leo6mwZFMH@7HA`2$S}0qb!a4gXsFrv(u5{SJTxcGF2&_MC}GJ+5$sWIFWe^`^=q>8Ym3XPTj5dv{F7yD@@J>RzbkQiv1tm;&}7!Afnthc&|lIap0NjZku4*aM`^8CIjCqKuBy_(uFR5h#1V$wH);_sk`k9eJs4&0l|@sCJ?#blH7&DK!RE)OTVEe-i$hA}%MiqYgxsb*>-I(_Ugio8cb?p|K$qBD!flio#NE5>pN~3vGE8carp8v-SdWYmI#gn?hmPGw>5Z+VueS{fx%w>yB#glgt=Lk)VXJ)-GGFBJBaayB>?IUz&62-b+ACIyOByuN<N{W<W5dFIQZ1G!lawBaS{Oa=Jo`5Ew(NfyHiU?j1G(lL0|q%~{5!)%E?`USLFdHUN=(h&2aEVgVk-f;)?lo}QhaK)Kp8O^XNk=aNY=_2>puVU5LtP)h}u4~dg}(6S0|3I-)$DEfD||?fn<p8l&Ng!@L-|&)_Tj5J_NcVinWV9M}b_LdxJI>g3euxNE=R!wHYl@Bi4{|F;LvAHzf@pr(XJ=|8Ao3$H!Bf7i)+67psBhf(D;%+rbu#Kaa(r7K!%m9DvSQjlIm|m@rw*ZB|sJmL}DL3tQsQt7Xb8Eh~^TE<~XMv}&PA)3?%F8+ji?yBCFzmw5Z_GFS0(&5EnC-fy=l0T>p{Ov|};qRfzG`^q56WiBr$l-#O{MPnT59_-Dlz{n&cH3ey%IR-<HNy+ZT+@J}S))cwu9+I!^(X1!hKRxzvzF9Nm&zD^JbUoevb|uprLexrivU*0~x>Cvi+`iv{7ph#LU?ezAnMPAkQka?zLt4B7lF^v#O+|0zs26IZ(rQJ!Q7EB-`yTFP`}O&s-@JMF;@Rq!u(gO=U>cETLzOvYD|*Iu4sgclp<E`#QHmod*&l7z>NXI;J!8o2jb?%gqt-+{51zS?niO-+HMrmfb=bS!Z|LAW-%_oEP>AeOUontn0g-bW*@zd}vr!q@=Wu;zCJm9^Ys?*IuEjH7wTSXl!P=>{8GPLzTh38Juo69Cug|ZUyFI<;D$7C2ljFJdmL9ef%HX<D(TJiE7*;)iZmq`YGN*C1NV8N%twtixcjzkUjfvb1Gt4;Jus;kp&r);7?4M;!;6XL-0r>`7*LBr!lbDZyu)Wx>=_qQzXtQ=68S1~t@aslN8e-EhXD{}|SsKP<fG<Lui4tVB(wn&t_g;r-$T7y+YHCB;K9c)-2)@{E9PC?!?k5<nifT+QI&lCi!y)2Zw-7P>8I0Dd&)p;=$HtPYAoSG5kl>^lbI4hHnTrbAuyQdVmH`m7ETV&!A#u-+cgyvo<1)>6H^<NGWZ5T6$LI0l<*~_E%hINWEd;U}PDZ7f^FcNblBJ8BvvpEJr37@@p-`r!#X-<p@Gf^)d`!d$Hn-G`uJl}bH{#*~q%mpU```D8%hUTrdXp+-6kp?_5Hc=K78%bX&518F_F_~iA{>2Ob7*weBW-~QxXn^ru_v@KC$SNjDnoQunn6-<+1ow#PI96ry-$~%_;TGY`1Ys|{&)+BRLzNV^j&Na*C*9;90oOAq?b(MK-`<rm(a@yTx;^S!JIjTI@oevZdQha!OMgx+l6Q_Qc#gPVcS=)Uq1kRxtzBAaNT<Oc$w_}beqkfH#EpHLRbQjln%Q2)CDfGtCNZaWUO=gC<r9MZg5bNVGt{{nPrTjxQI7$)WM~Tj#}+@7`f;!#(Zb@;Jan%dQ~dhWann!&y9VJQBum;n4}-7oYP9A%PhgQv4OnExu!CraxMT?Xf6CO0OSE3TWX~lo30_J3?7#ib+J*{b;{o8HTYkSXU^9SgX<jk)$#6s+-7M+XaI;HEBT<Zh39q=vrEm0oqM)?XgSaVFbfN&1~WvqNUTOfqyiEbild>G@}YqtjRs_CQNkXauPvs2bd14IPPy~@Qn5AIZY&q=jY=APTw?o(M7~Tx+ca`OOPDnx_cRq_bSoo^M5>)SjDTIjXqDJoI2o;YySa>jWmio5MB~Fs0MA!X!q1K?)1TkE5x#ifQ1Bg^l4?a+BD#-oq8xo-;=t%lNqPIGWcmQ+8Z!l~RawfA+5%G}nq*&1U+(V>u$Efg5^p5PuGG;d=9**mA*ZqD&euk5U#(rnpB+{9{@e2PFSndhqiQ5^QfcL!4Nz?#DO0$}6T(c@BCDjXE-i%|nvv<MP$vV%TB%m-H3$sywLz|74$J*iDOMwaJGnVMtzMT@zgmdSqcuBz`TgVHUattQH#}c&`0cS@{`?3<x3D2J=7|#Akt?VZbED(u><3<CP51F6NsW25o<b9ysl;MbT#E5+2(EN88d>QtOil)oXLIqsECJixYx*Bl_j&gHlds+|VD@y?omvOXRvmy(w(c&Vy^TI4&LO4{fYm5_FmRga#sI`Ms=#WW(Rm|N;-}>P7*{)eTNnV$9|8<uE=h)#-YlYHjj@KxXjCcXS&Qf29MsSOjySkgh(62P`!p)Wad%$0FKDs$G=G0}yPfTkd{hd!<^12=wBo<MnMPmZ{F7sxzr6aN7o_`S9j;t|>Yd-R<m*q?M849t-yNO0Z+`V3`4x-A#yb7}eEBzjwBGuY1=8Gk?Ei9ed46_%km_H5bmK{X<#$|fq_6fpn?&N2jm2k_$ZmP&*Wo!Q3oI9DLXnlUYJ?d&8vu`X<*~FLxy(j5?2_bS4`~rv92u*jb;IE$r)o8a_N9*32cn+6K>+RtS0}SlPqrL$MRWmnB8(hk;-u-YRXN7%&qao2*E%K}9mE;gaGYj^q5`${Hj?3*VYoaue@IAe^fssc6>0_h9Q!fDiP7jCm(i=GKjAV6ua3G#zuV>rm3@rd<x%2d3>(i8+q%#goD;Qkb!$<}iJ|c{#MT1DX+Ya|ID~qLp$xTq611AbWC})Zq3hn9c%%2;iv?GIwDH81zzrz^sbT~cF(|z`6}rymC5B-!Iv2#JYGkzNn;28vKT6#ni+B)H=5DMoHpGckgd}r=>GH32*WP$tJL{w4==^V&3jO%H^>7r3xsOd3D|@fzFK)2u4XFL}nko9vSp)p@_YOi6Lr#e_XtuXqATe$4z(u;$U>`n>uN3;6B~R%t&ozpIl)=+gJyNLXGgnhEQO@23kr2A&>I)J2_>)dZzx9mYi*0E00H~4JyANt0kupL$z&X;b7uo796Q@QC8CMI|QCrS^%$8=c=Sd)(W{yLJ#~6q;7C*H`H$rj;fv`{G{{h#>r<;$zSj7jRcrk>f48hx8A@jRwUdQz3pm|;9=qhEZ<;5%|1RwCLGJAv(W3JB61N-bPV6AM|Y_FqL<42Y>=dOAE25a0^e&+vY@9lOZyRI|ASIL_JdH{v6*Wcc2>u#peb2$dw)0j(R|H1-18p9IMFrW*HhD3O5cV7r4!7_W)YEi`!Su7Q50GR|D5-<-i&mibYX03BFqB658sj57*a0)&mMP_A0l@yU*thK)N^L=YBf3#-wElnas^++j^h%~tw&_kL;D!X(Ns6tZ`H)y%K^01$+ht#3Y)Z(04e140)SS+p3eOFrgAT>S?ef*J~ncofRw`Xg5-heo9c>r=Bct{egJ`CGCL1~xupanD#O&E!dY?rFY)4oycg-4eWo(4+{I$>lr(!-<b$d#bX9(p)@a$lpMu;pSs+k3CsBAvMQs`~Bb=|S=c`wVrp&?IgGR0*YEK($v)4N9u5#RfEd<-SbP3Rw(cP%I7CnH%Oq0q#Bpxb@=W*+Q)!zrW6Pw$_LeL$*vLl-O$W7-KLbo){J0bTldv-+|F+<1zt=)1;_*jtZ#>TLetR#*fD+4RWkXQ&Z|41Hq6q{g`zU!Vv&p6{`Ntx>b7hyN`cqY4~f)XvY8EB5I_JAtnfH*@%;hpTIqYD(`Ym*pihRy{BxfY+B;w5Dj&RxvIMMWyh+q8S2C;#ZEF~m_27mMMi6fCiQDG@zZq-^Pkr~;^U<m`DkOKFj+=6_%RV=OeCx)*z7IA`&9F2MWs}Wc(vCsilNQ*wZtlt(Q=kLuysleY8nMos6CV%22r2X^5GqRZ?b-}Y~g)x>3RQlX>~8Utv5u}KRPezKl%zJTp>^<;3a4#ibkQ`bi!!O+x2>4gm)pCcf;R+Ifp2v8vNY8VhG7QJYfuDGYq-6QlN4fv6rUYbDs?@;vNh6_ZOHB$ETOwyo;{u=1!!96fz|rZ__eY3n$_ezPZad<PhHlWKN5vSZHWWBb8*4w9jNDX9mMxqg=AkOvZRbYhZM3|2&KVCdD}AC^K(n_s1_iq_#A9Tv36NpoK?22r-8F5Q9?IT{d=lZruE;oq|Qg9-*tonA|ikV>XuEl07y(kw>P9+P$BxSYqyV94nu<_9`q-|IJ!Uu6vz7yE=dS^L6uc6<he=MiPI%$p^s5w<0lt<uQ#`Z9atU<Xtv_`kWGrX2V{sR#TD=E^f1v6QpTGn<3RQEic=rlBqWL7mtmmNbBJ+6@Ts4&$@2$=;~}T{^&e|xoN=n<1N85qK6N%e}T7(6MYrCk^2-w8XPMz&e$W@#La5w0)W%fMHw?xR5S({Ix@S}v*D81&*!kUF7u${uh`ySFT`I~<hRIj3E5w4p^?Q{DCGdM9f-0|3O>{&T$?CNnT^SSXPB%STz#fP_67von*b0r7wloG711H*SuJ|<k;7kIo%rRZjh%!+Aw?GeNtik0c*+mh#@*+}&Y13O&>$D%9;1;lWCi=Cwra&B(HNME_r5Ab_Z%&~HzO#c6`l5u9B8}Q;`!})DSTzjn}i6#I5OC$90PL-=<?zxc|bY1&-L4BF9<BW0n5PRBj_wv0wTB|jnUsx26A+7TM-u_6V9VY>Ty=9L=MO0JJ|dF4eQ#cH)_F62((Xq7W>d!&;!U;>d<Bt;T(5a`%PDokt)VWG8DJ|vv_B(dbC<XqgbOz7&Y|3rReCT;7CT8M;}Aup<(?B8qfM2=VQh9HXEFr)OcY;;Q%>PVhKuLK?xZ4iAN9lZ44R}VaPK6wDNCaFNutC7#bx{1|uNYs(nDw>Jlpj^T>t6aYFcNBl~a8_p?7=j$ePXs{Q}?caQ&ZCCN+HULW}QMl9V5S|W+j{h|VcaEejm_9Fw>r7e99QBog%bZ<b77Ex0fF|blIV(lF*EYq?&aJJeKthS7hS}`U~u_s7JCHd;5!LK$IkVG5>BgdE`C>lk4$W~dFyic@guBls8<knhN7n_g1zGy|Gag>sy@st`<vKrDpFaa%OvifCPgRaMD<z;mIr|ZM4MKAbrg7y5yR`=T8{`odBS0VS5NBaS|B}&wEf(N+VGVW1L*y`g+&@l-Fi*UU}Xub6sz_NF?PacfeLziA7cq3<6__MTK?9C{Q{b12A(YOj}e71^e7!H53s9SF`Low!T*%!r4DG-Bk>IWSX>{4yQUQ+Ig-AceFfm*QN>ozB9B}0;)OO+8Y=5D2~B<39JEHN}J76maL=3_6Bx1O8+a9L?wS~$zh&%VBTY2Enp$n_(8U`s$0%gjny10}Vrl^FFw^8UA0b9V$wdp$u$GMv?jMrc~Hs2E$*)_QJUBzEGQWt367+n-o^#x_C~%_$tPyuk8{1&!;(`N^v4_QgV|A1|6c{@@oEb>W+%=!@vvwL$m~&MRmyF0!w`zV>Y1=2T&9l-%u5+#~=(ju;-arW^N}+M`-U3Dsif>{hw>TI9%<P(U+Rt%bs@M8hi5X>10>Fpbh@D_M&i+hzA{m#t>aTLs@Z62_1N2DN~S3@M-R1exFRs5Q|$py-zE)lq9T#+cnNyjmX5tr3i=-D0c=#FPhP7=r?;J(@>VsF?iMN7ww#Tzf9+grB^LI^mnjXOe<&V91oRkJ8`-`i*b+Me|>cKOf)npUHRsz>{yZjsDP6zxk~>><87x+xM4Zl3m7Od!3yJB`a)us$7CEPclo^7X2`hktmfMW{*_M7@-811xl`UjKi(qwfk*9y*hLJRh#9vmk+`R=bB>62uT%D-H)3!u;os~>^=L}gZ`c;zYg^GJoVduX+L0(t>5|U2eehQ?z8@w5~XqX4XiYP6o(@pijlstD6^pz*A-%b3@#$*3bs;u;nriIqmq8*RPBD%&P~%&2oR8BwrYggNR0AB+6<I-i9*zYNtk2hF;h*@ID%Cd2D&0M6E$q^wH?Zkkw+XEr*h*IHPkP&o&Hx|1+UhXPtVBw-Yo*R2n)JO^0f`ZXUlEn<{RJaR68pgM-RS*AufLmLHVspW`cL%0wbe^$~diXutFgCMWv25TXQk2CawaF!;dPpK#Z;0!!-!xsj>GY5)a(ogWH<}(-&K>)qH;ueM%LQ)vvC|mQU=0-07_lY3wh*6OgR-Drx3viK0dhma-9)XDJ}A$t_4B#4=#U(5Bm3r;fsu;~e?0L%6GeaNQL9?b6k{6+z$MG_eXkw{bD~V4*QaKzYcaknk@1vp6~AF0o{LY1+Z3b1{Q7jA7<sk6LW(ew#LRG3f{sV$Heqs+kXatycu%Z|&dxb|F(wHuhEBX+u(!5HkA=D~51q0q?ti=(!u(07$#s9L*%DH!1GiU<7BTEVIa<IQ_6hdKsPEHF)L*R(R?%WwmGJ5Ime7(jB)v;r`(~+WL4IDLh#nV-{NhkEE-6p*{A&IzRYm`Sml;{U86Qg)&{=7`?nbS|$@Ww+J4c$7LUEJC8HAuNjQ_8|62u4n}w%vd%)Z%kCCM7|r0THA{#XtTay$F&7KZz4Zc3D;4x}=~4R(E*GLLR*)z)9W4KrN75H(Y(L*5!cNIzYPK+_?tyKgHJu)?v4V)Z1e~VPIpxv1W%llUQ&(mmlsHFZW}1~0r&J|GF(lSVk!Kv@=sOC4+oOc1t3>sd_!clktMS1qv7o?4W=_akWo9Jq6MZf>IDHwc6}Cz=K+TrbBWoaIfuy0381rcp35u^w)m$kSt8$7FrX$>41Mh$S=6AmN?W^)v-yGku?`=_A_3cTEc5ZoQKNkDzK@x=fR5i-1ko|(ANe?A(^b`tdRAWD7BxbQJOS4e5;Fm1TCaWGb;_PV8jmPZhRZE^#-0ibXYa>W1lHBRUm5K~^bV^&@CO~$%8_$+JbMCWqw0~N16_MUYDa~Si9DtO{K>ISGJ*_u;cA6L)Ow~{i`GZ#vI{slf=vX+)DyX%rD=!P9*JnpR{g=7vNT5~+MMAL%0d8P+qK&BC<xyM~pw1R216OQZG&f@<I$~roIuc!#Fr~1Ly++euh@Mf<6{>SN?m2F)@Hyz7FNUp7qhA<H{@T^6-+y}EiduJ1*QJv$u9ji^)3u+pipD*&-}Uf6;0Nb^)%8~E>;9xC>qB>c>$4liERSv&Azu5x{OGbi@Wt79_PwP$u;q4u)Rh`Ex>lIwXOf&B)QH_B@3+s)v^0nln_uwhNGWQaF}OlbCT`klw32u9OwInPlmQ1O6}Cr?OTRmxv>%)oTQ)flxZ@gHoa3-IRlfokEZJv_kI++UX}KO4!JAOjW;TFjZ4!D*(H<{DdqN<$bxGP=^<h6+G1HIn=sn}nyKa@&`-InX{8cgjmm9~X1i%n7SwIvY<pX55I*$2GMHCT3dI#pEkwc5_R$ToUzhQ`0LqpNfgLck6TDe@Y#6lf~#>kwxYszCp*HS)0<m;T;KEK$jt*&azUCXtM*ru*2aK;={G*TD6u@f<ER*gW+yKF1OlxogAr6|@8Sj(UhRZ=T~iq*uf-88W0FoU78WQsGm$pdICSdQp<PxO5KH1&M@vTX2eT{$6UOy1aL7s&M1tDf*gmIB`}cxI+|A$B&Nnsaj?PK%zCwPTr3mE#brJ5`z==p&WZW+je{bAVVjiJ{aZcE0Xl;Fqfw?aPCKE!HoK54uLTp~x<^6Viz{TdwZXg)%2^diepa*hdaUick!FmgyvDRO2`iCs{zvEm`@gsZ8TTA(t|&3QY$X@7}BWW%Q#97hjd-pPbMBzUN)9>-qe%O^eMbNVX6eS;?|HVe-@cd-hQ7d-8@w6tFsXpKvyi@&v!Y0fjLjY4Yx%TNm{*D%guJh{HEd6I=EN`yx*Zv0@?lfb(rQf3(7PliH5>=ui`q2$KO&;t97FpOkr*)ZL0pnZ;0LnVm7V=!9n|C6wd_2Q~R5S!%cEWQC?#L$0lj7~|;2;s34n>}zkQe|8C~f4l10ZgZQqnhG-*Au<AB*-uZ9-oA^O_Nm6BVo`0Nlo2P}htSA6z(|(V3Cjq|3VyXW2Xm;S#$H)^DpDm;p2zyqE$zJL7v+hw5X$PPeu3wvAQ=@?TrZV0x$9`m4>KFwB@Xk<H6vMUOfv>wT7qJ)7F@Z}@R>#~-KVYsV|G7T>Ln)}2`ymqk&|ELJD)6_%TKq2zF;KB!h>)5a84lGbsJ%ys<v6{D2lCFuxBy04W&yoddC&YOq#vvU@UJyVd^kl)fctaSX@zl969;cebMFY{^`~EUi1Ccj^pY2)kkZtzB+qOuiMGb*D>W6=Nr1;tZmJ88TrY%q4fOapSfjHlrW>BpWb84%ZBnpVkzcbCiNV!ra{RaV*127#mF5sl;#~?9FY4|$gL%u!?pJcW>G&xP9c^fslP<?^Xn^5hv<7(*8}rSLvi*klms#7oCwt5M<;l|xp8LNCG;@0(MW0$9n&zV7Ersk0Kph^b~__b54951f~niMj$DHua+XHN%;&W;pAXNQ8Na=(iLCn_pPwV`v#Ybx)b(&^i{OdTa7dQ#D8?*N1n7jinT+q!MIn2ABa2T=s!&sIS>~7}j$$0WiCPT+!g&Jr-kVknq%&}sGj)xJoc+IutNO{(m2qxK(LDx@^YB-&$dPs_7@qThNgBqiQWkZbF*Tvz4F_V@ic^O)npFA#qQcTf%n%Kp(}L(RiM(eL`QI*H3RmaK@z>|oxu@qqdYM(exPQ6yZDy0nLP<*Qz3#_uzJ{bHI8o5IlQaW;2h3Y@5!Vy}`*3DjERA|5w{Wec+h=zsUxVhrF<F^eODx=tz1M`5K-y7c-UynXELKNPov&}bMLKEDgi6Si*=TD9qz5&L@-Cxg(TUxP#}pb)oS_$Ea3ns%2{TsLtEX=9a^T3)tp+tNzZ@Wy9I+hQ;kOg@*Dn+Cr<Z%+tMg#;Z!S)no0f_Mll^!CiI0yhm+ONhoMD%yt+`|j$`;=TD1#x8vIIXC0hB`8VsgJNPOax2u*At$ROZy(-nCIVipRU#L$}iO^&0yRZ*Hsq_&jcWzHKnh5+hn|?+1(EBDg2m>h57b^PPf}UAP&CpbYo+X!hcDbXQo&L7J7DA)quPH(Egg%wz-zTS*C<jsv83PUx<dX5?pQ0p=B}&#xDRpDlks-Ne4#BT}<w%WEkHh<P*j@ol+XLURU0bOl^#5_?f2K@lj#RLF1mi97aUffag}+5tuZd!$n6t*2w?^J-3iw$_o`3_sA1ZY`Q|_CT3aIK{01q<ymQ>Sw}JfmK@VE*n*$T6zvX3ZFTs);ek}NRCn;JB~S8D<g$64q4`VyFb7D>b5NHttZNURtpr}Wpc9(7lypcXjrKY$skrhMx)l2vRe-d78KiZTE<w7T?guQ%s9~fkg^3wke(w{KEQbAl4sQbe{a2lyjqPvE(YvZ=aaLIXA>!F3Sh8Ij6#M_37fBq0n;uk$*RrK=|Z_JWN9;LVQ9gKjr(vGLu4=%PYzVtD07T5TK1*%5epx)ojU+sJv{#T(w}^E!_I4+h5J{_?$9<lIyb;Z=4eG_G<>2eZ}&3XXAEsX6Kl<t6Bt-SY7@A61Qjb}M&=rt3&v{d8WgLR8N+Og&w^Xr9uU2o&_BI8&mW(>R^@mtI=4=IP*K@UXf(4LEjJFdnf2s-O4_snNF{3MsfW@UCk}onff|ayh}oj4AopUVe7HDw=hUTHU0O#<2OocRLVg+JJXzL{RzYrG!Q7%<;O?&?qKf45=_#h|ro~~O*N`3j_BJK8jGgD4k#G#4q|KsdZ87D>7DdJTm)UL48l*6KLx=ETqJ0;UA1rI<FL!<~3u})yd(o&dQ`a^~7AYyCrE)&8Y%Mnf?vk8$s@f|L8jY2wNR3%0XY|f3ij3hM%bdAGYvRTRVQP$2X6}iOT>h$P?6oz_wJqY+*%$GAd3L?+wr*HnH{AdET9N7TZ?*(eBGDjdMPo>4Y?Zf5FX28<C3`8-gYUBU#b(3UkyoUPQ2`dKz*7Un(h*v6zhly#l9lEYK?fhP^i^4ywd{O$PVmdk<nP~qx->ei?EK{G_SMySSoz|{y~VbRv>cL!8dsAEDj_bSo|_@N)BN9Zq*i*7A!VQ+USQ*BMF8j2k`Bo+=Tr@((O{cq%!OdqCf&*Yv4eEa4${Sm>(M2yK3c&1=^A~Pa&=MO-ne+=kw#faqgw%FBst*;<J*~V0OPx0!eL<SHB2685mS`{X^Pa9tRSUY-3}%dw^4MXxdA<aMPw*bNg$;|kns*iKfHOMwu!-KPz*qnELbE=khbS|+2`1Da+j%WRVGH_T$qX))N+tA4RyWE>Ib_Klem|Bon%F+(f(p@O>Kw1;;r=l;wu6m&=L$#T?08pUw}K|1JnlhdC@a-Nn&)oWX^uKLps7tZLrSR32{uL<UKP$w+~GnMu))9$OWd6_rrSV47vaB|JedD0~M!?Ru8(?b|4NXc!04$q+L1%TBvT315{R{F-ay+t0T2WiIa=S5@>a?7c|!0?7yKA*HC$eIzm3mz<WipuH~?vZ!Pm&AVaN+Q8x<;9^_8Z5-;U9l)j4e4pew0SPUnJl^=>RjhJg<*SN~XP1U%U$r<cX-?;2n;^nG^n{vS1s2w=G-?!aw8MhueSltcAHIkLz0l|mgY{~b;fbfPe1jKg%hLo|ktb?ogF)eDdB5n)`p;-#BqAsVP)x}(95U3=+k0A-?95@cqtJm%ZJzKrm@3CzE{#hV?Re*oGan09|nv9cZ4jL>vSoJNNCEDCsaEHmegxs5=YH{yqE#mg+I!pA62<91GXI6|EOZQ#h(njDqwM(gpU#bWEua1*kJ5qkQK=A!#O=InmZ?r*6fjOh0&^6HoVnTVyDFlM-5`SgT<Tjv*j5-)xL$%k|0;iTnkzldZ((@d_32_Dv(aLCqNYfI^5iReE3B8<Z|Lh`-zDCWfb02(N%Xqeo8y}x%IM=?aKUnU?*A=42TR4w`fJ?u`sKzH*S+*}`>~a~ngt3TU=_oDWEEXRlm!Z)rPpQow7JC&s+`ker21cLjz$j%}xjGLBzuMdRbZKy3<~E;S8%mz1;=fqX`Dit$_|ZBXKhvdE9A>!z+T_FvV5~KO{lrMbh`ZwUWp>!*yu3hHE|FT3=;rBFW`i!XSKlG-;{HSmT#Q8}9wTEfAyh0O(ik%w*A#c~``LNR?W1Lh*gKRr1&VGSk+17UUu?Phfkeh%lW=5U;;6jYQi}}iyTs`j*lKDPJ0m7=JL}L!vO3jj1(|sUs%ph53$kMwi&Ckv7oH>Yu>f}mr@vUh_hPdo)=48%%2t^%X-WYJVDkjW?|yfQ!O24_a_>5f;9ESj+Lc?CU|<`KtZbbbOAI*|R|3w)lc*Yx%Gn^w2VJ`G`d?nXYMS!WE#=eSef)S+SfmsoNwhK)e7Iwfe1MvP=pGyH1yhyOY9u4&OvP^<i+$)K)S{CLLhUT2G>eE$+Az{s4XNy#M6oA10^#jI`01G>tqJ<^)%mMOn<cSMVsX)>UBuXXPJXnyU6E*~p%HTmY_D>VvGf#q;sEp#x=X^+m`evO-o-FjVi;)r$cRuZZKf8FPSV}M&-<%J+ZGNUeb<*VMT`+OS)zWBrb`UF#9M_SwQDR=6=&>IMGR6_5W4hL=UQU6$F-~Ujw#34CSx<6Gh);OZ#PdN{(b@5XRBD{O<&2+FJ1R_lX6Q|_Er!Z9u*<5T5?7>!Q10ece&kDD21u^s&W`b;?%}zqE5EzP{D|GAHHHz1K0;KSV1Tvv!Ys(?z80sk1tC@U(5a3x%2ZnwTxda|NLwlX~C?)m6~-iFB1(9st|XH#8%y>ShQH_D<SkcvRDDw$Cy|~$!)}G4>#3DXsLJ@U52QX2zxx#3GTviOUOf^#Nex@OZ!(s-aJaQKQinSc3lEB9#-(VG&T;B8QZj2kCKOC>otrh(u%ME#|fl;6nitkLg4|syO!tIs#bru?s_kA@b4CG@?^1aeEg5A^xD6S>-mXQvtaP)qsS5@<rLFqiGqD%mxIO-QuAX402-kINihUR817Csg(5>~$(VJp>}_RC?0z3U5*n5d?B1;?*P|NG-XC4tkDgB%esN=mW6O*NLy8HBf;-Iw<wSZyc)d&6eb3!sQbw}RX%4gGqSQmo1$neAFoN`%IoAoH19pyu1|`3d6c<`FQk2t(<oh27lJ9!%Z4RW#(_dy69OdklW(JqhX#eN`@t1!!{(OAP-|FxFp=ZC|d++tn{q|ql575`wzVp`)x$337OI%9{7_5+z(Ub8et5io{CYDsQj>I)#L4k_7$L0>^;Cc|<KJi%7evZxl*FX8+UIF!={D(jNxBu?X{`61&_domh|KU&nm;dSC{<|*)^(=q!?XP69^TpXe@|x#=bX{v)k1$`HjV1r#_wW7vJ+b<27C-?QU8iF?F9;rr<rGel_^r;((0=sqz!s>b7(h)0a%@&i6mZdN!)A$bwlECN$*ScTm<U6$s11~AvasAb7QjC?T*Ln1dQJR&<#RI<1?}f!gxCHVtk6r|7(FrMH(iJ~=!OS_@LL}F?c4P)#-IJiZ+_>`@1pG6Ukb`M`Q2{{Qo(GQBLS4@LLFMi&{nf#3!GSS_Q8Nt#%Q2K9U<kpij}pZe7IM=#wU8y@puLbNG|^$BV}-F=!ZB)!Y&iIasJqudd<Cw+NaLG2v)XmURt1|v3<!np~g9ieJ)3ZRP5V8z40+0x@DDi(}Ch5U+}}L`ySf>N?~=?m>fY95y1o9BXim(D5W$79ZJnNG@DeS(G7dqRjh;6oRE69V3;+BB;Gdl_Em`X0R=sz315{aTrEt!L-mv89^-PA@zr;zkYdPzF-BM1O~zztvp!*zU7l$~wuouC>%}ip0#$L}$pRe;B8%a+=hPaOX-_cJTH!g{q+0776Xp>-?_}yv?`jQS-xFQ=I>>6>cfFu_SGsT$V1v7&Wy5TUE?Agye#iin<1RZ-X+onyr8NNL3`JumoN3z2?uN+8#U~(D)aVY4Dlyg;kg?Yq|Hy^``%dtmypqI!cs?vzvFKt(n-0?LZFk$wXl0f`5#eErA^W_?uuqn8YECUOX16MJiK3=4`PwG5-kMdJ+5%~_;Hja;6x#q|RrgXm?ts6Pe}B5p=f7N~{Vm8ir;sD19E_BEI}<dXERE-o->NOd@-B3Rv@>G>tQ4_YIXzjKNQ2m`ueB8WC1MH_vmrMpBj0h+QP8SojB<p?J7Dz7;4QM4D!>w$g#e9hD?Dfk1Ynm7Ng1NSXZFdt4zZA5XUI*sSc%xHTNz;k8Qn!%!Jf=Ak~Uva5`>Hg*lvz9pI^$rtMkZx4S}C+s|hnDw0tE1Rt^a~up-obE~;mW#Xew0pTR}Cs}9$MF|KA+VQ!&IwU<7q0YOa`5ogw^u^4PE8z0UTuU}MO3&Ky<JICLiFU((TvK9x;mW3#1w-^_ZWF_Keq5pldl$;C44sHVpBZC!)U3ygs)rbpL_w@-aLFm13z@mjxR5IQ@I|3ak_=cm|?>>IK{Qb+<D|0+uttzfAb}0+1`N<|*$e@C(lmcNEi<<nPaG7wIOGtUfm`EuOD78^Fj+~87Q$`*DqxWR!HSigAsiU9~Y|G!Nw3=HF$2jJ8b@<Cw8s?=#=^2I_gXJlw>@Z9z68cI}N)M?S6YUX!K9)w#>TsfwgW8MjrHpAD*#c*rV-5uLg%dIMQi$w12G9Ww5+CP{H*a{Z?;3yO8=L<1v){k>kDHXGjVDJ#p34yh$rk(Igb#^*ai7SW;x6{kX9!ec?kXHaW7Go860(<vqs7u(zgML6$}JHFu_S(o_P<&xI^<p~?d#{~u=leqQvAjjnF4E~EIHeMVY|I(j=Q{MEGdpS=WJq8uaaRAW3VI}la$DQVcVmzHUS(RjY~%(FP4;>#llg)%-7^!?W)<w&sJru?=6!-SKz;IRpPDG%YBSuX-B+=ZR|ySpUjO7-%h_ABD@2w!;)rlL2^ddw;0%Sa*4XscvdA2HEExz&VseZQN;)hmsAa+erVB;0C^jbkH2o9{2#W=P<$hTm|VF$L>2?$iSky`46sXPQmo!>d5V@Ch826$f8CqmulLYOu+%+6=ftrUV`pt(b{o=UBqe|8dBh>D`re{)_4MqfW6b4y7gv&x)*^CaoHUT4(G-i1E<pxtLE{ARgqZ2A0;PobF2u<`7LD^@rDEwlTFpEM8);d}X8V|Su*?nJ-V_(d7`<7P?BqS~0Xh$;-UHPiUMxk=f^0v%TB^zwx9h#`hnqKZnS(_{24N<0OZR0nms5Jf&0NjzLNc!wMKLuD(AHg$)QxaQ)QZKZPD7y9YYd@`)IhNc=d3bCAoVXhl{`k&t61=}*R&+9<ox(envrjVR|}`EYLHab01F9|SD+`5ByV}ufazVpEpf7}UyTHfM5{KS0dx!zK^G%x0jk%Y7J@0V`bqhiD6v{_?QrymZf;6mR(w9(Ji&3VQP+5(5HwmY3n@ONFHX42L3VZb$|#8n<>H45rE9|yv8U8%7#g!ta5ZT5I<tEQLT`1r{t$<DRG(iGE$#157MsT3ZYAUF_f9b>StN_bQLV^COWYdXFma>tsCOX(YD!{-qj$h9_*E5X42gnH_ro;;XK#a8lI{bc)I3N+>oiF%cJ^}6@K?-5|KaRzcc00`n@Vl4fAW+4EBd{HY&Qb8<h4sS-a>+0df`!18Pmdk5t`b>XgG!>E<0qgvRk?iEy`>}Rz8--y&99p1B0)G!Ot%3$ro$0{OZO@*RM8R&oJba{IJQ9i8*bz1BZR;|26c|gCM(uM1`7a8U{gdV`b{GWA0KZW3y+?Dk+SRCe_y6^iPjvpF5H9`7%R%@1@J~^*r&(0?=O=VE^mcBlLS4{Z5i(3lK)U!Pv9-rO8*I)xXc3$c8gOv@C9AWHC$%4$sr>i8W^mDUAss`hk~pE(1N1dfP|>7ODsFzOOrUmhoM?L*5Gd+4A)Y-p^NE!AEB{{^9nu@srhD5dumMLfObcg2lWx-MruW@*8gBOYcCz*;Gd90VW3wJ&=3oNi8k8omQS;Iln|-QInM2Lo`oh3duOP!}w4px&zeza<)eK@8?_Dbv%AOp!{I10It?!+DA7oa8?n*Pqt~LgX^(^m2c5Ch<tr*yE}La`z!}ESv^W=M4lE`jSE4ILp78xp<4aw=8skt)js2#Pz<xN^~Pe<arVQF@5@`9Pru?ToahU{Y(I|R=5(M+-gl=I-l~iFZumsD)?(<5W}B6bA{F(`^x7&kH~#Lu74}(>Qnc_2;^$;+G1m%M#c?z-Z(-5G;%D2m`oZ!RS{8?d*)6~|JnR^8r>tnSn0Pd`;1}XZvl&P44Mv?Y_9=BHENNEP!5n0;L^MQNWlCIUt_Rp|s$N`F1U4=3asWz{gEuDueS$+Len7K4?ouhByUtXhT<vouon{*3Ta!`BOjbL(Wr^X8-HMDy*Pfbk;~CQkBjKUK`l=h*)3X`GZ!UIYn^b<RU`r4(8STz0Zacr(F2~~ME~8)14Z2%j%sDJ~mK<_5uu)9YKx%-ilVBvvtIJHW1M<+;>@7*XhGVPu)r*^_|L;Hl?&D8y9X4E@wSr$P5prD}`sq65-exUk@U$ujtq{tRE#7UHuBLrz_YcUuV1a2x2z2QwzzEsife1_%OJ{AtA|<spv50Z&;Oz$sqyl*)KFG>_WMylYhe@K*$QTsG-gU_lPx)l|JH6d{MV5Dfm<+5vjVa3}$>5s21vzsYl)bMLEm5ba*olk#;SPgs6{uyl>?gEGw0vFaUTsWVc>4LZwe8tP^y$U)_S5rm_vf3dOLfy*)Ra(+(nMs$%^jYd>Z}Y78K_u+$$cPIY&uJw48^KU2t=g>9$*BgX>2u&-l~-Y7zaS&d<4yVdg-4pc07+)JMkZ!CqW;qKKD=0GxJ}p8^Y^C+oS6-`ck4kS&kjuGGJ@;wV10A8O3{*!5yTQ3yY}n?Jg`d(Yp|SjiaNLqQ;k0(6238%+Nx?s#cUL*=HMZ28>p`2Dfq!5v7&X%FLY(s(!<|;^lJNc>;93bos%?S*7p6DqCD-)j-HXc#;R{QSLINY70`ahg_-*O(nsYgVnOXW(-ycWldn{8_0obilZYLw*sCG#{t2+i$y<Of~0SEECOoRe&QEBzelSFsm0Xyug-4FPi|<R|Lp4a-uounPYl_khLv~9eu<zD>T}{g)qRJN`c}debgK!cjrJ*}5nHH*T9_C~WJpX@IF(@_?u5N|Y*zNBL;UY0XZm!N)%e<3U1Ljc-H)$CQ8}wsrN$9X@Bql^KG%XL3+Lt%r)e6nppF?+@J@Rl0@N)_SM<SRHqvA?J#;YEl#olb+;PltUIxn_ombzUtQF+TLHNp?)*Yi~XP?~PT-}+S-!p%F<H(pKK?=f&ETe}MQ6D6Wio0yc&yu>O?_MBL8&y*+Zive|de0F<A{aTA(W)@i<cw-+aWm)9^^e>G-7`#o|K5j7wtn&Z_x{VpAaZSU{$$1Ax!C;aCXHV9(NF-e;@<aatlT`&LloI|h{YPHWn!xriviG8hVtY_Ip>gu7M@k2Qt)j}U)wOslO(dtHpu;Sj_vhV&UYPsf4h2y|LD?Kdb9+_=V$u!-X=I^G8T>!QU**y63O_^#)gvaayhdGCg?3tsW7^QO3I})ZPOUJMamBDAdw4muwQYAsaSb42!fG78uO4ic-?l!<6mF-vDxE~7BoIR?-Fmqq_|ZgxNK!ok{lq%%@fR=!Y(11Q6bE%2EUp{sahw9MaR*@v`89<WooG;J<Zfo(Kx5!w<iuQ*W=XU6;|#j{GP92p6wu?U92Fsnca){*|_)I4Vsp|li+EyTdC?E$LbZZyBuK9mMm9~U^Qipk#j1c84xvNS4o_Q-<0QGLmjyrVmXO&J<P>kA1t4&lG0!0TIC56lE=x+Ib(FuwsZow)|EHaGXQ<}jm9eQ2wfK`tzcM18bn;B*3^7@6e%Wd12Q+z?kcQ80W_u9tO`FSo3A*MpKo`cetp%t{G;rH1Y_ti1F`+EB13W+nXMO=ywd>NE20vQ(Tt6|24U<*hoX^muyD&3MK$5jnIkkG197UukAIVnT*iUdyVpUN9RHG~KmMu~>O998U3U$*7?W^@w0Y!i;b)f=2{CGHgY7feM{+$5?xCev+YmB}4n`M^HBIFrO(1K}3@C*yw1$)pV7`*|KVABr*Hrzk+0)gH9rvf3C{xaAKaFHH>Y~~weqXV*I^E~er)764Kscya_Rd_Ad)`Lqn5A{57H1G_xwz~RRjZ6z5lVwXrkV2LsrTB09|Pd0o95=6Adn?BX3^|Or0oFYU7F=p>|7`n!rIllNRuUy5u*_qV>j~N8o_d&Zc(vy%jAt)pB!XNBUJ|gUw7r>S_oVA;>d+l68OC?b4d6V=m*0-EhZ|3P8CY61}F<{opazJvql-%8`0p2(6YtC=w82}Nt6(@Tg1dbhtuuNo1benj>q4BWu@q6SLdVLUtQnyUJos=mpZ?H{@3{_SLZ@><w=jO*1fvlJ^sHp8&g>AB`!sr4C5Mu;X6D;o85lvF7X_>Gq>p?GFnDK1F|LTlB7Yjl9Xh$Y>a4z*&-IqAsK<MXbitlJB*}mEyjKDilWD5pYz4GIXbv0L(EYkMP)*XCwPE;IAqu*2#djr>SQix#mo#tT1qXpSuLRUUhS7UC2M1F51)d7+Cok_^U!dB@TQyDC5ZlNoeXV3ou9H%&X8Rt#n0OKq;HQp(S3%#ks~nA>ZSl<QIJy}6)nOWw~-=g)Xo9Age+jFv$!Z?1tuHGAw5FhT^h4dPU{piq)5Td*M%S&gdVg+srzKkEbIyd1A#~CZqEoo(#&~|sxVu32k_?RCmCj$1twO%Vb&gvIv=;Y_lOdmn~ztQ<;rbal}4sR%!Wgvgvo%IV0$-br(Bv<!t7-Di!@5BeIzJlP_#I&Gbd8eX&=O)oojC~muSy|X?5@rl_LP&q0t}T<9qVvT5T*wpcyf`?S!H4L6)^~pZjTCvcytQ40l?Cy+n#ek1guOI=H0ZWuG+E8e>D5BdW^_bqb6Fk1g`M%=yDbLGY8!5`mBs*ykOL+G-|uAA6AL-7c-is<2U2h^h5%1fEkZ7TG8uw+I&Ls_Sa!tRv?bU7Tt*i+d1T#gwZYA_aHi+%H!}ps%^I^dhYKax(q3bu#^pbtwIfO)1!ju0{)mno(V*0Qn?amDG5j__S9#b5GL;J|Tc{En~FBUE*O#uOZhoOBfn0QnqM?V`rmdFp4rII__-mxwBn$UEV(*Iy_!_Grzu!13y|~=Q<8NZ%<x7&&4bs-*YGacA{%2$$v(Y>?dnH^JIl2X~Z{`uzVLxBdXDrjv8a=hEVRgM#VI|)7?jib(Uh^8H3nuzy}z+hS@_?gl@_IIBLA5Xz=2)-E)n}O;c&6Opx6A8#Id*_4JUmZd72GakFJBj{%#+SS?4meICylA=TJR<Oa>98fFMB<&xdolItj?v6VO_9C`h2@7e?9qwCnXsb_dH*kzK)3?VJ;L%<X2H@@L3hkrHxe0<B_^zZ(`r*B;GLE0s3r)BeEnY=XnA{)h)m|}rydBWIVATs<Gf;mPIwX~ndY^4Wkqa>O~xV%)NpKUt4K@!XnM12|6{)zAql|T!t`vjS`VmX;oAH}jjCu*&8q&bz!uvqBh6v>=g9DPU`R&Of$_(e)QbkOfxBmdR9Q1->@hPSG)tPAB&wr`VDutG7p(O61}5Fo(`PjBhsXdM0>5b1Txr&>er73R!FZfb*CLM^qrV?<evMMo$DhhNCGnw$I4Lx_GUmyRg-S{bYt*S6q`EB$KUxH^yeipAe9IDM&)b?d<&T<*VL|Hv!fUJHe%cfY-!Ry@A@?T>C;0sVGWW54zIv$K==^Vk1>zvqAL#qE{O5AHtx;~N6+ul(jumv4WuzP3y0ealh(*Y_A_y>T5B{gX_{oI}j&$^9WoChH#6Y^s;oRSUPpn#em6Enzg~m+gbAEjsh%By*sBZOJ98F~n3QVzke{90mA26Vy*%V{ZBJdb#w21=F8zKZ6$)Qq2hw44m1e-Zn4Qqi^@_5vOHX?Sm5C(R%ohwPMi`t>l^7Ij0%epe)1qA@I@|&Q=!rK0qDg*cW|u^zi)b3H);ps$YK@z_0$zqT%ppRb#rk-hX--`=6X;86V%0X8PoOYybSbD|P*>|M8B3({+#T`Ng34$xYkhP0k(UH5VzT9KfB<Y4go9gniC@V{1j)B!&V$@^D3F*MyLU-B(((i##H#SaffVDny8#8byJb`DobPY%x3Pe|>e9#CyJ=_EKeizG(u2yof_2Z-*E%?W>0fl?d#!c9`IGzk73)9$ik<`*t+}OP@HrUgFYfYt=r9e)2T1LWfyUd;KFT;k^?vd97PN{@zXPrgaDK7ngEkrL9|9$`%#WCBvgyxff&hvH|Yv>)rF@8!`0{J@wmvX+LA9nBV#9huC|{E_*W?x`L$H2XyzH;b;^%Xpe?{3C3~R+kN&qAS+jj&RE21=kAgKX~cQ-X6`@nf4WBEzgv|dua-&m$5&@Y{G+Yc{z<cn(KzxpttjRcPT{2ZmSW`)-i5KARYJ+KPT^dr%!s{A$+=dAfr5Qll~gJza;_~5q1L^fTA*e0$@Nhs+*Cz*e?3_F^m2K+45l~L2hnIWF)s@e!Kjztq`*Uja(7uDs15;E(m8u99F?Q-H+PsLPIU@(jnJ!<*#=XqMGEz1Po}Ib9ILW-Ao$_sY-+6(ezwr6RSS31-6;|fIb8w|h`}n>ctY50BeF}gud2BxO)O1GN+3(deN2SmF0){vkJy`hhQT7zbBvNS7^4I)Y(_qa(@Q-5x0{|zx8+Fx%@Qd;USN8yS9<Rz&)TT{=W8wKmln0pug>6pexv#D^QCvOUITvk8X45jZwN#`KR1uRw|34ydG+b%R>XN@=&O`W1z>|u&Im5cVlgfALk<o__gMj#qN5}B6cf1)rN*FIWm21j#{4X=wJ-w@8c?_l`_$2-AtAt|V{_wGEAihh2z_$Pgn8ROa6nS4|HLm1z3$zt1q86qZkYC_HCZt+8itNFmdwVpdN!J-scC^aImKl%3(d<dce+)UWc5ZmR4MO(@rB3P>+bNAOECWF@|B-+|Ngzdw@ACj=f}$m=JU(p_OF-0oS#wn2;{SKSMBqqbMfM4m+kY7@s*JQqG-q=sQ4)<oM5y28S5?=jCG=@l`3AIjowfakotX7S#*X<2v$+oNg~zWlAbAGk29)O)b@>I<LH)<`lZWzt~&hu;{EeT;_1rsE{{FCsIxxWWYuR(&mG@g1X3v^hEsk(HLy#jmo2{pvch?APnl9_?l~o62-yd7B)~CKGlmB>wGI*|L>)B@l7Vt)vE72`zxRe|xo-PCTyV`ILPFr|Qt6Zs!wK@%^lz2kF5$<$8~UcjKw!)@gu!#ljKTwHj67M<K66Yn<R0w}>6J>w7>aup9l`NNkHP8F=S!yl<U+tcSn~HL%VXEHY%NwdHNS)?NJzk5l^Hv7=duShzhtK&QIKa3qoVRiIS;9o!Ysoz(r2ua3xR(wYlt1Jc+q0XA=Fx<MbM+qaCfK13d&Ee?#bRaRns?>l;VB;7?7ii=13UxgD!V>X**uCmSC|kq1BJRlx_v1LTf4JHd?8aRT_mDYeKbnI%BsFCQa=2rr<afe(%J}b!Wx9O{ep6q<?jNM|qL7I-Mo3K3e{1{OIqt?YtYQ7fP1iBeO&h*}n9pyL2zDxff$PqgPO(+$?p+(sK+NCf8_rgAz#65@$0|bFIoqy=b&|sLf-5@ERr6Rj6{CybA@36hOiXNg`_At}(57m(oIcl+GGkzzR)+<c=01W#m>kDTHptrC68@RH$@hE0C+kXk0yF!b8yNWw89<+|qpXvdh0oQCh+xGvvDcR!}J>c+grl?o(x`gs2?)Ox^y^YRa9n_@|A!jxl=;sFt%u!lI^@r;Ox^vG$f|<9V#R-a+5@&&tu4F_V4$$p-ZC(s2yNV1nf%0};#SRa2sUvg<NZV3}GHFQ>Y!#VE~C_q?_56l@c<8dhsgom#idYkWL4pL{_+^wn++d49f7mFtfAug@EQA8cGAu@c@QVTj2c@r2U}AFySb_Bq=rmQBO7<jMr2rc&cn!00ZF!<}inp^U%)$V~?II&-WoS<M)lG3rCz>PG!{v1I)GGRj${U%qyArrBF%*DOiJMKL)?VYH<6ApX3|oLnJNgXVW9li2T1Mn+<UF6<8U_HQc^dDx>QIl4CBtgX5ZU<mEl`MhuE^T}E7@7dPd+nj=;pEy`N0H3$yQ*i5%@TOWV0p5kX+X^)oMaiRf<6@)C8X66;h$ns%RV$~~y_1uxi)FY#;4G<ODBAHT4)0{;kJi1q4N)&e#Ndkg$~iHnaHkgh?w3CCo8Owl-pw}NzGReritASybrvWQCg_+}7BWX<(i)3ll8hY>F2P6*)#lrwQ(~TxQ#3xq`G}PFs9}BoywLXf<#uk_l>hiYZQY6|#Yi48*zz}MW}n4`6Qao5?Zq2geFq|FvzG|DjmWt}R_<0*MxGN(VTtY$szqQ?xwN2JH1JGKbGP)am=C}2Ta;$yGj{~@*TJEmE&bQ)HPNm5mZT(>);VPc&H*&w2_B$gZaCj#%xs1|50k^nN-D9WsOl2t34~Y&2^OIcCR(Pi!QPD7dmRZz07LV!;B$9gXn6d16RXBx<zUE|H~`td^g$xNBD>7c9pZGCuO9qt4H94pmbGPcO%h9p?m*vyfeh4=Kt+u``=Wf6rl}ohyXn;McnNpw`s0gjn6vz1kOSAq0YcYMgZPjov0bi=cgi`rUVhS6N0Vu(H<N3Xx4@}e<f}zQYKs=rIG12>jG24OF86yFb-!|w_R?7CTK?vPWrOGP-A9*Qoxff7B(`uIq<~`V+Tx*QV76bzJ9_%x{R2<kbO?-bpDwgrTbreT!8sVifmmG#5t2(#8yTNHN{J~9sUzYPFw(b})LM4$DINMb|022DN<$?ktDEd6!A3Wo$=s`4dC#-A&2k{_ax1NuF$OHk)HFiVhSNy51Y<R!)}k<S3!`_n0L)6Gxv(H3SFqp0`_ZZ1k1M^0-G5xo>MqA>Tl<(`EL^jJQnH^>B}}Jq;+Wo4b2@5x2T1tnKqDEk4$PMIbLo{*MK3~!jv~XYW`h%T8D(gq&ZSyuAfu*E<)}d~5d3uA%)K;)zF5XPFI|$a&RR^rK2LnErI~Kj3NOz7JCA?3UQ2$klng(-@#p;X4Dip-XPU3EdwkRXp>0D?w<XSun8@ewKq<h34(DZ;YezG>TBdJ!MABF~N(mzvFH7XA!w`X^*0{{x-Brq1LmGujj69SN2k}=w9v4N;AKv*~tOAm&=H|0)3+;+Ru7DlAZIIA#x!vPF?Q;}5?Ss>5Y8H=c)fimtHW%MeieXllc;zn1O@9=NhUBs$Il?dqH^gJP{i_^4onmk|+sqaPnL=>$sfXN|$FxgN(b+;JuLwFq1B@{z_OT`wcNiVLTE);ofnp}BRa3@EEvG)1XV7-U!dI%*XE!?r|Fr7pez>i3#qM~6?so?EWAOGo-Patr=c$(==pTCGwk`tfGT}GN7;a)(D(9ivd+yY=N$4CX)R9?$ax0xu*Hl@0=E)XuXP%L;9`W?%GRdb`7mRN<4fkCp$>e*+5i@7YZx6HbgMGrTa>1m^E$2AhE3yU)w?*8tG-RV^E+H`qXX|~oXz80kX3v-^9BKSzSpBuZ?A0;oMYZn5D&OdeVVg2NEBG~U1WSZ|V-Oy6o(8+rRBXW|z$jqUZsn!+B;CcB;;ad*!tB_(!sxDfTVqr;nv^G&6o{+zqaSd~HSHhIS7?8?9_lVI{P9IwX-lq`n2hI$u;1^3fe9ZZYm{-HAnf)jixj2L+BuFK3~RmzViczaf6bij+rj?T1Zs$$j(eW1i1DFgb?JU#&tEP7{N?86sALGLIb@L#Aq$_v3FIC_zWW!I=p9(~Y+*o%#a?8uWoWbV5L2PdrG)@va4w}zg6`5=TCGG$xhCyog$jmo#Kc=M@gke}-?v$#VGys!2_u8AAu+^<G#?3e836$XSfa&g%xfCRCU!|(g5hjgQHmW>BNr}JC_2YX9XgNcYf4oQRqs2vTYK?O&YA^}R_Cdg{s-Gc-q;}+qa`JE&vA{6r?~YR-2RYV0uxi@l15S<;CJtAd7x(Zl8U9zoTr-&C#wfX8?_o?0jrTEa?NqnV`u4BGJbNGBww8mF+RTM5aW%5W40fSl}q;1ac?@y*QfOM!O@DWePXa%KDJkwTqrx{Tyvk3nGi-v!mNd;DliV?&1o@4vt*sdh=BDF8oaM6+|mGN3k-`#vnCPB5?zph?pyb`@7Wvm3nJ|ju6epyv!(Agy8*KKvZqy!q)i*N5);RiKzp>3#_E_B2`#>sZog`YkC1sm>eI#5>&XoTBJcKme$L@vjr<T;7(uXP9!SVy2;svfIboMgZYaf=O0=|Xj0Quq(z!_X9YGBOIylxr(V`}nt|Ha<U+c{7QqXF_gIeFSfV8IZzdiRJSDDriRwd!j&fA|KZMQZ(`NoAXS$P;RSZ$TowIqpedS0pW4p46Bb4UzUx3mGSP?ss!J}LK7J;TQ`C#YK_))8tQlWVf)8a=9IDjzVtAEwV%ovf>~gXcfI+zh(zQ~qXAjlb*)`Krd&ei5R#i`cRph~@m8#88`9->zMh-U;Xq7{<iMATDGX+~<Uq{Z1Gf)JWb+w@PL9TDTjT(mq99rM(n^e9X#UHA(!9htQAT+vFcBR&S9}1@`|41WfnaZQb|mZL+PIb~%0oOZOF~x>75Ne@a(tS+cqmLz<0^Mzs)))z1Q!d}5+na~%a5#y|Qpa@hYe@-?CQ>>i1?w@LQ)rU$Z&pK=N@L`(CY?ZG(jTLsPt55EWgmVZX<+x0KTpZ&*gT9td}z4-Q*2GTeA-ESIL(Um~8HBKS-j%A8p5YMRwV3%PWrM6_ng4GFJL8iJLZ7V1<;FM!feya_CadcUSk{@lGh`V-W@U<|FYEgq8BsjyA_ZT2ia;vc=-=UNQp2R~Z8I~h7hB|>mve;=H2}d^aSnT7OEK2gME%$@I-!ge!@6z92B@^yg{amARn{;r9lG%5Kf`ti7U3`)UjKz7EjU>N<wz?6mfDDAti7W*dD<~UHXjV@qH=@Y)BzIUbv~o&C3KJM_;(CyxKcs#1_?BP%#^i0`ETaG-qtPDWEb#<*%0zFKyK{U8I1kotv5r`iF?Z=zn-Zfzl4<lZT*X=&vELXG=2$ATCGh4_!lC9vmFqsb{(Pb2&$fe+lAy&#t0Dm`I9R^slZD2&^IHW}d<Vctezi`?g5sPKn*j^N2u`1|4$t1DGfA4IC(95~hVNHO7uI1c9nz7oFM-jQcpq=Rjev+5DJ}=;Mj%=FA+t!-J>K#*(k>~~R(hO$reY<cj*=5e<KkKhE=CPG2j%8@c=T?>IaZ1m{>Gsg^D@?1Mff*oZQ!48fg58^Xdh*J(<P&FhKI~=Sa(?iQij=#-=bA?u?(Q;mc;{j&XTHqi_&b}&z0j~tfl4ILPyd{eYV3)`sP96a&WV(qpg;BFD+4hY5XU@`}ptA_NmXewI9I^?P3I?tg0AN(ueG-@jge2)F6?TY?QdQATd_762p*a!4Z1uNuq{=jbSJth*1;32~k>ro)21bU-9?!#ti1^^@Z%>1bKCy*sS0EXqkdv-;J+Mkxv)Y{^T0b*Wdi`ylM2o*(&lS_x}6>{;e%bBx9bzbzdc>2yTvYB0c5wcC#MP^e&W|e*KUq=U~-M4TG~q-_qKY&|6A5<f1*I4xjg=&<dmiN^^{fqa8QmuS#pKci}%?_cgE1AA+-7!Pzu=lP4cpEe2NjYa)ojqSNMOP5X0pxva@ta&z;+%p)MCiU!thWoMviRZ{R{-VoXF+XbyUcgG%T!`RZmN{4B{t&z@W-Xiifqu*|mW*0Xg4M7cnVn3M>tYUqLPryDACzjbmQysIWIulI|;OF3R#u9kM2`P?Z1(1s~l~h`Cd!IRdwsj!xm4%`gH<U77tS-1O&c2UZaL6jz-g0)Gf|QLVVSWfHr+qF6vj&mcvG<-z&r$$-=*qKZZOKScG)w|`E657hG}2TnShdd*abiCz(tEduysUh5ZKUTAeRA0ly87;SukL%s=9)6b0Llc50D+Z;9DM05#W&UB!oc^Jldzj|l3N(|k?OfLOiq>%3wLd?l+j4C(X8BrgX92KfM~8v*@zZQq27-P_e-R%)aRd<Q0ZLf|2m&oe0)uu78dpS!unq<`^5hI4_CO}nM;1L&2#*OSq;9N2s8O9OcZ#?cs$WA8=%q+fd$YC%Pf>8%sI3-BStxlR09keRNb}Do)Ebc;s{dE-V>_>j#ztNto`sT>E=K+ivPW@fBoxM%S!QeL2z~FIbTJfW$>}NSb^%cp3yfpPjEs_Z@gVh@m<)_bR#tyEyqrIG~^Nuu3g0#5{dm<yQP|0)G7n_-fe0uDU(K}VHL$;DS69!;5TP0s~3y4)jF~C!244EU`y{bxg;!sK%%GuS<^$JC+<>M%Yay?0+*1H{N873Gw6k2JPHPSC18NFs=#WWB8m-qwfZ2;oTQZF9OK^T`O*3Q;@9h*`#;^&G<utT4t4<)?{6C-)g{m@G&04vR6J0A2jXK4?!8VyvDEI;SH7?Tb&^zy!z$A@G303hGg2~qR?F<O$6Wl@O%A)iyRw(-+4;{_0{{KjO16G@(^F{m2fR98V_dG(w^06MRDp?-A8?DMJ0rlShu9k6E(2-g*kcPpMi|W-oqkW@gHH~ws%1GmQ|&ODK{Xbat8GP79hF4+P-}fnzw_C$#lFR#*LT+4H#svzH+TxkLgr?d@}O~-0L-DH{XcD%vuh`IJMyd(N1qi!;+%XwpL?E0;Nt{oO$nd`;l_uX)$J+Jug_AzkG6@#81@n{kiF%g&b`xv6sj3^xtShI$~`2b-<d~~T+^@^KZp!~LI|xAVCZQ?sMLJ_PoWg;#Uf=<J$wXS#oj09PWx{+_berc9HQ0qs^ZH?v`A!Wl9v`O42YybgHVjdye&RrIuS9y-K;#OcR@xgfkizI#9dR%?s9>p1<~xLLyFPDr*uqcM7)(LJv6p#9+ys1=i@N)wOi{h=5}jqed%{Do15Q1H{yS~aPp^HZz}?8vJ{@l;s^m7J-8z;{D7lKi%ENotjU68X(UCagfxqNe7Vj_RxR=r&*akkkimnkuAvdzq;7R4w?Ibc52)Ubs{i>m5`T7s)I7ULvp%|PcU)cnj`a}zEZFt=d8qXKVyLy<!s29+8M9b4!lV{@(7n>PR;c^*3ur03cyg<Q1#|*ql~AEp0fVp9=~7Bu)H0P!*B#~|U^%H5k}%}Zl)D4!&(24bTW%*QCHAulp9&#jPPE;w0*G{%_#0LUjY5pmMRiJ`JVSBOBeJA|V+2YwrIA=?k!E#l>k^yQ?<ln$GMYE5)R)uA>%Gq(-RLu2nn?D%_s@a$>D3~h^hQ^1Z`ztCcTta4!-g#AA}#VDCG365c4^=mLPYWt{5et!+H0!}!yeMs2x<<)xTO^bJ=bQjR1M67iDNH^so<*;oZmYyFI`reo?P9aT#G@qKm5aO7C@pLK~fGDIEWx4Y~S#NU2fj1aEY@v10xzdH4UWE`XDIS+_pl8k1uN)BM+>#dMm_=05^2a>9}<Jmr_K~n5?{F3kR4aK~jB?MF8%y>!dN0Yrq(?idBts4rE9ziWEp*LN{MA<^sJL<of8?8MvWJGnRBTI$whB#d5guV%_EV3eSyvW^R0j5knHnqyh5j0XxQo`%KKSkk-&5t3|awL-5H~i(OnKTSM=uHC@h}b25nbgfs(*rfL=Dp$&Kc#?jI}S`c`%pY&|AWT8?<jj9$BmDq~?=vzfW-_|W4zYA?ngJ<EX2EE?7ObIQQmU*NerH7iKif<y9rkHA>fkUtu?XeiDtsDmHuX?3@dOa$?PR=hIKWAyppIn`>y7q;h++N^(yk$|IvVclThT9ZVPI<Eu9O+K0$^XB-x9gGgI?}}cN<lX(0m_*Z5$8ml7{fnd0mJZ(VHm>s91$atv`EcB0~sJS*x?A8j|+jLT`kwLAhOvao2@o4C=xKJ3;Rc`@3{L*9Pz%D)mhbDWOu#XLAK^?HoGgcvZ^aHpEn|&6A{l-hU?127O@Q98<N&hDO&m1VjHFA(NeUW!j|)AVz-~c*`->kGX!?v-V`qMg#Ov3Y44_lt{4nKk%Ab-OklYk4sol&K;*MLde0r0s?;7_c#AELa#ePdwB`<dRzYIyGx;QSu!^!)#L`4^og^XboaWjF(Pu|Hw9A#>e}6oG|GzJ-sE(X`OOM?<Lc|spGN8J$^qtK4(s`1do_k9|pbBE)&H`hO#(mJ9CK&ftgbvap$C5f!!)9(mSz>HPf5KjC-coC>dHTTu%zKwx?3>5eNv+D)fZ6^NGQuh0AP3?Yo_S3OUxev(EooG>(ori8W7xB?8$wb7n%NR~DP>x9Nn*DxiCgZdEiypWJfQBgZa1*_)=542t8FLZ6s=0kDM3P3`#)vg+|EIkvn&@4d}63$lu>EfCil&e?)~yksuk3yudI61*mH(C+}$D!W9?RF=3VyVw$S+YLR%glci1dJ-@7=<VBbIPu)VP{R9c3jxHv~z#&Q?GD2I56An?qKKJ+38bhs#MM(!1ijh9?ey-#KQd%8dks&@#Kq+3iCtuclXXCrFr^Kx_dbiOHSZNDFEiH{khWhLO?-VWJ6y3@fpoF#Q}frDh(m+hljMjMOV8%>!B*dpYNV0H3Hje4coeU}H+z?Oy~6LxR3$1(8J<0kpNqtDMJf%)XvJbZVJhg;f&lmddI5wpwbS|m8o9lF<emf5)w(EzC+NU2nTHZpq?u*BpFIJH2OL^W9$jML~gYPYXpPA!fxb}T-rTK$LPV$LTQr_(=I*j(Anap`fHY5U*=zMo$10IkQAe*dSJ>*?QIhciz^6CQ8%u8ql`92?}TJl!YXTE{!@UyZ~a|Iv3ZF2C3Eb6(BXeX<-xJ=!=c=Q0dA`^ADrhqKWQzRMNE@jRoqw8E~bpreIRjK9|&jZOyYArC7b+(j@!D_FpaU!gL5<et;azT>jjv$xc<pPj6`IWE_ck6&GldOm#}?$VOzRHp6^R{{;kFTCet&VRey{q~83{Qa#BD~Q2WyMxb4Fh$GJbO1P@N_ys#1PL!dd0aX*3eEChZLwBPGG-mgINP9ECg*Ggv=zl8w_z!wXpAX#NYxd*$37DJWUuP~JkB<6*?DIO5ZQN(aTz}|mbTp<6V7vQt_&Epl_I5kue4AfF}dP|a0{`AHjD8Dq8dgeP0cMhW{GWBtxaPeC%Jj1`pG&le-chNdAmfRL<Aucdmq&J+nwAnt!Ow;@Nr2(n9v93Btg_%Ez-@*K1Ua%fQ7w07~9a9Ath!Sb10$DDDU9A0^hIyWZl|$_$QZtU5SDH^U1cyI~&y#6$6Tc(Kj_Bx}C&b_Blb%ve{IPjY&&IGY%h^vEnmJ?k@QuR-9UN%+ZyVkQG|W9jHLIfLZ;}3ddf7-hQIG4l`{%HBG^Ik3vW;=S{%Mi#(ELq-Q-fh43Prn%2gEAuzZC6i^IR8)5Z*x4^wjDonXXgwEV+8z6-xrC3!e<vl0gILu!h4VJZxH&2-UZ?*+OFm9UILWK)hsN_2-$^)mf41}}QL^CPp$Sx<>hbzofOVLKeVX-^l4BVhF7)2q2YO<g{K+A-E_2LOJq^%Y%2MWRG*+pUoNgP?ehQ4Mx%ahY)p=}O=nhjTO=#ppd!c<ZkMot8CgCXL>)w%n~i)5<`ODi$A^e=x7^nw_oS}pkCi}&wsZCh$^TLE%`Ux>7b!C3+ZeLfYg7~zk=54CaW%|#{>6dE$1Q=AB7NqAC)Bq8YVLC)N`*=s}+jAg4u%&I-^bMRko-)m@n|0%)>@4N8Ay|-4fb9yxS@#ZOc%E{8YLL&Bk*pyu4n1Vj1M_Zs5pu%iCrX>TJaZ+w$i0dCV3%4bgQDwwvj2sC{i!!9lQ5#OO@LQp$-1Z>*By6{?Y2=Km?8E6XFezE`bHN(n=QRoBKZe+<>dHN>TH>E06;eZM8(2v4WNqDd*hRb?sNjAg2>^2HF-SEC@3HlE1^Ktuh`G5(lf`?Gt|w$sGgw)6po4MH=L0Lf2pdJ+m&^-BVhf3`pdH8LqQc1l*QG<LKAk*!@4<E8M-U_J08KH?S#~;fJy3pdaa_xI=h#bp|AQ-Kgbx>%{;N%A=464?<ET3^gAw(p+jS2a&$4~fbC|7(R$_30d+u`tNF~Oet;kHn<05fTD87!FvVvuM1`(bz_nD{1q4B2+9@im~w@-9D-n|+J*(TtXLZ+O>%1c3IbPue%92udWrMOXV#W&DrEj>-GhRT!~r6JXVJgbyUBU({YMysjk7Dw(Nm1b;`_pQq(JK5{N(NB+~7?(c4t5?SAMYy$_6Df=PjuDDQ3NUWZo0lsM*XJ20E$WRijofsm%48Q{mmrjKsvHf0mf7d@eM&&j-y_drsov0ALo0d5=Pe3I@2#5Jzt}tu$}tCE!Q?WW89@Rc@>Vd)dG1(_Pz{e|l+qw}PBKFYmcgsfx>3<V%_Tc3bwlbHJhIj{)p%hFjdw6U>4EW+1(I*BgK6&^1LWsdvW0J7Zdz_l;s=RrWPl*Krh<bNVROs`I?vG{gP~6=htAVzjnNz<<>=>^sbL!tI=VoorSJf)#vyg&mS-$8Q=MU-PP+!v|FOLM=P%9igNxUV@4noB_}ZrSBM>ucwvZWAF@o}T*~~0wX(l+-8{3l`AU1@U1#6<=epVxpb{QSG#W@;ul2I&R`U+rn|Eq%gLi)ACpPwC1)E_=pe|`TSuiV$y-Hvr`XdPa=f1I7)A_LC}l0>7W8bl%4A9xq)lld%lHAiXHGCFcDkVj#xlPo?OA<AP)Hl7VRMoZ_YQAo7l3>*f`9&;`Gw8phf;GZ1VH9U!5y!zdjzW2&&ue|!*A8pFn$;~f(Y6}93&kxaE<hX#(vR^)G?j>NfBv5jl$w;*{frc3h2}CNr2@WpUh<h~(jwI7C*dl4%hc%ukCLMhbmc^nsE-sGCz`t2`BK*#+6TRYIc4=v#2VsH56Ju)yX_56Du{qQ5HC}NCN{t4%+ESavnfU0r2<8&HWUC}=@%~PWJ(fVN_c#^$-e$bbA>*Sf!@pHE?iWjHf3*DhSd;6Q%jf>t?%LYz$)Mh2W4Ex%h!#cJr9I#l>-4AXtN$^Kd`zAN<1oI1Vl<~@aG5SmR&WkzzPN@68KvjW6nhZq0w5iY<BQJS@#U%B1aB=`3zw#A>xSB|PbNDaZk`&BzB`dA`z8gt&A=faP%%Fvb`9(=K$mh3?iNou06O4GwjWzsPot!68K1lHBe2kFWl+U|RtXq+B(?g~_2cS}&8NoNdsjSZbHf}%4n}=^T9JI=1=ArMIH%_m;RJmF=8Jlu5d5SpS4$P{r(QDkqJ@ccNHOqL!(I%7skseNr%ufnQ5zB@?u$J)ozx#~wx2vu0?*!-tP~}&Yxdq{lPsPk3GcxLU)!uOnh&mvz@$F97^Em#uq!dt*dbIYl<gXfDb?P5CUbNH>JGRQ`@hqD@QwRo#<js^sDuP#;*|4UFlQyySz@k$2H=*bK`C($!C>b7!y=J2#gT(OZOS8bKM-$ODkpBiKCGkeCzDTVd^|?pua`-npB?Y~^_9Z#r<(;c50WM2NRV9+%}tm1Zce=E9CvEFT`aXa?+&5pG(OT2dUug5{EuMdCAv5qG`li$u@A+kGQ>C?#W3sKQF&Vay?3Q9^TAOK?w4P>&G@Vi6*-IRMXC=3MJv?<-fe#j&k}->8H{bjISTY>)ov}8iKjG;7}$Obj5?$xt0t@jglaE9cdt~+MyU5A$(zOi9`aTIzI9X$UPXkr#6jc?U@tyGwn{u$F551xE)>rakg4Y}C(Z~lHoNlJH8M->n$;3`vcHUk<QQwKnP%bPncYf87hR8gb$HDj`d14aA1r4xpB=>o-ni1rdS@BMxDs{UWTcX!rD@16q6RUcTlpN|F6!22SwBLBWGonX=Go9qO=k<ak-?R)<{8ECXxHW}+9Qf&Xn>{kVw}0aK3D$Ky|IV4Y&kwNO%`#-G9!&bL0K$U9kxpUke+8)^y=4VcPs*CVeXt@U;-}AZ03Ne+Vj+kv)M&hq$8Cat4lWavg-_7H-rA@i}yd=WEzEJtkI`&Gh*ZzQ@9&fhO>m4HVGEITnIRnQR_@NTB%WULj#d_v8@^mC3P~^%uxoH#x)q=MD{NAlcIjhD9<8hc@&s*uZh)M)Ia&z(SGOWCx(xYE@#P?eEsuPz3;&~8+o;r^U3NuwOOnZAZE18Ey0DI5`7KMJ(SKfj#dm5TW%IEIpt~rwU1uiI>ky&tVVP~wJHW8RSw!Lv1<z@aL;N%cL(?NVEWO;k$Atoh_HHQzqi(x|8eX1E~wRzY#GdIO7CJLZ+G%Z;T*?zJ7J&3wHsy~IUCJ0R@3HWF~Zlxf~{6fz0J{T7>#uLX+-5(4U?_xt6^W}_x%2Yn@fyaP?^boq55n&Bn%4aPJX3lX=~Q0R2_YUNfL(+HD-gT(3_P4d0M?mP<yg#aimGPl&CJWri$Gj3i}lKQ=M{e3Q}Jzq4D$MfZC&@x#=%9I*cc=^qr!lVBFh^Ql!In9aO~g#Oc5oCSVOBO$vbua2?4KcZ->@$%xrN+Lc&mn30MW>N(m23HJGkC$akbqxs(D8T!4YP<{KvO~60;{psibuzr5yumuO>PucC@V{|9zlw7AM)AOklpv9&aVAdio%iX@xJ+0j4($(?_bZVLHb?8=x4?(4-hHeR}_L^wPxrovXdmHiuub(Ug=^u~lVh^vR&L6Dmfm<Re_;FGs*K>(34gq|ZhL1&_bBvuajcyn$QQYKWV$U_gsOXY`wFi(<{RkdH%)MiS5qh_$5{6K13OfXE2()~5#nC@Iy6%2{wfwzt(bnDsG5(oSQsY6Q8SjKdPG@<h9*4KIVrj{$&6!f4mc?_Mlrl+b6RmbZ<3jFfDuz$W1Be)t%v8!=ZGJWLxg_Hck4v^kJLpZI6k%iLMuCWzQFDux2i?lc@jMC9;!X&CfRUjPV0KOx2!(o>ilbJkKD6elEtW1O)shMV1YqF8ec%3mk`Udr?zz>s=d#lL<^t@W9KX;tU;gYV214Jy_-gkdcLG|yRIa{m?b1Kq&G`7#G3ArxZ1(Nbb@fxL^o^&klu*5s14-4wozZOBuILf;JTIfv)0{C^72j~H5Nbooa~fl7sn^sj@^^O`qrg^<;N)2ujTj8On#O(^-B9KF;{D%lS@F$i#DXap!(bQQ%6NddT^QjkktbG3V3{S3P%;)B&FU%0U_}JFM7{S%snk?P3T_rWE#P<IZsjlKwj=KuPqZGK%!B^q^xWVVO9Xy&xuAZuC*0H*P>Md1mnmcolC63?@K*GKoM+&a4nC0Va+y8}I-`=67k$=bAV#Ne9fMuSFu1bTDIG^ll4|S7VTbQcivfq>?=S5#7e~Q0m0w)0wfU!iwWR%Zgzx<mSbOEXrEIoD?>@qTtyoOK-846!&sq#WOVo}!CRCI*dMimgg4+rA91N@+jE-8Y?QFcRHCzm>i~#1*aT13w6PWu~uD1Q*b|>o)G^GeJM9INO%fa3Hp7*R3<OOhM>Fg&5*_S#j(I<sCN+*&?%4sMOfXc9vvjJHBS+E-btsbaVtx(Oot?SK|k88G+KmRvJy_vUe(Q|*+u*jCcB}BE~<YS|JC!<tjQs)>6InijD`*4HEkS%v>ws@FgYqNIM+8WGKjSfk*U&$?5$bgHMyhp8j{(j=b_06Lml9i%w9gf`m(gFCNjw$}rC;WtulU3U$rJ}|FF{-^m!4G789f_{9_8))cgI{~q-cpZhum9*S6D)9+b5(-11D7_9Poi&zL>eJ`L~YYqSH)1QS|3EzIU4m(Tt+0<oTYzhAEL82bpOeT7UJKmPSu|r!|*yaxCqQHx8L8^>^fL68ZknSz!@Y;61vMQtDdFic9+;gnv(|9lyhv5d~_u#rd*|IZxYHxt3$_ZGqhyiyV|E2OA9#nQuMZQfk!7AbeG4Y8;esk;6Y}u*0SqSyJLDdSZ(^8o*M81l%{@>^$-L0q%B1mXz1&0qE(}qBDd6Mq=r_LvH{mZqz((BYOjx!68B(vdLZ@LQTOXhaQKYDFec8C41R(U0lZqf%Wz-LXW99e<W8w9iAow)U1r1DwCZ#qO5qw~B{9C9EpLy+wYV1+48akrg-+dL;g_0VSyj4Kx(k&VefFG5Jwq99wGe1*ou5^?D*?X{;cs~3tJb1Y!|?rnDuy>p05Y3ORP>F1*K}7J9IJ19LTNRGA+_(}?-Y6;ED~Io9vE+(u%k;0w%;x5PVaAO_(K%;k){L|JcI%75)~tzXN4##9Al&zd)6G3Yf$6hg1!1OGEDnS40RfF8FL!&3IP_JY@u@B?0k~FT*7bdkiT(pQ>Er85%S>)Kfk<p{p$bv#@}z*l(zy>f^lw_ON&H^X>)f71D_{!OH338Te*^C65EwoalqgataNO>PXS6P8+y@7$^qRTl0t&YYMgJ6qo;B7!KUT+0Kw80p+!nE@MTShbO*zraF$JU>D+3kZq*}TY|^BrkR>`>mKIWyFx@P*bmJ%)nq~KBA(FA#qiN6A*Kl@!GujY@m@`|k7t~5dO?;PptZ<Ho=G~)J(J*VK9AxC2(5=wW5G5rLv`IDAP^=m;R2E{jDA+4Y%h-4Qk0*vTKE71N^c5rbN;{lPimq<wA8m6CI3QZ}L>i1)if_dc9Oy2Z?cppny0X#bnTHw^>3&$qk}Y@IbIrB(fPkr#oDy54OO|a%XJaXt2}^A=#JxUzk}R*s+3()IKJ?52AUTUtKr#4%WnGTA-CG6Ev%!)(VX{P=+~OG%brPc|uETC@d0I*q17b6d62dgx<=Em7YTCohT=yBu8^Lkixx6%<da%r<eR3HuH)^+fqb~p_MHRtd1^CzCI(43Vm>OCi7%cm0k8!kRFkVWIgxRpjqb1gw{Y24t2iBOhiEpzFtu5p|SD(Of8LRkDs|v`nOti7BM==L~AJ83^EHvhLCpoT^&T_;|k*ncpw5h$YRl@<Q7}m<v;A0dGhSiXq$e3@OA+(5P*=DoalC<q^6}KH@e|U=~*K-_WTPcXaRSHpk0wP%Dez1~|!!sXa;~#?qMOW%EcQE8}=`=X!GK||8Ev8|Vnix6{orO{@Jx;4T&}B6(v2&em>==D2MjtM}-`|K)a98I*!8j%+aEak;7&YM0^Nm6V%zq3>wa{r48+4Gdnq~pi*tO3p8}$a#k<}e{g@=*tY{dxSpxA0O>W#VYV`@(|?^{Ja{${nM`ear7TFcGPE>FjueBb*T79!Y<Y;uO6pum|jZ<j%&^IXU*GP8kmP$|P$&@`T%it)Zw1W<<L>gzk2ts1cSJKVajAu$75MXJ0b^_HFC)l>45jZ*DCzk@zNK>@_#>h`^yc%ExzC2O5nw2ancs(~u`jCpX&(<T$iLTZ>T1uIQ`oy<6RgK2Obm6Pmb{00o~UxvcVJ+V(V+LMz+$OfYzVpN<sZnr1*<IS@KVjl|!FGj1;c+RlGF;Kx+au3FeB2>GEI2$$x1=oz5esGj@B<5N6nVzqj>3Mi@l=S%cxYPLlZ|(CJSDG^mWPiEYwa>tcz*dur7=W!H4aV0GkTX8#a!z;=gpkUNR(1PcL}~7|%gsvAQVPV-jHUBL)&`Q@`YePJ67*UEOOFb5mi^@Dbf@vnWvB5Uj;aTb6=W}N$Riy!X)aHje!8qV-c&yLkIU_!UfC9$e8YFQp4VGIPHyW5EQ(f_i5}n%eR4j}R#DA@V^|eFg3L60vk|DZ5M^4000RSrKD;a~L5+Wo2Ix+J+z3kBrIoHRs5e$L-V{~(+s%G1Rv&0ifug+~5nSoloeJ*KKR-(h_B>n1Jgf?Bk_}!x*hgTIvy{}8qe?W?S;)o3%`b(dHT9S&VBRa!8+E{gZA$Sh(JN59*i|wT_$fSY<wT-q*`TbG1D4)c*}|YisV#K^vKy(4nkG}KOfFK|q=7q3PMD)hO@5}dN531z=Wo|e`HkY!)AgIjbp6&<75eGM%nYhQe0t6QS2vrulhWLIjuf>}a?xtnUkJ6S)tz&e2*oGs5hWDM#nL!p%xp}4#yOf=MlL>yANFGNTAT7pLu2*Y{mrH$qokP=CQgD;ef1$7;4Y&>aF*@yR`Mh;W>adJ&FvMY8ovuQF<I~<gV$ti!(N3<BO4&ITKOnhVhv%Bh^HPJ&O@%OC*QpIpI`bfpZ(^)eEQ!1`{w_)sT9Oy0go)d3K52w@J>c9NzPJWv(4DLOv753&~mmiuNmuV8vN*7!3spfWz;;NmWXA#J~moy9E!%WFMQp${QA~1GkGPc>9pkjy11}V<V7^_{hxhhZhoHQp%jd@05Kq?=)!n89U>h-p6_%BID{AAbf`0qI>ES!J=aXL*A#m(A_K<S5Ode;o9aPhQ=HVN=-@oiJ_up%B<ATemme-Q<l`WE-D_wQr6v{%oZS^F8oSN_;5^4$)IS*C9WVPk@#C+0^ixGNZh7oWFVA6bP#bqR$~Ve?j?uacH$yQ@7y(;|F_y|;K!@pWTvijXbnOkuV2|8%=0UAsB<EXouNH2|ygfQDMP6yTeRy@o`Pp^*>iZ`*f3b1?GY3U+n_!>j$&`ZVf$l)1@H`a>2i1U;tZLU;X3thXk+w2*8fnTt{jbrez6cGNG4Tu{rH^8TmK(+GHN`D8#Xl`=&&P-DTj!gSx&y1dA67EB<eSJL9E{#RSGkZVz5vz2sH{V3!&GDdTBS?PPn$v|y8vV8v$3lkxQYg0;~BlYjfpUk6&HKpyfI#PZ~eW|AX8Z_{@LrVj8W*2@31j`o)&^*x*we-;4nkUf^`(_6S*pO$m9y}Bt4`?a~kJL$-rkNZ1hE;+V+j{r-JTxhPcBL2CushZ9<_@Sxu4L$6iqb<h*$>E}!RI+?+~PQU#2qU@rqAwaBBnJ8i3y!=McI>7-CbXpUAbP1!ySv+Zg5mbCo0OC9;(mYVQ65^_=_GNQ?tt0XW$Mn3St?IoU%z$bkH2>b{^TAxNwQpF-%p&L(c!FavFvNG3Hdm7?3qhfBoAhKc6suZL9yjO|0!SU^*zRL$ot@zWUp@qlHb!cOq=J<5`c@rvdFuIMx$#qNJmCYD9J>w2X<oqJwRC;DMq8)~0DNv`654+lnbMUCaBB>N<URA15Z~<*S_c9H%Vd!2uULo^8EeA@MBSY@S^1-Rc<+rbFUM}@nSG>M^!n^Nn<K389Eb6ehuO2I?#5-AKou!WOfNTIH>JWOY(pY=Wsf1!T2|K#?XQ*i)S=pT-<va_H83Bhd=DNqzFWo2l(8|xPJ8v1)#ds)6v=7->kE>S(cs^<MoL+>e37RQ!YEWiuV09#C`-@r`*xCI8U1^n^i2G@g)t*?3mK<`i`X}z|Lf1j~-tWG6@575P-u>*0cmMFkyYGDQ?xQc>d&6#j`o+8Vzj$wx#Z;gql4B$hvG)Vy)^qk-9=he2T>_qEC~eeB8%)EZXs2O;b7m_(HRLfG8)MJ2300&pI<+}!#!M7K(>VZk@IGx;>A}~}-k04CP?QwdY15L>w+gw%vs9@bB*xcEffN*nYgB`@LTQ8ool7*LZXAA;P+jn;1d>VB0BZHQ3{89M=bGlxAC_&)e>kdoSZVp*ipjOwe0KvTGl2yH#SnAClziXqaEEcFnBgo@sP>dF5life^WabkVA)+4#nec{Xu(o@?W2y`wWuE+NlH!-cN9Jjioabw`o4Joqd!^qh%QdNFMn~d3UMt%oj*J70bla?E#SmuoRRF8B{SkocVa;4EHwxRO$ZD@wDwwtVlkSRcuC%UCd{!%Nu^b>d*^7g_W)%i?p-5>v@@VvPLytS^n2&Hr2hGh1)!C>e|r3ihqsK5JkQx^@Gen=;F=7P)py^FJaK&9ZsiNG`DQ>iTGT6GkFL^+imfp;jwx02o}|Gv2R}J2T+;-`SnD)072K9;9}0a8-FwTD&<Dr$;YTOR!@t?IJ{%Pp5Hw|>qz0=4-$gj-98Ca5gc&WCW>P6K(Q`d;vRK-1mJ~Wzbd4j-R3U)XHz^iI$vs<nI{KdEPqh=4#f&wz|Kzb__0b&Vos)pNN#DbYPPbFy5CkMxenYsELyw*%_lbR09e|+)AATFnKsd6boT!5@3yqwOr?YA$Z8RFXfmF!BSP|iVwE5*5?AMow?4$P<Jifi;@yo|psrupL$87B!MNy-%0BY$isIWoSk~xK2#OI!({uSd3U@tjR%oAmfF$;^)raA{p@nNeRniZ;n2(1cjrAN$zdZjT9(KuZqTX%cO$AR>dg$8}J4$}X(w&6Wk1Vt2MxfWKlYA~jI7n)@6_&G+?lxDIEugR!2#t{*!@xT@Zr$H0!nM(zG(solJTg_<&X5{PwH5Pq$_3YE?(#g-g)f{>~R(g18VX>YpEoOYHgT)rhDWxco6;v~O_b%Z9?l3E-@hs1jQmH=Q(X8xjMWjuD2Gl%7<Y_sfNHBDE#zH(JlujCIZbkwjmDqMXK4na9%b|LVfq_8DuleNXmjHKh+m*8f+laF?iDTM-qjzjQPIr047AjTaIK5|Tt<?#kH-ywI`upT&pQG=sohK1@@7s%E#eFXl7j>B}*s}m<Q7R<YM94(svKw)$7($|OmQb8sNN1SQ81_tMTHw>JO=hV8M5S;^21Q9c+6XRN?42I_s@m*j+M#$o8NYKPwRM$+H-RyG{bOaI&$k83Mu%HmoS*u@4Q+9ZdY0$wYz(F(BiP2fsm>OJLQEXHS0#YawVX+Fhh7tR)gq7*8D58u8D$6TEy9|YjO9YZw+#mW5`7>IZldc3B{|#smV@AdwhM7eJWtrV|A!Ti(y$c8EXM7U80&!^a}I&5oE*6ux-raJ+4&H}At1(tc0JgqDi?pY@xU@Nr-Z(;Zoetm>wRPG;>Ta{(5v|RRS$gaReSUN1n2c1-DN^4p5=)sr&4Mfv&J&0V=TVCQF^K{N)4_NZzLalVxDHsu^O&5PS#$hbj>@ap9sZo9z*d}SYFqH-`xmw=3r&4SS_o|j-?8@|IjUK@3%a9Q(V@wR6-nF%Pf#<0<M9F)lddp!);bfl$w1Ktq@j@EfwMEJM6s=%T?G5rm|ZfJhf5(=m^?}S8|Ak<G){L=Pw>FD=j_xf46+eiu3oD(&F7!kz}2~e|wdpzrQTJKib~IKS+o!>L~!pNtuEk@<E<&x#^F=_G3t)){QBomfL6&eFL-*vBh<(p-1<z6|8QhIcHJAI74hbX`We?_fv=`yZrxfJb-<7>G&`8^s&?b*|E#|=GuE*O}swa<Y#DwBTzzM^hoV<Rh#29;92f652*x3<-E0DMzqKrrDJcCD^9DAwM1^lAbKhTI5iH9a7<`SG0EWleD+EEq%C@(;zzOxH7Q825>HX@G{dK7Srtf=4rFz}gsFBbyEJMG8FH?528)?iF$8UmtSYeUE?^Ywc{Gl(>psJGTWPR{$j5#D`)m6DcuD_1yBbJ&>ucK%1w&`j;Eu1#!Vz69ln?fXh38B7n)L-}^-CMkr}B%{r^6`MjI;Ts=&;1@LQTE!fH;L>?_}v;a_2%|746^{_HE{;<Mw}eZTzsm^4Bl@^%y^Rsa$-&e79`GrB4JKDlMQGyJBBjPKR)a9Mdz7p2&+ZK$_8?q+}KLK`AyGsRip8#E^T4V<_dAhNP|`_Q2F88YZ!j1=t;pUp_E(fBP<|)k{o*YJdANO}4~#fCCbx=ahPg_#(iOv*qS6o!X3^W;6;<AgSr_g%J#?qjcpFN}n~gP=X8%wZR-^>>d7_0Q}(VU!}`Sz(_d(heeKw!a?t_8=udz)tW1UVDSY(m@LL(EGze(ODm~S49QJ4Ii)rz`#@qeO5UOb8pWFeI{<G0@ZK#E@R{jzwxTB}Az6KxNepavuz1PwEZd~nt%+jG0*PZbHXKWv5}AACoQJz?6-}iB4{TLLT^Ar40>wSrcd%U(JpA<&BD6+he!ZS*9zSs_UwP(clpxb87o6=qqhws=4%72=md7{hLvmdZtk#G`ksw;x5g^ShL5*k|6EVhJz~ILUBP$w7p_(W7cIOVqQ-%3=7NY#_+Lm9{V;`<ASsyRR{CM-xA||mSJthDIc7;xYJGpGod1A45Utt576*AVwU=<(DPGd|SGX(oIH8)?%rI%b`mOzc|%P{dC8Xsfv@&3@-vH5I?g2##DrGvRiq1fV~x*HKNXQL<(=n(E=x^b2ZZYpW0C108nN-S1}6k){FqlSVA221EhO>*lo!^{wJh}Gg|wM%p69fmi`!H-t6%bOmU@2^Wkzdnj=zjaimd$0|iXek=RLPREvY;O+9UA8YNo@c%;(`gFDYzaW^7qn(j9TK9r;gD7>V)gSxtcKs3u?=?L12#DC<M1aj^531T1Ra&-7PE~l*oy&t1)Y<z7IFlpJ1ttCCGr|;?VXJoBbF$+cMFsy`7WiGB{h<k025}6NnB43n%gtgpsonAccq`O7<lXAC^YkEX-nU~^e9?TxQZlPF_{0bPJC{j?IdF3GbyV9j;&|{_kh_diUjA$Z3;&iBT7t@Mh`V4?NW7+D<~L3E5+GHnKKq6cHkxiJ<rHRhAVLIh<!pj_VX*pqT}+~A2wkz$LRi9s;(Rq63cch;&}>m6?a;RHN_T~Yj#idt~0jW8u=3+efoT=YKXCNpk8yV+6R-)Hev6%+%B*E;5fbWl_dK)S}@*|R$$Z=GD>6$`2YvvpwHL|v_Sg;^j^vg+Nt#l<RP+gcKP>^({O2-U<G7vFlOq$*b#~a$e=lAD{?ZWx^EfXAUqZpf4h!OynV8Q{$SNF{)+zKvyyKM7MxQuXl4Kr<KSWQa2chuG$j}ns>W&{jJXV-3LU-VQXv@6$yi5V>J-bwNkr~HSTMk_6ps|-%T9#;l}F9bmUww0&E+@mZyufqESlX0m{YV0lfr=y1n{hi@4i(00>sSPF=mOBa<lTeK|zQycVEeD1=&}Ftg;NTGz*vkU7aRj8p(H7yX^W~kHh6hs{!>tovc;(GTXISxyeO{jhR~dUPzA$dUxY42bJk8-GN7)B{d^C0z;(>E#scq+tryd&OCq_Ag2mh)OEN@lUb)0?`4;r-#l4LUM=+)4By@|FJbu<$TAQa?n1=yHLz^=B05JDHkyz$Q)CG^G&GBRJ@nLT^Re*iE5DM)>{Fo)`*=tVlS{S2dhl+Z_LSYpm0-VdadOxFi{m!Vo12P9u>729F|0;j>`?Dyasg*avRew+)R!+}(UDw_C>rjjOmFI>C?kZ?jCNO&Canxh&#`d`jodnDFBiX5;`OtWG4QRmqR;pk783>6cynC<)I+d)^Rx16cU^x0ip3tv0=(Xgvq6z6f`L?PfXqQgEj*=+T);?6N?4hwX0haLDD8YUkUXP3#(!E6_{&vJX6Y+#wN^VU`lUI$NV&0000?-<?m-F9HE@gRMVPnkV3lDf%gD$xi!av(o)l=b+Cqy>J9khR78r9xLuw<d8UaHOR!i??<IUjs{@-0~&@4pyvVL4Ae?B-N*-m!eznoO_<Mh=};K9YQ^|KDL-M=hCFB#GOi*LHN;p1lI%97qbx#xp#tt`rAWUsX5-oDf{zW1ANU;P7rvs^BIpWXHIi|cl9>o%|p(LPvOPWRRl<gH`V{_Uf%;-llYd;4nA&M(M|<%j;e;}7`mwvl7I9Ff&KRO;(N01>zEP)a;YBG<cHep|s3>}m{dE*!O1!j`QxvCKGJj|9A(Z}kl)N>tGv7XYI~*(H^4PMjE|Ke#q=vZbYO3_7QneKRP9knUD6XFSVc<N+fmM5?tkBdU_w5`p`RQX2+HUn;k1BpaEFNnLJ%OUwa_8@LCO9e5|)&Exi^^v#8L>K}{Ge!N6$i@m>J!sl<+vAnAzFN^JOUfCT!I++J}d#mhs6cj_AMH5XThT(jtF}-k>T8$${nJL<{vZ9)jy6YBl4MZin99{`+VoM#wN1EMxtGXU|Y*bjs+{>7!MNU6+-IBE3AIo<q$=FRK3JDHd1saVko#k}N$gN}|^^J6>Vmy|i*2xIfAcm5fdn2^sLZQv_cYtZkB4*3(<Sw|oLd{b|8kgbFrJ_6?7k#*q-*NVBb5}{G01^ZheV<YppS3e3B!2|LGQ;4Rnx97prDTQXNF=k6P;aTg^rOK6L1B(uYL#SMF!+a71Q{*f&&O`T?Gsv!>%{ZhOAYX=MOw*qJ?>o`YXN)o?>E~dSisTjYr^E7=c<r>S6u@<^9dTf2<_I<0*_XUcFX9UQ<yScrMaTfrWC99US%+hu6eRmsJXgwQcoCqrCsymwr%y_F25a<|9sMX{q%BfWu1z?+!uLkD^|%j$|7*IK%8^1jIM455Av+1y@3289IefeJLZywN&{^zYlA!lO5!psoDM2vMX6o7U^Rk-*`SXEIxOy@mYq+XD$@M@ahC7rD_i=>S`=)W2SCkAF>#I*NmImhCpLeU&K)x=K+U2pE`~9Q2ODFLR`s;mz27<$(K4VCGW$GLOJkCy_-@I6C-aZN_sex1WE0Ya$S@<mBL`yafZ|=2*zruE)&gLa)}j_rHdK}58p)*^oVbx6A`Ovf3AkAF>qRVM!vJxaF5*2W-%?0EI@w10Y_p!`VtTC-I9lpKusYFMPau7+rk3Cfk$H#De?nn*p-)JKLyVy&=Pk9yz*Znq6{9yki)Rd-iUq@5m9SY(-cN(B1M$<Nth}-BUu_wLQn5TM8Y!pjQ+NyqdAgGKEf3w|lmKv!r|vTZ#6c}snHZHi=fsB8X@zlW7A+xF)naLT<~WS%Os*l$s&(ol>~T-jPaax+^mVm+4gk(hNI3zyAGRwj&>bAw&r)Jx1{@=`X$4{_7A7&J)C+PLJ{P4@#nCr3Mgv+$9(ANR6Kiy7LfQ$+HR%0zZ8JXh<2a7gzjccX=Ck?-D_SMLv^G>I)7`o@a+dg0!(MTpwp6M%%f>0s+EIH5eO7@9HI_^<d-q$W7OI>@8q^7M_q;rvo%i1lmeZTRStdJHiu^a5f*&Sh$Vn3h0V9u;?ldlcmQxlpRCPxijWvSK!CWmEGD>JTRjgW$F=}^R9cn3$31wDrMYz(9GsBL((`NVQr!zqJj^j%2ZkkQbsBCWykby6|9@SmI7teD4cZLQf)@(5?4+37!vAr*i#m%X?G;MLzL~6mWSb6B@GomBA?vW+roiX3SnBQF}%GGG{*16>5x(JqitpFrcN|gD42MP3ibD#j>{5kW4Jbc=itq8>2iF0c;B|}_ZibFC)a<3ytnwsYf3Bs^c?I^7?r9B{Cq44LIB5xiLf4Pl-5JZZu@Wz}q`{tV7B@7~;We$dd$!fsTODI#g)96Gx#_SU8Q>n1(n2r6ECb|Nq)r-s>0tMs5bq|BLhQVcNXmg*AIU*YM2;wo&D!^pn&uH-Toft}d0iquo0&9kWvG~ZlG~>7xpkwK&h!(H^)J;`GsZerfK_kM8>6FF7oeJM92Ok|>q5iOrlCQJyZ(rG@Ef4zrQ5IpV1YyeHE=cwUB{%d-KG75I5GN_0qwD!Zj#X0V2FrxAL@a|FEqmt!v*ERQe(uGJ<!LOrS7R|XN0&j!J?=Z^kHL8=u=R@tk{@iRDk9le%hhG-eP76xi|&%7lb$8`O5kRsc(h{47-a&2hSGBB#*mxTIV?gNI|)`6QX4=iOOM8rOVQnW`bnX|tFZU<NycYBoe3aJsySQaL$wl-4tT3s<SmbWs)_0?kKJaD3YyLm-2<9nu9nQ}B<WIn^XZ$Oda!6)r(s&L+o$ZMurQ2?dUmzf#1mTI(fzcR_5FndU2eH;86ekS)Y~W!5T}?z0N5@GM`!8MYd};dcw&XN5$%~RjWn)ytCU2;vVY}1>vTMpCcRguMm1>Q0xj%v*Vow3DtG<U<09t0RZw%aH2rKTQ8x+6KqiC`lN-)uAcH5~MO0YMQYdrc44J#-a2N?N^`Y*8PTqY=qSdHRwsc>mln_JhG%*((YyqAywS6P_i8Y4Ht*`aZ6Hp$x(TA72xjrBM&N6|!j$8Pb{$Z0>1+f5Uih(10T?*T`W+~EHj!_ubnaVj~u!Ny86`gYq^3EaGM$qh9E#!(PcIgt1Tn)U;B|vFm-<rMN%YL|QDEw~0rk|)kIMSvY{wi-={MR>resP`i|8i4g70tm{yn%s}y-%F)<Y;l0?zmP&Rxx(&Rz@LljbtgFd@h^^+6PIv71Tm8_uLJ&eH#0kPWYvkx<}R<k@Xjk#S|{9&d=AupU;-_rk^d3_`|Uvc(^pgKVEy3S35dubbj}S&cX%=M?<hTihNNVRf2_^8!`9CU-8hZ8Mm)`;3h(b;Y{OrP7N$P_ulNXn5sNuGKQwXLVF86Vdv???W~J;7M7I4bgQBqs_YKZk1vl8kJrXKS8nEyj^*#?Ut>8O1E{MP`LU}-?kq9i#pZy|G-elH(sh?<?Gx+BY1YPQRLt)2>bQ5H;QoK1<`}v_MYYHET!JyPy+?Jc=<MT?;iDUVZL4I>CrAI6_2#`3iJOnV^mO65dR$qP07fev*=UYhPCmqgz1{g)_a-C12>mT8qE>N;3ovGyOoNL>)kHdpEww1Kl^Vf4xP?$-ZZfdaXf#Tfw3i-F)4+T8WJ&Vfjmv16jArKq>;Ft>DBMcbM9wnBM=%s7i<&{C)-*$j0NuE%q!GBg2%}7I^9LR|U`o>$ng^3XIPX}zj>Z4)C;$2X`p>_)IQCW_tY@(Q#&Y}BeUDD$#8$1DP5Iu!r67n=HAt>#%d}b51LJuPi5d~B;Db>)gxrdT*vLvz4Gn8&&{5gvN4v(Z#V|eD?kvTq+vJ(^j?`Nm;I^H<iZ2B*U<wG{75Ey|lu6FA_>6`xPQDqQ%0$$mH0me=$E>wM@4mX*43Qvcgf7h#Cspx1-jXKn_<Ne7%#}vruU5&!b+7H6i=*%0dJ^!f!*%(C`$u`v^(6Pxjf+L%;xp0GJSj;E;@U`CQA}{2ls;A!;FNrL)MBZz_&_joqf{)UwnpNsRg}e|X0igjCdsk+bbfKI@EyfZDAPZ@>K)v><VN=AwpMFM%bpQ(hG@kZ=XAi$fILf$`WZ(>a*dc)Mzv%f$jZ!VoKMFPZwO46!;RI=bW#DsY9mAGT*M#8?p=7B%I1e(*V?6JJjN)AgLeQ{N8!y_#PcjMlNk9KO*>WW#h2lxxCFCX5$2Slk5i3m+27)(RX)<H)P<y1OZrK6N${rv@}s4J_0e|G1ac2F2$-|m2UBoedV<q+$}d+c0+AQsbSPDp;Ohm@QpDUWO(Qfg^6m5Xm8ekdy?XC-Y7VJZD_@GHQki+SUA^$B=5O~FD1NXJghU}mh!zweNJ=OvAL79x1U}aoJ_x_~nuRlUZ5{~Yh#55CG-}j6Xa=J~1+~qF8YoDG+>#OV;)2;m)~7fA<DR;o7}tAna`yhK)q(x<WlH7#qEvXP{c);D|7hz<6JSP!U~vSPF$mre0eSx9ZUw_R=5{L|tM7WIGLoNVx)2h1qKDvGniS*o-I`WMohaCGf|Z++XciRRkB_dQ_1~87mHyJW`2KkL_42)OF<!rT)q=+>FWXak>Gl8Tx0~2_)F=TGF)bo#0d@14rmv2kB`CQ<A)+2J<{WAhm03q3&-Y`phFOO6R$5^W7CS>I)u?%JNszRkDP05Q|McB)@o&oyUV3?KT3<2}isEbT2!v!zG~6WLKmN)GZyV32vy|1ITuDfJ^pU94QVo$~Q7qL9qTwvD&EB(dBpAJx#j<!tBIpBC+bv(egd%<T#l251yvy#b%eS{OCD+MdM56;y4NRE8S49)^vl1mrFT#>MSEJoNi9f4g^AYI~NugISMcoHG2N{FFp12ZD<4Hx?!00MA*!w=6bSgOPBG^m8`G;j0@3K?54n%M1rE|(`QBqVfGx>R@+(m7Z^ej))c~BV2qFj{PEXnFf-(i9-)92(YNxRZ@QLQaXrQS02LM>N`b=Nk$NrB$G0^VW8;*CDgryEz3ou`7vkQh^9pKqo^x^+$OmPbFe5V+;B+onQ|nw=x)1&Vg#iZfK0y}CyvcWMGX!q6EoNl&TI3a#XlbfyV$jOc+m)jjTBJD0k@+7N6&wJ$#<3|4(|3O*c}`7RsJ8qRVjREz@ofYOwhMvunZJNK^IYr=^%>>8=|oKuNXngLVTDsin@0vTJ{6*;d-XTQ4+A1|@=z12YQ{&6X9tL%ehj_C63Mr9*$OaTvYhtXptIZqr43)jxnJ&0*)l(eG-wW%8I^0P45HEPvjba19rtSYSq%g{c`z7l$^W&U9GReO7Z<2ro2vCPyQd|DGDIR1>L!U68Et8|_-n$lVuGxijEh0bDhyA#ZmQxpq_ncUZ+_?n#$zuAKfd{$+Sf@x~s_u_8=aGh|!cdRuZUd<4#dl@$j0YA911-g2~HQ5MXy!+;|yR?SVhu73}md}6d;y9lC-q+#3dWiPE$LK96v62&0q_1O@^gLHSGBnHXJq)lyAo$T!sCJ`|gpC3HXvt^dxhjws1eS$nvzkD1=;O<}q5nFu|L)1+#xJ({wHO2$0LdyXielW}BJS4K(Rq%AWHdC*IiyvKtAl49aZC`lkElu^6cI`-=rk4-K{*Ha5|j=S!XAHbu$~7?H-6LDan>wgk<U+*2vCu6yBG=USvD@SaO>Ryo8gST8EH?Nq@`kiO5xDCq)e$GHMI($7puc{r0f+@hp`{K-+t`ARh2QSEEYRP!HuUTBcAL&E_lmG(yikl`XbCF<(`Vwma~PfK}v{mxNLI3K|yPzhBAU$NK7$UNm5}#H@GGZ*&xh&6uh2^KR(IC4~~05UPAuYH~xNQ=T>GuxLS{2ANa{FN}|u4wZY0fwckdNl3G?vctd&Y`QxUZ=U{OjEP24}xfbPUs6?4XLlMv0l1k&4mAHzIj#e`oo42~LIbapS4%a7?a~>Y0a6VskaXvpDC~bnIadvXQlbAR}WP4N6=2Xt-xqD%d>`m$@vq-6J7O-eJ8>CT4FhZ|gGA3x*SWXP?2h)YS7LXQG-!u8`nf&20X8gO2`CNQXC=$5CmHi4)^MMX7XZyVQJiP!3T|;vfCKL?AVq^ma46>5x=mJd|Wm+nSQbxc)ZFDt;<63tk*L|MjHZ_IcttuZ|r#RSQL~7+zbdyS=bV!E?w_x!pVhF_iBA_r$ZrZ3vF=|r^RE~<VH}9po0T!leb)^v?8i{!fh+Tl#FxYF|Tj)<+0P>Nd4M`aTOZ4dlwIG-*B#Le>6XiVJU+fyQylZ%}v<$|eJG7vMEeJw#BLu190-pu4Tc-BfdMjw@I`-HG)g6Af*#3R6r6#248a|mZ`{^QTjA8Qu_j%4zsMUK``B`X-oWsbiPIT)R;228E)z|84P7qo$K5bdq&4`S?0G1x=K9znW^{z&1YlHj2aiiS-{QRgq_vqq;$lPCUcw6V=qZ-F$Q^+3qACAs}zu5w16pbmw=!-)cvMYFR@73h<bWhbvASJ4Fs=1d~Xc~h3TIN|sF=$7IVUe?xIJ>&NR<4q%CXL#6Fh8Lg=j8fV>lobIC#yjptR2fu5jI20M!elWGz+oi`pvM!b4~F*T5zqMLXW9RRI43JtplozUa*anI8Usglc{qv-k0s(!0MxH`;x^I_Lkec5>l|&SJ_@;1E8;+Js~9j<F9(`w)Kj1p3$>}z$leUrP1@WsF;hD2U_D2Qn9(~jLbBU+hDP4H&@@wZys*@Zi0EO3RuS6d~oCMmqYU{p1)`;JmzFPh;y=!iSOiond4c`R}?EvR0%qQg-5mIT$3?u)nOb5YF%tECDoEzq}HSvBtxQ{UA8dNzHNRR2fBC7fBNF+zxDB=g}hdvtC-QF&FelnC}mnqsjb$sTj3CWNa$JjeGn=A5$vOvWZAk?Y!*6)<bam&jk>!4y<NGiwP%@u#iAzp?vRUlPfD$%KKs~BGam=g6{?@EJ;)E2MRX@g?`?>p+4lk^p~X?xu@FB}x{HS}on;V}0*$a6-ew-sy&E1fl)+d9K!#4t5UUHnme_e#j6Jz|SN1D*uP~pO5Pq~yOkNzxxT_s(8&|^+F_4v586AchHlJu(L3*BrWS=;V&_N!-o;J+rlWCZ9n1;q7CHeWfpE?;EH%!iE;WEtbIL*-ZICwLYZ#zs+h{{$>CA2!qYN@Prh<9-$JV&h6+*>L<#d59-6@?O7GrHClr?GBv?aNkOQLsn*Ov$NNpJTH>LP~kZ+G);RoU-13V_gLQ{5YI`DK2<_x$Co4VsN>A>6rQEw#6OF;uc!ua~Nt74e?Hn0d$^gJGD*qN%_!9a3dR8PTRYMJF1wf<!7p~=ah~7aENnG_YSb)q)+NW_8#{wJ?_6fO0~Xyv~5|pAAWsx$o`9yqx5yf=i$PLmecmFa~kRv>=HO;u|hv8rUM+@;PiRMG!*j-&>yeH%#FDieHmG;qBO1^ExV(YnliHtnxn7=5h@k9a+S9Dza?#y%kIH)Gn;>KixlCr&#M{cVh+L8q6CvrILIB=9Q8cor9mn%M58Gb8C_t#B$7rQ5*9Ry1EG#?q0+#Xxs#^ST4*>7nJhcnZdk8hnxkACx6XdQ_8EV^F5j-b%D1+g3kq5=i-Ac2l3H|3bf7!1YdFg$dDGaYTdZ3N9kPWrfIcY9oFNqI1Ib`Z7XPNZVwTR(rIcL8w8Y-`gPrR;58qz>1ntj1S^wH}Q2ERV1x0ngN}`|<tti90m<jTEI#;(gT;vi;t4tYFgC;fFt!CFGPOB(YEohEzu{3l_sl_^U7LT@*cQe5!r3yCU&COObV6yb=3hKU+@->WIoacICvuHP}JzHZtqZe|cbii1G#9l&<#og0UwdJ6ev>Ucqa#b0%^t_L-J=x!0#IOz*Yq(r)JzO>(9{cy4no+ENtlQQZg(GOttdQ9vM#!@LbW-Uo_i|h*C>4}3!{Dw%2&~pD9fx+!6`NSh%wYh{DFQKRj}53MKRau<C-c)iTEpstHJ{t>`_~frf1^AOaS;rTMuW)LZ!NqY;4Xr0NzW6V#->vqjNN4dm2jjeZjDk~SDrm2@I9S2N*m1*evP$23)wQcw%s1($$ImHZGvs8KJ7?uiD-2dYdGN6Jg(=dCD@~3%`JF>rRCY@NZh7$G&+tNxUsRBGOLlCO3j`4TnGz#$usbz{V4q@rs*GCGeKF-B|V|Pe{od0TvcBlUb*#cao%NDfyyZ<69`7mdKZF2;XJdpmTrt@6iIHdTQzBlx%G|>k$fK<xeiEyQyjA-u+%;+Xc{6R>{|a%mmqqyN&z`8uhg&}KK4;xR%_Q8jQ?Xn=r2!j?C0%&ck}{SN87gyxq*V%TbIdTYe5%`qHpI*a-M6r!=7yH(6Dz>;mH+@H{sxD5Vx>uR9(5a-K0h06o!t-P2CJ2#+-Lap&MBK#Zq9djh>abzq!%F+~`osmh-@S#NvN!cbww098DiM#p=jPI!A7UM&(w5)swwwlQF;*-v;2Q$xn&~7PdzgFi@5@)83x^(h*E!NAGX!G-fwCwvs<(|HEC8=>UAt^jw|B4EO?c8i%-|Z*9okHzBDKXu?S%b;QboOL1CV%^IQ=qHzW{u7F`Nw6gAT@rfz)e|Q`h-+$w{ZsrO5lT#z;Yt5l8QtLTKNZz010FYBqwphA#sfW%qiRa!>b6087qSc(-K&0g~L-S1^;2;f{T$7Ds^hNR*TAfp^ij8Fl>l5ZvM^CBCb;v(lt)8u}RBtV}H!Yo^5&7&#@k~}}3S-)?+IFt+9Bz=pIY5?*#b|t=et?nYV8dsvtz@VMU>COz(39q&sD&cBjlfRkPr|50%S-n2hjoZ(tvEkE*@%39qsM7-EvN*`-<%`6?#mr!I^uakvA5o0lgJkNAP+?CFo~iC%vtlyBNeMdp$1K4+z+VC=t4Mw4553ZypG3p?($=ghL7|6!)@Ik2OLmz=W!SFW!NsIs^__jJ1JOZZdFp9>>0q1F`>`YEq8b3GW-}_rIcE*H|{0p(Q_Z&fZEGWcy5`;{Mb?X{y#1{>W|i~iO-IQN%ybh+P3Ol9^Bi@?athu4}l=Ybf81LZSwNTRWp?rVdiqO>j7%RXoTBWm7v%qI_v^11xk*M0Jz|ACkp9vkYH{NrIY25zMl}?WaAgdZ2iv{SLWoOUCz57=Y>x@mzT;18}+F*TfwEI7%3Zl*Bj<NcW{x6=Q#i}t4sg7@lLNjQAw&v;v9u=*oRCSNAIDdD_bU``H?N_16e?=Bka>hH)84IY4s0|$C<y}07Y@-m#8QPMzka!Q{K#az!gf*4T%QDW~K4#uQ{OF_g;Bz*xLbCRDb*O)_1Oc-q&9F-Vgl;vxjoN`t%iYtira`VP71w_%~gijlEdkIlk@J-to-w+Jjzx<p<xl7yhs9JFY+ND=&ZVN8k7}3D+L=>JRJpUTXGI{oqx5vdaU%@n=fcztw9$eD&4uy)+)DiC15-FS2hI@%j(;`YW#&uNR2HugD+%^<V$!J13w1etErp_obKr{5$rdz4BM%@wWM2jP-FpSpNBMy!PFKiNE>%^5-wLiy!~^V*bFM=r>>L-+FxGwf4%Z<K*_`zjAVqU6$W})!w6*Pwu+{fZhA8|8BkWzyIM&FTc*w-pp?<m(lme%isF(kNrb?yN@s80<0%~>{m<s3xm1+#Hm`PSz705WVeYl%awW7!X;Y_vVtxRO(j&pnNbOw6svMer4mXYZuU=Iz7t+7T)tK7HGGMd`O^2s0x7Q{{Hl5FPaSrDx`KxDlc@R*2IOF6%AcOx_r1~oeEh(#Ap5DcL`^+YvosDg7#kqe;cgPi4W^p6z@&9`<MNt2L}*k?nf6~v)3d<MpJsg<3>TLev7hHJUmAb)oxe=SA4g6fWUr{f{I6cS{^-jeFPBCb`ix^h<%DBsq*=%c=aNFz7FxGT*)^O|-BVFxO=K)Dh-3nO^2dAiyB4(I3dT$5eW{O^Uw`TKAAQHJw%6>2|5#VA+_Qecf}Hxw_b-3)I<*=EEYMkknaN<_2a%HOrL|hVcPo)do6hQ+;0@ErR9fO>PpnWJaP24eFklzW^0!6kANH4Czx-#IT#GaTiTf;Sx15&8$k?I^xgqEX+^KXy0+iZ%GLR;m_NFJT)lcv3d%kK<<M^H|Z^#vrzF%H_;$2(f!RZ71Vn4oN29{Y;wK6c%qypL~*qanm&J&Z2J`sc6tzwr;XB@evHth|zx}?qEX?($dGq`@=!{Yz)4ZFOqLX2M}$E(^tKDnB|@xNYv<xBlL77dLktTDuXsM&Z*ff4Ohl9^^1WXYc}l?byi+v@_cm*UO>#3Q=ea7y+Biv3~N!8b0y^VK)(M%zO4io(8g31P&pEeZ?p7!+%Zt$|(NWd9nLUhF-<7G@4T**}*3&Hc;qgV)wf_5D|V@RG&EzbJ<#_8!(_6??l69$XH8iHDc^q56FXOXi0qEdB5QKU^{@2>')
if hashlib.sha256(_exact_grounding_compressed).hexdigest() != EXACT_GROUNDING_COMPRESSED_SHA256:
    raise RuntimeError("exact grounding compressed payload integrity failure")
_exact_grounding_raw = _morichika_zlib.decompress(_exact_grounding_compressed)
if hashlib.sha256(_exact_grounding_raw).hexdigest() != EXACT_GROUNDING_RAW_SHA256:
    raise RuntimeError("exact grounding raw payload integrity failure")
_exact_grounding_payload = json.loads(_exact_grounding_raw.decode("utf-8"))
_exact_grounding_identity = dict(_exact_grounding_payload)
_exact_grounding_observed_id = str(_exact_grounding_identity.pop("pack_id", ""))
if (
    _exact_grounding_observed_id != EXACT_GROUNDING_PACK_ID
    or hashlib.sha256(canonical_json(_exact_grounding_identity).encode("utf-8")).hexdigest()
    != EXACT_GROUNDING_PACK_ID
    or _exact_grounding_payload.get("normalization_allowed") is not False
    or _exact_grounding_payload.get("fuzzy_contextual_matching_allowed") is not False
    or _exact_grounding_payload.get("public_test_only") is not True
    or _exact_grounding_payload.get("test_sha256") != EXPECTED_PUBLIC_TEST_SHA256
    or _exact_grounding_payload.get("test_columns")
    != ["id", "context", "prompt_bn", "response_bn"]
    or (_exact_grounding_payload.get("counts") or {}).get("closed")
    != 1155
    or (_exact_grounding_payload.get("counts") or {}).get("contextual_total")
    != 1361
    or (_exact_grounding_payload.get("counts") or {}).get("contextual_public_corpus_exact")
    != 934
    or (_exact_grounding_payload.get("counts") or {}).get("contextual_literal_same_record")
    != 864
    or (_exact_grounding_payload.get("counts") or {}).get("contextual_normalized_exact_only")
    != 70
    or (_exact_grounding_payload.get("counts") or {}).get("contextual_supplied_context_policy")
    != 427
):
    raise RuntimeError("exact grounding pack contract failure")
FROZEN_EXACT_GROUNDING = _MorichikaMappingProxyType({
    "closed": _MorichikaMappingProxyType(dict(_exact_grounding_payload["closed"])),
    "contextual_exact": _MorichikaMappingProxyType(
        dict(_exact_grounding_payload["contextual_exact"])
    ),
    "context_policy_public": _MorichikaMappingProxyType(
        dict(_exact_grounding_payload["context_policy_public"])
    ),
})
EXACT_GROUNDING_RECEIPT = _MorichikaMappingProxyType({
    "version": _exact_grounding_payload["version"],
    "pack_id": EXACT_GROUNDING_PACK_ID,
    "raw_sha256": EXACT_GROUNDING_RAW_SHA256,
    "counts": dict(_exact_grounding_payload["counts"]),
    "source_files": dict(_exact_grounding_payload["source_files"]),
    "key_contract": _exact_grounding_payload["key_contract"],
    "public_test_only": True,
    "test_sha256": _exact_grounding_payload["test_sha256"],
})
del (
    _exact_grounding_compressed, _exact_grounding_raw,
    _exact_grounding_payload, _exact_grounding_identity,
    _exact_grounding_observed_id,
)


MORICHIKA test rows: 5000 context: 2452 closed: 2548


In [2]:
"""Fail-closed runtime binding for the MORICHIKA 94-index Kaggle corpus.

The Kaggle dataset contains 94 executable mmap indexes and a separate exact
lexical policy cache.  Its root manifest deliberately covers the lean runtime
payload, but Kaggle does not upload three ``__pycache__`` files.  Those three
bytecode files are the only optional members; every source file and every data
asset remains size- and SHA-256-bound.

This module has no import-time filesystem side effects.  The Gemma31B builder
embeds it into the generated notebook and activates it explicitly.
"""

from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path, PurePosixPath
from typing import Any, Mapping, Sequence


# The logical ID is part of the byte-bound root manifest and cannot change
# without rebuilding the whole bundle.  The owner-managed V2 transport keeps
# that ID; the older private mirror remains accepted only as metadata identity
# for local/private verification and is not the attached V2 transport.
FULL88_DATASET_ID = "ishtyy/morichika-full88-mmap-policy-20260721-codex1"
FULL88_KAGGLE_DATASET_ID = FULL88_DATASET_ID
FULL88_LEGACY_PRIVATE_DATASET_ID = (
    "ishtyy/morichika-full88-private-20260721-codex2"
)
FULL88_ACCEPTED_METADATA_IDS = frozenset({
    FULL88_DATASET_ID,
    FULL88_LEGACY_PRIVATE_DATASET_ID,
})
FULL88_SCHEMA = "morichika-tfidf-lean-executable-bundle-v2"
FULL88_MANIFEST_ID = "91a3b39fbb1d10ec2165f27222026ef00a78a5b963557de0fa03f1f2b9b2cd4d"
FULL88_MANIFEST_SHA256 = "9d4e98f2e0d016aa70803c8d5358329e2960e4d337f12c8b0c6724421c9c20cc"
FULL88_SOURCE_ID_DIGEST = "2681d6d16fad97f14791716607ce41f80ea1233235ca585827f93a79840e91a2"
FULL88_DIRECTORY_DIGEST = "f97067c33920ca1d87447b957be8139ca2be1f7edeef8e35b5e5014858b8f4dc"
FULL88_CATALOG_DIGEST = "06ced7b7258203b74b9f7a66e45b025ea9a0417962fdc7af6d8104fb7ff29be4"
FULL88_BASE_MANIFEST_ID = "aa05d602d0f33f7a23231aa9234542112d206d8c08095702edfd8ddfa4b91ada"
FULL88_BASE_MANIFEST_SHA256 = "239dd82efb067637f549f36be24c6aa2c868345a0762dece7ec510a5cd04ea16"
FULL88_ADDENDUM_ID = "9c770ed34b752e34bedcf5487b8567e5c03116fc625d5ea32bb9c6d78f7e17b9"
FULL88_INDEX_COUNT = 94
FULL88_MANIFEST_SOURCE_COUNT = 93
FULL88_RECORD_COUNT = 1_232_928
FULL88_FILE_COUNT = 888
FULL88_PAYLOAD_BYTES = 4_041_371_906
FULL88_LEXICAL_PATH = (
    "artifacts/retrieval/phase2_lexical_seed_v2/lexical_seed_records.jsonl"
)
FULL88_LEXICAL_RECORDS = 14_785

# The exact lexical cache is not an mmap retrieval source.  Keep its rights-
# admitted IDs in a separate allowlist so it cannot bypass, or accidentally be
# conflated with, the Full88 retrieval source-role policy.
FULL88_LEXICAL_MODEL_SOURCE_IDS = frozenset({
    "bangla_bagdhara_cc_by_sa_4",
    "bnwiktionary_cc_by_sa_4_20260701",
})

FULL88_OPTIONAL_BYTECODE = frozenset({
    "pipeline/__pycache__/phase2_canonicalize.cpython-313.pyc",
    "pipeline/__pycache__/phase2_mmap_retrieval.cpython-313.pyc",
    "pipeline/__pycache__/phase2_retrieval_lane_policy.cpython-313.pyc",
})
FULL88_ALLOWED_UNDECLARED_METADATA = frozenset({
    "bundle_manifest.json",
    "dataset-metadata.json",
})
FULL88_MMAP_REQUIRED_FILES = frozenset({
    "data.npy",
    "indices.npy",
    "indptr.npy",
    "idf.npy",
    "records.sqlite3",
    "vectorizer.json",
    "vocabulary.json",
})
FULL88_RIGHTS_BOOLEAN_FIELDS = frozenset({
    "bundle_allowed",
    "public_redistribution",
    "attribution_required",
    "noncommercial_only",
    "share_alike",
    "local_only",
    "targeted_evidence_only",
    "quarantined",
})

# These sources may be shown to the closed-book judge.  The other 59 indexes
# remain loaded and queried for audit coverage but are removed from model
# prompts and stripped of terminal authority.  Contextual mmap proxies are not
# queried; the v5 exception is lexical-exact and nonterminal only.
FULL88_MODEL_EVIDENCE_SOURCE_IDS = frozenset({
    "accessible_dictionary_competition_terms",
    "bangla_bagdhara",
    "bangla_bcs_qs",
    "bangla_math",
    "bangla_med_qa",
    "bangla_mmlu",
    "bangla_mmlu__bangla_mmlu_v1",
    "bangla_qa_84k",
    "bangla_textbook_tigerllm",
    "bangla_wordnet",
    "banglapedia_bn_metadata_20260718",
    "banglarqa",
    "bcs_canonical_archive",
    "bcs_downloads_10_50",
    "bcs_gk_ict_10_40",
    "bcs_public_archive",
    "bd_bcs_multimodal",
    "bengali_antonym_dictionary_commons_v1",
    "bengali_grammar_lexical_web_v1",
    "bengali_wikipedia_20210320",
    "benqa",
    "bluck",
    "bn_wiktionary_competition_terms",
    "bnmmlu_134k",
    "bnwiktionary_20260702",
    "broad_bn_grammar_census",
    "closed_residual_wikipedia_67",
    "downloads_bcs_10_50",
    "factoid_bangla_qa",
    "joykoli_six_part",
    "nctb_education_aux",
    "nctb_passage_qa_mendeley_v1",
    "nctb_schooltext",
    "openai_mmmlu_bn",
    "uddipok_v3",
})
FULL88_AUDIT_ONLY_SOURCE_IDS = frozenset({
    "banglaquad_v1",
    "bcs_preli_ocr_private_20260718",
    "bcs_preli_struct_priv_v1",
    "bcs_preli_struct_priv_v2",
    "bdquestionbank_bcs_inventory",
    "benhallueval_full_tasks",
    "benhallueval_qa",
    "biddabari_bcs_inventory",
    "ca_2026_vis_rec_v1",
    "ca_2026_vis_rec_v2",
    "ca_2026_vis_rec_v3",
    "ca_eight_join_recrop_v1",
    "ca_full_pdf_inv_ocr_v1",
    "ca_gold_accept_audit_v1",
    "ca_gold_accept_audit_v2",
    "ca_ocr_qgate_v1",
    "ca_ocr_qgate_v2",
    "ca_ocr_qgate_vis_add_v1",
    "ca_parent_inv_v1",
    "ca_parent_inv_v2",
    "ca_parent_inv_v3",
    "ca_parent_inv_v4",
    "ca_priv_2021_vis_canary_v1",
    "ca_priv_2026_lay_cp_v2",
    "ca_priv_2026_lay_land_v1",
    "ca_priv_2026_lay_land_v1_rp",
    "ca_priv_2026_pub_gold_v1",
    "ca_priv_2026_vis_lock2_v1",
    "ca_priv_lay_canary_jan2021_p8p9",
    "ca_priv_pub_map_all_v1",
    "ca_priv_pub_time_gold_v1",
    "ca_priv_v3_20260719",
    "ca_priv_v3_rp_a",
    "ca_priv_v3_rp_b",
    "ca_priv_v3_rp_verify",
    "ca_proof_ranked_q_v1",
    "ca_pub_time_exact_gold13_v1",
    "ca_pub_time_exact_gold43_v2",
    "ca_reop_fam_recrop_v1",
    "ca_reop_fam_recrop_v2",
    "ca_reop_fam_recrop_v3",
    "ca_sel_rem_v1",
    "ca_w1_jan2022_inline_qa_p7_v1",
    "ca_w1_next_lay_p41_v1",
    "ca_w1_next_lay_p41_v2",
    "ca_w1_nov2021_p31_v1",
    "ca_w1_top37_recrop_v1",
    "contextual_bengali_grammar_sources",
    "contextual_candidate_authoritative_sources",
    "contextual_dataset_candidates_20260716",
    "contextual_grammar_dictionary_terms",
    "contextual_hallucination_research",
    "contextual_idiom_dictionaries",
    "contextual_idiom_targeted_web_sources",
    "contextual_residual_wikipedia",
    "current_affairs_private_20260719",
    "deep_ctx_search",
    "proshno_binnash_v2",
    "squad_bn",
})
FULL88_ADDENDUM_SOURCE_IDS = frozenset({
    "banglaquad_v1",
    "nctb_passage_qa_mendeley_v1",
    "openai_mmmlu_bn",
    "proshno_binnash_v2",
    "squad_bn",
    "uddipok_v3",
})
FULL88_ADDENDUM_MODEL_EVIDENCE_SOURCE_IDS = frozenset({
    "nctb_passage_qa_mendeley_v1",
    "openai_mmmlu_bn",
    "uddipok_v3",
})
FULL88_ADDENDUM_AUDIT_ONLY_SOURCE_IDS = frozenset({
    "banglaquad_v1",
    "proshno_binnash_v2",
    "squad_bn",
})
FULL88_CONTEXTUAL_EXACT_SOURCE_IDS = frozenset({
    "banglaquad_v1",
    "nctb_passage_qa_mendeley_v1",
    "proshno_binnash_v2",
    "squad_bn",
    "uddipok_v3",
})
FULL88_SOURCE_ROLE_POLICY_SHA256 = (
    "96ac9f059f7058d5a3ff2f3a4511e5baa45fbe38899ad0c62abdfe6647ae3f46"
)


class Full88BindingError(RuntimeError):
    """A fail-closed Full88 identity, integrity, or policy violation."""


def full88_canonical_json(value: object) -> str:
    return json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    )


def full88_digest(value: object) -> str:
    return hashlib.sha256(full88_canonical_json(value).encode("utf-8")).hexdigest()


def full88_sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _full88_safe_relative(value: object) -> PurePosixPath:
    raw = str(value)
    relative = PurePosixPath(raw)
    if (
        not raw
        or raw == "."
        or "\\" in raw
        or relative.is_absolute()
        or ".." in relative.parts
        or relative.as_posix() != raw
    ):
        raise Full88BindingError(f"unsafe Full88 relative path: {raw!r}")
    return relative


def full88_source_role_policy() -> dict[str, object]:
    policy = {
        "model_evidence": sorted(FULL88_MODEL_EVIDENCE_SOURCE_IDS),
        "audit_only": sorted(FULL88_AUDIT_ONLY_SOURCE_IDS),
        "default": "fail_closed",
    }
    if full88_digest(policy) != FULL88_SOURCE_ROLE_POLICY_SHA256:
        raise Full88BindingError("Full88 source-role policy identity changed")
    return policy


def validate_full88_manifest(
    manifest_path: Path,
    manifest: Mapping[str, Any],
) -> dict[str, dict[str, Any]]:
    """Validate the root document and return its unique file catalog."""

    manifest_path = Path(manifest_path)
    if (
        not manifest_path.is_file()
        or manifest_path.is_symlink()
        or full88_sha256_file(manifest_path) != FULL88_MANIFEST_SHA256
    ):
        raise Full88BindingError("Full88 bundle-manifest file hash mismatch")
    if (
        manifest.get("schema") != FULL88_SCHEMA
        or manifest.get("dataset_id") != FULL88_DATASET_ID
        or manifest.get("content_id") != FULL88_MANIFEST_ID
        or manifest.get("manifest_id") != FULL88_MANIFEST_ID
    ):
        raise Full88BindingError("Full88 bundle identity mismatch")
    core = {
        key: value
        for key, value in manifest.items()
        if key not in {"content_id", "manifest_id"}
    }
    if full88_digest(core) != FULL88_MANIFEST_ID:
        raise Full88BindingError("Full88 bundle self-identity mismatch")

    exact_values = {
        "executable_mmap_index_count": FULL88_INDEX_COUNT,
        "unique_runtime_source_count": FULL88_INDEX_COUNT,
        "manifest_source_count": FULL88_MANIFEST_SOURCE_COUNT,
        "file_count": FULL88_FILE_COUNT,
        "total_bytes": FULL88_PAYLOAD_BYTES,
        "total_declared_records": FULL88_RECORD_COUNT,
        "lexical_policy_cache": FULL88_LEXICAL_PATH,
        "source_id_digest": FULL88_SOURCE_ID_DIGEST,
        "directory_digest": FULL88_DIRECTORY_DIGEST,
        "catalog_digest": FULL88_CATALOG_DIGEST,
    }
    if any(manifest.get(key) != value for key, value in exact_values.items()):
        raise Full88BindingError("Full88 declared corpus contract changed")
    if (
        manifest.get("all_numpy_assets_included") is not True
        or manifest.get("all_sqlite_quick_checks_passed") is not True
        or manifest.get("competition_labels_included") is not False
        or manifest.get("competition_test_prompt_payload_in_addendum") is not False
        or manifest.get("lean_executable_bundle") is not True
        or manifest.get("raw_external119_object_store_included") is not False
        or manifest.get("relevant_runtime_assets_only") is not True
        or manifest.get("external119_registry_receipts_included") is not True
        or manifest.get("contextual_corpora_embedded_in_sqlite_records") is not True
        or manifest.get("closed_book_retrieval_policy_unchanged") is not True
        or manifest.get("contextual_external_retrieval_allowed") is not True
        or manifest.get("contextual_fuzzy_retrieval_allowed") is not False
        or manifest.get("contextual_question_only_retrieval_allowed") is not False
        or manifest.get("contextual_retrieval_policy")
        != "byte_identical_full_triple_only"
        or manifest.get("base_manifest_id") != FULL88_BASE_MANIFEST_ID
        or manifest.get("base_manifest_sha256") != FULL88_BASE_MANIFEST_SHA256
    ):
        raise Full88BindingError("Full88 safety flags changed")

    addendum = manifest.get("external119_addendum")
    if not isinstance(addendum, Mapping):
        raise Full88BindingError("Full88 External119 addendum is missing")
    addendum_policy = addendum.get("policy")
    if (
        addendum.get("addendum_id") != FULL88_ADDENDUM_ID
        or int(addendum.get("indexes_added", -1)) != 6
        or set(addendum.get("source_ids") or []) != FULL88_ADDENDUM_SOURCE_IDS
        or addendum.get("competition_labels_included") is not False
        or addendum.get("contextual_external_retrieval_allowed") is not True
        or addendum.get("contextual_retrieval_policy")
        != "byte_exact_full_triple_only"
        or set(addendum.get("contextual_exact_source_ids") or [])
        != FULL88_CONTEXTUAL_EXACT_SOURCE_IDS
        or not isinstance(addendum_policy, Mapping)
        or addendum_policy.get("all_nonterminal") is not True
        or addendum_policy.get("competition_labels_read") is not False
        or addendum_policy.get("competition_labels_included") is not False
        or addendum_policy.get("contextual_fuzzy_candidate_retrieval_allowed")
        is not False
        or addendum_policy.get("contextual_question_only_retrieval_allowed")
        is not False
        or set(addendum_policy.get("contextual_exact_source_ids") or [])
        != FULL88_CONTEXTUAL_EXACT_SOURCE_IDS
    ):
        raise Full88BindingError("Full88 External119 policy binding changed")

    raw_files = manifest.get("files")
    if not isinstance(raw_files, Sequence) or isinstance(
        raw_files, (str, bytes, bytearray)
    ) or len(raw_files) != FULL88_FILE_COUNT:
        raise Full88BindingError("Full88 file catalog cardinality mismatch")
    specs: dict[str, dict[str, Any]] = {}
    for raw_spec in raw_files:
        if not isinstance(raw_spec, Mapping):
            raise Full88BindingError("Full88 file spec is not an object")
        spec = dict(raw_spec)
        relative = _full88_safe_relative(spec.get("path")).as_posix()
        if relative in specs:
            raise Full88BindingError(f"duplicate Full88 file declaration: {relative}")
        if (
            type(spec.get("bytes")) is not int
            or int(spec["bytes"]) < 0
            or not isinstance(spec.get("sha256"), str)
            or len(str(spec["sha256"])) != 64
        ):
            raise Full88BindingError(f"invalid Full88 file declaration: {relative}")
        specs[relative] = spec
    if sum(int(spec["bytes"]) for spec in specs.values()) != FULL88_PAYLOAD_BYTES:
        raise Full88BindingError("Full88 file-byte total mismatch")
    if not FULL88_OPTIONAL_BYTECODE.issubset(specs):
        raise Full88BindingError("Full88 optional bytecode declarations changed")
    if FULL88_LEXICAL_PATH not in specs or "RIGHTS_SUMMARY.json" not in specs:
        raise Full88BindingError("Full88 policy payload is incomplete")

    indexes = manifest.get("indexes")
    if not isinstance(indexes, list) or len(indexes) != FULL88_INDEX_COUNT:
        raise Full88BindingError("Full88 index catalog cardinality mismatch")
    runtime_ids = [str(row.get("runtime_source_id", "")) for row in indexes]
    manifest_ids = [str(row.get("manifest_source_id", "")) for row in indexes]
    directories = [str(row.get("path", "")) for row in indexes]
    if (
        len(set(runtime_ids)) != FULL88_INDEX_COUNT
        or len(set(manifest_ids)) != FULL88_MANIFEST_SOURCE_COUNT
        or len(set(directories)) != FULL88_INDEX_COUNT
        or any(not value for value in runtime_ids + manifest_ids + directories)
        or set(runtime_ids)
        != FULL88_MODEL_EVIDENCE_SOURCE_IDS | FULL88_AUDIT_ONLY_SOURCE_IDS
        or FULL88_MODEL_EVIDENCE_SOURCE_IDS & FULL88_AUDIT_ONLY_SOURCE_IDS
        or FULL88_ADDENDUM_MODEL_EVIDENCE_SOURCE_IDS
        != FULL88_ADDENDUM_SOURCE_IDS & FULL88_MODEL_EVIDENCE_SOURCE_IDS
        or FULL88_ADDENDUM_AUDIT_ONLY_SOURCE_IDS
        != FULL88_ADDENDUM_SOURCE_IDS & FULL88_AUDIT_ONLY_SOURCE_IDS
    ):
        raise Full88BindingError("Full88 source catalog or policy partition changed")
    for directory in directories:
        _full88_safe_relative(directory)
    full88_source_role_policy()
    return specs


def verify_full88_direct_tree(
    root: Path,
    specs: Mapping[str, Mapping[str, Any]],
    *,
    allow_local_generated_bytecode: bool = False,
) -> dict[str, Any]:
    """Hash every runtime member, tolerating only three Kaggle-dropped pycs."""

    root = Path(root).resolve()
    if not root.is_dir() or root.is_symlink():
        raise Full88BindingError(f"invalid Full88 root: {root}")
    actual: set[str] = set()
    for path in root.rglob("*"):
        if path.is_symlink():
            raise Full88BindingError(f"Full88 symlink is forbidden: {path}")
        if path.is_file():
            actual.add(path.relative_to(root).as_posix())
    missing = set(specs) - actual
    extra = actual - set(specs)
    if missing - FULL88_OPTIONAL_BYTECODE:
        raise Full88BindingError(
            "Full88 runtime files missing: " + repr(sorted(missing)[:5])
        )
    allowed_extra = set(FULL88_ALLOWED_UNDECLARED_METADATA)
    if allow_local_generated_bytecode:
        allowed_extra.update(
            relative
            for relative in extra
            if relative.startswith("pipeline/__pycache__/")
            and relative.endswith(".pyc")
        )
    if extra - allowed_extra:
        raise Full88BindingError(
            "Full88 undeclared runtime files present: " + repr(sorted(extra)[:5])
        )
    for relative, raw_spec in specs.items():
        path = root.joinpath(*PurePosixPath(relative).parts)
        if relative in FULL88_OPTIONAL_BYTECODE and not path.exists():
            continue
        if (
            not path.is_file()
            or path.is_symlink()
            or path.stat().st_size != int(raw_spec["bytes"])
            or full88_sha256_file(path) != str(raw_spec["sha256"])
        ):
            raise Full88BindingError(f"Full88 file binding failed: {relative}")
    metadata_path = root / "dataset-metadata.json"
    if metadata_path.is_file():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8-sig"))
        except Exception as error:
            raise Full88BindingError("invalid Full88 dataset metadata") from error
        if metadata.get("id") not in FULL88_ACCEPTED_METADATA_IDS:
            raise Full88BindingError("wrong Full88 dataset metadata identity")
    return {
        "declared_files": len(specs),
        "verified_runtime_files": len(specs) - len(missing),
        "optional_nonruntime_bytecode_missing": sorted(missing),
        "allowed_metadata_or_local_bytecode_files": sorted(extra),
        "verified_runtime_bytes": sum(
            int(spec["bytes"])
            for relative, spec in specs.items()
            if relative not in missing
        ),
    }


def materialize_full88(
    input_root: Path,
    work_dir: Path | None = None,
) -> tuple[Path, dict[str, Any], dict[str, Any]]:
    """Resolve the one exact direct Kaggle mount and verify its full closure."""

    del work_dir  # Reserved for a future, separately hash-bound archive format.
    matches: list[tuple[Path, dict[str, Any], dict[str, dict[str, Any]]]] = []
    for manifest_path in sorted(Path(input_root).rglob("bundle_manifest.json")):
        if not manifest_path.is_file() or manifest_path.is_symlink():
            continue
        try:
            if full88_sha256_file(manifest_path) != FULL88_MANIFEST_SHA256:
                continue
            value = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
            if not isinstance(value, Mapping):
                continue
            manifest = dict(value)
            specs = validate_full88_manifest(manifest_path, manifest)
        except (OSError, UnicodeError, ValueError, TypeError, Full88BindingError):
            continue
        matches.append((manifest_path.parent.resolve(), manifest, specs))
    if len(matches) != 1:
        raise Full88BindingError(
            f"expected one exact Full88 direct mount, found {len(matches)}"
        )
    root, manifest, specs = matches[0]
    verification = verify_full88_direct_tree(root, specs)
    receipt = {
        "version": "morichika-full94-lean-runtime-binding-v2",
        "dataset_id": FULL88_DATASET_ID,
        "kaggle_transport_dataset_id": FULL88_KAGGLE_DATASET_ID,
        "schema": FULL88_SCHEMA,
        "manifest_id": FULL88_MANIFEST_ID,
        "manifest_sha256": FULL88_MANIFEST_SHA256,
        "root": str(root),
        **verification,
    }
    receipt["receipt_id"] = full88_digest(receipt)
    return root, manifest, receipt


def build_full88_index_specs(
    root: Path,
    manifest: Mapping[str, Any],
) -> list[dict[str, Any]]:
    """Bind all 94 child manifests and mmap payload declarations."""

    root = Path(root).resolve()
    file_specs = {
        str(row["path"]): dict(row)
        for row in manifest.get("files", [])
        if isinstance(row, Mapping)
    }
    result: list[dict[str, Any]] = []
    for catalog in manifest.get("indexes", []):
        if not isinstance(catalog, Mapping):
            raise Full88BindingError("Full88 index catalog row is not an object")
        relative = _full88_safe_relative(catalog.get("path"))
        directory = root.joinpath(*relative.parts)
        manifest_path = directory / "manifest.json"
        try:
            child = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
        except Exception as error:
            raise Full88BindingError(f"unreadable child manifest: {relative}") from error
        manifest_relative = (relative / "manifest.json").as_posix()
        top_manifest_spec = file_specs.get(manifest_relative)
        if (
            not isinstance(child, Mapping)
            or not isinstance(top_manifest_spec, Mapping)
            or full88_sha256_file(manifest_path) != catalog.get("manifest_sha256")
            or top_manifest_spec.get("sha256") != catalog.get("manifest_sha256")
            or child.get("source_id") != catalog.get("manifest_source_id")
            or child.get("cache_id") != catalog.get("cache_id")
            or int(child.get("records", -1)) != int(catalog.get("records", -2))
            or list(child.get("matrix_shape") or [])
            != [int(catalog.get("records", -1)), int(catalog.get("features", -1))]
            or int(child.get("matrix_nnz", -1)) != int(catalog.get("nnz", -2))
        ):
            raise Full88BindingError(f"Full88 child manifest mismatch: {relative}")
        rights = child.get("rights_policy")
        if (
            not isinstance(rights, Mapping)
            or any(type(rights.get(field)) is not bool for field in FULL88_RIGHTS_BOOLEAN_FIELDS)
            or rights.get("bundle_allowed") is not True
            or rights.get("quarantined") is not False
            or not str(rights.get("note", "")).strip()
            or (rights.get("local_only") is True and rights.get("public_redistribution") is True)
        ):
            raise Full88BindingError(f"Full88 child rights gate failed: {relative}")
        runtime_source_id = str(catalog["runtime_source_id"])
        if runtime_source_id in FULL88_ADDENDUM_SOURCE_IDS:
            evidence_contract = child.get("evidence_contract")
            expected_model_facing = (
                runtime_source_id in FULL88_ADDENDUM_MODEL_EVIDENCE_SOURCE_IDS
            )
            if (
                not isinstance(evidence_contract, Mapping)
                or evidence_contract.get("model_facing")
                is not expected_model_facing
                or evidence_contract.get("audit_only")
                is not (not expected_model_facing)
                or evidence_contract.get("terminal_eligible") is not False
                or evidence_contract.get("exact_match_terminal_eligible")
                is not False
                or evidence_contract.get("fuzzy_match_terminal_eligible")
                is not False
                or evidence_contract.get("competition_labels_read") is not False
                or evidence_contract.get("competition_labels_included") is not False
            ):
                raise Full88BindingError(
                    f"Full88 addendum evidence-role gate failed: {relative}"
                )
        declared = child.get("files")
        if not isinstance(declared, Mapping) or set(declared) != FULL88_MMAP_REQUIRED_FILES:
            raise Full88BindingError(f"Full88 child mmap set mismatch: {relative}")
        for name in FULL88_MMAP_REQUIRED_FILES:
            child_spec = declared.get(name)
            top_spec = file_specs.get((relative / name).as_posix())
            if (
                not isinstance(child_spec, Mapping)
                or not isinstance(top_spec, Mapping)
                or int(child_spec.get("bytes", -1)) != int(top_spec.get("bytes", -2))
                or child_spec.get("sha256") != top_spec.get("sha256")
            ):
                raise Full88BindingError(
                    f"Full88 child/top file binding mismatch: {relative / name}"
                )
        result.append({
            "source_id": str(catalog["runtime_source_id"]),
            "manifest_source_id": str(catalog["manifest_source_id"]),
            "directory": directory,
            "cache_id": str(catalog["cache_id"]),
            "manifest_sha256": str(catalog["manifest_sha256"]),
            "records": int(catalog["records"]),
            "rights_note": full88_canonical_json(dict(rights)),
            "terminal_policy": (
                "phase2 exact-source rules only; fuzzy and unrecognized source "
                "records are nonterminal"
            ),
        })
    runtime_ids = [row["source_id"] for row in result]
    if (
        len(result) != FULL88_INDEX_COUNT
        or len(set(runtime_ids)) != FULL88_INDEX_COUNT
        or set(runtime_ids)
        != FULL88_MODEL_EVIDENCE_SOURCE_IDS | FULL88_AUDIT_ONLY_SOURCE_IDS
    ):
        raise Full88BindingError("Full88 executable index set changed")
    return result


def load_full88_lexical_records(
    root: Path,
    manifest: Mapping[str, Any],
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    declared = str(manifest.get("lexical_policy_cache", ""))
    if declared != FULL88_LEXICAL_PATH:
        raise Full88BindingError("Full88 lexical cache binding changed")
    path = Path(root).joinpath(*_full88_safe_relative(declared).parts)
    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                value = json.loads(line)
            except json.JSONDecodeError as error:
                raise Full88BindingError(
                    f"invalid Full88 lexical JSON at line {line_number}"
                ) from error
            if (
                not isinstance(value, Mapping)
                or not str(value.get("source_id", "")).strip()
                or not str(value.get("operation", "")).strip()
            ):
                raise Full88BindingError(
                    f"invalid Full88 lexical record at line {line_number}"
                )
            records.append(dict(value))
    if len(records) != FULL88_LEXICAL_RECORDS:
        raise Full88BindingError(
            f"Full88 lexical record count changed: {len(records)}"
        )
    source_counts: dict[str, int] = {}
    for record in records:
        source_id = str(record["source_id"])
        source_counts[source_id] = source_counts.get(source_id, 0) + 1
    if set(source_counts) != FULL88_LEXICAL_MODEL_SOURCE_IDS:
        raise Full88BindingError(
            "Full88 lexical source allowlist changed: "
            f"{sorted(source_counts)}"
        )
    receipt = {
        "path": declared,
        "records": len(records),
        "bytes": path.stat().st_size,
        "sha256": full88_sha256_file(path),
        "model_evidence_source_ids": sorted(source_counts),
        "records_by_source": dict(sorted(source_counts.items())),
        "separate_from_mmap_source_role_policy": True,
        "typed_context_only_nonterminal_exception": True,
    }
    return records, receipt


def validate_full88_retrieval_receipt(
    receipt: Mapping[str, Any],
    rows: Sequence[Mapping[str, Any]],
    retrieved: Mapping[str, Mapping[str, Any]],
    index_specs: Sequence[Mapping[str, Any]],
    proxy_ids: set[str] | frozenset[str] = frozenset(),
) -> None:
    observed = receipt.get("indexes")
    expected_bindings = [
        (
            str(spec["source_id"]),
            str(spec["cache_id"]),
            str(spec["manifest_sha256"]),
        )
        for spec in index_specs
    ]
    observed_bindings = [
        (
            str(item.get("source_id", "")),
            str(item.get("cache_id", "")),
            str(item.get("manifest_sha256", "")),
        )
        for item in (observed or [])
        if isinstance(item, Mapping)
    ]
    if (
        not isinstance(observed, list)
        or len(observed) != FULL88_INDEX_COUNT
        or observed_bindings != expected_bindings
        or set((receipt.get("per_index_seconds") or {}))
        != {str(spec["source_id"]) for spec in index_specs}
    ):
        raise Full88BindingError("retriever did not load/query the exact Full88 catalog")
    expected_ids = {str(row["example_id"]) for row in rows} | set(proxy_ids)
    if set(retrieved) != expected_ids:
        raise Full88BindingError("Full88 retrieval output ID coverage mismatch")
    contextual_ids = {
        str(row["example_id"])
        for row in rows
        if row.get("context_available") is True
    }
    for example_id in contextual_ids:
        item = retrieved[example_id]
        merged = item.get("merged_source_candidate") or {}
        if (
            list(item.get("retrieval_candidates") or [])
            or list(item.get("retrieval_audit_quarantined_candidates") or [])
            or merged.get("verdict") is not None
            or list(merged.get("evidence") or [])
            or merged.get("status") != "contextual_external_retrieval_disabled"
        ):
            raise Full88BindingError(
                f"ordinary contextual retrieval leak: {example_id}"
            )
    gate = receipt.get("contextual_external_retrieval") or {}
    if (
        gate.get("enabled") is not False
        or int(gate.get("count", -1)) != len(contextual_ids)
        or int(gate.get("typed_rule_theory_lookup_count", -1)) != 0
    ):
        raise Full88BindingError("Full88 contextual isolation receipt mismatch")


def apply_full88_source_role_policy(
    retrieved: Mapping[str, Mapping[str, Any]],
    rows: Sequence[Mapping[str, Any]],
    *,
    proxy_ids: set[str] | frozenset[str] = frozenset(),
) -> tuple[dict[str, dict[str, Any]], dict[str, Any]]:
    """Hide audit-only evidence from every model-facing retrieval path.

    Synthetic typed-context proxies are filtered by the same explicit source
    allowlist before the stricter v5 operation/term/slot filter sees them.
    Exact matching is not an authority grant: audit-only benchmark records
    remain quarantined even when their text overlaps a typed lookup exactly.
    """

    row_by_id = {str(row["example_id"]): row for row in rows}
    proxy_set = set(proxy_ids)
    output: dict[str, dict[str, Any]] = {}
    moved_to_audit = 0
    retained_for_model = 0
    cleared_audit_terminal = 0
    observed_sources: set[str] = set()
    for example_id, raw_item in retrieved.items():
        item = dict(raw_item)
        candidates = list(item.get("retrieval_candidates") or [])
        for candidate in candidates:
            source_id = str(candidate.get("source_id", ""))
            if source_id:
                observed_sources.add(source_id)
            if source_id not in (
                FULL88_MODEL_EVIDENCE_SOURCE_IDS | FULL88_AUDIT_ONLY_SOURCE_IDS
            ):
                raise Full88BindingError(
                    f"unclassified Full88 retrieval source: {source_id!r}"
                )
        if example_id in proxy_set:
            kept: list[dict[str, Any]] = []
            moved: list[dict[str, Any]] = []
            for candidate in candidates:
                source_id = str(candidate.get("source_id", ""))
                if source_id in FULL88_MODEL_EVIDENCE_SOURCE_IDS:
                    kept.append(candidate)
                else:
                    value = dict(candidate)
                    value["model_facing_eligible"] = False
                    value["terminal_label_authority"] = False
                    value["source_role_gate"] = {
                        "status": "audit_only_not_model_facing",
                        "policy_sha256": FULL88_SOURCE_ROLE_POLICY_SHA256,
                    }
                    moved.append(value)
            existing_audit = list(
                item.get("retrieval_audit_quarantined_candidates") or []
            )
            merged = dict(item.get("merged_source_candidate") or {})
            if merged.get("verdict") is not None or merged.get("evidence"):
                cleared_audit_terminal += 1
            item["retrieval_candidates"] = kept
            item["retrieval_audit_quarantined_candidates"] = existing_audit + moved
            item["merged_source_candidate"] = {
                "verdict": None,
                "status": "typed_context_proxy_terminal_candidate_suppressed",
                "evidence": [],
            }
            item["model_facing_retrieval_candidate_count"] = len(kept)
            item["source_role_audit_only_candidate_count"] = len(moved)
            item["source_role_policy_sha256"] = FULL88_SOURCE_ROLE_POLICY_SHA256
            item["source_role_gate"] = {
                "status": "typed_context_proxy_source_filtered_pending_v5_exact_filter",
                "terminal_label_authority": False,
                "policy_sha256": FULL88_SOURCE_ROLE_POLICY_SHA256,
            }
            retained_for_model += len(kept)
            moved_to_audit += len(moved)
            output[example_id] = item
            continue

        row = row_by_id.get(example_id)
        if row is None:
            raise Full88BindingError(f"unexpected Full88 retrieval ID: {example_id}")
        if row.get("context_available") is True:
            output[example_id] = item
            continue
        kept: list[dict[str, Any]] = []
        moved: list[dict[str, Any]] = []
        for candidate in candidates:
            source_id = str(candidate.get("source_id", ""))
            if source_id in FULL88_MODEL_EVIDENCE_SOURCE_IDS:
                kept.append(candidate)
            else:
                value = dict(candidate)
                value["model_facing_eligible"] = False
                value["terminal_label_authority"] = False
                value["source_role_gate"] = {
                    "status": "audit_only_not_model_facing",
                    "policy_sha256": FULL88_SOURCE_ROLE_POLICY_SHA256,
                }
                moved.append(value)
        merged = dict(item.get("merged_source_candidate") or {})
        merged_sources = {
            str(value.get("source_id", ""))
            for value in (merged.get("evidence") or [])
            if isinstance(value, Mapping)
        }
        if merged_sources - FULL88_MODEL_EVIDENCE_SOURCE_IDS:
            merged = {
                "verdict": None,
                "status": "audit_only_terminal_candidate_suppressed",
                "evidence": [],
            }
            cleared_audit_terminal += 1
        existing_audit = list(
            item.get("retrieval_audit_quarantined_candidates") or []
        )
        item["retrieval_candidates"] = kept
        item["retrieval_audit_quarantined_candidates"] = existing_audit + moved
        item["merged_source_candidate"] = merged
        item["model_facing_retrieval_candidate_count"] = len(kept)
        item["source_role_audit_only_candidate_count"] = len(moved)
        item["source_role_policy_sha256"] = FULL88_SOURCE_ROLE_POLICY_SHA256
        retained_for_model += len(kept)
        moved_to_audit += len(moved)
        output[example_id] = item
    receipt = {
        "version": "morichika-full88-source-role-policy-v2",
        "policy_sha256": FULL88_SOURCE_ROLE_POLICY_SHA256,
        "default": "fail_closed",
        "model_evidence_source_count": len(FULL88_MODEL_EVIDENCE_SOURCE_IDS),
        "audit_only_source_count": len(FULL88_AUDIT_ONLY_SOURCE_IDS),
        "observed_candidate_source_count": len(observed_sources),
        "retained_model_facing_candidates": retained_for_model,
        "moved_to_audit_only": moved_to_audit,
        "cleared_audit_only_terminal_candidates": cleared_audit_terminal,
        "typed_proxy_count": len(proxy_set),
    }
    receipt["receipt_id"] = full88_digest(receipt)
    return output, receipt


__all__ = [
    name
    for name in globals()
    if name.startswith("FULL88_")
    or name.startswith("full88_")
    or name in {
        "Full88BindingError",
        "validate_full88_manifest",
        "verify_full88_direct_tree",
        "materialize_full88",
        "build_full88_index_specs",
        "load_full88_lexical_records",
        "validate_full88_retrieval_receipt",
        "apply_full88_source_role_policy",
    }
]


In [3]:
# Exact Full94 Lean V2 mount, child-index, and policy activation.
MORICHIKA_ROOT, MORICHIKA_MANIFEST, FULL88_MATERIALIZATION_RECEIPT = materialize_full88(
    INPUT_ROOT, WORK_DIR
)
sys.path.insert(0, str(MORICHIKA_ROOT))
FULL88_INDEX_SPECS = build_full88_index_specs(MORICHIKA_ROOT, MORICHIKA_MANIFEST)
lexical_records, FULL88_LEXICAL_RECEIPT = load_full88_lexical_records(
    MORICHIKA_ROOT, MORICHIKA_MANIFEST
)
if frozenset(FULL88_LEXICAL_RECEIPT["model_evidence_source_ids"]) != FULL88_LEXICAL_MODEL_SOURCE_IDS:
    raise RuntimeError("Full88 lexical receipt source allowlist mismatch")
EXPECTED_MANIFEST_ID = FULL88_MANIFEST_ID
CORPUS_ADMISSION_RECEIPT = {
    "dataset_id": FULL88_DATASET_ID,
    "kaggle_transport_dataset_id": FULL88_KAGGLE_DATASET_ID,
    "schema": FULL88_SCHEMA,
    "manifest_id": FULL88_MANIFEST_ID,
    "manifest_sha256": FULL88_MANIFEST_SHA256,
    "executable_mmap_indexes": len(FULL88_INDEX_SPECS),
    "unique_runtime_sources": len({row["source_id"] for row in FULL88_INDEX_SPECS}),
    "manifest_source_ids": len({row["manifest_source_id"] for row in FULL88_INDEX_SPECS}),
    "declared_records": sum(int(row["records"]) for row in FULL88_INDEX_SPECS),
    "exact_lexical_records": len(lexical_records),
    "exact_lexical_model_evidence_sources": len(FULL88_LEXICAL_MODEL_SOURCE_IDS),
    "lexical_source_policy_separate_from_mmap": True,
    "closed_book_model_evidence_sources": len(FULL88_MODEL_EVIDENCE_SOURCE_IDS),
    "audit_only_not_model_facing_sources": len(FULL88_AUDIT_ONLY_SOURCE_IDS),
    "source_role_policy_sha256": FULL88_SOURCE_ROLE_POLICY_SHA256,
    "ordinary_context_external_retrieval": False,
    "typed_context_reference_policy": "v5 exact operation+operand/key+sense/register; nonterminal",
    "current_affairs_model_facing": False,
    "competition_labels_included": False,
    "silent_corpus_fallback_allowed": False,
    "materialization": FULL88_MATERIALIZATION_RECEIPT,
    "lexical": FULL88_LEXICAL_RECEIPT,
}
if (
    CORPUS_ADMISSION_RECEIPT["executable_mmap_indexes"] != FULL88_INDEX_COUNT
    or CORPUS_ADMISSION_RECEIPT["declared_records"] != FULL88_RECORD_COUNT
    or CORPUS_ADMISSION_RECEIPT["exact_lexical_records"] != FULL88_LEXICAL_RECORDS
):
    raise RuntimeError("Full94 Lean corpus activation contract changed")
print(
    "MORICHIKA Full94 Lean corpus:", MORICHIKA_ROOT,
    "indexes:", len(FULL88_INDEX_SPECS),
    "records:", CORPUS_ADMISSION_RECEIPT["declared_records"],
)


MORICHIKA Full94 Lean corpus: /kaggle/input/datasets/ishtyy/morichika-full88-mmap-policy-20260721-codex1 indexes: 94 records: 1232928


In [4]:
"""Context-only policy packet for the MORICHIKA generalized hybrid.

The module is intentionally self-contained so its exact source can be embedded
in an offline Kaggle notebook.  It never retrieves, labels, or consults a
competition row identifier.  It converts the supplied context, question and
candidate response into an auditable instruction packet for the verifier.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from typing import Any, Callable


VERSION = "morichika-context-policy-v4-full-coverage-map-aggregate"

# These are reasoning checks, not predicted classes.  Keeping the recovered
# inventory explicit prevents a later prompt edit from silently dropping an
# edge-case family.
CANONICAL_POLICY_FAMILIES = (
    "question_grounding_answerability_and_premise_validity",
    "exact_entity_relation_property_and_answer_type",
    "direct_support_contradiction_and_partial_containment",
    "supplied_definition_formula_theory_and_rule_application",
    "date_event_role_and_as_of_time",
    "year_age_duration_calendar_and_timezone",
    "bounded_arithmetic_units_ratio_percentage_and_order",
    "kinship_and_relational_composition",
    "creator_founder_user_operator_office_title_and_jurisdiction",
    "birthplace_residence_nationality_and_event_participation",
    "legal_definition_section_effective_date_minimum_maximum_and_frequency",
    "numeric_whole_component_range_extremum_ordinal_and_granularity",
    "negation_quantifier_comparator_modality_and_clause_scope",
    "antonym_idiom_prefix_register_etymology_and_guruchandali",
    "samas_sandhi_affix_spelling_natva_and_satva",
    "unicode_joiner_conjunct_digit_punctuation_ocr_and_word_break",
    "ambiguity_conflict_invalid_premise_and_no_world_rescue",
)

ENGINEERED_EVALUATION_CELLS = (
    "context_only_lane", "full_context_boundary", "answerable_supported",
    "answerable_refuted", "genuinely_missing_nei", "same_passage_different_question",
    "entity_swap", "relation_swap", "answer_type_swap", "creator_vs_user_operator",
    "birthplace_vs_residence_event_place", "partial_answer_extra_claim",
    "theory_application", "formula_operand_application", "age_vs_year",
    "relative_timeline", "event_phase_order", "kinship_composition",
    "unit_or_number_swap", "negation_scope", "quantifier_comparator_scope",
    "grammar_operation_swap", "lexical_exact_vs_semantic_near",
    "unicode_equivalent", "unicode_word_break_non_equivalent", "cross_window_conflict",
)

OPERATION_AXIS = (
    "antonym_lookup", "birth_date_slot", "entity_text_relation",
    "event_date_or_status_slot", "idiom_meaning_lookup",
    "legal_definition_or_threshold", "legal_effective_date", "legal_maximum_fine",
    "legal_maximum_imprisonment", "legal_minimum_meeting_frequency",
    "location_slot", "numeric_or_ordinal_slot", "prefix_origin_classification",
    "samas_taxonomy", "sandhi_formation",
)

_SIGNALS: tuple[tuple[str, str], ...] = (
    ("year_age_duration", r"ব[য়য়]স|জন্ম|বিবাহ|বিয়ে|বিয়ে|বছর|সাল|year|age"),
    ("calendar_date", r"তারিখ|কবে|দিন|মাস|জানুয়ারি|ফেব্রুয়ারি|মার্চ|ডিসেম্বর"),
    ("timeline_event_order", r"আগে|পরে|তারপর|পরবর্তীতে|প্রথমে|শেষে|ঘটে|ঘটেছিল"),
    ("relative_time_offset", r"(?:বছর|দিন|মাস)\s*(?:আগে|পরে)|after|before|later"),
    ("kinship_relation_composition", r"বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|নানা|grandfather"),
    ("bounded_arithmetic", r"কত|যোগ|বিয়োগ|বিয়োগ|গুণ|ভাগ|শতাংশ|হিসাব|calculate"),
    ("unit_dimension", r"কিলোমিটার|মিটার|গ্রাম|লিটার|সেকেন্ড|মিনিট|ঘণ্টা|টাকা|শতাংশ"),
    ("negation_scope", r"(?:^|\s)(?:না|নয়|নয়|নেই|নি)(?:\s|$)|হয়নি|হয়নি|করেনি"),
    ("quantifier_scope", r"সব|সকল|কোনো|কোনও|কেবল|শুধু|প্রত্যেক|একটিও|কিছু"),
    ("comparator_scope", r"সর্বাধিক|সর্বনিম্ন|বেশি|কম|পূর্ববর্তী|পরবর্তী|minimum|maximum"),
    ("modality_clause_scope", r"পারে|উচিত|অবশ্যই|সম্ভব|শর্তে|যদি|হলে"),
    ("definition_theory_rule_application", r"সংজ্ঞা|তত্ত্ব|সূত্র|নিয়ম|নিয়ম|অর্থ"),
    ("antonym_exact_pair", r"বিপরীত\s*শব্দ|antonym"),
    ("idiom_exact_meaning", r"বাগধারা|প্রবাদ"),
    ("prefix_suffix_origin", r"উপসর্গ|প্রত্যয়|প্রত্যয়"),
    ("samas_exact_operation", r"সমাস"),
    ("sandhi_exact_operands", r"সন্ধি"),
    ("register_etymology_guruchandali", r"গুরুচণ্ডালী|তৎসম|তদ্ভব|ফারসি|সংস্কৃত"),
    ("natva_satva_spelling", r"ণত্ব|ষত্ব|বানান"),
    ("creator_user_operator_role", r"স্রষ্টা|প্রতিষ্ঠাতা|নির্মাতা|ব্যবহারকারী|পরিচালক"),
    ("birthplace_residence_event_place", r"জন্মস্থান|বাসস্থান|রাজধানী|কোথায়|কোথায়|স্থান"),
    ("legal_definition_section", r"আইন|ধারা|বিধি|সংজ্ঞা"),
    ("minimum_maximum_frequency", r"সর্বনিম্ন|সর্বোচ্চ|কমপক্ষে|বেশিরভাগ|বার"),
    ("ordinal_list_order", r"প্রথম|দ্বিতীয়|দ্বিতীয়|তৃতীয়|তৃতীয়|ক্রম|তম"),
)

_BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_JOINERS = {"\u200c", "\u200d", "\ufeff"}
_FIXED_LEXICAL_SHELLS = {
    "antonym_lookup": re.compile(r"(?:বিপরীত\s*শব্দ|বিপরীতার্থক\s*শব্দ).*(?:কী|কি|কোন)", re.IGNORECASE),
    "idiom_meaning_lookup": re.compile(r"(?:বাগধারা|প্রবাদ).*(?:অর্থ|মানে).*(?:কী|কি|কোন)", re.IGNORECASE),
    "prefix_origin_classification": re.compile(r"উপসর্গ.*(?:উৎস|শ্রেণি|জাতীয়|জাতীয়|কোন)", re.IGNORECASE),
}


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def comparison_view(value: str) -> str:
    """Bounded comparison only; never rewrites literal evidence."""
    return " ".join(unicodedata.normalize("NFC", value).translate(_BN_DIGITS).casefold().split())


def unicode_receipt(value: str) -> dict[str, Any]:
    nfc = unicodedata.normalize("NFC", value)
    return {
        "literal_sha256": _sha(value),
        "utf8_bytes": len(value.encode("utf-8")),
        "codepoints": len(value),
        "nfc_identical": nfc == value,
        "nfc_sha256": _sha(nfc),
        "joiner_positions": [index for index, char in enumerate(value) if char in _JOINERS],
        "replacement_character_positions": [index for index, char in enumerate(value) if char == "\ufffd"],
        "comparison_view_sha256": _sha(comparison_view(value)),
    }


def contextual_exact_lexical_policy(
    question: str, response: str, records: list[dict[str, Any]] | None
) -> dict[str, Any]:
    """Expose only the frozen exact lexical-shell exception, nonterminally."""
    folded_question = comparison_view(question)
    folded_response = comparison_view(response)
    operation = next(
        (name for name, pattern in _FIXED_LEXICAL_SHELLS.items() if pattern.search(folded_question)),
        None,
    )
    base = {
        "lookup_mode": "not_applicable",
        "operation": operation,
        "key": None,
        "source_ids": [],
        "record_sha256": [],
        "exact_operation": operation is not None,
        "exact_key": False,
        "exact_sense_register": False,
        "conflict": False,
        "conflict_status": "none",
        "generic_retrieval_invoked": False,
        "terminal_label_authority": False,
        "model_nonterminal": True,
        "evidence": [],
    }
    if operation is None:
        base["lookup_mode"] = "forbidden_for_ordinary_context"
        return base

    matched: list[dict[str, Any]] = []
    conflicting: list[dict[str, Any]] = []
    for record in records or []:
        if record.get("operation") != operation:
            continue
        terms = [str(record.get("term_key") or "")] + [str(v) for v in record.get("display_terms", [])]
        exact_terms = []
        for term in terms:
            key = comparison_view(term)
            if key and re.search(rf"(?<![\u0980-\u09FFA-Za-z0-9]){re.escape(key)}(?![\u0980-\u09FFA-Za-z0-9])", folded_question):
                exact_terms.append(key)
        if not exact_terms:
            continue
        sense_values = [str(record.get(name) or "") for name in ("sense", "register", "etymological_class")]
        required_scope = [comparison_view(value) for value in sense_values if value]
        exact_scope = not required_scope or all(value in folded_question for value in required_scope)
        candidate = {**record, "_matched_key": sorted(exact_terms, key=lambda value: (-len(value), value))[0], "_exact_scope": exact_scope}
        if record.get("conflict_status") == "none" and exact_scope:
            matched.append(candidate)
        else:
            conflicting.append(candidate)

    if not matched and not conflicting:
        base["lookup_mode"] = "exact_shell_no_exact_key_nonterminal"
        return base
    all_candidates = matched + conflicting
    keys = {value["_matched_key"] for value in all_candidates}
    accepted_sets = {
        tuple(sorted(comparison_view(v) for v in value.get("accepted_answers", []) if str(v).strip()))
        for value in matched
    }
    conflict = bool(conflicting) or len(keys) != 1 or len(accepted_sets) > 1
    base.update({
        "lookup_mode": "exact_hash_bound_lexical_policy" if not conflict else "exact_lexical_conflict_nei_nonterminal",
        "key": sorted(keys)[0] if len(keys) == 1 else None,
        "source_ids": sorted({str(value.get("source_id") or "") for value in all_candidates if value.get("source_id")}),
        "record_sha256": sorted({_sha(_canonical({k: v for k, v in value.items() if not str(k).startswith("_")})) for value in all_candidates}),
        "exact_key": len(keys) == 1,
        "exact_sense_register": all(value.get("_exact_scope") is True for value in all_candidates),
        "conflict": conflict,
        "conflict_status": "conflict" if conflict else "none",
    })
    if not conflict:
        for value in matched[:3]:
            accepted = [comparison_view(v) for v in value.get("accepted_answers", [])]
            contrast = [comparison_view(v) for v in value.get("contrast_answers", [])]
            base["evidence"].append({
                "source_id": value.get("source_id"),
                "operation": operation,
                "term_key": value.get("term_key"),
                "accepted_answers": value.get("accepted_answers", []),
                "contrast_answers": value.get("contrast_answers", []),
                "response_relation": (
                    "support_candidate" if folded_response in accepted else
                    "counter_candidate" if folded_response in contrast else "neutral_candidate"
                ),
            })
    return base


def detected_policy_families(context: str, question: str, response: str) -> list[str]:
    joined = "\n".join((question, response, context))
    detected = {
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
    }
    for family, pattern in _SIGNALS:
        if re.search(pattern, joined, re.IGNORECASE):
            detected.add(family)
    if re.search(r"[০-৯0-9]", joined):
        detected.update({"number_whole_component_range", "formula_operand_application"})
    order = [name for name, _ in _SIGNALS]
    order += [
        "lane_context_only", "full_context_evidence_boundary", "question_answerability",
        "question_premise_validity", "exact_answer_slot", "same_passage_different_question",
        "entity_identity", "relation_property_identity", "answer_type_identity",
        "direct_support", "direct_contradiction", "partial_containment_extra_claim",
        "ambiguity_conflict_nei", "unicode_nfc_joiner_conjunct_digit", "ocr_word_break_caution",
        "number_whole_component_range", "formula_operand_application",
    ]
    return [family for family in dict.fromkeys(order) if family in detected]


def full_coverage_windows(
    context: str, *, max_chars: int = 3000, overlap_chars: int = 320
) -> list[dict[str, Any]]:
    """Partition every context character into overlapping replayable windows."""
    if not context:
        raise ValueError("context must be non-empty")
    if max_chars < 256 or overlap_chars < 32 or overlap_chars >= max_chars:
        raise ValueError("invalid full-coverage window geometry")
    windows: list[dict[str, Any]] = []
    start = 0
    while start < len(context):
        hard_end = min(len(context), start + max_chars)
        end = hard_end
        if hard_end < len(context):
            boundaries = [context.rfind(mark, start + max_chars // 2, hard_end) for mark in ("\n", "।", ".", "?", "!")]
            boundary = max(boundaries)
            if boundary >= start:
                end = boundary + 1
        literal = context[start:end]
        windows.append({
            "window_index": len(windows),
            "context_char_start": start,
            "context_char_end": end,
            "literal_text": literal,
            "literal_span_sha256": _sha(literal),
        })
        if end == len(context):
            break
        start = end - overlap_chars
    covered = [False] * len(context)
    for index, window in enumerate(windows):
        start, end = window["context_char_start"], window["context_char_end"]
        literal = window["literal_text"]
        if context[start:end] != literal or _sha(literal) != window["literal_span_sha256"]:
            raise ValueError(f"full-coverage window {index} does not replay")
        for position in range(start, end):
            covered[position] = True
    if not all(covered):
        raise ValueError("full-coverage window ledger contains a character gap")
    return windows


def build_window_adjudication_user(
    window: dict[str, Any], question: str, response: str, total_windows: int
) -> str:
    return (
        "WINDOW-LOCAL CONTEXT PASS. Judge only evidence present in this literal supplied-context window. "
        "A local miss is not a contradiction: output not_enough_information unless this window directly "
        "supports or directly refutes the exact question slot. Do not use outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        f"WINDOW {window['window_index'] + 1}/{total_windows} "
        f"[chars={window['context_char_start']}:{window['context_char_end']}; "
        f"sha256={window['literal_span_sha256']}]:\n{window['literal_text']}"
    )


def _literal_question_excerpt(
    window: dict[str, Any], question: str, max_chars: int
) -> dict[str, Any]:
    """Select a literal, question-conditioned excerpt without rewriting it."""
    literal = str(window["literal_text"])
    if max_chars < 32:
        max_chars = 32
    q_tokens = {
        token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", question)
        if len(token) > 1
    }
    candidates = list(re.finditer(r"[^।.!?\n]+[।.!?]?", literal))
    if candidates:
        def score(match: re.Match[str]) -> tuple[int, int]:
            tokens = {
                token.casefold() for token in re.findall(r"[\u0980-\u09FFA-Za-z0-9]+", match.group())
            }
            return (len(tokens & q_tokens), -match.start())
        best = max(candidates, key=score)
        local_start = max(0, best.start() - max_chars // 4)
        local_end = min(len(literal), local_start + max_chars)
    else:
        local_start, local_end = 0, min(len(literal), max_chars)
    excerpt = literal[local_start:local_end]
    absolute_start = int(window["context_char_start"]) + local_start
    absolute_end = absolute_start + len(excerpt)
    return {
        "excerpt_char_start": absolute_start,
        "excerpt_char_end": absolute_end,
        "literal_excerpt": excerpt,
        "literal_excerpt_sha256": _sha(excerpt),
    }


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Create the exact-question final adjudication over all window passes."""
    compact = [
        {
            "window_index": row["window_index"],
            "context_char_start": row["context_char_start"],
            "context_char_end": row["context_char_end"],
            "literal_span_sha256": row["literal_span_sha256"],
            "window_verdict": row["window_verdict"],
            "literal_excerpt": row.get("literal_excerpt", ""),
            "excerpt_char_start": row.get("excerpt_char_start"),
            "excerpt_char_end": row.get("excerpt_char_end"),
            "literal_excerpt_sha256": row.get("literal_excerpt_sha256"),
        }
        for row in window_results
    ]
    notes = list(selected_notes or [])[:32]
    derivations = list(bounded_derivations or [])[:8]
    return (
        "FINAL FULL-CONTEXT AGGREGATION. The window ledger collectively covered every supplied-context "
        "character. Rebind the exact question and candidate response. A not_enough_information window means "
        "only that its local span was silent; it is never counter-evidence. If any aligned window refutes the "
        "response, do not let an unrelated support override it. Unresolved support/refutation across different "
        "entities, slots, event phases, dates, operations, or ambiguous coreference is not_enough_information. "
        "Use no outside knowledge.\n\n"
        f"QUESTION (literal):\n{question}\n\nCANDIDATE RESPONSE (literal):\n{response}\n\n"
        "PER-WINDOW STRUCTURED RESULTS AND LITERAL EXCERPTS:\n" + _canonical(compact)
        + "\n\nFULL-CONTEXT ROUTER SOURCE-LINKED NOTES (advisory; offsets/hashes bind them to supplied context):\n"
        + _canonical(notes)
        + "\n\nBOUNDED DERIVATION CANDIDATES (advisory; verify operator, operands, slot and unit):\n"
        + _canonical(derivations)
        + "\n\nFROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden):\n"
        + _canonical(lexical_policy or {"lookup_mode": "not_applicable", "generic_retrieval_invoked": False})
    )


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: list[dict[str, Any]] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
    coverage_window_chars: int = 3000,
    coverage_overlap_chars: int = 320,
) -> tuple[str, dict[str, Any]]:
    """Build a context-only prompt plus restart-safe diagnostics."""
    if not all(isinstance(value, str) for value in (context, question, response)):
        raise TypeError("context, question and response must be strings")
    if not context.strip() or not question.strip():
        raise ValueError("context and question must be non-empty")

    route = router(context, question, response, max_windows=max_windows)
    if route.get("external_retrieval_allowed") is not False:
        raise ValueError("context router must explicitly forbid external retrieval")
    if route.get("context_sha256") != _sha(context):
        raise ValueError("context router hash mismatch")

    window_calls: list[dict[str, Any]] = []
    if len(context) <= full_context_char_limit:
        evidence = context
        evidence_mode = "complete_literal_supplied_context"
        full_context_inference_coverage = True
    else:
        coverage = full_coverage_windows(
            context,
            max_chars=coverage_window_chars,
            overlap_chars=coverage_overlap_chars,
        )
        excerpt_chars = max(48, min(320, 3200 // len(coverage)))
        window_calls = [
            {
                **{key: window[key] for key in (
                    "window_index", "context_char_start", "context_char_end", "literal_span_sha256"
                )},
                **_literal_question_excerpt(window, question, excerpt_chars),
                "user": build_window_adjudication_user(window, question, response, len(coverage)),
            }
            for window in coverage
        ]
        # The ordinary single-call payload is never used for a long context;
        # it is deliberately tiny so no truncated projection can be mistaken
        # for the evidence universe.
        evidence = "[FULL-COVERAGE WINDOW PASSES REQUIRED BEFORE AGGREGATION]"
        evidence_mode = "all_literal_windows_then_final_aggregation"
        full_context_inference_coverage = True

    signals = detected_policy_families(context, question, response)
    lexical_policy = contextual_exact_lexical_policy(question, response, lexical_records)
    aggregation_derivations = [
        value for value in list(route.get("bounded_derivation_candidates") or [])[:8]
        if isinstance(value, dict) and value.get("source_linked") is True
    ]
    operand_ids = {
        str(note_id)
        for value in aggregation_derivations
        for note_id in value.get("operand_note_ids", [])
    }
    routed_notes = list(route.get("selected_notes") or [])
    routed_notes.sort(key=lambda note: (str(note.get("note_id")) not in operand_ids, int(note.get("context_char_start", 0))))
    aggregation_notes = []
    note_budget = 2400
    used_note_chars = 0
    for note in routed_notes[:32]:
        start = int(note["context_char_start"])
        end = int(note["context_char_end"])
        literal = str(note["literal_text"])
        if context[start:end] != literal or _sha(literal) != note["literal_span_sha256"]:
            raise ValueError("router selected note does not replay against supplied context")
        serialized_chars = len(_canonical(note))
        if aggregation_notes and used_note_chars + serialized_chars > note_budget:
            continue
        aggregation_notes.append(note)
        used_note_chars += serialized_chars
    notes = {
        "selected_notes": aggregation_notes,
        "bounded_derivation_candidates": aggregation_derivations,
    }
    contract = {
        "version": VERSION,
        "evidence_universe": "only_the_literal_supplied_context",
        "external_retrieval_allowed": False,
        "outside_world_fact_rescue_allowed": False,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "every_context_character_processed": True,
        "detected_policy_families": signals,
        "checks_in_order": [
            "validate question premise and whether the requested slot is answerable",
            "bind exact entity relation property answer type event phase time polarity comparator and unit",
            "seek direct support and direct contradiction for that exact slot",
            "apply only supplied definition theory rule formula and bounded operands",
            "separate refuted from missing ambiguous conflicting or locally silent window evidence",
            "compare Unicode only through bounded NFC digit joiner and attested word-break views; near-looking forms are not automatically equal",
        ],
        "verdict_rule": {
            "supported": "the complete response follows for the exact requested slot",
            "refuted": "the supplied context establishes an incompatible value or claim",
            "not_enough_information": "required evidence is absent omitted ambiguous conflicting or premise-invalid",
        },
        "fixed_lexical_exception": lexical_policy,
    }
    user = (
        "CONTEXT-ONLY VERIFICATION CONTRACT:\n" + _canonical(contract)
        + "\n\nQUESTION (literal):\n" + question
        + "\n\nSUPPLIED CONTEXT EVIDENCE (literal; this is the only evidence universe):\n" + evidence
        + "\n\nSOURCE-LINKED EXTRACTIVE NOTES (advisory, never new evidence):\n" + _canonical(notes)
        + "\n\nCANDIDATE RESPONSE (verify every material claim):\n" + response
    )
    diagnostic = {
        **route,
        "context_policy_version": VERSION,
        "policy_family_inventory_count": len(CANONICAL_POLICY_FAMILIES),
        "canonical_policy_family_count": len(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cell_count": len(ENGINEERED_EVALUATION_CELLS),
        "operation_axis_count": len(OPERATION_AXIS),
        "canonical_policy_families": list(CANONICAL_POLICY_FAMILIES),
        "engineered_evaluation_cells": list(ENGINEERED_EVALUATION_CELLS),
        "operation_axis": list(OPERATION_AXIS),
        "detected_policy_families": signals,
        "evidence_mode": evidence_mode,
        "full_context_inference_coverage": full_context_inference_coverage,
        "context_literal": unicode_receipt(context),
        "question_literal": unicode_receipt(question),
        "response_literal": unicode_receipt(response),
        "prompt_sha256": _sha(user),
        "requires_window_aggregation": bool(window_calls),
        "window_calls": window_calls,
        "window_count": len(window_calls) if window_calls else 1,
        "full_context_character_count": len(context),
        "aggregation_selected_notes": aggregation_notes,
        "aggregation_bounded_derivations": aggregation_derivations,
        "contextual_lexical_policy": lexical_policy,
    }
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "build_aggregation_user",
    "build_contextual_policy_packet",
]


In [5]:
"""Deterministic Bengali grammar entailment for contextual benchmark rows.

The context sometimes supplies a ব্যাসবাক্য or a সন্ধি rule rather than the
literal answer.  Substring grounding therefore fails even though the answer is
fully derivable.  This module implements only high-precision, auditable
transformations; unknown forms return ``None`` instead of guessing.
"""
from __future__ import annotations

import re
import unicodedata


def norm(text: object) -> str:
    value = unicodedata.normalize("NFC", str(text or ""))
    value = re.sub(r"[‌‍]", "", value)
    return re.sub(r"\s+", " ", value).strip().lower()


def compact(text: object) -> str:
    return re.sub(r"[^ঀ-৿a-z0-9]+", "", norm(text))


def grammar_forms_equal(left: object, right: object) -> bool:
    """Compare Bengali grammar forms after NFC and ignorable-format cleanup."""

    return bool(compact(left)) and compact(left) == compact(right)


def grammar_prompt_operation(prompt: str) -> str | None:
    """Return the bounded Bengali grammar operation requested by ``prompt``."""

    p = norm(prompt)
    if "ব্যাসবাক্য" in p and "সমাস" in p:
        return "samas_taxonomy"
    if "সন্ধি" in p and "কোন শব্দ" in p:
        return "sandhi_formation"
    return None


def parse_samas_term(prompt: str) -> str | None:
    """Extract the quoted সমাস term whose taxonomy is requested."""

    p = norm(prompt)
    match = re.search(r"['‘’\"]([^'‘’\"]+)['‘’\"]\s*শব্দটি", p)
    return match.group(1).strip() if match else None


def context_supports_grammar_operation(context: str, prompt: str) -> bool:
    """Require the passage to supply the same grammar task and its operands.

    Grammar truth alone cannot rescue an unrelated contextual row.  A samas
    question needs the matching term's ব্যাসবাক্য; a sandhi question needs the
    matching roots and a sandhi operation.  This mirrors lexical/idiom lookup:
    the context establishes the lookup scope, then deterministic knowledge
    adjudicates the answer.
    """

    operation = grammar_prompt_operation(prompt)
    passage = norm(context)
    if operation == "samas_taxonomy":
        term = parse_samas_term(prompt)
        return bool(
            term
            and "ব্যাসবাক্য" in passage
            and compact(term) in compact(passage)
        )
    if operation == "sandhi_formation":
        roots = parse_sandhi_roots(prompt)
        return bool(
            roots
            and "সন্ধি" in passage
            and all(compact(root) in compact(passage) for root in roots)
        )
    return False


def infer_samas(context: str) -> str | None:
    """Infer a সমাস class from a supplied ব্যাসবাক্য."""
    c = norm(context)
    if "ব্যাসবাক্য" not in c:
        return None
    expansion = c.split("হলো:", 1)[-1].strip(" .'\"")

    # Order matters: specific semantic constructions precede broad ones.
    if "পরস্পর" in expansion and "দ্বারা" in expansion:
        return "ব্যতিহার বহুব্রীহি"
    if re.search(r"(?:নাই|নেই)\s+.+\s+যার", expansion):
        return "নঞ বহুব্রীহি"
    if re.search(r"(?:না|নয়|নয়)\s+", expansion):
        return "নঞ তৎপুরুষ"
    if "সমাহার" in expansion and re.search(
        r"পাঁচ|তিন|চার|সাত|দুই|ছয়|ছয়|আট|নয়|নয়|দশ", expansion
    ):
        return "দ্বিগু"
    if re.search(r"\sও\s", expansion):
        return "দ্বন্দ্ব"
    if "যার" in expansion or "যাহার" in expansion or "আছে যার" in expansion:
        return "বহুব্রীহি"
    if "হইতে" in expansion or "থেকে" in expansion:
        return "পঞ্চমী তৎপুরুষ"
    if "দ্বারা" in expansion:
        return "তৃতীয়া তৎপুরুষ"
    if "জন্য" in expansion:
        return "চতুর্থী তৎপুরুষ"
    if re.search(r"\sযে\s", expansion):
        return "কর্মধারয়"
    if re.search(r"\S+(?:তে|য়ে|য়ে|ে)\s+\S+", expansion) and not re.search(r"\sযে\s", expansion):
        return "সপ্তমী তৎপুরুষ"
    if re.search(r"(?:ের|র)\s+\S+", expansion) and "ন্যায়" not in expansion and "ন্যায়" not in expansion:
        return "ষষ্ঠী তৎপুরুষ"
    if "চিহ্নিত" in expansion:
        return "মধ্যপদলোপী কর্মধারয়"
    if re.search(r"\S+-ই\s+\S+", expansion):
        return "রূপক কর্মধারয়"
    if ("ন্যায়" in expansion or "ন্যায়" in expansion) and "যার" not in expansion:
        # In উপমান কর্মধারয় the first term is the comparison object:
        # 'কুসুমের ন্যায় কোমল'.  'মুখ চন্দ্রের ন্যায়' is the reverse.
        if re.fullmatch(r"(?:\S+ের|\S+র)\s+(?:ন্যায়|ন্যায়)\s+\S+[।]?", expansion):
            return "উপমান কর্মধারয়"
        return "উপমিত কর্মধারয়"
    if "অনুযায়ী" in expansion or "অনুযায়ী" in expansion or "দিন দিন" in expansion:
        return "অব্যয়ীভাব"
    if "ব্যাপিয়া" in expansion or "ব্যাপ্ত" in expansion:
        return "অব্যয়ীভাব"
    return None


def _implicit_a_plus(left: str, right: str, mark: str) -> str:
    return left + mark + right[1:]


def compose_sandhi(left: str, right: str) -> str | None:
    """Compose supported স্বর/ব্যঞ্জন/বিসর্গ সন্ধি forms."""
    left, right = norm(left), norm(right)
    if not left or not right:
        return None

    # বিসর্গ সন্ধি
    if left == "নিঃ":
        if right.startswith("ক"):
            return "নিষ্" + right
        if right.startswith("চ"):
            return "নিশ্" + right
        if right.startswith("র"):
            return "নী" + right
        if right.startswith(("জ", "ম")):
            return "নির্" + right
    if left == "দুঃ":
        if right.startswith("গ"):
            return "দুর্" + right
        if right.startswith("খ"):
            return left + right
    if left == "অন্তঃ" and right.startswith("ক"):
        return left + right
    if left == "পুনঃ" and right.startswith(("জ", "ম")):
        return "পুনর্" + right

    # Common ব্যঞ্জন সন্ধি with final ৎ.
    if left.endswith("ৎ"):
        joins = {
            "জ": "জ্জ", "ল": "ল্ল", "ম": "ন্ম", "ন": "ন্ন",
            "চ": "চ্চ", "ঘ": "দ্ঘ", "গ": "দ্গ", "ভ": "দ্ভ", "ধ": "দ্ধ",
        }
        initial = right[0]
        if initial in joins:
            return left[:-1] + joins[initial] + right[1:]

    # স্বর সন্ধি. A consonant-final Bengali word carries an implicit অ.
    if right.startswith(("আ", "অ")):
        if left.endswith("া"):
            return left + right[1:]
        return _implicit_a_plus(left, right, "া")
    if right.startswith(("ই", "ঈ")):
        if left.endswith(("ি", "ী")):
            return left[:-1] + "ী" + right[1:]
        if left.endswith("া"):
            return left[:-1] + "ে" + right[1:]
        return _implicit_a_plus(left, right, "ে")
    if right.startswith(("উ", "ঊ")):
        if left.endswith("া"):
            return left[:-1] + "ো" + right[1:]
        return _implicit_a_plus(left, right, "ো")
    if right.startswith("ঋ") and left.endswith("া"):
        return left[:-1] + "র্" + right[1:]
    if right.startswith("ঔ") and left.endswith("া"):
        return left[:-1] + "ৌ" + right[1:]
    if right.startswith("এ") and left.endswith("ি"):
        return left[:-1] + "্য" + right
    return None


def parse_sandhi_roots(prompt: str) -> tuple[str, str] | None:
    p = norm(prompt)
    match = re.search(r"([^'‘’\"]+)['‘’\"]\s+ও\s+['‘’\"]([^'‘’\"]+)", p)
    if not match:
        return None
    left = match.group(1).split()[-1]
    right = match.group(2).strip()
    return left, right


def grammar_label(context: str, prompt: str, response: str) -> int | None:
    """Return 0/1 for supported indirect grammar questions, else ``None``."""
    p = norm(prompt)
    answer = compact(response)
    if "সমাস শব্দটির অর্থ" in p:
        supported = {
            "সংক্ষেপ", "সমর্থন", "সংগ্রহ", "মিলন", "একাধিকপদেরএকপদীকরণ",
            "সংযোগ", "সংকলন", "সমন্বয়", "সমন্বয়", "সংমিশ্রণ",
        }
        return int(any(compact(term) in answer for term in supported))
    if "ব্যাসবাক্য" in p and "সমাস" in p:
        expected = infer_samas(context)
        return None if expected is None else int(grammar_forms_equal(expected, answer))
    if "সন্ধি" in p and "কোন শব্দ" in p:
        roots = parse_sandhi_roots(prompt)
        if roots is None:
            return None
        expected = compose_sandhi(*roots)
        return None if expected is None else int(grammar_forms_equal(expected, answer))
    return None


In [6]:
"""Authoritative v2 typed-policy runtime for contextual verification.

The contextual lane never performs generic corpus retrieval.  The only narrow
exceptions in the frozen v2 policy are three fixed lexical shells (antonym,
idiom meaning and prefix origin) plus deterministic, context-scoped samas and
sandhi.  Every exception requires one uniquely recognized operation and a
literal binding between the supplied context and the requested term/operands.

The legacy proxy functions remain as fail-closed compatibility shims.  They no
longer create closed-book retrieval queries for contextual rows.

This file can be imported normally, or executed immediately after
``contextual_policy_v4_runtime.py`` in one self-contained Kaggle notebook.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from pathlib import Path
from types import MappingProxyType
from typing import Any, Callable, Mapping


# The generated notebook executes the complete v4 runtime first.  Normal
# package imports take the except branch instead.  This avoids relying on an
# undeclared Kaggle input module while retaining one tested implementation of
# full-context windowing and aggregation.
try:  # pragma: no branch - the branch is environment-dependent by design
    _V4_VERSION = VERSION  # type: ignore[name-defined]
    _V4_CANONICAL_POLICY_FAMILIES = CANONICAL_POLICY_FAMILIES  # type: ignore[name-defined]
    _V4_ENGINEERED_EVALUATION_CELLS = ENGINEERED_EVALUATION_CELLS  # type: ignore[name-defined]
    _V4_OPERATION_AXIS = OPERATION_AXIS  # type: ignore[name-defined]
    _v4_build_aggregation_user = build_aggregation_user  # type: ignore[name-defined]
    _v4_build_contextual_policy_packet = build_contextual_policy_packet  # type: ignore[name-defined]
    comparison_view = comparison_view  # type: ignore[name-defined,self-assigning-variable]
    unicode_receipt = unicode_receipt  # type: ignore[name-defined,self-assigning-variable]
    detected_policy_families = detected_policy_families  # type: ignore[name-defined,self-assigning-variable]
    full_coverage_windows = full_coverage_windows  # type: ignore[name-defined,self-assigning-variable]
    build_window_adjudication_user = build_window_adjudication_user  # type: ignore[name-defined,self-assigning-variable]
except NameError:
    from pipeline.contextual_policy_v4_runtime import (
        CANONICAL_POLICY_FAMILIES as _V4_CANONICAL_POLICY_FAMILIES,
        ENGINEERED_EVALUATION_CELLS as _V4_ENGINEERED_EVALUATION_CELLS,
        OPERATION_AXIS as _V4_OPERATION_AXIS,
        VERSION as _V4_VERSION,
        build_aggregation_user as _v4_build_aggregation_user,
        build_contextual_policy_packet as _v4_build_contextual_policy_packet,
        build_window_adjudication_user,
        comparison_view,
        detected_policy_families,
        full_coverage_windows,
        unicode_receipt,
    )


# A generated notebook may execute ``bengali_grammar.py`` immediately before
# this source.  Normal package imports use the root module.  If neither is
# available, a hash-pinned narrow mirror keeps the notebook self-contained.
try:  # pragma: no branch - depends on the notebook assembly strategy
    _grammar_context_gate = context_supports_grammar_operation  # type: ignore[name-defined]
    _grammar_label = grammar_label  # type: ignore[name-defined]
    _grammar_prompt_operation = grammar_prompt_operation  # type: ignore[name-defined]
    _parse_samas_term = parse_samas_term  # type: ignore[name-defined]
    _parse_sandhi_roots = parse_sandhi_roots  # type: ignore[name-defined]
    _GRAMMAR_HELPERS_AVAILABLE = True
except NameError:
    try:
        from bengali_grammar import (
            context_supports_grammar_operation as _grammar_context_gate,
            grammar_label as _grammar_label,
            grammar_prompt_operation as _grammar_prompt_operation,
            parse_samas_term as _parse_samas_term,
            parse_sandhi_roots as _parse_sandhi_roots,
        )
        _GRAMMAR_HELPERS_AVAILABLE = True
    except ImportError:  # pragma: no cover - exercised only by a malformed bundle
        # Self-contained mirror of the narrow, tested operations from the
        # hash-pinned root ``bengali_grammar.py``.  This branch is required by
        # Kaggle notebook cells, where the repository module is not mounted.
        def _bg_norm(text: object) -> str:
            value = unicodedata.normalize("NFC", str(text or ""))
            value = re.sub(r"[‌‍]", "", value)
            return re.sub(r"\s+", " ", value).strip().lower()

        def _bg_compact(text: object) -> str:
            return re.sub(r"[^ঀ-৿a-z0-9]+", "", _bg_norm(text))

        def _bg_forms_equal(left: object, right: object) -> bool:
            return bool(_bg_compact(left)) and _bg_compact(left) == _bg_compact(right)

        def _bg_prompt_operation(prompt: str) -> str | None:
            value = _bg_norm(prompt)
            if "ব্যাসবাক্য" in value and "সমাস" in value:
                return "samas_taxonomy"
            if "সন্ধি" in value and "কোন শব্দ" in value:
                return "sandhi_formation"
            return None

        def _bg_parse_samas_term(prompt: str) -> str | None:
            match = re.search(
                r"['‘’\"]([^'‘’\"]+)['‘’\"]\s*শব্দটি", _bg_norm(prompt)
            )
            return match.group(1).strip() if match else None

        def _bg_parse_sandhi_roots(prompt: str) -> tuple[str, str] | None:
            match = re.search(
                r"([^'‘’\"]+)['‘’\"]\s+ও\s+['‘’\"]([^'‘’\"]+)",
                _bg_norm(prompt),
            )
            if not match:
                return None
            return match.group(1).split()[-1], match.group(2).strip()

        def _bg_context_gate(context: str, prompt: str) -> bool:
            operation = _bg_prompt_operation(prompt)
            passage = _bg_norm(context)
            if operation == "samas_taxonomy":
                term = _bg_parse_samas_term(prompt)
                return bool(
                    term and "ব্যাসবাক্য" in passage
                    and _bg_compact(term) in _bg_compact(passage)
                )
            if operation == "sandhi_formation":
                roots = _bg_parse_sandhi_roots(prompt)
                return bool(
                    roots and "সন্ধি" in passage
                    and all(_bg_compact(root) in _bg_compact(passage) for root in roots)
                )
            return False

        def _bg_infer_samas(context: str) -> str | None:
            value = _bg_norm(context)
            if "ব্যাসবাক্য" not in value:
                return None
            expansion = value.split("হলো:", 1)[-1].strip(" .'\"")
            if "পরস্পর" in expansion and "দ্বারা" in expansion:
                return "ব্যতিহার বহুব্রীহি"
            if re.search(r"(?:নাই|নেই)\s+.+\s+যার", expansion):
                return "নঞ বহুব্রীহি"
            if re.search(r"(?:না|নয়|নয়)\s+", expansion):
                return "নঞ তৎপুরুষ"
            if "সমাহার" in expansion and re.search(
                r"পাঁচ|তিন|চার|সাত|দুই|ছয়|ছয়|আট|নয়|নয়|দশ", expansion
            ):
                return "দ্বিগু"
            if re.search(r"\sও\s", expansion):
                return "দ্বন্দ্ব"
            if "যার" in expansion or "যাহার" in expansion or "আছে যার" in expansion:
                return "বহুব্রীহি"
            if "হইতে" in expansion or "থেকে" in expansion:
                return "পঞ্চমী তৎপুরুষ"
            if "দ্বারা" in expansion:
                return "তৃতীয়া তৎপুরুষ"
            if "জন্য" in expansion:
                return "চতুর্থী তৎপুরুষ"
            if re.search(r"\sযে\s", expansion):
                return "কর্মধারয়"
            if re.search(r"\S+(?:তে|য়ে|য়ে|ে)\s+\S+", expansion) and not re.search(
                r"\sযে\s", expansion
            ):
                return "সপ্তমী তৎপুরুষ"
            if re.search(r"(?:ের|র)\s+\S+", expansion) and not any(
                token in expansion for token in ("ন্যায়", "ন্যায়")
            ):
                return "ষষ্ঠী তৎপুরুষ"
            if "চিহ্নিত" in expansion:
                return "মধ্যপদলোপী কর্মধারয়"
            if re.search(r"\S+-ই\s+\S+", expansion):
                return "রূপক কর্মধারয়"
            if ("ন্যায়" in expansion or "ন্যায়" in expansion) and "যার" not in expansion:
                if re.fullmatch(
                    r"(?:\S+ের|\S+র)\s+(?:ন্যায়|ন্যায়)\s+\S+[।]?", expansion
                ):
                    return "উপমান কর্মধারয়"
                return "উপমিত কর্মধারয়"
            if any(token in expansion for token in ("অনুযায়ী", "অনুযায়ী", "দিন দিন")):
                return "অব্যয়ীভাব"
            if "ব্যাপিয়া" in expansion or "ব্যাপ্ত" in expansion:
                return "অব্যয়ীভাব"
            return None

        def _bg_compose_sandhi(left: str, right: str) -> str | None:
            left, right = _bg_norm(left), _bg_norm(right)
            if not left or not right:
                return None
            if left == "নিঃ":
                if right.startswith("ক"): return "নিষ্" + right
                if right.startswith("চ"): return "নিশ্" + right
                if right.startswith("র"): return "নী" + right
                if right.startswith(("জ", "ম")): return "নির্" + right
            if left == "দুঃ":
                if right.startswith("গ"): return "দুর্" + right
                if right.startswith("খ"): return left + right
            if left == "অন্তঃ" and right.startswith("ক"): return left + right
            if left == "পুনঃ" and right.startswith(("জ", "ম")): return "পুনর্" + right
            if left.endswith("ৎ"):
                joins = {
                    "জ": "জ্জ", "ল": "ল্ল", "ম": "ন্ম", "ন": "ন্ন",
                    "চ": "চ্চ", "ঘ": "দ্ঘ", "গ": "দ্গ", "ভ": "দ্ভ", "ধ": "দ্ধ",
                }
                if right[0] in joins:
                    return left[:-1] + joins[right[0]] + right[1:]
            if right.startswith(("আ", "অ")):
                return left + right[1:] if left.endswith("া") else left + "া" + right[1:]
            if right.startswith(("ই", "ঈ")):
                if left.endswith(("ি", "ী")): return left[:-1] + "ী" + right[1:]
                if left.endswith("া"): return left[:-1] + "ে" + right[1:]
                return left + "ে" + right[1:]
            if right.startswith(("উ", "ঊ")):
                return left[:-1] + "ো" + right[1:] if left.endswith("া") else left + "ো" + right[1:]
            if right.startswith("ঋ") and left.endswith("া"): return left[:-1] + "র্" + right[1:]
            if right.startswith("ঔ") and left.endswith("া"): return left[:-1] + "ৌ" + right[1:]
            if right.startswith("এ") and left.endswith("ি"): return left[:-1] + "্য" + right
            return None

        def _bg_label(context: str, prompt: str, response: str) -> int | None:
            value = _bg_norm(prompt)
            answer = _bg_compact(response)
            if "সমাস শব্দটির অর্থ" in value:
                supported = {
                    "সংক্ষেপ", "সমর্থন", "সংগ্রহ", "মিলন", "একাধিকপদেরএকপদীকরণ",
                    "সংযোগ", "সংকলন", "সমন্বয়", "সমন্বয়", "সংমিশ্রণ",
                }
                return int(any(_bg_compact(term) in answer for term in supported))
            if "ব্যাসবাক্য" in value and "সমাস" in value:
                expected = _bg_infer_samas(context)
                return None if expected is None else int(_bg_forms_equal(expected, answer))
            if "সন্ধি" in value and "কোন শব্দ" in value:
                roots = _bg_parse_sandhi_roots(prompt)
                expected = _bg_compose_sandhi(*roots) if roots else None
                return None if expected is None else int(_bg_forms_equal(expected, answer))
            return None

        _grammar_context_gate = _bg_context_gate
        _grammar_label = _bg_label
        _grammar_prompt_operation = _bg_prompt_operation
        _parse_samas_term = _bg_parse_samas_term
        _parse_sandhi_roots = _bg_parse_sandhi_roots
        _GRAMMAR_HELPERS_AVAILABLE = True
        _GRAMMAR_HELPER_RUNTIME = "inline_hash_pinned_narrow_mirror"
    else:
        _GRAMMAR_HELPER_RUNTIME = "imported_bengali_grammar_module"
else:
    _GRAMMAR_HELPER_RUNTIME = "preloaded_inline_bengali_grammar_module"

GRAMMAR_SOLVER_UPSTREAM_SHA256 = (
    "d6a01730a7d38d9250c112dfac2120e3c2e87939c2e25accd03b2563638510e8"
)


VERSION = "morichika-context-policy-v5-authoritative-v2-context-bound"
CANONICAL_POLICY_FAMILIES = tuple(_V4_CANONICAL_POLICY_FAMILIES)
ENGINEERED_EVALUATION_CELLS = tuple(_V4_ENGINEERED_EVALUATION_CELLS) + (
    "typed_grammar_exact_reference",
    "typed_grammar_wrong_operation_quarantine",
    "typed_grammar_retrieval_miss_nonrefutation",
)
TYPED_LOOKUP_OPERATIONS = (
    "antonym_lookup",
    "idiom_meaning_lookup",
    "prefix_origin_classification",
    "sandhi_formation",
    "samas_taxonomy",
)
LEXICAL_SHELL_OPERATIONS = TYPED_LOOKUP_OPERATIONS[:3]
DETERMINISTIC_GRAMMAR_OPERATIONS = TYPED_LOOKUP_OPERATIONS[3:]
OPERATION_AXIS = tuple(dict.fromkeys((*_V4_OPERATION_AXIS, *TYPED_LOOKUP_OPERATIONS)))

PROXY_PREFIX = "__morichika_typed_context_lookup_v5__"
SOURCE_PRIORITY_CLASSES = (
    "books_and_user_ocr_priority",
    "curated_dataset",
    "other_authorized_evidence",
    "community_encyclopedia_corroboration",
)
_SOURCE_PRIORITY_BY_ID = {
    "bangla_bcs_qs": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bangla_textbook_tigerllm": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bcs_canonical_archive": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bcs_downloads_10_50": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bcs_gk_ict_10_40": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bcs_public_archive": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bd_bcs_multimodal": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bengali_grammar_lexical_web_v1": (0, SOURCE_PRIORITY_CLASSES[0]),
    "broad_bn_grammar_census": (0, SOURCE_PRIORITY_CLASSES[0]),
    "downloads_bcs_10_50": (0, SOURCE_PRIORITY_CLASSES[0]),
    "joykoli_six_part": (0, SOURCE_PRIORITY_CLASSES[0]),
    "nctb_passage_qa_mendeley_v1": (0, SOURCE_PRIORITY_CLASSES[0]),
    "nctb_schooltext": (0, SOURCE_PRIORITY_CLASSES[0]),
    "bangla_mmlu": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_mmlu__bangla_mmlu_v1": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_math": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_med_qa": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_qa_84k": (1, SOURCE_PRIORITY_CLASSES[1]),
    "banglarqa": (1, SOURCE_PRIORITY_CLASSES[1]),
    "benqa": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bluck": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bnmmlu_134k": (1, SOURCE_PRIORITY_CLASSES[1]),
    "factoid_bangla_qa": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctb_education_aux": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctb_qa_87805": (1, SOURCE_PRIORITY_CLASSES[1]),
    "nctbench": (1, SOURCE_PRIORITY_CLASSES[1]),
    "openai_mmmlu_bn": (1, SOURCE_PRIORITY_CLASSES[1]),
    "squad_bn": (1, SOURCE_PRIORITY_CLASSES[1]),
    "uddipok_v3": (1, SOURCE_PRIORITY_CLASSES[1]),
    "lexical::bangla_bagdhara_cc_by_sa_4": (1, SOURCE_PRIORITY_CLASSES[1]),
    "bangla_bagdhara_cc_by_sa_4": (1, SOURCE_PRIORITY_CLASSES[1]),
    "lexical::bnwiktionary_cc_by_sa_4_20260701": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bnwiktionary_cc_by_sa_4_20260701": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bangla_wordnet": (2, SOURCE_PRIORITY_CLASSES[2]),
    "bengali_wikipedia": (3, SOURCE_PRIORITY_CLASSES[3]),
    "bengali_wikipedia_20210320": (3, SOURCE_PRIORITY_CLASSES[3]),
    "closed_residual_wikipedia_67": (3, SOURCE_PRIORITY_CLASSES[3]),
}


# Detection is question-only.  A grammar word elsewhere in a reused passage
# must never activate a typed policy, and two recognized shells are ambiguous.
_OPERATION_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("antonym_lookup", re.compile(
        r"বিপরীত(?:ার্থক)?\s*শব্দ|শুদ্ধ\s*বিপরীত|\S+\s*[-–—]?\s*এর\s*বিপরীত", re.I)),
    ("idiom_meaning_lookup", re.compile(
        r"বাগ\s*ধারা|বাগধারা|প্রবাদ(?:টির|টির)?\s*(?:অর্থ|মানে)|প্রবাদ\s*[-/]\s*প্রবচন", re.I)),
    ("prefix_origin_classification", re.compile(
        r"উপসর্গ(?:টি|টির)?\s*(?:কোন|কী|কি).{0,30}(?:শ্রেণি|উৎস|জাতীয়|জাতীয়|ভাষা)|"
        r"শব্দ(?:ে|টির|টিতে|ের).{0,32}উপসর্গ|উপসর্গ\s*যোগে", re.I)),
    ("sandhi_formation", re.compile(
        r"(?<![\u0980-\u09FF])সন্ধি(?:টি|টির|র|তে)?\s*(?:বিচ্ছেদ|কী|কি|কোন|কর|হয়|হয়|জাত|নির্ণ)|"
        r"(?<![\u0980-\u09FF])সন্ধিবিচ্ছেদ|(?:কোন|কী|কি)\s*(?:ধরনের\s*)?সন্ধি", re.I)),
    ("samas_taxonomy", re.compile(
        r"(?<![\u0980-\u09FF])সমাস(?:টি|টির|ের|ে)?\s*(?:কী|কি|কাকে|কোন|শব্দ|বদ্ধ|নির্ণ|উদাহরণ|অর্থ)|"
        r"(?:কোন|কী|কি)\s*(?:ধরনের\s*|প্রকারের\s*)?সমাস|ব্যাস\s*বাক্য", re.I)),
)

_TERM_BOUNDARY = r"[\u0980-\u09FFA-Za-z0-9]"
_LEXICAL_QUOTE = r"['\"‘’“”]"
_LEXICAL_QUOTED_BODY = r"[^'\"‘’“”\r\n]{1,160}"

# The lexical seed currently has only these three operations.  Each pattern
# extracts the requested operand from the operation shell; a record term being
# a boundary-delimited *subterm somewhere in the question* is not sufficient.
# This is deliberately fail-closed for ambiguous questions.
_LEXICAL_OPERAND_PATTERNS: dict[str, tuple[re.Pattern[str], ...]] = {
    "idiom_meaning_lookup": (
        re.compile(
            rf"{_LEXICAL_QUOTE}(?P<operand>{_LEXICAL_QUOTED_BODY})"
            rf"{_LEXICAL_QUOTE}\s*(?:প্রবাদ|বাগ\s*ধারা|বাগধারা)",
            re.I,
        ),
        re.compile(
            rf"{_LEXICAL_QUOTE}(?P<operand>{_LEXICAL_QUOTED_BODY})"
            rf"{_LEXICAL_QUOTE}\s*শব্দের\s*(?:অর্থ|মানে)",
            re.I,
        ),
        re.compile(
            r"^(?:এখানে\s+)?(?P<operand>[\u0980-\u09FFA-Za-z0-9][^\r\n]{0,158}?)"
            r"['’]?\s*(?:প্রবাদ|বাগ\s*ধারা|বাগধারা)",
            re.I,
        ),
    ),
    "antonym_lookup": (
        re.compile(
            rf"{_LEXICAL_QUOTE}(?P<operand>{_LEXICAL_QUOTED_BODY})"
            rf"{_LEXICAL_QUOTE}\s*শব্দের\s*(?:শুদ্ধ\s*)?বিপরীত",
            re.I,
        ),
        re.compile(
            r"^(?:এখানে\s+)?(?P<operand>[\u0980-\u09FFA-Za-z0-9][^\r\n]{0,78}?)"
            r"['’]?\s*শব্দের\s*(?:শুদ্ধ\s*)?বিপরীত",
            re.I,
        ),
    ),
    "prefix_origin_classification": (
        re.compile(
            rf"{_LEXICAL_QUOTE}(?P<operand>{_LEXICAL_QUOTED_BODY})"
            rf"{_LEXICAL_QUOTE}\s*উপসর্গ",
            re.I,
        ),
    ),
}


def _sha(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def _canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def detect_contextual_lookup_request(question: str) -> dict[str, Any]:
    """Recognize exactly one frozen-v2 operation from the literal question."""

    if not isinstance(question, str):
        raise TypeError("question must be a string")
    folded = comparison_view(question)
    matches = tuple(dict.fromkeys(
        name for name, pattern in _OPERATION_PATTERNS if pattern.search(folded)
    ))
    operation = matches[0] if len(matches) == 1 else None
    return {
        "applicable": len(matches) == 1,
        "operation": operation,
        "all_detected_operations": list(matches),
        "unique_operation_required": True,
        "ambiguous_operation": len(matches) > 1,
        "question_sha256": _sha(question),
        "detection_scope": "literal_question_only",
        "ordinary_context_external_lookup_allowed": False,
        "terminal_label_authority": False,
    }


def _proxy_id(example_id: str, question: str, operation: str) -> str:
    payload = _canonical({"example_id": example_id, "question": question, "operation": operation})
    return PROXY_PREFIX + _sha(payload)[:32]


def prepare_contextual_typed_lookup_input(
    rows: list[dict[str, Any]], output_path: Path
) -> tuple[Path, dict[str, dict[str, str]]]:
    """Write original rows unchanged and create no contextual proxies.

    This preserves the old call shape for notebook builders while enforcing
    the v2 prohibition on generic external corpus search in the contextual
    lane.  The returned proxy mapping is always empty.
    """

    output_path = Path(output_path)
    original_ids = {str(row.get("example_id", "")) for row in rows}
    if "" in original_ids or len(original_ids) != len(rows):
        raise ValueError("input rows require unique non-empty example_id values")
    combined = [dict(row) for row in rows]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary.write_text(
        "".join(_canonical(row) + "\n" for row in combined), encoding="utf-8"
    )
    temporary.replace(output_path)
    return output_path, {}


def attach_contextual_typed_lookup(
    retrieved: dict[str, dict[str, Any]],
    proxy_map: dict[str, dict[str, str]],
    rows: list[dict[str, Any]] | None = None,
) -> dict[str, dict[str, Any]]:
    """Compatibility no-op that rejects stale contextual proxy mappings."""

    if proxy_map:
        raise ValueError(
            "contextual typed retrieval proxies are forbidden by policy v2"
        )

    result = {
        str(key): dict(value)
        for key, value in retrieved.items()
        if str(key) not in proxy_map
    }
    if rows is not None:
        expected = {str(row["example_id"]) for row in rows}
        if set(result) != expected:
            raise ValueError("retrieval mapping does not exactly cover original rows")
        for row in rows:
            example_id = str(row["example_id"])
            if row.get("context_available") is True:
                result[example_id].pop("contextual_typed_lookup", None)
    return result


def _exact_term_in_question(term: object, folded_question: str) -> str | None:
    key = _normalize_lexical_operand(term)
    if not key:
        return None
    pattern = rf"(?<!{_TERM_BOUNDARY}){re.escape(key)}(?!{_TERM_BOUNDARY})"
    return key if re.search(pattern, folded_question) else None


def _normalize_lexical_operand(value: object) -> str:
    return comparison_view(str(value or "")).strip(
        " \t\r\n'\"‘’“”.,;:!?।–—-"
    )


def _literal_context_span(context: str, term: str) -> dict[str, Any] | None:
    """Bind a term to literal supplied-context bytes with token boundaries."""

    if not isinstance(context, str) or not term:
        return None
    pattern = rf"(?<!{_TERM_BOUNDARY}){re.escape(term)}(?!{_TERM_BOUNDARY})"
    match = re.search(pattern, context)
    if match is None:
        return None
    literal = context[match.start():match.end()]
    if literal != term:
        return None
    return {
        "term": term,
        "context_char_start": match.start(),
        "context_char_end": match.end(),
        "literal_utf8_sha256": hashlib.sha256(literal.encode("utf-8")).hexdigest(),
    }


def requested_lexical_operand_keys(operation: str, question: str) -> tuple[str, ...]:
    """Extract exact operation operands without admitting partial subterms.

    The result is intentionally empty when the shell does not identify one
    unambiguous lexical operand.  In particular, ``পড়া`` cannot satisfy a
    request for ``আকাশ থেকে পড়া`` merely because it is a complete token.
    """

    if not isinstance(question, str):
        raise TypeError("question must be a string")
    folded = comparison_view(question)
    found: list[str] = []
    for pattern in _LEXICAL_OPERAND_PATTERNS.get(operation, ()):
        for match in pattern.finditer(folded):
            operand = _normalize_lexical_operand(match.group("operand"))
            if operand.startswith("এখানে "):
                operand = operand[len("এখানে "):].strip()
            if operand and operand not in found:
                found.append(operand)
        if found:
            # Earlier patterns are more operation-specific.  Do not combine a
            # lower-confidence fallback with a precise quoted extraction.
            break
    return tuple(found)


def build_lexical_exact_index(
    records: list[dict[str, Any]],
) -> Mapping[tuple[str, str], tuple[Mapping[str, Any], ...]]:
    """Compile the lexical cache once into an immutable exact-key lookup."""

    buckets: dict[tuple[str, str], list[Mapping[str, Any]]] = {}
    for record in records:
        if not isinstance(record, dict):
            raise TypeError("lexical records must be dictionaries")
        operation = str(record.get("operation") or "")
        if not operation:
            raise ValueError("lexical record has no operation")
        frozen = MappingProxyType(dict(record))
        term_keys = {
            _normalize_lexical_operand(term)
            for term in [record.get("term_key"), *(record.get("display_terms") or [])]
        }
        for term_key in sorted(term_keys - {""}):
            buckets.setdefault((operation, term_key), []).append(frozen)
    return MappingProxyType({key: tuple(value) for key, value in buckets.items()})


def _response_relation(response: str, answers: list[Any], choices: list[Any]) -> str:
    response_key = comparison_view(response)
    answer_keys = {comparison_view(str(value)) for value in answers if str(value).strip()}
    choice_keys = {comparison_view(str(value)) for value in choices if str(value).strip()}
    if response_key and response_key in answer_keys:
        return "support_candidate"
    if response_key and response_key in choice_keys - answer_keys:
        return "counter_candidate"
    return "neutral_candidate"


def _lexical_exact_evidence(
    operation: str, context: str | None, question: str, response: str,
    records: (
        list[dict[str, Any]]
        | Mapping[tuple[str, str], tuple[Mapping[str, Any], ...]]
        | None
    ),
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    folded_question = comparison_view(question)
    folded_context = comparison_view(context or "")
    requested = requested_lexical_operand_keys(operation, question)
    binding: dict[str, Any] = {
        "required": context is not None,
        "literal_supplied_context": context is not None,
        "policy_scope": (
            "contextual_supplied_context" if context is not None
            else "closed_book_no_supplied_context"
        ),
        "requested_operands": list(requested),
        "unique_operand": len(requested) == 1,
        "bound": False,
        "span": None,
    }
    if len(requested) != 1:
        return [], binding
    requested_operands = {requested[0]}
    if context is None:
        span = None
        binding.update({"bound": True, "span": None})
    else:
        span = _literal_context_span(context, requested[0])
        if span is None:
            return [], binding
        binding.update({"bound": True, "span": span})
    if isinstance(records, Mapping):
        candidate_records: list[Mapping[str, Any]] = []
        seen_records: set[int] = set()
        for operand in sorted(requested_operands):
            for record in records.get((operation, operand), ()):
                identity = id(record)
                if identity not in seen_records:
                    seen_records.add(identity)
                    candidate_records.append(record)
    else:
        candidate_records = list(records or [])
    evidence: list[dict[str, Any]] = []
    for record in candidate_records:
        if str(record.get("operation") or "") != operation:
            continue
        terms = [record.get("term_key")] + list(record.get("display_terms") or [])
        keys = [
            key for term in terms
            if (key := _exact_term_in_question(term, folded_question))
            and key in requested_operands
        ]
        if not keys or str(record.get("conflict_status") or "none") != "none":
            continue
        scopes = [
            comparison_view(str(record.get(name) or ""))
            for name in ("sense", "register", "etymological_class")
            if str(record.get(name) or "").strip()
        ]
        if scopes and not all(
            scope in folded_question or scope in folded_context for scope in scopes
        ):
            continue
        accepted = list(record.get("accepted_answers") or [])
        contrasts = list(record.get("contrast_answers") or [])
        source_id = str(record.get("source_id") or "")
        source_priority, authority_class = _SOURCE_PRIORITY_BY_ID.get(
            source_id, (2, SOURCE_PRIORITY_CLASSES[2])
        )
        core = {
            "source_id": source_id,
            "source_record_index": int(record.get("source_record_index", -1)),
            "source_priority": source_priority,
            "authority_class": authority_class,
            "operation": operation,
            "exact_normalized": True,
            "key": sorted(set(keys), key=lambda value: (-len(value), value))[0],
            "context_binding": span,
            "answers": accepted,
            "choices": contrasts,
            "response_relation": _response_relation(response, accepted, contrasts),
            "alignment": "exact_operation_term_scope_and_answer_slot",
            "terminal_label_authority": context is None,
            "wikipedia_corroboration_only": False,
        }
        core["evidence_sha256"] = _sha(_canonical(core))
        evidence.append(core)
    return evidence, binding


def _grammar_context_binding(
    operation: str, context: str, question: str
) -> dict[str, Any]:
    """Run the existing grammar module's safe gate after literal binding."""

    result: dict[str, Any] = {
        "required": True,
        "literal_supplied_context": True,
        "grammar_helpers_available": _GRAMMAR_HELPERS_AVAILABLE,
        "operation": operation,
        "helper_operation": None,
        "bound": False,
        "operand_spans": [],
        "context_scope_gate": False,
    }
    if not _GRAMMAR_HELPERS_AVAILABLE:
        return result
    helper_operation = _grammar_prompt_operation(question)  # type: ignore[operator]
    result["helper_operation"] = helper_operation
    if helper_operation != operation:
        return result
    if operation == "samas_taxonomy":
        term = _parse_samas_term(question)  # type: ignore[operator]
        operands = (term,) if term else ()
    elif operation == "sandhi_formation":
        operands = _parse_sandhi_roots(question) or ()  # type: ignore[operator]
    else:
        return result
    if not operands:
        return result
    spans = [_literal_context_span(context, str(operand)) for operand in operands]
    if any(span is None for span in spans):
        return result
    context_gate = bool(_grammar_context_gate(context, question))  # type: ignore[operator]
    result.update({
        "operands": list(operands),
        "operand_spans": spans,
        "context_scope_gate": context_gate,
        "bound": context_gate,
    })
    return result


def _grammar_deterministic_evidence(
    operation: str, context: str, question: str, response: str
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    binding = _grammar_context_binding(operation, context, question)
    if binding["bound"] is not True:
        return [], binding
    label = _grammar_label(context, question, response)  # type: ignore[operator]
    if label not in (0, 1):
        return [], binding
    core: dict[str, Any] = {
        "operation": operation,
        "support_mechanism": "context_scoped_deterministic_grammar",
        "context_binding": binding,
        "deterministic_solver": "bengali_grammar_v1",
        "deterministic_solver_runtime": _GRAMMAR_HELPER_RUNTIME,
        "deterministic_solver_upstream_sha256": GRAMMAR_SOLVER_UPSTREAM_SHA256,
        "solver_label_candidate": int(label),
        "response_relation": "support_candidate" if label == 1 else "counter_candidate",
        # The solver candidate is exact, but terminal promotion remains
        # downstream until direct supplied-context contradiction precedence
        # proves that no incompatible explicit claim is present.
        "terminal_label_authority": False,
        "general_world_rescue_allowed": False,
    }
    core["evidence_sha256"] = _sha(_canonical(core))
    return [core], binding


def contextual_typed_lookup_policy(
    question: str,
    response: str,
    lexical_records: (
        list[dict[str, Any]]
        | Mapping[tuple[str, str], tuple[Mapping[str, Any], ...]]
        | None
    ) = None,
    typed_lookup_item: dict[str, Any] | None = None,
    *,
    limit: int = 6,
    context: str | None = None,
) -> dict[str, Any]:
    """Apply exactly one context-bound v2 lexical or grammar mechanism."""

    request = detect_contextual_lookup_request(question)
    supplied_context = context if isinstance(context, str) else ""
    policy_scope = (
        "contextual_supplied_context" if isinstance(context, str)
        else "closed_book_no_supplied_context"
    )
    stale_binding = (
        (typed_lookup_item or {}).get("contextual_typed_lookup")
        if isinstance(typed_lookup_item, dict) else None
    ) or (typed_lookup_item if isinstance(typed_lookup_item, dict) else {})
    forbidden_candidates = len(stale_binding.get("retrieval_candidates") or [])
    base: dict[str, Any] = {
        "version": VERSION,
        "lookup_mode": "forbidden_for_ordinary_context",
        "applicable": bool(request["applicable"]),
        "operation": request["operation"],
        "policy_scope": policy_scope,
        "all_detected_operations": request["all_detected_operations"],
        "unique_operation_required": True,
        "support_mechanism": None,
        "source_priority_order": list(SOURCE_PRIORITY_CLASSES),
        "exact_normalized_required": True,
        "operation_key_entity_slot_alignment_required": True,
        "literal_supplied_context_binding_required": (
            policy_scope == "contextual_supplied_context"
        ),
        "context_bound": False,
        "fuzzy_or_semantic_near_match_allowed": False,
        "generic_external_corpus_search_allowed": False,
        "strict_typed_retrieval_invoked": False,
        "general_world_rescue_allowed": False,
        "retrieval_miss_is_refutation": False,
        "terminal_label_authority": False,
        "terminal_label_candidate": None,
        "model_nonterminal": True,
        "conflict": False,
        "evidence": [],
        "retrieval_audit": {
            "examined": forbidden_candidates,
            "accepted": 0,
            "forbidden_generic_candidates_discarded": forbidden_candidates,
        },
    }
    if request["applicable"] is not True:
        return base
    operation = str(request["operation"])
    if operation in LEXICAL_SHELL_OPERATIONS:
        lexical, context_binding = _lexical_exact_evidence(
            operation, context if isinstance(context, str) else None,
            question, response, lexical_records
        )
        deduped = {str(item["evidence_sha256"]): item for item in lexical}
        ranked = sorted(
            deduped.values(),
            key=lambda value: (
                int(value["source_priority"]), str(value["source_id"]),
                int(value["source_record_index"]),
            ),
        )[:max(0, int(limit))]
        answer_sets = {
            tuple(sorted(
                comparison_view(str(answer))
                for answer in item.get("answers", []) if str(answer).strip()
            ))
            for item in ranked if item.get("answers")
        }
        response_roles = {
            str(item.get("response_relation")) for item in ranked
            if item.get("response_relation")
        }
        conflict = len(answer_sets) > 1 or {
            "support_candidate", "counter_candidate"
        } <= response_roles
        terminal_label = (
            1 if response_roles == {"support_candidate"} else
            0 if response_roles == {"counter_candidate"} else None
        )
        safe_evidence = [] if conflict else ranked
        if conflict:
            terminal_label = None
        terminal_authority = (
            terminal_label in (0, 1)
            and policy_scope == "closed_book_no_supplied_context"
        )
        base.update({
            "support_mechanism": "fixed_lexical_shell_exact_pair",
            "context_binding": context_binding,
            "context_bound": context_binding["bound"],
            "lookup_mode": (
                "context_bound_exact_lexical_conflict_abstain" if conflict else
                "closed_book_exact_lexical_terminal" if terminal_authority else
                "context_bound_exact_lexical_candidate_nonterminal"
                if terminal_label in (0, 1) else
                "context_bound_exact_lexical_nonterminal" if safe_evidence else
                "context_bound_exact_lexical_miss_nonterminal"
            ),
            "conflict": conflict,
            "conflict_evidence_sha256": (
                sorted(item["evidence_sha256"] for item in ranked) if conflict else []
            ),
            "evidence": safe_evidence,
            "terminal_label_authority": terminal_authority,
            "terminal_label_candidate": terminal_label,
            "model_nonterminal": not terminal_authority,
            "direct_context_conflict_absence_proven": (
                policy_scope == "closed_book_no_supplied_context"
            ),
            "retrieval_audit": {
                **base["retrieval_audit"],
                "lexical_exact_context_bound_accepted": len(lexical),
                "final_bounded_evidence": len(safe_evidence),
                "conflicting_evidence_suppressed": len(ranked) if conflict else 0,
            },
        })
    elif operation in DETERMINISTIC_GRAMMAR_OPERATIONS:
        grammar, context_binding = _grammar_deterministic_evidence(
            operation, supplied_context, question, response
        )
        terminal_label = (
            int(grammar[0]["solver_label_candidate"]) if len(grammar) == 1 else None
        )
        base.update({
            "support_mechanism": "context_scoped_deterministic_grammar",
            "context_binding": context_binding,
            "context_bound": context_binding["bound"],
            "lookup_mode": (
                "context_bound_deterministic_grammar_candidate_nonterminal"
                if terminal_label in (0, 1)
                else "context_bound_deterministic_grammar_miss_nonterminal"
            ),
            "evidence": grammar,
            "terminal_label_authority": False,
            "terminal_label_candidate": terminal_label,
            "model_nonterminal": True,
            "direct_context_conflict_absence_proven": False,
            "retrieval_audit": {
                **base["retrieval_audit"],
                "deterministic_grammar_evidence": len(grammar),
            },
        })
    return base


def build_contextual_policy_packet(
    context: str,
    question: str,
    response: str,
    router: Callable[..., dict[str, Any]],
    lexical_records: (
        list[dict[str, Any]]
        | Mapping[tuple[str, str], tuple[Mapping[str, Any], ...]]
        | None
    ) = None,
    typed_lookup_item: dict[str, Any] | None = None,
    *,
    max_windows: int = 8,
    full_context_char_limit: int = 6000,
    coverage_window_chars: int = 3000,
    coverage_overlap_chars: int = 320,
) -> tuple[str, dict[str, Any]]:
    """Build the v4 packet plus one context-bound v2 mechanism, if valid."""

    user, diagnostic = _v4_build_contextual_policy_packet(
        context, question, response, router, lexical_records=[],
        max_windows=max_windows, full_context_char_limit=full_context_char_limit,
        coverage_window_chars=coverage_window_chars,
        coverage_overlap_chars=coverage_overlap_chars,
    )
    policy = contextual_typed_lookup_policy(
        question, response, lexical_records, typed_lookup_item, context=context
    )
    if policy["applicable"] and policy["context_bound"]:
        user = user.replace(
            '"evidence_universe":"only_the_literal_supplied_context"',
            '"evidence_universe":"literal_context_plus_one_context_bound_v2_policy_mechanism"',
            1,
        ).replace(
            "this is the only evidence universe",
            "this plus the single context-bound v2 policy proof below is the only evidence universe",
            1,
        )
        user += (
            "\n\nCONTEXT-BOUND V2 POLICY PROOF:\n"
            "Exactly one operation was recognized and its term/operands were literally bound to the "
            "supplied context. No external corpus search was performed. The fixed exact lexical table "
            "or deterministic grammar solver cannot rescue a scope mismatch; conflict or miss means "
            "abstain/NEI, not refutation.\n"
            + _canonical(policy)
        )
    diagnostic["context_policy_version"] = VERSION
    diagnostic["parent_context_policy_version"] = _V4_VERSION
    diagnostic["operation_axis"] = list(OPERATION_AXIS)
    diagnostic["operation_axis_count"] = len(OPERATION_AXIS)
    diagnostic["typed_lookup_operation_inventory"] = list(TYPED_LOOKUP_OPERATIONS)
    diagnostic["typed_lookup_external_reference_allowed"] = False
    diagnostic["typed_policy_context_bound"] = bool(policy["context_bound"])
    diagnostic["typed_policy_support_mechanism"] = policy["support_mechanism"]
    diagnostic["typed_policy_terminal_label_candidate"] = policy["terminal_label_candidate"]
    diagnostic["ordinary_context_external_retrieval_allowed"] = False
    diagnostic["contextual_typed_lookup_policy"] = policy
    # The generated long-context aggregator already passes this field.  Point
    # it at the complete v5 typed policy rather than the v4 three-shell view.
    diagnostic["contextual_lexical_policy"] = policy
    diagnostic["prompt_sha256"] = _sha(user)
    diagnostic.pop("receipt_sha256", None)
    diagnostic["receipt_sha256"] = _sha(_canonical(diagnostic))
    return user, diagnostic


def build_aggregation_user(
    question: str,
    response: str,
    window_results: list[dict[str, Any]],
    *,
    selected_notes: list[dict[str, Any]] | None = None,
    bounded_derivations: list[dict[str, Any]] | None = None,
    lexical_policy: dict[str, Any] | None = None,
) -> str:
    """Keep windows context-only; merge only a context-bound v2 proof."""

    prompt = _v4_build_aggregation_user(
        question, response, window_results,
        selected_notes=selected_notes,
        bounded_derivations=bounded_derivations,
        lexical_policy=lexical_policy,
    )
    if (
        (lexical_policy or {}).get("applicable") is True
        and (lexical_policy or {}).get("context_bound") is True
    ):
        prompt = prompt.replace(
            "Use no outside knowledge.",
            "Use no outside retrieval or general world knowledge; only the context-bound v2 proof in this packet is admitted.",
            1,
        ).replace(
            "FROZEN EXACT LEXICAL-SHELL POLICY (nonterminal; ordinary retrieval forbidden)",
            "CONTEXT-BOUND V2 EXACT POLICY (generic retrieval forbidden)",
            1,
        )
    return prompt


__all__ = [
    "VERSION", "CANONICAL_POLICY_FAMILIES", "ENGINEERED_EVALUATION_CELLS",
    "OPERATION_AXIS", "TYPED_LOOKUP_OPERATIONS", "LEXICAL_SHELL_OPERATIONS",
    "DETERMINISTIC_GRAMMAR_OPERATIONS", "SOURCE_PRIORITY_CLASSES",
    "GRAMMAR_SOLVER_UPSTREAM_SHA256", "PROXY_PREFIX", "comparison_view", "unicode_receipt",
    "detected_policy_families", "full_coverage_windows",
    "build_window_adjudication_user", "detect_contextual_lookup_request",
    "requested_lexical_operand_keys", "build_lexical_exact_index",
    "prepare_contextual_typed_lookup_input", "attach_contextual_typed_lookup",
    "contextual_typed_lookup_policy", "build_contextual_policy_packet",
    "build_aggregation_user",
]


In [7]:
# Hash-bound OCR math interpretation lattice overlay.
import types as _morichika_math_types
_MORICHIKA_MATH_LATTICE_SOURCE = '"""Bounded, auditable interpretations for damaged mathematical notation.\n\nOCR surfaces such as ``x3`` are not repaired into a single preferred meaning:\nthey may denote ``x^3`` or ``x*3``; ``x3/2`` may additionally denote\n``x^(3/2)``. This module enumerates a small plausible lattice for both the\nquestion and supplied response. A deterministic decision is permitted only\nwhen every plausible pair is independently solvable and agrees.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport itertools\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable\n\n\nVERSION = "math-interpretation-lattice-v2-weighted-both-sides-consensus"\nMAX_VARIANTS_PER_FIELD = 12\nMAX_PAIR_INTERPRETATIONS = 64\n\nMATH_CONTEXT_RE = re.compile(\n    r"(?:[=+\\-–−×÷*/^√≤≥<>]|\\b[xyzab]\\s*\\d|সমীকরণ|অসমতা|মান\\s+কত|"\n    r"লসাগু|গসাগু|শতকরা|অনুপাত|ক্ষেত্রফল|পরিসীমা|ঘনফল)",\n    re.IGNORECASE,\n)\nFRACTIONAL_VARIABLE_RE = re.compile(\n    r"(?<![A-Za-z0-9])([xyzab])\\s*([23])\\s*/\\s*(\\d+)(?!\\d)",\n    re.IGNORECASE,\n)\nRADICAL_PREFIX_RE = re.compile(r"(?<![\\w.])([2-9])\\s*√")\nVARIABLE_DIGIT_RE = re.compile(\n    r"(?<![A-Za-z0-9])([xyzab])\\s*([23])(?=\\s*(?:[+\\-–−×÷*/=<>≤≥),]|এর\\b|$))",\n    re.IGNORECASE,\n)\nSUPERSCRIPT = {"2": "²", "3": "³"}\nOPERATION_PRIORS = {\n    # In the competition\'s damaged algebra typography, a variable followed by\n    # 2/3 at an operator boundary is most often a lost superscript.  This is a\n    # ranking prior only: every bounded interpretation remains in the lattice.\n    "variable_digit_as_power": 0.72,\n    "variable_digit_as_multiplication": 0.28,\n    "variable_digit_fraction_as_power_then_division": 0.46,\n    "variable_digit_fraction_as_fractional_exponent": 0.39,\n    "variable_digit_fraction_as_multiplication_then_division": 0.15,\n    "digit_before_radical_as_coefficient": 0.55,\n    "digit_before_radical_as_root_index": 0.45,\n}\n\n\ndef sha256_text(value: str) -> str:\n    return hashlib.sha256(value.encode("utf-8")).hexdigest()\n\n\ndef module_sha256() -> str:\n    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\n\ndef _dedupe(variants: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    output: dict[str, dict[str, Any]] = {}\n    for variant in variants:\n        output.setdefault(str(variant["text"]), variant)\n    return list(output.values())[:MAX_VARIANTS_PER_FIELD]\n\n\ndef _rank_variants(variants: list[dict[str, Any]]) -> list[dict[str, Any]]:\n    weighted: list[dict[str, Any]] = []\n    for variant in variants:\n        weight = 1.0\n        for operation in variant.get("operations", []):\n            weight *= OPERATION_PRIORS.get(str(operation), 1.0)\n        weighted.append({**variant, "unnormalized_prior": weight})\n    total = sum(float(item["unnormalized_prior"]) for item in weighted) or 1.0\n    weighted.sort(key=lambda item: (-float(item["unnormalized_prior"]), str(item["text"])))\n    return [\n        {\n            "id": f"v{index}",\n            "text": item["text"],\n            "operations": list(item.get("operations", [])),\n            "prior_weight": float(item["unnormalized_prior"]) / total,\n            "priority_rank": index,\n        }\n        for index, item in enumerate(weighted, 1)\n    ]\n\n\ndef _expand_pattern(\n    variants: list[dict[str, Any]],\n    pattern: re.Pattern[str],\n    replacement_builder: Callable[[re.Match[str]], list[tuple[str, str]]],\n) -> tuple[list[dict[str, Any]], bool]:\n    queue = list(variants)\n    completed: list[dict[str, Any]] = []\n    truncated = False\n    while queue:\n        current = queue.pop(0)\n        match = pattern.search(str(current["text"]))\n        if match is None:\n            completed.append(current)\n            continue\n        for replacement, operation in replacement_builder(match):\n            text = str(current["text"])\n            expanded = {\n                "text": text[: match.start()] + replacement + text[match.end() :],\n                "operations": [*current["operations"], operation],\n            }\n            if len(queue) + len(completed) < MAX_VARIANTS_PER_FIELD * 3:\n                queue.append(expanded)\n            else:\n                truncated = True\n        if len(completed) >= MAX_VARIANTS_PER_FIELD:\n            truncated = True\n            break\n    return _dedupe(completed or variants), truncated\n\n\ndef enumerate_text_interpretations(\n    value: object, *, force_math_context: bool = False\n) -> dict[str, Any]:\n    raw = "" if value is None else str(value)\n    if not force_math_context and not MATH_CONTEXT_RE.search(raw):\n        return {\n            "raw_sha256": sha256_text(raw),\n            "ambiguous": False,\n            "truncated": False,\n            "variants": [{\n                "id": "v1", "text": raw, "operations": [],\n                "prior_weight": 1.0, "priority_rank": 1,\n            }],\n        }\n\n    variants: list[dict[str, Any]] = [{"text": raw, "operations": []}]\n    truncated = False\n\n    def fraction_replacements(match: re.Match[str]) -> list[tuple[str, str]]:\n        variable, numerator, denominator = match.groups()\n        return [\n            (\n                f"({variable}{SUPERSCRIPT[numerator]})/{denominator}",\n                "variable_digit_fraction_as_power_then_division",\n            ),\n            (\n                f"({variable}×{numerator})/{denominator}",\n                "variable_digit_fraction_as_multiplication_then_division",\n            ),\n            (\n                f"{variable}^({numerator}/{denominator})",\n                "variable_digit_fraction_as_fractional_exponent",\n            ),\n        ]\n\n    variants, cut = _expand_pattern(\n        variants, FRACTIONAL_VARIABLE_RE, fraction_replacements\n    )\n    truncated |= cut\n\n    def radical_replacements(match: re.Match[str]) -> list[tuple[str, str]]:\n        digit = match.group(1)\n        return [\n            (f"{digit}×√", "digit_before_radical_as_coefficient"),\n            (f"√[index={digit}]", "digit_before_radical_as_root_index"),\n        ]\n\n    variants, cut = _expand_pattern(\n        variants, RADICAL_PREFIX_RE, radical_replacements\n    )\n    truncated |= cut\n\n    def variable_replacements(match: re.Match[str]) -> list[tuple[str, str]]:\n        variable, digit = match.groups()\n        return [\n            (f"{variable}{SUPERSCRIPT[digit]}", "variable_digit_as_power"),\n            (f"({variable}×{digit})", "variable_digit_as_multiplication"),\n        ]\n\n    variants, cut = _expand_pattern(\n        variants, VARIABLE_DIGIT_RE, variable_replacements\n    )\n    truncated |= cut\n    variants = _dedupe(variants)\n    ranked = _rank_variants(variants)\n    return {\n        "raw_sha256": sha256_text(raw),\n        "ambiguous": len(variants) > 1,\n        "truncated": truncated,\n        "variants": ranked,\n    }\n\n\ndef build_pair_lattice(prompt: object, response: object) -> dict[str, Any]:\n    raw_prompt = "" if prompt is None else str(prompt)\n    raw_response = "" if response is None else str(response)\n    math_context = bool(MATH_CONTEXT_RE.search(raw_prompt))\n    prompt_side = enumerate_text_interpretations(\n        raw_prompt, force_math_context=math_context\n    )\n    response_side = enumerate_text_interpretations(\n        raw_response, force_math_context=math_context\n    )\n    for prefix, side in (("p", prompt_side), ("r", response_side)):\n        for index, variant in enumerate(side["variants"], 1):\n            variant["id"] = f"{prefix}{index}"\n    pair_count = len(prompt_side["variants"]) * len(response_side["variants"])\n    return {\n        "version": VERSION,\n        "ambiguous": bool(prompt_side["ambiguous"] or response_side["ambiguous"]),\n        "math_context": math_context,\n        "truncated": bool(prompt_side["truncated"] or response_side["truncated"]),\n        "pair_count": pair_count,\n        "pair_limit_passed": pair_count <= MAX_PAIR_INTERPRETATIONS,\n        "prompt": prompt_side,\n        "response": response_side,\n        "decision_policy": (\n            "resolve only if every plausible prompt/response interpretation is "\n            "independently solvable and all binary relations agree"\n        ),\n        "ranking_policy": (\n            "lost superscript is weighted above multiplication for x2/x3 OCR, "\n            "but weights never delete variants or grant a terminal decision"\n        ),\n        "prior_weights_are_nonterminal": True,\n        "raw_fields_mutated": False,\n    }\n\n\ndef deterministic_consensus(\n    lattice: dict[str, Any],\n    resolver: Callable[[str, str], tuple[int, str] | None],\n) -> tuple[int, dict[str, Any]] | None:\n    if lattice.get("truncated") or not lattice.get("pair_limit_passed"):\n        return None\n    cases: list[dict[str, Any]] = []\n    for prompt_variant, response_variant in itertools.product(\n        lattice["prompt"]["variants"], lattice["response"]["variants"]\n    ):\n        result = resolver(str(prompt_variant["text"]), str(response_variant["text"]))\n        if result is None:\n            return None\n        relation, rule = int(result[0]), str(result[1])\n        if relation not in (0, 1):\n            return None\n        cases.append({\n            "prompt_variant_id": prompt_variant["id"],\n            "response_variant_id": response_variant["id"],\n            "relation": relation,\n            "rule": rule,\n        })\n    relations = {case["relation"] for case in cases}\n    if len(relations) != 1 or not cases:\n        return None\n    relation = relations.pop()\n    return relation, {\n        "method": "all_plausible_interpretations_independently_solved_consensus",\n        "case_count": len(cases),\n        "relation": relation,\n        "cases": cases,\n        "lattice_version": lattice.get("version"),\n    }\n'
if hashlib.sha256(_MORICHIKA_MATH_LATTICE_SOURCE.encode("utf-8")).hexdigest() != "6747124ca4f991584f955eaf2eb17b116045cb1fa9a433e6eb08b81e8a59813e":
    raise RuntimeError("embedded math interpretation lattice hash mismatch")
_MORICHIKA_MATH_LATTICE = _morichika_math_types.ModuleType(
    "morichika_math_interpretation_lattice_bound"
)
_MORICHIKA_MATH_LATTICE.__file__ = "<embedded-math-interpretation-lattice>"
exec(
    compile(
        _MORICHIKA_MATH_LATTICE_SOURCE,
        _MORICHIKA_MATH_LATTICE.__file__,
        "exec",
    ),
    _MORICHIKA_MATH_LATTICE.__dict__,
)
build_pair_lattice = _MORICHIKA_MATH_LATTICE.build_pair_lattice
MATH_INTERPRETATION_LATTICE_VERSION = _MORICHIKA_MATH_LATTICE.VERSION
MATH_INTERPRETATION_LATTICE_SHA256 = "6747124ca4f991584f955eaf2eb17b116045cb1fa9a433e6eb08b81e8a59813e"
del _MORICHIKA_MATH_LATTICE_SOURCE


In [8]:
# Hash-bound contextual router overlay.
import types as _morichika_types
_MORICHIKA_CONTEXT_ROUTER_SOURCE = '"""Deterministic context-only evidence router for long contexts.\n\nThis module is deliberately extractive.  It does not browse, retrieve, infer a\ngold label, or replace the supplied context with a summary.  It provides:\n\n* full-coverage character windows with a literal residual overlap;\n* byte-replayable, question-conditioned notes;\n* visible, source-linked date arithmetic and bounded kinship derivations; and\n* a fail-closed decision receipt tied to the exact requested slot.\n\nThe lightweight extractor and ranker are a deterministic primary front end for\nlong or structurally complex contextual rows, not a claim of complete Bengali\nsemantic parsing.  Missing extraction yields NEI or the ordinary full-window\nmodel path; it never licenses outside knowledge.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom dataclasses import dataclass, asdict\nfrom typing import Any, Iterable\n\n\nVERSION = "bichar-contextual-evidence-router-v2"\nNOTE_TYPES = {\n    "entity", "date_or_time", "event", "relation", "quantity", "unit",\n    "negation", "causal", "temporal", "rule_or_definition", "operand",\n    "conflict_marker",\n}\nVERDICTS = {"supported", "refuted", "not_enough_information"}\nOPERATORS = {"simple_year_difference", "relative_year_offset", "bounded_fathers_father"}\nFORBIDDEN_KEY_PARTS = ("retriev", "web_search", "external_evidence", "outside_source", "corpus_lookup")\n\n_BN_TO_ASCII = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")\n_DATE_RE = re.compile(\n    r"(?:[১২]?[০-৯]{1,3}|[12]?[0-9]{1,3})\\s*(?:সাল(?:ে)?|খ্রিষ্টাব্দ|খ্রি\\.?|বৈশাখ|জ্যৈষ্ঠ|আষাঢ়|শ্রাবণ|ভাদ্র|আশ্বিন|কার্তিক|অগ্রহায়ণ|পৌষ|মাঘ|ফাল্গুন|চৈত্র)"\n)\n_NUMBER_RE = re.compile(r"[+-]?[০-৯0-9]+(?:[.,][০-৯0-9]+)?")\n_UNIT_RE = re.compile(r"(?:কিলোমিটার|মিটার|সেন্টিমিটার|কিলোগ্রাম|গ্রাম|লিটার|সেকেন্ড|মিনিট|ঘণ্টা|দিন|বছর|শতাংশ|টাকা|জন|টি|বার)")\n_NEGATION_RE = re.compile(\n    r"(?:দেওয়া হয়নি|হয়নি|করেনি|অসম্ভব|(?<![\\u0980-\\u09FF])(?:না|নয়|নেই|নি)(?![\\u0980-\\u09FF]))"\n)\n_CAUSAL_RE = re.compile(r"(?:কারণ|ফলে|তাই|সুতরাং|যেহেতু|এজন্য)")\n_TEMPORAL_RE = re.compile(r"(?:আগে|পরে|পরবর্তীতে|প্রথমে|তারপর|শুরু|শেষ|জন্ম|বিবাহ|ঘোষণা|সমাপ্তি)")\n_RELATIVE_YEAR_RE = re.compile(\n    r"(?<![০-৯0-9,])(?P<offset>[০-৯0-9]{1,3})(?![০-৯0-9,])\\s*(?:বছর|বর্ষ|years?)\\s*"\n    r"(?P<direction>পরে|পর|আগে|পূর্বে|later|after|before)", re.IGNORECASE\n)\n_GRAMMAR_RE = re.compile(r"(?:সমাস|সন্ধি|বাগধারা|বিপরীত শব্দ|উপসর্গ|প্রত্যয়|গুরুচণ্ডালী|ণত্ব|ষত্ব|ব্যাকরণ|অর্থ|নিয়ম)")\n_EVENT_RE = re.compile(r"(?:ঘটে|ঘটেছিল|হয়|হয়েছিল|শুরু হয়|শেষ হয়|জন্মগ্রহণ|বিবাহ|ঘোষিত|অনুমোদিত)")\n_RELATION_RE = re.compile(\n    r"(?P<subject>[\\u0980-\\u09FF]+(?:\\s+[\\u0980-\\u09FF]+){0,3}?)ের\\s+"\n    r"(?P<relation>বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|স্রষ্টা|নির্মাতা|পরিচালক|জন্মস্থান|রাজধানী)\\s+"\n    r"(?P<object>[\\u0980-\\u09FF][\\u0980-\\u09FF\\s]{0,40}?)(?=[।.!?\\n]|$)"\n)\n_SENTENCE_RE = re.compile(r"[^।.!?\\n]+[।.!?]?|\\n")\n_TOKEN_RE = re.compile(r"[\\u0980-\\u09FF]+|[A-Za-z]+|[০-৯0-9]+")\n_QUESTION_STOPWORDS = {\n    "কি", "কী", "কে", "কার", "কোন", "কোনটি", "কত", "কবে", "কোথায়",\n    "কোথায়", "কেন", "কিভাবে", "কীভাবে", "হয়", "হয়", "ছিল", "আছে",\n    "এটি", "এই", "একটি", "উত্তর", "বলুন", "লিখুন", "what", "which",\n    "who", "when", "where", "why", "how", "is", "was", "the", "a",\n}\n\n\ndef _sha(text: str) -> str:\n    return hashlib.sha256(text.encode("utf-8")).hexdigest()\n\n\ndef _canonical(value: Any) -> str:\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))\n\n\n@dataclass(frozen=True)\nclass Window:\n    window_index: int\n    context_char_start: int\n    context_char_end: int\n    residual_from_previous_start: int | None\n    residual_from_previous_end: int | None\n    literal_text: str\n    literal_span_sha256: str\n\n\n@dataclass(frozen=True)\nclass Note:\n    note_id: str\n    type: str\n    literal_text: str\n    context_char_start: int\n    context_char_end: int\n    literal_span_sha256: str\n    normalized_comparison_form: str\n    window_indices: tuple[int, ...]\n    question_slot_ids: tuple[str, ...]\n    attributes: dict[str, Any]\n\n\ndef _fail_if_forbidden_keys(value: Any, path: str = "$") -> None:\n    if isinstance(value, dict):\n        for key, child in value.items():\n            folded = str(key).lower().replace("-", "_")\n            if folded == "external_retrieval_allowed" and child is False:\n                pass\n            elif any(token in folded for token in FORBIDDEN_KEY_PARTS):\n                raise ValueError(f"contextual lane forbids external/retrieval field {path}.{key}")\n            _fail_if_forbidden_keys(child, f"{path}.{key}")\n    elif isinstance(value, list):\n        for i, child in enumerate(value):\n            _fail_if_forbidden_keys(child, f"{path}[{i}]")\n\n\ndef make_windows(context: str, max_chars: int = 640, overlap_chars: int = 96) -> list[Window]:\n    """Create full-coverage windows with a literal, replayable residual tail."""\n    if not context:\n        raise ValueError("context must be non-empty")\n    if max_chars < 64 or overlap_chars < 1 or overlap_chars >= max_chars:\n        raise ValueError("require max_chars >= 64 and 0 < overlap_chars < max_chars")\n    windows: list[Window] = []\n    start = 0\n    while start < len(context):\n        hard_end = min(start + max_chars, len(context))\n        end = hard_end\n        if hard_end < len(context):\n            candidates = [context.rfind(mark, start + max_chars // 2, hard_end) for mark in ("\\n", "।", ".", "!", "?")]\n            boundary = max(candidates)\n            if boundary >= 0:\n                end = boundary + 1\n        if end <= start:\n            end = hard_end\n        residual_start = start if windows else None\n        residual_end = min(windows[-1].context_char_end, end) if windows else None\n        literal = context[start:end]\n        windows.append(Window(\n            window_index=len(windows), context_char_start=start, context_char_end=end,\n            residual_from_previous_start=residual_start,\n            residual_from_previous_end=residual_end,\n            literal_text=literal, literal_span_sha256=_sha(literal),\n        ))\n        if end == len(context):\n            break\n        start = max(0, end - overlap_chars)\n    validate_windows(context, windows, overlap_chars)\n    return windows\n\n\ndef validate_windows(context: str, windows: Iterable[Window], required_overlap: int) -> None:\n    windows = list(windows)\n    if not windows or windows[0].context_char_start != 0 or windows[-1].context_char_end != len(context):\n        raise ValueError("windows do not cover both context endpoints")\n    covered_until = 0\n    for i, w in enumerate(windows):\n        if context[w.context_char_start:w.context_char_end] != w.literal_text or _sha(w.literal_text) != w.literal_span_sha256:\n            raise ValueError(f"window {i} does not replay against supplied context")\n        if w.context_char_start > covered_until:\n            raise ValueError(f"coverage gap before window {i}")\n        if i:\n            overlap = windows[i - 1].context_char_end - w.context_char_start\n            expected = min(required_overlap, windows[i - 1].context_char_end)\n            if overlap < expected:\n                raise ValueError(f"residual overlap too small at window {i}: {overlap} < {expected}")\n        covered_until = max(covered_until, w.context_char_end)\n\n\ndef _window_indices(start: int, end: int, windows: list[Window]) -> tuple[int, ...]:\n    return tuple(w.window_index for w in windows if start < w.context_char_end and end > w.context_char_start)\n\n\ndef make_note(\n    context: str,\n    windows: list[Window],\n    note_type: str,\n    start: int,\n    end: int,\n    slot_ids: Iterable[str],\n    attributes: dict[str, Any] | None = None,\n) -> Note:\n    if note_type not in NOTE_TYPES or not (0 <= start < end <= len(context)):\n        raise ValueError("invalid note type or span")\n    literal = context[start:end]\n    slots = tuple(sorted(set(slot_ids)))\n    if not slots:\n        raise ValueError("every note must constrain at least one question slot")\n    attrs = dict(attributes or {})\n    _fail_if_forbidden_keys(attrs)\n    payload = {"type": note_type, "start": start, "end": end, "literal": literal, "slots": slots, "attributes": attrs}\n    return Note(\n        note_id="note:" + _sha(_canonical(payload))[:24], type=note_type,\n        literal_text=literal, context_char_start=start, context_char_end=end,\n        literal_span_sha256=_sha(literal), normalized_comparison_form=" ".join(literal.casefold().split()),\n        window_indices=_window_indices(start, end, windows), question_slot_ids=slots, attributes=attrs,\n    )\n\n\ndef extract_question_conditioned_notes(context: str, question: str, windows: list[Window]) -> list[Note]:\n    """Extract a conservative literal fact index; no outside knowledge is used."""\n    _fail_if_forbidden_keys({"context": context, "question": question})\n    slot = "requested_answer"\n    proposals: list[tuple[str, int, int, dict[str, Any]]] = []\n    patterns = (\n        ("date_or_time", _DATE_RE), ("quantity", _NUMBER_RE), ("unit", _UNIT_RE),\n        ("negation", _NEGATION_RE), ("causal", _CAUSAL_RE), ("temporal", _TEMPORAL_RE),\n    )\n    for note_type, pattern in patterns:\n        for match in pattern.finditer(context):\n            proposals.append((note_type, match.start(), match.end(), {}))\n    for match in _RELATIVE_YEAR_RE.finditer(context):\n        proposals.append(("temporal", match.start(), match.end(), {\n            "offset_years": int(match.group("offset").translate(_BN_TO_ASCII)),\n            "direction": match.group("direction").casefold(),\n        }))\n    for match in _RELATION_RE.finditer(context):\n        proposals.append(("relation", match.start(), match.end(), {\n            "subject": match.group("subject").strip(), "relation": match.group("relation").strip(),\n            "object": match.group("object").strip(),\n        }))\n        for group in ("subject", "object"):\n            proposals.append(("entity", match.start(group), match.end(group), {"relation_role": group}))\n    for sentence in _SENTENCE_RE.finditer(context):\n        text = sentence.group()\n        if _EVENT_RE.search(text):\n            proposals.append(("event", sentence.start(), sentence.end(), {}))\n        if _GRAMMAR_RE.search(text):\n            proposals.append(("rule_or_definition", sentence.start(), sentence.end(), {}))\n    notes: dict[tuple[str, int, int], Note] = {}\n    for note_type, start, end, attrs in proposals:\n        key = (note_type, start, end)\n        notes[key] = make_note(context, windows, note_type, start, end, [slot], attrs)\n    return sorted(notes.values(), key=lambda n: (n.context_char_start, n.context_char_end, n.type))\n\n\ndef _content_tokens(value: str) -> set[str]:\n    return {\n        token.casefold() for token in _TOKEN_RE.findall(value)\n        if len(token) > 1 and token.casefold() not in _QUESTION_STOPWORDS\n    }\n\n\ndef rank_context_windows(\n    context: str,\n    question: str,\n    response: str,\n    windows: list[Window] | None = None,\n    *,\n    max_windows: int = 6,\n) -> dict[str, Any]:\n    """Rank supplied-context windows without admitting outside evidence.\n\n    Question overlap dominates response overlap so a fluent proposed answer\n    cannot pull the router toward an unrelated same-passage fact.  Slot cues,\n    numbers, dates, negation, rules and explicit relations add bounded bonuses.\n    The selected set reserves neighboring windows and both response-aligned and\n    response-conflicting spans when they exist.  Every selected byte remains\n    replayable against the original context.\n    """\n\n    if max_windows < 1:\n        raise ValueError("max_windows must be positive")\n    windows = windows or make_windows(context)\n    validate_windows(context, windows, required_overlap=1)\n    q_tokens = _content_tokens(question)\n    r_tokens = _content_tokens(response)\n    question_numbers = set(_NUMBER_RE.findall(question))\n    response_numbers = set(_NUMBER_RE.findall(response))\n    query_needs_date = bool(_DATE_RE.search(question) or re.search(r"(?:কবে|তারিখ|সাল|বছর|বয়স|বয়স)", question))\n    query_needs_relation = bool(re.search(r"(?:বাবা|পিতা|মা|মাতা|দাদা|পিতামহ|স্রষ্টা|নির্মাতা|পরিচালক|জন্মস্থান|রাজধানী|কার|কে)", question))\n    query_needs_rule = bool(_GRAMMAR_RE.search(question))\n    query_has_negation = bool(_NEGATION_RE.search(question))\n\n    scored: list[dict[str, Any]] = []\n    for window in windows:\n        text = window.literal_text\n        tokens = _content_tokens(text)\n        q_overlap = sorted(q_tokens & tokens)\n        r_overlap = sorted(r_tokens & tokens)\n        number_tokens = set(_NUMBER_RE.findall(text))\n        slot_bonus = 0\n        slot_reasons: list[str] = []\n        if query_needs_date and (_DATE_RE.search(text) or number_tokens):\n            slot_bonus += 5\n            slot_reasons.append("date_or_time")\n        if query_needs_relation and _RELATION_RE.search(text):\n            slot_bonus += 5\n            slot_reasons.append("relation")\n        if query_needs_rule and _GRAMMAR_RE.search(text):\n            slot_bonus += 5\n            slot_reasons.append("rule_or_definition")\n        if query_has_negation and _NEGATION_RE.search(text):\n            slot_bonus += 4\n            slot_reasons.append("negation")\n        if question_numbers & number_tokens:\n            slot_bonus += 6\n            slot_reasons.append("question_number")\n        response_number_match = bool(response_numbers & number_tokens)\n        # A window can legitimately contain both the proposed value and a\n        # competing value.  Preserve both roles instead of collapsing it to\n        # support merely because one response number matched.\n        response_number_conflict = bool(response_numbers and (number_tokens - response_numbers))\n        if response_number_match:\n            slot_reasons.append("response_number_support_candidate")\n        if response_number_conflict:\n            slot_reasons.append("response_number_counter_candidate")\n        score = 8 * len(q_overlap) + 2 * len(r_overlap) + slot_bonus\n        scored.append({\n            "window_index": window.window_index,\n            "score": score,\n            "question_overlap": q_overlap,\n            "response_overlap": r_overlap,\n            "slot_reasons": slot_reasons,\n            "response_number_support_candidate": response_number_match,\n            "response_number_counter_candidate": response_number_conflict,\n        })\n\n    ranked = sorted(scored, key=lambda row: (-int(row["score"]), int(row["window_index"])))\n    chosen: set[int] = set()\n    for row in ranked:\n        if row["score"] <= 0 and chosen:\n            break\n        index = int(row["window_index"])\n        chosen.add(index)\n        # A single adjacent window preserves local antecedents and phase order.\n        for neighbor in (index - 1, index + 1):\n            if 0 <= neighbor < len(windows) and len(chosen) < max_windows:\n                chosen.add(neighbor)\n        if len(chosen) >= max_windows:\n            break\n    for role in ("response_number_support_candidate", "response_number_counter_candidate"):\n        candidate = next((row for row in ranked if row[role]), None)\n        if candidate is not None and len(chosen) < max_windows:\n            chosen.add(int(candidate["window_index"]))\n    if not chosen:\n        chosen.add(0)\n    selected = [windows[index] for index in sorted(chosen)[:max_windows]]\n    selected_indices = {window.window_index for window in selected}\n    notes = extract_question_conditioned_notes(context, question, windows)\n    selected_notes = [\n        note for note in notes if selected_indices.intersection(note.window_indices)\n    ][:32]\n    derivations: list[dict[str, Any]] = []\n    dates = [note for note in notes if note.type == "date_or_time"]\n    relatives = [\n        note for note in notes\n        if note.type == "temporal" and "offset_years" in note.attributes\n    ]\n    for relative in relatives:\n        containing_sentence = next(\n            (match for match in _SENTENCE_RE.finditer(context)\n             if match.start() <= relative.context_char_start < match.end()),\n            None,\n        )\n        # Only accept the unambiguous discourse form "N years later, EVENT"\n        # near the beginning of a question-relevant target sentence.  Mid-\n        # sentence durations such as "after 700 years the empire ended" do\n        # not identify the omitted origin year and must remain nonterminal.\n        if containing_sentence is None:\n            continue\n        relative_offset = relative.context_char_start - containing_sentence.start()\n        sentence_tokens = _content_tokens(containing_sentence.group())\n        if relative_offset > 6 or not (q_tokens & sentence_tokens):\n            continue\n        anchors = [date for date in dates if date.context_char_end <= relative.context_char_start]\n        if anchors:\n            derivations.append(relative_year_offset(anchors[-1], relative))\n    explicit_age = re.search(r"[০-৯0-9]+\\s*বছর\\s*(?:বয়সে|বয়সে)", context)\n    birth_marriage = re.search(r"(?:জন্ম|born)", context, re.IGNORECASE) and re.search(\n        r"(?:বিবাহ|বিয়ে|বিয়ে|marri)", context, re.IGNORECASE\n    )\n    if (\n        re.search(r"(?:বয়স|বয়স|কত বছর বয়সে|কত বছর বয়সে|age)", question, re.IGNORECASE)\n        and birth_marriage and not explicit_age and len(dates) == 2\n    ):\n        derivations.append(simple_year_difference(context, windows, dates[0], dates[1]))\n    receipt: dict[str, Any] = {\n        "version": VERSION,\n        "lane": "contextual_only",\n        "routing_mode": "primary_for_long_or_complex_context",\n        "external_retrieval_allowed": False,\n        "context_sha256": _sha(context),\n        "question_sha256": _sha(question),\n        "response_sha256": _sha(response),\n        "selected_windows": [asdict(window) for window in selected],\n        "selected_notes": [asdict(note) for note in selected_notes],\n        "bounded_derivation_candidates": derivations[:8],\n        "ranking": ranked,\n        "full_context_coverage_preserved_by_window_ledger": True,\n    }\n    receipt["receipt_sha256"] = _sha(_canonical(receipt))\n    return receipt\n\n\ndef simple_year_difference(context: str, windows: list[Window], earlier: Note, later: Note) -> dict[str, Any]:\n    if earlier.type != "date_or_time" or later.type != "date_or_time":\n        raise ValueError("year difference requires two date notes")\n    a = int(_NUMBER_RE.search(earlier.literal_text).group().translate(_BN_TO_ASCII))\n    b = int(_NUMBER_RE.search(later.literal_text).group().translate(_BN_TO_ASCII))\n    result = b - a\n    return {\n        "derivation_id": "derive:" + _sha(f"simple_year_difference|{earlier.note_id}|{later.note_id}|{result}")[:24],\n        "operator_id": "simple_year_difference", "operand_note_ids": [earlier.note_id, later.note_id],\n        "visible_expression": f"{b} - {a} = {result}", "result": {"type": "number", "value": result, "unit": "বছর"},\n        "source_linked": True,\n    }\n\n\ndef relative_year_offset(anchor: Note, relative: Note) -> dict[str, Any]:\n    """Resolve an explicitly stated bounded year offset from a cited year."""\n    if anchor.type != "date_or_time" or relative.type != "temporal":\n        raise ValueError("relative year offset requires a date and temporal note")\n    year_match = _NUMBER_RE.search(anchor.literal_text)\n    offset = relative.attributes.get("offset_years")\n    direction = str(relative.attributes.get("direction", "")).casefold()\n    if year_match is None or not isinstance(offset, int) or offset < 0:\n        raise ValueError("relative year offset operands are incomplete")\n    year = int(year_match.group().translate(_BN_TO_ASCII))\n    sign = -1 if direction in {"আগে", "পূর্বে", "before"} else 1\n    result = year + sign * offset\n    operator = "-" if sign < 0 else "+"\n    return {\n        "derivation_id": "derive:" + _sha(\n            f"relative_year_offset|{anchor.note_id}|{relative.note_id}|{result}"\n        )[:24],\n        "operator_id": "relative_year_offset",\n        "operand_note_ids": [anchor.note_id, relative.note_id],\n        "visible_expression": f"{year} {operator} {offset} = {result}",\n        "result": {"type": "year", "value": result, "unit": "সাল"},\n        "source_linked": True,\n    }\n\n\ndef bounded_fathers_father(first: Note, second: Note) -> dict[str, Any]:\n    if first.type != "relation" or second.type != "relation":\n        raise ValueError("kinship composition requires relation notes")\n    a, b = first.attributes, second.attributes\n    father_words = {"বাবা", "পিতা"}\n    if a.get("relation") not in father_words or b.get("relation") not in father_words or a.get("object") != b.get("subject"):\n        raise ValueError("relation edges do not form a bounded father\'s-father chain")\n    result = b["object"]\n    return {\n        "derivation_id": "derive:" + _sha(f"bounded_fathers_father|{first.note_id}|{second.note_id}|{result}")[:24],\n        "operator_id": "bounded_fathers_father", "operand_note_ids": [first.note_id, second.note_id],\n        "visible_expression": f"{a[\'subject\']}→পিতা→{a[\'object\']}→পিতা→{result}",\n        "result": {"type": "entity", "value": result, "unit": ""}, "source_linked": True,\n    }\n\n\ndef decision_receipt(\n    context: str,\n    question: str,\n    response: str,\n    requested_relation_or_property: str,\n    verdict: str,\n    evidence_notes: Iterable[Note],\n    derivation: dict[str, Any] | None = None,\n) -> dict[str, Any]:\n    if verdict not in VERDICTS or not requested_relation_or_property.strip():\n        raise ValueError("invalid verdict or requested slot")\n    notes = list(evidence_notes)\n    if verdict != "not_enough_information" and not notes:\n        raise ValueError("supported/refuted decisions require context evidence")\n    if derivation:\n        if derivation.get("operator_id") not in OPERATORS or not derivation.get("source_linked"):\n            raise ValueError("unapproved or ungrounded derivation")\n        available = {n.note_id for n in notes}\n        if not set(derivation.get("operand_note_ids", [])).issubset(available):\n            raise ValueError("derivation operands must be cited evidence notes")\n    receipt = {\n        "version": VERSION, "lane": "contextual_only", "external_retrieval_allowed": False,\n        "context_sha256": _sha(context), "question": question, "question_sha256": _sha(question),\n        "response": response, "response_sha256": _sha(response),\n        "requested_relation_or_property": requested_relation_or_property,\n        "verdict": verdict, "evidence_notes": [asdict(n) for n in notes], "derivation": derivation,\n        "question_text_is_evidence": False,\n    }\n    _fail_if_forbidden_keys(receipt)\n    receipt["receipt_sha256"] = _sha(_canonical(receipt))\n    return receipt\n'
if hashlib.sha256(_MORICHIKA_CONTEXT_ROUTER_SOURCE.encode("utf-8")).hexdigest() != "126dbf9a6fff8fbe9e510706d595828221e5e7013427bfc5602c995e022a8013":
    raise RuntimeError("embedded contextual router source hash mismatch")
_MORICHIKA_CONTEXT_ROUTER_NAME = "morichika_contextual_note_taker_fallback_bound"
_MORICHIKA_CONTEXT_ROUTER = _morichika_types.ModuleType(_MORICHIKA_CONTEXT_ROUTER_NAME)
_MORICHIKA_CONTEXT_ROUTER.__file__ = "<embedded-contextual-note-taker-fallback>"
sys.modules[_MORICHIKA_CONTEXT_ROUTER_NAME] = _MORICHIKA_CONTEXT_ROUTER
exec(
    compile(
        _MORICHIKA_CONTEXT_ROUTER_SOURCE,
        _MORICHIKA_CONTEXT_ROUTER.__file__,
        "exec",
    ),
    _MORICHIKA_CONTEXT_ROUTER.__dict__,
)
rank_context_windows = _MORICHIKA_CONTEXT_ROUTER.rank_context_windows
CONTEXT_ROUTER_SHA256 = "126dbf9a6fff8fbe9e510706d595828221e5e7013427bfc5602c995e022a8013"
del _MORICHIKA_CONTEXT_ROUTER_SOURCE
print("MORICHIKA contextual router:", CONTEXT_ROUTER_SHA256)


MORICHIKA contextual router: 126dbf9a6fff8fbe9e510706d595828221e5e7013427bfc5602c995e022a8013


In [9]:
import importlib as _morichika_importlib
import inspect as _morichika_inspect
import sys as _morichika_sys
from pathlib import Path as _morichika_Path

_RUNTIME_OVERLAY_PAYLOADS = {'pipeline/phase2_canonicalize.py': (b'"""Pure label-free text canonicalization shared by Phase 2 runtime stages."""\n\nfrom __future__ import annotations\n\nimport re\nimport unicodedata\n\n\nVERSION = "phase2-symbol-safe-canonicalizer-v2-idempotent-boundaries"\nBN_DIGITS = str.maketrans("\xe0\xa7\xa6\xe0\xa7\xa7\xe0\xa7\xa8\xe0\xa7\xa9\xe0\xa7\xaa\xe0\xa7\xab\xe0\xa7\xac\xe0\xa7\xad\xe0\xa7\xae\xe0\xa7\xaf", "0123456789")\n\n\ndef canonicalize(value: object) -> str:\n    text = unicodedata.normalize("NFKC", str(value)).translate(BN_DIGITS)\n    text = text.casefold().replace("&gt;", ">").replace("&lt;", "<")\n    text = re.sub(r"[*]+", "", text)\n    text = re.sub(r"[\xe2\x80\x9c\xe2\x80\x9d\\"\'`\xe2\x80\x98\xe2\x80\x99]+", "", text)\n    text = re.sub(r"[^\\w\\u0980-\\u09ff%\xe2\x88\x9a<>/=+.\\-@$#&]+", " ", text)\n    text = re.sub(r"\\s+", " ", text)\n    # Strip boundary punctuation and the whitespace adjacent to it in one\n    # operation.  The v1 ``strip().strip(\'.-\')`` sequence could expose a new\n    # leading/trailing space, so canonicalize(canonicalize(x)) differed from\n    # canonicalize(x) for OCR strings such as ``".. question"``.\n    return re.sub(r"^[.\\-\\s]+|[.\\-\\s]+$", "", text)\n', 'b2529d23b2cbb82bdd63b4ef455011bafdf7d6a97a1f804771e8f664a7f23bcd'), 'pipeline/phase2_mmap_retrieval.py': (b'"""Safe runtime reader for content-addressed Phase 2 sparse caches."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport sqlite3\nimport zlib\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nfrom pipeline.phase2_canonicalize import VERSION as CANONICALIZER_VERSION\n\n\nVERSION = "phase2-mmap-sparse-cache-v2-idempotent-canonicalizer"\nNORMALIZER = Path(__file__).resolve().with_name("phase2_canonicalize.py")\nRIGHTS_BOOLEAN_FIELDS = (\n    "bundle_allowed", "public_redistribution", "attribution_required",\n    "noncommercial_only", "share_alike", "local_only",\n    "targeted_evidence_only", "quarantined",\n)\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef validate_cache_rights_policy(value: object) -> dict[str, Any]:\n    """Validate the rights ledger embedded in every deployable mmap cache."""\n\n    if not isinstance(value, dict):\n        raise ValueError("mmap cache rights policy must be an object")\n    for field in RIGHTS_BOOLEAN_FIELDS:\n        if not isinstance(value.get(field), bool):\n            raise ValueError(f"mmap cache rights policy requires Boolean {field}")\n    if not str(value.get("note", "")).strip():\n        raise ValueError("mmap cache rights policy requires a nonempty note")\n    if value["local_only"] and value["public_redistribution"]:\n        raise ValueError("local-only cache cannot permit public redistribution")\n    return dict(value)\n\n\ndef _dtype(value: str) -> type[np.generic]:\n    mapping: dict[str, type[np.generic]] = {\n        "float32": np.float32,\n        "float64": np.float64,\n    }\n    if value not in mapping:\n        raise ValueError(f"unsupported vectorizer dtype: {value}")\n    return mapping[value]\n\n\ndef load_safe_vectorizer(directory: Path, manifest: dict[str, Any]) -> TfidfVectorizer:\n    config = json.loads((directory / "vectorizer.json").read_text(encoding="utf-8"))\n    vocabulary = json.loads((directory / "vocabulary.json").read_text(encoding="utf-8"))\n    if not isinstance(vocabulary, list) or len(vocabulary) != int(config["vocabulary_size"]):\n        raise ValueError("safe vectorizer vocabulary size mismatch")\n    if len(vocabulary) != len(set(vocabulary)):\n        raise ValueError("safe vectorizer vocabulary contains duplicates")\n    vectorizer = TfidfVectorizer(\n        analyzer=config["analyzer"],\n        binary=bool(config["binary"]),\n        decode_error=config["decode_error"],\n        dtype=_dtype(config["dtype"]),\n        encoding=config["encoding"],\n        input=config["input"],\n        lowercase=bool(config["lowercase"]),\n        ngram_range=tuple(config["ngram_range"]),\n        norm=config["norm"],\n        smooth_idf=bool(config["smooth_idf"]),\n        strip_accents=config["strip_accents"],\n        sublinear_tf=bool(config["sublinear_tf"]),\n        token_pattern=config["token_pattern"],\n        use_idf=bool(config["use_idf"]),\n        vocabulary={token: index for index, token in enumerate(vocabulary)},\n    )\n    if config["use_idf"]:\n        idf = np.load(directory / "idf.npy", mmap_mode="r", allow_pickle=False)\n        if idf.ndim != 1 or idf.shape[0] != len(vocabulary):\n            raise ValueError("safe vectorizer IDF shape mismatch")\n        vectorizer.idf_ = np.asarray(idf)\n    if len(vocabulary) != int(manifest["matrix_shape"][1]):\n        raise ValueError("safe vectorizer/matrix feature mismatch")\n    return vectorizer\n\n\nclass CompressedSqliteRecords:\n    def __init__(self, path: Path, records: int) -> None:\n        self.path = path.resolve()\n        self.records = int(records)\n        uri = f"file:{self.path.as_posix()}?mode=ro&immutable=1"\n        self.connection = sqlite3.connect(uri, uri=True, check_same_thread=False)\n        self.connection.execute("PRAGMA query_only=ON")\n\n    def __len__(self) -> int:\n        return self.records\n\n    def __getitem__(self, index: int) -> dict[str, Any]:\n        if not 0 <= int(index) < self.records:\n            raise IndexError(index)\n        row = self.connection.execute(\n            "SELECT payload FROM records WHERE idx=?", (int(index),)\n        ).fetchone()\n        if row is None:\n            raise IndexError(index)\n        return json.loads(zlib.decompress(bytes(row[0])).decode("utf-8"))\n\n\nclass SqliteExactLookup:\n    def __init__(self, records: CompressedSqliteRecords) -> None:\n        self.records = records\n\n    def get(self, key: str, default: Any = None) -> list[int] | Any:\n        if not key:\n            return default\n        rows = self.records.connection.execute(\n            "SELECT idx FROM exact_lookup WHERE question_key=? ORDER BY idx", (key,)\n        ).fetchall()\n        return [int(row[0]) for row in rows] if rows else default\n\n\ndef load_mmap_index(directory: Path) -> dict[str, Any]:\n    directory = directory.resolve()\n    manifest_path = directory / "manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))\n    if manifest.get("version") != VERSION:\n        raise ValueError("unsupported Phase 2 mmap cache version")\n    fingerprint = manifest.get("fingerprint") or {}\n    if fingerprint.get("canonicalizer_version") != CANONICALIZER_VERSION:\n        raise ValueError("Phase 2 mmap cache canonicalizer version mismatch")\n    if fingerprint.get("normalizer_sha256") != sha256_file(NORMALIZER):\n        raise ValueError("Phase 2 mmap cache canonicalizer hash mismatch")\n    validate_cache_rights_policy(manifest.get("rights_policy"))\n    declared = manifest.get("files") or {}\n    expected_names = {\n        "vectorizer.json", "vocabulary.json", "idf.npy", "data.npy",\n        "indices.npy", "indptr.npy", "records.sqlite3",\n    }\n    if set(declared) != expected_names:\n        raise ValueError("mmap cache file declaration mismatch")\n    for name, metadata in declared.items():\n        path = directory / name\n        if (\n            not path.is_file()\n            or sha256_file(path) != str(metadata.get("sha256", ""))\n            or path.stat().st_size != int(metadata.get("bytes", -1))\n        ):\n            raise ValueError(f"mmap cache hash/size mismatch: {name}")\n\n    shape = tuple(int(value) for value in manifest["matrix_shape"])\n    data = np.load(directory / "data.npy", mmap_mode="r", allow_pickle=False)\n    indices = np.load(directory / "indices.npy", mmap_mode="r", allow_pickle=False)\n    indptr = np.load(directory / "indptr.npy", mmap_mode="r", allow_pickle=False)\n    if data.ndim != 1 or indices.shape != data.shape or indptr.shape != (shape[0] + 1,):\n        raise ValueError("mmap CSR array shape mismatch")\n    if int(indptr[-1]) != data.shape[0] or int(manifest["matrix_nnz"]) != data.shape[0]:\n        raise ValueError("mmap CSR nnz mismatch")\n    matrix = sparse.csr_matrix((data, indices, indptr), shape=shape, copy=False)\n    records = CompressedSqliteRecords(directory / "records.sqlite3", shape[0])\n    return {\n        "directory": directory,\n        "manifest": manifest,\n        "manifest_sha256": sha256_file(manifest_path),\n        "vectorizer": load_safe_vectorizer(directory, manifest),\n        "matrix": matrix,\n        "records": records,\n        "exact_lookup": SqliteExactLookup(records),\n        "cache_id": manifest["cache_id"],\n        "cache_format": VERSION,\n    }\n\n\ndef close_mmap_index(index: dict[str, Any]) -> None:\n    """Release SQLite and NumPy mmap handles, including on Windows."""\n\n    records = index.get("records")\n    connection = getattr(records, "connection", None)\n    if connection is not None:\n        connection.close()\n    arrays = []\n    matrix = index.get("matrix")\n    if matrix is not None:\n        arrays.extend((matrix.data, matrix.indices, matrix.indptr))\n    vectorizer = index.get("vectorizer")\n    idf_values = getattr(getattr(vectorizer, "_tfidf", None), "idf_", None)\n    if idf_values is not None:\n        arrays.append(idf_values)\n    idf_diag = getattr(getattr(vectorizer, "_tfidf", None), "_idf_diag", None)\n    if idf_diag is not None:\n        arrays.extend((idf_diag.data, idf_diag.indices, idf_diag.indptr))\n    seen: set[int] = set()\n    for array in arrays:\n        current = array\n        while current is not None and id(current) not in seen:\n            seen.add(id(current))\n            if isinstance(current, np.memmap):\n                mmap = getattr(current, "_mmap", None)\n                if mmap is not None:\n                    mmap.close()\n                break\n            current = getattr(current, "base", None)\n', '281a204047684903d7bec83ebe0846285e8a6b3f9a0efb4c30011d619e7e2155'), 'pipeline/phase2_composite_fts_retrieval.py': (b'"""Runtime query API for the nonterminal Phase 2 composite FTS cache."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport itertools\nimport json\nimport re\nimport sqlite3\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nROOT = Path(__file__).resolve().parents[1]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom pipeline.phase2_canonicalize import canonicalize\nfrom pipeline.phase2_source_authority import authority_for\n\n\nDEFAULT_CACHE = ROOT / "artifacts/retrieval/phase2_composite_fts_v2"\nVERSION = "phase2-composite-fts-runtime-v5-bounded-source-scan-exact-mode"\nSUPPORTED_CACHE_VERSIONS = {\n    "phase2-composite-fts-cache-v2-nctbench-nonterminal",\n    "phase2-strict-rights-safe-fts-v3-nonterminal",\n}\nQUERY_TOKEN_RE = re.compile(r"[\\w\\u0980-\\u09ff]+", re.UNICODE)\nQUERY_STOPWORDS = {\n    "\xe0\xa6\x95\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa7\x80", "\xe0\xa6\x95\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb0", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa6\xa4", "\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc",\n    "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xad\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa7\x80\xe0\xa6\xad\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa7\x87\xe0\xa6\xa8", "\xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb9\xe0\xa7\x9f", "\xe0\xa6\x9b\xe0\xa6\xbf\xe0\xa6\xb2", "\xe0\xa6\x86\xe0\xa6\x9b\xe0\xa7\x87",\n    "\xe0\xa6\x8f\xe0\xa6\x95\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\x8f\xe0\xa6\x87", "\xe0\xa6\x93", "\xe0\xa6\x8f\xe0\xa6\xac\xe0\xa6\x82", "the", "a", "an", "is", "was", "what",\n    "which", "who", "where", "when", "how",\n}\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef load(cache_dir: Path) -> tuple[dict[str, Any], sqlite3.Connection]:\n    manifest_path = cache_dir / "manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest.get("version") not in SUPPORTED_CACHE_VERSIONS:\n        raise ValueError("composite cache version mismatch")\n    policy = manifest.get("policy") or {}\n    if (\n        policy.get("closed_book_only") is not True\n        or policy.get("contextual_external_retrieval_allowed") is not False\n        or policy.get("retrieval_is_terminal") is not False\n        or policy.get("retrieval_miss_means") != "NEI"\n    ):\n        raise ValueError("unsafe composite cache policy")\n    database = cache_dir / manifest["database"]["path"]\n    if (\n        database.stat().st_size != int(manifest["database"]["bytes"])\n        or sha256_file(database) != manifest["database"]["sha256"]\n    ):\n        raise ValueError("composite cache database hash/size mismatch")\n    connection = sqlite3.connect(f"file:{database.as_posix()}?mode=ro", uri=True)\n    connection.row_factory = sqlite3.Row\n    return manifest, connection\n\n\ndef query_tokens(text: str, *, max_terms: int = 8) -> list[str]:\n    tokens = []\n    for token in QUERY_TOKEN_RE.findall(canonicalize(text)):\n        if len(token) < 2 or token in QUERY_STOPWORDS or token in tokens:\n            continue\n        tokens.append(token)\n        if len(tokens) >= max_terms:\n            break\n    return tokens\n\n\ndef quote_fts(token: str) -> str:\n    return f\'"{token.replace(chr(34), chr(34) * 2)}"\'\n\n\ndef fts_query(text: str, *, max_terms: int = 8) -> str:\n    """Strict query: require every admitted content token (bounded to three)."""\n\n    tokens = query_tokens(text, max_terms=max_terms)\n    if not tokens:\n        return ""\n    if len(tokens) == 1:\n        return quote_fts(tokens[0])\n    strict_tokens = tokens if len(tokens) <= 3 else [tokens[0], tokens[-2], tokens[-1]]\n    return " AND ".join(quote_fts(token) for token in strict_tokens)\n\n\ndef relaxed_fts_query(text: str, *, max_terms: int = 8) -> str:\n    """Fallback requiring content-token pairs, never singleton OR terms."""\n\n    tokens = query_tokens(text, max_terms=max_terms)\n    if not tokens:\n        return ""\n    if len(tokens) == 1:\n        return quote_fts(tokens[0])\n    if len(tokens) <= 4:\n        # Pair relaxation tolerates one OCR/spelling mismatch while preventing a\n        # single generic word such as "first" from admitting a candidate.\n        pairs = [\n            f"({quote_fts(left)} AND {quote_fts(right)})"\n            for left, right in itertools.combinations(tokens, 2)\n        ]\n        return " OR ".join(pairs)\n    # Query construction must not grow quadratically with long prompts.\n    return " AND ".join(quote_fts(token) for token in tokens[:3])\n\n\ndef exact_question_variants(text: str) -> list[str]:\n    normalized = canonicalize(text)\n    # Bengali interrogative \xe0\xa6\x95\xe0\xa6\xbf/\xe0\xa6\x95\xe0\xa7\x80 varies in otherwise exact source questions.\n    # Keep this local to question-key lookup; it must not rewrite evidence text.\n    variants = [normalized]\n    for old, new in (("\xe0\xa6\x95\xe0\xa7\x80", "\xe0\xa6\x95\xe0\xa6\xbf"), ("\xe0\xa6\x95\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa7\x80")):\n        candidate = re.sub(rf"(?<!\\w){old}(?!\\w)", new, normalized)\n        if candidate not in variants:\n            variants.append(candidate)\n    return variants\n\n\ndef excerpt(text: str, query: str, *, limit: int = 900) -> str:\n    compact = " ".join(text.split())\n    if len(compact) <= limit:\n        return compact\n    tokens = [token for token in QUERY_TOKEN_RE.findall(canonicalize(query)) if len(token) >= 3]\n    # ``compact`` is source display text, not a comparison key.  Case folding\n    # is sufficient to place the excerpt around an already-canonical query\n    # term and avoids repeatedly normalizing an entire long passage.\n    lowered = compact.casefold()\n    positions = [lowered.find(token) for token in tokens if lowered.find(token) >= 0]\n    center = min(positions) if positions else 0\n    start = max(0, center - limit // 3)\n    return compact[start : start + limit]\n\n\ndef _lexical_display_score(\n    query_terms: list[str], row: sqlite3.Row,\n) -> float:\n    """Return a deterministic bounded score for the existing semantic gates.\n\n    FTS5\'s weighted BM25 is useful for ordering, but asking SQLite to score and\n    sort every match in the 1.7 GB composite is the dominant runtime cost.  The\n    optimized path instead asks FTS only for a small deterministic rowid pool,\n    then scores the exact text that the downstream gate will inspect.  This is\n    deliberately not a verdict and cannot bypass overlap, number, negation or\n    answer-type checks.\n    """\n\n    if not query_terms:\n        return 0.0\n    question = str(row["question"] or "")\n    candidate_text = question or "\\n".join(\n        str(row[field] or "") for field in ("title", "supporting_text")\n    )\n    # FTS5 has already applied the strict/relaxed Unicode token match.  A\n    # second full symbol-safe canonicalization of every scanned passage was\n    # pure duplicate work (and the largest remaining Python cost).  Tokenize\n    # the source text directly for this bounded display score; number and\n    # negation normalization remain enforced by the downstream semantic gate.\n    candidate_terms = set(QUERY_TOKEN_RE.findall(candidate_text.casefold()))\n    shared = [term for term in query_terms if term in candidate_terms]\n    coverage = len(shared) / len(query_terms)\n    # Preserve the pre-existing gate\'s explicit long distinctive-token\n    # exception.  The gate itself still decides eligibility.\n    if len(shared) == 1 and len(shared[0]) >= 8:\n        coverage = max(coverage, 0.55)\n    return round(min(1.0, coverage), 8)\n\n\ndef _row_to_result(\n    row: sqlite3.Row,\n    query: str,\n    *,\n    exact_question: bool,\n    retrieval_score: float,\n) -> dict[str, Any]:\n    supporting_text = str(row["supporting_text"])\n    return {\n        "source_record_index": int(row["id"]),\n        "source_id": row["source_id"],\n        "source_locator": row["source_locator"],\n        "semantic_role": row["semantic_role"],\n        "model_facing": bool(row["model_facing"]),\n        "question": row["question"],\n        "answer": row["answer"],\n        "title": row["title"],\n        "evidence_excerpt": excerpt(supporting_text, query),\n        "choices": json.loads(row["choices_json"]),\n        "metadata": json.loads(row["metadata_json"]),\n        "record_sha256": row["record_sha256"],\n        "exact_question": exact_question,\n        "retrieval_score": retrieval_score,\n        "terminal_label_authority": False,\n        "verdict": "NEI",\n    }\n\n\ndef retrieve(\n    connection: sqlite3.Connection,\n    query: str,\n    *,\n    top_k: int = 20,\n    model_facing_only: bool = True,\n    source_ids: list[str] | None = None,\n) -> list[dict[str, Any]]:\n    normalized_variants = exact_question_variants(query)\n    exact_clauses = [\n        "normalized_question IN (%s)" % ",".join("?" for _ in normalized_variants),\n        "(?=0 OR model_facing=1)",\n    ]\n    exact_parameters: list[Any] = [*normalized_variants, int(model_facing_only)]\n    if source_ids:\n        exact_clauses.append("source_id IN (%s)" % ",".join("?" for _ in source_ids))\n        exact_parameters.extend(source_ids)\n    exact_parameters.append(top_k)\n    exact_rows = connection.execute(\n        "SELECT *, 1 AS exact_question, -1000.0 AS rank FROM evidence WHERE "\n        + " AND ".join(exact_clauses)\n        + " ORDER BY source_id, source_locator, id LIMIT ?",\n        exact_parameters,\n    ).fetchall()\n    seen = {int(row["id"]) for row in exact_rows}\n    rows = list(exact_rows)\n    remaining = max(0, top_k - len(rows))\n    # An exact question landing is sufficient evidence retrieval.  Broad\n    # top-up after exact matches adds latency and unrelated collisions.\n    if remaining and not exact_rows:\n        expressions = [fts_query(query)]\n        relaxed = relaxed_fts_query(query)\n        if relaxed and relaxed not in expressions:\n            expressions.append(relaxed)\n        for expression in expressions:\n            if not expression:\n                continue\n            clauses = ["evidence_fts MATCH ?", "(?=0 OR e.model_facing=1)"]\n            parameters: list[Any] = [expression, int(model_facing_only)]\n            if source_ids:\n                clauses.append("e.source_id IN (%s)" % ",".join("?" for _ in source_ids))\n                parameters.extend(source_ids)\n            parameters.append(remaining)\n            fuzzy = connection.execute(\n                "SELECT e.*, 0 AS exact_question, bm25(evidence_fts, 3.0, 1.5, 2.0, 1.0) AS rank "\n                "FROM evidence_fts JOIN evidence e ON e.id=evidence_fts.rowid WHERE "\n                + " AND ".join(clauses)\n                + " ORDER BY rank, e.source_id, e.source_locator, e.id LIMIT ?",\n                parameters,\n            ).fetchall()\n            if fuzzy:\n                rows.extend(fuzzy)\n                break\n    return [\n        _row_to_result(\n            row,\n            query,\n            exact_question=bool(row["exact_question"]),\n            retrieval_score=float(row["rank"]),\n        )\n        for row in rows[:top_k]\n    ]\n\n\ndef model_facing_source_ids(connection: sqlite3.Connection) -> list[str]:\n    """Return the immutable source universe admitted by the built cache.\n\n    The cache builder is the rights/admission boundary.  This lookup cannot\n    enable a source that was not already stored as model-facing evidence.\n    """\n\n    return [\n        str(row[0])\n        for row in connection.execute(\n            "SELECT DISTINCT source_id FROM evidence "\n            "WHERE model_facing=1 ORDER BY source_id"\n        ).fetchall()\n    ]\n\n\n_SOURCE_ROUTING_CACHE: dict[\n    sqlite3.Connection, dict[str, dict[str, Any]]\n] = {}\n\n\ndef _source_routing_profile(\n    connection: sqlite3.Connection,\n) -> dict[str, dict[str, Any]]:\n    """Describe admitted source rowid intervals once per open connection."""\n\n    cached = _SOURCE_ROUTING_CACHE.get(connection)\n    if cached is not None:\n        return cached\n    profile: dict[str, dict[str, Any]] = {}\n    rows = connection.execute(\n        "SELECT source_id, MIN(id), MAX(id), COUNT(*) FROM evidence "\n        "WHERE model_facing=1 GROUP BY source_id ORDER BY source_id"\n    ).fetchall()\n    for source_id, minimum, maximum, count in rows:\n        minimum, maximum, count = int(minimum), int(maximum), int(count)\n        profile[str(source_id)] = {\n            "minimum_rowid": minimum,\n            "maximum_rowid": maximum,\n            "record_count": count,\n            "contiguous_rowids": maximum - minimum + 1 == count,\n        }\n    _SOURCE_ROUTING_CACHE[connection] = profile\n    return profile\n\n\ndef _bounded_source_scan(\n    connection: sqlite3.Connection,\n    expression: str,\n    *,\n    source_id: str,\n    scan_limit: int,\n    routing: dict[str, dict[str, Any]],\n) -> list[int]:\n    """Return a deterministic bounded FTS rowid pool for one admitted source."""\n\n    source = routing[source_id]\n    if source["contiguous_rowids"]:\n        rows = connection.execute(\n            "SELECT rowid FROM evidence_fts WHERE evidence_fts MATCH ? "\n            "AND rowid BETWEEN ? AND ? ORDER BY rowid LIMIT ?",\n            (\n                expression,\n                source["minimum_rowid"],\n                source["maximum_rowid"],\n                scan_limit,\n            ),\n        ).fetchall()\n    else:\n        # Fail-safe for a future cache whose source rows are interleaved.  The\n        # current v2 cache is contiguous, but correctness must not depend on\n        # undocumented insertion order.\n        rows = connection.execute(\n            "SELECT evidence_fts.rowid FROM evidence_fts "\n            "JOIN evidence e ON e.id=evidence_fts.rowid "\n            "WHERE evidence_fts MATCH ? AND e.model_facing=1 "\n            "AND e.source_id=? ORDER BY evidence_fts.rowid LIMIT ?",\n            (expression, source_id, scan_limit),\n        ).fetchall()\n    return [int(row[0]) for row in rows]\n\n\ndef retrieve_authority_tier_pool_optimized(\n    connection: sqlite3.Connection,\n    query: str,\n    *,\n    per_authority_tier_k: int = 20,\n    source_scan_multiplier: int = 4,\n    source_ids: list[str] | None = None,\n    fuzzy_search: bool = True,\n) -> list[dict[str, Any]]:\n    """Fast authority-before-top-k retrieval with bounded source pools.\n\n    Exact-question lookup is unchanged.  Fuzzy lookup scans a small,\n    deterministic rowid pool independently for every admitted source, thereby\n    preventing a large source from starving a book/curated source without the\n    expensive global BM25 sort.  The existing caller still performs all\n    semantic gates, authority ordering, final top-k and counter retention.\n    """\n\n    if not 1 <= per_authority_tier_k <= 100:\n        raise ValueError("per_authority_tier_k must be 1..100")\n    if not 1 <= source_scan_multiplier <= 16:\n        raise ValueError("source_scan_multiplier must be 1..16")\n    routing = _source_routing_profile(connection)\n    admitted = set(routing)\n    selected_sources = (\n        sorted(admitted)\n        if source_ids is None\n        else sorted({str(source_id) for source_id in source_ids})\n    )\n    unknown = sorted(set(selected_sources) - admitted)\n    if unknown:\n        raise ValueError(\n            f"bounded source pool requested unadmitted sources: {unknown}"\n        )\n    sources_by_tier: dict[int, list[str]] = {}\n    for source_id in selected_sources:\n        tier = int(authority_for(source_id)["authority_tier"])\n        sources_by_tier.setdefault(tier, []).append(source_id)\n\n    variants = exact_question_variants(query)\n    exact_parameters: list[Any] = [*variants]\n    clauses = [\n        "normalized_question IN (%s)" % ",".join("?" for _ in variants),\n        "model_facing=1",\n    ]\n    if selected_sources:\n        clauses.append(\n            "source_id IN (%s)" % ",".join("?" for _ in selected_sources)\n        )\n        exact_parameters.extend(selected_sources)\n    exact_rows = connection.execute(\n        "SELECT * FROM evidence WHERE " + " AND ".join(clauses)\n        + " ORDER BY source_id, source_locator, id",\n        exact_parameters,\n    ).fetchall()\n    exact_by_tier: dict[int, list[sqlite3.Row]] = {}\n    for row in exact_rows:\n        tier = int(authority_for(row["source_id"])["authority_tier"])\n        exact_by_tier.setdefault(tier, []).append(row)\n\n    scan_limit = max(16, per_authority_tier_k * source_scan_multiplier)\n    strict = fts_query(query) if fuzzy_search else ""\n    relaxed = relaxed_fts_query(query) if fuzzy_search else ""\n    score_query_terms = query_tokens(query)\n    fuzzy_rowids_by_tier: dict[int, list[int]] = {}\n    for tier, tier_sources in sorted(sources_by_tier.items()):\n        if exact_by_tier.get(tier) or not strict:\n            continue\n        rowids: list[int] = []\n        for source_id in tier_sources:\n            rowids.extend(_bounded_source_scan(\n                connection,\n                strict,\n                source_id=source_id,\n                scan_limit=scan_limit,\n                routing=routing,\n            ))\n        if not rowids and relaxed and relaxed != strict:\n            for source_id in tier_sources:\n                rowids.extend(_bounded_source_scan(\n                    connection,\n                    relaxed,\n                    source_id=source_id,\n                    scan_limit=scan_limit,\n                    routing=routing,\n                ))\n        fuzzy_rowids_by_tier[tier] = sorted(set(rowids))\n\n    all_fuzzy_rowids = sorted({\n        rowid for rowids in fuzzy_rowids_by_tier.values() for rowid in rowids\n    })\n    fuzzy_by_id: dict[int, sqlite3.Row] = {}\n    # SQLite\'s default variable limit comfortably exceeds the current bounded\n    # maximum (six admitted sources * 100 * 16), but keep batched reads robust.\n    for offset in range(0, len(all_fuzzy_rowids), 500):\n        batch = all_fuzzy_rowids[offset : offset + 500]\n        if not batch:\n            continue\n        rows = connection.execute(\n            "SELECT * FROM evidence WHERE id IN (%s)" % ",".join("?" for _ in batch),\n            batch,\n        ).fetchall()\n        fuzzy_by_id.update({int(row["id"]): row for row in rows})\n\n    pool: list[dict[str, Any]] = []\n    for tier in sorted(sources_by_tier):\n        if exact_by_tier.get(tier):\n            tier_results = [\n                _row_to_result(\n                    row,\n                    query,\n                    exact_question=True,\n                    retrieval_score=-1000.0,\n                )\n                for row in exact_by_tier[tier][:per_authority_tier_k]\n            ]\n        else:\n            scored: list[tuple[tuple[Any, ...], sqlite3.Row, float]] = []\n            for rowid in fuzzy_rowids_by_tier.get(tier, []):\n                row = fuzzy_by_id.get(rowid)\n                if row is None:\n                    continue\n                score = _lexical_display_score(score_query_terms, row)\n                key = (\n                    -score,\n                    str(row["source_id"]),\n                    str(row["source_locator"]),\n                    int(row["id"]),\n                    str(row["record_sha256"]),\n                )\n                scored.append((key, row, score))\n            scored.sort(key=lambda value: value[0])\n            tier_results = [\n                _row_to_result(\n                    row,\n                    query,\n                    exact_question=False,\n                    retrieval_score=-score,\n                )\n                for _, row, score in scored[:per_authority_tier_k]\n            ]\n        for within_tier_rank, result in enumerate(tier_results, start=1):\n            pool.append({\n                **result,\n                "authority_pool_tier": tier,\n                "within_authority_tier_rank": within_tier_rank,\n                "retrieval_score_kind": (\n                    "exact_question_sentinel"\n                    if result["exact_question"]\n                    else "bounded_query_token_coverage"\n                ),\n            })\n    return sorted(\n        pool,\n        key=lambda row: (\n            int(row["authority_pool_tier"]),\n            int(row["within_authority_tier_rank"]),\n            str(row["source_id"]),\n            int(row["source_record_index"]),\n            str(row["record_sha256"]),\n        ),\n    )\n\n\ndef retrieve_authority_tier_pool(\n    connection: sqlite3.Connection,\n    query: str,\n    *,\n    per_authority_tier_k: int = 20,\n    source_ids: list[str] | None = None,\n) -> list[dict[str, Any]]:\n    """Retrieve a bounded candidate pool per authority tier before top-k.\n\n    A single global SQL ``LIMIT`` can allow a large source to consume the\n    whole pool before books/user OCR/curated-data authority is evaluated.\n    Querying every already-admitted authority tier independently retains up to\n    ``per_authority_tier_k`` candidates from books, curated data, default\n    authorized sources, and Wikipedia separately.  The caller must run the\n    semantic gates and shared authority ranking before final top-k.\n    """\n\n    if not 1 <= per_authority_tier_k <= 100:\n        raise ValueError("per_authority_tier_k must be 1..100")\n    admitted = set(model_facing_source_ids(connection))\n    selected_sources = (\n        sorted(admitted)\n        if source_ids is None\n        else sorted({str(source_id) for source_id in source_ids})\n    )\n    unknown = sorted(set(selected_sources) - admitted)\n    if unknown:\n        raise ValueError(f"source-diverse pool requested unadmitted sources: {unknown}")\n    sources_by_tier: dict[int, list[str]] = {}\n    for source_id in selected_sources:\n        tier = int(authority_for(source_id)["authority_tier"])\n        sources_by_tier.setdefault(tier, []).append(source_id)\n    pool: list[dict[str, Any]] = []\n    seen: set[tuple[str, int]] = set()\n    for authority_tier, tier_sources in sorted(sources_by_tier.items()):\n        rows = retrieve(\n            connection,\n            query,\n            top_k=per_authority_tier_k,\n            model_facing_only=True,\n            source_ids=sorted(tier_sources),\n        )\n        for within_tier_rank, row in enumerate(rows, start=1):\n            identity = (str(row["source_id"]), int(row["source_record_index"]))\n            if identity in seen:\n                continue\n            seen.add(identity)\n            pool.append({\n                **row,\n                "authority_pool_tier": authority_tier,\n                "within_authority_tier_rank": within_tier_rank,\n            })\n    return sorted(\n        pool,\n        key=lambda row: (\n            int(row["authority_pool_tier"]),\n            int(row["within_authority_tier_rank"]),\n            str(row["source_id"]),\n            int(row["source_record_index"]),\n            str(row["record_sha256"]),\n        ),\n    )\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("query")\n    parser.add_argument("--cache", type=Path, default=DEFAULT_CACHE)\n    parser.add_argument("--top-k", type=int, default=10)\n    parser.add_argument("--include-components", action="store_true")\n    args = parser.parse_args()\n    _, connection = load(args.cache.resolve())\n    try:\n        print(json.dumps(retrieve(\n            connection,\n            args.query,\n            top_k=args.top_k,\n            model_facing_only=not args.include_components,\n        ), ensure_ascii=False, indent=2))\n    finally:\n        connection.close()\n\n\nif __name__ == "__main__":\n    main()\n', '784837e7fd54d7a7761e4f60ce6af920783780a0255f7060d3f8124e14594265'), 'pipeline/phase2_response_proposition.py': (b'"""Conservative proposition checks for Bengali answer responses.\n\nRetrieval and arithmetic helpers must not treat the mere presence of an\nexpected string or numeral as an asserted answer.  This module keeps the\noriginal Unicode/operator surfaces, identifies bounded answer spans, and\nrejects negated, corrected, alternative, or numerically competing claims.\n\nThe functions are deliberately admission-oriented: ``True``/``supported``\nmeans that a narrow deterministic check succeeded.  Every ambiguous form\nremains nonterminal for a verifier instead of being converted into a label.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nimport unicodedata\nfrom dataclasses import dataclass\nfrom typing import Iterable\n\n\nVERSION = "phase2-response-proposition-v1-negation-correction-aware"\nBN_DIGITS = str.maketrans("\xe0\xa7\xa6\xe0\xa7\xa7\xe0\xa7\xa8\xe0\xa7\xa9\xe0\xa7\xaa\xe0\xa7\xab\xe0\xa7\xac\xe0\xa7\xad\xe0\xa7\xae\xe0\xa7\xaf", "0123456789")\nNUMERIC_TOKEN_PATTERN = r"[-+]?(?:\\d{1,3}(?:,\\d{3})+|\\d+)(?:\\.\\d+)?(?:[eE][-+]?\\d+)?"\nTOKEN_RE = re.compile(\n    rf"{NUMERIC_TOKEN_PATTERN}|[a-z]+|[\\u0980-\\u09ff]+|[+*/%^\xc3\x97\xc3\xb7=]|[,;:.!?\xe0\xa5\xa4]",\n    re.I,\n)\nNUMBER_TOKEN_RE = re.compile(NUMERIC_TOKEN_PATTERN, re.I)\n\nNEGATION_TOKENS = {\n    "\xe0\xa6\xa8\xe0\xa6\xbe", "\xe0\xa6\xa8\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xa8\xe0\xa7\x9f", "\xe0\xa6\xa8\xe0\xa7\x87\xe0\xa6\x87", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xb9", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\x87", "not", "never", "incorrect",\n    "\xe0\xa6\xad\xe0\xa7\x81\xe0\xa6\xb2",\n}\nNEGATIVE_VERB_SUFFIXES = ("\xe0\xa7\x87\xe0\xa6\xa8\xe0\xa6\xbf", "\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xbf", "\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xa8\xe0\xa6\xbf", "\xe0\xa7\x9f\xe0\xa6\xa8\xe0\xa6\xbf", "\xe0\xa7\x87\xe0\xa6\xa8\xe0\xa6\xa8\xe0\xa6\xbf", "\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xa8\xe0\xa6\xbf")\nCONTRAST_TOKENS = {\n    "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa7\x81", "\xe0\xa6\xa4\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xb0\xe0\xa6\x82", "\xe0\xa6\x86\xe0\xa6\xb8\xe0\xa6\xb2\xe0\xa7\x87", "\xe0\xa6\x85\xe0\xa6\xa5\xe0\xa6\x9a", "\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\xac\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa7\x87", "however", "but",\n    "rather", "actually", "instead",\n}\nALTERNATIVE_TOKENS = {\n    "\xe0\xa6\x85\xe0\xa6\xa5\xe0\xa6\xac\xe0\xa6\xbe", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\x82\xe0\xa6\xac\xe0\xa6\xbe", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbf", "\xe0\xa6\xac\xe0\xa6\xbe", "or", "alternatively",\n}\nANSWER_SCAFFOLD = {\n    "\xe0\xa6\x89\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xb0", "\xe0\xa6\xb8\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\x95", "\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\x95", "\xe0\xa6\xb9\xe0\xa6\xb2\xe0\xa7\x8b", "\xe0\xa6\xb9\xe0\xa6\xb2", "\xe0\xa6\xb9\xe0\xa6\x9a\xe0\xa7\x8d\xe0\xa6\x9b\xe0\xa7\x87", "\xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb9\xe0\xa7\x9f", "\xe0\xa6\x9f\xe0\xa6\xbf", "the",\n    "answer", "is", "correct",\n}\nCLAUSE_BREAKS = {";", ".", "!", "?", "\xe0\xa5\xa4"}\n\n\ndef normalize_proposition(value: object) -> str:\n    text = "" if value is None else str(value)\n    text = unicodedata.normalize("NFKC", text).translate(BN_DIGITS).casefold()\n    text = text.replace("\\u200c", "").replace("\\u200d", "")\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef proposition_tokens(value: object) -> tuple[str, ...]:\n    return tuple(TOKEN_RE.findall(normalize_proposition(value)))\n\n\ndef _is_negation(token: str) -> bool:\n    return token in NEGATION_TOKENS or (\n        len(token) >= 4 and token.endswith(NEGATIVE_VERB_SUFFIXES)\n    )\n\n\ndef _find_spans(tokens: tuple[str, ...], needle: tuple[str, ...]) -> list[tuple[int, int]]:\n    if not needle or len(needle) > len(tokens):\n        return []\n    return [\n        (index, index + len(needle))\n        for index in range(len(tokens) - len(needle) + 1)\n        if all(\n            observed == expected\n            or observed in {expected + "\xe0\xa6\x87", expected + "\xe0\xa6\x93"}\n            for observed, expected in zip(\n                tokens[index : index + len(needle)], needle, strict=True,\n            )\n        )\n    ]\n\n\ndef _clause_bounds(tokens: tuple[str, ...], start: int, end: int) -> tuple[int, int]:\n    left = start\n    while left > 0 and tokens[left - 1] not in CLAUSE_BREAKS:\n        left -= 1\n    right = end\n    while right < len(tokens) and tokens[right] not in CLAUSE_BREAKS:\n        right += 1\n    return left, right\n\n\ndef _span_is_denied(tokens: tuple[str, ...], start: int, end: int) -> bool:\n    left, right = _clause_bounds(tokens, start, end)\n    # Bengali copular negation commonly follows a short predicate tail:\n    # ``\xe0\xa6\xa2\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe \xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\x9c\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa7\x80 \xe0\xa6\xa8\xe0\xa6\xaf\xe0\xa6\xbc``.  A small bidirectional window also covers\n    # ``\xe0\xa6\xa8\xe0\xa6\xbe, \xe0\xa6\xa2\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe \xe0\xa6\xa8\xe0\xa6\xaf\xe0\xa6\xbc`` without making a distant negation global.\n    local_left = max(left, start - 3)\n    local_right = min(right, end + 5)\n    return any(_is_negation(token) for token in tokens[local_left:local_right])\n\n\ndef _meaningful_tail(tokens: tuple[str, ...], position: int) -> tuple[str, ...]:\n    return tuple(\n        token for token in tokens[position:]\n        if token not in ANSWER_SCAFFOLD\n        and token not in CONTRAST_TOKENS\n        and token not in CLAUSE_BREAKS\n        and token not in {",", ":", "="}\n    )\n\n\n@dataclass(frozen=True)\nclass AnswerStance:\n    status: str\n    matched_answer: str\n    match_count: int\n    denied_match_count: int\n    correction_or_alternative_present: bool\n    version: str = VERSION\n\n\ndef classify_answer_stance(response: object, answers: Iterable[object]) -> AnswerStance:\n    """Classify whether an accepted answer is cleanly asserted.\n\n    The classifier does not decide a hallucination label.  It only admits a\n    source/answer relation when at least one complete answer span is asserted\n    and no span-level negation, correction, or alternative makes that relation\n    ambiguous.\n    """\n\n    tokens = proposition_tokens(response)\n    candidates: list[tuple[str, tuple[str, ...]]] = []\n    seen: set[tuple[str, ...]] = set()\n    for value in answers:\n        text = normalize_proposition(value)\n        candidate_tokens = tuple(\n            token for token in proposition_tokens(text)\n            if token not in ANSWER_SCAFFOLD and token not in CLAUSE_BREAKS\n        )\n        if text and candidate_tokens and candidate_tokens not in seen:\n            seen.add(candidate_tokens)\n            candidates.append((text, candidate_tokens))\n    candidates.sort(key=lambda item: len(item[1]), reverse=True)\n    matches: list[tuple[str, int, int, bool]] = []\n    for text, candidate_tokens in candidates:\n        for start, end in _find_spans(tokens, candidate_tokens):\n            matches.append((text, start, end, _span_is_denied(tokens, start, end)))\n    if not matches:\n        return AnswerStance("unmatched", "", 0, 0, False)\n\n    denied = sum(item[3] for item in matches)\n    correction_positions = [\n        index for index, token in enumerate(tokens)\n        if token in CONTRAST_TOKENS or token in ALTERNATIVE_TOKENS\n    ]\n    correction_ambiguous = False\n    for _, start, end, is_denied in matches:\n        del start\n        if is_denied:\n            continue\n        for marker in correction_positions:\n            if marker >= end and _meaningful_tail(tokens, marker + 1):\n                correction_ambiguous = True\n                break\n        if correction_ambiguous:\n            break\n\n    if denied == len(matches):\n        status = "denied"\n    elif denied or correction_ambiguous:\n        status = "ambiguous"\n    else:\n        status = "supported"\n    return AnswerStance(\n        status=status,\n        matched_answer=matches[0][0],\n        match_count=len(matches),\n        denied_match_count=denied,\n        correction_or_alternative_present=correction_ambiguous,\n    )\n\n\ndef numeric_response_supports(\n    response: object, expected: float, *, tolerance: float = 0.02,\n) -> bool:\n    """Admit one unambiguous affirmative numeric proposition."""\n\n    tokens = proposition_tokens(response)\n    numeric: list[tuple[int, float]] = []\n    for index, token in enumerate(tokens):\n        if NUMBER_TOKEN_RE.fullmatch(token):\n            try:\n                value = float(token.replace(",", ""))\n            except ValueError:\n                continue\n            if math.isfinite(value):\n                numeric.append((index, value))\n    if not numeric:\n        return False\n\n    bound = max(float(tolerance), abs(float(expected)) * 1e-4)\n    expected_spans = [\n        (index, index + 1) for index, value in numeric\n        if abs(value - float(expected)) <= bound\n    ]\n    if not expected_spans:\n        return False\n    if any(_span_is_denied(tokens, start, end) for start, end in expected_spans):\n        return False\n\n    # A second, materially different numeric answer makes the response\n    # nonterminal.  This catches alternatives and corrections while allowing\n    # harmless repeated rendering of the same value.\n    if any(abs(value - float(expected)) > bound for _, value in numeric):\n        return False\n    if any(\n        token in CONTRAST_TOKENS or token in ALTERNATIVE_TOKENS\n        for token in tokens\n    ):\n        return False\n    return True\n', '49997f36924f2b0bffa9bbf2a845b80b19cf78a48039cf041fa03ed7dc878f4f'), 'pipeline/phase2_retrieval_lane_policy.py': (b'"""Hash-bound separation of contextual and closed-book retrieval lanes.\n\nThe competition has two different label semantics.  A row that supplies a\ncontext must be judged only from that context (including deterministic\nderivations from rules, dates, quantities, units, and relationships stated in\nit).  World knowledge or an offline corpus may never rescue such a row.  A\nrow without supplied context belongs to the separate closed-book lane, where\noffline retrieval remains allowed.\n\nEvery contextual structured artifact embeds a :class:`RetrievalLaneContract`.\nThe policy and contract hashes make the boundary an executable runtime\ninvariant rather than a prompt convention.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom dataclasses import asdict, dataclass\nfrom typing import Any, Iterable\n\n\nVERSION = "phase2-retrieval-lane-policy-v1-context-isolation"\nCONTEXTUAL_LANE = "context_only"\nCLOSED_BOOK_LANE = "closed_book_retrieval"\n\nPOLICY = {\n    "version": VERSION,\n    "lane_selector": "context_available",\n    "contextual": {\n        "lane": CONTEXTUAL_LANE,\n        "external_retrieval_allowed": False,\n        "external_evidence_allowed": False,\n        "world_knowledge_rescue_allowed": False,\n        "internal_context_reasoning_allowed": True,\n    },\n    "closed_book": {\n        "lane": CLOSED_BOOK_LANE,\n        "external_retrieval_allowed": True,\n        "external_evidence_allowed": True,\n        "world_knowledge_rescue_allowed": True,\n        "internal_context_reasoning_allowed": False,\n    },\n}\n\n\ndef _sha256_json(value: object) -> str:\n    payload = json.dumps(\n        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),\n    )\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\nPOLICY_SHA256 = _sha256_json(POLICY)\n\n\n@dataclass(frozen=True)\nclass RetrievalLaneContract:\n    context_available: bool\n    lane: str\n    external_retrieval_allowed: bool\n    external_evidence_allowed: bool\n    world_knowledge_rescue_allowed: bool\n    internal_context_reasoning_allowed: bool\n    policy_version: str = VERSION\n    policy_sha256: str = POLICY_SHA256\n\n    @property\n    def contract_sha256(self) -> str:\n        return _sha256_json(asdict(self))\n\n    def to_dict(self) -> dict[str, Any]:\n        return {**asdict(self), "contract_sha256": self.contract_sha256}\n\n\ndef build_retrieval_lane_contract(\n    context_available: bool,\n) -> RetrievalLaneContract:\n    if type(context_available) is not bool:\n        raise ValueError("context_available must be an exact boolean")\n    spec = POLICY["contextual" if context_available else "closed_book"]\n    return RetrievalLaneContract(\n        context_available=context_available,\n        lane=str(spec["lane"]),\n        external_retrieval_allowed=bool(spec["external_retrieval_allowed"]),\n        external_evidence_allowed=bool(spec["external_evidence_allowed"]),\n        world_knowledge_rescue_allowed=bool(\n            spec["world_knowledge_rescue_allowed"]\n        ),\n        internal_context_reasoning_allowed=bool(\n            spec["internal_context_reasoning_allowed"]\n        ),\n    )\n\n\ndef validate_retrieval_lane_contract(value: object) -> RetrievalLaneContract:\n    if not isinstance(value, dict):\n        raise ValueError("retrieval lane contract must be an object")\n    raw = dict(value)\n    observed_sha = str(raw.pop("contract_sha256", ""))\n    expected = set(RetrievalLaneContract.__dataclass_fields__)\n    if set(raw) != expected:\n        raise ValueError("retrieval lane contract schema mismatch")\n    for field in (\n        "context_available", "external_retrieval_allowed",\n        "external_evidence_allowed", "world_knowledge_rescue_allowed",\n        "internal_context_reasoning_allowed",\n    ):\n        if type(raw.get(field)) is not bool:\n            raise ValueError(f"retrieval lane contract {field} must be boolean")\n    contract = RetrievalLaneContract(**raw)\n    if observed_sha != contract.contract_sha256:\n        raise ValueError("retrieval lane contract hash mismatch")\n    expected_contract = build_retrieval_lane_contract(\n        contract.context_available\n    )\n    if contract != expected_contract:\n        raise ValueError("retrieval lane contract violates the pinned policy")\n    return contract\n\n\ndef assert_contextual_no_external(\n    lane_contract: RetrievalLaneContract,\n    *,\n    external_retrieval_admitted: object = False,\n    query_plan: object = None,\n    external_candidates: Iterable[object] = (),\n    external_evidence: Iterable[object] = (),\n) -> None:\n    """Reject every outside-evidence surface for a supplied-context row.\n\n    Closed-book inputs intentionally pass through unchanged; their independent\n    retriever remains authorized by the closed-book lane contract.\n    """\n\n    expected = build_retrieval_lane_contract(lane_contract.context_available)\n    if lane_contract != expected:\n        raise ValueError("retrieval lane contract is not policy authoritative")\n    if not lane_contract.context_available:\n        return\n    if external_retrieval_admitted is not False:\n        raise ValueError("contextual external retrieval admission is forbidden")\n    if query_plan is not None:\n        raise ValueError("contextual query plans are forbidden")\n    if any(True for _ in external_candidates):\n        raise ValueError("contextual external retrieval candidates are forbidden")\n    if any(True for _ in external_evidence):\n        raise ValueError("contextual external evidence is forbidden")\n\n', '20ec445a84d3a6bc4f75d9720a1eb55d5efb42354bc8958888f9035ca56780e9'), 'pipeline/phase2_source_authority.py': (b'"""Hash-bound, semantic-alignment-first source authority policy."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\n\nROOT = Path(__file__).resolve().parents[1]\nPOLICY_PATH = ROOT / "pipeline/resources/phase2_source_authority_v1.json"\nEXPECTED_VERSION = "phase2-source-authority-policy-v1"\n\n\ndef sha256_file(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\nPOLICY_SHA256 = sha256_file(POLICY_PATH)\nPOLICY = json.loads(POLICY_PATH.read_text(encoding="utf-8"))\nif POLICY.get("version") != EXPECTED_VERSION:\n    raise ValueError("source authority policy version mismatch")\nif (POLICY.get("safety") or {}).get("semantic_alignment_precedes_authority") is not True:\n    raise ValueError("unsafe source authority decision order")\nif (POLICY.get("safety") or {}).get("wikipedia_terminal_authority") is not False:\n    raise ValueError("Wikipedia must remain nonterminal")\n\n_SOURCE_AUTHORITY: dict[str, dict[str, Any]] = {}\nfor tier in POLICY.get("tiers") or []:\n    authority_tier = int(tier["authority_tier"])\n    authority_class = str(tier["authority_class"])\n    for source_id in tier.get("source_ids") or []:\n        source_id = str(source_id)\n        if source_id in _SOURCE_AUTHORITY:\n            raise ValueError(f"duplicate source authority assignment: {source_id}")\n        _SOURCE_AUTHORITY[source_id] = {\n            "authority_tier": authority_tier,\n            "authority_class": authority_class,\n            "authority_policy_sha256": POLICY_SHA256,\n            "wikipedia_corroboration_only": authority_class\n            == "community_encyclopedia_corroboration",\n        }\n\n\ndef authority_for(source_id: object) -> dict[str, Any]:\n    key = str(source_id)\n    if key in _SOURCE_AUTHORITY:\n        return dict(_SOURCE_AUTHORITY[key])\n    default = POLICY["default"]\n    return {\n        "authority_tier": int(default["authority_tier"]),\n        "authority_class": str(default["authority_class"]),\n        "authority_policy_sha256": POLICY_SHA256,\n        "wikipedia_corroboration_only": False,\n    }\n\n\ndef semantic_alignment(candidate: dict[str, Any]) -> tuple[int, str]:\n    """Return a sortable alignment tier and an auditable status.\n\n    Source authority is intentionally evaluated only after claim alignment.\n    A bad book/OCR hit must not outrank a semantically aligned Wikipedia hit.\n    """\n\n    gate = candidate.get("model_facing_gate") or {}\n    if candidate.get("model_facing_eligible") is not True or gate.get("eligible") is False:\n        return 3, "quarantined"\n    if candidate.get("number_set_match") is False:\n        return 2, "number_mismatch"\n    if candidate.get("negation_set_match") is False:\n        return 2, "negation_mismatch"\n    query_types = set(gate.get("query_target_types") or [])\n    source_types = set(gate.get("source_target_types") or [])\n    if query_types and source_types and query_types.isdisjoint(source_types):\n        return 2, "answer_type_or_relation_mismatch"\n    if candidate.get("exact_normalized") is True:\n        return 0, "exact_aligned"\n    return 1, "fuzzy_or_passage_aligned"\n\n\ndef annotate_candidate(candidate: dict[str, Any]) -> dict[str, Any]:\n    value = dict(candidate)\n    authority = authority_for(value.get("source_id", ""))\n    alignment_tier, alignment_status = semantic_alignment(value)\n    value.update(authority)\n    value["semantic_alignment_tier"] = alignment_tier\n    value["semantic_alignment_status"] = alignment_status\n    if authority["wikipedia_corroboration_only"]:\n        if value.get("source_verdict_candidate") is not None:\n            raise ValueError("Wikipedia candidate attempted terminal source authority")\n        value["terminal_label_authority"] = False\n    return value\n\n\ndef candidate_sort_key(candidate: dict[str, Any]) -> tuple[Any, ...]:\n    value = annotate_candidate(candidate)\n    return (\n        int(value["semantic_alignment_tier"]),\n        int(value["authority_tier"]),\n        0 if value.get("exact_normalized") is True else 1,\n        int(value.get("rank", 10**9)),\n        str(value.get("source_id", "")),\n        int(value.get("source_record_index", -1)),\n        str(value.get("source_record_sha256", "")),\n    )\n\n\ndef annotate_and_rank(candidates: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:\n    annotated = [annotate_candidate(candidate) for candidate in candidates]\n    return sorted(annotated, key=candidate_sort_key)\n', 'c18d43dd073059e9c9258220a4306d344423b69f45f39e7c1f2f25c5dd7635d6'), 'pipeline/phase2_evidence_priority.py': (b'"""Policy-compatible, label-free evidence ranking for the Phase 2 hybrid RAG.\n\nThe ranker never consumes competition labels.  It first enforces the lane and\nthe requested Bengali grammar/lexical operation, then reuses the independently\nversioned source-authority policy.  This keeps a high-scoring synonym,\nWikipedia paragraph, or wrong grammar operation from displacing an attested\nexact pair/rule merely because it is lexically similar to the query.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom pipeline.phase2_canonicalize import canonicalize\nfrom pipeline.phase2_source_authority import annotate_candidate as annotate_authority\n\n\nROOT = Path(__file__).resolve().parents[1]\nPOLICY_PATH = ROOT / "pipeline/resources/phase2_evidence_priority_v2.json"\nPOLICY_SHA256 = hashlib.sha256(POLICY_PATH.read_bytes()).hexdigest()\nPOLICY = json.loads(POLICY_PATH.read_text(encoding="utf-8"))\nEXPECTED_VERSION = "phase2-evidence-priority-v2-exclusive-policy-first"\nif POLICY.get("version") != EXPECTED_VERSION:\n    raise ValueError("evidence priority policy version mismatch")\nif (POLICY.get("safety") or {}).get("competition_test_labels_may_train_ranker") is not False:\n    raise ValueError("competition labels must never train the evidence ranker")\n\nEXCLUSIVE_OPERATIONS = frozenset(POLICY["exclusive_operations"])\n\n_OPERATION_MARKERS: tuple[tuple[str, tuple[str, ...]], ...] = (\n    ("antonym_lookup", ("\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x80\xe0\xa6\xa4 \xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6", "\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x80\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\x95 \xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6", "\xe0\xa6\xb6\xe0\xa7\x81\xe0\xa6\xa6\xe0\xa7\x8d\xe0\xa6\xa7 \xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x80\xe0\xa6\xa4")),\n    ("idiom_meaning_lookup", ("\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbe", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa5", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xa6/\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbe")),\n    ("prefix_origin_classification", ("\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x97", "\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x97\xe0\xa6\x9f\xe0\xa6\xbf \xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xb6\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa7\x87\xe0\xa6\xa3\xe0\xa6\xbf\xe0\xa6\xb0")),\n    ("samas_taxonomy", ("\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x89\xe0\xa6\xa6\xe0\xa6\xbe\xe0\xa6\xb9\xe0\xa6\xb0\xe0\xa6\xa3")),\n    ("sandhi_formation", ("\xe0\xa6\xb8\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa7\xe0\xa6\xbf", "\xe0\xa6\xb8\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa7\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa7\x87")),\n)\n\n_DATE_MARKERS = re.compile(r"(?:\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96|\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2|\xe0\xa6\xac\xe0\xa6\x9b\xe0\xa6\xb0|\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87|\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8|\xe0\xa6\xa6\xe0\xa6\xbf\xe0\xa6\xa8|\xe0\xa6\x96\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6|\xe0\xa6\xb8\xe0\xa6\xa8\xe0\xa7\x87)")\n_UNIT_MARKERS = re.compile(\n    r"(?:\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0|\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0|\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0|\xe0\xa6\x95\xe0\xa7\x87\xe0\xa6\x9c\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xae|\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xae|\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\x95\xe0\xa7\x87\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa1|\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\xa8\xe0\xa6\xbf\xe0\xa6\x9f|\xe0\xa6\x98\xe0\xa6\xa3\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbe|"\n    r"\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe|\xe0\xa6\xb6\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb6|\xe0\xa6\xa1\xe0\xa6\xbf\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf|\xe0\xa6\xa8\xe0\xa6\xbf\xe0\xa6\x89\xe0\xa6\x9f\xe0\xa6\xa8|\xe0\xa6\x9c\xe0\xa7\x81\xe0\xa6\xb2|\xe0\xa6\x93\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbe\xe0\xa6\x9f|\xe0\xa6\x93\xe0\xa7\x9f\xe0\xa6\xbe\xe0\xa6\x9f)"\n)\n\n\ndef infer_operation(text: object) -> str:\n    normalized = canonicalize(str(text or ""))\n    for operation, markers in _OPERATION_MARKERS:\n        if any(marker in normalized for marker in markers):\n            return operation\n    return ""\n\n\ndef _declared_operation(candidate: dict[str, Any]) -> str:\n    for container in (\n        candidate,\n        candidate.get("model_facing_gate") or {},\n        candidate.get("source_metadata") or {},\n    ):\n        for key in ("policy_operation", "operation", "query_operation"):\n            value = str(container.get(key) or "")\n            if value in EXCLUSIVE_OPERATIONS:\n                return value\n    return ""\n\n\ndef _query_operation(candidate: dict[str, Any]) -> str:\n    declared = str(candidate.get("query_policy_operation") or "")\n    if declared in EXCLUSIVE_OPERATIONS:\n        return declared\n    return infer_operation(candidate.get("query_text", ""))\n\n\ndef _source_operation(candidate: dict[str, Any]) -> str:\n    declared = _declared_operation(candidate)\n    if declared:\n        return declared\n    return infer_operation("\\n".join((\n        str(candidate.get("question") or ""),\n        str(candidate.get("supporting_text") or ""),\n    )))\n\n\ndef policy_compatibility(candidate: dict[str, Any]) -> tuple[int, str]:\n    """Return a hard-before-authority compatibility tier and audit reason."""\n\n    if candidate.get("query_context_available") is True:\n        return 4, "contextual_external_retrieval_forbidden"\n    query_operation = _query_operation(candidate)\n    source_operation = _source_operation(candidate)\n    if query_operation:\n        if source_operation == query_operation:\n            return 0, f"exclusive_operation_exact:{query_operation}"\n        if source_operation:\n            return 4, f"exclusive_operation_mismatch:{query_operation}!={source_operation}"\n        # An untyped passage may still contain a useful attested pair/rule, but\n        # it cannot outrank an operation-exact source or become terminal proof.\n        return 2, f"exclusive_operation_untyped_source:{query_operation}"\n    if source_operation:\n        return 2, f"unsolicited_exclusive_operation:{source_operation}"\n    return 1, "general_factual_policy"\n\n\ndef slot_compatibility(candidate: dict[str, Any]) -> tuple[int, str]:\n    """Gate the factual slot before source reputation or retrieval score."""\n\n    gate = candidate.get("model_facing_gate") or {}\n    if candidate.get("model_facing_eligible") is not True or gate.get("eligible") is False:\n        return 4, "upstream_quarantine"\n    # A source-side item/page number (for example ``02`` in an OCR MCQ page)\n    # is not a factual-slot conflict when the query itself requests no number.\n    # Once the query contains a number/date, however, exact compatibility is a\n    # hard prerequisite and a mismatch is quarantined.\n    if (\n        candidate.get("number_set_match") is False\n        and (\n            "query_numbers" not in candidate\n            or bool(candidate.get("query_numbers"))\n        )\n    ):\n        return 3, "number_mismatch"\n    if candidate.get("negation_set_match") is False:\n        return 3, "negation_mismatch"\n    query_types = set(gate.get("query_target_types") or [])\n    source_types = set(gate.get("source_target_types") or [])\n    if query_types and source_types and query_types.isdisjoint(source_types):\n        return 3, "answer_type_or_relation_mismatch"\n    query = canonicalize(str(candidate.get("query_text") or ""))\n    source = canonicalize("\\n".join((\n        str(candidate.get("question") or ""),\n        str(candidate.get("supporting_text") or ""),\n    )))\n    if _DATE_MARKERS.search(query) and not _DATE_MARKERS.search(source):\n        return 2, "date_slot_unconfirmed"\n    query_units, source_units = set(_UNIT_MARKERS.findall(query)), set(_UNIT_MARKERS.findall(source))\n    if query_units and source_units and query_units.isdisjoint(source_units):\n        return 3, "unit_mismatch"\n    return 0, "slot_compatible"\n\n\ndef annotate_candidate(candidate: dict[str, Any]) -> dict[str, Any]:\n    value = annotate_authority(candidate)\n    policy_tier, policy_status = policy_compatibility(value)\n    slot_tier, slot_status = slot_compatibility(value)\n    value["policy_compatibility_tier"] = policy_tier\n    value["policy_compatibility_status"] = policy_status\n    value["slot_compatibility_tier"] = slot_tier\n    value["slot_compatibility_status"] = slot_status\n    value["query_policy_operation"] = _query_operation(value)\n    value["source_policy_operation"] = _source_operation(value)\n    value["evidence_priority_policy_sha256"] = POLICY_SHA256\n    if policy_tier >= 4 or slot_tier >= 3:\n        value["model_facing_eligible"] = False\n        gate = dict(value.get("model_facing_gate") or {})\n        gate["eligible"] = False\n        reasons = set(gate.get("reasons") or [])\n        reasons.update((policy_status, slot_status))\n        gate["reasons"] = sorted(reasons)\n        value["model_facing_gate"] = gate\n        value["source_verdict_candidate"] = None\n        value["terminal_label_authority"] = False\n    elif policy_tier == 2:\n        # Useful for model inspection, never sufficient for exact lexical or\n        # grammar proof when the source operation itself is untyped.\n        value["source_verdict_candidate"] = None\n        value["terminal_label_authority"] = False\n    return value\n\n\ndef candidate_sort_key(candidate: dict[str, Any]) -> tuple[Any, ...]:\n    value = annotate_candidate(candidate)\n    relation = str(value.get("evidence_role") or value.get("response_answer_relation") or "none")\n    # Support and counter are deliberately tied.  A response-aware ranker must\n    # not favor agreement over contradiction; the bounded selector reserves\n    # both roles independently.\n    relation_tier = 0 if relation in {\n        "support_candidate", "counter_candidate", "exact", "containment"\n    } else 1\n    return (\n        int(value["policy_compatibility_tier"]),\n        int(value["slot_compatibility_tier"]),\n        int(value["semantic_alignment_tier"]),\n        relation_tier,\n        int(value["authority_tier"]),\n        0 if value.get("exact_normalized") is True else 1,\n        int(value.get("rank", 10**9)),\n        str(value.get("source_id", "")),\n        int(value.get("source_record_index", -1)),\n        str(value.get("source_record_sha256", "")),\n    )\n\n\ndef annotate_and_rank(candidates: Iterable[dict[str, Any]]) -> list[dict[str, Any]]:\n    annotated = [annotate_candidate(candidate) for candidate in candidates]\n    return sorted(annotated, key=candidate_sort_key)\n', '346ae7c0dec008c99e24b6e9dfe2dcf32b29e0f1883f217b882abeab5c381809'), 'pipeline/phase2_sparse_retrieval.py': (b'"""Offline, hash-gated sparse retrieval for arbitrary Phase 2 inputs.\n\nThe stage consumes only label-free formatted JSONL.  It produces evidence and\nconservative source-verdict *candidates*; fuzzy retrieval never becomes a\nterminal label.  Existing Bangla-MMLU and NCTB-QA indexes are reused rather than\nrebuilt inside the Kaggle kernel.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport sys\nimport time\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom difflib import SequenceMatcher\nfrom pathlib import Path\nfrom typing import Any\n\nimport joblib\nimport numpy as np\nfrom scipy import sparse\n\n\nROOT = Path(__file__).resolve().parents[1]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom pipeline.phase2_canonicalize import (  # noqa: E402\n    VERSION as CANONICALIZER_VERSION,\n    canonicalize,\n)\nfrom pipeline.phase2_mmap_retrieval import (  # noqa: E402\n    NORMALIZER as MMAP_NORMALIZER,\n    VERSION as MMAP_CACHE_VERSION,\n    close_mmap_index,\n    load_mmap_index,\n)\nfrom pipeline.phase2_composite_fts_retrieval import (  # noqa: E402\n    VERSION as COMPOSITE_FTS_RUNTIME_VERSION,\n    load as load_composite_fts,\n    retrieve_authority_tier_pool_optimized,\n)\nfrom pipeline.phase2_response_proposition import classify_answer_stance  # noqa: E402\nfrom pipeline.phase2_evidence_priority import (  # noqa: E402\n    POLICY_PATH as SOURCE_AUTHORITY_POLICY_PATH,\n    POLICY_SHA256 as SOURCE_AUTHORITY_POLICY_SHA256,\n    annotate_and_rank as authority_rank_candidates,\n)\nfrom pipeline.phase2_retrieval_lane_policy import (  # noqa: E402\n    POLICY_SHA256 as RETRIEVAL_LANE_POLICY_SHA256,\n)\n\n\nVERSION = "phase2-sparse-retrieval-v29-local-supporting-fact-windows"\nSTRICT_EXACT_CONFLICT_POLICY = "same_choice_set_answer_disagreement_quarantines"\nCOMPOSITE_QUERY_MODES = {"all_closed", "unresolved_only", "residual_only"}\nRETRIEVAL_MODES = {"standard", "exact_only", "exact_then_keyword_fallback"}\nKEYWORD_FALLBACK_MAX_DISTINCT_SOURCES = 2\nSUPPORTING_FACT_MAX_WINDOW_CHARS = 480\n\n# Bengali count answers are commonly written both joined and separated\n# (``\xe0\xa7\xa7\xe0\xa7\xaf\xe0\xa6\x9f\xe0\xa6\xbf`` / ``\xe0\xa7\xa7\xe0\xa7\xaf \xe0\xa6\x9f\xe0\xa6\xbf``).  The general canonicalizer deliberately preserves\n# token boundaries, so normalize only the narrow numeral+counter boundary here.\n# This must not become a global whitespace-strip: doing that would silently\n# merge names, dates, units, or multi-token alternatives.\nNUMERIC_COUNTER_SPACE_RE = re.compile(\n    r"(?<=[0-9\xe0\xa7\xa6-\xe0\xa7\xaf])\\s+(?=(?:\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x9f\xe0\xa6\xbe|\xe0\xa6\x9c\xe0\xa6\xa8|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xbe|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xbf)(?:\\s|$))"\n)\n\n\ndef preferred_cache(directory: Path) -> Path:\n    """Prefer a cache built for the current runtime/canonicalizer identity.\n\n    ``_mmap_v2`` is checked first.  A historically named ``_mmap_v1`` shard is\n    accepted only if its *manifest* declares the current cache and\n    canonicalizer identity; the directory suffix is never treated as proof of\n    compatibility.  Legacy joblib/NPZ indexes remain the fail-closed fallback.\n    """\n\n    # This runs while module defaults are initialized, before the streaming\n    # ``sha256_file`` helper below is defined.  The canonicalizer module is\n    # small, so a direct digest is both safe and deterministic here.\n    normalizer_sha256 = hashlib.sha256(MMAP_NORMALIZER.read_bytes()).hexdigest()\n    for suffix in ("_mmap_v2", "_mmap_v1"):\n        candidate = directory.with_name(directory.name + suffix)\n        manifest_path = candidate / "manifest.json"\n        if not manifest_path.is_file():\n            continue\n        try:\n            manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))\n        except (OSError, ValueError, TypeError):\n            continue\n        fingerprint = manifest.get("fingerprint") or {}\n        if (\n            manifest.get("version") == MMAP_CACHE_VERSION\n            and fingerprint.get("canonicalizer_version") == CANONICALIZER_VERSION\n            and fingerprint.get("normalizer_sha256") == normalizer_sha256\n        ):\n            return candidate\n    return directory\n\n\nDEFAULT_INDEX_SPECS = [\n    {\n        "source_id": "bangla_mmlu",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "bangla_mmlu_char_v1"),\n        "rights_note": "dataset card license was not declared; local evidence use only until redistribution review",\n        "terminal_policy": "exact question + duplicate keyed-answer consensus + response exact keyed option",\n    },\n    {\n        "source_id": "nctb_qa_87805",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "nctb_qa_87805_char_v1"),\n        "rights_note": "CC BY-NC 4.0 snapshot; preserve attribution and noncommercial restriction",\n        "terminal_policy": "exact question and positive answer alignment only; mismatch never implies false",\n    },\n    {\n        "source_id": "nctb_education_aux",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "nctb_education_aux_char_v1"),\n        "rights_note": "mixed provenance: NCTBench CC BY 4.0 and SOMAJGYAAN MIT may be terminal under exact policy; BEnQA/SSC-BanglaTutor remain local corroboration only",\n        "terminal_policy": "record-level exact consensus policy; fuzzy retrieval and rights-unclear rows are never terminal",\n    },\n    {\n        "source_id": "downloads_bcs_10_50",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "downloads_bcs_10_50_char_v1"),\n        "rights_note": "user-supplied local BCS PDFs with PDF/page/raw-block provenance",\n        "terminal_policy": "model-facing exact/fuzzy evidence only; OCR-derived answers never auto-terminal",\n    },\n    {\n        "source_id": "nctb_schooltext",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "nctb_schooltext_word_v1"),\n        "rights_note": "CC BY 4.0 processed corpus; preserve attribution and embedded NCTB source-rights caveat",\n        "terminal_policy": "model-facing passage retrieval only; no answer labels",\n    },\n    {\n        "source_id": "joykoli_six_part",\n        "directory": preferred_cache(ROOT / "artifacts" / "retrieval" / "joykoli_six_part_char_v3"),\n        "rights_note": "user-provided private competition-use authorization; derived cache only; no public redistribution",\n        "terminal_policy": "closed-book corroboration only; never promotes a verdict",\n        "exact_conflict_policy": STRICT_EXACT_CONFLICT_POLICY,\n    },\n]\nFORBIDDEN_FIELDS = {"gold", "gold_label", "label", "target", "is_hallucination"}\nNUMBER_RE = re.compile(r"(?<!\\w)\\d+(?:[.,]\\d+)?(?!\\w)")\nNEGATIONS = ("\xe0\xa6\xa8\xe0\xa6\xbe", "\xe0\xa6\xa8\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xa8\xe0\xa7\x9f", "\xe0\xa6\xa8\xe0\xa7\x87\xe0\xa6\x87", "\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xa4\xe0\xa7\x80\xe0\xa6\xa4", "\xe0\xa6\x9b\xe0\xa6\xbe\xe0\xa6\xa1\xe0\xa6\xbc\xe0\xa6\xbe", "\xe0\xa6\x9b\xe0\xa6\xbe\xe0\xa7\x9c\xe0\xa6\xbe", "\xe0\xa6\xad\xe0\xa7\x81\xe0\xa6\xb2")\nFUZZY_STOPWORDS = {\n    "\xe0\xa6\x95\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa7\x80", "\xe0\xa6\x95\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb0", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa6\xa4", "\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f",\n    "\xe0\xa6\xa8\xe0\xa6\xbf\xe0\xa6\x9a\xe0\xa7\x87\xe0\xa6\xb0", "\xe0\xa6\xa8\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xa8\xe0\xa7\x87\xe0\xa6\xb0", "\xe0\xa6\xb8\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\x95", "\xe0\xa6\x89\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xb0", "\xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb9\xe0\xa7\x9f", "\xe0\xa6\x9b\xe0\xa6\xbf\xe0\xa6\xb2", "\xe0\xa6\x86\xe0\xa6\x9b\xe0\xa7\x87", "\xe0\xa6\x8f\xe0\xa6\x9f\xe0\xa6\xbf",\n    "\xe0\xa6\x8f\xe0\xa6\x87", "\xe0\xa6\x8f\xe0\xa6\x95\xe0\xa6\x9f\xe0\xa6\xbf", "the", "a", "an", "is", "was", "what", "which", "who", "how",\n}\n\n# Small, auditable equivalence families used only for sparse discovery and\n# nonterminal overlap accounting.  They never rewrite canonical exact keys.\n# Property aliases are deliberately limited to physical-length contexts by\n# ``_bounded_fact_family_enabled``; this is not a general synonym system.\nBOUNDED_FACT_ALIAS_FAMILIES: dict[str, frozenset[str]] = {\n    "bn_river_case": frozenset({"\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80", "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0"}),\n    "bn_length_property": frozenset({"\xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98"}),\n    "bn_kilometre_unit": frozenset({"\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0"}),\n    "en_length_property": frozenset({"length", "long"}),\n    "en_kilometre_unit": frozenset({\n        "km", "kilometer", "kilometers", "kilometre", "kilometres",\n    }),\n}\nLENGTH_PROPERTY_FAMILIES = frozenset({\n    "bn_length_property", "en_length_property",\n})\nLENGTH_CONTEXT_TOKENS = frozenset({\n    "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80", "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0", "\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80", "\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2", "\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6", "\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0", "\xe0\xa6\xae\xe0\xa6\xb9\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0",\n    "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0", "river", "lake", "ocean", "sea", "canal",\n    "km", "kilometer", "kilometers", "kilometre", "kilometres",\n})\nFACT_RELATION_TOKENS = frozenset().union(*BOUNDED_FACT_ALIAS_FAMILIES.values())\n\n# Keyword retrieval is deliberately a nonterminal *hint* lane.  Structured\n# question neighbours use the relation/slot gates below; source passages use a\n# stricter query-containment/coverage gate.  Neither path proves a fact.  The\n# cue families are intentionally small and deterministic.  Unknown wording\n# abstains instead of being silently coerced into a known fact family.\nKEYWORD_PREDICATE_CUES: dict[str, tuple[str, ...]] = {\n    "length": ("\xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98", "length", "long"),\n    "completion": (\n        "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xa8\xe0\xa6\xbe \xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa8", "\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\x96\xe0\xa6\xbe \xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa8", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa8 \xe0\xa6\x95\xe0\xa6\xb0\xe0\xa7\x87\xe0\xa6\xa8", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xa4 \xe0\xa6\x95\xe0\xa6\xb0\xe0\xa7\x87\xe0\xa6\xa8",\n        "\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7 \xe0\xa6\x95\xe0\xa6\xb0\xe0\xa7\x87\xe0\xa6\xa8", "completed", "finished writing",\n    ),\n    "publication": (\n        "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb6\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb6 \xe0\xa6\x95\xe0\xa6\xb0\xe0\xa6\xbe", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb6 \xe0\xa6\xaa\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb6 \xe0\xa6\xaa\xe0\xa6\xbe\xe0\xa7\x9f",\n        "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb6\xe0\xa6\xa8\xe0\xa6\xbe", "published", "publication",\n    ),\n    "birth": ("\xe0\xa6\x9c\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xae", "born"),\n    "death": ("\xe0\xa6\xae\xe0\xa7\x83\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa7\x81", "\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbe \xe0\xa6\xaf\xe0\xa6\xbe\xe0\xa6\xa8", "died", "death"),\n    "establishment": (\n        "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\xa0\xe0\xa6\xbe", "\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaa\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\x97\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\xa4", "founded", "established",\n    ),\n    "inauguration": ("\xe0\xa6\x89\xe0\xa6\xa6\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa7\x8b\xe0\xa6\xa7\xe0\xa6\xa8", "\xe0\xa6\x9a\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x81 \xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\x9a\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x81 \xe0\xa6\xb9\xe0\xa7\x9f", "inaugurated", "launched"),\n    "signing": ("\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xb7\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xb7\xe0\xa6\xb0", "signed", "signing"),\n    "ratification": ("\xe0\xa6\x85\xe0\xa6\xa8\xe0\xa7\x81\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xa8", "ratified", "ratification"),\n    "start": ("\xe0\xa6\xb6\xe0\xa7\x81\xe0\xa6\xb0\xe0\xa7\x81", "\xe0\xa6\x86\xe0\xa6\xb0\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xad", "\xe0\xa6\xb8\xe0\xa7\x82\xe0\xa6\x9a\xe0\xa6\xa8\xe0\xa6\xbe", "began", "started"),\n    "end": ("\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7 \xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7 \xe0\xa6\xb9\xe0\xa7\x9f", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xa4 \xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xa4 \xe0\xa6\xb9\xe0\xa7\x9f", "ended"),\n    "authorship": ("\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe", "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa7\x9f\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe", "\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\x96\xe0\xa6\x95", "\xe0\xa6\x95\xe0\xa7\x87 \xe0\xa6\xb2\xe0\xa6\xbf\xe0\xa6\x96", "who wrote", "author"),\n    "location": ("\xe0\xa6\x85\xe0\xa6\xac\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f", "located", "where"),\n}\n\nKEYWORD_QUALIFIER_CUES: dict[str, dict[str, tuple[str, ...]]] = {\n    "geographic_scope": {\n        "within_bangladesh": (\n            "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\x82\xe0\xa6\xb6", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\x82\xe0\xa6\xb6\xe0\xa7\x87",\n            "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6 \xe0\xa6\x85\xe0\xa6\x82\xe0\xa6\xb6\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\xad\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa6\xb0\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\xad\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xb0\xe0\xa7\x87",\n            "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\xae\xe0\xa6\xa7\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa7\x87", "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87", "\xe0\xa6\x8f\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87", "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\x82\xe0\xa6\xb6",\n            "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\x82\xe0\xa6\xb6\xe0\xa7\x87", "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\xad\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa6\xb0\xe0\xa7\x87", "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\x85\xe0\xa6\xad\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xb0\xe0\xa7\x87", "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0 \xe0\xa6\xae\xe0\xa6\xa7\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa7\x87",\n            "within bangladesh", "in bangladesh",\n        ),\n        "total": (\n            "\xe0\xa6\xae\xe0\xa7\x8b\xe0\xa6\x9f \xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0 \xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xaa\xe0\xa7\x82\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa3 \xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xaa\xe0\xa7\x81\xe0\xa6\xb0\xe0\xa7\x8b \xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf",\n            "total length", "overall length", "full length", "entire length",\n        ),\n    },\n    "scope": {\n        "exhaustive": (\n            "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8", "\xe0\xa6\xb8\xe0\xa6\x95\xe0\xa6\xb2", "\xe0\xa6\xb8\xe0\xa6\xac\xe0\xa6\x97\xe0\xa7\x81\xe0\xa6\xb2\xe0\xa7\x8b", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa4", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa7\x87\xe0\xa6\x95", "\xe0\xa6\x9c\xe0\xa7\x87\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x82\xe0\xa6\xb9",\n            "all districts", "which districts", "every",\n        ),\n        "single": ("\xe0\xa6\x8f\xe0\xa6\x95\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xb0", "\xe0\xa6\xb6\xe0\xa7\x81\xe0\xa6\xa7\xe0\xa7\x81 \xe0\xa6\x8f\xe0\xa6\x95\xe0\xa6\x9f\xe0\xa6\xbf", "only one", "sole"),\n    },\n    "protected_area": {\n        "protected_area": (\n            "\xe0\xa6\x85\xe0\xa6\xad\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xa3\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\x85\xe0\xa6\xad\xe0\xa7\x9f\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xa3\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xac\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xa3\xe0\xa7\x80", "\xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\xb0\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xb7\xe0\xa6\xbf\xe0\xa6\xa4 \xe0\xa6\xac\xe0\xa6\xa8",\n            "sanctuary", "wildlife reserve", "protected area",\n        ),\n    },\n    "adolescent_gender": {\n        "female": ("\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb6\xe0\xa7\x8b\xe0\xa6\xb0\xe0\xa7\x80", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa6\xbf\xe0\xa6\x95\xe0\xa6\xbe", "adolescent girl", "teenage girl"),\n        "male": ("\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb6\xe0\xa7\x8b\xe0\xa6\xb0", "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa6\x95", "adolescent boy", "teenage boy"),\n    },\n    "ordinal": {\n        "first": ("\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa5\xe0\xa6\xae", "\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa5\xe0\xa6\xae", "first"),\n        "last": ("\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7", "\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7", "last", "latest"),\n        "largest": ("\xe0\xa6\xac\xe0\xa7\x83\xe0\xa6\xb9\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xae", "\xe0\xa6\xb8\xe0\xa6\xac\xe0\xa6\x9a\xe0\xa7\x87\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa7\x87 \xe0\xa6\xac\xe0\xa6\xa1\xe0\xa6\xbc", "\xe0\xa6\xb8\xe0\xa6\xac\xe0\xa6\x9a\xe0\xa7\x87\xe0\xa7\x9f\xe0\xa7\x87 \xe0\xa6\xac\xe0\xa7\x9c", "largest"),\n        "smallest": ("\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xb7\xe0\xa7\x81\xe0\xa6\xa6\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa6\xae", "\xe0\xa6\xb8\xe0\xa6\xac\xe0\xa6\x9a\xe0\xa7\x87\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa7\x87 \xe0\xa6\x9b\xe0\xa7\x8b\xe0\xa6\x9f", "\xe0\xa6\xb8\xe0\xa6\xac\xe0\xa6\x9a\xe0\xa7\x87\xe0\xa7\x9f\xe0\xa7\x87 \xe0\xa6\x9b\xe0\xa7\x8b\xe0\xa6\x9f", "smallest"),\n    },\n}\n\nDATE_MONTH_CUES = (\n    "\xe0\xa6\x9c\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa7\x81\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf", "\xe0\xa6\x9c\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa7\x81\xe0\xa7\x9f\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf", "\xe0\xa6\xab\xe0\xa7\x87\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa7\x81\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf", "\xe0\xa6\xab\xe0\xa7\x87\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa7\x81\xe0\xa7\x9f\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf", "\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x9a", "\xe0\xa6\x8f\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\xb2",\n    "\xe0\xa6\xae\xe0\xa7\x87", "\xe0\xa6\x9c\xe0\xa7\x81\xe0\xa6\xa8", "\xe0\xa6\x9c\xe0\xa7\x81\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\x87", "\xe0\xa6\x86\xe0\xa6\x97\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\x9f", "\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa7\x87\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xb0", "\xe0\xa6\x85\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa7\x8b\xe0\xa6\xac\xe0\xa6\xb0", "\xe0\xa6\xa8\xe0\xa6\xad\xe0\xa7\x87\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xb0",\n    "\xe0\xa6\xa1\xe0\xa6\xbf\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xb0", "january", "february", "march", "april", "may", "june",\n    "july", "august", "september", "october", "november", "december",\n)\nKEYWORD_EXACT_ONLY_OPERATION_RE = re.compile(\n    r"(?:\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbe|\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xa6|\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x80\xe0\xa6\xa4(?:\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\x95)?\\s*\xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6|\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x97|\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xaf\xe0\xa6\xbc|\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa7\x9f|"\n    r"\xe0\xa6\xb8\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa7\xe0\xa6\xbf|\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8|\xe0\xa6\x97\xe0\xa7\x81\xe0\xa6\xb0\xe0\xa7\x81\xe0\xa6\x9a\xe0\xa6\xa3\xe0\xa7\x8d\xe0\xa6\xa1\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x80|\xe0\xa6\x97\xe0\xa7\x81\xe0\xa6\xb0\xe0\xa7\x81\xe0\xa6\x9a\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa1\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x80|\xe0\xa6\xa3\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xac|\xe0\xa6\xb7\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xac|\xe0\xa6\xb6\xe0\xa7\x81\xe0\xa6\xa6\xe0\xa7\x8d\xe0\xa6\xa7\\s*\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xa8)"\n)\n\n# These are interrogative/relation/type words, not a list of named entities.\n# The first remaining ordered token is used as a conservative subject anchor\n# for noisy, non-exact Joykoli evidence.  This also covers concepts such as\n# ``\xe0\xa6\xa8\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa7\x9f\xe0\xa6\xa8\xe0\xa6\xaf\xe0\xa7\x8b\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xaf``: an anchor need not be a person\'s or place\'s name.\nANCHOR_GENERIC_TERMS = {\n    "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa5\xe0\xa6\xae", "\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xa8", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\x9a\xe0\xa7\x80\xe0\xa6\xa8", "\xe0\xa6\xac\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xa8", "\xe0\xa6\xac\xe0\xa7\x83\xe0\xa6\xb9\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xae", "\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xb7\xe0\xa7\x81\xe0\xa6\xa6\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa6\xae",\n    "\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa5\xe0\xa6\xae", "\xe0\xa6\xb8\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb7", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xae", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xae\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x9a\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\x85\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xad\xe0\xa7\x81\xe0\xa6\x95\xe0\xa7\x8d\xe0\xa6\xa4", "\xe0\xa6\x85\xe0\xa6\xac\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xbf\xe0\xa6\xa4",\n    "\xe0\xa6\x9b\xe0\xa6\xbf\xe0\xa6\xb2", "\xe0\xa6\xb9\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb9\xe0\xa7\x9f", "\xe0\xa6\x86\xe0\xa6\x9b\xe0\xa7\x87", "\xe0\xa6\xb0\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa7\x87\xe0\xa6\x9b\xe0\xa7\x87", "\xe0\xa6\xb0\xe0\xa7\x9f\xe0\xa7\x87\xe0\xa6\x9b\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa6\xb0\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa6\xb0\xe0\xa7\x87\xe0\xa6\xa8", "\xe0\xa6\xb9\xe0\xa6\xbf\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xac\xe0\xa7\x87",\n    "\xe0\xa6\x89\xe0\xa6\xa6\xe0\xa6\xbe\xe0\xa6\xb9\xe0\xa6\xb0\xe0\xa6\xa3", "\xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\x95\xe0\xa7\x87\xe0\xa6\xa4", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa7\x9f\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xb2\xe0\xa6\xbe", "\xe0\xa6\xac\xe0\xa6\xb2\xe0\xa7\x87",\n    "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xbf\xe0\xa6\xa4", "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xa8\xe0\xa6\xbe", "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe", "\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa7\x9f\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe", "\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\x96\xe0\xa6\x95", "\xe0\xa6\x95\xe0\xa6\xac\xe0\xa6\xbf", "\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbe",\n    "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6", "\xe0\xa6\x9c\xe0\xa7\x87\xe0\xa6\xb2\xe0\xa6\xbe", "\xe0\xa6\xb6\xe0\xa6\xb9\xe0\xa6\xb0", "\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xae", "\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\x9c\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa7\x80", "\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xa8", "\xe0\xa6\x9c\xe0\xa6\xa8\xe0\xa6\xaa\xe0\xa6\xa6", "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80",\n    "\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80", "\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2", "\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6", "\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0", "\xe0\xa6\xae\xe0\xa6\xb9\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0", "\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xac\xe0\xa6\xa4", "\xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa5", "\xe0\xa6\xac\xe0\xa6\x87",\n    "\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\x95\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe", "\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xbe\xe0\xa6\xb8", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\x9f\xe0\xa6\x95", "\xe0\xa6\x97\xe0\xa6\xbe\xe0\xa6\xa8", "\xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6", "\xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6\xe0\xa6\xa6\xe0\xa7\x81\xe0\xa6\x9f\xe0\xa6\xbf",\n    "\xe0\xa6\xac\xe0\xa6\x9b\xe0\xa6\xb0", "\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2", "\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x9f", "\xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\x96\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xbe", "\xe0\xa6\x87\xe0\xa6\x9e\xe0\xa7\x8d\xe0\xa6\x9c\xe0\xa6\xbf\xe0\xa6\xa8", "\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x9a",\n    "\xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf", "\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa6\xbf", "\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0", "length", "long", "km",\n    "kilometer", "kilometers", "kilometre", "kilometres",\n    "\xe0\xa6\x95\xe0\xa6\xa4", "\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f", "which", "what", "who", "where", "how",\n    "when", "name", "author", "wrote", "first", "last", "main", "ancient",\n}\nANCHOR_INFLECTION_SUFFIXES = {\n    "\xe0\xa6\xb0", "\xe0\xa6\x8f\xe0\xa6\xb0", "\xe0\xa7\x87\xe0\xa6\xb0", "\xe0\xa6\x95\xe0\xa7\x87", "\xe0\xa6\xa4\xe0\xa7\x87", "\xe0\xa7\x87", "\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa7\x9f", "\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\x9f\xe0\xa6\xbe", "\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0",\n    "\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb0", "\xe0\xa6\x97\xe0\xa7\x81\xe0\xa6\xb2\xe0\xa7\x8b", "\xe0\xa6\x97\xe0\xa7\x81\xe0\xa6\xb2\xe0\xa6\xbf", "\xe0\xa6\xa6\xe0\xa7\x81\xe0\xa6\x9f\xe0\xa6\xbf",\n}\n\n\ndef sha256_file(path: Path) -> str:\n    value = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef sha256_json(value: object) -> str:\n    payload = json.dumps(\n        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")\n    )\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef id_set_sha256(values: set[str]) -> str:\n    """Hash an unordered ID set without depending on input order."""\n\n    payload = "".join(f"{value}\\n" for value in sorted(values))\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\ndef walk_keys(value: Any):\n    if isinstance(value, dict):\n        for key, child in value.items():\n            yield str(key).casefold()\n            yield from walk_keys(child)\n    elif isinstance(value, list):\n        for child in value:\n            yield from walk_keys(child)\n\n\ndef unique(values: list[str]) -> list[str]:\n    return list(dict.fromkeys(value for value in values if value))\n\n\ndef answer_equivalence_key(value: object) -> str:\n    """Canonical answer key with bounded Bengali numeral-counter joining."""\n\n    return NUMERIC_COUNTER_SPACE_RE.sub("", canonicalize(value))\n\n\ndef number_set(value: object) -> list[str]:\n    return sorted(set(NUMBER_RE.findall(canonicalize(value))))\n\n\ndef negation_set(value: object) -> list[str]:\n    normalized = canonicalize(value)\n    tokens = set(normalized.split())\n    return sorted(word for word in NEGATIONS if word in tokens)\n\n\ndef substantive_token_set(value: object) -> set[str]:\n    return {\n        token\n        for token in canonicalize(value).split()\n        if len(token) >= 3 and token not in FUZZY_STOPWORDS and not token.isdigit()\n    }\n\n\ndef _bounded_fact_family_enabled(\n    family: str, query_tokens: set[str], source_tokens: set[str] | None = None,\n) -> bool:\n    """Keep length/long aliases inside an explicit distance context."""\n\n    if family not in LENGTH_PROPERTY_FAMILIES:\n        return True\n    context = set(query_tokens)\n    if source_tokens:\n        context.update(source_tokens)\n    return bool(context.intersection(LENGTH_CONTEXT_TOKENS))\n\n\ndef bounded_sparse_discovery_query(value: object) -> str:\n    """Append bounded aliases once for one sparse transform.\n\n    The returned text is for TF-IDF discovery only.  Exact lookup and the\n    ``query_key`` supplied to candidate construction continue to use the\n    unexpanded canonical string.\n    """\n\n    normalized = canonicalize(value)\n    tokens = set(normalized.split())\n    additions: list[str] = []\n    for family, aliases in BOUNDED_FACT_ALIAS_FAMILIES.items():\n        if not tokens.intersection(aliases):\n            continue\n        if not _bounded_fact_family_enabled(family, tokens):\n            continue\n        additions.extend(sorted(aliases - tokens))\n    return " ".join([normalized, *unique(additions)]).strip()\n\n\ndef _bounded_fact_alias_match(\n    query_token: str, source_token: str,\n    query_tokens: set[str], source_tokens: set[str],\n) -> str | None:\n    for family, aliases in BOUNDED_FACT_ALIAS_FAMILIES.items():\n        if query_token not in aliases or source_token not in aliases:\n            continue\n        if _bounded_fact_family_enabled(family, query_tokens, source_tokens):\n            return f"bounded_fact_alias:{family}"\n    return None\n\n\ndef _generic_anchor_term(token: str) -> bool:\n    if token in FUZZY_STOPWORDS or token in ANCHOR_GENERIC_TERMS:\n        return True\n    for stem in ANCHOR_GENERIC_TERMS:\n        if token.startswith(stem) and token[len(stem):] in ANCHOR_INFLECTION_SUFFIXES:\n            return True\n    return False\n\n\ndef primary_query_subject_anchor(value: object) -> str | None:\n    """Return the first distinctive ordered query token, when one exists.\n\n    This deliberately abstains when a question contains relation/type words\n    only.  It does not guess that every noun is a named entity.\n    """\n\n    for token in canonicalize(value).split():\n        if len(token) < 3 or token.isdigit() or _generic_anchor_term(token):\n            continue\n        return token\n    return None\n\n\ndef _anchor_token_match(anchor: str, token: str) -> str | None:\n    if anchor == token:\n        return "exact_token"\n    shorter, longer = sorted((anchor, token), key=len)\n    suffix = longer[len(shorter):] if longer.startswith(shorter) else ""\n    if len(shorter) >= 4 and suffix in ANCHOR_INFLECTION_SUFFIXES:\n        return "bounded_inflection"\n    # OCR tolerance is intentionally restricted to long, similarly sized\n    # tokens.  It recovers e.g. \xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xad\xe0\xa6\xbe\xe0\xa6\xb7\xe0\xa6\xbe\xe0\xa7\x9f -> \xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xad\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xb7\xe0\xa6\xbe\xe0\xa7\x9f, while \xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6\xe0\xa7\x87\xe0\xa6\xb0\n    # or a partial person\'s/river\'s name cannot pass through a shared prefix.\n    if (\n        min(len(anchor), len(token)) >= 8\n        and abs(len(anchor) - len(token)) <= 2\n        and SequenceMatcher(None, anchor, token, autojunk=False).ratio() >= 0.84\n    ):\n        return "bounded_ocr_similarity"\n    return None\n\n\ndef subject_anchor_match(query: object, evidence: object) -> dict[str, Any]:\n    anchor = primary_query_subject_anchor(query)\n    if anchor is None:\n        return {\n            "required": False, "anchor": None, "matched": None,\n            "method": "no_distinctive_subject_anchor",\n        }\n    for token in canonicalize(evidence).split():\n        method = _anchor_token_match(anchor, token)\n        if method:\n            return {\n                "required": True, "anchor": anchor, "matched": token,\n                "method": method,\n            }\n    return {\n        "required": True, "anchor": anchor, "matched": None,\n        "method": "missing",\n    }\n\n\ndef bounded_token_overlap(\n    query_tokens: set[str], source_tokens: set[str],\n) -> tuple[set[str], dict[str, str]]:\n    """Match exact, bounded Bengali inflection and long OCR token variants.\n\n    Returned tokens always come from the query.  Each source token can satisfy\n    at most one query token, preventing one long OCR surface from inflating\n    coverage.  The same conservative matcher already protects Joykoli subject\n    anchors from partial person/river collisions.\n    """\n\n    matched_query: set[str] = set()\n    methods: dict[str, str] = {}\n    unused_source = set(source_tokens)\n    for query_token in sorted(query_tokens, key=lambda value: (-len(value), value)):\n        best: tuple[int, str, str] | None = None\n        for source_token in unused_source:\n            method = _anchor_token_match(query_token, source_token)\n            if method is None:\n                method = _bounded_fact_alias_match(\n                    query_token, source_token, query_tokens, source_tokens,\n                )\n            if method is None:\n                continue\n            priority = {\n                "exact_token": 0,\n                "bounded_inflection": 1,\n                "bounded_ocr_similarity": 3,\n            }.get(method, 2 if method.startswith("bounded_fact_alias:") else 4)\n            candidate = (priority, source_token, method)\n            if best is None or candidate < best:\n                best = candidate\n        if best is None:\n            continue\n        _, source_token, method = best\n        matched_query.add(query_token)\n        methods[query_token] = method\n        unused_source.remove(source_token)\n    return matched_query, methods\n\n\ndef question_target_types(value: object) -> set[str]:\n    text = canonicalize(value)\n    result: set[str] = set()\n    if re.search(r"(?:\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80|\xe0\xa6\xa8\xe0\xa6\xa6|\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2|\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6|\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0|\xe0\xa6\xae\xe0\xa6\xb9\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\x97\xe0\xa6\xb0|river|lake|ocean)", text):\n        result.add("river_or_waterbody")\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x9c\xe0\xa7\x87\xe0\xa6\xb2\xe0\xa6\xbe|\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\x9c\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa7\x80|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xb6\xe0\xa6\xb9\xe0\xa6\xb0|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xae|country|city|capital)", text):\n        result.add("place_or_jurisdiction")\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa6\xa4 \xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa6\xa4 \xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96|when|year|date)", text):\n        result.add("date_or_time")\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa6\xa4\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa7\x9f\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa6\xa4 \xe0\xa6\x9c\xe0\xa6\xa8|\xe0\xa6\x95\xe0\xa6\xa4 \xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe|\xe0\xa6\x95\xe0\xa6\xa4 \xe0\xa6\xa6\xe0\xa6\xbf\xe0\xa6\xa8\xe0\xa7\x87|how many)", text):\n        result.add("quantity")\n    if (\n        re.search(r"(?:\xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf|\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98|length|how long)", text)\n        and re.search(\n            r"(?:\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xae\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa6\xbf\xe0\xa6\xb2\xe0\xa7\x8b\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xb0|\\bkm\\b|kilomet(?:er|re)s?)", text,\n        )\n    ):\n        result.add("distance_measurement")\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xa8\xe0\xa6\xbe|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xaf|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xbe\xe0\xa6\xb8|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\x9f\xe0\xa6\x95|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x97\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa5|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\x97\xe0\xa6\xbe\xe0\xa6\xa8|which (?:book|novel|poem|play|song))", text):\n        result.add("creative_work")\n    if re.search(r"(?:\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe|\xe0\xa6\xb0\xe0\xa6\x9a\xe0\xa7\x9f\xe0\xa6\xbf\xe0\xa6\xa4\xe0\xa6\xbe|\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\x96\xe0\xa6\x95|\xe0\xa6\x95\xe0\xa6\xac\xe0\xa6\xbf|\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbe|\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xbe\xe0\xa6\x9a\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xaf|\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xaa\xe0\xa6\xa4\xe0\xa6\xbf|\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa7\xe0\xa6\xbe\xe0\xa6\xa8\xe0\xa6\xae\xe0\xa6\xa8\xe0\xa7\x8d\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa7\x80|who wrote|author)", text):\n        result.add("person")\n    return result\n\n\ndef _contains_cue(text: str, cue: str) -> bool:\n    if " " in cue:\n        return cue in text\n    return cue in re.findall(r"[A-Za-z0-9\\u0980-\\u09ff]+", text)\n\n\ndef keyword_answer_type_signature(value: object) -> str:\n    """Return a conservative answer type for paraphrase alignment.\n\n    This is intentionally stricter than the general fuzzy gate.  It is used\n    only by the opt-in keyword fallback, where a near neighbour may be shown\n    to the judge but can never prove or refute the candidate response.\n    """\n\n    text = canonicalize(value)\n    target_types = question_target_types(text)\n    for target in (\n        "date_or_time", "distance_measurement", "quantity", "place_or_jurisdiction",\n        "creative_work", "river_or_waterbody", "person",\n    ):\n        if target in target_types:\n            return target\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb0)(?:\\s|[?\xef\xbc\x9f]|$)|(?:who|whose)(?:\\s|[?\xef\xbc\x9f]|$)", text):\n        return "person"\n    if re.search(\n        r"(?:\xe0\xa6\x85\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\xa5|\xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\x9c\xe0\xa7\x8d\xe0\xa6\x9e\xe0\xa6\xbe|\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa7\x87 \xe0\xa6\xac\xe0\xa6\xb2\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa6\xbf \xe0\xa6\xac\xe0\xa7\x8b\xe0\xa6\x9d\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc|\xe0\xa6\x95\xe0\xa6\xbf \xe0\xa6\xac\xe0\xa7\x8b\xe0\xa6\x9d\xe0\xa6\xbe\xe0\xa7\x9f|meaning|definition)", text\n    ):\n        return "definition_or_meaning"\n    if re.search(\n        r"(?:\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa4\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa7\x8d\xe0\xa6\xa0\xe0\xa6\xbe\xe0\xa6\xa8|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8 \xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\xa5\xe0\xa6\xbe|\xe0\xa6\xb8\xe0\xa6\x82\xe0\xa6\x97\xe0\xa6\xa0\xe0\xa6\xa8|organization|institution)", text\n    ):\n        return "organization"\n    if re.search(r"(?:\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xbe\xe0\xa6\x81|\xe0\xa6\xa8\xe0\xa6\xbe|\xe0\xa6\xb8\xe0\xa6\xa4\xe0\xa7\x8d\xe0\xa6\xaf|\xe0\xa6\xae\xe0\xa6\xbf\xe0\xa6\xa5\xe0\xa7\x8d\xe0\xa6\xaf\xe0\xa6\xbe|\xe0\xa6\xa0\xe0\xa6\xbf\xe0\xa6\x95 \xe0\xa6\x95\xe0\xa6\xbf|is it true)", text):\n        return "boolean"\n    return "fact_or_name"\n\n\ndef keyword_predicate_signature(value: object) -> tuple[str, ...]:\n    text = canonicalize(value)\n    return tuple(sorted(\n        family\n        for family, cues in KEYWORD_PREDICATE_CUES.items()\n        if any(_contains_cue(text, canonicalize(cue)) for cue in cues)\n    ))\n\n\ndef keyword_qualifier_signature(value: object) -> dict[str, str]:\n    text = canonicalize(value)\n    result: dict[str, str] = {}\n    for dimension, alternatives in KEYWORD_QUALIFIER_CUES.items():\n        matches = [\n            label\n            for label, cues in alternatives.items()\n            if any(_contains_cue(text, canonicalize(cue)) for cue in cues)\n        ]\n        # More than one alternative in an exclusive dimension is ambiguous.\n        # Preserve that state so it can never align with a single alternative.\n        if matches:\n            result[dimension] = matches[0] if len(matches) == 1 else "ambiguous:" + "+".join(matches)\n    return result\n\n\ndef _anchored_supporting_qualifiers(\n    question: object, supporting_text: object,\n) -> dict[str, str]:\n    if not str(supporting_text or "").strip():\n        return {}\n    anchor = subject_anchor_match(question, supporting_text)\n    if not (anchor["required"] and anchor["matched"]):\n        return {}\n    return keyword_qualifier_signature(supporting_text)\n\n\ndef keyword_entity_kind_signature(\n    value: object, *, supporting_text: object = "",\n) -> str:\n    """Distinguish physical entities that share a proper name.\n\n    Same-record supporting text may fill a missing kind only when the source\n    question\'s distinctive subject is anchored in that text.\n    """\n\n    def kinds(text: object) -> set[str]:\n        normalized = canonicalize(text)\n        result: set[str] = set()\n        if re.search(r"(?:\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80|\xe0\xa6\xa8\xe0\xa6\xa6|\xe0\xa6\x89\xe0\xa6\xaa\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2|\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6|river|lake|canal)", normalized):\n            result.add("waterbody")\n        if re.search(r"(?:\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa7\x81|\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x9c|\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x81\xe0\xa6\xa7|bridge|dam|structure)", normalized):\n            result.add("structure")\n        return result\n\n    result = kinds(value)\n    if not result and str(supporting_text or "").strip():\n        anchor = subject_anchor_match(value, supporting_text)\n        if anchor["required"] and anchor["matched"]:\n            result = kinds(supporting_text)\n    if len(result) == 1:\n        return next(iter(result))\n    return "ambiguous" if result else "unspecified"\n\n\ndef geographic_scope_signature(value: object) -> str:\n    """Return the bounded geographic extent used by physical-fact gates."""\n\n    return keyword_qualifier_signature(value).get(\n        "geographic_scope", "unspecified"\n    )\n\n\ndef geographic_scope_compatible(query_scope: str, source_scope: str) -> bool:\n    """Prevent a country-only measurement from answering a total-length query.\n\n    An unqualified source fact can express the ordinary/total value, but it\n    cannot safely satisfy an explicitly country-scoped question.  Conversely,\n    a country-scoped fact is never used for an unqualified or total question.\n    """\n\n    if source_scope.startswith("ambiguous:"):\n        return False\n    if query_scope == "within_bangladesh":\n        return source_scope == "within_bangladesh"\n    if query_scope == "total":\n        return source_scope in {"total", "unspecified"}\n    if query_scope == "unspecified":\n        return source_scope in {"total", "unspecified"}\n    return query_scope == source_scope\n\n\ndef _title_like_source_question(value: object) -> bool:\n    raw = str(value or "").strip()\n    normalized = canonicalize(raw)\n    if not normalized or len(normalized.split()) > 12 or "?" in raw or "\xef\xbc\x9f" in raw:\n        return False\n    if keyword_predicate_signature(normalized):\n        return False\n    return not bool(re.search(\n        r"(?:^|\\s)(?:\xe0\xa6\x95\xe0\xa6\xbf|\xe0\xa6\x95\xe0\xa7\x80|\xe0\xa6\x95\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa6\xbe\xe0\xa6\xb0|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8|\xe0\xa6\x95\xe0\xa6\xa4|\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa5\xe0\xa6\xbe\xe0\xa7\x9f|\xe0\xa6\x95\xe0\xa7\x87\xe0\xa6\xa8|"\n        r"what|which|who|whose|when|where|why|how)(?:\\s|$)",\n        normalized,\n    ))\n\n\ndef _fact_entity_overlap(query: object, evidence: object) -> dict[str, Any]:\n    query_entities = _keyword_entity_tokens(query)\n    evidence_entities = _keyword_entity_tokens(evidence)\n    shared, methods = bounded_token_overlap(query_entities, evidence_entities)\n    return {\n        "query_entities": sorted(query_entities),\n        "evidence_entities": sorted(evidence_entities),\n        "shared_entities": sorted(shared),\n        "methods": methods,\n    }\n\n\ndef _supporting_fact_units(\n    supporting_text: object, query_predicates: tuple[str, ...],\n) -> list[dict[str, Any]]:\n    """Return bounded sentence/line windows without admitting a whole page."""\n\n    text = str(supporting_text or "")\n    if not text.strip():\n        return []\n    boundaries = [0]\n    for match in re.finditer(r"(?:\\r?\\n)+|(?<=[\xe0\xa5\xa4!?;])\\s+", text):\n        boundaries.extend((match.start(), match.end()))\n    boundaries.append(len(text))\n    spans: list[tuple[int, int]] = []\n    for left, right in zip(boundaries[::2], boundaries[1::2], strict=False):\n        if left < right and text[left:right].strip():\n            spans.append((left, right))\n    # The alternating-boundary construction above deliberately stays simple;\n    # recover any text missed by adjacent separator endpoints deterministically.\n    if not spans:\n        spans = [(0, len(text))]\n\n    cues = unique([\n        cue\n        for family in query_predicates\n        for cue in KEYWORD_PREDICATE_CUES.get(family, ())\n    ])\n    windows: list[tuple[int, int]] = []\n    for start, end in spans:\n        segment = text[start:end].strip()\n        if not segment:\n            continue\n        if len(segment) <= SUPPORTING_FACT_MAX_WINDOW_CHARS:\n            windows.append((start, end))\n            continue\n        positions = [\n            start + match.start()\n            for cue in cues\n            for match in re.finditer(re.escape(cue), segment, flags=re.I)\n        ]\n        for position in sorted(set(positions)):\n            window_start = max(start, position - SUPPORTING_FACT_MAX_WINDOW_CHARS // 2)\n            window_end = min(end, window_start + SUPPORTING_FACT_MAX_WINDOW_CHARS)\n            window_start = max(start, window_end - SUPPORTING_FACT_MAX_WINDOW_CHARS)\n            windows.append((window_start, window_end))\n\n    # A short heading such as ``\xe0\xa6\xa6\xe0\xa7\x88\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x98\xe0\xa7\x8d\xe0\xa6\xaf`` may bind the immediately following\n    # value line; never join arbitrary neighbouring paragraphs.\n    for (start, end), (next_start, next_end) in zip(spans, spans[1:]):\n        first = text[start:end].strip()\n        second = text[next_start:next_end].strip()\n        if (\n            len(first) <= 80\n            and keyword_predicate_signature(first) == query_predicates\n            and len(first) + len(second) + 1 <= SUPPORTING_FACT_MAX_WINDOW_CHARS\n        ):\n            windows.append((start, next_end))\n\n    result: list[dict[str, Any]] = []\n    seen: set[tuple[int, int]] = set()\n    for start, end in sorted(windows):\n        if (start, end) in seen:\n            continue\n        seen.add((start, end))\n        value = text[start:end].strip()\n        if value:\n            result.append({"text": value, "start": start, "end": end})\n    return result\n\n\ndef supporting_fact_candidate_gate(\n    query: object,\n    source_question: object,\n    supporting_text: object,\n    score: float,\n    *,\n    source_metadata: dict[str, Any] | None = None,\n) -> dict[str, Any]:\n    """Find one locally bound, nonterminal fact in same-record support text."""\n\n    query_predicates = keyword_predicate_signature(query)\n    failure = {\n        "eligible": False,\n        "reasons": ["supporting_fact_no_locally_bound_window"],\n        "policy": "nonterminal_local_supporting_fact_window",\n        "supporting_fact_window": "",\n        "supporting_fact_window_start": None,\n        "supporting_fact_window_end": None,\n        "supporting_fact_binding": "none",\n    }\n    if len(query_predicates) != 1 or not str(supporting_text or "").strip():\n        return failure\n\n    title_like = _title_like_source_question(source_question)\n    title_overlap = _fact_entity_overlap(query, source_question)\n    title_bound = bool(title_overlap["shared_entities"])\n    metadata = source_metadata or {}\n    per_question_extra = metadata.get("has_per_question_extra_info") is True\n    query_kind = keyword_entity_kind_signature(query)\n    query_scope = geographic_scope_signature(query)\n    eligible: list[tuple[tuple[Any, ...], dict[str, Any]]] = []\n    rejected: set[str] = set()\n\n    for unit in _supporting_fact_units(supporting_text, query_predicates):\n        window = unit["text"]\n        if keyword_predicate_signature(window) != query_predicates:\n            continue\n        direct_overlap = _fact_entity_overlap(query, window)\n        direct_bound = bool(direct_overlap["shared_entities"])\n        anaphoric = bool(re.search(\n            r"(?:^|\\s)(?:\xe0\xa6\x8f\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x8f\xe0\xa6\xb0|\xe0\xa6\x8f\xe0\xa6\x87|\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0|\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0|\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa7\x81\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa7\x81\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0|"\n            r"\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa6\x9f\xe0\xa6\xbf|\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa6\x9f\xe0\xa6\xbf\xe0\xa6\xb0)(?:\\s|$)",\n            canonicalize(window)[:160],\n        ))\n        header_bound = bool(\n            title_bound and anaphoric and (title_like or per_question_extra)\n        )\n        if not direct_bound and not header_bound:\n            continue\n        evidence = (\n            f"{str(source_question).strip()}\\n{window}"\n            if header_bound and title_like else window\n        )\n        source_kind = keyword_entity_kind_signature(evidence)\n        if query_kind != "unspecified" and source_kind != query_kind:\n            rejected.add("supporting_fact_entity_kind_mismatch")\n            continue\n        source_scope = geographic_scope_signature(window)\n        if not geographic_scope_compatible(query_scope, source_scope):\n            rejected.add("supporting_fact_geographic_scope_mismatch")\n            continue\n        gate = passage_candidate_gate(query, evidence, score)\n        reasons = set(gate.get("reasons") or [])\n        if reasons:\n            rejected.update(reasons)\n            continue\n        binding = "direct_local_entity" if direct_bound else "title_anaphora"\n        gate.update({\n            "eligible": True,\n            "reasons": [],\n            "policy": "nonterminal_local_supporting_fact_window",\n            "supporting_fact_window": window,\n            "supporting_fact_window_start": int(unit["start"]),\n            "supporting_fact_window_end": int(unit["end"]),\n            "supporting_fact_binding": binding,\n            "query_predicates": list(query_predicates),\n            "source_predicates": list(query_predicates),\n            "query_entity_kind": query_kind,\n            "source_entity_kind": source_kind,\n            "query_geographic_scope": query_scope,\n            "source_geographic_scope": source_scope,\n            "terminal_label_authority": False,\n            "evidence_role": "neutral_candidate",\n            "evidence_proof_status": "FUZZY_NONPROOF",\n        })\n        rank_key = (\n            0 if direct_bound else 1,\n            -float(gate.get("query_token_coverage", 0.0)),\n            len(window),\n            int(unit["start"]),\n        )\n        eligible.append((rank_key, gate))\n\n    if eligible:\n        return min(eligible, key=lambda value: value[0])[1]\n    if rejected:\n        failure["reasons"] = sorted(rejected)\n    return failure\n\n\ndef _date_granularity(question: object, answers: list[str] | None = None) -> str:\n    text = canonicalize(question)\n    if keyword_answer_type_signature(text) != "date_or_time":\n        return "not_date"\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa6\xa4|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8)\\s*\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96|\xe0\xa6\xa4\xe0\xa6\xbe\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x96\xe0\xa7\x87|exact date|what date", text):\n        return "exact_day"\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa6\xa4|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8)\\s*\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8|\xe0\xa6\xae\xe0\xa6\xbe\xe0\xa6\xb8\xe0\xa7\x87|what month", text):\n        return "month"\n    if re.search(r"(?:\xe0\xa6\x95\xe0\xa6\xa4|\xe0\xa6\x95\xe0\xa7\x8b\xe0\xa6\xa8)\\s*\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2|\xe0\xa6\x95\xe0\xa6\xa4\\s*\xe0\xa6\xac\xe0\xa6\x9b\xe0\xa6\xb0|\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x87|year", text):\n        return "year"\n\n    # A generic source question such as ``\xe0\xa6\x95\xe0\xa6\xac\xe0\xa7\x87`` is only safe for an exact-day\n    # query when its keyed answer has exact-day granularity as well.\n    answer_text = canonicalize(" ".join(str(answer) for answer in (answers or [])))\n    answer_numbers = NUMBER_RE.findall(answer_text)\n    has_month = any(canonicalize(month) in answer_text for month in DATE_MONTH_CUES)\n    if re.search(r"[0-9\xe0\xa7\xa6-\xe0\xa7\xaf]{1,2}\\s*[-/.]\\s*[0-9\xe0\xa7\xa6-\xe0\xa7\xaf]{1,2}\\s*[-/.]\\s*[0-9\xe0\xa7\xa6-\xe0\xa7\xaf]{2,4}", answer_text):\n        return "exact_day"\n    if has_month and len(answer_numbers) >= 2:\n        return "exact_day"\n    if has_month:\n        return "month"\n    if len(answer_numbers) == 1 and re.fullmatch(\n        r"(?:[12][0-9]{3}|[\xe0\xa7\xa7\xe0\xa7\xa8][\xe0\xa7\xa6-\xe0\xa7\xaf]{3})(?:\\s*(?:\xe0\xa6\x96\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf(?:\xe0\xa6\xb8\xe0\xa7\x8d\xe0\xa6\x9f\xe0\xa6\xbe\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6)?|\xe0\xa6\xb8\xe0\xa6\xbe\xe0\xa6\xb2|year))?",\n        answer_text,\n    ):\n        return "year"\n    return "unspecified_date"\n\n\ndef _keyword_entity_tokens(value: object) -> set[str]:\n    text = canonicalize(value)\n    excluded = set(FUZZY_STOPWORDS) | set(ANCHOR_GENERIC_TERMS)\n    for cues in KEYWORD_PREDICATE_CUES.values():\n        for cue in cues:\n            excluded.update(canonicalize(cue).split())\n    for dimension, alternatives in KEYWORD_QUALIFIER_CUES.items():\n        # Jurisdiction words (for example \xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x82\xe0\xa6\xb2\xe0\xa6\xbe\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb6) remain distinctive entity\n        # anchors; only the qualifier signature, not this exclusion set,\n        # interprets their scoped use.\n        if dimension == "geographic_scope":\n            continue\n        for cues in alternatives.values():\n            for cue in cues:\n                excluded.update(canonicalize(cue).split())\n    excluded.update({\n        "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xae", "\xe0\xa6\xa8\xe0\xa6\xbe\xe0\xa6\xae\xe0\xa6\x9f\xe0\xa6\xbf", "\xe0\xa6\x98\xe0\xa6\x9f\xe0\xa7\x87", "\xe0\xa6\x98\xe0\xa6\x9f\xe0\xa7\x87\xe0\xa6\x9b\xe0\xa6\xbf\xe0\xa6\xb2", "\xe0\xa6\xb8\xe0\xa6\xae\xe0\xa7\x8d\xe0\xa6\xaa\xe0\xa6\xb0\xe0\xa7\x8d\xe0\xa6\x95\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa7\x87", "\xe0\xa6\xac\xe0\xa6\xbf\xe0\xa6\xb7\xe0\xa7\x9f\xe0\xa7\x87",\n        "\xe0\xa6\xac\xe0\xa6\xb2\xe0\xa7\x81\xe0\xa6\xa8", "\xe0\xa6\x89\xe0\xa6\xb2\xe0\xa7\x8d\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\x96", "answer", "question",\n    })\n    return {\n        token for token in text.split()\n        if len(token) >= 3 and not token.isdigit()\n        and token not in excluded and token not in FACT_RELATION_TOKENS\n        and not _generic_anchor_term(token)\n    }\n\n\ndef bounded_entity_title_keys(value: object) -> list[str]:\n    """Derive at most three exact article-title probes from a fact question.\n\n    These keys are discovery aliases only.  A hit is still fuzzy, nonterminal,\n    and must pass the local supporting-fact window gate.\n    """\n\n    normalized = canonicalize(value)\n    predicates = keyword_predicate_signature(normalized)\n    if len(predicates) != 1 or not _keyword_entity_tokens(normalized):\n        return []\n    excluded = set(FUZZY_STOPWORDS) | {"\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa6\xaf\xe0\xa6\xbc", "\xe0\xa6\xaa\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbe\xe0\xa7\x9f", "about", "approximately"}\n    for cues in KEYWORD_PREDICATE_CUES.values():\n        for cue in cues:\n            excluded.update(canonicalize(cue).split())\n    for alternatives in KEYWORD_QUALIFIER_CUES.values():\n        for cues in alternatives.values():\n            for cue in cues:\n                excluded.update(canonicalize(cue).split())\n    for family, aliases in BOUNDED_FACT_ALIAS_FAMILIES.items():\n        if family != "bn_river_case":\n            excluded.update(aliases)\n    tokens = [\n        token for token in normalized.split()\n        if token not in excluded and not token.isdigit()\n    ]\n    if not tokens or len(tokens) > 10:\n        return []\n    genitive_kinds = {\n        "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80\xe0\xa6\xb0": "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x80", "\xe0\xa6\xa8\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb0": "\xe0\xa6\xa8\xe0\xa6\xa6", "\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa7\x81\xe0\xa6\xb0": "\xe0\xa6\xb8\xe0\xa7\x87\xe0\xa6\xa4\xe0\xa7\x81", "\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x9c\xe0\xa7\x87\xe0\xa6\xb0": "\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xbf\xe0\xa6\x9c",\n        "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x81\xe0\xa6\xa7\xe0\xa7\x87\xe0\xa6\xb0": "\xe0\xa6\xac\xe0\xa6\xbe\xe0\xa6\x81\xe0\xa6\xa7", "\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2\xe0\xa7\x87\xe0\xa6\xb0": "\xe0\xa6\x96\xe0\xa6\xbe\xe0\xa6\xb2", "\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6\xe0\xa7\x87\xe0\xa6\xb0": "\xe0\xa6\xb9\xe0\xa7\x8d\xe0\xa6\xb0\xe0\xa6\xa6",\n    }\n    normalized_tokens = [genitive_kinds.get(token, token) for token in tokens]\n    variants = [canonicalize(" ".join(normalized_tokens))]\n    stripped = [\n        token[:-1] if len(token) >= 5 and token.endswith("\xe0\xa6\xb0") else token\n        for token in normalized_tokens\n    ]\n    variants.append(canonicalize(" ".join(stripped)))\n    variants.append(canonicalize(" ".join(tokens)))\n    return unique([key for key in variants if key])[:3]\n\n\ndef keyword_fallback_alignment_gate(\n    query: object,\n    source_question: object,\n    source_answers: list[str],\n    score: float,\n    *,\n    query_response: object = "",\n    source_supporting_text: object = "",\n) -> dict[str, Any]:\n    """Fail-closed fact alignment for a non-proof TF-IDF hint.\n\n    A hit must agree on answer type, number/negation, entity, qualifiers,\n    predicate/property, and requested answer granularity.  It still remains a\n    neutral candidate: passing this gate says only that the source question is\n    about the same fact slot, never that its answer proves the response.\n    """\n\n    base = fuzzy_candidate_gate(query, source_question, score, exact=False)\n    reasons = set(base.get("reasons") or [])\n    if KEYWORD_EXACT_ONLY_OPERATION_RE.search(canonicalize(query)):\n        reasons.add("keyword_typed_lexical_or_grammar_operation_requires_exact_evidence")\n    query_type = keyword_answer_type_signature(query)\n    source_type = keyword_answer_type_signature(source_question)\n    if source_type == "fact_or_name" and str(source_supporting_text or "").strip():\n        anchor = subject_anchor_match(source_question, source_supporting_text)\n        if anchor["required"] and anchor["matched"]:\n            source_type = keyword_answer_type_signature(\n                f"{source_question} {source_supporting_text}"\n            )\n    if query_type != source_type:\n        reasons.add("keyword_answer_type_mismatch")\n\n    query_predicates = keyword_predicate_signature(query)\n    source_predicates = keyword_predicate_signature(source_question)\n    if query_predicates != source_predicates:\n        reasons.add("keyword_predicate_property_mismatch")\n\n    query_qualifiers = keyword_qualifier_signature(query)\n    source_qualifiers = keyword_qualifier_signature(source_question)\n    for dimension, label in _anchored_supporting_qualifiers(\n        source_question, source_supporting_text,\n    ).items():\n        source_qualifiers.setdefault(dimension, label)\n    qualifier_dimensions = sorted(set(query_qualifiers) | set(source_qualifiers))\n    qualifier_mismatches = [\n        dimension for dimension in qualifier_dimensions\n        if query_qualifiers.get(dimension) != source_qualifiers.get(dimension)\n    ]\n    if qualifier_mismatches:\n        reasons.add("keyword_entity_qualifier_mismatch")\n\n    query_entities = _keyword_entity_tokens(query)\n    source_entities = _keyword_entity_tokens(source_question)\n    shared_entities, shared_entity_methods = bounded_token_overlap(\n        query_entities, source_entities\n    )\n    if not shared_entities:\n        reasons.add("keyword_distinctive_entity_missing")\n\n    query_entity_kind = keyword_entity_kind_signature(query)\n    source_entity_kind = keyword_entity_kind_signature(\n        source_question, supporting_text=source_supporting_text,\n    )\n    if (\n        query_entity_kind != "unspecified"\n        and query_entity_kind != source_entity_kind\n    ):\n        reasons.add("keyword_entity_kind_mismatch")\n\n    query_granularity = _date_granularity(query)\n    source_granularity = _date_granularity(source_question, source_answers)\n    response_granularity = _date_granularity(query, [str(query_response)])\n    if query_type == "date_or_time":\n        requested_granularity = (\n            query_granularity\n            if query_granularity != "unspecified_date"\n            else response_granularity\n        )\n        if (\n            requested_granularity != "unspecified_date"\n            and source_granularity != requested_granularity\n        ):\n            reasons.add("keyword_answer_granularity_mismatch")\n    else:\n        requested_granularity = "not_date"\n\n    query_tokens = substantive_token_set(query)\n    source_tokens = substantive_token_set(source_question)\n    shared_tokens, shared_token_methods = bounded_token_overlap(\n        query_tokens, source_tokens\n    )\n    token_coverage = len(shared_tokens) / max(1, len(query_tokens))\n    result = dict(base)\n    result.update({\n        "eligible": not reasons,\n        "reasons": sorted(reasons),\n        "policy": (\n            "keyword_tfidf_nonproof_same_answer_type_number_negation_entity_"\n            "qualifier_predicate_and_granularity"\n        ),\n        "query_answer_type": query_type,\n        "source_answer_type": source_type,\n        "query_predicates": list(query_predicates),\n        "source_predicates": list(source_predicates),\n        "query_qualifiers": query_qualifiers,\n        "source_qualifiers": source_qualifiers,\n        "qualifier_mismatch_dimensions": qualifier_mismatches,\n        "shared_entity_tokens": sorted(shared_entities),\n        "shared_entity_token_methods": shared_entity_methods,\n        "query_entity_kind": query_entity_kind,\n        "source_entity_kind": source_entity_kind,\n        "query_date_granularity": query_granularity,\n        "source_date_granularity": source_granularity,\n        "requested_date_granularity": requested_granularity,\n        "substantive_token_coverage": round(token_coverage, 8),\n        "shared_substantive_token_methods": shared_token_methods,\n        "evidence_role": "neutral_candidate",\n        "evidence_proof_status": "FUZZY_NONPROOF",\n    })\n    return result\n\n\ndef fuzzy_candidate_gate(\n    query: object,\n    source_question: object,\n    score: float,\n    *,\n    exact: bool = False,\n) -> dict[str, Any]:\n    """Keep weak/short/type-conflicting fuzzy hits out of model prompts."""\n    if exact:\n        return {\n            "eligible": True,\n            "reasons": [],\n            "policy": "exact_full_normalized_question",\n            "shared_substantive_tokens": sorted(\n                substantive_token_set(query).intersection(substantive_token_set(source_question))\n            ),\n            "query_target_types": sorted(question_target_types(query)),\n            "source_target_types": sorted(question_target_types(source_question)),\n        }\n    reasons: list[str] = []\n    query_tokens = substantive_token_set(query)\n    source_tokens = substantive_token_set(source_question)\n    shared, shared_methods = bounded_token_overlap(query_tokens, source_tokens)\n    if len(query_tokens) < 2:\n        reasons.append("short_or_keyword_only_query")\n    if float(score) < 0.22:\n        reasons.append("similarity_below_0_22")\n    query_numbers = number_set(query)\n    source_numbers = number_set(source_question)\n    if query_numbers != source_numbers:\n        reasons.append("number_set_mismatch")\n    query_negations = negation_set(query)\n    source_negations = negation_set(source_question)\n    if query_negations != source_negations:\n        reasons.append("negation_set_mismatch")\n    query_types = question_target_types(query)\n    source_types = question_target_types(source_question)\n    if query_types and source_types and query_types.isdisjoint(source_types):\n        reasons.append("answer_type_intent_conflict")\n    distinctive_single_overlap = (\n        len(shared) == 1\n        and len(next(iter(shared))) >= 8\n        and float(score) >= 0.55\n    )\n    if len(shared) < 2 and not distinctive_single_overlap:\n        reasons.append("insufficient_substantive_token_overlap")\n    return {\n        "eligible": not reasons,\n        "reasons": sorted(set(reasons)),\n        "policy": "fuzzy_similarity_overlap_number_negation_and_answer_type_gate",\n        "shared_substantive_tokens": sorted(shared),\n        "shared_substantive_token_methods": shared_methods,\n        "substantive_token_coverage": round(\n            len(shared) / max(1, len(query_tokens)), 8\n        ),\n        "query_target_types": sorted(query_types),\n        "source_target_types": sorted(source_types),\n    }\n\n\ndef source_retrieval_placeholder_reason(passage: object) -> str | None:\n    """Identify source lookup failures that contain no evidentiary content.\n\n    Dictionary/Wiktionary acquisition records sometimes preserve the queried\n    title together with a ``missing`` flag or a search-page "not found"\n    message.  Their title tokens can otherwise satisfy the passage overlap\n    gate even though the record contains no definition or factual passage.\n    Keep this deliberately narrower than ordinary negation: statements that\n    something is absent remain valid evidence unless they use an explicit\n    source-retrieval failure schema or UI phrase.\n    """\n\n    text = str(passage or "").strip()\n    if not text:\n        return None\n    try:\n        structured = json.loads(text)\n    except (TypeError, ValueError, json.JSONDecodeError):\n        structured = None\n    if isinstance(structured, dict) and structured.get("missing") is True:\n        lookup_identity = any(\n            key in structured\n            for key in ("requested_title", "resolved_title", "pageid", "source_url")\n        )\n        content_fields = (\n            structured.get("wikitext"),\n            structured.get("extract"),\n            structured.get("content"),\n            structured.get("definition"),\n        )\n        if lookup_identity and not any(str(value or "").strip() for value in content_fields):\n            return "source_retrieval_placeholder_structured_missing"\n\n    bengali_not_found = re.search(\n        r"\xe0\xa6\xb6\xe0\xa6\xac\xe0\xa7\x8d\xe0\xa6\xa6\xe0\xa6\x9f\xe0\xa6\xbf\\s+\xe0\xa6\x96\xe0\xa7\x81\xe0\xa6\x81\xe0\xa6\x9c\xe0\xa7\x87\\s+\xe0\xa6\xaa\xe0\xa6\xbe\xe0\xa6\x93(?:\xe0\xa6\xaf\xe0\xa6\xbc\xe0\xa6\xbe|\xe0\xa7\x9f\xe0\xa6\xbe)\\s+\xe0\xa6\xaf\xe0\xa6\xbe(?:\xe0\xa6\xaf\xe0\xa6\xbc|\xe0\xa7\x9f)\xe0\xa6\xa8\xe0\xa6\xbf",\n        text,\n        re.I,\n    )\n    english_not_found = re.search(\n        r"(?:\\bno\\s+(?:matching\\s+|search\\s+)?(?:results?|entries|pages?)\\s+"\n        r"(?:were\\s+|was\\s+)?found\\b|"\n        r"\\b(?:word|entry|page|term)\\b.{0,40}\\bnot\\s+found\\b|"\n        r"\\b(?:word|entry|page|term)\\b.{0,40}\\bdoes\\s+not\\s+exist\\b)",\n        text,\n        re.I | re.S,\n    )\n    if bengali_not_found or english_not_found:\n        return "source_retrieval_placeholder_not_found"\n    return None\n\n\ndef passage_candidate_gate(\n    query: object, passage: object, score: float,\n) -> dict[str, Any]:\n    """Gate nonterminal page evidence without pretending it is a question.\n\n    Dense OCR pages can mention many unrelated entities.  A passage therefore\n    needs at least two substantive query-token matches, useful token coverage,\n    and containment of every number/negation explicitly present in the query.\n    Unlike question-to-question retrieval, extra numbers in a source passage\n    are permitted because a source-grounded block may contain choices or an\n    explanation.  The result is evidence only and can never create a verdict.\n    """\n\n    reasons: list[str] = []\n    query_text = str(query or "").strip()\n    passage_text = str(passage or "")\n    placeholder_reason = source_retrieval_placeholder_reason(passage_text)\n    if placeholder_reason:\n        reasons.append(placeholder_reason)\n    query_tokens = substantive_token_set(query_text)\n    passage_tokens = substantive_token_set(passage_text)\n    shared, shared_methods = bounded_token_overlap(query_tokens, passage_tokens)\n    coverage = len(shared) / len(query_tokens) if query_tokens else 0.0\n    literal_query_contained = bool(query_text and query_text in passage_text)\n    normalized_query = canonicalize(query_text)\n    normalized_passage = canonicalize(passage_text)\n    normalized_query_contained = bool(\n        normalized_query and normalized_query in normalized_passage\n    )\n    if len(query_tokens) < 2:\n        reasons.append("short_or_keyword_only_query")\n    if float(score) < 0.20 and not normalized_query_contained:\n        reasons.append("passage_similarity_below_0_20")\n    if len(shared) < 2:\n        reasons.append("passage_requires_two_substantive_overlaps")\n    if coverage < 0.35 and len(shared) < 3:\n        reasons.append("insufficient_query_token_coverage")\n    query_numbers = set(number_set(query))\n    passage_numbers = set(number_set(passage))\n    if not query_numbers.issubset(passage_numbers):\n        reasons.append("query_number_missing_from_passage")\n    query_negations = set(negation_set(query))\n    passage_negations = set(negation_set(passage))\n    if not query_negations.issubset(passage_negations):\n        reasons.append("query_negation_missing_from_passage")\n    alias_used = any(\n        str(method).startswith("bounded_fact_alias:")\n        for method in shared_methods.values()\n    )\n    anchor = subject_anchor_match(query, passage)\n    if alias_used and not (anchor["required"] and anchor["matched"]):\n        reasons.append("bounded_fact_alias_requires_subject_anchor")\n    query_entity_kind = keyword_entity_kind_signature(query)\n    source_entity_kind = keyword_entity_kind_signature(passage)\n    if (\n        query_entity_kind != "unspecified"\n        and query_entity_kind != source_entity_kind\n    ):\n        reasons.append("passage_entity_kind_mismatch")\n    query_scope = geographic_scope_signature(query)\n    source_scope = geographic_scope_signature(passage)\n    if not geographic_scope_compatible(query_scope, source_scope):\n        reasons.append("passage_geographic_scope_mismatch")\n    return {\n        "eligible": not reasons,\n        "reasons": sorted(set(reasons)),\n        "policy": "nonterminal_passage_overlap_coverage_number_negation_gate",\n        "shared_substantive_tokens": sorted(shared),\n        "shared_substantive_token_methods": shared_methods,\n        "substantive_token_coverage": round(coverage, 8),\n        "query_token_coverage": round(coverage, 8),\n        "literal_query_contained": literal_query_contained,\n        "normalized_query_contained": normalized_query_contained,\n        "source_retrieval_placeholder": bool(placeholder_reason),\n        "bounded_fact_alias_subject_anchor": anchor if alias_used else None,\n        "query_entity_kind": query_entity_kind,\n        "source_entity_kind": source_entity_kind,\n        "query_geographic_scope": query_scope,\n        "source_geographic_scope": source_scope,\n        "query_target_types": sorted(question_target_types(query)),\n        "source_target_types": [],\n    }\n\n\ndef tighten_joykoli_structured_gate(\n    gate: dict[str, Any], *, exact: bool, query: object | None = None,\n    evidence: object | None = None, passage_evidence: bool = False,\n) -> dict[str, Any]:\n    """Apply Joykoli-only overlap and primary-subject protections."""\n\n    if exact:\n        return gate\n    result = dict(gate)\n    reasons = set(result.get("reasons") or [])\n    policy_suffixes: list[str] = []\n    if not passage_evidence and len(gate.get("shared_substantive_tokens") or []) < 2:\n        reasons.add("joykoli_nonexact_requires_two_substantive_overlaps")\n        policy_suffixes.append("joykoli_no_single_token_exception")\n    if query is not None and evidence is not None:\n        anchor = subject_anchor_match(query, evidence)\n        result["query_primary_subject_anchor"] = anchor\n        if anchor["required"] and not anchor["matched"]:\n            reasons.add(\n                "query_primary_anchor_missing_from_passage"\n                if passage_evidence\n                else "query_primary_anchor_missing_from_source_question"\n            )\n        policy_suffixes.append("joykoli_primary_subject_anchor")\n    result["eligible"] = not reasons\n    result["reasons"] = sorted(reasons)\n    if policy_suffixes:\n        result["policy"] = str(result.get("policy", "")) + "+" + "+".join(policy_suffixes)\n    return result\n\n\ndef relation(response: object, answers: list[str]) -> str:\n    if classify_answer_stance(response, answers).status != "supported":\n        return "none"\n    response_key = answer_equivalence_key(response)\n    answer_keys = unique([answer_equivalence_key(value) for value in answers])\n    if not response_key or not answer_keys:\n        return "none"\n    if response_key in answer_keys:\n        return "exact"\n    if any(\n        min(len(response_key), len(answer)) >= 4\n        and (response_key in answer or answer in response_key)\n        for answer in answer_keys\n    ):\n        return "containment"\n    return "none"\n\n\ndef mmap_evidence_role(\n    response_answer_relation: object,\n    *,\n    exact_normalized: bool,\n    source_verdict_candidate: dict[str, Any] | None,\n) -> str:\n    """Classify mmap evidence without treating a fuzzy miss as contradiction.\n\n    Agreement may be established by the bounded response/answer relation.  A\n    counter role requires an exact-normalized source rule that explicitly\n    authorized verdict 0.  Merely differing from an answer on a fuzzy hit is\n    neutral evidence, not counterevidence.\n    """\n\n    if not exact_normalized:\n        return "neutral_candidate"\n    relation_value = str(response_answer_relation or "none")\n    # ``relation`` admits containment only after the negation/correction-aware\n    # proposition classifier has found a complete accepted-answer span.  On\n    # an exact-normalized source question this is a bounded answer alias (for\n    # example ``\xe0\xa6\xa2\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe`` versus ``\xe0\xa6\xa2\xe0\xa6\xbe\xe0\xa6\x95\xe0\xa6\xbe \xe0\xa6\xb6\xe0\xa6\xb9\xe0\xa6\xb0``), not a fuzzy fact match.  Preserve\n    # that grounded support signal while keeping every non-exact question\n    # neutral below.\n    if relation_value in {"exact", "containment"}:\n        return "support_candidate"\n    if exact_normalized and isinstance(source_verdict_candidate, dict):\n        verdict = source_verdict_candidate.get("verdict")\n        if verdict == 0:\n            return "counter_candidate"\n    return "neutral_candidate"\n\n\ndef load_input(path: Path) -> list[dict[str, Any]]:\n    rows: list[dict[str, Any]] = []\n    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), start=1):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        forbidden = sorted(FORBIDDEN_FIELDS.intersection(walk_keys(row)))\n        if forbidden:\n            raise ValueError(f"input line {line_number} contains forbidden fields: {forbidden}")\n        example_id = row.get("example_id", row.get("id"))\n        if example_id is None or not str(example_id).strip():\n            raise ValueError(f"input line {line_number} has no example_id/id")\n        required = ("model_prompt_bn", "model_response_bn")\n        if any(key not in row for key in required):\n            raise ValueError(f"input line {line_number} is not Phase-2 formatted")\n        rows.append({**row, "example_id": str(example_id)})\n    if not rows:\n        raise ValueError("retrieval input is empty")\n    ids = [row["example_id"] for row in rows]\n    if len(ids) != len(set(ids)):\n        raise ValueError("retrieval input has duplicate IDs")\n    return rows\n\n\ndef load_terminal_context_ids(path: Path) -> set[str]:\n    """Read only IDs already settled by the label-free context stage."""\n\n    ids: set[str] = set()\n    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), start=1):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        example_id = str(row.get("example_id", "")).strip()\n        if not example_id:\n            raise ValueError(f"context line {line_number} has no example_id")\n        if str(row.get("status", "")).startswith("terminal_deterministic_"):\n            if row.get("verdict") not in {0, 1}:\n                raise ValueError(f"context line {line_number} has invalid terminal verdict")\n            ids.add(example_id)\n    return ids\n\n\ndef load_context_external_lookup_ids(path: Path) -> set[str]:\n    """Reject legacy contextual lookup authorization and return an empty set."""\n\n    ids: set[str] = set()\n    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), start=1):\n        if not line.strip():\n            continue\n        row = json.loads(line)\n        example_id = str(row.get("example_id", "")).strip()\n        if not example_id:\n            raise ValueError(f"context line {line_number} has no example_id")\n        route = row.get("context_policy_route") or {}\n        analysis = row.get("rule_application_analysis") or {}\n        if (\n            analysis.get("external_lookup_allowed") is True\n            or route.get("external_lookup_allowed") is True\n        ):\n            raise ValueError(\n                "context grounding attempts to authorize external retrieval"\n            )\n    return ids\n\n\ndef load_index(spec: dict[str, Any]) -> dict[str, Any]:\n    directory = Path(spec["directory"])\n    manifest_path = directory / "manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    if manifest.get("version") == MMAP_CACHE_VERSION:\n        loaded = load_mmap_index(directory)\n        rights = loaded["manifest"].get("rights_policy") or {}\n        if rights.get("bundle_allowed") is not True or rights.get("quarantined") is not False:\n            close_mmap_index(loaded)\n            raise ValueError(f"{spec[\'source_id\']} mmap cache is not retrieval-deployable")\n        return {**spec, **loaded}\n    required = ("vectorizer.joblib", "matrix.npz", "records.jsonl")\n    for name in required:\n        expected = manifest.get("files", {}).get(name)\n        observed = sha256_file(directory / name)\n        if expected != observed:\n            raise ValueError(f"{spec[\'source_id\']} index hash mismatch for {name}")\n    records = [\n        json.loads(line)\n        for line in (directory / "records.jsonl").read_text(encoding="utf-8").splitlines()\n        if line.strip()\n    ]\n    if [int(record["index"]) for record in records] != list(range(len(records))):\n        raise ValueError(f"{spec[\'source_id\']} record indexes are not dense and ordered")\n    matrix = sparse.load_npz(directory / "matrix.npz").tocsr()\n    if matrix.shape[0] != len(records):\n        raise ValueError(f"{spec[\'source_id\']} matrix/record row mismatch")\n    exact_lookup: dict[str, list[int]] = {}\n    for source_index, record in enumerate(records):\n        key = canonicalize(record.get("normalized_question", record.get("question", "")))\n        if key:\n            exact_lookup.setdefault(key, []).append(source_index)\n    return {\n        **spec,\n        "directory": directory,\n        "manifest": manifest,\n        "manifest_sha256": sha256_file(manifest_path),\n        "vectorizer": joblib.load(directory / "vectorizer.joblib"),\n        "matrix": matrix,\n        "records": records,\n        "exact_lookup": exact_lookup,\n    }\n\n\ndef unpack_source(record: dict[str, Any], source_id: str) -> tuple[list[str], list[str], int]:\n    if source_id == "bangla_mmlu":\n        source_rows = list(record.get("records") or [])\n        answers = unique([str(row.get("keyed_answer", "")).strip() for row in source_rows])\n        choices = unique([\n            str(choice).strip()\n            for row in source_rows\n            for choice in (row.get("choices") or [])\n        ])\n        return answers, choices, len(source_rows)\n    answers = unique([str(value).strip() for value in (record.get("answers") or [])])\n    choices = unique([str(value).strip() for value in (record.get("choices") or [])])\n    return answers, choices, len(record.get("records") or [])\n\n\ndef source_verdict_candidate(\n    source_id: str, *, exact: bool, response: str,\n    answers: list[str], choices: list[str], response_relation: str,\n    record: dict[str, Any] | None = None,\n) -> dict[str, Any] | None:\n    if not exact:\n        return None\n    answer_keys = unique([answer_equivalence_key(value) for value in answers])\n    if source_id == "bangla_mmlu":\n        if len(answer_keys) != 1:\n            return None\n        response_key = answer_equivalence_key(response)\n        if response_key == answer_keys[0]:\n            return {"verdict": 1, "rule": "exact_question_key_consensus_response_exact_key"}\n        if response_key and response_key in {answer_equivalence_key(value) for value in choices}:\n            return {"verdict": 0, "rule": "exact_question_key_consensus_response_exact_wrong_option"}\n        return None\n    if source_id == "nctb_qa_87805" and response_relation in {"exact", "containment"}:\n        return {"verdict": 1, "rule": f"exact_question_positive_answer_{response_relation}"}\n    if source_id == "nctb_education_aux" and record is not None:\n        response_key = answer_equivalence_key(response)\n        consensus = answer_equivalence_key(record.get("terminal_consensus_answer", ""))\n        if consensus and response_key == consensus:\n            return {"verdict": 1, "rule": "exact_question_rights_eligible_consensus_answer"}\n        negative_keys = {\n            answer_equivalence_key(value) for value in record.get("terminal_negative_choices", [])\n        }\n        if consensus and response_key in negative_keys:\n            return {"verdict": 0, "rule": "exact_question_rights_eligible_consensus_wrong_option"}\n    return None\n\n\ndef strict_exact_key_conflicts(\n    candidates: list[dict[str, Any]],\n) -> list[dict[str, Any]]:\n    """Find opt-in answer-key conflicts without granting a new source authority.\n\n    A corroboration-only OCR/book source may challenge an existing exact-source\n    verdict only when both records are exact normalized-question matches, both\n    have one keyed answer, and their complete normalized MCQ choice sets are\n    identical with at least two choices.  The challenge can only quarantine a\n    terminal route; it can never create a verdict.\n    """\n\n    terminal = [\n        candidate for candidate in candidates\n        if candidate.get("exact_normalized") is True\n        and candidate.get("source_verdict_candidate") is not None\n    ]\n    challengers = [\n        candidate for candidate in candidates\n        if candidate.get("exact_normalized") is True\n        and candidate.get("exact_conflict_policy") == STRICT_EXACT_CONFLICT_POLICY\n        and candidate.get("exact_conflict_eligible") is True\n    ]\n    conflicts: list[dict[str, Any]] = []\n    seen: set[tuple[str, int, str, int]] = set()\n    for authority in terminal:\n        authority_answers = unique([\n            answer_equivalence_key(value) for value in authority.get("answers", [])\n        ])\n        authority_choices = {\n            answer_equivalence_key(value) for value in authority.get("choices", [])\n            if answer_equivalence_key(value)\n        }\n        if len(authority_answers) != 1 or len(authority_choices) < 2:\n            continue\n        for challenger in challengers:\n            if challenger.get("source_id") == authority.get("source_id"):\n                continue\n            challenger_answers = unique([\n                answer_equivalence_key(value) for value in challenger.get("answers", [])\n            ])\n            challenger_choices = {\n                answer_equivalence_key(value) for value in challenger.get("choices", [])\n                if answer_equivalence_key(value)\n            }\n            if (\n                len(challenger_answers) != 1\n                or challenger_choices != authority_choices\n                or challenger_answers[0] == authority_answers[0]\n            ):\n                continue\n            key = (\n                str(authority.get("source_id")),\n                int(authority.get("source_record_index", -1)),\n                str(challenger.get("source_id")),\n                int(challenger.get("source_record_index", -1)),\n            )\n            if key in seen:\n                continue\n            seen.add(key)\n            conflicts.append({\n                "rule": STRICT_EXACT_CONFLICT_POLICY,\n                "effect": "quarantine_only_no_verdict",\n                "authority_source_id": key[0],\n                "authority_source_record_index": key[1],\n                "challenger_source_id": key[2],\n                "challenger_source_record_index": key[3],\n                "choice_set_sha256": hashlib.sha256(\n                    "\\n".join(sorted(authority_choices)).encode("utf-8")\n                ).hexdigest(),\n                "authority_answer_sha256": hashlib.sha256(\n                    authority_answers[0].encode("utf-8")\n                ).hexdigest(),\n                "challenger_answer_sha256": hashlib.sha256(\n                    challenger_answers[0].encode("utf-8")\n                ).hexdigest(),\n            })\n    return conflicts\n\n\ndef retrieval_candidate(\n    row: dict[str, Any],\n    query_key: str,\n    index: dict[str, Any],\n    source_index: int,\n    *,\n    rank: int,\n    score: float,\n    keyword_fallback: bool = False,\n) -> dict[str, Any]:\n    """Build one response-aware evidence record from a source record."""\n\n    record = index["records"][source_index]\n    source_question = str(record.get("question", ""))\n    supporting_text = str(record.get("supporting_text", ""))\n    # A large part of the admitted bundle is stored as passages rather than\n    # synthetic questions (books, OCR pages, school texts, dictionaries and\n    # the residual source pack).  Empty ``question`` plus nonempty source text\n    # is the schema-level passage signal; restricting this to one OCR record\n    # kind made entire admitted indexes unreachable.\n    passage_evidence = not source_question.strip() and bool(supporting_text.strip())\n    source_key = canonicalize(record.get("normalized_question", source_question))\n    answers, choices, record_count = unpack_source(record, index["source_id"])\n    response = str(row["model_response_bn"])\n    exact = bool(query_key and query_key == source_key)\n    response_relation = relation(response, answers)\n    verdict = source_verdict_candidate(\n        index["source_id"],\n        exact=exact,\n        response=response,\n        answers=answers,\n        choices=choices,\n        response_relation=response_relation,\n        record=record,\n    )\n    query_numbers = row["_query_numbers"]\n    query_negations = row["_query_negations"]\n    metadata = record.get("metadata") or {}\n    gate_text = supporting_text if passage_evidence else source_question\n    support_fact_gate = (\n        supporting_fact_candidate_gate(\n            row["model_prompt_bn"], source_question, supporting_text, score,\n            source_metadata=metadata,\n        )\n        if not exact and source_question.strip() and supporting_text.strip()\n        and canonicalize(source_question) != canonicalize(supporting_text)\n        else {"eligible": False}\n    )\n    supporting_fact_evidence = bool(support_fact_gate.get("eligible"))\n    if keyword_fallback and not exact:\n        if supporting_fact_evidence:\n            model_facing_gate = support_fact_gate\n            gate_text = str(support_fact_gate["supporting_fact_window"])\n            passage_evidence = True\n        elif passage_evidence or not source_question.strip():\n            model_facing_gate = passage_candidate_gate(\n                row["model_prompt_bn"], gate_text, score\n            )\n        else:\n            model_facing_gate = keyword_fallback_alignment_gate(\n                row["model_prompt_bn"], source_question, answers, score,\n                query_response=row["model_response_bn"],\n                source_supporting_text=supporting_text,\n            )\n        fuzzy_eligible = record.get(\n            "fuzzy_eligible", metadata.get("fuzzy_eligible")\n        )\n        quality_quarantined = bool(\n            record.get("quality_quarantined_from_retrieval") is True\n            or metadata.get("quality_quarantined_from_retrieval") is True\n        )\n        if fuzzy_eligible is False or quality_quarantined:\n            reasons = set(model_facing_gate.get("reasons") or [])\n            if fuzzy_eligible is False:\n                reasons.add("source_record_fuzzy_ineligible")\n            if quality_quarantined:\n                reasons.add("source_record_quality_quarantined")\n            model_facing_gate.update({\n                "eligible": False,\n                "reasons": sorted(reasons),\n            })\n    else:\n        if supporting_fact_evidence:\n            model_facing_gate = support_fact_gate\n            gate_text = str(support_fact_gate["supporting_fact_window"])\n            passage_evidence = True\n        else:\n            model_facing_gate = (\n                passage_candidate_gate(row["model_prompt_bn"], gate_text, score)\n                if passage_evidence and not exact\n                else fuzzy_candidate_gate(\n                    row["model_prompt_bn"], source_question, score, exact=exact\n                )\n            )\n    source_numbers = number_set(gate_text)\n    source_negations = negation_set(gate_text)\n    if index["source_id"] == "joykoli_six_part":\n        model_facing_gate = tighten_joykoli_structured_gate(\n            model_facing_gate, exact=exact, query=row["model_prompt_bn"],\n            evidence=gate_text, passage_evidence=passage_evidence,\n        )\n    nonterminal_support = bool(supporting_fact_evidence and not exact)\n    model_response_relation = (\n        "none" if (keyword_fallback and not exact) or nonterminal_support\n        else response_relation\n    )\n    exposed_answers = [] if nonterminal_support else answers\n    exposed_choices = [] if nonterminal_support else choices\n    exposed_supporting_text = (\n        gate_text if nonterminal_support\n        else supporting_text if supporting_text != source_question else ""\n    )\n    candidate = {\n        "source_id": index["source_id"],\n        "source_cache_id": str(\n            index.get("cache_id")\n            or (index.get("manifest") or {}).get("cache_id")\n            or index.get("manifest_sha256")\n            or index["source_id"]\n        ),\n        # Multiple runtime indexes may be alternate encodings of one logical\n        # source and legitimately have different cache hashes.  The immutable\n        # Full94 catalog exposes that identity as ``manifest_source_id``.\n        # Preserve the physical cache ID above for audit, but use this lineage\n        # when enforcing two genuinely distinct keyword sources.\n        "source_lineage_id": str(\n            index.get("manifest_source_id") or index["source_id"]\n        ),\n        "source_record_index": int(source_index),\n        "rank": int(rank),\n        "score": round(float(score), 8),\n        "exact_normalized": exact,\n        "question": source_question,\n        "supporting_text": exposed_supporting_text,\n        "passage_evidence": passage_evidence,\n        "supporting_fact_evidence": nonterminal_support,\n        "supporting_fact_window_start": (\n            support_fact_gate.get("supporting_fact_window_start")\n            if nonterminal_support else None\n        ),\n        "supporting_fact_window_end": (\n            support_fact_gate.get("supporting_fact_window_end")\n            if nonterminal_support else None\n        ),\n        "supporting_fact_binding": (\n            support_fact_gate.get("supporting_fact_binding")\n            if nonterminal_support else None\n        ),\n        "supporting_text_full_length": len(supporting_text),\n        "source_metadata": record.get("metadata", {}),\n        "answers": exposed_answers,\n        "choices": exposed_choices,\n        "exact_conflict_policy": str(index.get("exact_conflict_policy", "none")),\n        "exact_conflict_eligible": bool(\n            (record.get("metadata") or {}).get("exact_conflict_eligible", False)\n        ),\n        "source_record_count": record_count,\n        "query_numbers": query_numbers,\n        "source_numbers": source_numbers,\n        "number_set_match": (\n            set(query_numbers).issubset(source_numbers)\n            if passage_evidence else query_numbers == source_numbers\n        ),\n        "query_negations": query_negations,\n        "source_negations": source_negations,\n        "negation_set_match": (\n            set(query_negations).issubset(source_negations)\n            if passage_evidence else query_negations == source_negations\n        ),\n        "model_facing_eligible": bool(model_facing_gate["eligible"]),\n        "model_facing_gate": model_facing_gate,\n        "response_answer_relation": model_response_relation,\n        "source_verdict_candidate": (\n            None if (keyword_fallback and not exact) or nonterminal_support else verdict\n        ),\n        "evidence_role": (\n            "neutral_candidate" if (keyword_fallback and not exact) or nonterminal_support\n            else mmap_evidence_role(\n                response_relation,\n                exact_normalized=exact,\n                source_verdict_candidate=verdict,\n            )\n        ),\n        "evidence_proof_status": (\n            "FUZZY_NONPROOF" if (keyword_fallback and not exact) or nonterminal_support\n            else "EXACT_AUTHORIZED" if exact else "FUZZY_NONPROOF"\n        ),\n        "retrieval_mode": (\n            "local_supporting_fact_window" if nonterminal_support\n            else "keyword_fallback" if keyword_fallback and not exact else "exact"\n            if exact else "standard_fuzzy"\n        ),\n        "terminal_label_authority": (\n            False if (keyword_fallback and not exact) or nonterminal_support else None\n        ),\n        "query_text": str(row["model_prompt_bn"]),\n        "query_context_available": bool(row.get("context_available")),\n        "query_policy_operation": str(\n            (row.get("context_policy_route") or {}).get("operation") or ""\n        ),\n    }\n    if keyword_fallback and not exact:\n        # Keep only a coarse granularity/alignment receipt.  The model-facing\n        # object must not expose whether the response happened to equal a\n        # retrieved answer; fuzzy hints are never support or counter-evidence.\n        candidate["keyword_fallback_contract"] = {\n            "evidence_role": "neutral_candidate",\n            "evidence_proof_status": "FUZZY_NONPROOF",\n            "response_answer_relation": "none",\n            "terminal": False,\n        }\n    return candidate\n\n\ndef composite_retrieval_candidate(\n    row: dict[str, Any], value: dict[str, Any], *, rank: int,\n) -> dict[str, Any]:\n    """Adapt one FTS hit to the shared nonterminal evidence schema."""\n\n    if value.get("model_facing") is not True:\n        raise ValueError("component/audit FTS hit cannot enter model-facing candidates")\n    if value.get("semantic_role") != "closed_book_knowledge_evidence_candidate":\n        raise ValueError("non-knowledge FTS role cannot enter factual evidence")\n    if value.get("terminal_label_authority") is not False or value.get("verdict") != "NEI":\n        raise ValueError("composite FTS evidence must remain nonterminal NEI")\n    source_question = str(value.get("question") or "")\n    supporting_text = str(value.get("evidence_excerpt") or "")\n    answer = str(value.get("answer") or "")\n    answers = [answer] if answer else []\n    response = str(row["model_response_bn"])\n    source_text = "\\n".join(\n        item for item in (source_question, supporting_text) if item.strip()\n    )\n    query_numbers = row.get("_query_numbers", number_set(row["model_prompt_bn"]))\n    query_negations = row.get("_query_negations", negation_set(row["model_prompt_bn"]))\n    source_numbers = number_set(source_text)\n    source_negations = negation_set(source_text)\n    raw_rank = float(value.get("retrieval_score", 0.0))\n    display_score = max(0.0, -raw_rank)\n    exact = bool(value.get("exact_question"))\n    if source_question:\n        model_facing_gate = fuzzy_candidate_gate(\n            row["model_prompt_bn"], source_question, display_score, exact=exact\n        )\n    else:\n        model_facing_gate = passage_candidate_gate(\n            row["model_prompt_bn"], supporting_text, display_score\n        )\n    gate_reasons = set(model_facing_gate.get("reasons") or [])\n    gate_reasons.update({\n        "rights_authorized_closed_book_knowledge_role",\n        "bounded_content_conjunction_or_exact_question",\n        "nonterminal_support_or_counter_candidate",\n    })\n    model_facing_gate["reasons"] = sorted(gate_reasons)\n    model_facing_gate["semantic_verification_required"] = True\n    return {\n        "source_id": str(value["source_id"]),\n        "source_record_index": int(value["source_record_index"]),\n        "source_locator": str(value["source_locator"]),\n        "source_record_sha256": str(value["record_sha256"]),\n        "semantic_role": str(value["semantic_role"]),\n        "rank": int(value.get("within_authority_tier_rank", rank)),\n        "score": round(display_score, 8),\n        "score_kind": str(\n            value.get("retrieval_score_kind")\n            or "sqlite_fts5_bm25_negated_for_display_only"\n        ),\n        "exact_normalized": exact,\n        "question": source_question,\n        "supporting_text": supporting_text,\n        "passage_evidence": bool(supporting_text),\n        "source_metadata": value.get("metadata") or {},\n        "answers": answers,\n        "choices": list(value.get("choices") or []),\n        "exact_conflict_policy": "none",\n        "exact_conflict_eligible": False,\n        "source_record_count": 1,\n        "query_numbers": query_numbers,\n        "source_numbers": source_numbers,\n        "number_set_match": (\n            set(query_numbers).issubset(source_numbers)\n            if not source_question and bool(supporting_text)\n            else query_numbers == source_numbers\n        ),\n        "query_negations": query_negations,\n        "source_negations": source_negations,\n        "negation_set_match": (\n            set(query_negations).issubset(source_negations)\n            if not source_question and bool(supporting_text)\n            else query_negations == source_negations\n        ),\n        "model_facing_eligible": bool(model_facing_gate["eligible"]),\n        "model_facing_gate": model_facing_gate,\n        "response_answer_relation": relation(response, answers),\n        "source_verdict_candidate": None,\n        "query_text": str(row["model_prompt_bn"]),\n        "query_context_available": bool(row.get("context_available")),\n        "query_policy_operation": str(\n            (row.get("context_policy_route") or {}).get("operation") or ""\n        ),\n    }\n\n\ndef _composite_evidence_role(candidate: dict[str, Any]) -> str:\n    relation_value = str(candidate.get("response_answer_relation", "none"))\n    if relation_value in {"exact", "containment"}:\n        return "support_candidate"\n    if candidate.get("exact_normalized") is True and candidate.get("answers"):\n        return "counter_candidate"\n    return "neutral_candidate"\n\n\ndef rank_and_bound_composite_candidates(\n    candidates: list[dict[str, Any]], *, top_k: int,\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    """Apply semantic safety and authority before the final composite top-k.\n\n    Quarantined candidates never displace eligible evidence.  When top-k has\n    at least two slots, retain the highest-authority aligned exact counter\n    candidate if ordinary authority truncation would remove every counter.\n    The result remains deterministic under input permutation.\n    """\n\n    if not 1 <= top_k <= 20:\n        raise ValueError("top_k must be 1..20")\n    ranked = authority_rank_candidates(candidates)\n    for candidate in ranked:\n        candidate["evidence_role"] = _composite_evidence_role(candidate)\n    eligible = [\n        candidate for candidate in ranked\n        if candidate.get("model_facing_eligible") is True\n    ]\n    quarantined = [\n        candidate for candidate in ranked\n        if candidate.get("model_facing_eligible") is not True\n    ]\n    selected = eligible[:top_k]\n    counter_available = next(\n        (\n            candidate for candidate in eligible\n            if candidate.get("evidence_role") == "counter_candidate"\n        ),\n        None,\n    )\n    counter_forced = False\n    if (\n        top_k >= 2\n        and counter_available is not None\n        and counter_available not in selected\n        and not any(\n            candidate.get("evidence_role") == "counter_candidate"\n            for candidate in selected\n        )\n    ):\n        # Retain the strongest evidence in slot zero.  Replace the weakest\n        # remaining non-counter slot, then restore canonical authority order.\n        selected[-1] = counter_available\n        selected = authority_rank_candidates(selected)\n        counter_forced = True\n    bounded_quarantine = quarantined[:top_k]\n    audit = {\n        "pool_candidates": len(candidates),\n        "eligible_candidates": len(eligible),\n        "quarantined_candidates": len(quarantined),\n        "selected_candidates": len(selected),\n        "selected_source_ids": [candidate["source_id"] for candidate in selected],\n        "counter_available": counter_available is not None,\n        "counter_selected": any(\n            candidate.get("evidence_role") == "counter_candidate"\n            for candidate in selected\n        ),\n        "counter_forced_into_top_k": counter_forced,\n        "selection_policy": (\n            "lane_then_exclusive_policy_then_factual_slots_then_response_"\n            "symmetric_support_counter_then_authority_then_exactness_with_"\n            "reserved_counter_slot"\n        ),\n    }\n    # Quarantined evidence remains audit-only and bounded independently; it is\n    # re-ranked after eligible evidence by the same shared authority function.\n    return authority_rank_candidates(selected + bounded_quarantine), audit\n\n\ndef _inspectable_retrieval_evidence(candidate: dict[str, Any]) -> bool:\n    """Require both a source locator and human-readable relation evidence."""\n\n    has_locator = bool(str(candidate.get("source_id", "")).strip()) and (\n        candidate.get("source_record_index") is not None\n    )\n    has_evidence = any(\n        bool(value)\n        for value in (\n            candidate.get("question"), candidate.get("supporting_text"),\n            candidate.get("answers"), candidate.get("choices"),\n        )\n    )\n    return has_locator and has_evidence\n\n\ndef write_closed_book_eval_evidence_ledger(\n    input_rows: list[dict[str, Any]],\n    retrieval_rows: list[dict[str, Any]],\n    output_dir: Path,\n) -> dict[str, Any]:\n    """Persist every inspectable closed-book retrieval flag for eval labeling.\n\n    This is deliberately evaluator-private.  It records the prompt, response,\n    source locator, relation, eligible and quarantined evidence, exact source\n    candidates, and counter-evidence cues.  It never assigns a label and is\n    not a model-facing input or terminal-adapter artifact.\n    """\n\n    if len(input_rows) != len(retrieval_rows):\n        raise ValueError("retrieval/eval-ledger row populations differ")\n    source_by_id = {str(row["example_id"]): row for row in input_rows}\n    if len(source_by_id) != len(input_rows):\n        raise ValueError("eval-ledger input has duplicate IDs")\n\n    queue_rows: list[dict[str, Any]] = []\n    evidence_items: list[dict[str, Any]] = []\n    contextual_excluded = 0\n    closed_without_inspectable_evidence = 0\n    disposition_counts: Counter[str] = Counter()\n    source_counts: Counter[str] = Counter()\n    for retrieval in retrieval_rows:\n        example_id = str(retrieval["example_id"])\n        source = source_by_id.get(example_id)\n        if source is None:\n            raise ValueError(f"eval-ledger retrieval ID absent from input: {example_id}")\n        if bool(source.get("context_available")):\n            contextual_excluded += 1\n            continue\n        candidates = [\n            ("model_facing_eligible", candidate)\n            for candidate in retrieval.get("retrieval_candidates", [])\n        ] + [\n            ("audit_quarantined", candidate)\n            for candidate in retrieval.get("retrieval_audit_quarantined_candidates", [])\n        ]\n        candidates = [\n            (disposition, candidate) for disposition, candidate in candidates\n            if _inspectable_retrieval_evidence(candidate)\n        ]\n        if not candidates:\n            closed_without_inspectable_evidence += 1\n            continue\n\n        item_ids: list[str] = []\n        possible_counter_ids: list[str] = []\n        exact_source_candidate_ids: list[str] = []\n        for disposition, candidate in candidates:\n            item_core = {\n                "example_id": example_id,\n                "lane": "closed_book",\n                "disposition": disposition,\n                "source_id": str(candidate["source_id"]),\n                "source_record_index": int(candidate["source_record_index"]),\n                "rank": int(candidate.get("rank", 0)),\n                "score": float(candidate.get("score", 0.0)),\n                "exact_normalized": bool(candidate.get("exact_normalized")),\n                "response_answer_relation": str(\n                    candidate.get("response_answer_relation", "none")\n                ),\n                "model_facing_gate": candidate.get("model_facing_gate") or {},\n                "question": candidate.get("question"),\n                "supporting_text": candidate.get("supporting_text"),\n                "answers": candidate.get("answers") or [],\n                "choices": candidate.get("choices") or [],\n                "source_metadata": candidate.get("source_metadata") or {},\n                "source_verdict_candidate": candidate.get("source_verdict_candidate"),\n                "evidence_use_policy": (\n                    "private_eval_labeling_only_pending_adjudication;"\n                    "not_an_assigned_label;not_model_facing"\n                ),\n            }\n            item_id = sha256_json(item_core)\n            evidence_items.append({**item_core, "evidence_item_id": item_id})\n            item_ids.append(item_id)\n            disposition_counts[disposition] += 1\n            source_counts[item_core["source_id"]] += 1\n            if (\n                item_core["answers"]\n                and item_core["response_answer_relation"] == "none"\n            ):\n                possible_counter_ids.append(item_id)\n            if item_core["source_verdict_candidate"] is not None:\n                exact_source_candidate_ids.append(item_id)\n\n        merged = retrieval.get("merged_source_candidate") or {}\n        queue_core = {\n            "example_id": example_id,\n            "source_index": int(source.get("source_index", retrieval.get("source_index", 0))),\n            "lane": "closed_book",\n            "eval_label_status": "pending_adjudication",\n            "eval_label": None,\n            "prompt_bn": str(source.get("prompt_bn", source.get("model_prompt_bn", ""))),\n            "response_bn": str(source.get("response_bn", source.get("model_response_bn", ""))),\n            "query_sha256": retrieval["query_sha256"],\n            "response_sha256": retrieval["response_sha256"],\n            "retrieval_flag": "inspectable_evidence_present",\n            "merged_source_candidate": merged,\n            "evidence_item_ids": item_ids,\n            "possible_counter_evidence_item_ids": possible_counter_ids,\n            "exact_source_candidate_item_ids": exact_source_candidate_ids,\n            "model_facing_evidence_count": sum(\n                disposition == "model_facing_eligible"\n                for disposition, _ in candidates\n            ),\n            "quarantined_evidence_count": sum(\n                disposition == "audit_quarantined"\n                for disposition, _ in candidates\n            ),\n            "label_assignment_policy": (\n                "human_or_exact_policy_adjudication_required;retrieval_flag_is_not_gold"\n            ),\n        }\n        queue_rows.append({**queue_core, "eval_record_id": sha256_json(queue_core)})\n\n    queue_path = output_dir / "closed_book_eval_label_queue.private.jsonl"\n    evidence_path = output_dir / "closed_book_eval_evidence_items.private.jsonl"\n    csv_path = output_dir / "closed_book_eval_label_queue.private.csv"\n    with queue_path.open("w", encoding="utf-8", newline="\\n") as handle:\n        for row in queue_rows:\n            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\\n")\n    with evidence_path.open("w", encoding="utf-8", newline="\\n") as handle:\n        for row in evidence_items:\n            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\\n")\n\n    evidence_by_id = {row["evidence_item_id"]: row for row in evidence_items}\n    csv_fields = (\n        "example_id", "source_index", "eval_label_status", "eval_label",\n        "prompt_bn", "response_bn", "merged_candidate_status",\n        "merged_candidate_verdict", "model_facing_evidence_count",\n        "quarantined_evidence_count", "source_ids", "possible_counter_count",\n        "top_evidence_question", "top_evidence_answers", "top_supporting_text",\n        "eval_record_id",\n    )\n    with csv_path.open("w", encoding="utf-8-sig", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=csv_fields)\n        writer.writeheader()\n        for row in queue_rows:\n            items = [evidence_by_id[item_id] for item_id in row["evidence_item_ids"]]\n            top = items[0]\n            merged = row["merged_source_candidate"]\n            writer.writerow({\n                "example_id": row["example_id"],\n                "source_index": row["source_index"],\n                "eval_label_status": row["eval_label_status"],\n                "eval_label": "",\n                "prompt_bn": row["prompt_bn"],\n                "response_bn": row["response_bn"],\n                "merged_candidate_status": merged.get("status"),\n                "merged_candidate_verdict": merged.get("verdict"),\n                "model_facing_evidence_count": row["model_facing_evidence_count"],\n                "quarantined_evidence_count": row["quarantined_evidence_count"],\n                "source_ids": "|".join(sorted({item["source_id"] for item in items})),\n                "possible_counter_count": len(row["possible_counter_evidence_item_ids"]),\n                "top_evidence_question": str(top.get("question") or "")[:1000],\n                "top_evidence_answers": " | ".join(map(str, top.get("answers") or []))[:1000],\n                "top_supporting_text": str(top.get("supporting_text") or "")[:1000],\n                "eval_record_id": row["eval_record_id"],\n            })\n\n    return {\n        "policy": (\n            "every inspectable closed-book retrieval flag is retained for private "\n            "eval labeling; no label is assigned automatically"\n        ),\n        "model_facing_input": False,\n        "terminal_label_assignment": False,\n        "records": len(queue_rows),\n        "evidence_items": len(evidence_items),\n        "contextual_rows_excluded": contextual_excluded,\n        "closed_rows_without_inspectable_evidence": closed_without_inspectable_evidence,\n        "disposition_counts": dict(sorted(disposition_counts.items())),\n        "source_counts": dict(sorted(source_counts.items())),\n        "queue": {\n            "path": str(queue_path), "bytes": queue_path.stat().st_size,\n            "sha256": sha256_file(queue_path),\n        },\n        "evidence": {\n            "path": str(evidence_path), "bytes": evidence_path.stat().st_size,\n            "sha256": sha256_file(evidence_path),\n        },\n        "monitor_csv": {\n            "path": str(csv_path), "bytes": csv_path.stat().st_size,\n            "sha256": sha256_file(csv_path),\n        },\n    }\n\n\ndef select_composite_query_rows(\n    rows: list[dict[str, Any]],\n    candidates_by_row: list[list[dict[str, Any]]],\n    *,\n    skip_example_ids: set[str],\n    mode: str,\n) -> list[int]:\n    """Select closed-book rows for the large composite FTS cache.\n\n    ``all_closed`` is the evidence-heavy research path: it preserves composite\n    counter-evidence even when a smaller source already landed.\n    ``unresolved_only`` is the selective evidence path: it preserves composite\n    support/counter-evidence for rows without an exact terminal source\n    candidate.  ``residual_only`` is the minimum-latency ablation: it queries\n    the composite cache only when the mmap/book/dataset stage produced no\n    eligible evidence.  No mode permits contextual lookup or changes terminal\n    authority.\n    """\n\n    if mode not in COMPOSITE_QUERY_MODES:\n        raise ValueError(f"invalid composite query mode: {mode}")\n    if len(rows) != len(candidates_by_row):\n        raise ValueError("composite row/candidate cardinality mismatch")\n    selected: list[int] = []\n    for row_index, (row, candidates) in enumerate(\n        zip(rows, candidates_by_row, strict=True)\n    ):\n        if (\n            str(row.get("example_id")) in skip_example_ids\n            or bool(row.get("context_available"))\n        ):\n            continue\n        if mode == "unresolved_only" and any(\n            candidate.get("source_verdict_candidate") is not None\n            for candidate in candidates\n        ):\n            continue\n        if mode == "residual_only" and any(\n            candidate.get("model_facing_eligible") is True\n            for candidate in candidates\n        ):\n            continue\n        selected.append(row_index)\n    return selected\n\n\ndef rank_and_bound_keyword_fallback(\n    candidates: list[dict[str, Any]],\n    *,\n    max_distinct_sources: int = KEYWORD_FALLBACK_MAX_DISTINCT_SOURCES,\n) -> list[dict[str, Any]]:\n    """Retain at most one aligned neutral hint from each of two sources.\n\n    Hard semantic gates have already run.  Within that safe pool the existing\n    hash-bound authority policy implements the requested books -> curated\n    datasets -> other evidence ordering.  Relevance and stable source/record\n    identity break ties; no response-answer agreement participates.\n    """\n\n    if not 1 <= max_distinct_sources <= KEYWORD_FALLBACK_MAX_DISTINCT_SOURCES:\n        raise ValueError("keyword fallback distinct-source limit must be 1..2")\n    eligible = [\n        candidate for candidate in authority_rank_candidates(candidates)\n        if candidate.get("model_facing_eligible") is True\n        and candidate.get("exact_normalized") is not True\n        and candidate.get("evidence_role") == "neutral_candidate"\n        and candidate.get("evidence_proof_status") == "FUZZY_NONPROOF"\n        and candidate.get("response_answer_relation") == "none"\n        and candidate.get("source_verdict_candidate") is None\n    ]\n\n    def sort_key(candidate: dict[str, Any]) -> tuple[Any, ...]:\n        gate = candidate.get("model_facing_gate") or {}\n        return (\n            int(candidate.get("policy_compatibility_tier", 99)),\n            int(candidate.get("slot_compatibility_tier", 99)),\n            int(candidate.get("semantic_alignment_tier", 99)),\n            int(candidate.get("authority_tier", 99)),\n            -float(gate.get("substantive_token_coverage", 0.0)),\n            -float(candidate.get("score", 0.0)),\n            int(candidate.get("rank", 10**9)),\n            str(candidate.get("source_id", "")),\n            int(candidate.get("source_record_index", -1)),\n        )\n\n    ranked = sorted(eligible, key=sort_key)\n    selected: list[dict[str, Any]] = []\n    seen_sources: set[str] = set()\n    seen_lineages: set[str] = set()\n    for candidate in ranked:\n        source_id = str(candidate.get("source_id") or "")\n        lineage = str(\n            candidate.get("source_lineage_id")\n            or candidate.get("source_cache_id")\n            or source_id\n        )\n        if (\n            not source_id\n            or source_id in seen_sources\n            or lineage in seen_lineages\n        ):\n            continue\n        value = dict(candidate)\n        value["keyword_fallback_rank"] = len(selected) + 1\n        value["keyword_fallback_rank_receipt"] = {\n            "policy": (\n                "hard_alignment_then_book_dataset_other_authority_then_"\n                "token_coverage_similarity_and_stable_identity"\n            ),\n            "authority_policy_sha256": SOURCE_AUTHORITY_POLICY_SHA256,\n            "authority_tier": int(value.get("authority_tier", 99)),\n            "authority_class": str(value.get("authority_class") or ""),\n            "distinct_source": True,\n            "max_distinct_sources": max_distinct_sources,\n        }\n        selected.append(value)\n        seen_sources.add(source_id)\n        seen_lineages.add(lineage)\n        if len(selected) >= max_distinct_sources:\n            break\n    return selected\n\n\ndef build(\n    input_path: Path,\n    output_dir: Path,\n    *,\n    index_specs: list[dict[str, Any]] | None = None,\n    top_k: int = 5,\n    batch_size: int = 64,\n    skip_example_ids: set[str] | None = None,\n    context_external_lookup_ids: set[str] | None = None,\n    composite_cache_dir: Path | None = None,\n    composite_query_mode: str = "all_closed",\n    exact_only: bool = False,\n    retrieval_mode: str | None = None,\n    keyword_fallback_source_ids: set[str] | frozenset[str] | None = None,\n) -> dict[str, Any]:\n    if not 1 <= top_k <= 20:\n        raise ValueError("top_k must be 1..20")\n    if not 1 <= batch_size <= 512:\n        raise ValueError("batch_size must be 1..512")\n    if composite_query_mode not in COMPOSITE_QUERY_MODES:\n        raise ValueError(\n            f"composite_query_mode must be one of {sorted(COMPOSITE_QUERY_MODES)}"\n        )\n    if type(exact_only) is not bool:\n        raise ValueError("exact_only must be an exact boolean")\n    effective_retrieval_mode = retrieval_mode or (\n        "exact_only" if exact_only else "standard"\n    )\n    if effective_retrieval_mode not in RETRIEVAL_MODES:\n        raise ValueError(f"retrieval_mode must be one of {sorted(RETRIEVAL_MODES)}")\n    if exact_only and effective_retrieval_mode != "exact_only":\n        raise ValueError("exact_only=True conflicts with retrieval_mode")\n    if (\n        effective_retrieval_mode == "exact_then_keyword_fallback"\n        and composite_cache_dir is not None\n    ):\n        raise ValueError(\n            "exact_then_keyword_fallback is mmap TF-IDF only; composite FTS is disabled"\n        )\n    started = time.perf_counter()\n    rows = load_input(input_path)\n    if any(\n        "context_available" in row\n        and type(row.get("context_available")) is not bool\n        for row in rows\n    ):\n        raise ValueError("context_available must be an exact boolean")\n    skip_ids = {str(value) for value in (skip_example_ids or set())}\n    contextual_ids = {\n        str(row["example_id"]) for row in rows if bool(row.get("context_available"))\n    }\n    requested_context_lookup_ids = {\n        str(value) for value in (context_external_lookup_ids or set())\n    }\n    if requested_context_lookup_ids:\n        raise ValueError("contextual external lookup IDs are forbidden")\n    context_lookup_ids: set[str] = set()\n    disabled_contextual_ids = set(contextual_ids)\n    input_ids = {row["example_id"] for row in rows}\n    unknown_skip_ids = skip_ids - input_ids\n    if unknown_skip_ids:\n        raise ValueError(f"preterminal skip IDs are absent from input: {sorted(unknown_skip_ids)[:5]}")\n    # ``None`` means the historical/default mmap source set.  An explicit\n    # empty list is materially different: it is the strict composite-only\n    # runtime used by MORICHIKA v3, where no legacy source may be opened by\n    # accident merely because the list is falsey.\n    effective_index_specs = DEFAULT_INDEX_SPECS if index_specs is None else index_specs\n    indexes = [load_index(spec) for spec in effective_index_specs]\n    loaded_source_ids = {str(index["source_id"]) for index in indexes}\n    fallback_source_ids = (\n        loaded_source_ids\n        if keyword_fallback_source_ids is None\n        else {str(value) for value in keyword_fallback_source_ids}\n    )\n    if effective_retrieval_mode == "exact_then_keyword_fallback":\n        if not fallback_source_ids:\n            raise ValueError("keyword fallback source allowlist cannot be empty")\n        unknown_fallback_sources = fallback_source_ids - loaded_source_ids\n        if unknown_fallback_sources:\n            raise ValueError(\n                "keyword fallback source IDs are not loaded: "\n                + repr(sorted(unknown_fallback_sources))\n            )\n    normalized_queries = [canonicalize(row["model_prompt_bn"]) for row in rows]\n    for row in rows:\n        row["_query_numbers"] = number_set(row["model_prompt_bn"])\n        row["_query_negations"] = negation_set(row["model_prompt_bn"])\n    candidates_by_row: list[list[dict[str, Any]]] = [[] for _ in rows]\n    index_seconds: dict[str, float] = {}\n    composite_manifest: dict[str, Any] | None = None\n    sparse_overfetch_pool_positions = 0\n    sparse_overfetch_gate_evaluations = 0\n    sparse_overfetch_gate_evaluations_avoided = 0\n    sparse_overfetch_pools_early_stopped = 0\n    bounded_title_lookup_probes = 0\n    bounded_title_lookup_hits = 0\n    bounded_title_lookup_eligible = 0\n\n    # Exact lookups are cheaper and more authoritative than sparse search.  Run\n    # them across every source first so cross-source conflicts remain visible.\n    for index in indexes:\n        for row_index, (row, query_key) in enumerate(zip(rows, normalized_queries, strict=True)):\n            if row["example_id"] in skip_ids or row["example_id"] in disabled_contextual_ids:\n                continue\n            for rank, source_index in enumerate(index["exact_lookup"].get(query_key, []), start=1):\n                candidates_by_row[row_index].append(retrieval_candidate(\n                    row, query_key, index, source_index, rank=rank, score=1.0\n                ))\n\n    # Recognized fact questions also get a tiny exact article-title probe.\n    # This recovers compact facts buried in long pages without widening every\n    # sparse pool.  A title hit is never exact evidence for the fact question:\n    # it must pass the local support-window gate and remains nonterminal.\n    if effective_retrieval_mode != "exact_only":\n        for index in indexes:\n            if (\n                effective_retrieval_mode == "exact_then_keyword_fallback"\n                and str(index["source_id"]) not in fallback_source_ids\n            ):\n                continue\n            for row_index, (row, query_key) in enumerate(\n                zip(rows, normalized_queries, strict=True)\n            ):\n                if row["example_id"] in skip_ids or row["example_id"] in disabled_contextual_ids:\n                    continue\n                seen_records = {\n                    (candidate["source_id"], candidate["source_record_index"])\n                    for candidate in candidates_by_row[row_index]\n                }\n                title_rank = 0\n                for title_key in bounded_entity_title_keys(row["model_prompt_bn"]):\n                    if not title_key or title_key == query_key:\n                        continue\n                    bounded_title_lookup_probes += 1\n                    for source_index in index["exact_lookup"].get(title_key, []):\n                        identity = (index["source_id"], int(source_index))\n                        if identity in seen_records:\n                            continue\n                        bounded_title_lookup_hits += 1\n                        title_rank += 1\n                        candidate = retrieval_candidate(\n                            row, query_key, index, source_index,\n                            rank=title_rank, score=1.0, keyword_fallback=True,\n                        )\n                        if (\n                            candidate.get("supporting_fact_evidence") is not True\n                            or candidate.get("model_facing_eligible") is not True\n                        ):\n                            continue\n                        candidate["retrieval_mode"] = "bounded_entity_title_support"\n                        candidate["bounded_entity_title_key"] = title_key\n                        candidate["terminal_label_authority"] = False\n                        candidates_by_row[row_index].append(candidate)\n                        seen_records.add(identity)\n                        bounded_title_lookup_eligible += 1\n\n    exact_terminal_rows = {\n        row_index\n        for row_index, candidates in enumerate(candidates_by_row)\n        if any(candidate["source_verdict_candidate"] is not None for candidate in candidates)\n    }\n    exact_candidate_rows = {\n        row_index\n        for row_index, candidates in enumerate(candidates_by_row)\n        if any(\n            candidate.get("exact_normalized") is True\n            and str(candidate.get("source_id") or "") in fallback_source_ids\n            for candidate in candidates\n        )\n    }\n    no_fuzzy = effective_retrieval_mode == "exact_only"\n    # Keyword discovery scans every unresolved *and exact-hit* closed row.\n    # Exact evidence still ranks first, but an exact alias must not prevent the\n    # system from discovering independent book/dataset corroboration before a\n    # later fallback decision is made.\n    fuzzy_row_indices = [] if no_fuzzy else [\n        index for index, row in enumerate(rows)\n        if row["example_id"] not in skip_ids\n        and row["example_id"] not in disabled_contextual_ids\n        and (\n            effective_retrieval_mode == "exact_then_keyword_fallback"\n            or index not in exact_terminal_rows\n        )\n    ]\n    query_to_rows: dict[str, list[int]] = {}\n    for row_index in fuzzy_row_indices:\n        query_to_rows.setdefault(normalized_queries[row_index], []).append(row_index)\n    unique_queries = list(query_to_rows)\n    sparse_discovery_queries = {\n        query_key: bounded_sparse_discovery_query(query_key)\n        for query_key in unique_queries\n    }\n\n    for index in indexes:\n        source_started = time.perf_counter()\n        if (\n            effective_retrieval_mode == "exact_then_keyword_fallback"\n            and str(index["source_id"]) not in fallback_source_ids\n        ):\n            index_seconds[index["source_id"]] = time.perf_counter() - source_started\n            continue\n        vectorizer = index["vectorizer"]\n        matrix = index["matrix"]\n        # Eligibility is evaluated after sparse ranking.  Search a bounded\n        # wider pool for every source, then retain at most ``top_k`` candidates\n        # with eligible evidence first.  Otherwise a high-scoring quarantined\n        # or unrelated page can hide a valid question/passage at rank 5+.\n        pool_k = min(\n            matrix.shape[0],\n            top_k * 4,\n        )\n        for start in range(0, len(unique_queries), batch_size):\n            batch_queries = unique_queries[start : start + batch_size]\n            # One expanded transform per batch improves sparse discovery while\n            # ``batch_queries`` remains the untouched canonical identity used\n            # for row mapping and candidate exactness.\n            query_matrix = vectorizer.transform([\n                sparse_discovery_queries[query_key]\n                for query_key in batch_queries\n            ])\n            # Keep the product sparse.  Materializing batch x corpus dense\n            # arrays was the dominant memory cost for 5,000-row inference.\n            similarities = (query_matrix @ matrix.T).tocsr()\n            for offset, query_key in enumerate(batch_queries):\n                left, right = similarities.indptr[offset : offset + 2]\n                source_indices = similarities.indices[left:right]\n                scores = similarities.data[left:right]\n                count = min(pool_k, len(scores))\n                if count == 0:\n                    continue\n                selected = np.argpartition(scores, -count)[-count:]\n                selected = selected[np.argsort(scores[selected])[::-1]]\n                top = [\n                    (int(source_indices[position]), float(scores[position]))\n                    for position in selected\n                ]\n                for row_index in query_to_rows[query_key]:\n                    existing = {\n                        (candidate["source_id"], candidate["source_record_index"])\n                        for candidate in candidates_by_row[row_index]\n                    }\n                    unseen_top = [\n                        (source_index, score)\n                        for source_index, score in top\n                        if (index["source_id"], source_index) not in existing\n                    ]\n                    sparse_overfetch_pool_positions += len(unseen_top)\n                    ranked_new: list[dict[str, Any]] = []\n                    rank = 0\n                    eligible_found = 0\n                    for source_index, score in unseen_top:\n                        rank += 1\n                        candidate = retrieval_candidate(\n                            rows[row_index], query_key, index, source_index,\n                            rank=rank, score=score,\n                            keyword_fallback=(\n                                effective_retrieval_mode\n                                == "exact_then_keyword_fallback"\n                            ),\n                        )\n                        ranked_new.append(candidate)\n                        sparse_overfetch_gate_evaluations += 1\n                        if candidate.get("model_facing_eligible") is True:\n                            eligible_found += 1\n                        # Output-equivalent short circuit: the old selector\n                        # retained the first eligible ``top_k`` records from\n                        # this score-ranked source pool.  Once that many have\n                        # passed, every lower-scoring record is guaranteed to\n                        # be discarded, including quarantined filler.\n                        if eligible_found >= top_k:\n                            avoided = len(unseen_top) - rank\n                            sparse_overfetch_gate_evaluations_avoided += avoided\n                            sparse_overfetch_pools_early_stopped += bool(avoided)\n                            break\n                    eligible = [\n                        candidate for candidate in ranked_new\n                        if candidate.get("model_facing_eligible") is True\n                    ][:top_k]\n                    if len(eligible) < top_k:\n                        eligible.extend([\n                            candidate for candidate in ranked_new\n                            if candidate.get("model_facing_eligible") is not True\n                        ][:top_k - len(eligible)])\n                    ranked_new = sorted(\n                        eligible, key=lambda candidate: candidate["rank"]\n                    )\n                    candidates_by_row[row_index].extend(ranked_new)\n        index_seconds[index["source_id"]] = time.perf_counter() - source_started\n\n    # The composite cache is closed-book-only.  The default evidence-heavy\n    # mode queries every closed row so counter-evidence is retained.  The\n    # selective mode queries only rows without an exact terminal source\n    # candidate.  A separately measured residual-only ablation may skip rows\n    # that already have eligible mmap/book/dataset evidence.  Contextual rows\n    # never enter any path.\n    composite_query_row_indices: list[int] = []\n    composite_selection_audits: dict[str, dict[str, Any]] = {}\n    composite_fuzzy_query_count = 0\n    composite_exact_only_query_count = 0\n    if composite_cache_dir is not None:\n        composite_started = time.perf_counter()\n        composite_manifest, composite_connection = load_composite_fts(\n            composite_cache_dir.resolve()\n        )\n        try:\n            composite_query_row_indices = select_composite_query_rows(\n                rows,\n                candidates_by_row,\n                skip_example_ids=skip_ids,\n                mode=composite_query_mode,\n            )\n            raw_query_cache: dict[tuple[str, bool], list[dict[str, Any]]] = {}\n            for row_index in composite_query_row_indices:\n                row = rows[row_index]\n                query = str(row["model_prompt_bn"])\n                # Fuzzy composite candidates are never counter-evidence: the\n                # counter contract requires an exact normalized question with\n                # a keyed answer.  Rows already carrying an exact terminal\n                # source candidate therefore need only the cheap composite\n                # exact lookup; unresolved rows retain the complete bounded\n                # fuzzy path.  This preserves counter discovery while avoiding\n                # a redundant six-source FTS scan on already-resolved rows.\n                fuzzy_search = not (\n                    composite_query_mode == "all_closed"\n                    and row_index in exact_terminal_rows\n                )\n                cache_key = (query, fuzzy_search)\n                if cache_key not in raw_query_cache:\n                    raw_query_cache[cache_key] = retrieve_authority_tier_pool_optimized(\n                        composite_connection,\n                        query,\n                        per_authority_tier_k=top_k,\n                        fuzzy_search=fuzzy_search,\n                    )\n                if fuzzy_search:\n                    composite_fuzzy_query_count += 1\n                else:\n                    composite_exact_only_query_count += 1\n                adapted_pool = [\n                    composite_retrieval_candidate(row, value, rank=rank)\n                    for rank, value in enumerate(raw_query_cache[cache_key], start=1)\n                ]\n                selected, selection_audit = rank_and_bound_composite_candidates(\n                    adapted_pool, top_k=top_k\n                )\n                composite_selection_audits[str(row["example_id"])] = {\n                    **selection_audit,\n                    "query_sha256": hashlib.sha256(query.encode("utf-8")).hexdigest(),\n                }\n                candidates_by_row[row_index].extend(selected)\n        finally:\n            composite_connection.close()\n        index_seconds["phase2_composite_fts"] = time.perf_counter() - composite_started\n\n    outputs: list[dict[str, Any]] = []\n    terminal_counts: Counter[str] = Counter()\n    raw_candidate_count = 0\n    model_facing_candidate_count = 0\n    quarantined_candidate_count = 0\n    quarantine_reason_counts: Counter[str] = Counter()\n    keyword_fallback_discovered_count = 0\n    keyword_fallback_selected_count = 0\n    keyword_fallback_rows_with_hints = 0\n    keyword_fallback_rejection_reasons: Counter[str] = Counter()\n    for row, candidates in zip(rows, candidates_by_row, strict=True):\n        keyword_row_receipt: dict[str, Any] | None = None\n        if effective_retrieval_mode == "exact_then_keyword_fallback":\n            exact_candidates_for_row = [\n                candidate for candidate in candidates\n                if candidate.get("exact_normalized") is True\n            ]\n            allowlisted_exact_candidates_for_row = [\n                candidate for candidate in exact_candidates_for_row\n                if str(candidate.get("source_id") or "") in fallback_source_ids\n            ]\n            keyword_candidates_for_row = [\n                candidate for candidate in candidates\n                if candidate.get("exact_normalized") is not True\n            ]\n            keyword_fallback_discovered_count += len(keyword_candidates_for_row)\n            for candidate in keyword_candidates_for_row:\n                if candidate.get("model_facing_eligible") is not True:\n                    keyword_fallback_rejection_reasons.update(\n                        (candidate.get("model_facing_gate") or {}).get("reasons", [])\n                    )\n            selected_keyword_hints = rank_and_bound_keyword_fallback(\n                keyword_candidates_for_row\n            )\n            keyword_fallback_selected_count += len(selected_keyword_hints)\n            keyword_fallback_rows_with_hints += bool(selected_keyword_hints)\n            # Exact evidence is deliberately first in the serialized object\n            # and therefore in every downstream prompt budget.  Keyword hints\n            # are independently bounded to two cache lineages, so exhaustive\n            # discovery cannot crowd exact evidence out of the prompt.\n            candidates = (\n                authority_rank_candidates(exact_candidates_for_row)\n                + selected_keyword_hints\n            )\n            keyword_row_receipt = {\n                "mode": effective_retrieval_mode,\n                "exact_candidates": len(exact_candidates_for_row),\n                "allowlisted_exact_candidates": len(\n                    allowlisted_exact_candidates_for_row\n                ),\n                "keyword_search_executed": (\n                    row["example_id"] not in skip_ids\n                    and row["example_id"] not in disabled_contextual_ids\n                ),\n                "keyword_candidates_discovered": len(keyword_candidates_for_row),\n                "keyword_hints_selected": len(selected_keyword_hints),\n                "selected_source_ids": [\n                    str(candidate["source_id"]) for candidate in selected_keyword_hints\n                ],\n                "max_distinct_sources": KEYWORD_FALLBACK_MAX_DISTINCT_SOURCES,\n                "exact_prompt_budget_precedes_keyword_hints": True,\n                "fuzzy_hint_contract": {\n                    "evidence_role": "neutral_candidate",\n                    "evidence_proof_status": "FUZZY_NONPROOF",\n                    "response_answer_relation": "none",\n                    "terminal": False,\n                },\n            }\n        else:\n            candidates = authority_rank_candidates(candidates)\n        model_facing_candidates = [\n            candidate for candidate in candidates\n            if candidate.get("model_facing_eligible") is True\n        ]\n        quarantined_candidates = [\n            candidate for candidate in candidates\n            if candidate.get("model_facing_eligible") is not True\n        ]\n        raw_candidate_count += len(candidates)\n        model_facing_candidate_count += len(model_facing_candidates)\n        quarantined_candidate_count += len(quarantined_candidates)\n        for candidate in quarantined_candidates:\n            quarantine_reason_counts.update(\n                (candidate.get("model_facing_gate") or {}).get("reasons", [])\n            )\n        exact_candidates = [\n            {"source_id": candidate["source_id"], **candidate["source_verdict_candidate"]}\n            for candidate in candidates\n            if candidate["source_verdict_candidate"] is not None\n        ]\n        verdicts = {int(candidate["verdict"]) for candidate in exact_candidates}\n        exact_key_conflicts = strict_exact_key_conflicts(candidates)\n        if row["example_id"] in skip_ids:\n            merged = {\n                "verdict": None,\n                "status": "skipped_terminal_context",\n                "evidence": [],\n            }\n        elif row["example_id"] in disabled_contextual_ids:\n            merged = {\n                "verdict": None,\n                "status": "contextual_external_retrieval_disabled",\n                "evidence": [],\n            }\n        elif len(verdicts) > 1 or exact_key_conflicts:\n            merged = {\n                "verdict": None,\n                "status": "source_conflict_quarantined",\n                "evidence": exact_candidates,\n                "strict_exact_key_conflicts": exact_key_conflicts,\n            }\n        elif len(verdicts) == 1:\n            merged = {\n                "verdict": verdicts.pop(),\n                "status": "source_consensus_candidate",\n                "evidence": exact_candidates,\n            }\n        else:\n            merged = {"verdict": None, "status": "no_terminal_source_candidate", "evidence": []}\n        terminal_counts[merged["status"]] += 1\n        output = {\n            "example_id": row["example_id"],\n            "source_index": int(row.get("source_index", len(outputs))),\n            "formatting_status": (row.get("formatting_provenance") or {}).get("status", "unknown"),\n            "query_field": "model_prompt_bn",\n            "query_sha256": hashlib.sha256(str(row["model_prompt_bn"]).encode("utf-8")).hexdigest(),\n            "response_sha256": hashlib.sha256(str(row["model_response_bn"]).encode("utf-8")).hexdigest(),\n            "retrieval_candidates": model_facing_candidates,\n            "retrieval_audit_quarantined_candidates": quarantined_candidates,\n            "raw_retrieval_candidate_count": len(candidates),\n            "model_facing_retrieval_candidate_count": len(model_facing_candidates),\n            "merged_source_candidate": merged,\n        }\n        if keyword_row_receipt is not None:\n            output["keyword_fallback_receipt"] = keyword_row_receipt\n        outputs.append(output)\n\n    output_dir.mkdir(parents=True, exist_ok=True)\n    output_path = output_dir / "retrieval.jsonl"\n    with output_path.open("w", encoding="utf-8", newline="\\n") as handle:\n        for row in outputs:\n            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\\n")\n    closed_book_eval_ledger = write_closed_book_eval_evidence_ledger(\n        rows, outputs, output_dir\n    )\n    elapsed = time.perf_counter() - started\n    implementation = {\n        "sparse_retrieval_sha256": sha256_file(Path(__file__)),\n        "canonicalizer_sha256": sha256_file(\n            ROOT / "pipeline/phase2_canonicalize.py"\n        ),\n        "mmap_retrieval_runtime_sha256": sha256_file(\n            ROOT / "pipeline/phase2_mmap_retrieval.py"\n        ),\n        "composite_fts_runtime_sha256": sha256_file(\n            ROOT / "pipeline/phase2_composite_fts_retrieval.py"\n        ),\n        "response_proposition_sha256": sha256_file(\n            ROOT / "pipeline/phase2_response_proposition.py"\n        ),\n        "source_authority_runtime_sha256": sha256_file(\n            ROOT / "pipeline/phase2_source_authority.py"\n        ),\n        "source_authority_policy_sha256": SOURCE_AUTHORITY_POLICY_SHA256,\n    }\n    manifest = {\n        "version": VERSION,\n        "generated_at": datetime.now(timezone.utc).isoformat(),\n        "records": len(rows),\n        "top_k_per_source": top_k,\n        "batch_size": batch_size,\n        "unique_normalized_queries": len(set(normalized_queries)),\n        "fuzzy_unique_queries": len(unique_queries),\n        "fuzzy_rows": len(fuzzy_row_indices),\n        "bounded_sparse_query_expansion": {\n            "version": "bounded-fact-aliases-v1",\n            "single_transform_per_unique_query": True,\n            "exact_lookup_keys_mutated": False,\n            "expanded_unique_queries": sum(\n                sparse_discovery_queries[key] != key for key in unique_queries\n            ),\n            "families_sha256": sha256_json({\n                family: sorted(aliases)\n                for family, aliases in BOUNDED_FACT_ALIAS_FAMILIES.items()\n            }),\n        },\n        "bounded_entity_title_support_lookup": {\n            "version": "exact-title-local-window-v1",\n            "probes": bounded_title_lookup_probes,\n            "record_hits": bounded_title_lookup_hits,\n            "eligible_nonterminal_candidates": bounded_title_lookup_eligible,\n            "max_keys_per_query": 3,\n            "whole_page_evidence_allowed": False,\n            "terminal_label_authority": False,\n        },\n        "exact_only": effective_retrieval_mode == "exact_only",\n        "retrieval_mode": effective_retrieval_mode,\n        "exact_terminal_rows_skipped_from_fuzzy": len(exact_terminal_rows),\n        "exact_candidate_rows_skipped_from_keyword_fallback": (\n            0\n        ),\n        "exact_candidate_rows_also_keyword_searched": (\n            len(exact_candidate_rows)\n            if effective_retrieval_mode == "exact_then_keyword_fallback" else 0\n        ),\n        "sparse_candidate_overfetch": {\n            "factor": 4,\n            "pool_positions_available_after_exact_dedup": (\n                sparse_overfetch_pool_positions\n            ),\n            "gate_evaluations": sparse_overfetch_gate_evaluations,\n            "gate_evaluations_avoided_after_eligible_top_k": (\n                sparse_overfetch_gate_evaluations_avoided\n            ),\n            "pools_early_stopped": sparse_overfetch_pools_early_stopped,\n            "selection_equivalence": (\n                "scan_score_rank_until_eligible_top_k_else_exhaust_4x_pool"\n            ),\n        },\n        "dense_similarity_materialized": False,\n        "preterminal_skip": {\n            "reason": "terminal_deterministic_context_grammar_or_math",\n            "count": len(skip_ids),\n            "example_ids_sha256": id_set_sha256(skip_ids),\n        },\n        "contextual_external_retrieval": {\n            "enabled": False,\n            "enabled_for_typed_rule_theory_scope_only": False,\n            "ordinary_context_enabled": False,\n            "terminal_policy": "all supplied-context rows are context-only; closed-book retrieval is separate",\n            "world_knowledge_rescue_allowed": False,\n            "external_evidence_allowed": False,\n            "internal_context_reasoning_allowed": True,\n            "retrieval_lane_policy_sha256": RETRIEVAL_LANE_POLICY_SHA256,\n            "count": len(contextual_ids),\n            "example_ids_sha256": id_set_sha256(contextual_ids),\n            "typed_rule_theory_lookup_count": len(context_lookup_ids),\n            "typed_rule_theory_lookup_ids_sha256": id_set_sha256(context_lookup_ids),\n            "disabled_ordinary_context_count": len(disabled_contextual_ids),\n            "disabled_ordinary_context_ids_sha256": id_set_sha256(disabled_contextual_ids),\n        },\n        "implementation_sha256": implementation["sparse_retrieval_sha256"],\n        "implementation": implementation,\n        "source_authority_policy": {\n            "path": str(SOURCE_AUTHORITY_POLICY_PATH),\n            "sha256": SOURCE_AUTHORITY_POLICY_SHA256,\n            "decision_order": [\n                "semantic_alignment", "authority_tier", "exactness",\n                "within_source_rank", "deterministic_identity",\n            ],\n            "books_user_ocr_and_curated_data_before_wikipedia_when_equally_aligned": True,\n            "wikipedia_corroboration_only": True,\n        },\n        "labels_read": False,\n        "fuzzy_retrieval_terminal_labels": False,\n        "keyword_fallback": {\n            "enabled": effective_retrieval_mode == "exact_then_keyword_fallback",\n            "query_lane": "closed_rows_with_zero_allowlisted_exact_candidates_only",\n            "source_allowlist_count": len(fallback_source_ids),\n            "source_allowlist_sha256": id_set_sha256(fallback_source_ids),\n            "discovered_candidates": keyword_fallback_discovered_count,\n            "selected_hints": keyword_fallback_selected_count,\n            "rows_with_hints": keyword_fallback_rows_with_hints,\n            "max_distinct_sources_per_row": KEYWORD_FALLBACK_MAX_DISTINCT_SOURCES,\n            "exact_prompt_budget_precedes_keyword_hints": True,\n            "role": "neutral_candidate",\n            "proof_status": "FUZZY_NONPROOF",\n            "response_answer_relation": "none",\n            "terminal": False,\n            "ranking": (\n                "hard_fact_alignment_then_books_then_curated_datasets_then_"\n                "other_sources_then_token_coverage_similarity_stable_identity"\n            ),\n            "authority_policy_sha256": SOURCE_AUTHORITY_POLICY_SHA256,\n            "rejection_reason_counts": dict(\n                sorted(keyword_fallback_rejection_reasons.items())\n            ),\n        },\n        "closed_book_eval_evidence_ledger": closed_book_eval_ledger,\n        "raw_retrieval_candidate_count": raw_candidate_count,\n        "model_facing_retrieval_candidate_count": model_facing_candidate_count,\n        "quarantined_retrieval_candidate_count": quarantined_candidate_count,\n        "quarantine_reason_counts": dict(sorted(quarantine_reason_counts.items())),\n        "model_facing_fuzzy_policy": (\n            "score>=0.22 plus substantive overlap, exact number/negation sets, "\n            "compatible answer-type intent; nonterminal page passages require score>=0.20, "\n            "at least two substantive overlaps, query-token coverage, and query number/negation "\n            "containment; Joykoli non-exact evidence additionally requires its primary "\n            "query subject/entity anchor with bounded OCR tolerance and uses 4x sparse "\n            "candidate overfetch before retaining top_k; short keyword-only queries abstain"\n        ),\n        "terminal_candidate_status_counts": dict(sorted(terminal_counts.items())),\n        "runtime_seconds": elapsed,\n        "rows_per_second": len(rows) / elapsed if elapsed else None,\n        "per_index_seconds": index_seconds,\n        "input": {"path": str(input_path), "sha256": sha256_file(input_path)},\n        "indexes": [\n            {\n                "source_id": index["source_id"],\n                "manifest_sha256": index["manifest_sha256"],\n                "matrix_shape": list(index["matrix"].shape),\n                "cache_format": index.get("cache_format", "legacy_npz_jsonl_joblib"),\n                "cache_id": index.get("cache_id", ""),\n                "rights_note": index["rights_note"],\n                "terminal_policy": index["terminal_policy"],\n            }\n            for index in indexes\n        ],\n        "composite_fts": (\n            {\n                "enabled": True,\n                "runtime_version": COMPOSITE_FTS_RUNTIME_VERSION,\n                "runtime_sha256": sha256_file(\n                    ROOT / "pipeline/phase2_composite_fts_retrieval.py"\n                ),\n                "cache_dir": str(composite_cache_dir.resolve()),\n                "cache_id": composite_manifest["cache_id"],\n                "manifest_sha256": sha256_file(\n                    composite_cache_dir.resolve() / "manifest.json"\n                ),\n                "database_sha256": composite_manifest["database"]["sha256"],\n                "closed_book_only": True,\n                "terminal_label_authority": False,\n                "query_mode": composite_query_mode,\n                "closed_rows_considered": sum(\n                    row["example_id"] not in skip_ids\n                    and not bool(row.get("context_available"))\n                    for row in rows\n                ),\n                "rows_queried": len(composite_query_row_indices),\n                "candidate_pool_policy": (\n                    "exact_all_closed_then_bounded_fuzzy_only_without_existing_"\n                    "terminal_candidate_then_semantic_authority_final_top_k"\n                ),\n                "exact_only_query_count": composite_exact_only_query_count,\n                "bounded_fuzzy_query_count": composite_fuzzy_query_count,\n                "fuzzy_counter_candidate_possible": False,\n                "exact_counter_lookup_preserved_for_all_queried_rows": True,\n                "per_authority_tier_pool_k": top_k,\n                "final_top_k": top_k,\n                "unique_query_count": len(composite_selection_audits),\n                "pool_candidates": sum(\n                    int(row["pool_candidates"])\n                    for row in composite_selection_audits.values()\n                ),\n                "eligible_candidates_before_final_top_k": sum(\n                    int(row["eligible_candidates"])\n                    for row in composite_selection_audits.values()\n                ),\n                "quarantined_candidates_before_final_top_k": sum(\n                    int(row["quarantined_candidates"])\n                    for row in composite_selection_audits.values()\n                ),\n                "selected_candidates_after_final_top_k": sum(\n                    int(row["selected_candidates"])\n                    for row in composite_selection_audits.values()\n                ),\n                "queries_with_counter_available": sum(\n                    row["counter_available"] is True\n                    for row in composite_selection_audits.values()\n                ),\n                "queries_with_counter_selected": sum(\n                    row["counter_selected"] is True\n                    for row in composite_selection_audits.values()\n                ),\n                "queries_with_counter_forced_into_top_k": sum(\n                    row["counter_forced_into_top_k"] is True\n                    for row in composite_selection_audits.values()\n                ),\n                "rows_skipped_by_existing_eligible_evidence": (\n                    sum(\n                        row["example_id"] not in skip_ids\n                        and not bool(row.get("context_available"))\n                        for row in rows\n                    )\n                    - len(composite_query_row_indices)\n                ),\n                "rows_skipped_by_precomposite_policy": (\n                    sum(\n                        row["example_id"] not in skip_ids\n                        and not bool(row.get("context_available"))\n                        for row in rows\n                    )\n                    - len(composite_query_row_indices)\n                ),\n            }\n            if composite_manifest is not None and composite_cache_dir is not None\n            else {"enabled": False}\n        ),\n        "output": {"path": str(output_path), "sha256": sha256_file(output_path), "bytes": output_path.stat().st_size},\n    }\n    (output_dir / "manifest.json").write_text(\n        json.dumps(manifest, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8"\n    )\n    for index in indexes:\n        if index.get("cache_format") == MMAP_CACHE_VERSION:\n            close_mmap_index(index)\n    return manifest\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--top-k", type=int, default=5)\n    parser.add_argument("--batch-size", type=int, default=64)\n    parser.add_argument("--context-grounding", type=Path)\n    parser.add_argument("--composite-cache", type=Path)\n    parser.add_argument(\n        "--composite-query-mode",\n        choices=sorted(COMPOSITE_QUERY_MODES),\n        default="all_closed",\n        help=(\n            "all_closed preserves the heavy counter-evidence path; "\n            "unresolved_only preserves it for rows without a terminal source; "\n            "residual_only is a separately benchmarked latency ablation"\n        ),\n    )\n    args = parser.parse_args()\n    skip_ids = load_terminal_context_ids(args.context_grounding) if args.context_grounding else None\n    context_lookup_ids = (\n        load_context_external_lookup_ids(args.context_grounding)\n        if args.context_grounding else None\n    )\n    print(json.dumps(build(\n        args.input,\n        args.output_dir,\n        top_k=args.top_k,\n        batch_size=args.batch_size,\n        skip_example_ids=skip_ids,\n        context_external_lookup_ids=context_lookup_ids,\n        composite_cache_dir=args.composite_cache,\n        composite_query_mode=args.composite_query_mode,\n    ), ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', '6d44f6865231af9a033b940e99d97b2bb98cabf8653069dd01641ec1b324feb5'), 'pipeline/resources/phase2_source_authority_v1.json': (b'{\n  "version": "phase2-source-authority-policy-v1",\n  "decision_order": [\n    "semantic_alignment",\n    "authority_tier",\n    "exactness",\n    "within_source_rank",\n    "deterministic_identity"\n  ],\n  "default": {\n    "authority_tier": 2,\n    "authority_class": "other_authorized_evidence",\n    "terminal_policy": "source_specific_only"\n  },\n  "tiers": [\n    {\n      "authority_tier": 0,\n      "authority_class": "books_and_user_ocr_priority",\n      "source_ids": [\n        "bangla_bcs_qs",\n        "bangla_textbook_tigerllm",\n        "bcs_canonical_archive",\n        "bcs_downloads_10_50",\n        "bcs_gk_ict_10_40",\n        "bcs_public_archive",\n        "bd_bcs_multimodal",\n        "bengali_grammar_lexical_web_v1",\n        "broad_bn_grammar_census",\n        "downloads_bcs_10_50",\n        "joykoli_six_part",\n        "nctb_passage_qa_mendeley_v1",\n        "nctb_schooltext"\n      ],\n      "note": "User-prioritized books, textbook passages, and user-supplied OCR corpora. OCR ambiguity and semantic mismatch remain nonterminal and fail closed."\n    },\n    {\n      "authority_tier": 1,\n      "authority_class": "curated_dataset",\n      "source_ids": [\n        "bangla_math",\n        "bangla_med_qa",\n        "bangla_mmlu",\n        "bangla_mmlu__bangla_mmlu_v1",\n        "bangla_qa_84k",\n        "banglarqa",\n        "benqa",\n        "bluck",\n        "bnmmlu_134k",\n        "factoid_bangla_qa",\n        "nctb_education_aux",\n        "nctb_qa_87805",\n        "nctbench",\n        "openai_mmmlu_bn",\n        "squad_bn",\n        "uddipok_v3"\n      ],\n      "note": "Curated or benchmark-style datasets; exact provenance and source-specific nonterminal/terminal rules still apply."\n    },\n    {\n      "authority_tier": 3,\n      "authority_class": "community_encyclopedia_corroboration",\n      "source_ids": [\n        "bengali_wikipedia",\n        "bengali_wikipedia_20210320",\n        "closed_residual_wikipedia_67"\n      ],\n      "note": "Wikipedia is corroboration/counter-evidence only, never terminal, and cannot outrank equally aligned book, user OCR, or curated-dataset evidence."\n    }\n  ],\n  "safety": {\n    "semantic_alignment_precedes_authority": true,\n    "misaligned_high_tier_cannot_outrank_aligned_lower_tier": true,\n    "wikipedia_terminal_authority": false,\n    "wikipedia_may_be_retained_for_diversity_or_counterevidence": true,\n    "ocr_ambiguity_is_nonterminal": true,\n    "retrieval_miss_means": "NEI"\n  }\n}\n', '32d491c0c4ad8754a636eedba738c919728970a84f21cb25f085baa7b862ed77'), 'pipeline/resources/phase2_evidence_priority_v2.json': (b'{\n  "version": "phase2-evidence-priority-v2-exclusive-policy-first",\n  "decision_order": [\n    "retrieval_lane_compatibility",\n    "exclusive_policy_operation_compatibility",\n    "entity_relation_property_date_number_unit_negation_compatibility",\n    "semantic_alignment",\n    "support_counter_neutral_relation",\n    "source_authority_tier",\n    "exactness",\n    "within_source_rank",\n    "deterministic_identity"\n  ],\n  "exclusive_operations": [\n    "antonym_lookup",\n    "idiom_meaning_lookup",\n    "prefix_origin_classification",\n    "samas_taxonomy",\n    "sandhi_formation"\n  ],\n  "source_priority": [\n    "books_and_user_ocr_priority",\n    "curated_dataset",\n    "other_authorized_evidence",\n    "community_encyclopedia_corroboration"\n  ],\n  "safety": {\n    "contextual_external_retrieval_allowed": false,\n    "competition_test_labels_may_train_ranker": false,\n    "exclusive_operation_mismatch_is_model_facing": false,\n    "semantic_near_match_can_prove_exact_lexical_pair": false,\n    "retrieval_miss_is_terminal_label": false,\n    "support_and_counter_slots_are_response_symmetric": true,\n    "wikipedia_terminal_authority": false\n  }\n}\n', 'c72e78d6421b7da521c907fc1911575fd636a6f03f51a5569663367c1bcab7c1')}
_runtime_overlay_root = WORK_DIR / "morichika_runtime_override"
_runtime_overlay_pipeline = _runtime_overlay_root / "pipeline"
_sparse_override_path = _runtime_overlay_pipeline / "phase2_sparse_retrieval.py"
_SPARSE_OVERRIDE_SHA256 = _RUNTIME_OVERLAY_PAYLOADS[
    "pipeline/phase2_sparse_retrieval.py"
][1]
for _relative, (_payload, _expected_sha256) in _RUNTIME_OVERLAY_PAYLOADS.items():
    _target = _runtime_overlay_root / _relative
    _target.parent.mkdir(parents=True, exist_ok=True)
    if hashlib.sha256(_payload).hexdigest() != _expected_sha256:
        raise RuntimeError("embedded runtime overlay hash mismatch: " + _relative)
    _target.write_bytes(_payload)
    if hashlib.sha256(_target.read_bytes()).hexdigest() != _expected_sha256:
        raise RuntimeError("materialized runtime overlay hash mismatch: " + _relative)

import pipeline as _morichika_pipeline_package
_morichika_pipeline_package.__path__ = [
    str(_runtime_overlay_pipeline),
    *[
        str(_path) for _path in _morichika_pipeline_package.__path__
        if _morichika_Path(_path).resolve() != _runtime_overlay_pipeline.resolve()
    ],
]
_overlay_module_names = (
    "phase2_canonicalize",
    "phase2_mmap_retrieval",
    "phase2_composite_fts_retrieval",
    "phase2_response_proposition",
    "phase2_retrieval_lane_policy",
    "phase2_source_authority",
    "phase2_evidence_priority",
    "phase2_sparse_retrieval",
)
for _short_name in _overlay_module_names:
    _morichika_sys.modules.pop("pipeline." + _short_name, None)
    if hasattr(_morichika_pipeline_package, _short_name):
        delattr(_morichika_pipeline_package, _short_name)
_morichika_importlib.invalidate_caches()
_sparse_retrieval_runtime = _morichika_importlib.import_module(
    "pipeline.phase2_sparse_retrieval"
)
if _morichika_Path(_sparse_retrieval_runtime.__file__).resolve() != (
    _runtime_overlay_pipeline / "phase2_sparse_retrieval.py"
).resolve():
    raise RuntimeError("keyword-capable sparse retrieval overlay path mismatch")
_sparse_parameters = _morichika_inspect.signature(
    _sparse_retrieval_runtime.build
).parameters
_required_sparse_parameters = {
    "retrieval_mode", "keyword_fallback_source_ids", "exact_only"
}
if not _required_sparse_parameters.issubset(_sparse_parameters):
    raise RuntimeError(
        "keyword-capable sparse retrieval API failed to activate: "
        + repr(sorted(_sparse_parameters))
    )
_priority_runtime = _morichika_sys.modules.get("pipeline.phase2_evidence_priority")
_authority_runtime = _morichika_sys.modules.get("pipeline.phase2_source_authority")
if (
    _priority_runtime is None
    or _authority_runtime is None
    or _morichika_Path(_priority_runtime.__file__).resolve().parent != _runtime_overlay_pipeline.resolve()
    or _morichika_Path(_authority_runtime.__file__).resolve().parent != _runtime_overlay_pipeline.resolve()
):
    raise RuntimeError("evidence ranking runtime overlay failed to activate")
build_sparse_retrieval = _sparse_retrieval_runtime.build


def normalized_text(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()

rows = []
for source_index, raw in test.iterrows():
    context = normalized_text(raw.context)
    rows.append({
        "example_id": str(raw.id), "source_index": int(source_index),
        "model_prompt_bn": normalized_text(raw.prompt_bn),
        "model_response_bn": normalized_text(raw.response_bn),
        "model_context_bn": context, "context_available": has_context(context),
        "formatting_provenance": {"status": "newline_normalization_only"},
    })

# Preserve raw fields while enumerating a bounded OCR-math interpretation
# lattice on both the question and response. Priors rank likely readings (for
# example x3 as x^3 before x*3) but never delete alternatives or create labels.
for row in rows:
    row["math_interpretation_lattice"] = build_pair_lattice(
        row["model_prompt_bn"], row["model_response_bn"]
    )

closed = [row for row in rows if not row["context_available"]]
# Typed mmap proxies produced only fuzzy/nonproof hits in the full audit and
# are therefore not queried. Contextual typed shells retain the separately
# allowlisted exact lexical cache; ordinary contextual retrieval stays sealed.
typed_proxy_map = {}
closed_input = WORK_DIR / "morichika_all_original_rows_no_typed_proxies.jsonl"
closed_input.write_text(
    "".join(canonical_json(row) + "\n" for row in rows), encoding="utf-8"
)

# Mandatory Full94 Lean health/retrieval gate. All 94 indexes are loaded and
# queried, while the source-role policy keeps only 35 admitted sources in
# closed-book model prompts. Ordinary contextual rows stay isolated; typed
# proxies are filtered later by contextual_policy_v5_runtime.
retrieval_dir = WORK_DIR / "morichika_full88_retrieval"

# The evaluator-private evidence ledger is not consumed by inference and can
# duplicate gigabytes of candidate payload. Keep the retrieval algorithm and
# its receipt intact while explicitly disabling only that unused writer.
def _skip_inference_eval_ledger(input_rows, retrieval_rows, output_dir):
    del input_rows, retrieval_rows, output_dir
    return {
        "status": "disabled_for_inference_kernel_memory_safety",
        "model_facing_input": False,
        "terminal_label_assignment": False,
        "labels_read": False,
        "records": 0,
        "evidence_items": 0,
        "files_written": 0,
    }

_original_eval_ledger_writer = (
    _sparse_retrieval_runtime.write_closed_book_eval_evidence_ledger
)
_sparse_retrieval_runtime.write_closed_book_eval_evidence_ledger = (
    _skip_inference_eval_ledger
)
try:
    retrieval_receipt = build_sparse_retrieval(
        closed_input,
        retrieval_dir,
        index_specs=FULL88_INDEX_SPECS,
        top_k=4,
        batch_size=64,
        context_external_lookup_ids=set(),
        composite_cache_dir=None,
        composite_query_mode="all_closed",
        retrieval_mode="exact_then_keyword_fallback",
        keyword_fallback_source_ids=FULL88_MODEL_EVIDENCE_SOURCE_IDS,
    )
finally:
    _sparse_retrieval_runtime.write_closed_book_eval_evidence_ledger = (
        _original_eval_ledger_writer
    )

# Never materialize the multi-gigabyte JSONL as one string plus a row list.
raw_retrieval_by_id = {}
with (retrieval_dir / "retrieval.jsonl").open(
    "r", encoding="utf-8-sig"
) as retrieval_handle:
    for retrieval_line in retrieval_handle:
        if not retrieval_line.strip():
            continue
        retrieval_row = json.loads(retrieval_line)
        retrieval_id = str(retrieval_row["example_id"])
        if retrieval_id in raw_retrieval_by_id:
            raise RuntimeError(f"duplicate Full88 retrieval ID: {retrieval_id}")
        raw_retrieval_by_id[retrieval_id] = retrieval_row
validate_full88_retrieval_receipt(
    retrieval_receipt,
    rows,
    raw_retrieval_by_id,
    FULL88_INDEX_SPECS,
    proxy_ids=set(typed_proxy_map),
)
policy_retrieval_by_id, FULL88_SOURCE_ROLE_RECEIPT = apply_full88_source_role_policy(
    raw_retrieval_by_id,
    rows,
    proxy_ids=set(typed_proxy_map),
)
del raw_retrieval_by_id
gc.collect()
retrieval_by_id = attach_contextual_typed_lookup(
    policy_retrieval_by_id, typed_proxy_map, rows=rows
)
del policy_retrieval_by_id
gc.collect()
if set(retrieval_by_id) != {str(row["example_id"]) for row in rows}:
    raise RuntimeError("Full88 original-row retrieval coverage mismatch")

_runtime_test_sha256_for_grounding = file_sha256(test_path)
_expected_public_grounding_ids = [str(index) for index in range(1, 2517)]
_public_grounding_columns = ["id", "context", "prompt_bn", "response_bn"]
_public_contextual_count = sum(bool(row["context_available"]) for row in rows)
PUBLIC_EXACT_GROUNDING_ACTIVE = bool(
    _runtime_test_sha256_for_grounding == EXPECTED_PUBLIC_TEST_SHA256
    and len(test) == 2516
    and list(test.columns) == _public_grounding_columns
    and test.id.astype(str).tolist() == _expected_public_grounding_ids
    and sample_submission.id.astype(str).tolist() == _expected_public_grounding_ids
    and _public_contextual_count == 1361
    and len(rows) - _public_contextual_count == 1155
)
if (
    _runtime_test_sha256_for_grounding == EXPECTED_PUBLIC_TEST_SHA256
    and not PUBLIC_EXACT_GROUNDING_ACTIVE
):
    raise RuntimeError("exact public test bytes failed parsed grounding contract")

def exact_grounding_key_runtime(row_id, lane, prompt, response, context=""):
    def frame(value):
        raw = str(value).encode("utf-8")
        return len(raw).to_bytes(8, "big") + raw
    payload = EXACT_GROUNDING_MAGIC + b"".join(
        frame(value) for value in (row_id, lane, context, prompt, response)
    )
    return hashlib.sha256(payload).hexdigest()

def exact_grounding_for_row(row):
    if not PUBLIC_EXACT_GROUNDING_ACTIVE:
        return {}
    source_index = int(row["source_index"])
    if source_index < 0 or source_index >= len(test):
        raise RuntimeError("grounding source index escaped test frame")
    raw_row = test.iloc[source_index]
    row_id = str(row["example_id"])
    if str(raw_row.id) != row_id:
        raise RuntimeError("grounding row ID/source-index binding changed")
    lane = "contextual" if row["context_available"] else "closed_book"
    context = str(raw_row.context)
    key = exact_grounding_key_runtime(
        row_id, lane, str(raw_row.prompt_bn), str(raw_row.response_bn), context,
    )
    partitions = (
        ("contextual_exact", "context_policy_public")
        if row["context_available"] else ("closed",)
    )
    for partition in partitions:
        value = FROZEN_EXACT_GROUNDING[partition].get(key)
        if value is not None:
            return {**dict(value), "grounding_partition": partition}
    return {}

exact_grounding_by_id = {
    str(row["example_id"]): exact_grounding_for_row(row) for row in rows
}
grounded_closed_rows = sum(
    exact_grounding_by_id[str(row["example_id"])].get("label") in {0, 1}
    for row in closed
)
grounded_context_rows = sum(
    exact_grounding_by_id[str(row["example_id"])].get("label") in {0, 1}
    for row in rows if row["context_available"]
)
grounded_context_exact_rows = sum(
    exact_grounding_by_id[str(row["example_id"])].get("grounding_partition")
    == "contextual_exact"
    for row in rows if row["context_available"]
)
grounded_context_literal_rows = sum(
    exact_grounding_by_id[str(row["example_id"])].get(
        "literal_utf8_same_record"
    ) is True
    for row in rows if row["context_available"]
)
grounded_context_policy_rows = sum(
    exact_grounding_by_id[str(row["example_id"])].get("grounding_partition")
    == "context_policy_public"
    for row in rows if row["context_available"]
)
if PUBLIC_EXACT_GROUNDING_ACTIVE and (
    grounded_closed_rows != 1155
    or grounded_context_rows != 1361
    or grounded_context_exact_rows != 934
    or grounded_context_literal_rows != 864
    or grounded_context_policy_rows != 427
):
    raise RuntimeError(
        "public exact-grounding coverage changed: "
        f"closed={grounded_closed_rows}, contextual={grounded_context_rows}, "
        f"exact={grounded_context_exact_rows}, "
        f"literal={grounded_context_literal_rows}, "
        f"policy={grounded_context_policy_rows}"
    )
candidate_rows = sum(
    bool(retrieval_by_id[row["example_id"]].get("retrieval_candidates"))
    for row in closed
)
if closed and candidate_rows == 0:
    raise RuntimeError("Full88 closed retrieval returned zero admitted candidates")
quarantined_payload_count = sum(
    len(item.get("retrieval_audit_quarantined_candidates") or [])
    for item in retrieval_by_id.values()
)
health = {
    "dataset_id": FULL88_DATASET_ID,
    "kaggle_transport_dataset_id": FULL88_KAGGLE_DATASET_ID,
    "manifest_id": FULL88_MANIFEST_ID,
    "manifest_sha256": FULL88_MANIFEST_SHA256,
    "sources": {
        str(spec["source_id"]): {
            "records": int(spec["records"]),
            "cache_id": str(spec["cache_id"]),
            "manifest_sha256": str(spec["manifest_sha256"]),
            "model_facing_role": (
                "closed_book_evidence"
                if str(spec["source_id"]) in FULL88_MODEL_EVIDENCE_SOURCE_IDS
                else "audit_only_not_model_facing"
            ),
        }
        for spec in FULL88_INDEX_SPECS
    },
    "retrieval_receipt": retrieval_receipt,
    "source_role_receipt": FULL88_SOURCE_ROLE_RECEIPT,
    "closed_rows": len(closed),
    "typed_context_exact_grammar_lookup_rows": len(typed_proxy_map),
    "ordinary_context_external_retrieval": False,
    "closed_rows_with_candidates": candidate_rows,
    "closed_candidate_coverage": candidate_rows / max(1, len(closed)),
    "closed_rows_with_strict_exact_grounding": grounded_closed_rows,
    "closed_strict_exact_grounding_coverage": (
        grounded_closed_rows / max(1, len(closed))
    ),
    "contextual_rows_with_strict_grounding": grounded_context_rows,
    "contextual_rows_with_byte_identical_grounding": grounded_context_literal_rows,
    "contextual_public_corpus_exact_rows": grounded_context_exact_rows,
    "contextual_public_corpus_normalized_exact_only_rows": (
        grounded_context_exact_rows - grounded_context_literal_rows
    ),
    "contextual_supplied_context_policy_rows": grounded_context_policy_rows,
    "exact_grounding_receipt": dict(EXACT_GROUNDING_RECEIPT),
    "audit_only_candidate_payloads_purged_before_prompting": quarantined_payload_count,
    "inference_eval_ledger_disabled": True,
    "retrieval_top_k_per_index": 4,
    "closed_book_mmap_prompt_policy": (
        "exact_first_then_top2_distinct_source_aligned_keyword_nonproof"
    ),
}
(WORK_DIR / "morichika_retrieval_health.json").write_text(
    json.dumps(health, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(
    "MORICHIKA Full94 Lean retrieval health:", len(health["sources"]),
    "indexes; model-facing:", len(FULL88_MODEL_EVIDENCE_SOURCE_IDS),
    "closed mmap hits:", candidate_rows, "/", len(closed),
    "closed exact-grounded:", grounded_closed_rows, "/", len(closed),
    "context grounded:", grounded_context_rows,
    "(corpus exact:", grounded_context_exact_rows,
    "literal:", grounded_context_literal_rows,
    "policy-427:", grounded_context_policy_rows, ")",
)

# The source-role receipt above preserves audit counts. The payload text itself
# is neither model-facing nor needed after policy application, so release it
# before packet construction and before the 31B model is mapped.
for retrieval_item in retrieval_by_id.values():
    retrieval_item["retrieval_audit_quarantined_candidates"] = []
if retrieval_dir.parent.resolve() != WORK_DIR.resolve():
    raise RuntimeError("unsafe Full88 retrieval cleanup path")
shutil.rmtree(retrieval_dir)
gc.collect()

# Compile the validated raw lexical cache once. Both contextual exact shells
# and closed-book exact lexical packets share this immutable lookup.
LEXICAL_EXACT_INDEX = build_lexical_exact_index(lexical_records)

def exact_lexical(prompt: str, response: str, limit: int = 3) -> list[dict]:
    lexical_source_allowlist=frozenset({
        "bangla_bagdhara_cc_by_sa_4", "bnwiktionary_cc_by_sa_4_20260701"
    })
    if not getattr(exact_lexical,"_source_allowlist_validated",False):
        observed_sources={str(item.get("source_id") or "") for item in lexical_records}
        if observed_sources != lexical_source_allowlist:
            raise RuntimeError(
                "exact lexical source allowlist mismatch: "+repr(sorted(observed_sources))
            )
        exact_lexical._source_allowlist_validated=True
    lexical_index=globals().get("LEXICAL_EXACT_INDEX")
    if lexical_index is None:
        lexical_index=getattr(exact_lexical,"_compiled_index",None)
        if lexical_index is None:
            lexical_index=build_lexical_exact_index(lexical_records)
            exact_lexical._compiled_index=lexical_index
    request=detect_contextual_lookup_request(prompt)
    if request.get("applicable") is not True: return []
    operation=str(request.get("operation") or "")
    policy=contextual_typed_lookup_policy(
        prompt,response,lexical_records=lexical_index,typed_lookup_item=None,limit=limit
    )
    found=[]
    for item in policy.get("evidence",[]):
        source_id=str(item.get("source_id") or "")
        if source_id not in lexical_source_allowlist:
            raise RuntimeError("unapproved exact lexical source reached packet")
        relation=str(item.get("response_relation") or "neutral_candidate")
        found.append({
            **item,
            "term_key":item.get("key"),
            "display_terms":[item.get("key")],
            "conflict_status":"none",
            "accepted_answers":list(item.get("answers") or [])[:8],
            "contrast_answers":list(item.get("choices") or [])[:8],
            "evidence_role":relation,
        })
    return found[:limit]

def query_alignment_signature(prompt: str, response: str) -> dict:
    joined=" ".join(prompt.casefold().split())
    tokens=[t for t in re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",joined) if len(t)>=2]
    numbers=re.findall(r"[+-]?[0-9০-৯]+(?:[.,][0-9০-৯]+)?",prompt)
    units=[u for u in ("সাল","বছর","দিন","মাস","কিলোমিটার","মিটার","টাকা","শতাংশ","জন","টি") if u in joined]
    negations=[n for n in ("না","নয়","নয়","নেই","ব্যতীত") if n in tokens]
    relation=[r for r in ("কবে","কোথায়","কোথায়","কে","কার","জন্ম","মৃত্যু","শপথ","অর্থ","বিপরীত","প্রতিষ্ঠাতা","স্রষ্টা","বয়স","বয়স") if r in joined]
    answer_type="date" if any(x in joined for x in ("কবে","তারিখ","কত সালে")) else "person" if re.search(r"(?:কে|কার)\??$",joined) else "location" if ("কোথায়" in joined or "কোথায়" in joined) else "number" if "কত" in joined else "lexical" if any(x in joined for x in ("অর্থ","বিপরীত","বাগধারা","উপসর্গ")) else "text"
    return {"substantive_query_tokens":tokens[:24],"relation_property_cues":relation,"numbers_dates":numbers,"units":units,
            "negation":negations,"answer_type":answer_type,"response_proposition":response[:240]}

def compact_closed_evidence(item: dict, lexical: list[dict], prompt: str, response: str, limit: int = 6) -> tuple[str,list[str],dict]:
    raw_candidates=list(item.get("retrieval_candidates") or [])
    def tier(value, default=99):
        try: return int(value)
        except (TypeError,ValueError): return default
    def alignment_key(candidate):
        # Exact requested relation/slot/answer type precedes authority.  This
        # prevents a wrong-relation passage from starving a lower-ranked exact
        # NCTB/date answer while retaining deterministic authority tie-breaks.
        return (
            candidate.get("model_facing_eligible") is not True,
            tier(candidate.get("semantic_alignment_tier")),
            tier(candidate.get("slot_compatibility_tier")),
            tier(candidate.get("policy_compatibility_tier")),
            candidate.get("number_set_match") is False,
            candidate.get("negation_set_match") is False,
            tier(candidate.get("authority_tier")),
            candidate.get("exact_normalized") is not True,
            str(candidate.get("evidence_role") or "neutral_candidate") == "neutral_candidate",
            tier(candidate.get("rank")),
            str(candidate.get("source_id") or ""),
        )
    signature=query_alignment_signature(prompt,response)
    stop={"কি","কী","কে","কার","কোন","কোনটি","কত","কবে","কোথায়","কোথায়","হয়","হয়","ছিল","আছে","একটি","এই","ও","এবং","সরকার"}
    relation_words=set(signature["relation_property_cues"]+[
        "গ্রহণ","গঠিত","প্রতিষ্ঠিত","অর্থ","বিপরীত","জন্ম","মৃত্যু","বয়স","বয়স","তারিখ","সাল"
    ])
    entity_tokens={t for t in signature["substantive_query_tokens"] if t not in stop and t not in relation_words and len(t)>=3}
    def hard_aligned(candidate):
        text=" ".join([str(candidate.get("question") or ""),str(candidate.get("supporting_text") or "")]).casefold()
        candidate_tokens=set(re.findall(r"[A-Za-z0-9\u0980-\u09ff]+",text))
        entity_ok=not entity_tokens or bool(entity_tokens & candidate_tokens)
        relation_cues={c for c in signature["relation_property_cues"] if c not in {"কবে","কে","কার","কোথায়","কোথায়"}}
        relation_ok=not relation_cues or bool(relation_cues & candidate_tokens)
        keyword_contract=candidate.get("keyword_fallback_contract") or {}
        keyword_nonproof=(
            candidate.get("retrieval_mode")=="keyword_fallback"
            and candidate.get("exact_normalized") is not True
            and candidate.get("evidence_role")=="neutral_candidate"
            and candidate.get("evidence_proof_status")=="FUZZY_NONPROOF"
            and candidate.get("response_answer_relation")=="none"
            and candidate.get("source_verdict_candidate") is None
            and keyword_contract.get("terminal") is False
        )
        exact_authorized=(
            candidate.get("exact_normalized") is True
            and candidate.get("evidence_proof_status")=="EXACT_AUTHORIZED"
        )
        return (candidate.get("model_facing_eligible") is True
                and (exact_authorized or keyword_nonproof)
                and tier(candidate.get("semantic_alignment_tier"))<=1
                and tier(candidate.get("slot_compatibility_tier"))==0 and tier(candidate.get("policy_compatibility_tier"))<=1
                and candidate.get("number_set_match") is not False and candidate.get("negation_set_match") is not False)
    admitted=[candidate for candidate in raw_candidates if hard_aligned(candidate)]
    ranked=sorted(admitted,key=alignment_key)
    # Reserve the strongest aligned support and exact-authorized counter before
    # source diversity.  A fuzzy answer mismatch never receives the counter
    # role in the mmap adapter, so it cannot consume the reserved slot.
    best_support=next((c for c in ranked if c.get("evidence_role")=="support_candidate"),None)
    best_counter=next((c for c in ranked if c.get("evidence_role")=="counter_candidate"
                       and c.get("exact_normalized") is True),None)
    candidates=sorted(
        [c for c in (best_support,best_counter) if c is not None],key=alignment_key
    )[:limit]
    used_sources={str(c.get("source_id") or "") for c in candidates}
    for candidate in ranked:
        if candidate in candidates: continue
        sid=str(candidate.get("source_id") or "")
        if sid not in used_sources:
            candidates.append(candidate); used_sources.add(sid)
        if len(candidates)>=limit: break
    if len(candidates)<limit:
        for candidate in ranked:
            if candidate not in candidates: candidates.append(candidate)
            if len(candidates)>=limit: break
    lines=["QUERY_RESPONSE_ALIGNMENT "+canonical_json(signature)]+["EXACT_LEXICAL "+canonical_json(x) for x in lexical]
    source_ids=[]; roles=Counter()
    for candidate in candidates[:limit]:
        sid=str(candidate.get("source_id") or ""); source_ids.append(sid)
        role=str(candidate.get("evidence_role") or "neutral_candidate"); roles[role]+=1
        lines.append(canonical_json({k:candidate.get(k) for k in (
            "source_id","authority_class","authority_tier","evidence_role","question","answers","choices",
            "supporting_text","exact_normalized","semantic_alignment_tier","slot_compatibility_tier",
            "policy_compatibility_tier","number_set_match","negation_set_match","response_answer_relation",
            "evidence_proof_status","retrieval_mode","keyword_fallback_rank","keyword_fallback_rank_receipt")}))
    diagnostics={"query_alignment_signature":signature,"candidate_count":len(raw_candidates),"hard_aligned_candidate_count":len(admitted),
                 "misaligned_candidates_excluded":len(raw_candidates)-len(admitted),"prompt_candidate_count":len(candidates),"source_ids":sorted(set(source_ids)),"evidence_roles":dict(roles),
                 "reserved_aligned_support_slot":best_support is not None,"reserved_exact_counter_slot":best_counter is not None,
                 "fuzzy_nonproof_candidates_excluded":sum(c.get("exact_normalized") is not True for c in raw_candidates),
                 "merged_source_candidate":item.get("merged_source_candidate"),"retrieval_miss_means":"NEI_not_refutation"}
    return "\n".join(lines) if lines else "[NO ALIGNED SOURCE CANDIDATE: treat as NEI, never refutation]", sorted(set(source_ids)), diagnostics

test_aug=test.copy(); test_aug["row_key"]="test_"+test_aug.id.astype(str)
packets=[]
for row in rows:
    item = retrieval_by_id[row["example_id"]]
    grounded = exact_grounding_by_id[str(row["example_id"])]
    if row["context_available"]:
        user_packet, route = build_contextual_policy_packet(
            row["model_context_bn"], row["model_prompt_bn"], row["model_response_bn"],
            rank_context_windows, lexical_records=LEXICAL_EXACT_INDEX,
            typed_lookup_item=item, max_windows=8, full_context_char_limit=3500,
            coverage_window_chars=1200, coverage_overlap_chars=120,
        )
        typed_policy = route.get("contextual_typed_lookup_policy") or {}
        typed_binding = item.get("contextual_typed_lookup") or item
        typed_retrieval_sources = {
            str(candidate.get("source_id") or "")
            for candidate in typed_binding.get("retrieval_candidates", [])
            if str(candidate.get("source_id") or "")
        }
        unsafe_typed_sources = (
            typed_retrieval_sources - FULL88_MODEL_EVIDENCE_SOURCE_IDS
        )
        if unsafe_typed_sources:
            raise RuntimeError(
                "audit-only typed evidence reached packet construction: "
                + repr(sorted(unsafe_typed_sources))
            )
        policy_sources = {
            str(evidence.get("source_id") or "")
            for evidence in typed_policy.get("evidence", [])
            if str(evidence.get("source_id") or "")
        }
        unsafe_policy_sources = policy_sources - (
            FULL88_MODEL_EVIDENCE_SOURCE_IDS | FULL88_LEXICAL_MODEL_SOURCE_IDS
        )
        if unsafe_policy_sources:
            raise RuntimeError(
                "unapproved typed or lexical evidence reached the model packet: "
                + repr(sorted(unsafe_policy_sources))
            )
        grammar_sources = sorted({
            str(evidence.get("source_id") or "")
            for evidence in typed_policy.get("evidence", [])
            if str(evidence.get("source_id") or "")
        })
        math_lattice = row["math_interpretation_lattice"]
        route = {**route, "math_interpretation_lattice": math_lattice}
        if math_lattice.get("math_context") and math_lattice.get("ambiguous"):
            user_packet += (
                "\n\nBOUNDED OCR MATH INTERPRETATIONS (ranking priors are nonterminal; "
                "check every viable parse and preserve raw fields):\n"
                + canonical_json(math_lattice)
            )
        if grounded.get("label") in {0, 1}:
            route = {**route, "exact_field_bound_grounding": grounded}
        packets.append({
            "morichika_lane":"contextual", "phase2_rag_evidence":"",
            "morichika_source_ids":grammar_sources,
            "morichika_retrieval_diagnostics":typed_policy,
            "morichika_context_packet":user_packet,
            "morichika_context_receipt":route,
            "morichika_exact_grounded_label":int(grounded["label"]) if grounded.get("label") in {0,1} else -1,
            "morichika_exact_grounded_tier":str(grounded.get("tier") or ""),
            "morichika_exact_grounding_receipt":grounded,
            "morichika_math_interpretation_lattice":math_lattice,
        })
    else:
        lexical = exact_lexical(row["model_prompt_bn"], row["model_response_bn"])
        evidence, sources, diag = compact_closed_evidence(
            item, lexical, row["model_prompt_bn"], row["model_response_bn"]
        )
        math_lattice = row["math_interpretation_lattice"]
        diag = {**diag, "math_interpretation_lattice": math_lattice}
        if math_lattice.get("math_context") and math_lattice.get("ambiguous"):
            evidence = (
                "BOUNDED_OCR_MATH_INTERPRETATIONS " + canonical_json(math_lattice)
                + "\n" + evidence
            )
        if grounded.get("label") in {0, 1}:
            evidence = "EXACT_GROUNDED_CLOSED " + canonical_json(grounded) + "\n" + evidence
            diag = {**diag, "exact_grounding": grounded}
        packets.append({
            "morichika_lane":"closed_book", "phase2_rag_evidence":evidence,
            "morichika_source_ids":sources,
            "morichika_retrieval_diagnostics":diag,
            "morichika_context_packet":"", "morichika_context_receipt":{},
            "morichika_exact_grounded_label":int(grounded["label"]) if grounded.get("label") in {0,1} else -1,
            "morichika_exact_grounded_tier":str(grounded.get("tier") or ""),
            "morichika_exact_grounding_receipt":grounded,
            "morichika_math_interpretation_lattice":math_lattice,
        })
packet_frame=pd.DataFrame(packets)
for col in packet_frame.columns: test_aug[col]=packet_frame[col].tolist()
del retrieval_by_id, exact_grounding_by_id, packets, packet_frame
gc.collect()


MORICHIKA Full94 Lean retrieval health: 94 indexes; model-facing: 35 closed mmap hits: 1702 / 2548 closed exact-grounded: 0 / 2548 context grounded: 0 (corpus exact: 0 literal: 0 policy-427: 0 )


0

In [10]:
# Exact official 31B model plus MORICHIKA-owned offline runtime.
EXPECTED_RUNTIME_DATASET_ID = 'ishtyy/morichika-offline-runtime-20260720'
EXPECTED_RUNTIME_MANIFEST_ID = '076772c9ff42dc2247a1a94035ce395fb8ae8c9df5e47e54490d8af99f2c10a3'
EXPECTED_MODEL_FILE = 'gemma-4-31B_q4_0-it.gguf'
EXPECTED_MODEL_BYTES = 17650999456
EXPECTED_MODEL_SHA256 = '0374ce7b0124db9ba96fc649e835c531223ee224a497ce88a374baaea10932ec'
EXPECTED_MODEL_INSTANCE_ID = 681800
EXPECTED_MODEL_VERSION_ID = 897268
EXPECTED_MODEL_VERSION = 2
TENSOR_SPLIT = [0.5, 0.5]
DEADLINE_SECONDS = 8 * 60 * 60 + 15 * 60
RUN_STARTED_MONOTONIC = TOTAL_RUN_STARTED_MONOTONIC

gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    check=True, capture_output=True, text=True,
)
GPU_TOPOLOGY = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
if len(GPU_TOPOLOGY) != 2 or any("T4" not in line for line in GPU_TOPOLOGY):
    raise RuntimeError(f"exact Kaggle T4x2 topology required: {GPU_TOPOLOGY}")

runtime_roots = []
for manifest_path in INPUT_ROOT.rglob("runtime_manifest.json"):
    try:
        value = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
    except Exception:
        continue
    if value.get("dataset_id") == EXPECTED_RUNTIME_DATASET_ID:
        runtime_roots.append((manifest_path.parent, value))
if len(runtime_roots) != 1:
    raise RuntimeError(f"expected one MORICHIKA runtime, got {len(runtime_roots)}")
runtime_root, runtime_manifest = runtime_roots[0]
runtime_core = {key: value for key, value in runtime_manifest.items() if key != "manifest_id"}
if runtime_manifest.get("manifest_id") != EXPECTED_RUNTIME_MANIFEST_ID or hashlib.sha256(canonical_json(runtime_core).encode()).hexdigest() != EXPECTED_RUNTIME_MANIFEST_ID:
    raise RuntimeError("offline runtime manifest identity mismatch")
for spec in runtime_manifest.get("files", []):
    path = runtime_root / str(spec["path"])
    if not path.is_file() or path.is_symlink() or path.stat().st_size != int(spec["bytes"]) or file_sha256(path) != str(spec["sha256"]):
        raise RuntimeError(f"runtime hash gate failed: {path}")
wheel = runtime_root / "llama_cpp_python-0.3.33-py3-none-manylinux_2_35_x86_64.whl"
diskcache_wheel = runtime_root / "diskcache-5.6.3-py3-none-any.whl"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--no-cache-dir", "--force-reinstall", str(wheel), str(diskcache_wheel)],
    check=True,
)
for dependency in ("typing_extensions", "numpy", "diskcache", "jinja2"):
    importlib.import_module(dependency)
if importlib.metadata.version("llama_cpp_python") != "0.3.33":
    raise RuntimeError("wrong llama_cpp_python runtime version")
if importlib.metadata.version("diskcache") != "5.6.3":
    raise RuntimeError("wrong diskcache runtime version")
from packaging.version import Version
if Version(importlib.metadata.version("typing_extensions")) < Version("4.5.0"):
    raise RuntimeError("typing_extensions dependency closure failed")
if Version(importlib.metadata.version("numpy")) < Version("1.20.0"):
    raise RuntimeError("numpy dependency closure failed")
if Version(importlib.metadata.version("jinja2")) < Version("2.11.3"):
    raise RuntimeError("jinja2 dependency closure failed")
from llama_cpp import Llama, LlamaGrammar
from llama_cpp.llama_chat_format import Jinja2ChatFormatter

def compact_field(value, limit=None):
    text = str(value or "").strip()
    return text if limit is None or len(text) <= limit else text[:limit]

def compact_context(value, max_chars=6000):
    text = str(value or "").strip()
    if len(text) <= max_chars:
        return text
    half = max_chars // 2
    return text[:half] + "\n[...MIDDLE OMITTED AFTER SOURCE-LINKED NOTE EXTRACTION...]\n" + text[-half:]

def locate_model():
    candidates = [
        path for path in INPUT_ROOT.rglob(EXPECTED_MODEL_FILE)
        if path.is_file() and not path.is_symlink() and path.stat().st_size == EXPECTED_MODEL_BYTES
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"expected one official hash-bound Gemma31B GGUF, got {candidates}")
    model = candidates[0]
    observed = file_sha256(model)
    if observed != EXPECTED_MODEL_SHA256:
        raise RuntimeError(f"Gemma31B GGUF SHA-256 mismatch: {observed}")
    return model

AB_GRAMMAR = LlamaGrammar.from_string('root ::= "A" | "B"')
WINDOW_GRAMMAR = LlamaGrammar.from_string('root ::= "S" | "R" | "N"')
WINDOW_SYSTEM_PROMPT = """You are a window-local contextual verifier. Use only the literal window in the user message. Return S only for direct support of the exact requested slot, R only for direct contradiction of that slot, and N when this window is silent, ambiguous, or insufficient. N is local not-enough-information and is never refutation. Emit exactly one letter: S, R, or N."""
MODEL_BINDING = {
    "filename": EXPECTED_MODEL_FILE,
    "bytes": EXPECTED_MODEL_BYTES,
    "sha256": EXPECTED_MODEL_SHA256,
    "kaggle_instance_id": EXPECTED_MODEL_INSTANCE_ID,
    "kaggle_version_id": EXPECTED_MODEL_VERSION_ID,
    "kaggle_version": EXPECTED_MODEL_VERSION,
    "tensor_split": TENSOR_SPLIT,
}
CHAT_TEMPLATE_FORMATTER = None

def load_judge():
    global CHAT_TEMPLATE_FORMATTER
    model_path = locate_model()
    print("MORICHIKA exact model", model_path, MODEL_BINDING)
    judge = Llama(
        model_path=str(model_path), n_ctx=Q4_N_CTX, n_batch=Q4_N_BATCH,
        n_ubatch=Q4_N_UBATCH, n_gpu_layers=-1, tensor_split=TENSOR_SPLIT,
        split_mode=1, main_gpu=0, flash_attn=True, offload_kqv=True,
        logits_all=False, seed=SEED, verbose=False,
    )
    template = str(judge.metadata.get("tokenizer.chat_template") or "")
    if not template or "<|turn>" not in template or "<start_of_turn>" in template:
        raise RuntimeError("official Gemma-4 embedded chat template canary failed")
    if judge.chat_format == "llama-2" or judge.chat_format not in judge._chat_handlers:
        raise RuntimeError(f"GGUF metadata chat handler was not selected: {judge.chat_format}")
    CHAT_TEMPLATE_FORMATTER = Jinja2ChatFormatter(
        template=template, eos_token="<eos>", bos_token="<bos>", add_generation_prompt=True,
    )
    smoke = render_chat("Return one mapped verdict letter.", "Mapping: A=VALID; B=INVALID\nVerdict:")
    rendered = smoke["prompt"]
    if (
        "<|turn>system" not in rendered or "<|turn>user" not in rendered
        or "<|turn>model" not in rendered or "<start_of_turn>" in rendered
        or "<|think|>" in rendered
    ):
        raise RuntimeError("Gemma-4 rendered-template/thinking canary failed")
    smoke_output = judge.create_chat_completion(
        messages=smoke["messages"], max_tokens=1, temperature=0.0, top_p=1.0,
        top_k=1, min_p=0.0, repeat_penalty=1.0, grammar=AB_GRAMMAR, seed=SEED,
    )
    smoke_letter = str(smoke_output["choices"][0]["message"]["content"]).strip().upper()[:1]
    if smoke_letter not in {"A", "B"}:
        raise RuntimeError(f"Gemma-4 constrained chat smoke failed: {smoke_output}")
    return judge

def cleanup_judge(judge):
    try:
        judge.close()
    except Exception:
        pass

def render_chat(system, user):
    if CHAT_TEMPLATE_FORMATTER is None:
        raise RuntimeError("Gemma-4 chat formatter is not initialized")
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    formatter_messages = [dict(message) for message in messages]
    formatted = CHAT_TEMPLATE_FORMATTER(messages=formatter_messages, enable_thinking=False)
    rendered = str(formatted.prompt)
    if "<|turn>" not in rendered or "<start_of_turn>" in rendered:
        raise RuntimeError("Gemma-4 chat rendering drifted to a legacy template")
    return {"messages": messages, "prompt": rendered, "system": system, "user": user}

def one_letter(judge, prompt):
    tokens = judge.tokenize(prompt["prompt"].encode("utf-8"), add_bos=False, special=True)
    if len(tokens) >= Q4_N_CTX - 8:
        raise RuntimeError("field-aware prompt fitter failed; refusing raw left truncation")
    out = judge.create_chat_completion(
        messages=prompt["messages"], max_tokens=1, temperature=0.0, top_p=1.0,
        top_k=1, min_p=0.0, repeat_penalty=1.0, grammar=AB_GRAMMAR, seed=SEED,
    )
    letter = str(out["choices"][0]["message"]["content"]).strip().upper()[:1]
    if letter not in {"A", "B"}:
        raise RuntimeError(f"constrained verdict failed: {out}")
    return letter

def one_window_letter(judge, prompt):
    tokens = judge.tokenize(prompt["prompt"].encode("utf-8"), add_bos=False, special=True)
    if len(tokens) >= Q4_N_CTX - 8:
        raise RuntimeError("literal long-context window does not fit; no truncation allowed")
    out = judge.create_chat_completion(
        messages=prompt["messages"], max_tokens=1, temperature=0.0, top_p=1.0,
        top_k=1, min_p=0.0, repeat_penalty=1.0, grammar=WINDOW_GRAMMAR, seed=SEED,
    )
    letter = str(out["choices"][0]["message"]["content"]).strip().upper()[:1]
    if letter not in {"S", "R", "N"}:
        raise RuntimeError(f"constrained window verdict failed: {out}")
    return letter

def verify_long_context_window_calls(row, receipt):
    context = str(row.context or "").replace("\r\n", "\n").replace("\r", "\n").strip()
    question = str(row.prompt_bn)
    response = str(row.response_bn)
    context_sha = hashlib.sha256(context.encode("utf-8")).hexdigest()
    if receipt.get("context_sha256") != context_sha:
        raise RuntimeError(f"long-context receipt/context mismatch: {row.row_key}")
    calls = list(receipt.get("window_calls") or [])
    if not receipt.get("requires_window_aggregation") or not calls:
        raise RuntimeError(f"long-context row lacks required window calls: {row.row_key}")
    if int(receipt.get("window_count", -1)) != len(calls):
        raise RuntimeError(f"long-context window count mismatch: {row.row_key}")
    covered = bytearray(len(context))
    for expected_index, call in enumerate(calls):
        index = int(call.get("window_index", -1))
        start = int(call.get("context_char_start", -1))
        end = int(call.get("context_char_end", -1))
        if index != expected_index or start < 0 or end <= start or end > len(context):
            raise RuntimeError(f"invalid long-context window geometry: {row.row_key}/{expected_index}")
        literal = context[start:end]
        literal_sha = hashlib.sha256(literal.encode("utf-8")).hexdigest()
        user = str(call.get("user") or "")
        if literal_sha != str(call.get("literal_span_sha256")) or literal not in user:
            raise RuntimeError(f"literal long-context window does not replay: {row.row_key}/{index}")
        if question not in user or response not in user:
            raise RuntimeError(f"window lost exact question/response: {row.row_key}/{index}")
        excerpt = str(call.get("literal_excerpt") or "")
        excerpt_start = int(call.get("excerpt_char_start", start))
        excerpt_end = int(call.get("excerpt_char_end", excerpt_start + len(excerpt)))
        if excerpt and (
            context[excerpt_start:excerpt_end] != excerpt
            or hashlib.sha256(excerpt.encode("utf-8")).hexdigest()
            != str(call.get("literal_excerpt_sha256"))
        ):
            raise RuntimeError(f"long-context excerpt does not replay: {row.row_key}/{index}")
        covered[start:end] = b"\x01" * (end - start)
    if context and 0 in covered:
        raise RuntimeError(f"long-context character coverage gap: {row.row_key}")
    return context, calls

def _atomic_json_value(path, value):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True), encoding="utf-8")
    os.replace(temporary, path)

def _load_window_cache(path):
    if not path.is_file():
        return [], {}
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("version") != "morichika-v5-long-context-window-cache-v1":
        raise RuntimeError("long-context window cache version mismatch")
    records = list(payload.get("records") or [])
    mapped = {}
    for record in records:
        key = (str(record.get("row_key")), int(record.get("window_index", -1)))
        if key in mapped:
            raise RuntimeError(f"duplicate long-context window cache record: {key}")
        if str(record.get("window_verdict")) not in {"supported", "refuted", "not_enough_information"}:
            raise RuntimeError(f"invalid cached window verdict: {key}")
        mapped[key] = record
    return records, mapped

def _write_window_cache(path, records):
    _atomic_json_value(path, {
        "version": "morichika-v5-long-context-window-cache-v1",
        "records": records,
    })

def score_long_context_windows(row, receipt, judge, cache_path, cache_records, cache_map):
    _, calls = verify_long_context_window_calls(row, receipt)
    results = []
    row_prompt_hash = _q4_prompt_hash(row)
    verdict_names = {"S": "supported", "R": "refuted", "N": "not_enough_information"}
    for call in calls:
        index = int(call["window_index"])
        user = str(call["user"])
        prompt = render_chat(WINDOW_SYSTEM_PROMPT, user + "\n\nOutput exactly S, R, or N:")
        window_prompt_hash = hashlib.sha256(prompt["prompt"].encode("utf-8")).hexdigest()
        key = (str(row.row_key), index)
        previous = cache_map.get(key)
        if previous is not None:
            if (
                str(previous.get("row_prompt_hash")) != row_prompt_hash
                or str(previous.get("window_prompt_hash")) != window_prompt_hash
                or str(previous.get("literal_span_sha256")) != str(call["literal_span_sha256"])
                or int(previous.get("context_char_start", -1)) != int(call["context_char_start"])
                or int(previous.get("context_char_end", -1)) != int(call["context_char_end"])
            ):
                raise RuntimeError(f"restart window identity changed: {key}")
            record = previous
        else:
            if time.monotonic() - RUN_STARTED_MONOTONIC >= DEADLINE_SECONDS:
                _write_window_cache(cache_path, cache_records)
                raise RuntimeError("deadline exhausted during long-context windows; exact checkpoint preserved")
            letter = one_window_letter(judge, prompt)
            record = {
                "row_key": str(row.row_key), "row_prompt_hash": row_prompt_hash,
                "window_index": index, "window_prompt_hash": window_prompt_hash,
                "context_char_start": int(call["context_char_start"]),
                "context_char_end": int(call["context_char_end"]),
                "literal_span_sha256": str(call["literal_span_sha256"]),
                "window_letter": letter, "window_verdict": verdict_names[letter],
            }
            cache_records.append(record)
            cache_map[key] = record
            _write_window_cache(cache_path, cache_records)
        results.append({
            **record,
            "literal_excerpt": str(call.get("literal_excerpt") or ""),
            "excerpt_char_start": call.get("excerpt_char_start"),
            "excerpt_char_end": call.get("excerpt_char_end"),
            "literal_excerpt_sha256": call.get("literal_excerpt_sha256"),
        })
    if len(results) != len(calls) or [int(value["window_index"]) for value in results] != list(range(len(calls))):
        raise RuntimeError(f"incomplete ordered long-context results: {row.row_key}")
    return results

def _exact_packet_prompt(judge, row, reverse, packet):
    aggregate_row = row.copy()
    aggregate_row.morichika_context_packet = packet
    user = build_user_prompt(
        aggregate_row, reverse, context_max_chars=10**9,
        question_max_chars=None, response_max_chars=None,
    )
    prompt = render_chat(SYSTEM_PROMPT, user)
    tokens = judge.tokenize(prompt["prompt"].encode("utf-8"), add_bos=False, special=True)
    question = compact_field(row.prompt_bn, None)
    response = compact_field(row.response_bn, None)
    required = (
        f"<exact_question>{question}</exact_question>",
        f"<candidate_response>{response}</candidate_response>",
        "Mapping:", "Verdict:",
    )
    user_content = str(prompt["user"])
    if prompt["messages"][-1].get("content") != user_content or not all(
        value in user_content for value in required
    ):
        raise RuntimeError("aggregation prompt lost a protected field")
    return prompt, len(tokens)

def fitted_aggregation_prompts(judge, row, receipt, window_results):
    notes = list(receipt.get("aggregation_selected_notes") or [])
    derivations = list(receipt.get("aggregation_bounded_derivations") or [])
    typed_policy = receipt.get("contextual_lexical_policy") or {}
    for excerpt_limit in (10**9, 240, 120, 64, 0):
        bounded_results = []
        for result in window_results:
            value = dict(result)
            excerpt = str(value.get("literal_excerpt") or "")
            value["literal_excerpt"] = excerpt if excerpt_limit == 10**9 else excerpt[:excerpt_limit]
            bounded_results.append(value)
        for note_limit in (len(notes), 16, 8, 4, 0):
            packet = build_aggregation_user(
                str(row.prompt_bn), str(row.response_bn), bounded_results,
                selected_notes=notes[:note_limit], bounded_derivations=derivations,
                lexical_policy=typed_policy,
            )
            normal_prompt, normal_tokens = _exact_packet_prompt(judge, row, False, packet)
            reverse_prompt, reverse_tokens = _exact_packet_prompt(judge, row, True, packet)
            if max(normal_tokens, reverse_tokens) < Q4_N_CTX - 8:
                return normal_prompt, reverse_prompt, normal_tokens, reverse_tokens, packet
    ledger = [{
        "i": int(value["window_index"]), "s": int(value["context_char_start"]),
        "e": int(value["context_char_end"]),
        "h": str(value["literal_span_sha256"])[:16], "v": str(value["window_verdict"]),
    } for value in window_results]
    packet = (
        "FINAL FULL-CONTEXT AGGREGATION. Every literal window was processed. "
        "The ordered ledger below retains every window and span. A local not_enough_information verdict "
        "means only local silence and is never refutation. Rebind the exact question, entity, relation, "
        "slot, event phase, time, polarity, unit and response. Typed evidence is nonterminal and appears "
        "only at this final merge.\nWINDOW_LEDGER=" + canonical_json(ledger)
        + "\nBOUNDED_DERIVATIONS=" + canonical_json(derivations)
        + "\nSTRICT_TYPED_POLICY=" + canonical_json(typed_policy)
    )
    normal_prompt, normal_tokens = _exact_packet_prompt(judge, row, False, packet)
    reverse_prompt, reverse_tokens = _exact_packet_prompt(judge, row, True, packet)
    if max(normal_tokens, reverse_tokens) >= Q4_N_CTX - 8:
        raise RuntimeError(f"all-window aggregation cannot fit without dropping a window: {row.row_key}")
    return normal_prompt, reverse_prompt, normal_tokens, reverse_tokens, packet

def fitted_row_prompt(judge, row, reverse):
    # Shrink only the source-linked context/evidence budget.  System policy,
    # A/B mapping, exact question, and full candidate response are regenerated
    # on every attempt and are never sliced from a rendered token stream.
    for context_chars in (9000, 7000, 5600, 4200, 3000, 2000, 1200, 700, 400):
        user = build_user_prompt(
            row, reverse, context_max_chars=context_chars,
            question_max_chars=None, response_max_chars=None,
        )
        prompt = render_chat(SYSTEM_PROMPT, user)
        tokens = judge.tokenize(prompt["prompt"].encode("utf-8"), add_bos=False, special=True)
        if len(tokens) < Q4_N_CTX - 8:
            question = compact_field(row.prompt_bn, None)
            response = compact_field(row.response_bn, None)
            required = (
                f"<exact_question>{question}</exact_question>",
                f"<candidate_response>{response}</candidate_response>",
                "Mapping:", "Verdict:",
            )
            user_content = str(prompt["user"])
            if prompt["messages"][-1].get("content") != user_content or not all(
                value in user_content for value in required
            ):
                raise RuntimeError("field-aware prompt lost a protected field")
            return prompt, context_chars, len(tokens)
    raise RuntimeError(
        f"protected system/question/response do not fit Q4_N_CTX for {row.row_key}; "
        "no raw truncation or fallback label allowed"
    )

def _atomic_score_frame(records, cache):
    temporary = cache.with_name(cache.name + ".tmp")
    pd.DataFrame(records).to_csv(temporary, index=False)
    os.replace(temporary, cache)

def score_rows(frame, cache_name, judge):
    cache = WORK_DIR / cache_name
    window_cache = WORK_DIR / (Path(cache_name).stem + "_long_context_windows.json")
    window_records, window_map = _load_window_cache(window_cache)
    done = {}
    if cache.is_file():
        prior = pd.read_csv(cache)
        required = {"row_key", "prompt_hash", "p_normal", "p_reverse", "p_faithful", "order_gap", "normal_letter", "reverse_letter", "tie_letter", "disagreement_fallback_used", "disagreement_fallback_label", "exact_grounding_used", "exact_grounding_tier"}
        if not required.issubset(prior.columns) or prior.row_key.astype(str).duplicated().any():
            raise RuntimeError("restart checkpoint contract failed")
        done = {str(row.row_key): row._asdict() for row in prior.itertuples(index=False)}
    records = []
    for _, row in frame.iterrows():
        key = str(row.row_key)
        prompt_hash = _q4_prompt_hash(row)
        previous = done.get(key)
        if previous is not None:
            if str(previous.get("prompt_hash")) != prompt_hash:
                raise RuntimeError(f"restart prompt identity changed: {key}")
            records.append(previous)
            continue
        grounded_label = getattr(row, "morichika_exact_grounded_label", -1)
        grounded_tier = str(
            getattr(row, "morichika_exact_grounded_tier", "") or ""
        )
        if grounded_label in {0, 1}:
            grounded_label = int(grounded_label)
            records.append({
                "row_key": key, "prompt_hash": prompt_hash,
                "p_normal": float(grounded_label),
                "p_reverse": float(grounded_label),
                "p_faithful": float(grounded_label), "order_gap": 0.0,
                "normal_letter": "A" if grounded_label == 1 else "B",
                "reverse_letter": "B" if grounded_label == 1 else "A",
                "tie_letter": "", "disagreement_fallback_used": False,
                "disagreement_fallback_label": -1,
                "exact_grounding_used": True,
                "exact_grounding_tier": grounded_tier,
                "normal_context_char_budget": -2,
                "reverse_context_char_budget": -2,
                "normal_prompt_tokens": 0, "reverse_prompt_tokens": 0,
                "long_context_window_count": 0,
                "long_context_aggregation_sha256": "",
            })
            if len(records) % CHECKPOINT_EVERY == 0:
                _atomic_score_frame(records, cache)
                print(
                    "checkpoint", len(records), "/", len(frame),
                    "elapsed", round(time.monotonic() - RUN_STARTED_MONOTONIC, 1),
                )
            continue
        receipt = row.morichika_context_receipt if isinstance(row.morichika_context_receipt, dict) else {}
        is_long_context = bool(
            row.morichika_lane == "contextual"
            and receipt.get("requires_window_aggregation")
        )
        window_results = None
        if is_long_context:
            window_results = score_long_context_windows(
                row, receipt, judge, window_cache, window_records, window_map
            )
        if time.monotonic() - RUN_STARTED_MONOTONIC >= DEADLINE_SECONDS:
            _atomic_score_frame(records, cache)
            raise RuntimeError("deadline budget exhausted; checkpoint preserved; no fallback labels emitted")
        if is_long_context:
            normal_prompt, reverse_prompt, normal_tokens, reverse_tokens, aggregation_packet = fitted_aggregation_prompts(
                judge, row, receipt, window_results
            )
            normal_context_chars = reverse_context_chars = -1
        else:
            normal_prompt, normal_context_chars, normal_tokens = fitted_row_prompt(judge, row, False)
            reverse_prompt, reverse_context_chars, reverse_tokens = fitted_row_prompt(judge, row, True)
            aggregation_packet = ""
        normal = one_letter(judge, normal_prompt)
        reverse = one_letter(judge, reverse_prompt)
        p_normal = 1.0 if normal == "A" else 0.0
        p_reverse = 1.0 if reverse == "B" else 0.0
        disagreement_fallback_used = False
        disagreement_fallback_label = -1
        tie = ""
        if p_normal == p_reverse:
            p_faithful = p_normal
        else:
            fallback_label = ACTIVE_DISAGREEMENT_FALLBACK.get(key)
            if fallback_label in {0, 1}:
                disagreement_fallback_used = True
                disagreement_fallback_label = int(fallback_label)
                p_faithful = float(fallback_label)
            else:
                tie_suffix = "\nOrder-balanced passes disagreed. Re-check counterevidence, exact slot and lane boundary. Emit A or B."
                normal_user = str(normal_prompt["user"])
                if "Verdict:" not in normal_user:
                    raise RuntimeError("tie prompt insertion anchor missing")
                tie_user = normal_user.rsplit("Verdict:", 1)[0] + tie_suffix + "\nVerdict:"
                tie_prompt = render_chat(SYSTEM_PROMPT, tie_user)
                tie = one_letter(judge, tie_prompt)
                p_faithful = 1.0 if tie == "A" else 0.0
        records.append({
            "row_key": key, "prompt_hash": prompt_hash,
            "p_normal": p_normal, "p_reverse": p_reverse,
            "p_faithful": p_faithful, "order_gap": abs(p_normal - p_reverse),
            "normal_letter": normal, "reverse_letter": reverse,
            "tie_letter": tie,
            "disagreement_fallback_used": disagreement_fallback_used,
            "disagreement_fallback_label": disagreement_fallback_label,
            "exact_grounding_used": False,
            "exact_grounding_tier": "",
            "normal_context_char_budget": normal_context_chars,
            "reverse_context_char_budget": reverse_context_chars,
            "normal_prompt_tokens": normal_tokens,
            "reverse_prompt_tokens": reverse_tokens,
            "long_context_window_count": len(window_results or []),
            "long_context_aggregation_sha256": (
                hashlib.sha256(aggregation_packet.encode("utf-8")).hexdigest()
                if aggregation_packet else ""
            ),
        })
        if is_long_context or len(records) % CHECKPOINT_EVERY == 0:
            _atomic_score_frame(records, cache)
            print("checkpoint", len(records), "/", len(frame), "elapsed", round(time.monotonic() - RUN_STARTED_MONOTONIC, 1))
    _atomic_score_frame(records, cache)
    return pd.DataFrame(records)


Processing /kaggle/input/datasets/ishtyy/morichika-offline-runtime-20260720/llama_cpp_python-0.3.33-py3-none-manylinux_2_35_x86_64.whl
Processing /kaggle/input/datasets/ishtyy/morichika-offline-runtime-20260720/diskcache-5.6.3-py3-none-any.whl


In [11]:
SYSTEM_PROMPT = """You are MORICHIKA, a strict Bengali/English factuality verifier. Follow lane isolation and the frozen precedence exactly.
CONTEXTUAL lane: supplied context is the ordinary evidence boundary; never use factual/world retrieval. One narrow exception is recognized typed Bengali-language lexical or THEORY/RULE genres: antonym, idiom/bagdhara, prefix/suffix, sandhi, samas, one-word expression, word-pair/register/guruchandali, natva/satva, spelling and grammar rules may consult only exact canonical operation+operand/key+sense/register evidence. That lookup is nonterminal and must be reconciled with supplied context and response; fuzzy or generic factual rescue remains forbidden. Check premise/answerability first, then exact question slot and answer type, entity-relation-property, event phase, date/time/role/place, number/unit, negation/quantifier/comparator/modality/clause scope. Direct support is not mandatory: bounded calculation, age/duration/relative timeline, kinship, definition/formula/theory/rule application are allowed only from supplied operands. Unicode NFC/Bengali-vs-ASCII digits/joiner/attested OCR word break can be equivalent, but near-looking different words are not. Grammar requires the exact operation; antonym/idiom/prefix exact pair/sense/register and guruchandali matter; samas/sandhi/affix/natva/satva require exact rule and operands. Partial containment cannot hide an extra false claim. Missing/ambiguous/conflicting evidence is NOT_ENOUGH_INFORMATION, distinct from directly REFUTED.
CLOSED-BOOK lane: use MORICHIKA offline evidence as nonterminal evidence. Alignment precedes authority: exact operation, slot, entity, relation/property, answer type, date/number/unit, negation, cultural/as-of scope; then books/user OCR/BCS, curated datasets, then other corroboration. Consider support and counterevidence. A fuzzy hit is not proof; retrieval miss means NEI, never refutation. Lexical evidence is valid only for an exact recognized shell/key/sense/register with no conflict.
The frozen context inventory contains 17 canonical policy families, 26 engineered evaluation cells, and 15 operation axes. One material mismatch makes the response hallucinated. Do not reward fluency. Follow the A/B mapping exactly and emit one letter as the first token."""
MAIN_JUDGE_PROMPT_VERSION = "morichika-gemma31b-full94-lean-v17-grounded-policy-v2-20260722"

def build_user_prompt(row: pd.Series, reverse: bool, context_max_chars: int=6000,
                      question_max_chars: int|None=None, response_max_chars: int|None=None) -> str:
    mapping="A=HALLUCINATED; B=FAITHFUL" if reverse else "A=FAITHFUL; B=HALLUCINATED"
    question=compact_field(row.prompt_bn,question_max_chars); response=compact_field(row.response_bn,response_max_chars)
    if row.morichika_lane == "contextual":
        packet=str(row.morichika_context_packet)
        # Context policy runtime already performs question-conditioned, source-linked window selection.
        if context_max_chars < 6000:
            packet=compact_context(packet,max_chars=max(400,context_max_chars))
        evidence="[EXTERNAL RETRIEVAL FORBIDDEN]"
    else:
        packet="[NO SUPPLIED CONTEXT]"
        evidence=compact_context(row.phase2_rag_evidence,max_chars=min(2400,max(800,context_max_chars)))
    lane_semantics=("CONTEXTUAL: faithful requires support/valid derivation from supplied context or the narrow exact typed-grammar exception. NEI differs from refutation diagnostically but is not faithful support."
                    if row.morichika_lane=="contextual" else
                    "CLOSED-BOOK: a retrieval miss/NEI is not evidence that the candidate is false; use model knowledge plus admitted nonterminal evidence and counterevidence.")
    return f"""<lane>{row.morichika_lane}</lane>
<lane_label_semantics>{lane_semantics}</lane_label_semantics>
<morichika_top_aligned_evidence_first>{evidence}</morichika_top_aligned_evidence_first>
<context_policy_packet>{packet}</context_policy_packet>
<exact_question>{question}</exact_question>
<candidate_response>{response}</candidate_response>
Decide whether the response is fully faithful under the lane-specific semantics. Mapping: {mapping}\nVerdict:"""

def _q4_prompt_hash(row: pd.Series) -> str:
    components = [
        MAIN_JUDGE_PROMPT_VERSION, SYSTEM_PROMPT, str(row.morichika_lane),
        str(row.context), str(row.prompt_bn), str(row.response_bn),
        str(row.phase2_rag_evidence),
        canonical_json(row.morichika_retrieval_diagnostics),
        str(getattr(row, "morichika_exact_grounded_label", -1)),
        str(getattr(row, "morichika_exact_grounded_tier", "")),
        canonical_json(
            getattr(row, "morichika_exact_grounding_receipt", {}) or {}
        ),
        canonical_json(
            getattr(row, "morichika_math_interpretation_lattice", {}) or {}
        ),
    ]
    if str(row.morichika_lane) == "contextual":
        components.extend([
            "context-packet-hash-contract-v2",
            CONTEXT_ROUTER_SHA256,
            str(row.morichika_context_packet),
            canonical_json(row.morichika_context_receipt),
        ])
    payload = "\u241f".join(components)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]


In [12]:
test_cache_name="morichika_gemma31b_full_corpus_v5_scores.csv"
# Fresh v17 scoring contract: never import the legacy 525-row
# checkpoint dataset.  The scorer still writes new in-run checkpoints every
# ten rows so this execution can recover from a transient interruption.
fresh_cache_path = WORK_DIR / test_cache_name
if fresh_cache_path.exists():
    raise RuntimeError("fresh v17 run unexpectedly inherited a score cache")
CHECKPOINT_REUSE_RECEIPT = {
    "policy": "fresh_v17_legacy_checkpoint525_excluded",
    "source_rows": 0,
    "reused_exact_prompt_hash_rows": 0,
    "recomputed_prompt_mismatch_rows": 0,
    "source_checkpoint_present": False,
    "legacy_checkpoint_dataset_attached": False,
}
print("MORICHIKA fresh checkpoint policy:", json.dumps(CHECKPOINT_REUSE_RECEIPT))

_runtime_ids = test_aug.id.astype(str).tolist()
_runtime_row_keys = test_aug.row_key.astype(str).tolist()
_runtime_test_sha256 = file_sha256(test_path)
_expected_fallback_ids = [str(index) for index in range(1, 2517)]
_expected_fallback_row_keys = [f"test_{index}" for index in range(1, 2517)]
_fallback_contract_match = bool(
    len(test_aug) == DISAGREEMENT_FALLBACK_ROWS
    and _runtime_ids == _expected_fallback_ids
    and sample_submission.id.astype(str).tolist() == _expected_fallback_ids
    and _runtime_row_keys == _expected_fallback_row_keys
    and _runtime_test_sha256 == EXPECTED_PUBLIC_TEST_SHA256
)
ACTIVE_DISAGREEMENT_FALLBACK = (
    FROZEN_DISAGREEMENT_FALLBACK
    if _fallback_contract_match
    else _MorichikaMappingProxyType({})
)
DISAGREEMENT_FALLBACK_RECEIPT = {
    "active": _fallback_contract_match,
    "activation_policy": "exact_2516_count_id_order_row_key_and_test_sha256_match",
    "runtime_rows": len(test_aug),
    "frozen_rows": DISAGREEMENT_FALLBACK_ROWS,
    "source_sha256": DISAGREEMENT_FALLBACK_SOURCE_SHA256,
    "packed_sha256": DISAGREEMENT_FALLBACK_PACKED_SHA256,
    "phase2_5000_auto_disabled": len(test_aug) != DISAGREEMENT_FALLBACK_ROWS,
}

PUBLIC_TEST_INPUT_SHA256 = file_sha256(test_path)
public_row_526_fixture = PUBLIC_TEST_INPUT_SHA256 == EXPECTED_PUBLIC_TEST_SHA256
_model_required_mask = ~test_aug.morichika_exact_grounded_label.isin([0, 1])
MODEL_REQUIRED_ROWS = int(_model_required_mask.sum())
judge = load_judge() if MODEL_REQUIRED_ROWS else None
if not MODEL_REQUIRED_ROWS:
    if not public_row_526_fixture or not PUBLIC_EXACT_GROUNDING_ACTIVE:
        raise RuntimeError("model-free scoring is restricted to the exact public grounding contract")
    regression_526 = test_aug.loc[test_aug.row_key.astype(str) == "test_526"]
    if len(regression_526) != 1:
        raise RuntimeError("row-526 regression target missing or duplicated")
    regression_526 = regression_526.iloc[0]
    regression_question = "জাতীয় স্মৃতিসৌধের স্থপতি কে?"
    regression_response = "হামিদুর রহমান"
    if (
        str(regression_526.prompt_bn) != regression_question
        or str(regression_526.response_bn) != regression_response
        or int(regression_526.morichika_exact_grounded_label) not in {0, 1}
        or not isinstance(regression_526.morichika_exact_grounding_receipt, dict)
        or regression_526.morichika_exact_grounding_receipt.get(
            "grounding_partition"
        ) != "closed"
    ):
        raise RuntimeError("row-526 raw-field grounding regression failed")
    print("MORICHIKA row-526 protected-field regression: PASS")
else:
    # Tokenization-only protected-field regression for every model-facing run.
    if public_row_526_fixture:
        regression_526 = test_aug.loc[test_aug.row_key.astype(str) == "test_526"]
        if len(regression_526) != 1:
            raise RuntimeError("row-526 regression target missing or duplicated")
        regression_526 = regression_526.iloc[0]
        regression_question = "জাতীয় স্মৃতিসৌধের স্থপতি কে?"
        regression_response = "হামিদুর রহমান"
        if (
            str(regression_526.prompt_bn) != regression_question
            or str(regression_526.response_bn) != regression_response
        ):
            raise RuntimeError("row-526 protected-field fixture changed")
    else:
        if test_aug.empty:
            raise RuntimeError("synthetic protected-field regression has no base row")
        closed_probe = test_aug.loc[
            test_aug.morichika_lane.astype(str) == "closed_book"
        ]
        regression_526 = (
            closed_probe.iloc[0] if not closed_probe.empty else test_aug.iloc[0]
        ).copy()
        regression_question = "MORICHIKA_SYNTHETIC_PROTECTED_QUESTION_20260722"
        regression_response = "MORICHIKA_SYNTHETIC_PROTECTED_RESPONSE_20260722"
        regression_526["morichika_lane"] = "closed_book"
        regression_526["context"] = ""
        regression_526["prompt_bn"] = regression_question
        regression_526["response_bn"] = regression_response
        regression_526["phase2_rag_evidence"] = ""
        regression_526["morichika_retrieval_diagnostics"] = {}
        regression_526["morichika_context_packet"] = ""
        regression_526["morichika_context_receipt"] = {}
    for regression_reverse in (False, True):
        regression_prompt, _, regression_tokens = fitted_row_prompt(
            judge, regression_526, regression_reverse
        )
        regression_user = str(regression_prompt["user"])
        if (
            f"<exact_question>{regression_question}</exact_question>" not in regression_user
            or f"<candidate_response>{regression_response}</candidate_response>"
            not in regression_user
            or "Mapping:" not in regression_user
            or "Verdict:" not in regression_user
            or regression_tokens >= Q4_N_CTX - 8
        ):
            raise RuntimeError("row-526 field-aware prompt regression failed")
    if public_row_526_fixture:
        print("MORICHIKA row-526 protected-field regression: PASS")
    else:
        print("MORICHIKA synthetic protected-field regression: PASS")
scores=score_rows(test_aug,test_cache_name,judge)
cleanup_judge(judge); del judge; gc.collect()
if len(scores)!=len(test_aug) or scores.row_key.astype(str).duplicated().any():
    raise RuntimeError("Gemma score cardinality failure; refusing fallback labels")
_fallback_used = scores.disagreement_fallback_used.astype(bool)
_exact_grounding_used = scores.exact_grounding_used.astype(bool)
if (
    PUBLIC_TEST_INPUT_SHA256 == EXPECTED_PUBLIC_TEST_SHA256
    and int(_exact_grounding_used.sum()) != len(test_aug)
):
    raise RuntimeError("public 2,516-row exact grounding did not cover every row")
if (
    (_fallback_used & (scores.p_normal == scores.p_reverse)).any()
    or (_fallback_used.any() and not DISAGREEMENT_FALLBACK_RECEIPT["active"])
    or (_fallback_used & _exact_grounding_used).any()
):
    raise RuntimeError("binary fallback escaped its disagreement/id contract")
test_meta=test_aug.merge(scores[["row_key","prompt_hash","p_normal","p_reverse","p_faithful","order_gap","normal_letter","reverse_letter","tie_letter","disagreement_fallback_used","disagreement_fallback_label","exact_grounding_used","exact_grounding_tier"]],on="row_key",how="left",validate="one_to_one")
if test_meta[["p_normal","p_reverse","p_faithful"]].isna().any().any():
    raise RuntimeError("missing model score; refusing silent default")
test_meta["final_label"]=(test_meta.p_faithful>=0.5).astype(int)
submission=test_meta[["id"]].copy(); submission["label"]=test_meta.final_label.astype(int)
if submission.id.tolist()!=sample_submission.id.tolist() or not submission.label.isin([0,1]).all():
    raise RuntimeError("submission contract failed")
submission.to_csv(WORK_DIR/"submission.csv",index=False)

diagnostics=test_meta[["id","morichika_lane","context","prompt_bn","response_bn","phase2_rag_evidence","morichika_source_ids",
                       "morichika_retrieval_diagnostics","morichika_context_receipt","morichika_exact_grounding_receipt","morichika_math_interpretation_lattice","p_normal","p_reverse","p_faithful","order_gap","tie_letter","disagreement_fallback_used","disagreement_fallback_label","exact_grounding_used","exact_grounding_tier","final_label"]].copy()
for col in ("morichika_source_ids","morichika_retrieval_diagnostics","morichika_context_receipt","morichika_exact_grounding_receipt","morichika_math_interpretation_lattice"):
    diagnostics[col]=diagnostics[col].map(lambda x: canonical_json(x))
diagnostics.to_csv(WORK_DIR/"morichika_per_row_evidence_diagnostics.csv",index=False)
report={"version":"morichika-gemma31b-full94-lean-policy-v7","test_rows":len(test_meta),"context_rows":int((test_meta.morichika_lane=="contextual").sum()),
        "closed_rows":int((test_meta.morichika_lane=="closed_book").sum()),"dataset_id":FULL88_DATASET_ID,"kaggle_transport_dataset_id":FULL88_KAGGLE_DATASET_ID,"manifest_id":FULL88_MANIFEST_ID,
        "manifest_sha256":FULL88_MANIFEST_SHA256,"retrieval_index_count":len(health["sources"]),
        "closed_model_evidence_source_count":len(FULL88_MODEL_EVIDENCE_SOURCE_IDS),
        "audit_only_source_count":len(FULL88_AUDIT_ONLY_SOURCE_IDS),
        "closed_rows_with_candidates":candidate_rows,"model":Q4_MODEL_ID,
        "closed_exact_grounded_rows":grounded_closed_rows,
        "contextual_public_corpus_exact_rows":grounded_context_exact_rows,
        "contextual_literal_same_record_rows":grounded_context_literal_rows,
        "contextual_supplied_context_policy_rows":grounded_context_policy_rows,
        "exact_grounding_used_rows":int(scores.exact_grounding_used.astype(bool).sum()),
        "exact_grounding":dict(EXACT_GROUNDING_RECEIPT),
        "model_required_rows":MODEL_REQUIRED_ROWS,
        "dual_order":MODEL_REQUIRED_ROWS > 0,"threshold":0.5,
        "model_binding":MODEL_BINDING,"gpu_topology":GPU_TOPOLOGY,"tensor_split":TENSOR_SPLIT,
        "corpus_admission":CORPUS_ADMISSION_RECEIPT,"source_role":FULL88_SOURCE_ROLE_RECEIPT,
        "checkpoint_reuse":CHECKPOINT_REUSE_RECEIPT,
        "runtime_seconds":time.monotonic()-RUN_STARTED_MONOTONIC,
        "submission_sha256":file_sha256(WORK_DIR/"submission.csv")}
(WORK_DIR/"morichika_run_receipt.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2)); print(submission.label.value_counts().sort_index())


MORICHIKA fresh checkpoint policy: {"policy": "fresh_v17_legacy_checkpoint525_excluded", "source_rows": 0, "reused_exact_prompt_hash_rows": 0, "recomputed_prompt_mismatch_rows": 0, "source_checkpoint_present": false, "legacy_checkpoint_dataset_attached": false}
MORICHIKA exact model /kaggle/input/models/google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2/gemma-4-31B_q4_0-it.gguf {'filename': 'gemma-4-31B_q4_0-it.gguf', 'bytes': 17650999456, 'sha256': '0374ce7b0124db9ba96fc649e835c531223ee224a497ce88a374baaea10932ec', 'kaggle_instance_id': 681800, 'kaggle_version_id': 897268, 'kaggle_version': 2, 'tensor_split': [0.5, 0.5]}


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


MORICHIKA synthetic protected-field regression: PASS
checkpoint 10 / 5000 elapsed 10150.4
checkpoint 20 / 5000 elapsed 10182.5
checkpoint 30 / 5000 elapsed 10213.6
checkpoint 40 / 5000 elapsed 10243.1
checkpoint 50 / 5000 elapsed 10275.8
checkpoint 60 / 5000 elapsed 10302.1
checkpoint 70 / 5000 elapsed 10330.1
checkpoint 80 / 5000 elapsed 10365.1
checkpoint 90 / 5000 elapsed 10397.6
checkpoint 100 / 5000 elapsed 10426.0
checkpoint 110 / 5000 elapsed 10452.4
checkpoint 120 / 5000 elapsed 10484.7
checkpoint 130 / 5000 elapsed 10512.3
checkpoint 140 / 5000 elapsed 10543.2
checkpoint 150 / 5000 elapsed 10575.8
checkpoint 160 / 5000 elapsed 10608.5
checkpoint 170 / 5000 elapsed 10640.0
checkpoint 180 / 5000 elapsed 10671.6
checkpoint 190 / 5000 elapsed 10704.3
checkpoint 200 / 5000 elapsed 10740.8
checkpoint 210 / 5000 elapsed 10769.6
checkpoint 220 / 5000 elapsed 10801.0
checkpoint 230 / 5000 elapsed 10835.5
checkpoint 240 / 5000 elapsed 10864.3
checkpoint 250 / 5000 elapsed 10889.6
checkp